# Procesamiento histórico del DENUE – Morelia, Michoacán

## Proyecto doctoral

Objetivo:
Procesar las distintas ediciones históricas del DENUE para reconstruir la
dinámica de los establecimientos de comercio y servicios en Morelia.

### Reglas de procesamiento

- Trabajar una edición del DENUE a la vez.
- No modificar los archivos originales.
- Utilizar la clase SCIAN de 6 dígitos como máximo nivel de desagregación.
- No interpretar automáticamente apariciones como nacimientos.
- No interpretar automáticamente ausencias como muertes.
- Documentar la versión SCIAN correspondiente a cada edición.
- Conservar los establecimientos individuales antes de realizar agregaciones.

In [62]:
from pathlib import Path
import pandas as pd
import numpy as np

print("Cuaderno DENUE listo.")

Cuaderno DENUE listo.


In [63]:
# Rutas principales del proyecto

BASE = Path.cwd()

ORIGINALES = BASE / "00_ORIGINALES"
MORELIA = BASE / "01_MORELIA_LIMPIO"
CONTEOS = BASE / "02_CONTEOS_SCIAN"
HOMOLOGACION = BASE / "03_HOMOLOGACION_SCIAN"
COMPARACIONES = BASE / "04_COMPARACIONES"
RESULTADOS = BASE / "05_RESULTADOS"

print("Carpeta de trabajo:")
print(BASE)

Carpeta de trabajo:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS


In [64]:
# ==========================================
# DENUE 2010
# Crear carpeta para archivo original
# ==========================================

CARPETA_2010 = ORIGINALES / "2010"
CARPETA_2010.mkdir(parents=True, exist_ok=True)

print("Carpeta 2010 lista:")
print(CARPETA_2010)

Carpeta 2010 lista:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\00_ORIGINALES\2010


In [65]:
import zipfile

zip_2010 = ORIGINALES / "2010" / "denue_16_2010_csv.zip"

print("ZIP encontrado:", zip_2010.exists())
print("Nombre:", zip_2010.name)

with zipfile.ZipFile(zip_2010, "r") as z:
    print("\nContenido del ZIP:")
    for nombre in z.namelist():
        print("-", nombre)

ZIP encontrado: True
Nombre: denue_16_2010_csv.zip

Contenido del ZIP:
- DenueCSV16.csv
- Diccionario_de_datos_CSV_y_DBF.pdf


In [66]:
# ==========================================
# DENUE 2010
# Inspección inicial del archivo CSV
# ==========================================

with zipfile.ZipFile(zip_2010, "r") as z:
    with z.open("DenueCSV16.csv") as archivo:
        muestra_2010 = pd.read_csv(
            archivo,
            nrows=5,
            encoding="latin1",
            dtype=str
        )

print("Número de columnas:", len(muestra_2010.columns))

print("\nNombres de las columnas:")
for columna in muestra_2010.columns:
    print("-", columna)

print("\nPrimeros 5 registros:")
display(muestra_2010)

Número de columnas: 24

Nombres de las columnas:
- Nombre de la unidad económica
- Razón social
- Código de la clase de actividad
- Nombre de la clase de actividad
- Personal ocupado (estrato)
- Calle, avenida, andador, carretera, manzana u otro
- Número exterior o km
- Edificio, piso o nivel
- Número o letra interior
- Colonia, fraccionamiento, unidad habitacional o barrio
- Corredor industrial, centro comercial o mercado público
- Número de local
- Código postal
- Entidad federativa
- Municipio
- Localidad
- Área geoestadística básica
- Manzana
- Número de teléfono
- Correo electrónico
- Sitio en Internet
- Tipo de unidad económica
- Latitud
- Longitud

Primeros 5 registros:


,Nombre de la unidad económica,Razón social,Código de la clase de actividad,Nombre de la clase de actividad,Personal ocupado (estrato),"Calle, avenida, andador, carretera, manzana u otro",Número exterior o km,"Edificio, piso o nivel",Número o letra interior,"Colonia, fraccionamiento, unidad habitacional o barrio",...,Municipio,Localidad,Área geoestadística básica,Manzana,Número de teléfono,Correo electrónico,Sitio en Internet,Tipo de unidad económica,Latitud,Longitud
0,COMITE DE AGUA POTABLE Y ALCANTARILLADO DE ACU...,COMITE DE AGUA POTABLE Y ALCANTARILLADO DE ACU...,222111,"CAPTACION, TRATAMIENTO Y SUMINISTRO DE AGUA RE...",6 A 10 PERSONAS,VICENTE RIVA PALACIO,SN,NaN,NaN,CENTRO,...,ACUITZIO,Acuitzio del Canje,0028,007,434343882750,NaN,NaN,FIJO,19.49588,-101.33301419999999
1,PALETERIA CARIBE,NaN,311520,ELABORACION DE HELADOS Y PALETAS,0 A 5 PERSONAS,VICENTE RIVA PALACIO,37,NaN,NaN,COLONIA CENTRO,...,ACUITZIO,Acuitzio del Canje,0028,006,0,NaN,NaN,FIJO,19.496142200000001,-101.333173
2,EL ANGEL,NaN,311812,PANIFICACION TRADICIONAL,0 A 5 PERSONAS,VICENTE RIVA PALACIO,133,NaN,NaN,CENTRO,...,ACUITZIO,Acuitzio del Canje,0028,001,4343882388,NaN,NaN,FIJO,19.495215900000002,-101.33330119999999
3,PANADERIA,NaN,311812,PANIFICACION TRADICIONAL,0 A 5 PERSONAS,VICENTE RIVA PALACIO,359,NaN,NaN,CENTRO,...,ACUITZIO,Acuitzio del Canje,0028,013,0,NaN,NaN,FIJO,19.492876200000001,-101.3336043
4,PANADERIA AISSE,NaN,311812,PANIFICACION TRADICIONAL,0 A 5 PERSONAS,VICENTE RIVA PALACIO,52,NaN,NaN,CENTRO,...,ACUITZIO,Acuitzio del Canje,0028,007,3882053,NaN,NaN,FIJO,19.4962403,-101.3327432


In [67]:
# ==========================================
# DENUE 2010
# Revisar municipios contenidos en el archivo
# ==========================================

with zipfile.ZipFile(zip_2010, "r") as z:
    with z.open("DenueCSV16.csv") as archivo:
        municipios_2010 = pd.read_csv(
            archivo,
            usecols=["Municipio"],
            encoding="latin1",
            dtype=str
        )

print("Total de registros en Michoacán:", len(municipios_2010))

print("\nNúmero de municipios distintos:")
print(municipios_2010["Municipio"].nunique())

print("\nCoincidencias con la palabra MORELIA:")
print(
    municipios_2010[
        municipios_2010["Municipio"]
        .str.contains("MORELIA", case=False, na=False)
    ]["Municipio"]
    .value_counts()
)

Total de registros en Michoacán: 201915

Número de municipios distintos:
113

Coincidencias con la palabra MORELIA:
Municipio
MORELIA    41342
Name: count, dtype: int64


In [68]:
# ==========================================
# DENUE 2010
# Extraer todos los establecimientos de Morelia
# ==========================================

with zipfile.ZipFile(zip_2010, "r") as z:
    with z.open("DenueCSV16.csv") as archivo:
        denue_2010 = pd.read_csv(
            archivo,
            encoding="latin1",
            dtype=str
        )

# Filtrar únicamente Morelia
filtro_morelia = (
    denue_2010["Municipio"]
    .str.strip()
    .str.upper()
    .eq("MORELIA")
)

morelia_2010 = denue_2010.loc[filtro_morelia].copy()

print("Registros originales Michoacán:", len(denue_2010))
print("Registros Morelia:", len(morelia_2010))
print("Columnas conservadas:", len(morelia_2010.columns))

# Verificación de seguridad
assert len(morelia_2010) == 41342, "ATENCIÓN: el número de registros no coincide"

print("\n✓ Verificación correcta: Morelia tiene 41,342 registros.")

Registros originales Michoacán: 201915
Registros Morelia: 41342
Columnas conservadas: 24

✓ Verificación correcta: Morelia tiene 41,342 registros.


In [69]:
# Guardar DENUE 2010 filtrado para Morelia

archivo_salida_2010 = MORELIA / "DENUE_2010_MORELIA.csv"

morelia_2010.to_csv(
    archivo_salida_2010,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado correctamente en:")
print(archivo_salida_2010)

Archivo guardado correctamente en:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2010_MORELIA.csv


In [70]:
# ==========================================
# DENUE 2010
# Conteo por clase SCIAN de 6 dígitos
# ==========================================

conteo_scian_2010 = (
    morelia_2010
    .groupby(
        ["Código de la clase de actividad",
         "Nombre de la clase de actividad"],
        dropna=False
    )
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2010 = conteo_scian_2010.sort_values(
    "Número de establecimientos",
    ascending=False
)

print(
    "Número de clases SCIAN distintas:",
    conteo_scian_2010["Código de la clase de actividad"].nunique()
)

print("\n20 clases con más establecimientos:")
display(conteo_scian_2010.head(20))

Número de clases SCIAN distintas: 646

20 clases con más establecimientos:


,Código de la clase de actividad,Nombre de la clase de actividad,Número de establecimientos
261,461110,"COMERCIO AL POR MENOR EN TIENDAS DE ABARROTES,...",4179
592,722212,RESTAURANTES DE COMIDA PARA LLEVAR,1975
593,722219,OTROS RESTAURANTES CON SERVICIO LIMITADO,1707
623,812110,SALONES Y CLINICAS DE BELLEZA Y PELUQUERIAS,1582
280,463211,"COMERCIO AL POR MENOR DE ROPA, EXCEPTO DE BEBE...",1481
265,461130,COMERCIO AL POR MENOR DE FRUTAS Y VERDURAS FRE...,1174
58,311830,ELABORACION DE TORTILLAS DE MAIZ Y MOLIENDA DE...,1069
510,621211,CONSULTORIOS DENTALES DEL SECTOR PRIVADO,913
302,465311,COMERCIO AL POR MENOR DE ARTICULOS DE PAPELERIA,849
597,811111,REPARACION MECANICA EN GENERAL DE AUTOMOVILES ...,656


In [71]:
# ==========================================
# DENUE 2010
# Identificar sector SCIAN
# ==========================================

conteo_scian_2010["Sector SCIAN"] = (
    conteo_scian_2010["Código de la clase de actividad"]
    .astype(str)
    .str[:2]
)

resumen_sector_2010 = (
    conteo_scian_2010
    .groupby("Sector SCIAN")
    .agg(
        Clases_SCIAN=("Código de la clase de actividad", "nunique"),
        Establecimientos=("Número de establecimientos", "sum")
    )
    .reset_index()
    .sort_values("Sector SCIAN")
)

display(resumen_sector_2010)

,Sector SCIAN,Clases_SCIAN,Establecimientos
0,11,5,9
1,21,1,16
2,22,2,7
3,23,30,248
4,31,56,2073
5,32,51,1180
6,33,49,1080
7,43,67,1189
8,46,80,16693
9,48,19,249


In [72]:
# ==========================================
# DENUE 2010
# Etiquetas de sectores SCIAN
# ==========================================

nombres_sector = {
    "11": "Agricultura, cría y explotación de animales, aprovechamiento forestal, pesca y caza",
    "21": "Minería",
    "22": "Generación y distribución de energía, agua y gas",
    "23": "Construcción",
    "31": "Industrias manufactureras",
    "32": "Industrias manufactureras",
    "33": "Industrias manufactureras",
    "43": "Comercio al por mayor",
    "46": "Comercio al por menor",
    "48": "Transportes, correos y almacenamiento",
    "49": "Transportes, correos y almacenamiento",
    "51": "Información",
    "52": "Servicios financieros y de seguros",
    "53": "Servicios inmobiliarios y de alquiler",
    "54": "Servicios profesionales, científicos y técnicos",
    "56": "Servicios de apoyo a los negocios",
    "61": "Servicios educativos",
    "62": "Servicios de salud y asistencia social",
    "71": "Servicios de esparcimiento, culturales y deportivos",
    "72": "Servicios de alojamiento temporal y preparación de alimentos y bebidas",
    "81": "Otros servicios excepto actividades gubernamentales",
    "93": "Actividades gubernamentales",
    "99": "Por revisar"
}

resumen_sector_2010["Nombre del sector"] = (
    resumen_sector_2010["Sector SCIAN"].map(nombres_sector)
)

display(
    resumen_sector_2010[
        ["Sector SCIAN", "Nombre del sector",
         "Clases_SCIAN", "Establecimientos"]
    ]
)


,Sector SCIAN,Nombre del sector,Clases_SCIAN,Establecimientos
0,11,"Agricultura, cría y explotación de animales, a...",5,9
1,21,Minería,1,16
2,22,"Generación y distribución de energía, agua y gas",2,7
3,23,Construcción,30,248
4,31,Industrias manufactureras,56,2073
5,32,Industrias manufactureras,51,1180
6,33,Industrias manufactureras,49,1080
7,43,Comercio al por mayor,67,1189
8,46,Comercio al por menor,80,16693
9,48,"Transportes, correos y almacenamiento",19,249


In [73]:
# ==========================================
# DENUE 2010
# Separar comercio al por menor y servicios
# ==========================================

# COMERCIO AL POR MENOR
comercio_2010 = conteo_scian_2010[
    conteo_scian_2010["Sector SCIAN"] == "46"
].copy()

# SECTORES DE SERVICIOS considerados en el estudio
sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2010 = conteo_scian_2010[
    conteo_scian_2010["Sector SCIAN"].isin(sectores_servicios)
].copy()

print("COMERCIO AL POR MENOR")
print("Clases SCIAN:", comercio_2010["Código de la clase de actividad"].nunique())
print("Establecimientos:", comercio_2010["Número de establecimientos"].sum())

print("\nSERVICIOS")
print("Clases SCIAN:", servicios_2010["Código de la clase de actividad"].nunique())
print("Establecimientos:", servicios_2010["Número de establecimientos"].sum())

COMERCIO AL POR MENOR
Clases SCIAN: 80
Establecimientos: 16693

SERVICIOS
Clases SCIAN: 296
Establecimientos: 17963


In [74]:
# ==========================================
# Diagnóstico: clases de servicios con > 50
# establecimientos en 2010
# ==========================================

servicios_mayores_50_2010 = servicios_2010[
    servicios_2010["Número de establecimientos"] > 50
].copy()

servicios_mayores_50_2010 = servicios_mayores_50_2010.sort_values(
    "Número de establecimientos",
    ascending=False
)

print(
    "Clases de servicios con más de 50 establecimientos:",
    len(servicios_mayores_50_2010)
)

print(
    "Establecimientos contenidos en esas clases:",
    servicios_mayores_50_2010["Número de establecimientos"].sum()
)

display(servicios_mayores_50_2010)

Clases de servicios con más de 50 establecimientos: 57
Establecimientos contenidos en esas clases: 15274


,Código de la clase de actividad,Nombre de la clase de actividad,Número de establecimientos,Sector SCIAN
592,722212,RESTAURANTES DE COMIDA PARA LLEVAR,1975,72
593,722219,OTROS RESTAURANTES CON SERVICIO LIMITADO,1707,72
623,812110,SALONES Y CLINICAS DE BELLEZA Y PELUQUERIAS,1582,81
510,621211,CONSULTORIOS DENTALES DEL SECTOR PRIVADO,913,62
597,811111,REPARACION MECANICA EN GENERAL DE AUTOMOVILES ...,656,81
457,561432,SERVICIOS DE ACCESO A COMPUTADORAS,578,56
509,621113,CONSULTORIOS DE MEDICINA ESPECIALIZADA DEL SEC...,499,62
417,541110,BUFETES JURIDICOS,493,54
507,621111,CONSULTORIOS DE MEDICINA GENERAL DEL SECTOR PR...,441,62
604,811121,HOJALATERIA Y PINTURA DE AUTOMOVILES Y CAMIONES,335,81


In [75]:
# ==========================================
# DENUE 2010
# Guardar conteos y diagnósticos
# ==========================================

# 1. Todas las clases SCIAN de Morelia
conteo_scian_2010.to_csv(
    CONTEOS / "CONTEO_SCIAN_2010_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Todas las clases de comercio al por menor
comercio_2010.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2010.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Todas las clases de servicios
servicios_2010.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2010.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Diagnóstico de servicios con más de 50 establecimientos
servicios_mayores_50_2010.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2010.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos 2010 guardados correctamente.")

Archivos 2010 guardados correctamente.


# DENUE 2011


In [76]:
print("Contenido exacto de la carpeta 2011:")

for archivo in CARPETA_2011.iterdir():
    print(archivo.name)

Contenido exacto de la carpeta 2011:


NameError: name 'CARPETA_2011' is not defined

In [ ]:
import zipfile

zip_2011 = CARPETA_2011 / "denue_16_2011_csv.zip"

print("ZIP encontrado:", zip_2011.exists())
print("Nombre:", zip_2011.name)

with zipfile.ZipFile(zip_2011, "r") as z:
    print("\nContenido del ZIP:")
    for nombre in z.namelist():
        print("-", nombre)

ZIP encontrado: True
Nombre: denue_16_2011_csv.zip

Contenido del ZIP:
- denueCSV1116c.csv
- diccionario_de_datos_CSV_y_DBF.pdf


In [ ]:
# ==========================================
# DENUE 2011
# Inspección inicial y comparación con 2010
# ==========================================

with zipfile.ZipFile(zip_2011, "r") as z:
    with z.open("denueCSV1116c.csv") as archivo:
        muestra_2011 = pd.read_csv(
            archivo,
            nrows=5,
            encoding="latin1",
            dtype=str
        )

print("Número de columnas 2011:", len(muestra_2011.columns))

print("\nColumnas 2011:")
for columna in muestra_2011.columns:
    print("-", columna)

print("\n¿Las columnas de 2011 son exactamente iguales a las de 2010?")
print(list(muestra_2011.columns) == list(muestra_2010.columns))

print("\nPrimeros 5 registros:")
display(muestra_2011)

Número de columnas 2011: 30

Columnas 2011:
- Nombre de la unidad económica
- Razón social
- Código de la clase de actividad
- Nombre de la clase de actividad
- Personal ocupado (estrato)
- Tipo de vialidad
- Nombre de la vialidad
- Número exterior o km
- Edificio, piso o nivel
- Número o letra interior
- Tipo y nombre del asentamiento humano
- Corredor industrial, centro comercial o mercado público
- Número de local
- Código postal
- Clave entidad federativa
- Entidad federativa
- Clave municipio
- Municipio
- Clave localidad
- Localidad
- Área geoestadística básica
- Manzana
- Número de teléfono
- Correo electrónico
- Sitio en Internet
- Tipo de unidad económica
- Fecha de incorporación al DENUE
- Estatus de actualización
- Latitud
- Longitud

¿Las columnas de 2011 son exactamente iguales a las de 2010?
False

Primeros 5 registros:


,Nombre de la unidad económica,Razón social,Código de la clase de actividad,Nombre de la clase de actividad,Personal ocupado (estrato),Tipo de vialidad,Nombre de la vialidad,Número exterior o km,"Edificio, piso o nivel",Número o letra interior,...,Área geoestadística básica,Manzana,Número de teléfono,Correo electrónico,Sitio en Internet,Tipo de unidad económica,Fecha de incorporación al DENUE,Estatus de actualización,Latitud,Longitud
0,COMITE DE AGUA POTABLE Y ALCANTARILLADO DE ACU...,COMITE DE AGUA POTABLE Y ALCANTARILLADO DE ACU...,222111,"CAPTACION, TRATAMIENTO Y SUMINISTRO DE AGUA RE...",6 A 10 PERSONAS,CALLE,PORTAL MIGUEL HIDALGO,SN,NaN,NaN,...,0028,019,434343882750,luisvillazarco_capa@hotmail.com,NaN,NaN,JULIO 2010,2,19.496943089599998,-101.33316345999999
1,PALETERIA CARIBE,NaN,311520,ELABORACION DE HELADOS Y PALETAS,0 A 5 PERSONAS,AVENIDA,VICENTE RIVA PALACIO,37,NaN,NaN,...,0028,006,0,NaN,NaN,FIJO,JULIO 2010,0,19.496175369199999,-101.3331687
2,EL ANGEL,NaN,311812,PANIFICACION TRADICIONAL,0 A 5 PERSONAS,AVENIDA,VICENTE RIVA PALACIO,133,NaN,NaN,...,0028,001,4343882388,NaN,NaN,FIJO,JULIO 2010,0,19.4952722618,-101.333291923
3,PANADERIA,NaN,311812,PANIFICACION TRADICIONAL,0 A 5 PERSONAS,AVENIDA,VICENTE RIVA PALACIO,359,NaN,NaN,...,0028,013,0,NaN,NaN,FIJO,JULIO 2010,0,19.493217876599999,-101.33356208799999
4,PANADERIA AISSE,NaN,311812,PANIFICACION TRADICIONAL,0 A 5 PERSONAS,CALLE,GUADALUPE VICTORIA,52,NaN,NaN,...,0028,007,3882053,NaN,NaN,FIJO,JULIO 2010,0,19.496372911000002,-101.332303301


In [ ]:
# ==========================================
# DENUE 2011
# Comparar estructura con DENUE 2010
# ==========================================

columnas_2010 = set(muestra_2010.columns)
columnas_2011 = set(muestra_2011.columns)

solo_2010 = sorted(columnas_2010 - columnas_2011)
solo_2011 = sorted(columnas_2011 - columnas_2010)
comunes = sorted(columnas_2010 & columnas_2011)

print("COLUMNAS COMUNES:", len(comunes))

print("\nCOLUMNAS QUE SOLO EXISTEN EN 2010:")
for c in solo_2010:
    print("-", c)

print("\nCOLUMNAS QUE SOLO EXISTEN EN 2011:")
for c in solo_2011:
    print("-", c)

COLUMNAS COMUNES: 22

COLUMNAS QUE SOLO EXISTEN EN 2010:
- Calle, avenida, andador, carretera, manzana u otro
- Colonia, fraccionamiento, unidad habitacional o barrio

COLUMNAS QUE SOLO EXISTEN EN 2011:
- Clave entidad federativa
- Clave localidad
- Clave municipio
- Estatus de actualización
- Fecha de incorporación al DENUE
- Nombre de la vialidad
- Tipo de vialidad
- Tipo y nombre del asentamiento humano


In [ ]:
# ==========================================
# DENUE 2011
# Validar identificación de Morelia
# ==========================================

columnas_revision_2011 = [
    "Clave entidad federativa",
    "Entidad federativa",
    "Clave municipio",
    "Municipio",
    "Clave localidad",
    "Localidad",
    "Estatus de actualización",
    "Fecha de incorporación al DENUE"
]

with zipfile.ZipFile(zip_2011, "r") as z:
    with z.open("denueCSV1116c.csv") as archivo:
        revision_2011 = pd.read_csv(
            archivo,
            usecols=columnas_revision_2011,
            encoding="latin1",
            dtype=str
        )

# Limpiar claves sin alterar el archivo original
revision_2011["Clave municipio"] = (
    revision_2011["Clave municipio"]
    .str.strip()
    .str.zfill(3)
)

revision_2011["Municipio_limpio"] = (
    revision_2011["Municipio"]
    .str.strip()
    .str.upper()
)

print("Total de registros Michoacán 2011:", len(revision_2011))

print("\nRegistros con clave municipal 053:")
print((revision_2011["Clave municipio"] == "053").sum())

print("\nRegistros cuyo municipio dice MORELIA:")
print((revision_2011["Municipio_limpio"] == "MORELIA").sum())

print("\n¿Hay discrepancias entre clave 053 y nombre MORELIA?")

discrepancias = revision_2011[
    (revision_2011["Clave municipio"] == "053")
    !=
    (revision_2011["Municipio_limpio"] == "MORELIA")
]

print(len(discrepancias))

Total de registros Michoacán 2011: 203442

Registros con clave municipal 053:
42004

Registros cuyo municipio dice MORELIA:
42004

¿Hay discrepancias entre clave 053 y nombre MORELIA?
0


In [ ]:
# ==========================================
# DENUE 2011
# Revisar variables de actualización
# ==========================================

print("ESTATUS DE ACTUALIZACIÓN:")
display(
    revision_2011["Estatus de actualización"]
    .value_counts(dropna=False)
)

print("\nFECHA DE INCORPORACIÓN AL DENUE:")
display(
    revision_2011["Fecha de incorporación al DENUE"]
    .value_counts(dropna=False)
    .head(30)
)

ESTATUS DE ACTUALIZACIÓN:


Estatus de actualización
0    198570
1      2166
3      1728
2       978
Name: count, dtype: int64


FECHA DE INCORPORACIÓN AL DENUE:


Fecha de incorporación al DENUE
JULIO 2010    201714
MARZO 2011      1728
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2011
# Cruce entre estatus e incorporación
# ==========================================

tabla_estatus_fecha_2011 = pd.crosstab(
    revision_2011["Estatus de actualización"],
    revision_2011["Fecha de incorporación al DENUE"],
    margins=True
)

display(tabla_estatus_fecha_2011)

Fecha de incorporación al DENUE,JULIO 2010,MARZO 2011,All
Estatus de actualización,,,
0,198570,0,198570
1,2166,0,2166
2,978,0,978
3,0,1728,1728
All,201714,1728,203442


In [ ]:
# ==========================================
# DENUE 2011
# Estatus e incorporación SOLO MORELIA
# ==========================================

morelia_revision_2011 = revision_2011[
    revision_2011["Clave municipio"] == "053"
].copy()

print("Registros de Morelia:", len(morelia_revision_2011))

display(
    pd.crosstab(
        morelia_revision_2011["Estatus de actualización"],
        morelia_revision_2011["Fecha de incorporación al DENUE"],
        margins=True
    )
)

Registros de Morelia: 42004


Fecha de incorporación al DENUE,JULIO 2010,MARZO 2011,All
Estatus de actualización,,,
0,40249,0,40249
1,793,0,793
2,238,0,238
3,0,724,724
All,41280,724,42004


In [ ]:
# ==========================================
# DENUE 2011
# Extraer diccionario de datos
# ==========================================

nombre_diccionario_2011 = "diccionario_de_datos_CSV_y_DBF.pdf"
ruta_diccionario_2011 = CARPETA_2011 / nombre_diccionario_2011

with zipfile.ZipFile(zip_2011, "r") as z:
    contenido_pdf = z.read(nombre_diccionario_2011)

with open(ruta_diccionario_2011, "wb") as f:
    f.write(contenido_pdf)

print("Diccionario extraído correctamente:")
print(ruta_diccionario_2011)
print("Existe:", ruta_diccionario_2011.exists())

Diccionario extraído correctamente:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\00_ORIGINALES\2011\diccionario_de_datos_CSV_y_DBF.pdf
Existe: True


In [ ]:
# ==========================================
# DENUE 2011
# Buscar "Estatus de actualización"
# dentro del diccionario PDF
# ==========================================

from pypdf import PdfReader

reader = PdfReader(ruta_diccionario_2011)

print("Páginas del diccionario:", len(reader.pages))

for i, pagina in enumerate(reader.pages):
    texto = pagina.extract_text() or ""

    if "Estatus de actualización" in texto:
        print("\n==============================")
        print("PÁGINA:", i + 1)
        print("==============================")
        print(texto)
        

Páginas del diccionario: 2


In [ ]:
# ==========================================
# DENUE 2011
# Buscar términos relacionados con estatus
# ==========================================

for i, pagina in enumerate(reader.pages):
    texto = pagina.extract_text() or ""

    if (
        "estatus" in texto.lower()
        or "actualización" in texto.lower()
        or "actualizacion" in texto.lower()
    ):
        print("\n==============================")
        print("PÁGINA:", i + 1)
        print("==============================")
        print(texto)


PÁGINA: 1
Directorio Estadístico Nacional de Unidades Económicas
DESCRIPCIÓN DE ARCHIVOS, DENUE MARZO 2011
Tipo de dato Longitud Descripción
alfanumérico 75
Razón social alfanumérico 75
alfanumérico 6
alfanumérico 160 Descripción del código de actividad conforme al SCIAN 2007.
alfanumérico 40
Tipo de vialidad tipo_calle alfanumérico 25
calle alfanumérico 75 Es el sustantivo propio con el cual se identifica la vialidad.
alfanumérico 8
Edificio, piso o nivel edificio alfanumérico 12
alfanumérico 8
colonia alfanumérico 75
alfanumérico 75
Nombre de la 
columna en el 
arvhivo CSV
Nombre de la 
columna en el 
archivo DBF
Nombre de la Unidad 
Económica nom_estab
Es el nombre comercial o nombre exterior con el que se identifica o anuncia la unidad 
económica.
Generalmente es fácil reconocerlo porque está visible y escrito en rótulos, fachadas o 
anuncios luminosos. Ejemplo: ABARROTES LUPITA.
Cuando la unidad económica no tiene ningún apelativo comercial, entonces se 
registra con el nombre ge

In [ ]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ==========================================
# DENUE 2011
# Localizar definiciones relevantes
# en el diccionario
# ==========================================

import unicodedata

def normalizar_texto(texto):
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )
    return texto.lower()

for i, pagina in enumerate(reader.pages):

    texto = pagina.extract_text() or ""
    lineas = texto.splitlines()

    for j, linea in enumerate(lineas):

        linea_normalizada = normalizar_texto(linea)

        if any(
            palabra in linea_normalizada
            for palabra in ["estatus", "actualizacion", "incorporacion"]
        ):

            print("\n====================================")
            print(f"PÁGINA {i+1} - alrededor de línea {j+1}")
            print("====================================")

            inicio = max(0, j - 4)
            fin = min(len(lineas), j + 6)

            for k in range(inicio, fin):
                print(lineas[k])


PÁGINA 1 - alrededor de línea 42
para proteger la confidencialidad de la información.
Código de la clase de 
actividad clase_act
La clasificación de las actividades desarrolladas por las unidades económicas debe 
hacerse utilizando el SCIAN y sus actualizaciones, con base en lo dispuesto en el 
acuerdo para el uso del SCIAN y en la recopilación, análisis y presentación de 
estadísticas económicas, publicado en el Diario Oficial de la Federación del 10 de julio 
de 2009.
El código está asignado con base en la actividad principal que desarrolla la unidad 
económica y de acuerdo con el Sistema de Clasificación Industrial para América del 

PÁGINA 2 - alrededor de línea 25
Correo electrónico alfanumérico 75
Sitio en Internet alfanumérico 75
alfanumérico 39
alta alfanumérico 10
estatus alfanumérico 1
Latitud latitud alfanumérico 12
Longitud longitud alfanumérico 12
num_local Dato alfanumérico que corresponde al local en donde se localizó  la unidad 
económica.
cod_postal Clave numérica int

In [ ]:
# ==========================================
# DENUE 2011
# Cargar archivo completo y filtrar Morelia
# ==========================================

with zipfile.ZipFile(zip_2011, "r") as z:
    with z.open("denueCSV1116c.csv") as archivo:
        denue_2011 = pd.read_csv(
            archivo,
            encoding="latin1",
            dtype=str
        )

# Normalizar clave municipal
denue_2011["Clave municipio"] = (
    denue_2011["Clave municipio"]
    .str.strip()
    .str.zfill(3)
)

# Filtrar Morelia
morelia_2011 = denue_2011[
    denue_2011["Clave municipio"] == "053"
].copy()

print("Registros Michoacán 2011:", len(denue_2011))
print("Registros Morelia 2011:", len(morelia_2011))
print("Columnas conservadas:", len(morelia_2011.columns))

assert len(morelia_2011) == 42004

# Guardar archivo derivado
archivo_salida_2011 = MORELIA / "DENUE_2011_MORELIA.csv"

morelia_2011.to_csv(
    archivo_salida_2011,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ DENUE 2011 Morelia guardado correctamente.")
print(archivo_salida_2011)

Registros Michoacán 2011: 203442
Registros Morelia 2011: 42004
Columnas conservadas: 30

✓ DENUE 2011 Morelia guardado correctamente.
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2011_MORELIA.csv


In [ ]:
# ==========================================
# DENUE 2011
# Conteo por clase SCIAN de 6 dígitos
# ==========================================

conteo_scian_2011 = (
    morelia_2011
    .groupby(
        ["Código de la clase de actividad",
         "Nombre de la clase de actividad"],
        dropna=False
    )
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2011 = conteo_scian_2011.sort_values(
    "Número de establecimientos",
    ascending=False
)

# Crear sector SCIAN con los primeros 2 dígitos
conteo_scian_2011["Sector SCIAN"] = (
    conteo_scian_2011["Código de la clase de actividad"]
    .astype(str)
    .str[:2]
)

print(
    "Número de clases SCIAN distintas:",
    conteo_scian_2011["Código de la clase de actividad"].nunique()
)

print("\n20 clases con más establecimientos en 2011:")
display(conteo_scian_2011.head(20))

Número de clases SCIAN distintas: 670

20 clases con más establecimientos en 2011:


,Código de la clase de actividad,Nombre de la clase de actividad,Número de establecimientos,Sector SCIAN
271,461110,"COMERCIO AL POR MENOR EN TIENDAS DE ABARROTES,...",4194,46
616,722212,RESTAURANTES DE COMIDA PARA LLEVAR,1975,72
617,722219,OTROS RESTAURANTES CON SERVICIO LIMITADO,1708,72
647,812110,SALONES Y CLINICAS DE BELLEZA Y PELUQUERIAS,1582,81
290,463211,"COMERCIO AL POR MENOR DE ROPA, EXCEPTO DE BEBE...",1495,46
275,461130,COMERCIO AL POR MENOR DE FRUTAS Y VERDURAS FRE...,1179,46
66,311830,ELABORACION DE TORTILLAS DE MAIZ Y MOLIENDA DE...,1071,31
535,621211,CONSULTORIOS DENTALES DEL SECTOR PRIVADO,914,62
312,465311,COMERCIO AL POR MENOR DE ARTICULOS DE PAPELERIA,857,46
621,811111,REPARACION MECANICA EN GENERAL DE AUTOMOVILES ...,657,81


In [ ]:
# ==========================================
# DENUE 2011
# Comercio al por menor y servicios
# ==========================================

# COMERCIO AL POR MENOR
comercio_2011 = conteo_scian_2011[
    conteo_scian_2011["Sector SCIAN"] == "46"
].copy()

# SERVICIOS
sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2011 = conteo_scian_2011[
    conteo_scian_2011["Sector SCIAN"].isin(sectores_servicios)
].copy()

# Servicios con más de 50 establecimientos
servicios_mayores_50_2011 = servicios_2011[
    servicios_2011["Número de establecimientos"] > 50
].copy()

servicios_mayores_50_2011 = servicios_mayores_50_2011.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("COMERCIO AL POR MENOR")
print("Clases SCIAN:",
      comercio_2011["Código de la clase de actividad"].nunique())
print("Establecimientos:",
      comercio_2011["Número de establecimientos"].sum())

print("\nSERVICIOS")
print("Clases SCIAN:",
      servicios_2011["Código de la clase de actividad"].nunique())
print("Establecimientos:",
      servicios_2011["Número de establecimientos"].sum())

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:",
      len(servicios_mayores_50_2011))
print("Establecimientos:",
      servicios_mayores_50_2011["Número de establecimientos"].sum())

COMERCIO AL POR MENOR
Clases SCIAN: 79
Establecimientos: 16824

SERVICIOS
Clases SCIAN: 310
Establecimientos: 18269

SERVICIOS CON MÁS DE 50
Clases: 59
Establecimientos: 15505


In [ ]:
# ==========================================
# DENUE 2011
# Guardar conteos
# ==========================================

conteo_scian_2011.to_csv(
    CONTEOS / "CONTEO_SCIAN_2011_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_2011.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2011.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_2011.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2011.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_2011.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2011.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Archivos 2011 guardados correctamente.")

✓ Archivos 2011 guardados correctamente.


In [ ]:
# ==========================================
# COMPARACIÓN DENUE 2010 - 2011
# Por clase SCIAN de 6 dígitos
# ==========================================

comparacion_2010_2011 = pd.merge(
    
    conteo_scian_2010[
        ["Código de la clase de actividad",
         "Nombre de la clase de actividad",
         "Número de establecimientos"]
    ].rename(columns={
        "Nombre de la clase de actividad": "Actividad_2010",
        "Número de establecimientos": "2010"
    }),
    
    conteo_scian_2011[
        ["Código de la clase de actividad",
         "Nombre de la clase de actividad",
         "Número de establecimientos"]
    ].rename(columns={
        "Nombre de la clase de actividad": "Actividad_2011",
        "Número de establecimientos": "2011"
    }),
    
    on="Código de la clase de actividad",
    how="outer",
    indicator=True
)

# Crear un solo nombre de actividad
comparacion_2010_2011["Actividad"] = (
    comparacion_2010_2011["Actividad_2011"]
    .combine_first(comparacion_2010_2011["Actividad_2010"])
)

# Sustituir ausencias por cero SOLO para el conteo
comparacion_2010_2011["2010"] = (
    comparacion_2010_2011["2010"].fillna(0).astype(int)
)

comparacion_2010_2011["2011"] = (
    comparacion_2010_2011["2011"].fillna(0).astype(int)
)

# Variación neta de registros entre ediciones
comparacion_2010_2011["Variación_registros"] = (
    comparacion_2010_2011["2011"] -
    comparacion_2010_2011["2010"]
)

# Identificar presencia en las ediciones
comparacion_2010_2011["Presencia"] = (
    comparacion_2010_2011["_merge"]
    .map({
        "both": "Ambas ediciones",
        "left_only": "Solo 2010",
        "right_only": "Solo 2011"
    })
)

# Seleccionar columnas finales
comparacion_2010_2011 = comparacion_2010_2011[
    [
        "Código de la clase de actividad",
        "Actividad",
        "2010",
        "2011",
        "Variación_registros",
        "Presencia"
    ]
].sort_values(
    "Variación_registros",
    ascending=False
)

display(comparacion_2010_2011.head(30))

,Código de la clase de actividad,Actividad,2010,2011,Variación_registros,Presencia
406,522110,BANCA MULTIPLE,0,161,161,Solo 2011
674,999999,NO ESPECIFICADO,310,327,17,Ambas ediciones
16,236221,EDIFICACION DE INMUEBLES COMERCIALES Y DE SERV...,62,79,17,Ambas ediciones
274,461110,"COMERCIO AL POR MENOR EN TIENDAS DE ABARROTES,...",4179,4194,15,Ambas ediciones
673,931810,ACTIVIDADES DE SEGURIDAD NACIONAL,9,24,15,Ambas ediciones
293,463211,"COMERCIO AL POR MENOR DE ROPA, EXCEPTO DE BEBE...",1481,1495,14,Ambas ediciones
29,237999,OTRAS CONSTRUCCIONES DE INGENIERIA CIVIL,0,13,13,Solo 2011
23,237212,CONSTRUCCION DE OBRAS DE URBANIZACION,21,34,13,Ambas ediciones
12,236111,EDIFICACION DE VIVIENDA UNIFAMILIAR,30,41,11,Ambas ediciones
377,492110,SERVICIOS DE MENSAJERIA Y PAQUETERIA FORANEA,30,41,11,Ambas ediciones


In [ ]:
# ==========================================
# Caso de control: BANCA MÚLTIPLE 522110
# ==========================================

banca_2011 = morelia_2011[
    morelia_2011["Código de la clase de actividad"] == "522110"
].copy()

print("Registros 522110 en Morelia 2011:", len(banca_2011))

print("\nFecha de incorporación al DENUE:")
display(
    banca_2011["Fecha de incorporación al DENUE"]
    .value_counts(dropna=False)
)

print("\nEstatus de actualización:")
display(
    banca_2011["Estatus de actualización"]
    .value_counts(dropna=False)
)

print("\nCruce fecha × estatus:")
display(
    pd.crosstab(
        banca_2011["Fecha de incorporación al DENUE"],
        banca_2011["Estatus de actualización"],
        margins=True
    )
)

Registros 522110 en Morelia 2011: 161

Fecha de incorporación al DENUE:


Fecha de incorporación al DENUE
MARZO 2011    161
Name: count, dtype: int64


Estatus de actualización:


Estatus de actualización
3    161
Name: count, dtype: int64


Cruce fecha × estatus:


Estatus de actualización,3,All
Fecha de incorporación al DENUE,,
MARZO 2011,161,161
All,161,161


### Nota metodológica sobre las comparaciones entre ediciones

La aparición de establecimientos o clases de actividad en una edición posterior
del DENUE no se interpreta automáticamente como nacimiento empresarial.

Como comprobación, la clase SCIAN 522110 (Banca múltiple) registró 0
establecimientos en Morelia en la edición 2010 y 161 en la edición 2011.
Los 161 registros presentan fecha de incorporación "MARZO 2011" y estatus
de actualización 3, definido en el diccionario de datos como "Nueva alta,
marzo 2011".

Por tanto, la aparición en una edición posterior representa una incorporación
al directorio y no necesariamente el inicio de operaciones de la unidad económica.

In [ ]:
# ==========================================
# Guardar comparación 2010 - 2011
# ==========================================

archivo_comparacion = (
    COMPARACIONES / "COMPARACION_SCIAN_2010_2011.csv"
)

comparacion_2010_2011.to_csv(
    archivo_comparacion,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Comparación 2010-2011 guardada:")
print(archivo_comparacion)


✓ Comparación 2010-2011 guardada:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\04_COMPARACIONES\COMPARACION_SCIAN_2010_2011.csv


# DENUE 2012

In [ ]:
# ==========================================
# DENUE 2012
# Localizar archivo ZIP
# ==========================================

CARPETA_2012 = ORIGINALES / "2012"

print("Existe carpeta 2012:", CARPETA_2012.exists())

print("\nContenido de la carpeta 2012:")
for archivo in CARPETA_2012.iterdir():
    print("-", archivo.name)

Existe carpeta 2012: True

Contenido de la carpeta 2012:
- denue_16_2012_csv.zip
- Diccionario de Datos DENUE 2012.pdf


In [ ]:
# ==========================================
# DENUE 2012
# Revisar contenido del ZIP
# ==========================================

zip_2012 = CARPETA_2012 / "denue_16_2012_csv.zip"

print("ZIP encontrado:", zip_2012.exists())
print("Nombre:", zip_2012.name)

with zipfile.ZipFile(zip_2012, "r") as z:
    print("\nContenido del ZIP:")
    for nombre in z.namelist():
        print("-", nombre)

ZIP encontrado: True
Nombre: denue_16_2012_csv.zip

Contenido del ZIP:
- denue_1_16.csv
- Diccionario de Datos DENUE 2012.pdf


In [ ]:
# ==========================================
# DENUE 2012
# Inspección inicial y comparación con 2011
# ==========================================

with zipfile.ZipFile(zip_2012, "r") as z:
    with z.open("denue_1_16.csv") as archivo:
        muestra_2012 = pd.read_csv(
            archivo,
            nrows=5,
            encoding="latin1",
            dtype=str
        )

print("Número de columnas 2012:", len(muestra_2012.columns))

print("\nColumnas 2012:")
for columna in muestra_2012.columns:
    print("-", columna)

print("\n¿Las columnas de 2012 son exactamente iguales a las de 2011?")
print(list(muestra_2012.columns) == list(muestra_2011.columns))

print("\nPrimeros 5 registros:")
display(muestra_2012)

Número de columnas 2012: 47

Columnas 2012:
- Llave DENUE
- NIC
- NOP
- Clave entidad
- Entidad federativa
- Clave municipio
- Municipio
- Clave localidad
- Localidad
- Área geoestadística básica 
- Manzana
- Nombre de la Unidad Económica
- Razón social
- Tipo de vialidad
- Nombre de la vialidad
- Tipo entre vialidad 1
- Nombre entre vialidad 1
- Tipo entre vialidad 2
- Nombre entre vialidad 2
- Tipo vialidad posterior
- Nombre vialidad posterior
- Número exterior o kilómetro
- Edificio, piso o nivel
- Número o letra interior
- Tipo de asentamiento humano
- Nombre de asentamiento humano
- Código Postal
- Número de teléfono 1
- Extensión 1
- Número de teléfono 2
- Extensión 2
- Número de FAX
- Código de la clase de actividad SCIAN
- Nombre de clase de la actividad
- Corredor industrial, centro comercial o mercado público
- Número de local
- Correo electrónico 1
- Sitio en Internet
- Correo electrónico 2
- Tipo de establecimiento
- Tipo de unidad económica
- Personal ocupado (estrato)
- 

,Llave DENUE,NIC,NOP,Clave entidad,Entidad federativa,Clave municipio,Municipio,Clave localidad,Localidad,Área geoestadística básica,...,Sitio en Internet,Correo electrónico 2,Tipo de establecimiento,Tipo de unidad económica,Personal ocupado (estrato),Descripcion estrato personal ocupado,Estatus de control de registros,Fecha de incorporación al DENUE,Latitud,Longitud
0,4254914,MIC997016,GPM99531250,16,MICHOACAN DE OCAMPO,104,VILLAMAR,0028,El Platanal,0167,...,NaN,NaN,1,,2,6 a 10 personas,0,JULIO 2010,20.0653664,-102.57693148
1,740472,MIC425330,P1816425330,16,MICHOACAN DE OCAMPO,102,URUAPAN,0001,Uruapan,0514,...,NaN,NaN,1,,6,101 a 250 personas,0,JULIO 2010,19.39313127,-102.06953718
2,1238871,MIC419004,P0916419004,16,MICHOACAN DE OCAMPO,102,URUAPAN,0001,Uruapan,0336,...,NaN,NaN,1,,3,11 a 30 personas,0,JULIO 2010,19.41205594,-102.05528683
3,4365195,MIC520148,P1916520148,16,MICHOACAN DE OCAMPO,077,SAN LUCAS,0067,Vicente Riva Palacio,0223,...,NaN,NaN,1,,1,0 a 5 personas,0,JULIO 2010,18.36541478,-100.66330431
4,4253699,MIC882026,GPM12100007,16,MICHOACAN DE OCAMPO,112,ZITÁCUARO,0051,Las Rosas,0099,...,NaN,NaN,1,,1,0 a 5 personas,0,JULIO 2010,19.41222222,-100.34194444


In [ ]:
# ==========================================
# Comparar estructura 2012 vs 2011
# ==========================================

columnas_2011 = set(muestra_2011.columns)
columnas_2012 = set(muestra_2012.columns)

solo_2011 = sorted(columnas_2011 - columnas_2012)
solo_2012 = sorted(columnas_2012 - columnas_2011)
comunes_2011_2012 = sorted(columnas_2011 & columnas_2012)

print("COLUMNAS COMUNES:", len(comunes_2011_2012))

print("\nCOLUMNAS QUE SOLO EXISTEN EN 2011:")
for c in solo_2011:
    print("-", c)

print("\nCOLUMNAS QUE SOLO EXISTEN EN 2012:")
for c in solo_2012:
    print("-", c)

COLUMNAS COMUNES: 19

COLUMNAS QUE SOLO EXISTEN EN 2011:
- Clave entidad federativa
- Correo electrónico
- Código de la clase de actividad
- Código postal
- Estatus de actualización
- Nombre de la clase de actividad
- Nombre de la unidad económica
- Número de teléfono
- Número exterior o km
- Tipo y nombre del asentamiento humano
- Área geoestadística básica

COLUMNAS QUE SOLO EXISTEN EN 2012:
- Clave entidad
- Correo electrónico 1
- Correo electrónico 2
- Código Postal
- Código de la clase de actividad SCIAN
- Descripcion estrato personal ocupado
- Estatus de control de registros
- Extensión 1
- Extensión 2
- Llave DENUE
- NIC
- NOP
- Nombre de asentamiento humano
- Nombre de clase de la actividad
- Nombre de la Unidad Económica
- Nombre entre vialidad 1
- Nombre entre vialidad 2
- Nombre vialidad posterior
- Número de FAX
- Número de teléfono 1
- Número de teléfono 2
- Número exterior o kilómetro
- Tipo de asentamiento humano
- Tipo de establecimiento
- Tipo entre vialidad 1
- Tipo e

In [ ]:
# ==========================================
# DENUE 2012
# Buscar columnas clave
# ==========================================

palabras_clave = [
    "llave",
    "nic",
    "nop",
    "entidad",
    "municipio",
    "localidad",
    "actividad",
    "scian",
    "estatus",
    "control",
    "incorpor",
    "fecha",
    "personal",
    "latitud",
    "longitud"
]

print("COLUMNAS RELEVANTES DE 2012:\n")

for columna in muestra_2012.columns:
    if any(palabra in columna.lower() for palabra in palabras_clave):
        print(repr(columna))

COLUMNAS RELEVANTES DE 2012:

'Llave DENUE'
'NIC'
'NOP'
'Clave entidad'
'Entidad federativa'
'Clave municipio'
'Municipio'
'Clave localidad'
'Localidad'
'Código de la clase de actividad SCIAN'
'Nombre de clase de la actividad'
'Correo electrónico 1'
'Correo electrónico 2'
'Personal ocupado (estrato)'
'Descripcion estrato personal ocupado'
'Estatus de control de registros'
'Fecha de incorporación al DENUE'
'Latitud'
'Longitud'


In [ ]:
# ==========================================
# DENUE 2012
# Extraer diccionario de datos
# ==========================================

nombre_diccionario_2012 = "Diccionario de Datos DENUE 2012.pdf"
ruta_diccionario_2012 = CARPETA_2012 / nombre_diccionario_2012

with zipfile.ZipFile(zip_2012, "r") as z:
    contenido_pdf = z.read(nombre_diccionario_2012)

with open(ruta_diccionario_2012, "wb") as f:
    f.write(contenido_pdf)

print("Diccionario 2012 extraído:")
print(ruta_diccionario_2012)
print("Existe:", ruta_diccionario_2012.exists())

Diccionario 2012 extraído:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\00_ORIGINALES\2012\Diccionario de Datos DENUE 2012.pdf
Existe: True


In [ ]:
# ==========================================
# DENUE 2012
# Leer diccionario
# ==========================================

from pypdf import PdfReader

reader_2012 = PdfReader(ruta_diccionario_2012)

print("Páginas del diccionario 2012:", len(reader_2012.pages))

Páginas del diccionario 2012: 4


In [ ]:
# ==========================================
# DENUE 2012
# Buscar definiciones clave
# ==========================================

terminos_2012 = [
    "llave denue",
    "nic",
    "nop",
    "estatus de control",
    "fecha de incorporación",
    "fecha de incorporacion"
]

for i, pagina in enumerate(reader_2012.pages):

    texto = pagina.extract_text() or ""
    lineas = texto.splitlines()

    for j, linea in enumerate(lineas):

        linea_norm = normalizar_texto(linea)

        if any(termino in linea_norm for termino in terminos_2012):

            print("\n====================================")
            print(f"PÁGINA {i+1} - alrededor de línea {j+1}")
            print("====================================")

            inicio = max(0, j - 4)
            fin = min(len(lineas), j + 8)

            for k in range(inicio, fin):
                print(lineas[k])


PÁGINA 1 - alrededor de línea 2
                                                                                                                   Junio 2012
Llave DENUE d_llave numérico 10
Número de identificación del DENUE, es una clave numérica única para cada registro 
de la base de datos del Directorio Estadístico Nacional de Unidades Económicas 
(DENUE). 
NIC nic alfanumérico 9 Número de identificación Censal.
NOP nop alfanumérico 11 Número operativo Censal.
Clave entidad cve_ent alfanumérico 2 Clave que identifica a la entidad federativa.
Entidad federativa entidad alfanumérico 32 Es el sustantivo propio que identifica a la entidad federativa.

PÁGINA 1 - alrededor de línea 3
                                                                                                                   Junio 2012
Llave DENUE d_llave numérico 10
Número de identificación del DENUE, es una clave numérica única para cada registro 
de la base de datos del Directorio Estadístico Nacional de Unidad

In [ ]:
# ==========================================
# DENUE 2012
# Buscar definiciones exactas pendientes
# ==========================================

terminos_buscar = [
    "estatus de control de registros",
    "fecha de incorporacion al denue"
]

for i, pagina in enumerate(reader_2012.pages):

    texto = pagina.extract_text() or ""
    lineas = texto.splitlines()

    for j, linea in enumerate(lineas):

        linea_norm = normalizar_texto(linea)

        if any(termino in linea_norm for termino in terminos_buscar):

            print("\n====================================")
            print(f"PÁGINA {i+1} - línea aproximada {j+1}")
            print("====================================")

            inicio = max(0, j - 2)
            fin = min(len(lineas), j + 15)

            for k in range(inicio, fin):
                print(lineas[k])

In [ ]:
# ==========================================
# DENUE 2012
# Buscar palabras sueltas en el diccionario
# ==========================================

for i, pagina in enumerate(reader_2012.pages):

    texto = pagina.extract_text() or ""
    lineas = texto.splitlines()

    for j, linea in enumerate(lineas):

        linea_norm = normalizar_texto(linea)

        if (
            "estatus" in linea_norm
            or "control" in linea_norm
            or "incorpor" in linea_norm
            or "denue" in linea_norm
        ):

            print(f"\nPÁGINA {i+1} - línea {j+1}")
            print("--------------------------------")

            inicio = max(0, j - 3)
            fin = min(len(lineas), j + 10)

            for k in range(inicio, fin):
                print(lineas[k])


PÁGINA 1 - línea 2
--------------------------------
                                                                                                                   Junio 2012
Llave DENUE d_llave numérico 10
Número de identificación del DENUE, es una clave numérica única para cada registro 
de la base de datos del Directorio Estadístico Nacional de Unidades Económicas 
(DENUE). 
NIC nic alfanumérico 9 Número de identificación Censal.
NOP nop alfanumérico 11 Número operativo Censal.
Clave entidad cve_ent alfanumérico 2 Clave que identifica a la entidad federativa.
Entidad federativa entidad alfanumérico 32 Es el sustantivo propio que identifica a la entidad federativa.
Clave municipio cve_mun alfanumérico 3 Clave que identifica al Municipio, y en el caso del Distrito Federal los que identifican 
a las Delegaciones.

PÁGINA 1 - línea 3
--------------------------------
                                                                                                                   Jun

In [ ]:
# ==========================================
# DENUE 2012
# Búsqueda robusta dentro del PDF
# ==========================================

texto_completo_2012 = ""

for pagina in reader_2012.pages:
    texto_completo_2012 += "\n" + (pagina.extract_text() or "")

texto_norm = normalizar_texto(texto_completo_2012)

terminos = [
    "estatus",
    "control de registros",
    "incorporacion",
    "fecha de incorporacion"
]

for termino in terminos:
    termino_norm = normalizar_texto(termino)
    posicion = texto_norm.find(termino_norm)

    print("\n===================================")
    print("BÚSQUEDA:", termino)
    print("===================================")

    if posicion == -1:
        print("No encontrado")
    else:
        inicio = max(0, posicion - 400)
        fin = min(len(texto_completo_2012), posicion + 800)
        print(texto_completo_2012[inicio:fin])


BÚSQUEDA: estatus
jó para la unidad económica, 
sujeto a su dirección y control y que cubrió, como mínimo, una tercera parte de la 
jornada laboral. Puede ser personal de planta, eventual remunerado o no 
remunerado.
Rango de personal ocupado en cada estrato, que permiten identificar el tamaño de 
Unidades Económicas por el número de personal que emplean, con base en el 
personal ocupado total reportado por éstas.
Estatus de control 
de registros estatus alfanumérico 1
Estatus de la situación del registro en la pasada actualización, bajo las siguientes 
definiciones:   0 = No visitada,  1 = Visitada sin cambios,  2 = Visitada con cambios,  3 
= Alta.
Fecha de 
incorporación al 
DENUE
alta alfanumérico 15 Fecha en la que la unidad económica se integró al Directorio Nacional de Unidades 
Económicas.
Longitud longitud numérico 19 Es la distancia que existe entre la Unidad económica y el meridiano de Greenwich, 
medida sobre el paralelo que pasa por dicho punto.
Latitud latitud numérico 1

In [ ]:
# ==========================================
# DENUE 2012
# Cargar archivo completo y filtrar Morelia
# ==========================================

with zipfile.ZipFile(zip_2012, "r") as z:
    with z.open("denue_1_16.csv") as archivo:
        denue_2012 = pd.read_csv(
            archivo,
            encoding="latin1",
            dtype=str
        )

# Normalizar clave municipal
denue_2012["Clave municipio"] = (
    denue_2012["Clave municipio"]
    .str.strip()
    .str.zfill(3)
)

# Filtrar Morelia
morelia_2012 = denue_2012[
    denue_2012["Clave municipio"] == "053"
].copy()

print("Registros Michoacán 2012:", len(denue_2012))
print("Registros Morelia 2012:", len(morelia_2012))
print("Columnas:", len(morelia_2012.columns))

Registros Michoacán 2012: 204166
Registros Morelia 2012: 42251
Columnas: 47


In [ ]:
# ==========================================
# DENUE 2012
# Revisar identificadores y actualización
# ==========================================

print("LLAVE DENUE")
print("Registros:", len(morelia_2012))
print("Llaves no nulas:", morelia_2012["Llave DENUE"].notna().sum())
print("Llaves únicas:", morelia_2012["Llave DENUE"].nunique())

print("\nESTATUS DE CONTROL:")
display(
    morelia_2012["Estatus de control de registros"]
    .value_counts(dropna=False)
)

print("\nFECHA DE INCORPORACIÓN AL DENUE:")
display(
    morelia_2012["Fecha de incorporación al DENUE"]
    .value_counts(dropna=False)
)

LLAVE DENUE
Registros: 42251
Llaves no nulas: 42251
Llaves únicas: 42251

ESTATUS DE CONTROL:


Estatus de control de registros
0    41421
3      374
1      246
2      210
Name: count, dtype: int64


FECHA DE INCORPORACIÓN AL DENUE:


Fecha de incorporación al DENUE
JULIO 2010         41157
MARZO 2011           720
ABRIL 2012           374
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2012
# Cruce fecha de incorporación × estatus
# ==========================================

tabla_estatus_fecha_2012 = pd.crosstab(
    morelia_2012["Estatus de control de registros"],
    morelia_2012["Fecha de incorporación al DENUE"],
    margins=True
)

display(tabla_estatus_fecha_2012)

Fecha de incorporación al DENUE,ABRIL 2012,JULIO 2010,MARZO 2011,All
Estatus de control de registros,,,,
0,0,40737,684,41421
1,0,232,14,246
2,0,188,22,210
3,374,0,0,374
All,374,41157,720,42251


In [ ]:
# ==========================================
# DENUE 2012
# Guardar Morelia
# ==========================================

archivo_salida_2012 = MORELIA / "DENUE_2012_MORELIA.csv"

morelia_2012.to_csv(
    archivo_salida_2012,
    index=False,
    encoding="utf-8-sig"
)

print("✓ DENUE 2012 Morelia guardado correctamente:")
print(archivo_salida_2012)

✓ DENUE 2012 Morelia guardado correctamente:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2012_MORELIA.csv


In [ ]:
# ==========================================
# DENUE 2012
# Conteo por clase SCIAN de 6 dígitos
# ==========================================

conteo_scian_2012 = (
    morelia_2012
    .groupby(
        [
            "Código de la clase de actividad SCIAN",
            "Nombre de la clase de actividad"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2012 = conteo_scian_2012.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2012["Sector SCIAN"] = (
    conteo_scian_2012["Código de la clase de actividad SCIAN"]
    .astype(str)
    .str[:2]
)

print(
    "Número de clases SCIAN distintas:",
    conteo_scian_2012[
        "Código de la clase de actividad SCIAN"
    ].nunique()
)

print("\n20 clases con más establecimientos en 2012:")
display(conteo_scian_2012.head(20))

KeyError: 'Nombre de la clase de actividad'

In [ ]:
# ==========================================
# DENUE 2012
# Normalizar nombres de columnas
# ==========================================

import re

morelia_2012.columns = [
    re.sub(r"\s+", " ", str(c)).strip()
    for c in morelia_2012.columns
]

print("Columnas relacionadas con actividad:")

for c in morelia_2012.columns:
    if "actividad" in c.lower() or "scian" in c.lower():
        print(repr(c))

Columnas relacionadas con actividad:
'Código de la clase de actividad SCIAN'
'Nombre de clase de la actividad'


In [ ]:
# ==========================================
# DENUE 2012
# Conteo por clase SCIAN de 6 dígitos
# ==========================================

conteo_scian_2012 = (
    morelia_2012
    .groupby(
        [
            "Código de la clase de actividad SCIAN",
            "Nombre de la clase de actividad"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2012 = conteo_scian_2012.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2012["Sector SCIAN"] = (
    conteo_scian_2012["Código de la clase de actividad SCIAN"]
    .astype(str)
    .str[:2]
)

print(
    "Número de clases SCIAN distintas:",
    conteo_scian_2012[
        "Código de la clase de actividad SCIAN"
    ].nunique()
)

print("\n20 clases con más establecimientos en 2012:")
display(conteo_scian_2012.head(20))

KeyError: 'Nombre de la clase de actividad'

In [ ]:
# ==========================================
# DENUE 2012
# Conteo por clase SCIAN de 6 dígitos
# ==========================================

conteo_scian_2012 = (
    morelia_2012
    .groupby(
        [
            "Código de la clase de actividad SCIAN",
            "Nombre de clase de actividad"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2012 = conteo_scian_2012.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2012["Sector SCIAN"] = (
    conteo_scian_2012["Código de la clase de actividad SCIAN"]
    .astype(str)
    .str[:2]
)

print(
    "Número de clases SCIAN distintas:",
    conteo_scian_2012[
        "Código de la clase de actividad SCIAN"
    ].nunique()
)

print("\n20 clases con más establecimientos en 2012:")
display(conteo_scian_2012.head(20))

KeyError: 'Nombre de clase de actividad'

In [ ]:
# ==========================================
# DENUE 2012
# Detectar y estandarizar columnas SCIAN
# ==========================================

# Encontrar automáticamente las columnas
col_codigo_2012 = next(
    c for c in morelia_2012.columns
    if "actividad" in c.lower() and "scian" in c.lower()
)

col_nombre_2012 = next(
    c for c in morelia_2012.columns
    if "nombre" in c.lower() and "actividad" in c.lower()
)

print("Columna de código encontrada:")
print(repr(col_codigo_2012))

print("\nColumna de nombre encontrada:")
print(repr(col_nombre_2012))

# Crear una copia con nombres estandarizados
morelia_2012_std = morelia_2012.rename(
    columns={
        col_codigo_2012: "SCIAN6",
        col_nombre_2012: "Actividad"
    }
)

print("\n¿Existe SCIAN6?", "SCIAN6" in morelia_2012_std.columns)
print("¿Existe Actividad?", "Actividad" in morelia_2012_std.columns)

Columna de código encontrada:
'Código de la clase de actividad SCIAN'

Columna de nombre encontrada:
'Nombre de clase de la actividad'

¿Existe SCIAN6? True
¿Existe Actividad? True


In [ ]:
# ==========================================
# DENUE 2012
# Conteo por clase SCIAN de 6 dígitos
# ==========================================

conteo_scian_2012 = (
    morelia_2012_std
    .groupby(
        ["SCIAN6", "Actividad"],
        dropna=False
    )
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2012 = conteo_scian_2012.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2012["Sector SCIAN"] = (
    conteo_scian_2012["SCIAN6"]
    .astype(str)
    .str[:2]
)

print(
    "Número de clases SCIAN distintas:",
    conteo_scian_2012["SCIAN6"].nunique()
)

print("\n20 clases con más establecimientos en 2012:")
display(conteo_scian_2012.head(20))

Número de clases SCIAN distintas: 676

20 clases con más establecimientos en 2012:


,SCIAN6,Actividad,Número de establecimientos,Sector SCIAN
273,461110,Comercio al por menor ...,4199,46
621,722212,Servicios de alojamiento temporal y de prepara...,1977,72
622,722219,Servicios de alojamiento temporal y de prepara...,1718,72
653,812110,Otros servicios excepto actividades gubernamen...,1583,81
292,463211,Comercio al por menor ...,1500,46
277,461130,Comercio al por menor ...,1180,46
67,311830,Industrias manufactureras ...,1071,31
540,621211,Servicios de salud y de asistencia social ...,915,62
314,465311,Comercio al por menor ...,857,46
627,811111,Otros servicios excepto actividades gubernamen...,657,81


In [ ]:
# ==========================================
# DENUE 2012
# Verificar nombres completos de actividades
# ==========================================

codigos_prueba = ["461110", "722212", "722219", "812110", "621211"]

for codigo in codigos_prueba:
    print("\nSCIAN:", codigo)

    valores = (
        morelia_2012_std.loc[
            morelia_2012_std["SCIAN6"] == codigo,
            "Actividad"
        ]
        .dropna()
        .unique()
    )

    for valor in valores:
        print(repr(valor))


SCIAN: 461110
'Comercio al por menor                                                                                                                                                                                                                                      -Comercio al por menor de abarrotes, alimentos, bebidas, hielo y tabaco-Comercio al por menor de abarrotes y alimentos-Comercio al por menor en tiendas de abarrotes, ultramarinos y misceláneas'

SCIAN: 722212
'Servicios de alojamiento temporal y de preparación de alimentos y bebidas                                                                                                                                                                                  -Servicios de preparación de alimentos y bebidas                                                                                                                                                                                                           -Restaurantes de auto

In [ ]:
# ==========================================
# DENUE 2012
# Revisar todas las columnas y un registro
# para localizar descripción SCIAN específica
# ==========================================

registro_prueba = morelia_2012[
    morelia_2012["Código de la clase de actividad SCIAN"] == "722212"
].iloc[0]

for i, columna in enumerate(morelia_2012.columns, start=1):
    print(f"{i}. {repr(columna)}")
    print("   →", repr(registro_prueba[columna]))

1. 'Llave DENUE'
   → '1191731'
2. 'NIC'
   → 'MIC227029'
3. 'NOP'
   → 'P1416227029'
4. 'Clave entidad'
   → '16'
5. 'Entidad federativa'
   → 'MICHOACAN DE OCAMPO'
6. 'Clave municipio'
   → '053'
7. 'Municipio'
   → 'MORELIA'
8. 'Clave localidad'
   → '0001'
9. 'Localidad'
   → 'Morelia'
10. 'Área geoestadística básica'
   → '0749'
11. 'Manzana'
   → '034'
12. 'Nombre de la Unidad Económica'
   → '3 TIEMPOS ALIMENTOS PREPARADOS SEMI PREPARADOS'
13. 'Razón social'
   → nan
14. 'Tipo de vialidad'
   → 'CALLE'
15. 'Nombre de la vialidad'
   → 'JOSÉ JUAN TABLADA'
16. 'Tipo entre vialidad 1'
   → 'PRIVADA'
17. 'Nombre entre vialidad 1'
   → 'NINGUNO'
18. 'Tipo entre vialidad 2'
   → 'CALLE'
19. 'Nombre entre vialidad 2'
   → 'XAVIER SORONDO'
20. 'Tipo vialidad posterior'
   → nan
21. 'Nombre vialidad posterior'
   → nan
22. 'Número exterior o kilómetro'
   → '530 2'
23. 'Edificio, piso o nivel'
   → nan
24. 'Número o letra interior'
   → nan
25. 'Tipo de asentamiento humano'
   → 'COLONIA

In [ ]:
# ==========================================
# DENUE 2012
# Localizar campos relacionados con SCIAN
# ==========================================

for columna, valor in registro_prueba.items():

    nombre = normalizar_texto(str(columna))

    if any(
        palabra in nombre
        for palabra in [
            "actividad",
            "scian",
            "descripcion",
            "sector",
            "subsector",
            "rama",
            "subrama",
            "clase"
        ]
    ):
        print(repr(columna))
        print("   →", repr(valor))

'Código de la clase de actividad SCIAN'
   → '722212'
'Nombre de clase de la actividad'
   → 'Servicios de alojamiento temporal y de preparación de alimentos y bebidas                                                                                                                                                                                  -Servicios de preparación de alimentos y bebidas                                                                                                                                                                                                           -Restaurantes de autoservicio, comida para llevar y otros restaurantes con servicio limitado                                                                                                                                                              -Restaurantes de autoservicio, comida para llevar y otros restaurantes con servicio limitado                                                              

In [ ]:
# ==========================================
# DENUE 2012
# Estandarización correcta de variables SCIAN
# ==========================================

morelia_2012_std = morelia_2012.rename(
    columns={
        col_codigo_2012: "SCIAN6",
        col_nombre_2012: "Descripcion_general"
    }
).copy()

print("SCIAN6 disponible:", "SCIAN6" in morelia_2012_std.columns)
print("Descripción general disponible:",
      "Descripcion_general" in morelia_2012_std.columns)

SCIAN6 disponible: True
Descripción general disponible: True


In [ ]:
# ==========================================
# DENUE 2012
# Conteo correcto por clase SCIAN
# ==========================================

conteo_scian_2012 = (
    morelia_2012_std
    .groupby("SCIAN6", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2012 = conteo_scian_2012.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2012["Sector SCIAN"] = (
    conteo_scian_2012["SCIAN6"]
    .astype(str)
    .str[:2]
)

print(
    "Número de clases SCIAN distintas:",
    conteo_scian_2012["SCIAN6"].nunique()
)

display(conteo_scian_2012.head(20))

Número de clases SCIAN distintas: 676


,SCIAN6,Número de establecimientos,Sector SCIAN
273,461110,4199,46
621,722212,1977,72
622,722219,1718,72
653,812110,1583,81
292,463211,1500,46
277,461130,1180,46
67,311830,1071,31
540,621211,915,62
314,465311,857,46
627,811111,657,81


In [ ]:
# ==========================================
# DENUE 2012
# Comercio al por menor y servicios
# ==========================================

# COMERCIO AL POR MENOR
comercio_2012 = conteo_scian_2012[
    conteo_scian_2012["Sector SCIAN"] == "46"
].copy()

# SERVICIOS considerados en el estudio
sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2012 = conteo_scian_2012[
    conteo_scian_2012["Sector SCIAN"].isin(sectores_servicios)
].copy()

# Diagnóstico: servicios con más de 50 establecimientos
servicios_mayores_50_2012 = servicios_2012[
    servicios_2012["Número de establecimientos"] > 50
].copy()

servicios_mayores_50_2012 = servicios_mayores_50_2012.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("COMERCIO AL POR MENOR")
print("Clases SCIAN:", comercio_2012["SCIAN6"].nunique())
print("Establecimientos:", comercio_2012["Número de establecimientos"].sum())

print("\nSERVICIOS")
print("Clases SCIAN:", servicios_2012["SCIAN6"].nunique())
print("Establecimientos:", servicios_2012["Número de establecimientos"].sum())

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_2012))
print(
    "Establecimientos:",
    servicios_mayores_50_2012["Número de establecimientos"].sum()
)

COMERCIO AL POR MENOR
Clases SCIAN: 79
Establecimientos: 16860

SERVICIOS
Clases SCIAN: 314
Establecimientos: 18429

SERVICIOS CON MÁS DE 50
Clases: 60
Establecimientos: 15600


In [ ]:
# ==========================================
# DENUE 2012
# Guardar conteos y diagnósticos
# ==========================================

conteo_scian_2012.to_csv(
    CONTEOS / "CONTEO_SCIAN_2012_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_2012.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2012.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_2012.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2012.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_2012.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2012.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Archivos 2012 guardados correctamente.")

✓ Archivos 2012 guardados correctamente.


In [ ]:
# ==========================================
# Resumen acumulado 2010-2012
# ==========================================

resumen_historico = pd.DataFrame({
    "Año": [2010, 2011, 2012],

    "Total_Morelia": [
        41342,
        42004,
        42251
    ],

    "Clases_SCIAN_Total": [
        646,
        670,
        676
    ],

    "Clases_Comercio_Minorista": [
        80,
        79,
        79
    ],

    "Establecimientos_Comercio_Minorista": [
        16693,
        16824,
        16860
    ],

    "Clases_Servicios": [
        296,
        310,
        314
    ],

    "Establecimientos_Servicios": [
        17963,
        18269,
        18429
    ],

    "Clases_Servicios_Mayores_50": [
        57,
        59,
        60
    ],

    "Establecimientos_Servicios_Mayores_50": [
        15274,
        15505,
        15600
    ]
})

display(resumen_historico)

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50
0,2010,41342,646,80,16693,296,17963,57,15274
1,2011,42004,670,79,16824,310,18269,59,15505
2,2012,42251,676,79,16860,314,18429,60,15600


In [ ]:
resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Resumen histórico guardado.")

✓ Resumen histórico guardado.


In [ ]:
# ==========================================
# BITÁCORA MAESTRA DE EDICIONES DENUE
# ==========================================

cortes_denue = pd.DataFrame({
    "Corte": [
        "2010",
        "2011",
        "2012",
        "2013_07",
        "2013_10",
        "2015",
        "2016_01",
        "2016_10",
        "2017_03",
        "2017_11",
        "2018_03",
        "2018_11",
        "2019_04",
        "2019_11",
        "2020_04",
        "2020_11",
        "2021_05",
        "2021_11",
        "2022_05",
        "2022_11",
        "2023_11",
        "2024_05",
        "2024_11",
        "2025_05",
        "2026_05"
    ]
})

cortes_denue["Estado"] = "Pendiente"

cortes_denue.loc[
    cortes_denue["Corte"].isin(["2010", "2011", "2012"]),
    "Estado"
] = "Procesado"

display(cortes_denue)

,Corte,Estado
0,2010,Procesado
1,2011,Procesado
2,2012,Procesado
3,2013_07,Pendiente
4,2013_10,Pendiente
5,2015,Pendiente
6,2016_01,Pendiente
7,2016_10,Pendiente
8,2017_03,Pendiente
9,2017_11,Pendiente


In [ ]:
cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora de ediciones guardada.")

✓ Bitácora de ediciones guardada.


In [ ]:
# ==========================================
# DENUE 2013
# Localizar ambos cortes
# ==========================================

CARPETA_2013_07 = ORIGINALES / "2013_07"
CARPETA_2013_10 = ORIGINALES / "2013_10"

print("Carpeta 2013/07 existe:", CARPETA_2013_07.exists())
print("Carpeta 2013/10 existe:", CARPETA_2013_10.exists())

print("\nArchivos 2013/07:")
for archivo in CARPETA_2013_07.iterdir():
    print("-", archivo.name)

print("\nArchivos 2013/10:")
for archivo in CARPETA_2013_10.iterdir():
    print("-", archivo.name)

Carpeta 2013/07 existe: True
Carpeta 2013/10 existe: True

Archivos 2013/07:
- denue_16_2013_csv.zip

Archivos 2013/10:
- denue_16_2013_csv (1).zip


In [ ]:
# ==========================================
# DENUE 2013
# Revisar contenido interno de ambos ZIP
# ==========================================

zip_2013_07 = next(CARPETA_2013_07.glob("*.zip"))
zip_2013_10 = next(CARPETA_2013_10.glob("*.zip"))

print("===== 2013/07 =====")
print("ZIP:", zip_2013_07.name)

with zipfile.ZipFile(zip_2013_07, "r") as z:
    contenido_07 = z.namelist()
    for nombre in contenido_07:
        print("-", nombre)

print("\n===== 2013/10 =====")
print("ZIP:", zip_2013_10.name)

with zipfile.ZipFile(zip_2013_10, "r") as z:
    contenido_10 = z.namelist()
    for nombre in contenido_10:
        print("-", nombre)

===== 2013/07 =====
ZIP: denue_16_2013_csv.zip
- denue_1_16.csv
- diccionario de datos denue 2013.pdf

===== 2013/10 =====
ZIP: denue_16_2013_csv (1).zip
- DENUE_Nacional_16.csv
- Diccionario de datos denue 2013.pdf


In [ ]:
# ==========================================
# DENUE 2013
# Auditoría estructural 2013/07 vs 2013/10
# ==========================================

# Detectar automáticamente el CSV dentro de cada ZIP
with zipfile.ZipFile(zip_2013_07, "r") as z:
    csv_2013_07 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
    )

with zipfile.ZipFile(zip_2013_10, "r") as z:
    csv_2013_10 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
    )

print("CSV interno 2013/07:", csv_2013_07)
print("CSV interno 2013/10:", csv_2013_10)

# Cargar ambos archivos
with zipfile.ZipFile(zip_2013_07, "r") as z:
    with z.open(csv_2013_07) as archivo:
        denue_2013_07 = pd.read_csv(
            archivo,
            encoding="latin1",
            dtype=str
        )

with zipfile.ZipFile(zip_2013_10, "r") as z:
    with z.open(csv_2013_10) as archivo:
        denue_2013_10 = pd.read_csv(
            archivo,
            encoding="latin1",
            dtype=str
        )

print("\n===== 2013/07 =====")
print("Filas:", len(denue_2013_07))
print("Columnas:", len(denue_2013_07.columns))

print("\n===== 2013/10 =====")
print("Filas:", len(denue_2013_10))
print("Columnas:", len(denue_2013_10.columns))

print("\n¿Tienen exactamente las mismas columnas?")
print(list(denue_2013_07.columns) == list(denue_2013_10.columns))

CSV interno 2013/07: denue_1_16.csv
CSV interno 2013/10: DENUE_Nacional_16.csv

===== 2013/07 =====
Filas: 205109
Columnas: 28

===== 2013/10 =====
Filas: 205109
Columnas: 44

¿Tienen exactamente las mismas columnas?
False


In [ ]:
# ==========================================
# Comparar columnas 2013/07 vs 2013/10
# ==========================================

cols_07 = set(denue_2013_07.columns)
cols_10 = set(denue_2013_10.columns)

print("Columnas comunes:", len(cols_07 & cols_10))

print("\nSolo en 2013/07:")
for c in sorted(cols_07 - cols_10):
    print("-", repr(c))

print("\nSolo en 2013/10:")
for c in sorted(cols_10 - cols_07):
    print("-", repr(c))

Columnas comunes: 28

Solo en 2013/07:

Solo en 2013/10:
- 'Clave entidad'
- 'Clave localidad'
- 'Clave municipio'
- 'Correo electrónico 2'
- 'Estatus de control de registros'
- 'Extensión 1'
- 'Extensión 2'
- 'Nombre entre vialidad 1'
- 'Nombre entre vialidad 2'
- 'Nombre vialidad posterior'
- 'Número de FAX'
- 'Número de teléfono 2'
- 'Personal ocupado (estrato)'
- 'Tipo entre vialidad 1'
- 'Tipo entre vialidad 2'
- 'Tipo vialidad posterior'


In [ ]:
# ==========================================
# DENUE 2013
# Comparar registros de Morelia
# ==========================================

for df in [denue_2013_07, denue_2013_10]:
    if "Clave municipio" in df.columns:
        df["Clave municipio"] = (
            df["Clave municipio"]
            .astype(str)
            .str.strip()
            .str.zfill(3)
        )

morelia_2013_07 = denue_2013_07[
    denue_2013_07["Clave municipio"] == "053"
].copy()

morelia_2013_10 = denue_2013_10[
    denue_2013_10["Clave municipio"] == "053"
].copy()

print("Morelia 2013/07:", len(morelia_2013_07))
print("Morelia 2013/10:", len(morelia_2013_10))

KeyError: 'Clave municipio'

In [ ]:
# ==========================================
# DENUE 2013
# Comparar Morelia entre julio y octubre
# usando variable común
# ==========================================

# Normalizar nombre del municipio
for df in [denue_2013_07, denue_2013_10]:
    df["Municipio_normalizado"] = (
        df["Municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_2013_07 = denue_2013_07[
    denue_2013_07["Municipio_normalizado"] == "MORELIA"
].copy()

morelia_2013_10 = denue_2013_10[
    denue_2013_10["Municipio_normalizado"] == "MORELIA"
].copy()

print("Morelia 2013/07:", len(morelia_2013_07))
print("Morelia 2013/10:", len(morelia_2013_10))

Morelia 2013/07: 42672
Morelia 2013/10: 42672


In [ ]:
# Validar Morelia en octubre con clave municipal 053

morelia_clave_2013_10 = (
    denue_2013_10["Clave municipio"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
    .eq("053")
)

print(
    "Morelia octubre por clave 053:",
    morelia_clave_2013_10.sum()
)

print(
    "¿Coincide nombre MORELIA con clave 053?:",
    morelia_clave_2013_10.sum() == len(morelia_2013_10)
)

Morelia octubre por clave 053: 42672
¿Coincide nombre MORELIA con clave 053?: True


In [ ]:
print(
    "Llave DENUE en julio:",
    "Llave DENUE" in morelia_2013_07.columns
)

print(
    "Llave DENUE en octubre:",
    "Llave DENUE" in morelia_2013_10.columns
)

Llave DENUE en julio: True
Llave DENUE en octubre: True


In [ ]:
# ==========================================
# DENUE 2013
# Comparar establecimientos mediante Llave DENUE
# ==========================================

llaves_07 = set(
    morelia_2013_07["Llave DENUE"].dropna().astype(str)
)

llaves_10 = set(
    morelia_2013_10["Llave DENUE"].dropna().astype(str)
)

print("Llaves Morelia julio:", len(llaves_07))
print("Llaves Morelia octubre:", len(llaves_10))

print("\nLlaves presentes en ambos cortes:",
      len(llaves_07 & llaves_10))

print("Llaves solo en julio:",
      len(llaves_07 - llaves_10))

print("Llaves solo en octubre:",
      len(llaves_10 - llaves_07))

Llaves Morelia julio: 42672
Llaves Morelia octubre: 42672

Llaves presentes en ambos cortes: 42630
Llaves solo en julio: 42
Llaves solo en octubre: 42


In [ ]:
# ==========================================
# DENUE 2013
# Registros exclusivos de cada corte
# ==========================================

solo_julio_2013 = morelia_2013_07[
    morelia_2013_07["Llave DENUE"].astype(str).isin(llaves_07 - llaves_10)
].copy()

solo_octubre_2013 = morelia_2013_10[
    morelia_2013_10["Llave DENUE"].astype(str).isin(llaves_10 - llaves_07)
].copy()

print("Registros solo julio:", len(solo_julio_2013))
print("Registros solo octubre:", len(solo_octubre_2013))

Registros solo julio: 42
Registros solo octubre: 42


In [ ]:
# ==========================================
# Columnas comunes útiles para comparar
# los 42 registros exclusivos
# ==========================================

columnas_comunes = [
    c for c in morelia_2013_07.columns
    if c in morelia_2013_10.columns
]

palabras_utiles = [
    "nombre",
    "razón",
    "razon",
    "actividad",
    "scian",
    "municipio",
    "localidad",
    "calle",
    "vialidad",
    "numero",
    "número",
    "latitud",
    "longitud",
    "nic",
    "nop",
    "llave"
]

columnas_utiles = [
    c for c in columnas_comunes
    if any(p in c.lower() for p in palabras_utiles)
]

print("Columnas útiles comunes:")
for c in columnas_utiles:
    print("-", repr(c))

Columnas útiles comunes:
- 'Llave DENUE'
- 'Municipio'
- 'Localidad'
- 'Nombre de la Unidad Económica'
- 'Razón social'
- 'Tipo de vialidad'
- 'Nombre de la vialidad'
- 'Número exterior o kilómetro'
- 'Número o letra interior'
- 'Nombre de asentamiento humano'
- 'Número de teléfono 1'
- 'Código de la clase de actividad SCIAN'
- 'Nombre de clase de la actividad'
- 'Número de local'
- 'Correo electrónico 1'
- 'Latitud'
- 'Longitud'
- 'Municipio_normalizado'


In [ ]:
# ==========================================
# DENUE 2013
# Comparar registros comunes por Llave DENUE
# ==========================================

comunes_07 = morelia_2013_07[
    morelia_2013_07["Llave DENUE"].astype(str).isin(llaves_07 & llaves_10)
].copy()

comunes_10 = morelia_2013_10[
    morelia_2013_10["Llave DENUE"].astype(str).isin(llaves_07 & llaves_10)
].copy()

# Dejar únicamente las 28 columnas originales comunes
cols_comunes_reales = [
    c for c in denue_2013_07.columns
    if c in denue_2013_10.columns
    and c != "Municipio_normalizado"
]

# Ordenar por llave
comunes_07 = comunes_07.sort_values("Llave DENUE").reset_index(drop=True)
comunes_10 = comunes_10.sort_values("Llave DENUE").reset_index(drop=True)

# Comparar campo por campo
comparacion_campos = {}

for col in cols_comunes_reales:
    a = comunes_07[col].fillna("").astype(str).str.strip()
    b = comunes_10[col].fillna("").astype(str).str.strip()

    comparacion_campos[col] = (a != b).sum()

cambios_campos = pd.Series(comparacion_campos).sort_values(ascending=False)

print("Cambios detectados entre julio y octubre")
print("en las 42,630 llaves comunes:\n")

display(cambios_campos[cambios_campos > 0])

Cambios detectados entre julio y octubre
en las 42,630 llaves comunes:



Nombre de asentamiento humano    1994
Tipo de asentamiento humano      1172
Nombre de la Unidad Económica    1039
dtype: int64

In [ ]:
# ==========================================
# DENUE 2013
# ¿Los 42 exclusivos son los mismos
# establecimientos con distinta Llave DENUE?
# ==========================================

# Preparar campos de comparación
for df in [solo_julio_2013, solo_octubre_2013]:
    df["Latitud_cmp"] = df["Latitud"].astype(str).str.strip()
    df["Longitud_cmp"] = df["Longitud"].astype(str).str.strip()
    df["SCIAN_cmp"] = (
        df["Código de la clase de actividad SCIAN"]
        .astype(str)
        .str.strip()
    )

# Comparar por coordenadas exactas + SCIAN
posibles_mismos = pd.merge(
    solo_julio_2013,
    solo_octubre_2013,
    on=["Latitud_cmp", "Longitud_cmp", "SCIAN_cmp"],
    how="inner",
    suffixes=("_julio", "_octubre")
)

print("Registros julio exclusivos:", len(solo_julio_2013))
print("Registros octubre exclusivos:", len(solo_octubre_2013))

print(
    "\nCoincidencias exactas por coordenadas + SCIAN:",
    len(posibles_mismos)
)

Registros julio exclusivos: 42
Registros octubre exclusivos: 42

Coincidencias exactas por coordenadas + SCIAN: 50


In [ ]:
# ==========================================
# DENUE 2013
# Comparar los 42 exclusivos mediante NIC y NOP
# ==========================================

for campo in ["NIC", "NOP"]:
    print("\n==============================")
    print(campo)
    print("==============================")

    print("Existe en julio:", campo in solo_julio_2013.columns)
    print("Existe en octubre:", campo in solo_octubre_2013.columns)

    if campo in solo_julio_2013.columns and campo in solo_octubre_2013.columns:

        julio = set(
            solo_julio_2013[campo]
            .dropna()
            .astype(str)
            .str.strip()
        )

        octubre = set(
            solo_octubre_2013[campo]
            .dropna()
            .astype(str)
            .str.strip()
        )

        print("Valores julio:", len(julio))
        print("Valores octubre:", len(octubre))
        print("Coincidencias:", len(julio & octubre))
        print("Solo julio:", len(julio - octubre))
        print("Solo octubre:", len(octubre - julio))


NIC
Existe en julio: False
Existe en octubre: False

NOP
Existe en julio: False
Existe en octubre: False


In [ ]:
# ==========================================
# DENUE 2013
# Crear huella compuesta para los 42 exclusivos
# ==========================================

import unicodedata
import re

def normalizar_valor(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"\s+", " ", x)
    return x

campos_huella = [
    "Código de la clase de actividad SCIAN",
    "Latitud",
    "Longitud",
    "Nombre de la Unidad Económica",
    "Nombre de la vialidad",
    "Número exterior o kilómetro",
    "Número o letra interior",
    "Localidad"
]

def crear_huella(df):
    copia = df.copy()

    for campo in campos_huella:
        copia[campo + "_norm"] = copia[campo].apply(normalizar_valor)

    columnas_norm = [c + "_norm" for c in campos_huella]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

solo_julio_h = crear_huella(solo_julio_2013)
solo_octubre_h = crear_huella(solo_octubre_2013)

huellas_julio = set(solo_julio_h["HUELLA"])
huellas_octubre = set(solo_octubre_h["HUELLA"])

print("Registros julio:", len(solo_julio_h))
print("Registros octubre:", len(solo_octubre_h))

print("\nHuellas únicas julio:", len(huellas_julio))
print("Huellas únicas octubre:", len(huellas_octubre))

print("\nCoincidencias exactas de huella:",
      len(huellas_julio & huellas_octubre))

print("Huellas solo julio:",
      len(huellas_julio - huellas_octubre))

print("Huellas solo octubre:",
      len(huellas_octubre - huellas_julio))

Registros julio: 42
Registros octubre: 42

Huellas únicas julio: 42
Huellas únicas octubre: 42

Coincidencias exactas de huella: 42
Huellas solo julio: 0
Huellas solo octubre: 0


In [ ]:
# ==========================================
# DENUE 2013
# Comparar distribución SCIAN
# Julio vs Octubre
# ==========================================

# Detectar columna SCIAN común
col_scian_2013 = next(
    c for c in morelia_2013_07.columns
    if "actividad" in c.lower()
    and "scian" in c.lower()
)

print("Columna SCIAN:", repr(col_scian_2013))

conteo_07 = (
    morelia_2013_07[col_scian_2013]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_10 = (
    morelia_2013_10[col_scian_2013]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

print("\nClases SCIAN julio:", conteo_07.size)
print("Clases SCIAN octubre:", conteo_10.size)

print(
    "¿Distribución SCIAN exactamente igual?:",
    conteo_07.equals(conteo_10)
)

# Mostrar diferencias si existieran
comparacion_scian_2013 = pd.concat(
    [conteo_07.rename("2013_07"),
     conteo_10.rename("2013_10")],
    axis=1
).fillna(0).astype(int)

comparacion_scian_2013["Diferencia"] = (
    comparacion_scian_2013["2013_10"]
    - comparacion_scian_2013["2013_07"]
)

diferencias_scian_2013 = comparacion_scian_2013[
    comparacion_scian_2013["Diferencia"] != 0
]

print("\nClases con diferencias:")
print(len(diferencias_scian_2013))

display(diferencias_scian_2013)

Columna SCIAN: 'Código de la clase de actividad SCIAN'

Clases SCIAN julio: 684
Clases SCIAN octubre: 684
¿Distribución SCIAN exactamente igual?: True

Clases con diferencias:
0


,2013_07,2013_10,Diferencia
Código de la clase de actividad SCIAN,,,


### Decisión metodológica para 2013

Se auditaron las ediciones DENUE 2013/07 y 2013/10 antes de seleccionar
el corte anual.

Ambas ediciones contienen 42,672 registros correspondientes al municipio
de Morelia y 684 clases SCIAN, con una distribución por clase de actividad
exactamente igual.

De las Llaves DENUE, 42,630 permanecieron sin cambio entre ambas ediciones.
Los 42 registros cuya Llave DENUE fue diferente coincidieron exactamente
mediante una huella compuesta por código SCIAN, coordenadas, nombre de la
unidad económica, domicilio y localidad, por lo que no fueron interpretados
como incorporaciones o desapariciones de establecimientos.

La edición 2013/10 incorpora variables adicionales y actualizaciones de
información descriptiva. Por ello, se utiliza DENUE 2013/10 como corte
representativo de 2013 para la serie histórica, conservando DENUE 2013/07
como edición auditada.

In [ ]:
# ==========================================
# DENUE 2013
# Guardar corte representativo 2013/10
# ==========================================

archivo_salida_2013 = MORELIA / "DENUE_2013_MORELIA.csv"

morelia_2013_10.to_csv(
    archivo_salida_2013,
    index=False,
    encoding="utf-8-sig"
)

print("✓ DENUE 2013/10 seleccionado y guardado como corte 2013:")
print(archivo_salida_2013)

✓ DENUE 2013/10 seleccionado y guardado como corte 2013:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2013_MORELIA.csv


In [ ]:
# ==========================================
# DENUE 2013
# Estandarizar SCIAN y hacer conteo
# ==========================================

col_scian_2013 = next(
    c for c in morelia_2013_10.columns
    if "actividad" in c.lower()
    and "scian" in c.lower()
)

morelia_2013_std = morelia_2013_10.rename(
    columns={col_scian_2013: "SCIAN6"}
).copy()

conteo_scian_2013 = (
    morelia_2013_std
    .groupby("SCIAN6", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2013 = conteo_scian_2013.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2013["Sector SCIAN"] = (
    conteo_scian_2013["SCIAN6"]
    .astype(str)
    .str.strip()
    .str[:2]
)

print("Total Morelia 2013:", len(morelia_2013_std))
print("Clases SCIAN:", conteo_scian_2013["SCIAN6"].nunique())

display(conteo_scian_2013.head(20))

Total Morelia 2013: 42672
Clases SCIAN: 684


,SCIAN6,Número de establecimientos,Sector SCIAN
278,461110,4214,46
630,722212,1971,72
631,722219,1729,72
662,812110,1587,81
297,463211,1514,46
282,461130,1179,46
67,311830,1070,31
549,621211,914,62
319,465311,861,46
636,811111,662,81


In [ ]:
# ==========================================
# DENUE 2013
# Comercio al por menor y servicios
# ==========================================

# COMERCIO AL POR MENOR
comercio_2013 = conteo_scian_2013[
    conteo_scian_2013["Sector SCIAN"] == "46"
].copy()

# SERVICIOS
sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2013 = conteo_scian_2013[
    conteo_scian_2013["Sector SCIAN"].isin(sectores_servicios)
].copy()

# Diagnóstico > 50 establecimientos
servicios_mayores_50_2013 = servicios_2013[
    servicios_2013["Número de establecimientos"] > 50
].copy()

servicios_mayores_50_2013 = servicios_mayores_50_2013.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("COMERCIO AL POR MENOR")
print("Clases SCIAN:", comercio_2013["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_2013["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases SCIAN:", servicios_2013["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_2013["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_2013))
print(
    "Establecimientos:",
    servicios_mayores_50_2013[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases SCIAN: 79
Establecimientos: 17045

SERVICIOS
Clases SCIAN: 318
Establecimientos: 18752

SERVICIOS CON MÁS DE 50
Clases: 62
Establecimientos: 15828


In [ ]:
# ==========================================
# DENUE 2013
# Guardar conteos y diagnósticos
# ==========================================

conteo_scian_2013.to_csv(
    CONTEOS / "CONTEO_SCIAN_2013_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_2013.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_2013.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_2013.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Archivos 2013 guardados correctamente.")

✓ Archivos 2013 guardados correctamente.


In [ ]:
# ==========================================
# Agregar 2013 al resumen histórico
# ==========================================

fila_2013 = pd.DataFrame([{
    "Año": 2013,
    "Total_Morelia": 42672,
    "Clases_SCIAN_Total": 684,
    "Clases_Comercio_Minorista": 79,
    "Establecimientos_Comercio_Minorista": 17045,
    "Clases_Servicios": 318,
    "Establecimientos_Servicios": 18752,
    "Clases_Servicios_Mayores_50": 62,
    "Establecimientos_Servicios_Mayores_50": 15828
}])

# Evitar duplicar 2013 si ejecutamos la celda otra vez
resumen_historico = resumen_historico[
    resumen_historico["Año"] != 2013
]

resumen_historico = pd.concat(
    [resumen_historico, fila_2013],
    ignore_index=True
).sort_values("Año")

display(resumen_historico)

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Resumen histórico actualizado hasta 2013.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50
0,2010,41342,646,80,16693,296,17963,57,15274
1,2011,42004,670,79,16824,310,18269,59,15505
2,2012,42251,676,79,16860,314,18429,60,15600
3,2013,42672,684,79,17045,318,18752,62,15828


✓ Resumen histórico actualizado hasta 2013.


In [ ]:
cortes_denue.loc[
    cortes_denue["Corte"] == "2013_07",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == "2013_10",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(cortes_denue.head(6))

,Corte,Estado
0,2010,Procesado
1,2011,Procesado
2,2012,Procesado
3,2013_07,Auditado - no seleccionado
4,2013_10,Procesado - corte seleccionado
5,2015,Pendiente


# DENUE 2015

In [ ]:
# ==========================================
# DENUE 2015
# Auditoría inicial
# ==========================================

CARPETA_2015 = ORIGINALES / "2015"

print("Existe carpeta 2015:", CARPETA_2015.exists())

print("\nContenido:")
for archivo in CARPETA_2015.iterdir():
    print("-", archivo.name)

Existe carpeta 2015: True

Contenido:
- denue_16_25022015_csv.zip


In [ ]:
# ==========================================
# DENUE 2015
# Revisar contenido interno del ZIP
# ==========================================

zip_2015 = CARPETA_2015 / "denue_16_25022015_csv.zip"

print("ZIP encontrado:", zip_2015.exists())
print("Nombre:", zip_2015.name)

with zipfile.ZipFile(zip_2015, "r") as z:
    print("\nContenido del ZIP:")
    for nombre in z.namelist():
        print("-", nombre)

ZIP encontrado: True
Nombre: denue_16_25022015_csv.zip

Contenido del ZIP:
- DENUE_INEGI_16_.csv


In [ ]:
# ==========================================
# DENUE 2015
# Auditoría estructural inicial
# ==========================================

with zipfile.ZipFile(zip_2015, "r") as z:
    with z.open("DENUE_INEGI_16_.csv") as archivo:
        denue_2015 = pd.read_csv(
            archivo,
            encoding="latin1",
            dtype=str
        )

print("===== DENUE 2015 =====")
print("Filas:", len(denue_2015))
print("Columnas:", len(denue_2015.columns))

print("\nNombres de columnas:")
for i, columna in enumerate(denue_2015.columns, start=1):
    print(i, repr(columna))

===== DENUE 2015 =====
Filas: 225998
Columnas: 41

Nombres de columnas:
1 'ID'
2 'Nombre de la Unidad EconÃ³mica'
3 'RazÃ³n social'
4 'CÃ³digo de la clase de actividad SCIAN'
5 'Nombre de clase de la actividad'
6 'Descripcion estrato personal ocupado'
7 'Tipo de vialidad'
8 'Nombre de la vialidad'
9 'Tipo de entre vialidad 1'
10 'Nombre de entre vialidad 1'
11 'Tipo de entre vialidad 2'
12 'Nombre de entre vialidad 2'
13 'Tipo de entre vialidad 3'
14 'Nombre de entre vialidad 3'
15 'NÃºmero exterior o kilÃ³metro'
16 'Letra exterior'
17 'Edificio'
18 'Edificio Piso'
19 'NÃºmero interior'
20 'Letra interior'
21 'Tipo de asentamiento humano'
22 'Nombre de asentamiento humano'
23 'Tipo centro comercial'
24 'Corredor industrial, centro comercial o mercado pÃºblico'
25 'NÃºmero de local'
26 'CÃ³digo Postal'
27 'Clave entidad'
28 'Entidad federativa'
29 'Clave municipio'
30 'Municipio'
31 'Clave localidad'
32 'Localidad'
33 'Ã\x81rea geoestadÃ\xadstica bÃ¡sica '
34 'Manzana'
35 'NÃºmero de te

In [ ]:
# ==========================================
# Comparar estructura 2015 vs 2013/10
# ==========================================

cols_2013 = set(denue_2013_10.columns)
cols_2015 = set(denue_2015.columns)

print("Columnas comunes:", len(cols_2013 & cols_2015))

print("\nSolo en 2013/10:")
for c in sorted(cols_2013 - cols_2015):
    print("-", repr(c))

print("\nSolo en 2015:")
for c in sorted(cols_2015 - cols_2013):
    print("-", repr(c))

Columnas comunes: 17

Solo en 2013/10:
- 'Corredor industrial, centro comercial o mercado público'
- 'Correo electrónico 1'
- 'Correo electrónico 2'
- 'Código Postal'
- 'Código de la clase de actividad SCIAN'
- 'Edificio, piso o nivel'
- 'Estatus de control de registros'
- 'Extensión 1'
- 'Extensión 2'
- 'Fecha de incorporación al DENUE'
- 'Llave DENUE'
- 'Municipio_normalizado'
- 'Nombre de la Unidad Económica'
- 'Nombre entre vialidad 1'
- 'Nombre entre vialidad 2'
- 'Nombre vialidad posterior'
- 'Número de FAX'
- 'Número de local'
- 'Número de teléfono 1'
- 'Número de teléfono 2'
- 'Número exterior o kilómetro'
- 'Número o letra interior'
- 'Personal ocupado (estrato)'
- 'Razón social'
- 'Tipo entre vialidad 1'
- 'Tipo entre vialidad 2'
- 'Tipo vialidad posterior'
- 'Área geoestadística básica '

Solo en 2015:
- 'Corredor industrial, centro comercial o mercado pÃºblico'
- 'Correo electrÃ³nico'
- 'CÃ³digo Postal'
- 'CÃ³digo de la clase de actividad SCIAN'
- 'Edificio'
- 'Edificio Pis

In [ ]:
# ==========================================
# DENUE 2015
# Recargar con codificación correcta
# ==========================================

with zipfile.ZipFile(zip_2015, "r") as z:
    with z.open("DENUE_INEGI_16_.csv") as archivo:
        denue_2015 = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            dtype=str
        )

print("Filas:", len(denue_2015))
print("Columnas:", len(denue_2015.columns))

print("\nPrimeras columnas:")
for i, columna in enumerate(denue_2015.columns, start=1):
    print(i, repr(columna))

Filas: 225998
Columnas: 41

Primeras columnas:
1 'ID'
2 'Nombre de la Unidad Económica'
3 'Razón social'
4 'Código de la clase de actividad SCIAN'
5 'Nombre de clase de la actividad'
6 'Descripcion estrato personal ocupado'
7 'Tipo de vialidad'
8 'Nombre de la vialidad'
9 'Tipo de entre vialidad 1'
10 'Nombre de entre vialidad 1'
11 'Tipo de entre vialidad 2'
12 'Nombre de entre vialidad 2'
13 'Tipo de entre vialidad 3'
14 'Nombre de entre vialidad 3'
15 'Número exterior o kilómetro'
16 'Letra exterior'
17 'Edificio'
18 'Edificio Piso'
19 'Número interior'
20 'Letra interior'
21 'Tipo de asentamiento humano'
22 'Nombre de asentamiento humano'
23 'Tipo centro comercial'
24 'Corredor industrial, centro comercial o mercado público'
25 'Número de local'
26 'Código Postal'
27 'Clave entidad'
28 'Entidad federativa'
29 'Clave municipio'
30 'Municipio'
31 'Clave localidad'
32 'Localidad'
33 'Área geoestadística básica '
34 'Manzana'
35 'Número de teléfono'
36 'Correo electrónico'
37 'Sitio en

In [ ]:
# ==========================================
# Comparar estructura 2015 vs 2013/10
# CON CODIFICACIÓN CORRECTA
# ==========================================

cols_2013 = set(
    c for c in denue_2013_10.columns
    if c != "Municipio_normalizado"
)

cols_2015 = set(denue_2015.columns)

print("Columnas comunes:", len(cols_2013 & cols_2015))

print("\nSolo en 2013/10:")
for c in sorted(cols_2013 - cols_2015):
    print("-", repr(c))

print("\nSolo en 2015:")
for c in sorted(cols_2015 - cols_2013):
    print("-", repr(c))

Columnas comunes: 26

Solo en 2013/10:
- 'Correo electrónico 1'
- 'Correo electrónico 2'
- 'Edificio, piso o nivel'
- 'Estatus de control de registros'
- 'Extensión 1'
- 'Extensión 2'
- 'Llave DENUE'
- 'Nombre entre vialidad 1'
- 'Nombre entre vialidad 2'
- 'Nombre vialidad posterior'
- 'Número de FAX'
- 'Número de teléfono 1'
- 'Número de teléfono 2'
- 'Número o letra interior'
- 'Personal ocupado (estrato)'
- 'Tipo entre vialidad 1'
- 'Tipo entre vialidad 2'
- 'Tipo vialidad posterior'

Solo en 2015:
- 'Correo electrónico'
- 'Edificio'
- 'Edificio Piso'
- 'ID'
- 'Letra exterior'
- 'Letra interior'
- 'Nombre de entre vialidad 1'
- 'Nombre de entre vialidad 2'
- 'Nombre de entre vialidad 3'
- 'Número de teléfono'
- 'Número interior'
- 'Tipo centro comercial'
- 'Tipo de entre vialidad 1'
- 'Tipo de entre vialidad 2'
- 'Tipo de entre vialidad 3'


In [ ]:
# ==========================================
# DENUE 2015
# Auditoría de identificador y Morelia
# ==========================================

print("COLUMNAS RELEVANTES 2015:\n")

for c in denue_2015.columns:
    if any(p in c.lower() for p in [
        "id", "municipio", "localidad",
        "actividad", "scian",
        "incorpor", "fecha",
        "estatus", "control",
        "latitud", "longitud"
    ]):
        print(repr(c))

# ------------------------------------------
# Identificador ID
# ------------------------------------------

print("\n===== ID =====")

print("Registros totales:", len(denue_2015))
print("ID no nulos:", denue_2015["ID"].notna().sum())
print("ID únicos:", denue_2015["ID"].nunique())

print("\nEjemplos de ID:")
print(denue_2015["ID"].dropna().head(10).tolist())

print("\nLongitudes de ID:")
print(
    denue_2015["ID"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

# ------------------------------------------
# Municipio
# ------------------------------------------

print("\n===== MORELIA =====")

print(
    "¿Existe Clave municipio?:",
    "Clave municipio" in denue_2015.columns
)

print(
    "¿Existe Municipio?:",
    "Municipio" in denue_2015.columns
)


COLUMNAS RELEVANTES 2015:

'ID'
'Nombre de la Unidad Económica'
'Código de la clase de actividad SCIAN'
'Nombre de clase de la actividad'
'Tipo de vialidad'
'Nombre de la vialidad'
'Tipo de entre vialidad 1'
'Nombre de entre vialidad 1'
'Tipo de entre vialidad 2'
'Nombre de entre vialidad 2'
'Tipo de entre vialidad 3'
'Nombre de entre vialidad 3'
'Clave entidad'
'Entidad federativa'
'Clave municipio'
'Municipio'
'Clave localidad'
'Localidad'
'Latitud'
'Longitud'
'Fecha de incorporación al DENUE'

===== ID =====
Registros totales: 225998
ID no nulos: 225998
ID únicos: 225998

Ejemplos de ID:
['6171723', '6441320', '2599326', '2600292', '2599927', '2599381', '2591399', '2727451', '2736836', '6171159']

Longitudes de ID:
ID
7    225998
Name: count, dtype: int64

===== MORELIA =====
¿Existe Clave municipio?: True
¿Existe Municipio?: True


In [ ]:
print("Registros totales:", len(denue_2015))
print("ID no nulos:", denue_2015["ID"].notna().sum())
print("ID únicos:", denue_2015["ID"].nunique())

print("\nPrimeros 10 ID:")
print(denue_2015["ID"].dropna().head(10).tolist())

print("\nLongitud de los ID:")
print(
    denue_2015["ID"]
    .dropna()
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

Registros totales: 225998
ID no nulos: 225998
ID únicos: 225998

Primeros 10 ID:
['6171723', '6441320', '2599326', '2600292', '2599927', '2599381', '2591399', '2727451', '2736836', '6171159']

Longitud de los ID:
ID
7    225998
Name: count, dtype: int64


In [ ]:
denue_2015["Clave municipio"] = (
    denue_2015["Clave municipio"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

morelia_2015 = denue_2015[
    denue_2015["Clave municipio"] == "053"
].copy()

print("Registros Michoacán 2015:", len(denue_2015))
print("Registros Morelia 2015:", len(morelia_2015))
print("ID no nulos en Morelia:", morelia_2015["ID"].notna().sum())
print("ID únicos en Morelia:", morelia_2015["ID"].nunique())

Registros Michoacán 2015: 225998
Registros Morelia 2015: 44814
ID no nulos en Morelia: 44814
ID únicos en Morelia: 44814


In [ ]:
# ==========================================
# DENUE 2013/10 vs 2015
# Comparar Llave DENUE con ID
# ==========================================

llaves_2013 = set(
    morelia_2013_10["Llave DENUE"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_2015 = set(
    morelia_2015["ID"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Llaves únicas Morelia 2013:", len(llaves_2013))
print("ID únicos Morelia 2015:", len(ids_2015))

print("\nCoincidencias exactas:", len(llaves_2013 & ids_2015))
print("Solo en 2013:", len(llaves_2013 - ids_2015))
print("Solo en 2015:", len(ids_2015 - llaves_2013))

print(
    "\nPorcentaje de Llaves 2013 que reaparecen como ID en 2015:",
    round(len(llaves_2013 & ids_2015) / len(llaves_2013) * 100, 2),
    "%"
)

Llaves únicas Morelia 2013: 42672
ID únicos Morelia 2015: 44814

Coincidencias exactas: 320
Solo en 2013: 42352
Solo en 2015: 44494

Porcentaje de Llaves 2013 que reaparecen como ID en 2015: 0.75 %


In [ ]:
# ==========================================
# DENUE 2015
# Fechas de incorporación al DENUE
# ==========================================

print("FECHA DE INCORPORACIÓN AL DENUE - MORELIA 2015")

display(
    morelia_2015["Fecha de incorporación al DENUE"]
    .value_counts(dropna=False)
)

FECHA DE INCORPORACIÓN AL DENUE - MORELIA 2015


Fecha de incorporación al DENUE
JULIO 2010        25925
DICIEMBRE 2014    17932
JULIO 2013          408
MARZO 2011          352
JUNIO 2012          197
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2013 vs 2015
# Crear huella comparable de establecimiento
# ==========================================

campos_huella_13_15 = [
    "Código de la clase de actividad SCIAN",
    "Latitud",
    "Longitud",
    "Nombre de la Unidad Económica",
    "Nombre de la vialidad",
    "Número exterior o kilómetro",
    "Localidad"
]

def crear_huella_comparable(df):
    copia = df.copy()

    for campo in campos_huella_13_15:
        copia[campo + "_norm"] = copia[campo].apply(normalizar_valor)

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella_13_15
    ]

    copia["HUELLA_13_15"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

m13 = crear_huella_comparable(morelia_2013_10)
m15 = crear_huella_comparable(morelia_2015)

huellas_2013 = set(m13["HUELLA_13_15"])
huellas_2015 = set(m15["HUELLA_13_15"])

print("Registros 2013:", len(m13))
print("Registros 2015:", len(m15))

print("\nHuellas únicas 2013:", len(huellas_2013))
print("Huellas únicas 2015:", len(huellas_2015))

print("\nHuellas presentes en ambos:")
print(len(huellas_2013 & huellas_2015))

print("\nHuellas solo 2013:")
print(len(huellas_2013 - huellas_2015))

print("\nHuellas solo 2015:")
print(len(huellas_2015 - huellas_2013))

Registros 2013: 42672
Registros 2015: 44814

Huellas únicas 2013: 42042
Huellas únicas 2015: 44101

Huellas presentes en ambos:
17

Huellas solo 2013:
42025

Huellas solo 2015:
44084


### Ruptura metodológica observada en DENUE 2015

La comparación entre DENUE 2013/10 y DENUE 2015 muestra un cambio importante
en la estructura e identificación de los registros.

Morelia pasa de 42,672 registros en 2013 a 44,814 en 2015. Sin embargo,
la variable Llave DENUE de 2013 no presenta continuidad directa con la
variable ID de 2015.

En DENUE 2015, 17,932 registros presentan diciembre de 2014 como fecha
de incorporación al DENUE. Esta fecha corresponde a incorporación al
directorio y no se interpreta como fecha de nacimiento de la empresa.

Una comparación mediante una huella estricta basada en SCIAN, coordenadas,
nombre y domicilio produjo muy pocas coincidencias exactas entre ambas
ediciones, lo que evidencia que no resulta metodológicamente válido inferir
permanencias, incorporaciones o desapariciones mediante una comparación
directa de identificadores entre 2013 y 2015.

Por ello, el corte 2015 se considera un punto de ruptura en la comparabilidad
individual y se mantiene la comparación histórica a nivel agregado por
clase SCIAN mientras se evalúan procedimientos de homologación posteriores.

In [ ]:
# ==========================================
# DENUE 2015
# Estandarizar código SCIAN
# ==========================================

col_scian_2015 = next(
    c for c in morelia_2015.columns
    if "actividad" in c.lower()
    and "scian" in c.lower()
)

print("Columna SCIAN encontrada:", repr(col_scian_2015))

morelia_2015_std = morelia_2015.rename(
    columns={col_scian_2015: "SCIAN6"}
).copy()

morelia_2015_std["SCIAN6"] = (
    morelia_2015_std["SCIAN6"]
    .astype(str)
    .str.strip()
)

print("SCIAN6 disponible:", "SCIAN6" in morelia_2015_std.columns)

Columna SCIAN encontrada: 'Código de la clase de actividad SCIAN'
SCIAN6 disponible: True


In [ ]:
# ==========================================
# DENUE 2015
# Conteo por clase SCIAN
# ==========================================

conteo_scian_2015 = (
    morelia_2015_std
    .groupby("SCIAN6", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2015 = conteo_scian_2015.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2015["Sector SCIAN"] = (
    conteo_scian_2015["SCIAN6"].str[:2]
)

print("Total Morelia 2015:", len(morelia_2015_std))
print(
    "Clases SCIAN distintas:",
    conteo_scian_2015["SCIAN6"].nunique()
)

display(conteo_scian_2015.head(20))

Total Morelia 2015: 44814
Clases SCIAN distintas: 660


,SCIAN6,Número de establecimientos,Sector SCIAN
255,461110,3870,46
636,812110,1767,81
275,463211,1412,46
604,722514,1287,72
59,311830,1165,31
521,621211,1055,62
603,722513,968,72
259,461130,894,46
297,465311,894,46
610,811111,754,81


In [ ]:
# ==========================================
# DENUE 2015
# Registrar versión SCIAN
# ==========================================

VERSION_SCIAN_2015 = "SCIAN 2013"

print("Versión utilizada en DENUE 2015:")
print(VERSION_SCIAN_2015)

Versión utilizada en DENUE 2015:
SCIAN 2013


In [ ]:
# ==========================================
# DENUE 2015
# Comercio al por menor y servicios
# ==========================================

comercio_2015 = conteo_scian_2015[
    conteo_scian_2015["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2015 = conteo_scian_2015[
    conteo_scian_2015["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_2015 = servicios_2015[
    servicios_2015["Número de establecimientos"] > 50
].copy()

servicios_mayores_50_2015 = servicios_mayores_50_2015.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("COMERCIO AL POR MENOR")
print("Clases SCIAN:", comercio_2015["SCIAN6"].nunique())
print("Establecimientos:",
      comercio_2015["Número de establecimientos"].sum())

print("\nSERVICIOS")
print("Clases SCIAN:", servicios_2015["SCIAN6"].nunique())
print("Establecimientos:",
      servicios_2015["Número de establecimientos"].sum())

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_2015))
print("Establecimientos:",
      servicios_mayores_50_2015["Número de establecimientos"].sum())

COMERCIO AL POR MENOR
Clases SCIAN: 82
Establecimientos: 16955

SERVICIOS
Clases SCIAN: 314
Establecimientos: 20722

SERVICIOS CON MÁS DE 50
Clases: 65
Establecimientos: 17981


In [ ]:
# ==========================================
# DENUE 2015
# Guardar archivo Morelia
# ==========================================

archivo_salida_2015 = MORELIA / "DENUE_2015_MORELIA.csv"

morelia_2015.to_csv(
    archivo_salida_2015,
    index=False,
    encoding="utf-8-sig"
)

print("✓ DENUE 2015 Morelia guardado:")
print(archivo_salida_2015)

✓ DENUE 2015 Morelia guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2015_MORELIA.csv


In [ ]:
# ==========================================
# DENUE 2015
# Guardar conteos y diagnósticos
# ==========================================

conteo_scian_2015.to_csv(
    CONTEOS / "CONTEO_SCIAN_2015_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_2015.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_2015.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_2015.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2015.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Archivos 2015 guardados correctamente.")

✓ Archivos 2015 guardados correctamente.


In [ ]:
# ==========================================
# Agregar 2015 al resumen histórico
# ==========================================

# Añadir columnas si todavía no existen
if "Version_SCIAN" not in resumen_historico.columns:
    resumen_historico["Version_SCIAN"] = ""

if "Observacion_metodologica" not in resumen_historico.columns:
    resumen_historico["Observacion_metodologica"] = ""

# Etiquetar años anteriores
resumen_historico.loc[
    resumen_historico["Año"].isin([2010, 2011, 2012, 2013]),
    "Version_SCIAN"
] = "SCIAN 2007"

fila_2015 = pd.DataFrame([{
    "Año": 2015,
    "Total_Morelia": 44814,
    "Clases_SCIAN_Total": 660,
    "Clases_Comercio_Minorista": 82,
    "Establecimientos_Comercio_Minorista": 16955,
    "Clases_Servicios": 314,
    "Establecimientos_Servicios": 20722,
    "Clases_Servicios_Mayores_50": 65,
    "Establecimientos_Servicios_Mayores_50": 17981,
    "Version_SCIAN": "SCIAN 2013",
    "Observacion_metodologica":
        "Ruptura respecto de 2013: actualización integral y cambio de SCIAN 2007 a SCIAN 2013."
}])

# Evitar duplicar 2015
resumen_historico = resumen_historico[
    resumen_historico["Año"] != 2015
]

resumen_historico = pd.concat(
    [resumen_historico, fila_2015],
    ignore_index=True
).sort_values("Año")

display(resumen_historico)

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Resumen actualizado hasta 2015.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...


✓ Resumen actualizado hasta 2015.


# DENUE 2016

In [ ]:
# ==========================================
# DENUE 2016
# Localizar ambos cortes
# ==========================================

CARPETA_2016_01 = ORIGINALES / "2016_01"
CARPETA_2016_10 = ORIGINALES / "2016_10"

print("Carpeta 2016/01 existe:", CARPETA_2016_01.exists())
print("Carpeta 2016/10 existe:", CARPETA_2016_10.exists())

print("\nArchivos 2016/01:")
for archivo in CARPETA_2016_01.iterdir():
    print("-", archivo.name)

print("\nArchivos 2016/10:")
for archivo in CARPETA_2016_10.iterdir():
    print("-", archivo.name)

Carpeta 2016/01 existe: True
Carpeta 2016/10 existe: True

Archivos 2016/01:
- denue_16_0116_csv.zip

Archivos 2016/10:
- denue_16_1016_csv.zip


In [ ]:
# ==========================================
# DENUE 2016
# Revisar contenido interno de ambos ZIP
# ==========================================

zip_2016_01 = next(CARPETA_2016_01.glob("*.zip"))
zip_2016_10 = next(CARPETA_2016_10.glob("*.zip"))

print("===== 2016/01 =====")
print("ZIP:", zip_2016_01.name)

with zipfile.ZipFile(zip_2016_01, "r") as z:
    contenido_01 = z.namelist()
    for nombre in contenido_01:
        print("-", nombre)

print("\n===== 2016/10 =====")
print("ZIP:", zip_2016_10.name)

with zipfile.ZipFile(zip_2016_10, "r") as z:
    contenido_10 = z.namelist()
    for nombre in contenido_10:
        print("-", nombre)

===== 2016/01 =====
ZIP: denue_16_0116_csv.zip
- denue_16_csv/conjunto_de_datos/
- denue_16_csv/conjunto_de_datos/denue_inegi_16.csv
- denue_16_csv/diccionario_de_datos/
- denue_16_csv/diccionario_de_datos/denue_diccionario_de_datos.csv
- denue_16_csv/metadatos/
- denue_16_csv/metadatos/metadatos_denue.txt

===== 2016/10 =====
ZIP: denue_16_1016_csv.zip
- denue_16_csv/conjunto_de_datos/
- denue_16_csv/conjunto_de_datos/denue_inegi_16_.csv
- denue_16_csv/diccionario_de_datos/
- denue_16_csv/diccionario_de_datos/denue_diccionario_de_datos.csv
- denue_16_csv/metadatos/
- denue_16_csv/metadatos/metadatos_denue.txt


In [ ]:
# ==========================================
# DENUE 2016
# Detectar CSV principal de ambos cortes
# ==========================================

with zipfile.ZipFile(zip_2016_01, "r") as z:
    csv_2016_01 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_2016_10, "r") as z:
    csv_2016_10 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print("CSV principal 2016/01:")
print(csv_2016_01)

print("\nCSV principal 2016/10:")
print(csv_2016_10)

CSV principal 2016/01:
denue_16_csv/conjunto_de_datos/denue_inegi_16.csv

CSV principal 2016/10:
denue_16_csv/conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ==========================================
# DENUE 2016
# Inspección inicial de ambos CSV
# ==========================================

def leer_muestra_zip(zip_path, csv_interno):
    for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(csv_interno) as archivo:
                    muestra = pd.read_csv(
                        archivo,
                        nrows=5,
                        encoding=codificacion,
                        dtype=str
                    )
            return muestra, codificacion
        except UnicodeDecodeError:
            continue

    raise ValueError("No se pudo detectar la codificación.")

muestra_2016_01, enc_01 = leer_muestra_zip(
    zip_2016_01,
    csv_2016_01
)

muestra_2016_10, enc_10 = leer_muestra_zip(
    zip_2016_10,
    csv_2016_10
)

print("2016/01")
print("Codificación:", enc_01)
print("Columnas:", len(muestra_2016_01.columns))

print("\n2016/10")
print("Codificación:", enc_10)
print("Columnas:", len(muestra_2016_10.columns))

print("\n¿Mismas columnas exactamente?")
print(
    list(muestra_2016_01.columns)
    ==
    list(muestra_2016_10.columns)
)

2016/01
Codificación: utf-8-sig
Columnas: 41

2016/10
Codificación: utf-8-sig
Columnas: 41

¿Mismas columnas exactamente?
True


In [ ]:
# ==========================================
# DENUE 2016
# Cargar ambos cortes completos
# ==========================================

with zipfile.ZipFile(zip_2016_01, "r") as z:
    with z.open(csv_2016_01) as archivo:
        denue_2016_01 = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            dtype=str
        )

with zipfile.ZipFile(zip_2016_10, "r") as z:
    with z.open(csv_2016_10) as archivo:
        denue_2016_10 = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            dtype=str
        )

print("2016/01")
print("Filas:", len(denue_2016_01))
print("Columnas:", len(denue_2016_01.columns))

print("\n2016/10")
print("Filas:", len(denue_2016_10))
print("Columnas:", len(denue_2016_10.columns))

2016/01
Filas: 227818
Columnas: 41

2016/10
Filas: 228322
Columnas: 41


In [ ]:
# ==========================================
# DENUE 2016
# Comparar Morelia enero vs octubre
# ==========================================

for df in [denue_2016_01, denue_2016_10]:
    df["Clave municipio"] = (
        df["Clave municipio"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

morelia_2016_01 = denue_2016_01[
    denue_2016_01["Clave municipio"] == "053"
].copy()

morelia_2016_10 = denue_2016_10[
    denue_2016_10["Clave municipio"] == "053"
].copy()

print("Morelia 2016/01:", len(morelia_2016_01))
print("Morelia 2016/10:", len(morelia_2016_10))

KeyError: 'Clave municipio'

In [ ]:
# ==========================================
# DENUE 2016
# Identificar variables geográficas
# ==========================================

print("Columnas relacionadas con municipio, entidad y localidad:\n")

for c in denue_2016_01.columns:
    nombre = normalizar_texto(str(c))

    if any(x in nombre for x in [
        "mun", "municipio",
        "ent", "entidad",
        "loc", "localidad"
    ]):
        print(repr(c))

Columnas relacionadas con municipio, entidad y localidad:

'tipo_asent'
'nomb_asent'
'num_local'
'cve_ent'
'entidad'
'cve_mun'
'municipio'
'cve_loc'
'localidad'


In [ ]:
# ==========================================
# DENUE 2016
# Filtrar Morelia en ambos cortes
# ==========================================

for df in [denue_2016_01, denue_2016_10]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_2016_01 = denue_2016_01[
    denue_2016_01["cve_mun"] == "053"
].copy()

morelia_2016_10 = denue_2016_10[
    denue_2016_10["cve_mun"] == "053"
].copy()

print("MORELIA 2016/01:", len(morelia_2016_01))
print("MORELIA 2016/10:", len(morelia_2016_10))

print("\nValidación por nombre:")

print(
    "Enero discrepancias:",
    (
        (denue_2016_01["cve_mun"] == "053")
        !=
        (denue_2016_01["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    "Octubre discrepancias:",
    (
        (denue_2016_10["cve_mun"] == "053")
        !=
        (denue_2016_10["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2016/01: 45393
MORELIA 2016/10: 45511

Validación por nombre:
Enero discrepancias: 0
Octubre discrepancias: 0


In [ ]:
# ==========================================
# DENUE 2016
# Buscar identificadores
# ==========================================

print("Columnas posibles de identificación:\n")

for c in denue_2016_01.columns:
    nombre = normalizar_texto(str(c))

    if any(x in nombre for x in [
        "id", "llave", "clee", "nic", "nop"
    ]):
        print(repr(c))

Columnas posibles de identificación:

'id'
'entidad'
'municipio'
'localidad'
'municipio_norm'


In [ ]:
# ==========================================
# DENUE 2016
# Buscar columna SCIAN
# ==========================================

print("Columnas relacionadas con SCIAN/actividad:\n")

for c in denue_2016_01.columns:
    nombre = normalizar_texto(str(c))

    if "scian" in nombre or "actividad" in nombre:
        print(repr(c))

Columnas relacionadas con SCIAN/actividad:



In [ ]:
# ==========================================
# DENUE 2016
# Ver nombres exactos de las 41 columnas
# ==========================================

for i, c in enumerate(denue_2016_01.columns, start=1):
    print(i, repr(c))

1 'id'
2 'nom_estab'
3 'raz_social'
4 'codigo_act'
5 'nombre_act'
6 'per_ocu'
7 'tipo_vial'
8 'nom_vial'
9 'tipo_v_e_1'
10 'nom_v_e_1'
11 'tipo_v_e_2'
12 'nom_v_e_2'
13 'tipo_v_e_3'
14 'nom_v_e_3'
15 'numero_ext'
16 'letra_ext'
17 'edificio'
18 'edificio_e'
19 'numero_int'
20 'letra_int'
21 'tipo_asent'
22 'nomb_asent'
23 'tipoCenCom'
24 'nom_CenCom'
25 'num_local'
26 'cod_postal'
27 'cve_ent'
28 'entidad'
29 'cve_mun'
30 'municipio'
31 'cve_loc'
32 'localidad'
33 'ageb'
34 'manzana'
35 'telefono'
36 'correoelec'
37 'www'
38 'tipoUniEco'
39 'latitud'
40 'longitud'
41 'fecha_alta'
42 'municipio_norm'


In [ ]:
# Buscar posibles columnas de actividad

print("Posibles columnas de actividad:\n")

for c in denue_2016_01.columns:
    nombre = normalizar_texto(str(c))

    if any(x in nombre for x in [
        "cod", "act", "giro", "nombre_act"
    ]):
        print(repr(c))

Posibles columnas de actividad:

'codigo_act'
'nombre_act'
'cod_postal'


In [ ]:
# ==========================================
# DENUE 2016
# Comparar identificadores enero vs octubre
# ==========================================

ids_2016_01 = set(
    morelia_2016_01["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_2016_10 = set(
    morelia_2016_10["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Registros Morelia enero:", len(morelia_2016_01))
print("ID únicos enero:", len(ids_2016_01))

print("\nRegistros Morelia octubre:", len(morelia_2016_10))
print("ID únicos octubre:", len(ids_2016_10))

print("\nID comunes:", len(ids_2016_01 & ids_2016_10))
print("Solo enero:", len(ids_2016_01 - ids_2016_10))
print("Solo octubre:", len(ids_2016_10 - ids_2016_01))

Registros Morelia enero: 45393
ID únicos enero: 45393

Registros Morelia octubre: 45511
ID únicos octubre: 45511

ID comunes: 45278
Solo enero: 115
Solo octubre: 233


In [ ]:
# ==========================================
# DENUE 2016
# Comparar distribución SCIAN
# ==========================================

conteo_scian_2016_01 = (
    morelia_2016_01["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_scian_2016_10 = (
    morelia_2016_10["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

print("Clases SCIAN enero:", conteo_scian_2016_01.size)
print("Clases SCIAN octubre:", conteo_scian_2016_10.size)

print(
    "¿Distribución SCIAN exactamente igual?:",
    conteo_scian_2016_01.equals(conteo_scian_2016_10)
)

comparacion_scian_2016 = pd.concat(
    [
        conteo_scian_2016_01.rename("2016_01"),
        conteo_scian_2016_10.rename("2016_10")
    ],
    axis=1
).fillna(0).astype(int)

comparacion_scian_2016["Diferencia"] = (
    comparacion_scian_2016["2016_10"]
    - comparacion_scian_2016["2016_01"]
)

diferencias_scian_2016 = comparacion_scian_2016[
    comparacion_scian_2016["Diferencia"] != 0
]

print(
    "\nClases SCIAN con diferencias:",
    len(diferencias_scian_2016)
)

display(
    diferencias_scian_2016
    .sort_values("Diferencia", ascending=False)
)

Clases SCIAN enero: 667
Clases SCIAN octubre: 672
¿Distribución SCIAN exactamente igual?: False

Clases SCIAN con diferencias: 293


,2016_01,2016_10,Diferencia
codigo_act,,,
611312,60,106,46
931210,251,280,29
722511,467,485,18
621113,574,586,12
722519,461,473,12
...,...,...,...
621111,448,434,-14
517110,58,43,-15
722518,529,512,-17


In [ ]:
# ==========================================
# DENUE 2016
# Revisar fechas de alta
# ==========================================

print("FECHA ALTA 2016/01")
display(
    morelia_2016_01["fecha_alta"]
    .value_counts(dropna=False)
)

print("\nFECHA ALTA 2016/10")
display(
    morelia_2016_10["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2016/01


fecha_alta
2010-07       26116
2014-12       18129
2013-07         412
2011-03         365
JUNIO 2012      205
2016-01         166
Name: count, dtype: int64


FECHA ALTA 2016/10


fecha_alta
JULIO 2010        25941
DICIEMBRE 2014    17903
ENERO 2016          475
JULIO 2013          406
MARZO 2011          355
OCTUBRE 2016        233
JUNIO 2012          198
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2016
# Comparar ID enero vs octubre
# ==========================================

ids_2016_01 = set(
    morelia_2016_01["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_2016_10 = set(
    morelia_2016_10["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Registros enero:", len(morelia_2016_01))
print("ID únicos enero:", len(ids_2016_01))

print("\nRegistros octubre:", len(morelia_2016_10))
print("ID únicos octubre:", len(ids_2016_10))

print("\nID comunes:", len(ids_2016_01 & ids_2016_10))
print("Solo enero:", len(ids_2016_01 - ids_2016_10))
print("Solo octubre:", len(ids_2016_10 - ids_2016_01))

Registros enero: 45393
ID únicos enero: 45393

Registros octubre: 45511
ID únicos octubre: 45511

ID comunes: 45278
Solo enero: 115
Solo octubre: 233


In [ ]:
# ==========================================
# DENUE 2016
# Registros exclusivos enero y octubre
# ==========================================

solo_enero_2016 = morelia_2016_01[
    morelia_2016_01["id"].astype(str).isin(
        ids_2016_01 - ids_2016_10
    )
].copy()

solo_octubre_2016 = morelia_2016_10[
    morelia_2016_10["id"].astype(str).isin(
        ids_2016_10 - ids_2016_01
    )
].copy()

print("Solo enero:", len(solo_enero_2016))
print("Solo octubre:", len(solo_octubre_2016))


Solo enero: 115
Solo octubre: 233


In [ ]:
print("FECHA DE ALTA - SOLO ENERO")
display(
    solo_enero_2016["fecha_alta"]
    .value_counts(dropna=False)
)

print("\nFECHA DE ALTA - SOLO OCTUBRE")
display(
    solo_octubre_2016["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA DE ALTA - SOLO ENERO


fecha_alta
2010-07       70
2014-12       30
JUNIO 2012     7
2011-03        3
2013-07        3
2016-01        2
Name: count, dtype: int64


FECHA DE ALTA - SOLO OCTUBRE


fecha_alta
OCTUBRE 2016    233
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2016
# Comprobar cambios de ID mediante huella
# ==========================================

campos_huella_2016 = [
    "codigo_act",
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_2016(df):
    copia = df.copy()

    for campo in campos_huella_2016:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella_2016
    ]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

enero_h = crear_huella_2016(solo_enero_2016)
octubre_h = crear_huella_2016(solo_octubre_2016)

huellas_enero = set(enero_h["HUELLA"])
huellas_octubre = set(octubre_h["HUELLA"])

print("Huellas únicas enero:", len(huellas_enero))
print("Huellas únicas octubre:", len(huellas_octubre))

print("\nCoincidencias exactas entre exclusivos:")
print(len(huellas_enero & huellas_octubre))

print("\nSolo enero después de comparar huellas:")
print(len(huellas_enero - huellas_octubre))

print("\nSolo octubre después de comparar huellas:")
print(len(huellas_octubre - huellas_enero))

Huellas únicas enero: 115
Huellas únicas octubre: 233

Coincidencias exactas entre exclusivos:
0

Solo enero después de comparar huellas:
115

Solo octubre después de comparar huellas:
233


### Decisión metodológica para 2016

Se auditaron las ediciones DENUE 2016/01 y 2016/10.

El corte de enero contiene 45,393 registros de Morelia y el de octubre
45,511. Se identificaron 45,278 registros con el mismo ID en ambas
ediciones, 115 registros presentes únicamente en enero y 233 presentes
únicamente en octubre.

Los 233 registros exclusivos de octubre presentan fecha de alta
"OCTUBRE 2016". Por tanto, se interpretan como altas incorporadas al
DENUE en dicho corte, sin equipararlas automáticamente con nacimientos
empresariales.

Los 115 registros presentes en enero y ausentes en octubre corresponden
a distintas cohortes de incorporación anteriores. No se interpretan como
cierres o muertes empresariales.

Una comparación mediante huella compuesta de código SCIAN, coordenadas,
nombre del establecimiento, vialidad, número exterior y localidad no
detectó coincidencias entre los registros exclusivos de ambos cortes,
por lo que no se observaron cambios simples de identificador entre estos
grupos.

Para la serie anual se selecciona DENUE 2016/10 como corte representativo
de 2016, conservando DENUE 2016/01 como edición intraanual auditada.

In [ ]:
# ==========================================
# DENUE 2016
# Guardar corte representativo 2016/10
# ==========================================

archivo_salida_2016 = MORELIA / "DENUE_2016_MORELIA.csv"

morelia_2016_10.to_csv(
    archivo_salida_2016,
    index=False,
    encoding="utf-8-sig"
)

print("✓ DENUE 2016/10 guardado como corte 2016:")
print(archivo_salida_2016)

✓ DENUE 2016/10 guardado como corte 2016:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2016_MORELIA.csv


In [ ]:
# ==========================================
# DENUE 2016
# Conteo por clase SCIAN
# ==========================================

conteo_scian_2016 = (
    morelia_2016_10
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2016 = conteo_scian_2016.sort_values(
    "Número de establecimientos",
    ascending=False
)

conteo_scian_2016["SCIAN6"] = (
    conteo_scian_2016["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_2016["Sector SCIAN"] = (
    conteo_scian_2016["SCIAN6"].str[:2]
)

print("Total Morelia 2016:", len(morelia_2016_10))
print(
    "Clases SCIAN distintas:",
    conteo_scian_2016["SCIAN6"].nunique()
)

display(conteo_scian_2016.head(20))

Total Morelia 2016: 45511
Clases SCIAN distintas: 672


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
261,461110,3998,461110,46
648,812110,1767,812110,81
281,463211,1429,463211,46
616,722514,1298,722514,72
68,311830,1176,311830,31
533,621211,1057,621211,62
265,461130,963,461130,46
615,722513,958,722513,72
303,465311,900,465311,46
622,811111,750,811111,81


In [ ]:
# ==========================================
# DENUE 2016
# Comercio al por menor y servicios
# ==========================================

comercio_2016 = conteo_scian_2016[
    conteo_scian_2016["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2016 = conteo_scian_2016[
    conteo_scian_2016["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_2016 = servicios_2016[
    servicios_2016["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_2016["SCIAN6"].nunique())
print("Establecimientos:",
      comercio_2016["Número de establecimientos"].sum())

print("\nSERVICIOS")
print("Clases:", servicios_2016["SCIAN6"].nunique())
print("Establecimientos:",
      servicios_2016["Número de establecimientos"].sum())

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_2016))
print("Establecimientos:",
      servicios_mayores_50_2016["Número de establecimientos"].sum())

COMERCIO AL POR MENOR
Clases: 82
Establecimientos: 17313

SERVICIOS
Clases: 320
Establecimientos: 21059

SERVICIOS CON MÁS DE 50
Clases: 66
Establecimientos: 18225


In [ ]:
# ==========================================
# DENUE 2016
# Guardar conteos y diagnósticos
# ==========================================

conteo_scian_2016.to_csv(
    CONTEOS / "CONTEO_SCIAN_2016_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_2016.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2016.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_2016.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2016.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_2016.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2016.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Archivos 2016 guardados correctamente.")

✓ Archivos 2016 guardados correctamente.


In [ ]:
# ==========================================
# Agregar 2016 al resumen histórico
# ==========================================

fila_2016 = pd.DataFrame([{
    "Año": 2016,
    "Total_Morelia": 45511,
    "Clases_SCIAN_Total": 672,
    "Clases_Comercio_Minorista": 82,
    "Establecimientos_Comercio_Minorista": 17313,
    "Clases_Servicios": 320,
    "Establecimientos_Servicios": 21059,
    "Clases_Servicios_Mayores_50": 66,
    "Establecimientos_Servicios_Mayores_50": 18225,
    "Version_SCIAN": "SCIAN 2013",
    "Observacion_metodologica":
        "Se auditaron 2016/01 y 2016/10; se seleccionó 2016/10 como corte anual."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != 2016
]

resumen_historico = pd.concat(
    [resumen_historico, fila_2016],
    ignore_index=True
).sort_values("Año")

display(resumen_historico)

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Resumen actualizado hasta 2016.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...


✓ Resumen actualizado hasta 2016.


In [ ]:
cortes_denue.loc[
    cortes_denue["Corte"] == "2016_01",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == "2016_10",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora actualizada.")

✓ Bitácora actualizada.


# DENUE 2017

In [ ]:
# ==========================================
# DENUE 2017
# Localizar ambos cortes
# ==========================================

CARPETA_2017_03 = ORIGINALES / "2017_03"
CARPETA_2017_11 = ORIGINALES / "2017_11"

print("Carpeta 2017/03 existe:", CARPETA_2017_03.exists())
print("Carpeta 2017/11 existe:", CARPETA_2017_11.exists())

print("\nArchivos 2017/03:")
for archivo in CARPETA_2017_03.iterdir():
    print("-", archivo.name)

print("\nArchivos 2017/11:")
for archivo in CARPETA_2017_11.iterdir():
    print("-", archivo.name)

Carpeta 2017/03 existe: True
Carpeta 2017/11 existe: True

Archivos 2017/03:
- denue_16_0317_csv.zip

Archivos 2017/11:
- denue_16_1117_csv.zip


In [ ]:
# ==========================================
# DENUE 2017
# Revisar contenido interno de ambos ZIP
# ==========================================

zip_2017_03 = next(CARPETA_2017_03.glob("*.zip"))
zip_2017_11 = next(CARPETA_2017_11.glob("*.zip"))

print("===== 2017/03 =====")
print("ZIP:", zip_2017_03.name)

with zipfile.ZipFile(zip_2017_03, "r") as z:
    contenido_03 = z.namelist()
    for nombre in contenido_03:
        print("-", nombre)

print("\n===== 2017/11 =====")
print("ZIP:", zip_2017_11.name)

with zipfile.ZipFile(zip_2017_11, "r") as z:
    contenido_11 = z.namelist()
    for nombre in contenido_11:
        print("-", nombre)

===== 2017/03 =====
ZIP: denue_16_0317_csv.zip
- denue_16_csv/conjunto_de_datos/
- denue_16_csv/conjunto_de_datos/denue_inegi_16_.csv
- denue_16_csv/diccionario_de_datos/
- denue_16_csv/diccionario_de_datos/denue_diccionario_de_datos.csv
- denue_16_csv/metadatos/
- denue_16_csv/metadatos/metadatos_denue.txt

===== 2017/11 =====
ZIP: denue_16_1117_csv.zip
- denue_16_csv/conjunto_de_datos/
- denue_16_csv/conjunto_de_datos/denue_inegi_16_.csv
- denue_16_csv/diccionario_de_datos/
- denue_16_csv/diccionario_de_datos/denue_diccionario_de_datos.csv
- denue_16_csv/metadatos/
- denue_16_csv/metadatos/metadatos_denue.txt


In [ ]:
# ==========================================
# DENUE 2017
# Detectar CSV principal de ambos cortes
# ==========================================

with zipfile.ZipFile(zip_2017_03, "r") as z:
    csv_2017_03 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_2017_11, "r") as z:
    csv_2017_11 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print("CSV principal 2016/01:")
print(csv_2017_03)

print("\nCSV principal 2016/10:")
print(csv_2017_11)

CSV principal 2016/01:
denue_16_csv/conjunto_de_datos/denue_inegi_16_.csv

CSV principal 2016/10:
denue_16_csv/conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ==========================================
# DENUE 2017
# Inspección inicial de ambos CSV
# ==========================================

def leer_muestra_zip(zip_path, csv_interno):
    for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(csv_interno) as archivo:
                    muestra = pd.read_csv(
                        archivo,
                        nrows=5,
                        encoding=codificacion,
                        dtype=str
                    )
            return muestra, codificacion
        except UnicodeDecodeError:
            continue

    raise ValueError("No se pudo detectar la codificación.")

muestra_2017_03, enc_03 = leer_muestra_zip(
    zip_2017_03,
    csv_2017_03
)

muestra_2017_11, enc_11 = leer_muestra_zip(
    zip_2017_11,
    csv_2017_11
)

print("2017/03")
print("Codificación:", enc_03)
print("Columnas:", len(muestra_2017_03.columns))

print("\n2017/11")
print("Codificación:", enc_11)
print("Columnas:", len(muestra_2017_11.columns))

print("\n¿Mismas columnas exactamente?")
print(
    list(muestra_2017_03.columns)
    ==
    list(muestra_2017_11.columns)
)

2017/03
Codificación: utf-8-sig
Columnas: 41

2017/11
Codificación: utf-8-sig
Columnas: 41

¿Mismas columnas exactamente?
True


In [ ]:
# ==========================================
# DENUE 2017
# Cargar ambos cortes completos
# ==========================================

with zipfile.ZipFile(zip_2017_03, "r") as z:
    with z.open(csv_2017_03) as archivo:
        denue_2017_03 = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            dtype=str
        )

with zipfile.ZipFile(zip_2017_11, "r") as z:
    with z.open(csv_2017_11) as archivo:
        denue_2017_11 = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            dtype=str
        )

print("2017/03")
print("Filas:", len(denue_2017_03))
print("Columnas:", len(denue_2017_03.columns))

print("\n2017/11")
print("Filas:", len(denue_2017_11))
print("Columnas:", len(denue_2017_11.columns))

2017/03
Filas: 228564
Columnas: 41

2017/11
Filas: 228817
Columnas: 41


In [ ]:
# ==========================================
# DENUE 2017
# Comparar estructura marzo vs noviembre
# ==========================================

print(
    "¿Mismas columnas exactamente?:",
    list(denue_2017_03.columns) == list(denue_2017_11.columns)
)

print("Columnas marzo:", len(denue_2017_03.columns))
print("Columnas noviembre:", len(denue_2017_11.columns))

¿Mismas columnas exactamente?: True
Columnas marzo: 41
Columnas noviembre: 41


In [ ]:
# ==========================================
# DENUE 2017
# Filtrar Morelia en ambos cortes
# ==========================================

for df in [denue_2017_03, denue_2017_11]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_2017_03 = denue_2017_03[
    denue_2017_03["cve_mun"] == "053"
].copy()

morelia_2017_11 = denue_2017_11[
    denue_2017_11["cve_mun"] == "053"
].copy()

print("MORELIA 2017/03:", len(morelia_2017_03))
print("MORELIA 2017/11:", len(morelia_2017_11))

print("\nValidación por nombre:")

print(
    "Marzo discrepancias:",
    (
        (denue_2017_03["cve_mun"] == "053")
        !=
        (denue_2017_03["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    "Noviembre discrepancias:",
    (
        (denue_2017_11["cve_mun"] == "053")
        !=
        (denue_2017_11["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2017/03: 45612
MORELIA 2017/11: 45715

Validación por nombre:
Marzo discrepancias: 0
Noviembre discrepancias: 0


In [ ]:
# ==========================================
# DENUE 2017
# Comparar ID marzo vs noviembre
# ==========================================

ids_2017_03 = set(
    morelia_2017_03["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_2017_11 = set(
    morelia_2017_11["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Registros Morelia marzo:", len(morelia_2017_03))
print("ID únicos marzo:", len(ids_2017_03))

print("\nRegistros Morelia noviembre:", len(morelia_2017_11))
print("ID únicos noviembre:", len(ids_2017_11))

print("\nID comunes:", len(ids_2017_03 & ids_2017_11))
print("Solo marzo:", len(ids_2017_03 - ids_2017_11))
print("Solo noviembre:", len(ids_2017_11 - ids_2017_03))

Registros Morelia marzo: 45612
ID únicos marzo: 45612

Registros Morelia noviembre: 45715
ID únicos noviembre: 45715

ID comunes: 45541
Solo marzo: 71
Solo noviembre: 174


In [ ]:
# ==========================================
# DENUE 2017
# Comparar ID marzo vs noviembre
# ==========================================

ids_2017_03 = set(
    morelia_2017_03["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_2017_11 = set(
    morelia_2017_11["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Registros Morelia marzo:", len(morelia_2017_03))
print("ID únicos marzo:", len(ids_2017_03))

print("\nRegistros Morelia noviembre:", len(morelia_2017_11))
print("ID únicos noviembre:", len(ids_2017_11))

print("\nID comunes:", len(ids_2017_03 & ids_2017_11))
print("Solo marzo:", len(ids_2017_03 - ids_2017_11))
print("Solo noviembre:", len(ids_2017_11 - ids_2017_03))

Registros Morelia marzo: 45612
ID únicos marzo: 45612

Registros Morelia noviembre: 45715
ID únicos noviembre: 45715

ID comunes: 45541
Solo marzo: 71
Solo noviembre: 174


In [ ]:
# ==========================================
# DENUE 2017
# Revisar fechas de alta
# ==========================================

print("FECHA ALTA 2017/03")
display(
    morelia_2017_03["fecha_alta"]
    .value_counts(dropna=False)
)

print("\nFECHA ALTA 2017/11")
display(
    morelia_2017_11["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2017/03


fecha_alta
JULIO 2010        25914
DICIEMBRE 2014    17895
ENERO 2016          469
JULIO 2013          406
MARZO 2011          355
OCTUBRE 2016        225
JUNIO 2012          197
MARZO 2017          151
Name: count, dtype: int64


FECHA ALTA 2017/11


fecha_alta
JULIO 2010        25909
DICIEMBRE 2014    17882
ENERO 2016          470
JULIO 2013          407
MARZO 2011          351
OCTUBRE 2016        222
JUNIO 2012          196
NOVIEMBRE 2017      163
MARZO 2017          115
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2017
# Comparar distribución SCIAN
# ==========================================

conteo_scian_2017_03 = (
    morelia_2017_03["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_scian_2017_11 = (
    morelia_2017_11["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

print("Clases SCIAN marzo:", conteo_scian_2017_03.size)
print("Clases SCIAN noviembre:", conteo_scian_2017_11.size)

print(
    "¿Distribución SCIAN exactamente igual?:",
    conteo_scian_2017_03.equals(conteo_scian_2017_11)
)

Clases SCIAN marzo: 672
Clases SCIAN noviembre: 672
¿Distribución SCIAN exactamente igual?: False


In [ ]:
# ==========================================
# DENUE 2017
# Fecha de alta SOLO de registros exclusivos
# ==========================================

solo_marzo_2017 = morelia_2017_03[
    morelia_2017_03["id"].astype(str).isin(
        ids_2017_03 - ids_2017_11
    )
].copy()

solo_noviembre_2017 = morelia_2017_11[
    morelia_2017_11["id"].astype(str).isin(
        ids_2017_11 - ids_2017_03
    )
].copy()

print("SOLO MARZO:", len(solo_marzo_2017))
display(
    solo_marzo_2017["fecha_alta"]
    .value_counts(dropna=False)
)

print("\nSOLO NOVIEMBRE:", len(solo_noviembre_2017))
display(
    solo_noviembre_2017["fecha_alta"]
    .value_counts(dropna=False)
)

SOLO MARZO: 71


fecha_alta
MARZO 2017        36
DICIEMBRE 2014    13
JULIO 2010        10
OCTUBRE 2016       5
MARZO 2011         4
ENERO 2016         2
JUNIO 2012         1
Name: count, dtype: int64


SOLO NOVIEMBRE: 174


fecha_alta
NOVIEMBRE 2017    163
JULIO 2010          5
ENERO 2016          3
OCTUBRE 2016        2
JULIO 2013          1
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2017
# Comprobar posibles cambios de ID
# ==========================================

campos_huella_2017 = [
    "codigo_act",
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_2017(df):
    copia = df.copy()

    for campo in campos_huella_2017:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella_2017
    ]

    copia["HUELLA"] = copia[columnas_norm].agg("|".join, axis=1)

    return copia

marzo_h = crear_huella_2017(solo_marzo_2017)
noviembre_h = crear_huella_2017(solo_noviembre_2017)

huellas_marzo = set(marzo_h["HUELLA"])
huellas_noviembre = set(noviembre_h["HUELLA"])

print("Huellas únicas marzo:", len(huellas_marzo))
print("Huellas únicas noviembre:", len(huellas_noviembre))
print("Coincidencias exactas:", len(huellas_marzo & huellas_noviembre))
print("Solo marzo tras huella:", len(huellas_marzo - huellas_noviembre))
print("Solo noviembre tras huella:", len(huellas_noviembre - huellas_marzo))

Huellas únicas marzo: 71
Huellas únicas noviembre: 174
Coincidencias exactas: 0
Solo marzo tras huella: 71
Solo noviembre tras huella: 174


In [ ]:
# ==========================================
# DENUE 2017
# Auditoría fina de SCIAN
# ==========================================

scian_marzo = set(conteo_scian_2017_03.index)
scian_noviembre = set(conteo_scian_2017_11.index)

print("Clases comunes:", len(scian_marzo & scian_noviembre))
print("Solo marzo:", len(scian_marzo - scian_noviembre))
print("Solo noviembre:", len(scian_noviembre - scian_marzo))

comparacion_scian_2017 = pd.concat(
    [
        conteo_scian_2017_03.rename("2017_03"),
        conteo_scian_2017_11.rename("2017_11")
    ],
    axis=1
).fillna(0).astype(int)

comparacion_scian_2017["Diferencia"] = (
    comparacion_scian_2017["2017_11"]
    - comparacion_scian_2017["2017_03"]
)

diferencias_scian_2017 = comparacion_scian_2017[
    comparacion_scian_2017["Diferencia"] != 0
]

print("\nClases cuyo número de registros cambió:",
      len(diferencias_scian_2017))

display(
    diferencias_scian_2017
    .sort_values("Diferencia", ascending=False)
)

Clases comunes: 671
Solo marzo: 1
Solo noviembre: 1

Clases cuyo número de registros cambió: 119


,2017_03,2017_11,Diferencia
codigo_act,,,
464111,473,499,26
461123,72,82,10
465215,125,129,4
462112,223,226,3
236111,59,62,3
...,...,...,...
624121,4,3,-1
813230,99,98,-1
463216,76,74,-2


In [ ]:
# ==========================================
# DENUE 2017
# Identificar clases exclusivas de cada corte
# ==========================================

print("Clase presente solo en marzo:")
print(scian_marzo - scian_noviembre)

print("\nClase presente solo en noviembre:")
print(scian_noviembre - scian_marzo)

Clase presente solo en marzo:
{'322132'}

Clase presente solo en noviembre:
{'237313'}


### Decisión metodológica para 2017

Se auditaron las ediciones DENUE 2017/03 y DENUE 2017/11.

El corte de marzo contiene 45,612 registros correspondientes a Morelia,
mientras que el corte de noviembre contiene 45,715. Se identificaron
45,541 registros con el mismo ID en ambos cortes, 71 registros presentes
únicamente en marzo y 174 presentes únicamente en noviembre.

De los registros exclusivos de noviembre, 163 presentan fecha de alta
"NOVIEMBRE 2017"; los 11 restantes corresponden a cohortes de incorporación
anteriores. Los registros exclusivos de marzo también pertenecen a distintas
cohortes de incorporación.

La comparación mediante una huella compuesta de código SCIAN, coordenadas,
nombre del establecimiento, vialidad, número exterior y localidad no
identificó coincidencias exactas entre los registros exclusivos de ambos
cortes, por lo que no se detectaron simples cambios de identificador.

Ambos cortes contienen 672 clases SCIAN; 671 son comunes, una aparece
únicamente en marzo y otra únicamente en noviembre. Además, 119 clases
presentaron cambios en el número de establecimientos registrados.

Para la serie anual se selecciona DENUE 2017/11 como corte representativo
de 2017, conservando DENUE 2017/03 como corte intraanual auditado.

In [ ]:
# ==========================================
# DENUE 2017
# Guardar corte representativo 2017/11
# ==========================================

archivo_salida_2017 = MORELIA / "DENUE_2017_MORELIA.csv"

morelia_2017_11.to_csv(
    archivo_salida_2017,
    index=False,
    encoding="utf-8-sig"
)

print("✓ DENUE 2017/11 guardado como corte 2017.")

✓ DENUE 2017/11 guardado como corte 2017.


In [ ]:
# ==========================================
# DENUE 2017
# Conteo por clase SCIAN
# ==========================================

conteo_scian_2017 = (
    morelia_2017_11
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2017["SCIAN6"] = (
    conteo_scian_2017["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_2017["Sector SCIAN"] = (
    conteo_scian_2017["SCIAN6"].str[:2]
)

conteo_scian_2017 = conteo_scian_2017.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia 2017:", len(morelia_2017_11))
print("Clases SCIAN:", conteo_scian_2017["SCIAN6"].nunique())

display(conteo_scian_2017.head(20))

Total Morelia 2017: 45715
Clases SCIAN: 672


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
262,461110,3998,461110,46
648,812110,1767,812110,81
282,463211,1436,463211,46
616,722514,1297,722514,72
69,311830,1175,311830,31
534,621211,1056,621211,62
266,461130,964,461130,46
615,722513,958,722513,72
304,465311,900,465311,46
622,811111,751,811111,81


In [ ]:
# ==========================================
# DENUE 2017
# Comercio al por menor y servicios
# ==========================================

# COMERCIO AL POR MENOR
comercio_2017 = conteo_scian_2017[
    conteo_scian_2017["Sector SCIAN"] == "46"
].copy()

# SERVICIOS
sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2017 = conteo_scian_2017[
    conteo_scian_2017["Sector SCIAN"].isin(sectores_servicios)
].copy()

# Servicios con más de 50 establecimientos
servicios_mayores_50_2017 = servicios_2017[
    servicios_2017["Número de establecimientos"] > 50
].copy()

servicios_mayores_50_2017 = servicios_mayores_50_2017.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_2017["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_2017["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_2017["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_2017["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_2017))
print(
    "Establecimientos:",
    servicios_mayores_50_2017[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 82
Establecimientos: 17427

SERVICIOS
Clases: 319
Establecimientos: 21109

SERVICIOS CON MÁS DE 50
Clases: 65
Establecimientos: 18192


In [ ]:
# ==========================================
# DENUE 2017
# Guardar conteos y diagnósticos
# ==========================================

conteo_scian_2017.to_csv(
    CONTEOS / "CONTEO_SCIAN_2017_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_2017.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2017.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_2017.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2017.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_2017.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2017.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Archivos 2017 guardados correctamente.")

✓ Archivos 2017 guardados correctamente.


In [ ]:
# ==========================================
# Agregar 2017 al resumen histórico
# ==========================================

fila_2017 = pd.DataFrame([{
    "Año": 2017,
    "Total_Morelia": 45715,
    "Clases_SCIAN_Total": 672,
    "Clases_Comercio_Minorista": 82,
    "Establecimientos_Comercio_Minorista": 17427,
    "Clases_Servicios": 319,
    "Establecimientos_Servicios": 21109,
    "Clases_Servicios_Mayores_50": 65,
    "Establecimientos_Servicios_Mayores_50": 18192,
    "Version_SCIAN": "SCIAN 2013",
    "Observacion_metodologica":
        "Se auditaron 2017/03 y 2017/11; se seleccionó 2017/11 como corte anual."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != 2017
]

resumen_historico = pd.concat(
    [resumen_historico, fila_2017],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print("✓ Resumen actualizado hasta 2017.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...


✓ Resumen actualizado hasta 2017.


In [ ]:
cortes_denue.loc[
    cortes_denue["Corte"] == "2017_03",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == "2017_11",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora actualizada.")

✓ Bitácora actualizada.


# DENUE 2018

In [ ]:
# ==========================================
# DENUE 2018
# Localizar ambos cortes
# ==========================================

CARPETA_2018_03 = ORIGINALES / "2018_03"
CARPETA_2018_11 = ORIGINALES / "2018_11"

print("Carpeta 2018/03 existe:", CARPETA_2018_03.exists())
print("Carpeta 2018/11 existe:", CARPETA_2018_11.exists())

print("\nArchivos 2018/03:")
for archivo in CARPETA_2018_03.iterdir():
    print("-", archivo.name)

print("\nArchivos 2018/11:")
for archivo in CARPETA_2018_11.iterdir():
    print("-", archivo.name)

Carpeta 2018/03 existe: True
Carpeta 2018/11 existe: True

Archivos 2018/03:
- denue_16_0318_csv.zip

Archivos 2018/11:
- denue_16_1118_csv.zip


In [ ]:
# ==========================================
# DENUE 2018
# Revisar contenido interno de ambos ZIP
# ==========================================

zip_2018_03 = next(CARPETA_2018_03.glob("*.zip"))
zip_2018_11 = next(CARPETA_2018_11.glob("*.zip"))

print("===== 2018/03 =====")
print("ZIP:", zip_2018_03.name)

with zipfile.ZipFile(zip_2018_03, "r") as z:
    contenido_03 = z.namelist()
    for nombre in contenido_03:
        print("-", nombre)

print("\n===== 2018/11 =====")
print("ZIP:", zip_2018_11.name)

with zipfile.ZipFile(zip_2018_11, "r") as z:
    contenido_11 = z.namelist()
    for nombre in contenido_11:
        print("-", nombre)

===== 2018/03 =====
ZIP: denue_16_0318_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt

===== 2018/11 =====
ZIP: denue_16_1118_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ==========================================
# DENUE 2018
# Detectar CSV principal de ambos cortes
# ==========================================

with zipfile.ZipFile(zip_2018_03, "r") as z:
    csv_2018_03 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_2018_11, "r") as z:
    csv_2018_11 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print("CSV principal 2018/03:")
print(csv_2018_03)

print("\nCSV principal 2018/11:")
print(csv_2018_11)

CSV principal 2018/03:
conjunto_de_datos/denue_inegi_16_.csv

CSV principal 2018/11:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ==========================================
# DENUE 2018
# Revisar columnas de ambos cortes
# ==========================================

with zipfile.ZipFile(zip_2018_03, "r") as z:
    with z.open(csv_2018_03) as archivo:
        muestra_2018_03 = pd.read_csv(
            archivo,
            nrows=5,
            encoding="utf-8-sig",
            dtype=str
        )

with zipfile.ZipFile(zip_2018_11, "r") as z:
    with z.open(csv_2018_11) as archivo:
        muestra_2018_11 = pd.read_csv(
            archivo,
            nrows=5,
            encoding="utf-8-sig",
            dtype=str
        )

print("2018/03 columnas:", len(muestra_2018_03.columns))
print("2018/11 columnas:", len(muestra_2018_11.columns))

print(
    "¿Mismas columnas exactamente?:",
    list(muestra_2018_03.columns) ==
    list(muestra_2018_11.columns)
)

2018/03 columnas: 41
2018/11 columnas: 41
¿Mismas columnas exactamente?: True


In [ ]:
# ==========================================
# DENUE 2018
# Localizar ZIP y revisar estructura
# ==========================================

zip_2018_03 = next(CARPETA_2018_03.glob("*.zip"))
zip_2018_11 = next(CARPETA_2018_11.glob("*.zip"))

csv_2018 = "conjunto_de_datos/denue_inegi_16_.csv"
meta_2018 = "metadatos/metadatos_denue.txt"

# Leer muestras de ambos CSV
with zipfile.ZipFile(zip_2018_03, "r") as z:
    with z.open(csv_2018) as archivo:
        muestra_2018_03 = pd.read_csv(
            archivo,
            nrows=5,
            encoding="utf-8-sig",
            dtype=str
        )

with zipfile.ZipFile(zip_2018_11, "r") as z:
    with z.open(csv_2018) as archivo:
        muestra_2018_11 = pd.read_csv(
            archivo,
            nrows=5,
            encoding="utf-8-sig",
            dtype=str
        )

print("2018/03 columnas:", len(muestra_2018_03.columns))
print("2018/11 columnas:", len(muestra_2018_11.columns))

print(
    "¿Mismas columnas exactamente?:",
    list(muestra_2018_03.columns)
    ==
    list(muestra_2018_11.columns)
)

2018/03 columnas: 41
2018/11 columnas: 41
¿Mismas columnas exactamente?: True


In [ ]:
# ==========================================
# DENUE 2018
# Revisar metadatos de ambos cortes
# ==========================================

for etiqueta, zip_path in [
    ("2018/03", zip_2018_03),
    ("2018/11", zip_2018_11)
]:
    print("\n===================================")
    print(etiqueta)
    print("===================================")

    with zipfile.ZipFile(zip_path, "r") as z:
        contenido = z.read(meta_2018)

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                texto = contenido.decode(encoding)
                break
            except UnicodeDecodeError:
                continue

    # Mostrar únicamente líneas relacionadas con
    # SCIAN, versión, fecha o actualización
    for linea in texto.splitlines():
        linea_norm = normalizar_texto(linea)

        if any(
            palabra in linea_norm
            for palabra in [
                "scian",
                "version",
                "actualizacion",
                "fecha"
            ]
        ):
            print(linea)


2018/03
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados del operativo de actualización realizado por el INEGI en el segundo semestre del 2016. Resultado de ello, para esta versión se dieron de baja 4 mil 969 establecimientos y se incorporaron 32 486, dando un total de 5 millones 32 mil 503 establecimientos en el Directorio.

2018/11
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados del operativo de actualización realiza

In [ ]:
# ==========================================
# DENUE 2018
# Cargar ambos cortes completos
# ==========================================

with zipfile.ZipFile(zip_2018_03, "r") as z:
    with z.open(csv_2018_03) as archivo:
        denue_2018_03 = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            dtype=str
        )

with zipfile.ZipFile(zip_2018_11, "r") as z:
    with z.open(csv_2018_11) as archivo:
        denue_2018_11 = pd.read_csv(
            archivo,
            encoding="utf-8-sig",
            dtype=str
        )

print("2018/03")
print("Filas:", len(denue_2018_03))
print("Columnas:", len(denue_2018_03.columns))

print("\n2018/11")
print("Filas:", len(denue_2018_11))
print("Columnas:", len(denue_2018_11.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(denue_2018_03.columns) ==
    list(denue_2018_11.columns)
)

2018/03
Filas: 229212
Columnas: 41

2018/11
Filas: 229253
Columnas: 41

¿Mismas columnas exactamente?: True


In [ ]:
# ==========================================
# DENUE 2018
# Filtrar Morelia
# ==========================================

for df in [denue_2018_03, denue_2018_11]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_2018_03 = denue_2018_03[
    denue_2018_03["cve_mun"] == "053"
].copy()

morelia_2018_11 = denue_2018_11[
    denue_2018_11["cve_mun"] == "053"
].copy()

print("MORELIA 2018/03:", len(morelia_2018_03))
print("MORELIA 2018/11:", len(morelia_2018_11))

print("\nDiscrepancias marzo:",
      (
          (denue_2018_03["cve_mun"] == "053")
          !=
          (denue_2018_03["municipio_norm"] == "MORELIA")
      ).sum()
)

print("Discrepancias noviembre:",
      (
          (denue_2018_11["cve_mun"] == "053")
          !=
          (denue_2018_11["municipio_norm"] == "MORELIA")
      ).sum()
)

MORELIA 2018/03: 45892
MORELIA 2018/11: 45900

Discrepancias marzo: 0
Discrepancias noviembre: 0


In [ ]:
# ==========================================
# DENUE 2018
# Comparar ID marzo vs noviembre
# ==========================================

ids_2018_03 = set(
    morelia_2018_03["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_2018_11 = set(
    morelia_2018_11["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Registros marzo:", len(morelia_2018_03))
print("ID únicos marzo:", len(ids_2018_03))

print("\nRegistros noviembre:", len(morelia_2018_11))
print("ID únicos noviembre:", len(ids_2018_11))

print("\nID comunes:", len(ids_2018_03 & ids_2018_11))
print("Solo marzo:", len(ids_2018_03 - ids_2018_11))
print("Solo noviembre:", len(ids_2018_11 - ids_2018_03))

Registros marzo: 45892
ID únicos marzo: 45892

Registros noviembre: 45900
ID únicos noviembre: 45900

ID comunes: 45855
Solo marzo: 37
Solo noviembre: 45


In [ ]:
# ==========================================
# DENUE 2018
# Fechas de alta
# ==========================================

print("FECHA ALTA 2018/03")
display(
    morelia_2018_03["fecha_alta"]
    .value_counts(dropna=False)
)

print("\nFECHA ALTA 2018/11")
display(
    morelia_2018_11["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2018/03


fecha_alta
JULIO 2010        25867
DICIEMBRE 2014    17849
ENERO 2016          470
JULIO 2013          406
MARZO 2011          339
MARZO 2018          266
OCTUBRE 2016        221
JUNIO 2012          196
NOVIEMBRE 2017      163
MARZO 2017          115
Name: count, dtype: int64


FECHA ALTA 2018/11


fecha_alta
2010-07    25846
2014-12    17846
2016-01      471
2013-07      401
2011-03      338
2018-03      266
2016-10      220
2012-06      196
2017-11      163
2017-03      114
2018-11       39
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2018
# Filtrar Morelia en ambos cortes
# ==========================================

for df in [denue_2018_03, denue_2018_11]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_2018_03 = denue_2018_03[
    denue_2018_03["cve_mun"] == "053"
].copy()

morelia_2018_11 = denue_2018_11[
    denue_2018_11["cve_mun"] == "053"
].copy()

print("MORELIA 2018/03:", len(morelia_2018_03))
print("MORELIA 2018/11:", len(morelia_2018_11))

print("\nDiscrepancias marzo:",
      (
          (denue_2018_03["cve_mun"] == "053")
          !=
          (denue_2018_03["municipio_norm"] == "MORELIA")
      ).sum()
)

print("Discrepancias noviembre:",
      (
          (denue_2018_11["cve_mun"] == "053")
          !=
          (denue_2018_11["municipio_norm"] == "MORELIA")
      ).sum()
)

MORELIA 2018/03: 45892
MORELIA 2018/11: 45900

Discrepancias marzo: 0
Discrepancias noviembre: 0


In [ ]:
# ==========================================
# DENUE 2018
# Revisar fechas de alta
# ==========================================

print("FECHA ALTA 2018/03")
display(
    morelia_2018_03["fecha_alta"]
    .value_counts(dropna=False)
)

print("\nFECHA ALTA 2018/11")
display(
    morelia_2018_11["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2018/03


fecha_alta
JULIO 2010        25867
DICIEMBRE 2014    17849
ENERO 2016          470
JULIO 2013          406
MARZO 2011          339
MARZO 2018          266
OCTUBRE 2016        221
JUNIO 2012          196
NOVIEMBRE 2017      163
MARZO 2017          115
Name: count, dtype: int64


FECHA ALTA 2018/11


fecha_alta
2010-07    25846
2014-12    17846
2016-01      471
2013-07      401
2011-03      338
2018-03      266
2016-10      220
2012-06      196
2017-11      163
2017-03      114
2018-11       39
Name: count, dtype: int64

### Cambio de versión SCIAN en 2018

DENUE 03/2018 clasifica las actividades económicas con SCIAN México 2013,
mientras que DENUE 11/2018 utiliza SCIAN México 2018.

Por tanto, las diferencias en códigos SCIAN de seis dígitos entre ambos
cortes no se interpretan automáticamente como cambios en la actividad del
establecimiento. La comparación de clases requiere utilizar las tablas
oficiales de equivalencia SCIAN 2013–2018.

In [ ]:
# ==========================================
# DENUE 2018
# Registros exclusivos marzo y noviembre
# ==========================================

solo_marzo_2018 = morelia_2018_03[
    morelia_2018_03["id"].astype(str).isin(
        ids_2018_03 - ids_2018_11
    )
].copy()

solo_noviembre_2018 = morelia_2018_11[
    morelia_2018_11["id"].astype(str).isin(
        ids_2018_11 - ids_2018_03
    )
].copy()

print("Solo marzo:", len(solo_marzo_2018))
print("Solo noviembre:", len(solo_noviembre_2018))

print("\nFECHA ALTA - SOLO MARZO")
display(
    solo_marzo_2018["fecha_alta"]
    .value_counts(dropna=False)
)

print("\nFECHA ALTA - SOLO NOVIEMBRE")
display(
    solo_noviembre_2018["fecha_alta"]
    .value_counts(dropna=False)
)

Solo marzo: 37
Solo noviembre: 45

FECHA ALTA - SOLO MARZO


fecha_alta
JULIO 2010        22
DICIEMBRE 2014     6
JULIO 2013         5
MARZO 2017         1
MARZO 2011         1
NOVIEMBRE 2017     1
OCTUBRE 2016       1
Name: count, dtype: int64


FECHA ALTA - SOLO NOVIEMBRE


fecha_alta
2018-11    39
2014-12     3
2016-01     1
2010-07     1
2017-11     1
Name: count, dtype: int64

In [ ]:
# ==========================================
# DENUE 2018
# Comprobar posibles cambios de ID
# SIN utilizar SCIAN
# ==========================================

campos_huella_2018 = [
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_2018(df):
    copia = df.copy()

    for campo in campos_huella_2018:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella_2018
    ]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

marzo_h_2018 = crear_huella_2018(solo_marzo_2018)
noviembre_h_2018 = crear_huella_2018(solo_noviembre_2018)

huellas_marzo_2018 = set(marzo_h_2018["HUELLA"])
huellas_noviembre_2018 = set(noviembre_h_2018["HUELLA"])

print("Huellas únicas marzo:", len(huellas_marzo_2018))
print("Huellas únicas noviembre:", len(huellas_noviembre_2018))

print("\nCoincidencias exactas:")
print(len(huellas_marzo_2018 & huellas_noviembre_2018))

print("\nSolo marzo tras huella:")
print(len(huellas_marzo_2018 - huellas_noviembre_2018))

print("\nSolo noviembre tras huella:")
print(len(huellas_noviembre_2018 - huellas_marzo_2018))

Huellas únicas marzo: 37
Huellas únicas noviembre: 45

Coincidencias exactas:
0

Solo marzo tras huella:
37

Solo noviembre tras huella:
45


In [ ]:
# ==========================================
# DENUE 2018
# Cambio de código SCIAN en ID persistentes
# SCIAN 2013 -> SCIAN 2018
# ==========================================

ids_comunes_2018 = ids_2018_03 & ids_2018_11

comunes_marzo_2018 = (
    morelia_2018_03[
        morelia_2018_03["id"].astype(str).isin(ids_comunes_2018)
    ][["id", "codigo_act"]]
    .copy()
)

comunes_noviembre_2018 = (
    morelia_2018_11[
        morelia_2018_11["id"].astype(str).isin(ids_comunes_2018)
    ][["id", "codigo_act"]]
    .copy()
)

comparacion_codigos_2018 = pd.merge(
    comunes_marzo_2018,
    comunes_noviembre_2018,
    on="id",
    suffixes=("_SCIAN2013", "_SCIAN2018")
)

comparacion_codigos_2018["Cambio_codigo"] = (
    comparacion_codigos_2018["codigo_act_SCIAN2013"].astype(str)
    !=
    comparacion_codigos_2018["codigo_act_SCIAN2018"].astype(str)
)

print("ID comunes:", len(comparacion_codigos_2018))

print(
    "Con mismo código SCIAN:",
    (~comparacion_codigos_2018["Cambio_codigo"]).sum()
)

print(
    "Con código SCIAN diferente:",
    comparacion_codigos_2018["Cambio_codigo"].sum()
)

ID comunes: 45855
Con mismo código SCIAN: 45577
Con código SCIAN diferente: 278


In [ ]:
transiciones_scian_2018 = (
    comparacion_codigos_2018[
        comparacion_codigos_2018["Cambio_codigo"]
    ]
    .groupby(
        ["codigo_act_SCIAN2013", "codigo_act_SCIAN2018"]
    )
    .size()
    .reset_index(name="Establecimientos")
    .sort_values("Establecimientos", ascending=False)
)

display(transiciones_scian_2018.head(30))

,codigo_act_SCIAN2013,codigo_act_SCIAN2018,Establecimientos
26,532291,532282,71
20,517110,517311,43
3,222111,221311,42
24,532230,532289,35
21,517210,517312,15
23,532220,532281,15
27,532299,532289,14
0,221110,221111,12
4,222112,221312,6
29,722515,722511,3


### Decisión metodológica para 2018

Se auditaron las ediciones DENUE 2018/03 y DENUE 2018/11.

El corte de marzo contiene 45,892 registros correspondientes a Morelia y
el de noviembre 45,900. Se identificaron 45,855 registros con el mismo ID
en ambos cortes, 37 registros presentes únicamente en marzo y 45 presentes
únicamente en noviembre.

De los 45 registros exclusivos de noviembre, 39 presentan fecha de alta
2018-11; los restantes corresponden a cohortes de incorporación anteriores.
Los registros exclusivos de marzo pertenecen también a distintas cohortes.

Una comparación mediante huella compuesta de ubicación, nombre del
establecimiento y domicilio, excluyendo el código SCIAN, no identificó
coincidencias exactas entre los registros exclusivos de ambos cortes.

Entre los 45,855 registros con ID persistente, 45,577 conservaron el mismo
código de actividad y 278 presentaron un código diferente. Estos cambios no
se interpretan automáticamente como modificaciones reales de actividad,
debido al cambio de versión SCIAN y a posibles procesos de actualización o
reclasificación del directorio.

Para la serie anual se selecciona DENUE 2018/11 como corte representativo
de 2018, conservando DENUE 2018/03 como corte intraanual auditado.

In [ ]:
# ==========================================
# DENUE 2018
# Guardar corte representativo 2018/11
# ==========================================

archivo_salida_2018 = MORELIA / "DENUE_2018_MORELIA.csv"

morelia_2018_11.to_csv(
    archivo_salida_2018,
    index=False,
    encoding="utf-8-sig"
)

print("✓ DENUE 2018/11 guardado como corte 2018.")

✓ DENUE 2018/11 guardado como corte 2018.


In [ ]:
transiciones_scian_2018.to_csv(
    HOMOLOGACION / "TRANSICIONES_SCIAN_2013_2018_OBSERVADAS.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Transiciones observadas 2013→2018 guardadas.")

✓ Transiciones observadas 2013→2018 guardadas.


In [ ]:
# ==========================================
# DENUE 2018
# Conteo por clase SCIAN
# ==========================================

conteo_scian_2018 = (
    morelia_2018_11
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_2018["SCIAN6"] = (
    conteo_scian_2018["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_2018["Sector SCIAN"] = (
    conteo_scian_2018["SCIAN6"].str[:2]
)

conteo_scian_2018 = conteo_scian_2018.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia 2018:", len(morelia_2018_11))
print(
    "Clases SCIAN:",
    conteo_scian_2018["SCIAN6"].nunique()
)

display(conteo_scian_2018.head(20))

Total Morelia 2018: 45900
Clases SCIAN: 671


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
263,461110,3999,461110,46
647,812110,1767,812110,81
283,463211,1440,463211,46
615,722514,1297,722514,72
70,311830,1175,311830,31
533,621211,1056,621211,62
267,461130,964,461130,46
614,722513,958,722513,72
305,465311,904,465311,46
621,811111,751,811111,81


In [ ]:
# ==========================================
# DENUE 2018
# Comercio al por menor y servicios
# ==========================================

comercio_2018 = conteo_scian_2018[
    conteo_scian_2018["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_2018 = conteo_scian_2018[
    conteo_scian_2018["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_2018 = servicios_2018[
    servicios_2018["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_2018["SCIAN6"].nunique())
print("Establecimientos:",
      comercio_2018["Número de establecimientos"].sum())

print("\nSERVICIOS")
print("Clases:", servicios_2018["SCIAN6"].nunique())
print("Establecimientos:",
      servicios_2018["Número de establecimientos"].sum())

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_2018))
print("Establecimientos:",
      servicios_mayores_50_2018["Número de establecimientos"].sum())

COMERCIO AL POR MENOR
Clases: 82
Establecimientos: 17452

SERVICIOS
Clases: 317
Establecimientos: 21276

SERVICIOS CON MÁS DE 50
Clases: 65
Establecimientos: 18360


In [ ]:
print(
    "Clases SCIAN totales 2018/11:",
    conteo_scian_2018["SCIAN6"].nunique()
)

Clases SCIAN totales 2018/11: 671


In [ ]:
# ==========================================
# DENUE 2018
# Guardar conteos y diagnósticos
# ==========================================

conteo_scian_2018.to_csv(
    CONTEOS / "CONTEO_SCIAN_2018_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_2018.to_csv(
    CONTEOS / "COMERCIO_MINORISTA_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_2018.to_csv(
    CONTEOS / "SERVICIOS_TODAS_CLASES_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_2018.to_csv(
    CONTEOS / "SERVICIOS_MAYORES_50_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Archivos 2018 guardados correctamente.")

✓ Archivos 2018 guardados correctamente.


In [ ]:
# ==========================================
# Agregar 2018 al resumen histórico
# ==========================================

fila_2018 = pd.DataFrame([{
    "Año": 2018,
    "Total_Morelia": 45900,
    "Clases_SCIAN_Total": 671,
    "Clases_Comercio_Minorista": 82,
    "Establecimientos_Comercio_Minorista": 17452,
    "Clases_Servicios": 317,
    "Establecimientos_Servicios": 21276,
    "Clases_Servicios_Mayores_50": 65,
    "Establecimientos_Servicios_Mayores_50": 18360,
    "Version_SCIAN": "SCIAN 2018",
    "Observacion_metodologica":
        "Se auditaron 2018/03 y 2018/11; se seleccionó 2018/11. Cambio de SCIAN 2013 a SCIAN 2018."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != 2018
]

resumen_historico = pd.concat(
    [resumen_historico, fila_2018],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print("✓ Resumen actualizado hasta 2018.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...


✓ Resumen actualizado hasta 2018.


In [ ]:
cortes_denue.loc[
    cortes_denue["Corte"] == "2018_03",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == "2018_11",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora actualizada.")

✓ Bitácora actualizada.


# DENUE 2019

In [ ]:
# ============================================================
# DENUE 2019
# CONFIGURACIÓN DEL AÑO
# ============================================================

ANIO = "2019"
CORTE_1 = "04"
CORTE_2 = "11"

ETIQUETA_1 = f"{ANIO}/{CORTE_1}"
ETIQUETA_2 = f"{ANIO}/{CORTE_2}"

CARPETA_CORTE_1 = ORIGINALES / f"{ANIO}_{CORTE_1}"
CARPETA_CORTE_2 = ORIGINALES / f"{ANIO}_{CORTE_2}"

print("Año de trabajo:", ANIO)
print("Corte 1:", ETIQUETA_1)
print("Corte 2:", ETIQUETA_2)

Año de trabajo: 2019
Corte 1: 2019/04
Corte 2: 2019/11


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ARCHIVOS DE AMBOS CORTES
# ============================================================

print(f"Carpeta {ETIQUETA_1} existe:", CARPETA_CORTE_1.exists())
print(f"Carpeta {ETIQUETA_2} existe:", CARPETA_CORTE_2.exists())

print(f"\nArchivos {ETIQUETA_1}:")
for archivo in CARPETA_CORTE_1.iterdir():
    print("-", archivo.name)

print(f"\nArchivos {ETIQUETA_2}:")
for archivo in CARPETA_CORTE_2.iterdir():
    print("-", archivo.name)

Carpeta 2019/04 existe: True
Carpeta 2019/11 existe: True

Archivos 2019/04:
- denue_16_0419_csv.zip

Archivos 2019/11:
- denue_16_1119_csv.zip


In [ ]:
# ============================================================
# PASO 2. REVISAR CONTENIDO INTERNO DE LOS ZIP
# ============================================================

zip_corte_1 = next(CARPETA_CORTE_1.glob("*.zip"))
zip_corte_2 = next(CARPETA_CORTE_2.glob("*.zip"))

print(f"===== {ETIQUETA_1} =====")
print("ZIP:", zip_corte_1.name)

with zipfile.ZipFile(zip_corte_1, "r") as z:
    contenido_corte_1 = z.namelist()
    for nombre in contenido_corte_1:
        print("-", nombre)

print(f"\n===== {ETIQUETA_2} =====")
print("ZIP:", zip_corte_2.name)

with zipfile.ZipFile(zip_corte_2, "r") as z:
    contenido_corte_2 = z.namelist()
    for nombre in contenido_corte_2:
        print("-", nombre)

===== 2019/04 =====
ZIP: denue_16_0419_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt

===== 2019/11 =====
ZIP: denue_16_1119_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 3. DETECTAR CSV PRINCIPAL DE AMBOS CORTES
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    csv_corte_1 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    csv_corte_2 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print(f"CSV principal {ETIQUETA_1}:")
print(csv_corte_1)

print(f"\nCSV principal {ETIQUETA_2}:")
print(csv_corte_2)

CSV principal 2019/04:
conjunto_de_datos/denue_inegi_16_.csv

CSV principal 2019/11:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 4. INSPECCIÓN ESTRUCTURAL DE AMBOS CORTES
# ============================================================

def leer_muestra_zip(zip_path, csv_interno):
    for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(csv_interno) as archivo:
                    muestra = pd.read_csv(
                        archivo,
                        nrows=5,
                        encoding=codificacion,
                        dtype=str
                    )
            return muestra, codificacion
        except UnicodeDecodeError:
            continue

    raise ValueError("No se pudo detectar la codificación.")

muestra_corte_1, encoding_corte_1 = leer_muestra_zip(
    zip_corte_1,
    csv_corte_1
)

muestra_corte_2, encoding_corte_2 = leer_muestra_zip(
    zip_corte_2,
    csv_corte_2
)

print(ETIQUETA_1)
print("Codificación:", encoding_corte_1)
print("Columnas:", len(muestra_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Codificación:", encoding_corte_2)
print("Columnas:", len(muestra_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(muestra_corte_1.columns)
    ==
    list(muestra_corte_2.columns)
)

2019/04
Codificación: latin1
Columnas: 41

2019/11
Codificación: latin1
Columnas: 41

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 5. CARGAR AMBOS CORTES COMPLETOS
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    with z.open(csv_corte_1) as archivo:
        denue_corte_1 = pd.read_csv(
            archivo,
            encoding=encoding_corte_1,
            dtype=str
        )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    with z.open(csv_corte_2) as archivo:
        denue_corte_2 = pd.read_csv(
            archivo,
            encoding=encoding_corte_2,
            dtype=str
        )

print(ETIQUETA_1)
print("Filas:", len(denue_corte_1))
print("Columnas:", len(denue_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Filas:", len(denue_corte_2))
print("Columnas:", len(denue_corte_2.columns))

2019/04
Filas: 229968
Columnas: 41

2019/11
Filas: 258067
Columnas: 41


In [ ]:
# ============================================================
# PASO 6. REVISAR METADATOS DE AMBOS CORTES
# ============================================================

def leer_metadatos(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:

        archivo_meta = next(
            nombre for nombre in z.namelist()
            if nombre.lower().endswith(".txt")
            and "metadato" in nombre.lower()
        )

        contenido = z.read(archivo_meta)

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                return contenido.decode(encoding)
            except UnicodeDecodeError:
                continue

    return ""

meta_corte_1 = leer_metadatos(zip_corte_1)
meta_corte_2 = leer_metadatos(zip_corte_2)

print(f"===== {ETIQUETA_1} =====")
for linea in meta_corte_1.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

print(f"\n===== {ETIQUETA_2} =====")
for linea in meta_corte_2.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

===== 2019/04 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados del operativo de actualización realizado por el INEGI en el segundo semestre del 2018. 

===== 2019/11 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados del operativo de actualización realizado por el INEGI en el segundo semestre del 2018. 


In [ ]:
# ============================================================
# PASO 7. FILTRAR MORELIA EN AMBOS CORTES
# ============================================================

for df in [denue_corte_1, denue_corte_2]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_corte_1 = denue_corte_1[
    denue_corte_1["cve_mun"] == "053"
].copy()

morelia_corte_2 = denue_corte_2[
    denue_corte_2["cve_mun"] == "053"
].copy()

print(f"MORELIA {ETIQUETA_1}:", len(morelia_corte_1))
print(f"MORELIA {ETIQUETA_2}:", len(morelia_corte_2))

print(
    f"\nDiscrepancias {ETIQUETA_1}:",
    (
        (denue_corte_1["cve_mun"] == "053")
        !=
        (denue_corte_1["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    f"Discrepancias {ETIQUETA_2}:",
    (
        (denue_corte_2["cve_mun"] == "053")
        !=
        (denue_corte_2["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2019/04: 46097
MORELIA 2019/11: 47934

Discrepancias 2019/04: 0
Discrepancias 2019/11: 0


In [ ]:
# ============================================================
# PASO 8. COMPARAR IDENTIFICADORES ENTRE CORTES
# ============================================================

ids_corte_1 = set(
    morelia_corte_1["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_corte_2 = set(
    morelia_corte_2["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"Registros {ETIQUETA_1}:", len(morelia_corte_1))
print("ID únicos:", len(ids_corte_1))

print(f"\nRegistros {ETIQUETA_2}:", len(morelia_corte_2))
print("ID únicos:", len(ids_corte_2))

print("\nID comunes:", len(ids_corte_1 & ids_corte_2))
print(f"Solo {ETIQUETA_1}:", len(ids_corte_1 - ids_corte_2))
print(f"Solo {ETIQUETA_2}:", len(ids_corte_2 - ids_corte_1))

Registros 2019/04: 46097
ID únicos: 46097

Registros 2019/11: 47934
ID únicos: 47934

ID comunes: 26986
Solo 2019/04: 19111
Solo 2019/11: 20948


In [ ]:
# ============================================================
# PASO 9. REVISAR FECHA_ALTA
# ============================================================

print(f"FECHA ALTA {ETIQUETA_1}")
display(
    morelia_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA {ETIQUETA_2}")
display(
    morelia_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2019/04


fecha_alta
2010-07    25802
2014-12    17811
2016-01      449
2013-07      398
2011-03      337
2019-04      319
2018-03      266
2016-10      216
2012-06      192
2017-11      157
2017-03      111
2018-11       39
Name: count, dtype: int64


FECHA ALTA 2019/11


fecha_alta
2019-11    20939
2010-07    17349
2014-12     8113
2019-04      271
2018-03      263
2016-01      219
2013-07      213
2011-03      155
2012-06      124
2016-10      111
2017-11       76
2017-03       60
2018-11       41
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 6. REVISAR METADATOS DE AMBOS CORTES
# ============================================================

def leer_metadatos(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:

        archivo_meta = next(
            nombre for nombre in z.namelist()
            if nombre.lower().endswith(".txt")
            and "metadato" in nombre.lower()
        )

        contenido = z.read(archivo_meta)

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                return contenido.decode(encoding)
            except UnicodeDecodeError:
                continue

    return ""

meta_corte_1 = leer_metadatos(zip_corte_1)
meta_corte_2 = leer_metadatos(zip_corte_2)

print(f"===== {ETIQUETA_1} =====")
for linea in meta_corte_1.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

print(f"\n===== {ETIQUETA_2} =====")
for linea in meta_corte_2.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

===== 2019/04 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados del operativo de actualización realizado por el INEGI en el segundo semestre del 2018. 

===== 2019/11 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados del operativo de actualización realizado por el INEGI en el segundo semestre del 2018. 


In [ ]:
# ============================================================
# PASO 7. FILTRAR MORELIA EN AMBOS CORTES
# ============================================================

for df in [denue_corte_1, denue_corte_2]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_corte_1 = denue_corte_1[
    denue_corte_1["cve_mun"] == "053"
].copy()

morelia_corte_2 = denue_corte_2[
    denue_corte_2["cve_mun"] == "053"
].copy()

print(f"MORELIA {ETIQUETA_1}:", len(morelia_corte_1))
print(f"MORELIA {ETIQUETA_2}:", len(morelia_corte_2))

print(
    f"\nDiscrepancias {ETIQUETA_1}:",
    (
        (denue_corte_1["cve_mun"] == "053")
        !=
        (denue_corte_1["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    f"Discrepancias {ETIQUETA_2}:",
    (
        (denue_corte_2["cve_mun"] == "053")
        !=
        (denue_corte_2["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2019/04: 46097
MORELIA 2019/11: 47934

Discrepancias 2019/04: 0
Discrepancias 2019/11: 0


### Ruptura metodológica observada en DENUE 2019

La comparación entre DENUE 2019/04 y DENUE 2019/11 muestra una ruptura
importante asociada con la actualización total del Directorio mediante los
Censos Económicos 2019.

En Morelia, el número de registros pasa de 46,097 en abril a 47,934 en
noviembre. Únicamente 26,986 identificadores permanecen en ambos cortes;
19,111 registros se encuentran únicamente en abril y 20,948 únicamente
en noviembre.

De estos últimos, 20,939 presentan fecha de alta 2019-11. Estas altas se
interpretan como incorporaciones al DENUE derivadas del proceso de
actualización del Directorio y no como nacimientos empresariales.

Por tanto, las diferencias de identificadores entre DENUE 2019/04 y
DENUE 2019/11 no se utilizan directamente para inferir permanencias,
nacimientos o cierres empresariales.

In [ ]:
# ============================================================
# PASO 10. SEPARAR REGISTROS EXCLUSIVOS DE AMBOS CORTES
# ============================================================

solo_corte_1 = morelia_corte_1[
    morelia_corte_1["id"].astype(str).isin(
        ids_corte_1 - ids_corte_2
    )
].copy()

solo_corte_2 = morelia_corte_2[
    morelia_corte_2["id"].astype(str).isin(
        ids_corte_2 - ids_corte_1
    )
].copy()

print(f"Solo {ETIQUETA_1}:", len(solo_corte_1))
print(f"Solo {ETIQUETA_2}:", len(solo_corte_2))

print(f"\nFECHA ALTA - SOLO {ETIQUETA_1}")
display(
    solo_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA - SOLO {ETIQUETA_2}")
display(
    solo_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

Solo 2019/04: 19111
Solo 2019/11: 20948

FECHA ALTA - SOLO 2019/04


fecha_alta
2014-12    9699
2010-07    8457
2016-01     231
2013-07     185
2011-03     182
2016-10     105
2017-11      81
2012-06      68
2017-03      51
2019-04      49
2018-03       3
Name: count, dtype: int64


FECHA ALTA - SOLO 2019/11


fecha_alta
2019-11    20939
2010-07        4
2018-11        2
2016-01        1
2014-12        1
2019-04        1
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 11. BUSCAR POSIBLES CAMBIOS DE ID MEDIANTE HUELLA
# ============================================================

campos_huella = [
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_generica(df):
    copia = df.copy()

    for campo in campos_huella:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella
    ]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

corte_1_h = crear_huella_generica(solo_corte_1)
corte_2_h = crear_huella_generica(solo_corte_2)

huellas_corte_1 = set(corte_1_h["HUELLA"])
huellas_corte_2 = set(corte_2_h["HUELLA"])

print(f"Huellas únicas {ETIQUETA_1}:", len(huellas_corte_1))
print(f"Huellas únicas {ETIQUETA_2}:", len(huellas_corte_2))

print("\nCoincidencias exactas entre exclusivos:")
print(len(huellas_corte_1 & huellas_corte_2))

print(f"\nSolo {ETIQUETA_1} tras huella:")
print(len(huellas_corte_1 - huellas_corte_2))

print(f"\nSolo {ETIQUETA_2} tras huella:")
print(len(huellas_corte_2 - huellas_corte_1))

Huellas únicas 2019/04: 18850
Huellas únicas 2019/11: 20654

Coincidencias exactas entre exclusivos:
36

Solo 2019/04 tras huella:
18814

Solo 2019/11 tras huella:
20618


In [ ]:
# ============================================================
# PASO 12. COINCIDENCIAS EXACTAS UNO-A-UNO ENTRE EXCLUSIVOS
# ============================================================

frecuencias_corte_1 = (
    corte_1_h["HUELLA"]
    .value_counts()
    .rename("n_corte_1")
)

frecuencias_corte_2 = (
    corte_2_h["HUELLA"]
    .value_counts()
    .rename("n_corte_2")
)

comparacion_huellas = pd.concat(
    [frecuencias_corte_1, frecuencias_corte_2],
    axis=1
).fillna(0).astype(int)

comparacion_huellas["coincidencias_posibles"] = (
    comparacion_huellas[
        ["n_corte_1", "n_corte_2"]
    ].min(axis=1)
)

coincidencias_uno_a_uno = (
    comparacion_huellas["coincidencias_posibles"].sum()
)

print("Registros exclusivos corte 1:", len(solo_corte_1))
print("Registros exclusivos corte 2:", len(solo_corte_2))

print(
    "\nHuellas distintas presentes en ambos:",
    (
        (comparacion_huellas["n_corte_1"] > 0)
        &
        (comparacion_huellas["n_corte_2"] > 0)
    ).sum()
)

print(
    "Máximo de coincidencias exactas uno-a-uno:",
    coincidencias_uno_a_uno
)

Registros exclusivos corte 1: 19111
Registros exclusivos corte 2: 20948

Huellas distintas presentes en ambos: 36
Máximo de coincidencias exactas uno-a-uno: 36


In [ ]:
# ============================================================
# PASO 13. COMPARAR DISTRIBUCIÓN SCIAN
# ============================================================

conteo_scian_corte_1 = (
    morelia_corte_1["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_scian_corte_2 = (
    morelia_corte_2["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

scian_corte_1 = set(conteo_scian_corte_1.index)
scian_corte_2 = set(conteo_scian_corte_2.index)

print(f"Clases SCIAN {ETIQUETA_1}:", len(scian_corte_1))
print(f"Clases SCIAN {ETIQUETA_2}:", len(scian_corte_2))

print("\nClases comunes:", len(scian_corte_1 & scian_corte_2))
print(f"Solo {ETIQUETA_1}:", len(scian_corte_1 - scian_corte_2))
print(f"Solo {ETIQUETA_2}:", len(scian_corte_2 - scian_corte_1))

comparacion_scian = pd.concat(
    [
        conteo_scian_corte_1.rename(ETIQUETA_1),
        conteo_scian_corte_2.rename(ETIQUETA_2)
    ],
    axis=1
).fillna(0).astype(int)

comparacion_scian["Diferencia"] = (
    comparacion_scian[ETIQUETA_2]
    - comparacion_scian[ETIQUETA_1]
)

diferencias_scian = comparacion_scian[
    comparacion_scian["Diferencia"] != 0
]

print(
    "\nClases cuyo número de registros cambió:",
    len(diferencias_scian)
)


Clases SCIAN 2019/04: 675
Clases SCIAN 2019/11: 709

Clases comunes: 625
Solo 2019/04: 50
Solo 2019/11: 84

Clases cuyo número de registros cambió: 648


In [ ]:
# ============================================================
# PASO 13. COMPARAR DISTRIBUCIÓN SCIAN ENTRE CORTES
# ============================================================

conteo_scian_corte_1 = (
    morelia_corte_1["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_scian_corte_2 = (
    morelia_corte_2["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

scian_corte_1 = set(conteo_scian_corte_1.index)
scian_corte_2 = set(conteo_scian_corte_2.index)

print(f"Clases SCIAN {ETIQUETA_1}:", len(scian_corte_1))
print(f"Clases SCIAN {ETIQUETA_2}:", len(scian_corte_2))

print("\nClases comunes:", len(scian_corte_1 & scian_corte_2))
print(f"Solo {ETIQUETA_1}:", len(scian_corte_1 - scian_corte_2))
print(f"Solo {ETIQUETA_2}:", len(scian_corte_2 - scian_corte_1))

comparacion_scian = pd.concat(
    [
        conteo_scian_corte_1.rename(ETIQUETA_1),
        conteo_scian_corte_2.rename(ETIQUETA_2)
    ],
    axis=1
).fillna(0).astype(int)

comparacion_scian["Diferencia"] = (
    comparacion_scian[ETIQUETA_2]
    - comparacion_scian[ETIQUETA_1]
)

diferencias_scian = comparacion_scian[
    comparacion_scian["Diferencia"] != 0
]

print(
    "\nClases cuyo número de registros cambió:",
    len(diferencias_scian)
)

print("\n20 mayores aumentos:")
display(
    diferencias_scian
    .sort_values("Diferencia", ascending=False)
    .head(20)
)

print("\n20 mayores disminuciones:")
display(
    diferencias_scian
    .sort_values("Diferencia", ascending=True)
    .head(20)
)

Clases SCIAN 2019/04: 675
Clases SCIAN 2019/11: 709

Clases comunes: 625
Solo 2019/04: 50
Solo 2019/11: 84

Clases cuyo número de registros cambió: 648

20 mayores aumentos:


,2019/04,2019/11,Diferencia
codigo_act,,,
812110,1767,2328,561
722518,512,713,201
722515,589,752,163
722516,56,210,154
465111,166,302,136
722514,1298,1409,111
722513,958,1064,106
621211,1056,1159,103
327111,619,721,102



20 mayores disminuciones:


,2019/04,2019/11,Diferencia
codigo_act,,,
561432,567,291,-276
461130,964,838,-126
461110,3999,3880,-119
931210,275,158,-117
522110,546,439,-107
524210,140,46,-94
811499,402,315,-87
541110,422,362,-60
461212,216,156,-60


### Decisión metodológica para 2019

Se auditaron las ediciones DENUE 2019/04 y DENUE 2019/11.

En Morelia, el corte de abril contiene 46,097 registros y el de noviembre
47,934. Se identificaron 26,986 ID comunes, 19,111 registros presentes
únicamente en abril y 20,948 presentes únicamente en noviembre.

De los registros exclusivos de noviembre, 20,939 presentan fecha de alta
2019-11. Estas altas corresponden a incorporaciones al DENUE y no se
interpretan automáticamente como nacimientos empresariales.

La comparación mediante una huella compuesta de ubicación, nombre y
domicilio encontró únicamente 36 coincidencias exactas uno-a-uno entre los
registros exclusivos de ambos cortes. Por tanto, el amplio recambio de
identificadores no puede explicarse como un simple cambio de ID.

El número de clases SCIAN representadas pasa de 675 en abril a 709 en
noviembre. Se observan 625 clases comunes, 50 presentes únicamente en abril
y 84 únicamente en noviembre; 648 clases presentan cambios en el número de
establecimientos registrados.

Por tratarse de una actualización amplia del directorio, estas diferencias
no se interpretan directamente como nacimientos, cierres o cambios reales
de actividad empresarial.

Para la serie anual se selecciona DENUE 2019/11 como corte representativo
de 2019, conservando DENUE 2019/04 como corte auditado.

In [ ]:
# ============================================================
# PASO 14. SELECCIONAR CORTE ANUAL
# ============================================================

CORTE_SELECCIONADO = ETIQUETA_2
morelia_seleccionado = morelia_corte_2.copy()

print("Año:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)
print("Registros Morelia:", len(morelia_seleccionado))

archivo_salida = (
    MORELIA / f"DENUE_{ANIO}_MORELIA.csv"
)

morelia_seleccionado.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Archivo Morelia guardado:")
print(archivo_salida)

Año: 2019
Corte seleccionado: 2019/11
Registros Morelia: 47934

✓ Archivo Morelia guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2019_MORELIA.csv


In [ ]:
# ============================================================
# PASO 15. CONTEO POR CLASE SCIAN DEL CORTE SELECCIONADO
# ============================================================

conteo_scian_anual = (
    morelia_seleccionado
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Año:", ANIO)
print("Corte:", CORTE_SELECCIONADO)
print("Total Morelia:", len(morelia_seleccionado))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

display(conteo_scian_anual.head(20))

Año: 2019
Corte: 2019/11
Total Morelia: 47934
Clases SCIAN distintas: 709


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
284,461110,3880,461110,46
684,812110,2328,812110,81
304,463211,1512,463211,46
651,722514,1409,722514,72
68,311830,1176,311830,31
561,621211,1159,621211,62
650,722513,1064,722513,72
326,465311,946,465311,46
288,461130,838,461130,46
657,811111,789,811111,81


In [ ]:
# ============================================================
# PASO 16. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(
        sectores_servicios
    )
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

servicios_mayores_50_anual = (
    servicios_mayores_50_anual
    .sort_values(
        "Número de establecimientos",
        ascending=False
    )
)

print("COMERCIO AL POR MENOR")
print(
    "Clases:",
    comercio_anual["SCIAN6"].nunique()
)
print(
    "Establecimientos:",
    comercio_anual[
        "Número de establecimientos"
    ].sum()
)

print("\nSERVICIOS")
print(
    "Clases:",
    servicios_anual["SCIAN6"].nunique()
)
print(
    "Establecimientos:",
    servicios_anual[
        "Número de establecimientos"
    ].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print(
    "Clases:",
    len(servicios_mayores_50_anual)
)
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 17642

SERVICIOS
Clases: 332
Establecimientos: 22842

SERVICIOS CON MÁS DE 50
Clases: 72
Establecimientos: 20155


In [ ]:
# ============================================================
# PASO 17. GUARDAR RESULTADOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Resultados {ANIO} guardados correctamente.")

✓ Resultados 2019 guardados correctamente.


In [ ]:
# ============================================================
# PASO 15. CONTEO POR CLASE SCIAN DEL CORTE SELECCIONADO
# ============================================================

conteo_scian_anual = (
    morelia_seleccionado
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Año:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)
print("Total Morelia:", len(morelia_seleccionado))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Año: 2020
Corte seleccionado: 2019/11
Total Morelia: 47934
Clases SCIAN distintas: 709


In [ ]:
# ============================================================
# PASO 16. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual["Número de establecimientos"].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 17642

SERVICIOS
Clases: 332
Establecimientos: 22842

SERVICIOS CON MÁS DE 50
Clases: 72
Establecimientos: 20155


In [ ]:
# ============================================================
# PASO 18. ACTUALIZAR RESUMEN HISTÓRICO AUTOMÁTICAMENTE
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),

    "Total_Morelia":
        len(morelia_seleccionado),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN":
        "SCIAN 2018",

    "Observacion_metodologica":
        f"Se auditaron {ETIQUETA_1} y {ETIQUETA_2}; "
        f"se seleccionó {CORTE_SELECCIONADO} como corte anual."
}])

# Evitar duplicar el año si ejecutamos la celda otra vez
resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

# Guardar
resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen histórico actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2020,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...


✓ Resumen histórico actualizado hasta 2020.


In [ ]:
# ============================================================
# PASO 19. ACTUALIZAR BITÁCORA DE CORTES
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_1}",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_2}",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Bitácora {ANIO} actualizada.")

✓ Bitácora 2020 actualizada.


In [ ]:
# ============================================================
# PASO 20. GUARDAR PRODUCTOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Todos los productos {ANIO} guardados.")

✓ Todos los productos 2020 guardados.


In [ ]:
# ============================================================
# CORRECCIÓN: LA FILA MARCADA COMO 2020 CORRESPONDE A 2019
# ============================================================

# Eliminar la fila incorrectamente etiquetada como 2020
resumen_historico = resumen_historico[
    resumen_historico["Año"] != 2020
].copy()

# Corregir el año de trabajo
ANIO = "2019"
CORTE_1 = "04"
CORTE_2 = "11"

ETIQUETA_1 = f"{ANIO}/{CORTE_1}"
ETIQUETA_2 = f"{ANIO}/{CORTE_2}"
CORTE_SELECCIONADO = ETIQUETA_2

print("Año corregido:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)

Año corregido: 2019
Corte seleccionado: 2019/11


In [ ]:
# ============================================================
# PASO 18. ACTUALIZAR RESUMEN HISTÓRICO AUTOMÁTICAMENTE
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),

    "Total_Morelia":
        len(morelia_seleccionado),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN":
        "SCIAN 2018",

    "Observacion_metodologica":
        f"Se auditaron {ETIQUETA_1} y {ETIQUETA_2}; "
        f"se seleccionó {CORTE_SELECCIONADO} como corte anual."
}])

# Evitar duplicar el año si ejecutamos la celda otra vez
resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

# Guardar
resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen histórico actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...


✓ Resumen histórico actualizado hasta 2019.


In [ ]:
# ============================================================
# PASO 19. ACTUALIZAR BITÁCORA DE CORTES
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_1}",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_2}",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Bitácora {ANIO} actualizada.")

✓ Bitácora 2019 actualizada.


In [ ]:
# ============================================================
# PASO 20. GUARDAR PRODUCTOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Todos los productos {ANIO} guardados.")

✓ Todos los productos 2019 guardados.


# DENUE 2020

In [ ]:
# ============================================================
# DENUE 2020
# CONFIGURACIÓN DEL AÑO
# ============================================================

ANIO = "2020"
CORTE_1 = "04"
CORTE_2 = "11"

ETIQUETA_1 = f"{ANIO}/{CORTE_1}"
ETIQUETA_2 = f"{ANIO}/{CORTE_2}"

CARPETA_CORTE_1 = ORIGINALES / f"{ANIO}_{CORTE_1}"
CARPETA_CORTE_2 = ORIGINALES / f"{ANIO}_{CORTE_2}"

print("Año de trabajo:", ANIO)
print("Corte 1:", ETIQUETA_1)
print("Corte 2:", ETIQUETA_2)

Año de trabajo: 2020
Corte 1: 2020/04
Corte 2: 2020/11


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ARCHIVOS DE AMBOS CORTES
# ============================================================

print(f"Carpeta {ETIQUETA_1} existe:", CARPETA_CORTE_1.exists())
print(f"Carpeta {ETIQUETA_2} existe:", CARPETA_CORTE_2.exists())

print(f"\nArchivos {ETIQUETA_1}:")
for archivo in CARPETA_CORTE_1.iterdir():
    print("-", archivo.name)

print(f"\nArchivos {ETIQUETA_2}:")
for archivo in CARPETA_CORTE_2.iterdir():
    print("-", archivo.name)

Carpeta 2020/04 existe: True
Carpeta 2020/11 existe: True

Archivos 2020/04:
- denue_16_0420_csv.zip

Archivos 2020/11:
- denue_16_1120_csv.zip


In [ ]:
# ============================================================
# PASO 2. REVISAR CONTENIDO INTERNO DE LOS ZIP
# ============================================================

zip_corte_1 = next(CARPETA_CORTE_1.glob("*.zip"))
zip_corte_2 = next(CARPETA_CORTE_2.glob("*.zip"))

print(f"===== {ETIQUETA_1} =====")
print("ZIP:", zip_corte_1.name)

with zipfile.ZipFile(zip_corte_1, "r") as z:
    contenido_corte_1 = z.namelist()
    for nombre in contenido_corte_1:
        print("-", nombre)

print(f"\n===== {ETIQUETA_2} =====")
print("ZIP:", zip_corte_2.name)

with zipfile.ZipFile(zip_corte_2, "r") as z:
    contenido_corte_2 = z.namelist()
    for nombre in contenido_corte_2:
        print("-", nombre)

===== 2020/04 =====
ZIP: denue_16_0420_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt

===== 2020/11 =====
ZIP: denue_16_1120_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 3. DETECTAR CSV PRINCIPAL DE AMBOS CORTES
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    csv_corte_1 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    csv_corte_2 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print(f"CSV principal {ETIQUETA_1}:")
print(csv_corte_1)

print(f"\nCSV principal {ETIQUETA_2}:")
print(csv_corte_2)

CSV principal 2020/04:
conjunto_de_datos/denue_inegi_16_.csv

CSV principal 2020/11:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 4. INSPECCIÓN ESTRUCTURAL DE AMBOS CORTES
# ============================================================

def leer_muestra_zip(zip_path, csv_interno):
    for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(csv_interno) as archivo:
                    muestra = pd.read_csv(
                        archivo,
                        nrows=5,
                        encoding=codificacion,
                        dtype=str
                    )
            return muestra, codificacion
        except UnicodeDecodeError:
            continue

    raise ValueError("No se pudo detectar la codificación.")

muestra_corte_1, encoding_corte_1 = leer_muestra_zip(
    zip_corte_1,
    csv_corte_1
)

muestra_corte_2, encoding_corte_2 = leer_muestra_zip(
    zip_corte_2,
    csv_corte_2
)

print(ETIQUETA_1)
print("Codificación:", encoding_corte_1)
print("Columnas:", len(muestra_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Codificación:", encoding_corte_2)
print("Columnas:", len(muestra_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(muestra_corte_1.columns)
    ==
    list(muestra_corte_2.columns)
)

2020/04
Codificación: latin1
Columnas: 41

2020/11
Codificación: latin1
Columnas: 41

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 5. CARGAR AMBOS CORTES COMPLETOS
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    with z.open(csv_corte_1) as archivo:
        denue_corte_1 = pd.read_csv(
            archivo,
            encoding=encoding_corte_1,
            dtype=str
        )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    with z.open(csv_corte_2) as archivo:
        denue_corte_2 = pd.read_csv(
            archivo,
            encoding=encoding_corte_2,
            dtype=str
        )

print(ETIQUETA_1)
print("Filas:", len(denue_corte_1))
print("Columnas:", len(denue_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Filas:", len(denue_corte_2))
print("Columnas:", len(denue_corte_2.columns))

2020/04
Filas: 259070
Columnas: 41

2020/11
Filas: 260767
Columnas: 41


In [ ]:
# ============================================================
# PASO 6. REVISAR METADATOS DE AMBOS CORTES
# ============================================================

def leer_metadatos(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:

        archivo_meta = next(
            nombre for nombre in z.namelist()
            if nombre.lower().endswith(".txt")
            and "metadato" in nombre.lower()
        )

        contenido = z.read(archivo_meta)

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                return contenido.decode(encoding)
            except UnicodeDecodeError:
                continue

    return ""

meta_corte_1 = leer_metadatos(zip_corte_1)
meta_corte_2 = leer_metadatos(zip_corte_2)

print(f"===== {ETIQUETA_1} =====")
for linea in meta_corte_1.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

print(f"\n===== {ETIQUETA_2} =====")
for linea in meta_corte_2.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

===== 2020/04 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados definitivos de los Censos Económicos 2019.

===== 2020/11 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados definitivos de los Censos Económicos 2019.


In [ ]:
# ============================================================
# PASO 7. FILTRAR MORELIA EN AMBOS CORTES
# ============================================================

for df in [denue_corte_1, denue_corte_2]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_corte_1 = denue_corte_1[
    denue_corte_1["cve_mun"] == "053"
].copy()

morelia_corte_2 = denue_corte_2[
    denue_corte_2["cve_mun"] == "053"
].copy()

print(f"MORELIA {ETIQUETA_1}:", len(morelia_corte_1))
print(f"MORELIA {ETIQUETA_2}:", len(morelia_corte_2))

print(
    f"\nDiscrepancias {ETIQUETA_1}:",
    (
        (denue_corte_1["cve_mun"] == "053")
        !=
        (denue_corte_1["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    f"Discrepancias {ETIQUETA_2}:",
    (
        (denue_corte_2["cve_mun"] == "053")
        !=
        (denue_corte_2["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2020/04: 48242
MORELIA 2020/11: 48981

Discrepancias 2020/04: 0
Discrepancias 2020/11: 0


In [ ]:
# ============================================================
# PASO 8. COMPARAR IDENTIFICADORES ENTRE CORTES
# ============================================================

ids_corte_1 = set(
    morelia_corte_1["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_corte_2 = set(
    morelia_corte_2["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"Registros {ETIQUETA_1}:", len(morelia_corte_1))
print("ID únicos:", len(ids_corte_1))

print(f"\nRegistros {ETIQUETA_2}:", len(morelia_corte_2))
print("ID únicos:", len(ids_corte_2))

print("\nID comunes:", len(ids_corte_1 & ids_corte_2))
print(f"Solo {ETIQUETA_1}:", len(ids_corte_1 - ids_corte_2))
print(f"Solo {ETIQUETA_2}:", len(ids_corte_2 - ids_corte_1))

Registros 2020/04: 48242
ID únicos: 48242

Registros 2020/11: 48981
ID únicos: 48981

ID comunes: 46918
Solo 2020/04: 1324
Solo 2020/11: 2063


In [ ]:
# ============================================================
# PASO 9. REVISAR FECHA_ALTA
# ============================================================

print(f"FECHA ALTA {ETIQUETA_1}")
display(
    morelia_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA {ETIQUETA_2}")
display(
    morelia_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2020/04


fecha_alta
2019-11    20974
2010-07    17353
2014-12     8122
2019-04      287
2018-03      263
2016-01      221
2013-07      214
2011-03      204
2020-04      189
2012-06      125
2016-10      117
2017-11       76
2017-03       60
2018-11       37
Name: count, dtype: int64


FECHA ALTA 2020/11


fecha_alta
2019-11    19732
2010-07    17990
2014-12     8507
2020-11      941
2019-04      275
2018-03      263
2016-01      231
2013-07      217
2011-03      215
2020-04      174
2012-06      131
2016-10      120
2017-11       81
2017-03       53
2018-11       37
2017 03        7
2013 07        7
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 10. REGISTROS EXCLUSIVOS DE AMBOS CORTES
# ============================================================

solo_corte_1 = morelia_corte_1[
    morelia_corte_1["id"].astype(str).isin(
        ids_corte_1 - ids_corte_2
    )
].copy()

solo_corte_2 = morelia_corte_2[
    morelia_corte_2["id"].astype(str).isin(
        ids_corte_2 - ids_corte_1
    )
].copy()

print(f"Solo {ETIQUETA_1}:", len(solo_corte_1))
print(f"Solo {ETIQUETA_2}:", len(solo_corte_2))

print(f"\nFECHA ALTA - SOLO {ETIQUETA_1}")
display(
    solo_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA - SOLO {ETIQUETA_2}")
display(
    solo_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

Solo 2020/04: 1324
Solo 2020/11: 2063

FECHA ALTA - SOLO 2020/04


fecha_alta
2019-11    1240
2020-04      23
2010-07      21
2014-12      14
2019-04      12
2016-01       6
2016-10       5
2012-06       1
2017-03       1
2013-07       1
Name: count, dtype: int64


FECHA ALTA - SOLO 2020/11


fecha_alta
2020-11    938
2010-07    658
2014-12    399
2016-01     16
2013-07     11
2011-03     11
2016-10      8
2020-04      8
2012-06      7
2017-11      5
2019-11      1
2017 03      1
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 11. BUSCAR POSIBLES CAMBIOS DE ID MEDIANTE HUELLA
# ============================================================

campos_huella = [
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_generica(df):
    copia = df.copy()

    for campo in campos_huella:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella
    ]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

corte_1_h = crear_huella_generica(solo_corte_1)
corte_2_h = crear_huella_generica(solo_corte_2)

frecuencias_corte_1 = (
    corte_1_h["HUELLA"]
    .value_counts()
    .rename("n_corte_1")
)

frecuencias_corte_2 = (
    corte_2_h["HUELLA"]
    .value_counts()
    .rename("n_corte_2")
)

comparacion_huellas = pd.concat(
    [frecuencias_corte_1, frecuencias_corte_2],
    axis=1
).fillna(0).astype(int)

comparacion_huellas["coincidencias_posibles"] = (
    comparacion_huellas[
        ["n_corte_1", "n_corte_2"]
    ].min(axis=1)
)

print(
    "Huellas distintas presentes en ambos:",
    (
        (comparacion_huellas["n_corte_1"] > 0)
        &
        (comparacion_huellas["n_corte_2"] > 0)
    ).sum()
)

print(
    "Máximo de coincidencias exactas uno-a-uno:",
    comparacion_huellas["coincidencias_posibles"].sum()
)

Huellas distintas presentes en ambos: 1044
Máximo de coincidencias exactas uno-a-uno: 1075


In [ ]:
# ============================================================
# PASO 12. COMPARAR DISTRIBUCIÓN SCIAN
# ============================================================

conteo_scian_corte_1 = (
    morelia_corte_1["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_scian_corte_2 = (
    morelia_corte_2["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

scian_corte_1 = set(conteo_scian_corte_1.index)
scian_corte_2 = set(conteo_scian_corte_2.index)

print(f"Clases SCIAN {ETIQUETA_1}:", len(scian_corte_1))
print(f"Clases SCIAN {ETIQUETA_2}:", len(scian_corte_2))

print("\nClases comunes:", len(scian_corte_1 & scian_corte_2))
print(f"Solo {ETIQUETA_1}:", len(scian_corte_1 - scian_corte_2))
print(f"Solo {ETIQUETA_2}:", len(scian_corte_2 - scian_corte_1))

comparacion_scian = pd.concat(
    [
        conteo_scian_corte_1.rename(ETIQUETA_1),
        conteo_scian_corte_2.rename(ETIQUETA_2)
    ],
    axis=1
).fillna(0).astype(int)

comparacion_scian["Diferencia"] = (
    comparacion_scian[ETIQUETA_2]
    - comparacion_scian[ETIQUETA_1]
)

diferencias_scian = comparacion_scian[
    comparacion_scian["Diferencia"] != 0
]

print(
    "\nClases cuyo número de registros cambió:",
    len(diferencias_scian)
)

Clases SCIAN 2020/04: 686
Clases SCIAN 2020/11: 685

Clases comunes: 672
Solo 2020/04: 14
Solo 2020/11: 13

Clases cuyo número de registros cambió: 279


In [ ]:
# ============================================================
# PASO 13. SELECCIONAR CORTE ANUAL
# ============================================================

CORTE_SELECCIONADO = ETIQUETA_2
morelia_seleccionado = morelia_corte_2.copy()

print("Año:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)
print("Registros Morelia:", len(morelia_seleccionado))

archivo_salida = (
    MORELIA / f"DENUE_{ANIO}_MORELIA.csv"
)

morelia_seleccionado.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Archivo Morelia guardado:")
print(archivo_salida)

Año: 2020
Corte seleccionado: 2020/11
Registros Morelia: 48981

✓ Archivo Morelia guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2020_MORELIA.csv


In [ ]:
# ============================================================
# PASO 14. CONTEO POR CLASE SCIAN
# ============================================================

conteo_scian_anual = (
    morelia_seleccionado
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_seleccionado))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Total Morelia: 48981
Clases SCIAN distintas: 685


In [ ]:
# ============================================================
# PASO 15. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 17984

SERVICIOS
Clases: 327
Establecimientos: 23464

SERVICIOS CON MÁS DE 50
Clases: 77
Establecimientos: 20876


In [ ]:
# ============================================================
# PASO 16. GUARDAR PRODUCTOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Productos {ANIO} guardados.")

✓ Productos 2020 guardados.


In [ ]:
# ============================================================
# PASO 17. ACTUALIZAR RESUMEN HISTÓRICO
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),
    "Total_Morelia": len(morelia_seleccionado),
    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),
    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),
    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),
    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),
    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),
    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),
    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),
    "Version_SCIAN": "SCIAN 2018",
    "Observacion_metodologica":
        f"Se auditaron {ETIQUETA_1} y {ETIQUETA_2}; "
        f"se seleccionó {CORTE_SELECCIONADO} como corte anual."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen actualizado hasta 2020.


In [ ]:
# ============================================================
# PASO 18. ACTUALIZAR BITÁCORA
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_1}",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_2}",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Bitácora {ANIO} actualizada.")

✓ Bitácora 2020 actualizada.


### Decisión metodológica para 2020

Se auditaron las ediciones DENUE 2020/04 y DENUE 2020/11.

El corte de abril contiene 48,242 registros correspondientes a Morelia,
mientras que el corte de noviembre contiene 48,981, lo que representa una
variación neta de 739 registros.

La comparación mediante identificadores mostró 1,324 registros presentes
únicamente en abril y 2,063 únicamente en noviembre. Sin embargo, mediante
una huella compuesta de ubicación, nombre del establecimiento y domicilio
se identificaron hasta 1,075 coincidencias exactas uno-a-uno entre estos
grupos.

Este resultado muestra que la ausencia de coincidencia en el identificador
no implica necesariamente que se trate de establecimientos diferentes, por
lo que el ID no se utiliza de manera aislada para inferir permanencia,
incorporación o desaparición.

Después de considerar las coincidencias exactas por huella quedan 249
registros de abril y 988 registros de noviembre sin correspondencia exacta,
cuya diferencia coincide con la variación neta de 739 registros entre ambos
cortes.

En cuanto a la clasificación económica, abril registra 686 clases SCIAN y
noviembre 685; 672 clases están presentes en ambos cortes, 14 únicamente en
abril y 13 únicamente en noviembre. En 279 clases cambió el número de
establecimientos registrados.

Para la serie anual se selecciona DENUE 2020/11 como corte representativo
de 2020, conservando DENUE 2020/04 como corte intraanual auditado.

In [ ]:
# ============================================================
# PASO 13. SELECCIONAR CORTE ANUAL
# ============================================================

CORTE_SELECCIONADO = ETIQUETA_2
morelia_seleccionado = morelia_corte_2.copy()

print("Año:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)
print("Registros Morelia:", len(morelia_seleccionado))

archivo_salida = MORELIA / f"DENUE_{ANIO}_MORELIA.csv"

morelia_seleccionado.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Archivo Morelia guardado:")
print(archivo_salida)

Año: 2020
Corte seleccionado: 2020/11
Registros Morelia: 48981

✓ Archivo Morelia guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2020_MORELIA.csv


In [ ]:
# ============================================================
# PASO 14. CONTEO POR CLASE SCIAN
# ============================================================

conteo_scian_anual = (
    morelia_seleccionado
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_seleccionado))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Total Morelia: 48981
Clases SCIAN distintas: 685


In [ ]:
# ============================================================
# PASO 15. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 17984

SERVICIOS
Clases: 327
Establecimientos: 23464

SERVICIOS CON MÁS DE 50
Clases: 77
Establecimientos: 20876


In [ ]:
# ============================================================
# PASO 16. GUARDAR PRODUCTOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Productos {ANIO} guardados.")

✓ Productos 2020 guardados.


In [ ]:
# ============================================================
# PASO 17. ACTUALIZAR RESUMEN HISTÓRICO
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),
    "Total_Morelia": len(morelia_seleccionado),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN": "SCIAN 2018",

    "Observacion_metodologica":
        f"Se auditaron {ETIQUETA_1} y {ETIQUETA_2}; "
        f"se seleccionó {CORTE_SELECCIONADO} como corte anual."
}])

# Evitar duplicar el año
resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen actualizado hasta 2020.


In [ ]:
# ============================================================
# PASO 18. ACTUALIZAR BITÁCORA
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_1}",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_2}",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Bitácora {ANIO} actualizada.")

✓ Bitácora 2020 actualizada.


# DENUE 2021

In [ ]:
# ============================================================
# DENUE 2021
# CONFIGURACIÓN DEL AÑO
# ============================================================

ANIO = "2021"
CORTE_1 = "05"
CORTE_2 = "11"

ETIQUETA_1 = f"{ANIO}/{CORTE_1}"
ETIQUETA_2 = f"{ANIO}/{CORTE_2}"

CARPETA_CORTE_1 = ORIGINALES / f"{ANIO}_{CORTE_1}"
CARPETA_CORTE_2 = ORIGINALES / f"{ANIO}_{CORTE_2}"

print("Año de trabajo:", ANIO)
print("Corte 1:", ETIQUETA_1)
print("Corte 2:", ETIQUETA_2)

Año de trabajo: 2021
Corte 1: 2021/05
Corte 2: 2021/11


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ARCHIVOS
# ============================================================

print(f"Carpeta {ETIQUETA_1} existe:", CARPETA_CORTE_1.exists())
print(f"Carpeta {ETIQUETA_2} existe:", CARPETA_CORTE_2.exists())

print(f"\nArchivos {ETIQUETA_1}:")
for archivo in CARPETA_CORTE_1.iterdir():
    print("-", archivo.name)

print(f"\nArchivos {ETIQUETA_2}:")
for archivo in CARPETA_CORTE_2.iterdir():
    print("-", archivo.name)

Carpeta 2021/05 existe: True
Carpeta 2021/11 existe: True

Archivos 2021/05:
- denue_16_0521_csv.zip

Archivos 2021/11:
- denue_16_1121_csv.zip


In [ ]:
# ============================================================
# PASO 2. REVISAR CONTENIDO DE LOS ZIP
# ============================================================

zip_corte_1 = next(CARPETA_CORTE_1.glob("*.zip"))
zip_corte_2 = next(CARPETA_CORTE_2.glob("*.zip"))

print(f"===== {ETIQUETA_1} =====")
print("ZIP:", zip_corte_1.name)

with zipfile.ZipFile(zip_corte_1, "r") as z:
    for nombre in z.namelist():
        print("-", nombre)

print(f"\n===== {ETIQUETA_2} =====")
print("ZIP:", zip_corte_2.name)

with zipfile.ZipFile(zip_corte_2, "r") as z:
    for nombre in z.namelist():
        print("-", nombre)

===== 2021/05 =====
ZIP: denue_16_0521_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt

===== 2021/11 =====
ZIP: denue_16_1121_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 3. DETECTAR CSV PRINCIPAL
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    csv_corte_1 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    csv_corte_2 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print(f"CSV principal {ETIQUETA_1}:")
print(csv_corte_1)

print(f"\nCSV principal {ETIQUETA_2}:")
print(csv_corte_2)

CSV principal 2021/05:
conjunto_de_datos/denue_inegi_16_.csv

CSV principal 2021/11:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 4. INSPECCIÓN ESTRUCTURAL
# ============================================================

def leer_muestra_zip(zip_path, csv_interno):
    for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(csv_interno) as archivo:
                    muestra = pd.read_csv(
                        archivo,
                        nrows=5,
                        encoding=codificacion,
                        dtype=str
                    )
            return muestra, codificacion
        except UnicodeDecodeError:
            continue

    raise ValueError("No se pudo detectar la codificación.")

muestra_corte_1, encoding_corte_1 = leer_muestra_zip(
    zip_corte_1,
    csv_corte_1
)

muestra_corte_2, encoding_corte_2 = leer_muestra_zip(
    zip_corte_2,
    csv_corte_2
)

print(ETIQUETA_1)
print("Codificación:", encoding_corte_1)
print("Columnas:", len(muestra_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Codificación:", encoding_corte_2)
print("Columnas:", len(muestra_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(muestra_corte_1.columns)
    ==
    list(muestra_corte_2.columns)
)

2021/05
Codificación: latin1
Columnas: 42

2021/11
Codificación: latin1
Columnas: 42

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 5. CARGAR AMBOS CORTES COMPLETOS
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    with z.open(csv_corte_1) as archivo:
        denue_corte_1 = pd.read_csv(
            archivo,
            encoding=encoding_corte_1,
            dtype=str
        )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    with z.open(csv_corte_2) as archivo:
        denue_corte_2 = pd.read_csv(
            archivo,
            encoding=encoding_corte_2,
            dtype=str
        )

print(ETIQUETA_1)
print("Filas:", len(denue_corte_1))
print("Columnas:", len(denue_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Filas:", len(denue_corte_2))
print("Columnas:", len(denue_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(denue_corte_1.columns)
    ==
    list(denue_corte_2.columns)
)

2021/05
Filas: 259749
Columnas: 42

2021/11
Filas: 260228
Columnas: 42

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 6. REVISAR METADATOS
# ============================================================

def leer_metadatos(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:

        candidatos = [
            nombre for nombre in z.namelist()
            if nombre.lower().endswith(".txt")
            and "metadato" in nombre.lower()
        ]

        if not candidatos:
            return "No se encontró archivo de metadatos."

        contenido = z.read(candidatos[0])

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                return contenido.decode(encoding)
            except UnicodeDecodeError:
                continue

    return ""

meta_corte_1 = leer_metadatos(zip_corte_1)
meta_corte_2 = leer_metadatos(zip_corte_2)

print(f"===== {ETIQUETA_1} =====")
for linea in meta_corte_1.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

print(f"\n===== {ETIQUETA_2} =====")
for linea in meta_corte_2.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

===== 2021/05 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados definitivos de los Censos Económicos 2019.

===== 2021/11 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados definitivos de los Censos Económicos 2019.


In [ ]:
# ============================================================
# PASO 7. FILTRAR MORELIA
# ============================================================

for df in [denue_corte_1, denue_corte_2]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_corte_1 = denue_corte_1[
    denue_corte_1["cve_mun"] == "053"
].copy()

morelia_corte_2 = denue_corte_2[
    denue_corte_2["cve_mun"] == "053"
].copy()

print(f"MORELIA {ETIQUETA_1}:", len(morelia_corte_1))
print(f"MORELIA {ETIQUETA_2}:", len(morelia_corte_2))

print(
    f"\nDiscrepancias {ETIQUETA_1}:",
    (
        (denue_corte_1["cve_mun"] == "053")
        !=
        (denue_corte_1["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    f"Discrepancias {ETIQUETA_2}:",
    (
        (denue_corte_2["cve_mun"] == "053")
        !=
        (denue_corte_2["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2021/05: 48725
MORELIA 2021/11: 48834

Discrepancias 2021/05: 0
Discrepancias 2021/11: 0


In [ ]:
# ============================================================
# PASO 8. COMPARAR ID
# ============================================================

ids_corte_1 = set(
    morelia_corte_1["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_corte_2 = set(
    morelia_corte_2["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"Registros {ETIQUETA_1}:", len(morelia_corte_1))
print("ID únicos:", len(ids_corte_1))

print(f"\nRegistros {ETIQUETA_2}:", len(morelia_corte_2))
print("ID únicos:", len(ids_corte_2))

print("\nID comunes:", len(ids_corte_1 & ids_corte_2))
print(f"Solo {ETIQUETA_1}:", len(ids_corte_1 - ids_corte_2))
print(f"Solo {ETIQUETA_2}:", len(ids_corte_2 - ids_corte_1))

Registros 2021/05: 48725
ID únicos: 48725

Registros 2021/11: 48834
ID únicos: 48834

ID comunes: 48690
Solo 2021/05: 35
Solo 2021/11: 144


In [ ]:
# ============================================================
# PASO 9. REVISAR FECHA_ALTA
# ============================================================

print(f"FECHA ALTA {ETIQUETA_1}")
display(
    morelia_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA {ETIQUETA_2}")
display(
    morelia_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2021/05


fecha_alta
2019-11    19464
2010-07    17886
2014-12     8414
2020-11      938
2019-04      267
2018-03      259
2021-05      236
2016-01      230
2013-07      217
2011-03      213
2020-04      173
2012-06      130
2016-10      115
2017-11       81
2017-03       53
2018-11       36
2013 07        7
2017 03        6
Name: count, dtype: int64


FECHA ALTA 2021/11


fecha_alta
2019-11    19457
2010-07    17876
2014-12     8411
2020-11      926
2019-04      269
2018-03      259
2021-05      236
2016-01      229
2013-07      217
2011-03      213
2020-04      172
2021-11      141
2012-06      130
2016-10      115
2017-11       81
2017-03       53
2018-11       36
2013 07        7
2017 03        6
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 10. REGISTROS EXCLUSIVOS
# ============================================================

solo_corte_1 = morelia_corte_1[
    morelia_corte_1["id"].astype(str).isin(
        ids_corte_1 - ids_corte_2
    )
].copy()

solo_corte_2 = morelia_corte_2[
    morelia_corte_2["id"].astype(str).isin(
        ids_corte_2 - ids_corte_1
    )
].copy()

print(f"Solo {ETIQUETA_1}:", len(solo_corte_1))
print(f"Solo {ETIQUETA_2}:", len(solo_corte_2))

print(f"\nFECHA ALTA - SOLO {ETIQUETA_1}")
display(
    solo_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA - SOLO {ETIQUETA_2}")
display(
    solo_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

Solo 2021/05: 35
Solo 2021/11: 144

FECHA ALTA - SOLO 2021/05


fecha_alta
2020-11    13
2010-07    10
2019-11     7
2014-12     3
2016-01     1
2020-04     1
Name: count, dtype: int64


FECHA ALTA - SOLO 2021/11


fecha_alta
2021-11    141
2019-04      2
2020-11      1
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 11. HUELLA COMPUESTA
# ============================================================

campos_huella = [
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_generica(df):
    copia = df.copy()

    for campo in campos_huella:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella
    ]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

corte_1_h = crear_huella_generica(solo_corte_1)
corte_2_h = crear_huella_generica(solo_corte_2)

frecuencias_corte_1 = (
    corte_1_h["HUELLA"]
    .value_counts()
    .rename("n_corte_1")
)

frecuencias_corte_2 = (
    corte_2_h["HUELLA"]
    .value_counts()
    .rename("n_corte_2")
)

comparacion_huellas = pd.concat(
    [frecuencias_corte_1, frecuencias_corte_2],
    axis=1
).fillna(0).astype(int)

comparacion_huellas["coincidencias_posibles"] = (
    comparacion_huellas[
        ["n_corte_1", "n_corte_2"]
    ].min(axis=1)
)

print(
    "Huellas distintas presentes en ambos:",
    (
        (comparacion_huellas["n_corte_1"] > 0)
        &
        (comparacion_huellas["n_corte_2"] > 0)
    ).sum()
)

print(
    "Máximo de coincidencias exactas uno-a-uno:",
    comparacion_huellas["coincidencias_posibles"].sum()
)

Huellas distintas presentes en ambos: 0
Máximo de coincidencias exactas uno-a-uno: 0


In [ ]:
# ============================================================
# PASO 12. COMPARAR DISTRIBUCIÓN SCIAN
# ============================================================

conteo_scian_corte_1 = (
    morelia_corte_1["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_scian_corte_2 = (
    morelia_corte_2["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

scian_corte_1 = set(conteo_scian_corte_1.index)
scian_corte_2 = set(conteo_scian_corte_2.index)

print(f"Clases SCIAN {ETIQUETA_1}:", len(scian_corte_1))
print(f"Clases SCIAN {ETIQUETA_2}:", len(scian_corte_2))

print("\nClases comunes:", len(scian_corte_1 & scian_corte_2))
print(f"Solo {ETIQUETA_1}:", len(scian_corte_1 - scian_corte_2))
print(f"Solo {ETIQUETA_2}:", len(scian_corte_2 - scian_corte_1))

comparacion_scian = pd.concat(
    [
        conteo_scian_corte_1.rename(ETIQUETA_1),
        conteo_scian_corte_2.rename(ETIQUETA_2)
    ],
    axis=1
).fillna(0).astype(int)

comparacion_scian["Diferencia"] = (
    comparacion_scian[ETIQUETA_2]
    - comparacion_scian[ETIQUETA_1]
)

diferencias_scian = comparacion_scian[
    comparacion_scian["Diferencia"] != 0
]

print(
    "\nClases cuyo número de registros cambió:",
    len(diferencias_scian)
)

Clases SCIAN 2021/05: 688
Clases SCIAN 2021/11: 689

Clases comunes: 688
Solo 2021/05: 0
Solo 2021/11: 1

Clases cuyo número de registros cambió: 95


In [ ]:
# ============================================================
# PASO 13. SELECCIONAR CORTE ANUAL
# ============================================================

CORTE_SELECCIONADO = ETIQUETA_2
morelia_seleccionado = morelia_corte_2.copy()

print("Año:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)
print("Registros Morelia:", len(morelia_seleccionado))

archivo_salida = MORELIA / f"DENUE_{ANIO}_MORELIA.csv"

morelia_seleccionado.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Archivo Morelia guardado:")
print(archivo_salida)

Año: 2021
Corte seleccionado: 2021/11
Registros Morelia: 48834

✓ Archivo Morelia guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2021_MORELIA.csv


In [ ]:
# ============================================================
# PASO 14. CONTEO POR CLASE SCIAN
# ============================================================

conteo_scian_anual = (
    morelia_seleccionado
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_seleccionado))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Total Morelia: 48834
Clases SCIAN distintas: 689


In [ ]:
# ============================================================
# PASO 15. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 17992

SERVICIOS
Clases: 327
Establecimientos: 23298

SERVICIOS CON MÁS DE 50
Clases: 77
Establecimientos: 20726


In [ ]:
# ============================================================
# PASO 16. GUARDAR PRODUCTOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Productos {ANIO} guardados.")

✓ Productos 2021 guardados.


In [ ]:
# ============================================================
# PASO 17. ACTUALIZAR RESUMEN HISTÓRICO
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),
    "Total_Morelia": len(morelia_seleccionado),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN": "SCIAN 2018",

    "Observacion_metodologica":
        f"Se auditaron {ETIQUETA_1} y {ETIQUETA_2}; "
        f"se seleccionó {CORTE_SELECCIONADO} como corte anual."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen actualizado hasta 2021.


In [ ]:
# ============================================================
# PASO 18. ACTUALIZAR BITÁCORA
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_1}",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_2}",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Bitácora {ANIO} actualizada.")

✓ Bitácora 2021 actualizada.


### Decisión metodológica para 2021

Se auditaron las ediciones DENUE 2021/05 y DENUE 2021/11.

El corte de mayo contiene 48,725 registros correspondientes a Morelia,
mientras que el corte de noviembre contiene 48,834, lo que representa una
variación neta de 109 registros.

Se identificaron 48,690 ID comunes entre ambos cortes, 35 registros presentes
únicamente en mayo y 144 presentes únicamente en noviembre.

De los 144 registros exclusivos de noviembre, 141 presentan fecha de alta
2021-11; los tres restantes corresponden a cohortes de incorporación
anteriores. Los 35 registros exclusivos de mayo también pertenecen a
distintas cohortes de incorporación.

La comparación mediante una huella compuesta de ubicación, nombre del
establecimiento y domicilio no identificó coincidencias exactas entre los
registros exclusivos de ambos cortes, por lo que no se detectaron simples
cambios de identificador entre estos grupos.

En cuanto a la clasificación económica, el corte de mayo contiene 688 clases
SCIAN y el de noviembre 689. Las 688 clases de mayo también están presentes
en noviembre, mientras que una clase aparece únicamente en el corte de
noviembre. En 95 clases cambió el número de establecimientos registrados.

Las diferencias observadas entre ambos cortes se interpretan como cambios
en los registros del DENUE y no automáticamente como nacimientos o cierres
empresariales.

Para la serie anual se selecciona DENUE 2021/11 como corte representativo
de 2021, conservando DENUE 2021/05 como corte intraanual auditado.

# DENUE 2022

In [ ]:
# ============================================================
# DENUE 2022
# CONFIGURACIÓN DEL AÑO
# ============================================================

ANIO = "2022"
CORTE_1 = "05"
CORTE_2 = "11"

ETIQUETA_1 = f"{ANIO}/{CORTE_1}"
ETIQUETA_2 = f"{ANIO}/{CORTE_2}"

CARPETA_CORTE_1 = ORIGINALES / f"{ANIO}_{CORTE_1}"
CARPETA_CORTE_2 = ORIGINALES / f"{ANIO}_{CORTE_2}"

print("Año de trabajo:", ANIO)
print("Corte 1:", ETIQUETA_1)
print("Corte 2:", ETIQUETA_2)

Año de trabajo: 2022
Corte 1: 2022/05
Corte 2: 2022/11


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ARCHIVOS
# ============================================================

print(f"Carpeta {ETIQUETA_1} existe:", CARPETA_CORTE_1.exists())
print(f"Carpeta {ETIQUETA_2} existe:", CARPETA_CORTE_2.exists())

print(f"\nArchivos {ETIQUETA_1}:")
for archivo in CARPETA_CORTE_1.iterdir():
    print("-", archivo.name)

print(f"\nArchivos {ETIQUETA_2}:")
for archivo in CARPETA_CORTE_2.iterdir():
    print("-", archivo.name)

Carpeta 2022/05 existe: True
Carpeta 2022/11 existe: True

Archivos 2022/05:
- denue_16_0522_csv.zip

Archivos 2022/11:
- denue_16_1122_csv.zip


In [ ]:
# ============================================================
# PASO 2. REVISAR CONTENIDO DE LOS ZIP
# ============================================================

zip_corte_1 = next(CARPETA_CORTE_1.glob("*.zip"))
zip_corte_2 = next(CARPETA_CORTE_2.glob("*.zip"))

print(f"===== {ETIQUETA_1} =====")
print("ZIP:", zip_corte_1.name)

with zipfile.ZipFile(zip_corte_1, "r") as z:
    for nombre in z.namelist():
        print("-", nombre)

print(f"\n===== {ETIQUETA_2} =====")
print("ZIP:", zip_corte_2.name)

with zipfile.ZipFile(zip_corte_2, "r") as z:
    for nombre in z.namelist():
        print("-", nombre)

===== 2022/05 =====
ZIP: denue_16_0522_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt

===== 2022/11 =====
ZIP: denue_16_1122_csv.zip
- conjunto_de_datos/denue_inegi_16_.csv
- diccionario_de_datos/denue_diccionario_de_datos.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 3. DETECTAR CSV PRINCIPAL
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    csv_corte_1 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    csv_corte_2 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print(f"CSV principal {ETIQUETA_1}:")
print(csv_corte_1)

print(f"\nCSV principal {ETIQUETA_2}:")
print(csv_corte_2)

CSV principal 2022/05:
conjunto_de_datos/denue_inegi_16_.csv

CSV principal 2022/11:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 4. INSPECCIÓN ESTRUCTURAL
# ============================================================

def leer_muestra_zip(zip_path, csv_interno):
    for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(csv_interno) as archivo:
                    muestra = pd.read_csv(
                        archivo,
                        nrows=5,
                        encoding=codificacion,
                        dtype=str
                    )
            return muestra, codificacion
        except UnicodeDecodeError:
            continue

    raise ValueError("No se pudo detectar la codificación.")

muestra_corte_1, encoding_corte_1 = leer_muestra_zip(
    zip_corte_1,
    csv_corte_1
)

muestra_corte_2, encoding_corte_2 = leer_muestra_zip(
    zip_corte_2,
    csv_corte_2
)

print(ETIQUETA_1)
print("Codificación:", encoding_corte_1)
print("Columnas:", len(muestra_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Codificación:", encoding_corte_2)
print("Columnas:", len(muestra_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(muestra_corte_1.columns)
    ==
    list(muestra_corte_2.columns)
)

2022/05
Codificación: latin1
Columnas: 42

2022/11
Codificación: latin1
Columnas: 42

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 5. CARGAR AMBOS CORTES COMPLETOS
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    with z.open(csv_corte_1) as archivo:
        denue_corte_1 = pd.read_csv(
            archivo,
            encoding=encoding_corte_1,
            dtype=str
        )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    with z.open(csv_corte_2) as archivo:
        denue_corte_2 = pd.read_csv(
            archivo,
            encoding=encoding_corte_2,
            dtype=str
        )

print(ETIQUETA_1)
print("Filas:", len(denue_corte_1))
print("Columnas:", len(denue_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Filas:", len(denue_corte_2))
print("Columnas:", len(denue_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(denue_corte_1.columns)
    ==
    list(denue_corte_2.columns)
)

2022/05
Filas: 260135
Columnas: 42

2022/11
Filas: 260311
Columnas: 42

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 6. REVISAR METADATOS
# ============================================================

def leer_metadatos(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:

        candidatos = [
            nombre for nombre in z.namelist()
            if nombre.lower().endswith(".txt")
            and "metadato" in nombre.lower()
        ]

        if not candidatos:
            return "No se encontró archivo de metadatos."

        contenido = z.read(candidatos[0])

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                return contenido.decode(encoding)
            except UnicodeDecodeError:
                continue

    return ""

meta_corte_1 = leer_metadatos(zip_corte_1)
meta_corte_2 = leer_metadatos(zip_corte_2)

print(f"===== {ETIQUETA_1} =====")
for linea in meta_corte_1.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

print(f"\n===== {ETIQUETA_2} =====")
for linea in meta_corte_2.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "incorpor",
            "baja"
        ]
    ):
        print(linea)

===== 2022/05 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados definitivos de los Censos Económicos 2019.

===== 2022/11 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes. Esta edición del DENUE se actualizó con base en los resultados definitivos de los Censos Económicos 2019.


In [ ]:
# ============================================================
# PASO 7. FILTRAR MORELIA
# ============================================================

for df in [denue_corte_1, denue_corte_2]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_corte_1 = denue_corte_1[
    denue_corte_1["cve_mun"] == "053"
].copy()

morelia_corte_2 = denue_corte_2[
    denue_corte_2["cve_mun"] == "053"
].copy()

print(f"MORELIA {ETIQUETA_1}:", len(morelia_corte_1))
print(f"MORELIA {ETIQUETA_2}:", len(morelia_corte_2))

print(
    f"\nDiscrepancias {ETIQUETA_1}:",
    (
        (denue_corte_1["cve_mun"] == "053")
        !=
        (denue_corte_1["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    f"Discrepancias {ETIQUETA_2}:",
    (
        (denue_corte_2["cve_mun"] == "053")
        !=
        (denue_corte_2["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2022/05: 48799
MORELIA 2022/11: 48849

Discrepancias 2022/05: 0
Discrepancias 2022/11: 0


In [ ]:
# ============================================================
# PASO 8. COMPARAR ID
# ============================================================

ids_corte_1 = set(
    morelia_corte_1["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_corte_2 = set(
    morelia_corte_2["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"Registros {ETIQUETA_1}:", len(morelia_corte_1))
print("ID únicos:", len(ids_corte_1))

print(f"\nRegistros {ETIQUETA_2}:", len(morelia_corte_2))
print("ID únicos:", len(ids_corte_2))

print("\nID comunes:", len(ids_corte_1 & ids_corte_2))
print(f"Solo {ETIQUETA_1}:", len(ids_corte_1 - ids_corte_2))
print(f"Solo {ETIQUETA_2}:", len(ids_corte_2 - ids_corte_1))

Registros 2022/05: 48799
ID únicos: 48799

Registros 2022/11: 48849
ID únicos: 48849

ID comunes: 48788
Solo 2022/05: 11
Solo 2022/11: 61


In [ ]:
# ============================================================
# PASO 9. REVISAR FECHA_ALTA
# ============================================================

print(f"FECHA ALTA {ETIQUETA_1}")
display(
    morelia_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA {ETIQUETA_2}")
display(
    morelia_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2022/05


fecha_alta
2019-11    19444
2010-07    17865
2014-12     8407
2020-11      924
2019-04      266
2018-03      259
2021-05      236
2016-01      228
2013-07      216
2011-03      214
2020-04      172
2021-11      141
2012-06      129
2016-10      114
2017-11       81
2017-03       53
2018-11       36
2013 07        7
2017 03        5
2022-05        2
Name: count, dtype: int64


FECHA ALTA 2022/11


fecha_alta
2019-11    19443
2010-07    17867
2014-12     8406
2020-11      926
2019-04      264
2018-03      259
2021-05      236
2016-01      229
2013-07      216
2011-03      215
2020-04      172
2021-11      141
2012-06      129
2016-10      113
2017-11       80
2017-03       54
2022-11       49
2018-11       36
2013 07        7
2017 03        5
2022-05        2
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 10. REGISTROS EXCLUSIVOS Y SUS FECHAS DE ALTA
# ============================================================

solo_corte_1 = morelia_corte_1[
    morelia_corte_1["id"].astype(str).isin(
        ids_corte_1 - ids_corte_2
    )
].copy()

solo_corte_2 = morelia_corte_2[
    morelia_corte_2["id"].astype(str).isin(
        ids_corte_2 - ids_corte_1
    )
].copy()

print(f"Solo {ETIQUETA_1}:", len(solo_corte_1))
print(f"Solo {ETIQUETA_2}:", len(solo_corte_2))

print(f"\nFECHA ALTA - SOLO {ETIQUETA_1}")
display(
    solo_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA - SOLO {ETIQUETA_2}")
display(
    solo_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

Solo 2022/05: 11
Solo 2022/11: 61

FECHA ALTA - SOLO 2022/05


fecha_alta
2010-07    3
2019-11    3
2019-04    2
2016-10    1
2014-12    1
2017-11    1
Name: count, dtype: int64


FECHA ALTA - SOLO 2022/11


fecha_alta
2022-11    49
2010-07     5
2019-11     2
2020-11     2
2011-03     1
2017-03     1
2016-01     1
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 11. HUELLA COMPUESTA
# ============================================================

campos_huella = [
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_generica(df):
    copia = df.copy()

    for campo in campos_huella:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella
    ]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

corte_1_h = crear_huella_generica(solo_corte_1)
corte_2_h = crear_huella_generica(solo_corte_2)

frecuencias_corte_1 = (
    corte_1_h["HUELLA"]
    .value_counts()
    .rename("n_corte_1")
)

frecuencias_corte_2 = (
    corte_2_h["HUELLA"]
    .value_counts()
    .rename("n_corte_2")
)

comparacion_huellas = pd.concat(
    [frecuencias_corte_1, frecuencias_corte_2],
    axis=1
).fillna(0).astype(int)

comparacion_huellas["coincidencias_posibles"] = (
    comparacion_huellas[
        ["n_corte_1", "n_corte_2"]
    ].min(axis=1)
)

print(
    "Huellas distintas presentes en ambos:",
    (
        (comparacion_huellas["n_corte_1"] > 0)
        &
        (comparacion_huellas["n_corte_2"] > 0)
    ).sum()
)

print(
    "Máximo de coincidencias exactas uno-a-uno:",
    comparacion_huellas["coincidencias_posibles"].sum()
)

Huellas distintas presentes en ambos: 0
Máximo de coincidencias exactas uno-a-uno: 0


In [ ]:
# ============================================================
# PASO 12. COMPARAR DISTRIBUCIÓN SCIAN
# ============================================================

conteo_scian_corte_1 = (
    morelia_corte_1["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

conteo_scian_corte_2 = (
    morelia_corte_2["codigo_act"]
    .astype(str)
    .str.strip()
    .value_counts()
    .sort_index()
)

scian_corte_1 = set(conteo_scian_corte_1.index)
scian_corte_2 = set(conteo_scian_corte_2.index)

print(f"Clases SCIAN {ETIQUETA_1}:", len(scian_corte_1))
print(f"Clases SCIAN {ETIQUETA_2}:", len(scian_corte_2))

print("\nClases comunes:", len(scian_corte_1 & scian_corte_2))
print(f"Solo {ETIQUETA_1}:", len(scian_corte_1 - scian_corte_2))
print(f"Solo {ETIQUETA_2}:", len(scian_corte_2 - scian_corte_1))

comparacion_scian = pd.concat(
    [
        conteo_scian_corte_1.rename(ETIQUETA_1),
        conteo_scian_corte_2.rename(ETIQUETA_2)
    ],
    axis=1
).fillna(0).astype(int)

comparacion_scian["Diferencia"] = (
    comparacion_scian[ETIQUETA_2]
    - comparacion_scian[ETIQUETA_1]
)

diferencias_scian = comparacion_scian[
    comparacion_scian["Diferencia"] != 0
]

print(
    "\nClases cuyo número de registros cambió:",
    len(diferencias_scian)
)

Clases SCIAN 2022/05: 689
Clases SCIAN 2022/11: 690

Clases comunes: 688
Solo 2022/05: 1
Solo 2022/11: 2

Clases cuyo número de registros cambió: 55


In [ ]:
# ============================================================
# PASO 13. SELECCIONAR CORTE ANUAL
# ============================================================

CORTE_SELECCIONADO = ETIQUETA_2
morelia_seleccionado = morelia_corte_2.copy()

print("Año:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)
print("Registros Morelia:", len(morelia_seleccionado))

archivo_salida = MORELIA / f"DENUE_{ANIO}_MORELIA.csv"

morelia_seleccionado.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Archivo Morelia guardado:")
print(archivo_salida)

Año: 2022
Corte seleccionado: 2022/11
Registros Morelia: 48849

✓ Archivo Morelia guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2022_MORELIA.csv


In [ ]:
# ============================================================
# PASO 14. CONTEO POR CLASE SCIAN
# ============================================================

conteo_scian_anual = (
    morelia_seleccionado
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_seleccionado))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Total Morelia: 48849
Clases SCIAN distintas: 690


In [ ]:
# ============================================================
# PASO 15. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 17997

SERVICIOS
Clases: 328
Establecimientos: 23306

SERVICIOS CON MÁS DE 50
Clases: 77
Establecimientos: 20735


In [ ]:
# ============================================================
# PASO 16. GUARDAR PRODUCTOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Productos {ANIO} guardados.")

✓ Productos 2022 guardados.


In [ ]:
# ============================================================
# PASO 17. ACTUALIZAR RESUMEN HISTÓRICO
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),

    "Total_Morelia":
        len(morelia_seleccionado),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN": "SCIAN 2018",

    "Observacion_metodologica":
        f"Se auditaron {ETIQUETA_1} y {ETIQUETA_2}; "
        f"se seleccionó {CORTE_SELECCIONADO} como corte anual."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen actualizado hasta 2022.


In [ ]:
# ============================================================
# PASO 18. ACTUALIZAR BITÁCORA
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_1}",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == f"{ANIO}_{CORTE_2}",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Bitácora {ANIO} actualizada.")

✓ Bitácora 2022 actualizada.


### Decisión metodológica para 2022

Se auditaron las ediciones DENUE 2022/05 y DENUE 2022/11.

El corte de mayo contiene 48,799 registros correspondientes a Morelia,
mientras que el corte de noviembre contiene 48,849, lo que representa una
variación neta de 50 registros.

Se identificaron 48,788 ID comunes entre ambos cortes, 11 registros presentes
únicamente en mayo y 61 presentes únicamente en noviembre.

De los 61 registros exclusivos de noviembre, 49 presentan fecha de alta
2022-11; los restantes corresponden a cohortes de incorporación anteriores.
Los 11 registros exclusivos de mayo también pertenecen a diferentes cohortes
de incorporación.

La comparación mediante una huella compuesta de ubicación, nombre del
establecimiento y domicilio no identificó coincidencias exactas entre los
registros exclusivos de ambos cortes, por lo que no se detectaron simples
cambios de identificador entre estos grupos.

En cuanto a la clasificación económica, el corte de mayo contiene 689 clases
SCIAN y el de noviembre 690. Se identificaron 688 clases comunes, una
presente únicamente en mayo y dos únicamente en noviembre. En 55 clases
cambió el número de establecimientos registrados.

Las diferencias observadas entre ambos cortes se interpretan como cambios
en los registros del DENUE y no automáticamente como nacimientos o cierres
empresariales.

Para la serie anual se selecciona DENUE 2022/11 como corte representativo
de 2022, conservando DENUE 2022/05 como corte intraanual auditado.

# DENUE 2023

In [ ]:
# ============================================================
# DENUE 2023
# CONFIGURACIÓN - UN SOLO CORTE
# ============================================================

ANIO = "2023"
CORTE = "11"

ETIQUETA = f"{ANIO}/{CORTE}"
CARPETA_CORTE = ORIGINALES / f"{ANIO}_{CORTE}"

print("Año de trabajo:", ANIO)
print("Corte:", ETIQUETA)
print("Carpeta existe:", CARPETA_CORTE.exists())

Año de trabajo: 2023
Corte: 2023/11
Carpeta existe: True


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ZIP Y REVISAR CONTENIDO
# ============================================================

zip_anual = next(CARPETA_CORTE.glob("*.zip"))

print("ZIP:", zip_anual.name)

with zipfile.ZipFile(zip_anual, "r") as z:
    print("\nContenido:")
    for nombre in z.namelist():
        print("-", nombre)

ZIP: denue_16_1123_csv.zip

Contenido:
- conjunto_de_datos/denue_inegi_16_.csv
- diccionario_de_datos/denue_diccionario_de_datos.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 2. DETECTAR CSV PRINCIPAL
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:
    csv_anual = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print("CSV principal:")
print(csv_anual)

CSV principal:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 3. INSPECCIÓN ESTRUCTURAL
# ============================================================

for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
    try:
        with zipfile.ZipFile(zip_anual, "r") as z:
            with z.open(csv_anual) as archivo:
                muestra_anual = pd.read_csv(
                    archivo,
                    nrows=5,
                    encoding=codificacion,
                    dtype=str
                )

        encoding_anual = codificacion
        break

    except UnicodeDecodeError:
        continue

print("Codificación:", encoding_anual)
print("Columnas:", len(muestra_anual.columns))

print("\nColumnas:")
for i, c in enumerate(muestra_anual.columns, start=1):
    print(i, repr(c))

Codificación: utf-8-sig
Columnas: 42

Columnas:
1 'id'
2 'clee'
3 'nom_estab'
4 'raz_social'
5 'codigo_act'
6 'nombre_act'
7 'per_ocu'
8 'tipo_vial'
9 'nom_vial'
10 'tipo_v_e_1'
11 'nom_v_e_1'
12 'tipo_v_e_2'
13 'nom_v_e_2'
14 'tipo_v_e_3'
15 'nom_v_e_3'
16 'numero_ext'
17 'letra_ext'
18 'edificio'
19 'edificio_e'
20 'numero_int'
21 'letra_int'
22 'tipo_asent'
23 'nomb_asent'
24 'tipoCenCom'
25 'nom_CenCom'
26 'num_local'
27 'cod_postal'
28 'cve_ent'
29 'entidad'
30 'cve_mun'
31 'municipio'
32 'cve_loc'
33 'localidad'
34 'ageb'
35 'manzana'
36 'telefono'
37 'correoelec'
38 'www'
39 'tipoUniEco'
40 'latitud'
41 'longitud'
42 'fecha_alta'


In [ ]:
# ============================================================
# PASO 4. REVISAR METADATOS
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:

    archivo_meta = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".txt")
        and "metadato" in nombre.lower()
    )

    contenido = z.read(archivo_meta)

    for enc in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            texto_meta = contenido.decode(enc)
            break
        except UnicodeDecodeError:
            continue

print("===== METADATOS 2023/11 =====")

for linea in texto_meta.splitlines():

    linea_norm = normalizar_texto(linea)

    if any(
        palabra in linea_norm
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "version"
        ]
    ):
        print(linea)

===== METADATOS 2023/11 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes.


In [ ]:
# ============================================================
# PASO 5. CARGAR DENUE 2023 COMPLETO
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:
    with z.open(csv_anual) as archivo:
        denue_anual = pd.read_csv(
            archivo,
            encoding=encoding_anual,
            dtype=str
        )

print("Filas Michoacán:", len(denue_anual))
print("Columnas:", len(denue_anual.columns))

Filas Michoacán: 260596
Columnas: 42


In [ ]:
# ============================================================
# PASO 6. FILTRAR MORELIA
# ============================================================

denue_anual["cve_mun"] = (
    denue_anual["cve_mun"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

denue_anual["municipio_norm"] = (
    denue_anual["municipio"]
    .astype(str)
    .str.strip()
    .str.upper()
)

morelia_anual = denue_anual[
    denue_anual["cve_mun"] == "053"
].copy()

print("Registros Morelia 2023:", len(morelia_anual))

print(
    "Discrepancias clave 053 vs MORELIA:",
    (
        (denue_anual["cve_mun"] == "053")
        !=
        (denue_anual["municipio_norm"] == "MORELIA")
    ).sum()
)

Registros Morelia 2023: 48889
Discrepancias clave 053 vs MORELIA: 0


In [ ]:
# ============================================================
# PASO 7. AUDITORÍA BÁSICA DEL CORTE
# ============================================================

print("Registros Morelia:", len(morelia_anual))
print("ID no nulos:", morelia_anual["id"].notna().sum())
print("ID únicos:", morelia_anual["id"].nunique())

print("\nFECHA ALTA:")
display(
    morelia_anual["fecha_alta"]
    .value_counts(dropna=False)
)

Registros Morelia: 48889
ID no nulos: 48889
ID únicos: 48889

FECHA ALTA:


fecha_alta
2019-11    19360
2010-07    17761
2014-12     8336
2020-11      895
2023-11      433
2018-03      259
2019-04      249
2021-05      236
2016-01      212
2013-07      207
2011-03      204
2020-04      154
2021-11      141
2012-06      121
2016-10      106
2017-11       74
2022-11       48
2017-03       46
2018-11       33
2013 07        7
2017 03        5
2022-05        2
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 8. CONTEO SCIAN 2023
# ============================================================

conteo_scian_anual = (
    morelia_anual
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_anual))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

display(conteo_scian_anual.head(20))

Total Morelia: 48889
Clases SCIAN distintas: 695


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
273,461110,3765,461110,46
671,812110,2334,812110,81
293,463211,1490,463211,46
638,722514,1396,722514,72
63,311830,1183,311830,31
548,621211,1171,621211,62
637,722513,1075,722513,72
315,465311,951,465311,46
277,461130,854,461130,46
644,811111,797,811111,81


In [ ]:
# ============================================================
# PASO 9. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(
        sectores_servicios
    )
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual[
        "Número de establecimientos"
    ].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual[
        "Número de establecimientos"
    ].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 17996

SERVICIOS
Clases: 330
Establecimientos: 23327

SERVICIOS CON MÁS DE 50
Clases: 77
Establecimientos: 20746


In [ ]:
# ============================================================
# PASO 10. GUARDAR MORELIA 2023
# ============================================================

archivo_salida = MORELIA / f"DENUE_{ANIO}_MORELIA.csv"

morelia_anual.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("✓ DENUE 2023 Morelia guardado.")

✓ DENUE 2023 Morelia guardado.


In [ ]:
# ============================================================
# PASO 11. GUARDAR PRODUCTOS 2023
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Productos 2023 guardados.")

✓ Productos 2023 guardados.


In [ ]:
# ============================================================
# PASO 12. AGREGAR 2023 AL RESUMEN HISTÓRICO
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),

    "Total_Morelia":
        len(morelia_anual),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN":
        "SCIAN 2018",

    "Observacion_metodologica":
        "Único corte disponible para 2023: DENUE 2023/11."
}])

# Evitar duplicar 2023
resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen actualizado hasta 2023.


In [ ]:
# ============================================================
# PASO 13. ACTUALIZAR BITÁCORA 2023
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == "2023_11",
    "Estado"
] = "Procesado - único corte disponible"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora 2023 actualizada.")

✓ Bitácora 2023 actualizada.


### Tratamiento metodológico de DENUE 2023

Para 2023 se dispone únicamente de la edición DENUE 2023/11 en la serie
histórica analizada, por lo que no fue posible realizar una auditoría
intraanual entre dos cortes como en años anteriores.

El archivo contiene 48,889 registros correspondientes al municipio de
Morelia, todos con identificador no nulo y único dentro de la edición.

La variable fecha de alta registra 433 unidades con fecha 2023-11. Esta
información se interpreta como fecha de incorporación al DENUE y no
automáticamente como nacimiento empresarial.

La edición 2023/11 se utiliza como corte representativo de 2023 para la
serie histórica.

# DENUE 2024

In [ ]:
# ============================================================
# DENUE 2024
# CONFIGURACIÓN DEL AÑO
# ============================================================

ANIO = "2024"
CORTE_1 = "05"
CORTE_2 = "11"

ETIQUETA_1 = f"{ANIO}/{CORTE_1}"
ETIQUETA_2 = f"{ANIO}/{CORTE_2}"

CARPETA_CORTE_1 = ORIGINALES / f"{ANIO}_{CORTE_1}"
CARPETA_CORTE_2 = ORIGINALES / f"{ANIO}_{CORTE_2}"

print("Año de trabajo:", ANIO)
print("Corte 1:", ETIQUETA_1)
print("Corte 2:", ETIQUETA_2)

Año de trabajo: 2024
Corte 1: 2024/05
Corte 2: 2024/11


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ARCHIVOS
# ============================================================

print(f"Carpeta {ETIQUETA_1} existe:", CARPETA_CORTE_1.exists())
print(f"Carpeta {ETIQUETA_2} existe:", CARPETA_CORTE_2.exists())

print(f"\nArchivos {ETIQUETA_1}:")
for archivo in CARPETA_CORTE_1.iterdir():
    print("-", archivo.name)

print(f"\nArchivos {ETIQUETA_2}:")
for archivo in CARPETA_CORTE_2.iterdir():
    print("-", archivo.name)

Carpeta 2024/05 existe: True
Carpeta 2024/11 existe: True

Archivos 2024/05:
- denue_16_0524_csv.zip

Archivos 2024/11:
- denue_16_1124_csv.zip


In [ ]:
# ============================================================
# PASO 2. REVISAR CONTENIDO DE LOS ZIP
# ============================================================

zip_corte_1 = next(CARPETA_CORTE_1.glob("*.zip"))
zip_corte_2 = next(CARPETA_CORTE_2.glob("*.zip"))

print(f"===== {ETIQUETA_1} =====")
print("ZIP:", zip_corte_1.name)

with zipfile.ZipFile(zip_corte_1, "r") as z:
    for nombre in z.namelist():
        print("-", nombre)

print(f"\n===== {ETIQUETA_2} =====")
print("ZIP:", zip_corte_2.name)

with zipfile.ZipFile(zip_corte_2, "r") as z:
    for nombre in z.namelist():
        print("-", nombre)

===== 2024/05 =====
ZIP: denue_16_0524_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt

===== 2024/11 =====
ZIP: denue_16_1124_csv.zip
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 3. DETECTAR CSV PRINCIPAL
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    csv_corte_1 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    csv_corte_2 = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print(f"CSV principal {ETIQUETA_1}:")
print(csv_corte_1)

print(f"\nCSV principal {ETIQUETA_2}:")
print(csv_corte_2)

CSV principal 2024/05:
conjunto_de_datos/denue_inegi_16_.csv

CSV principal 2024/11:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 4. INSPECCIÓN ESTRUCTURAL
# ============================================================

def leer_muestra_zip(zip_path, csv_interno):
    for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(csv_interno) as archivo:
                    muestra = pd.read_csv(
                        archivo,
                        nrows=5,
                        encoding=codificacion,
                        dtype=str
                    )
            return muestra, codificacion
        except UnicodeDecodeError:
            continue

    raise ValueError("No se pudo detectar la codificación.")

muestra_corte_1, encoding_corte_1 = leer_muestra_zip(
    zip_corte_1,
    csv_corte_1
)

muestra_corte_2, encoding_corte_2 = leer_muestra_zip(
    zip_corte_2,
    csv_corte_2
)

print(ETIQUETA_1)
print("Codificación:", encoding_corte_1)
print("Columnas:", len(muestra_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Codificación:", encoding_corte_2)
print("Columnas:", len(muestra_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(muestra_corte_1.columns)
    ==
    list(muestra_corte_2.columns)
)

2024/05
Codificación: latin1
Columnas: 42

2024/11
Codificación: latin1
Columnas: 42

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 5. CARGAR AMBOS CORTES COMPLETOS
# ============================================================

with zipfile.ZipFile(zip_corte_1, "r") as z:
    with z.open(csv_corte_1) as archivo:
        denue_corte_1 = pd.read_csv(
            archivo,
            encoding=encoding_corte_1,
            dtype=str
        )

with zipfile.ZipFile(zip_corte_2, "r") as z:
    with z.open(csv_corte_2) as archivo:
        denue_corte_2 = pd.read_csv(
            archivo,
            encoding=encoding_corte_2,
            dtype=str
        )

print(ETIQUETA_1)
print("Filas:", len(denue_corte_1))
print("Columnas:", len(denue_corte_1.columns))

print("\n" + ETIQUETA_2)
print("Filas:", len(denue_corte_2))
print("Columnas:", len(denue_corte_2.columns))

print(
    "\n¿Mismas columnas exactamente?:",
    list(denue_corte_1.columns)
    ==
    list(denue_corte_2.columns)
)


2024/05
Filas: 261255
Columnas: 42

2024/11
Filas: 285324
Columnas: 42

¿Mismas columnas exactamente?: True


In [ ]:
# ============================================================
# PASO 6. REVISAR METADATOS
# ============================================================

def leer_metadatos(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:

        candidatos = [
            nombre for nombre in z.namelist()
            if nombre.lower().endswith(".txt")
            and "metadato" in nombre.lower()
        ]

        if not candidatos:
            return "No se encontró archivo de metadatos."

        contenido = z.read(candidatos[0])

        for encoding in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                return contenido.decode(encoding)
            except UnicodeDecodeError:
                continue

    return ""

meta_corte_1 = leer_metadatos(zip_corte_1)
meta_corte_2 = leer_metadatos(zip_corte_2)

print(f"===== {ETIQUETA_1} =====")
for linea in meta_corte_1.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "version"
        ]
    ):
        print(linea)

print(f"\n===== {ETIQUETA_2} =====")
for linea in meta_corte_2.splitlines():
    if any(
        palabra in normalizar_texto(linea)
        for palabra in [
            "description",
            "actualiz",
            "censo",
            "operativo",
            "scian",
            "version"
        ]
    ):
        print(linea)

===== 2024/05 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes.

===== 2024/11 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes.


In [ ]:
# ============================================================
# PASO 7. FILTRAR MORELIA
# ============================================================

for df in [denue_corte_1, denue_corte_2]:

    df["cve_mun"] = (
        df["cve_mun"]
        .astype(str)
        .str.strip()
        .str.zfill(3)
    )

    df["municipio_norm"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

morelia_corte_1 = denue_corte_1[
    denue_corte_1["cve_mun"] == "053"
].copy()

morelia_corte_2 = denue_corte_2[
    denue_corte_2["cve_mun"] == "053"
].copy()

print(f"MORELIA {ETIQUETA_1}:", len(morelia_corte_1))
print(f"MORELIA {ETIQUETA_2}:", len(morelia_corte_2))

print(
    f"\nDiscrepancias {ETIQUETA_1}:",
    (
        (denue_corte_1["cve_mun"] == "053")
        !=
        (denue_corte_1["municipio_norm"] == "MORELIA")
    ).sum()
)

print(
    f"Discrepancias {ETIQUETA_2}:",
    (
        (denue_corte_2["cve_mun"] == "053")
        !=
        (denue_corte_2["municipio_norm"] == "MORELIA")
    ).sum()
)

MORELIA 2024/05: 49043
MORELIA 2024/11: 52441

Discrepancias 2024/05: 0
Discrepancias 2024/11: 0


In [ ]:
# ============================================================
# PASO 8. COMPARAR ID
# ============================================================

ids_corte_1 = set(
    morelia_corte_1["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

ids_corte_2 = set(
    morelia_corte_2["id"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"Registros {ETIQUETA_1}:", len(morelia_corte_1))
print("ID únicos:", len(ids_corte_1))

print(f"\nRegistros {ETIQUETA_2}:", len(morelia_corte_2))
print("ID únicos:", len(ids_corte_2))

print("\nID comunes:", len(ids_corte_1 & ids_corte_2))
print(f"Solo {ETIQUETA_1}:", len(ids_corte_1 - ids_corte_2))
print(f"Solo {ETIQUETA_2}:", len(ids_corte_2 - ids_corte_1))

Registros 2024/05: 49043
ID únicos: 49043

Registros 2024/11: 52441
ID únicos: 52441

ID comunes: 33307
Solo 2024/05: 15736
Solo 2024/11: 19134


In [ ]:
# ============================================================
# PASO 9. REVISAR FECHA_ALTA
# ============================================================

print(f"FECHA ALTA {ETIQUETA_1}")
display(
    morelia_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA {ETIQUETA_2}")
display(
    morelia_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

FECHA ALTA 2024/05


fecha_alta
2019-11    19351
2010-07    17750
2014-12     8331
2020-11      886
2023-11      431
2018-03      257
2019-04      243
2021-05      235
2016-01      212
2024-05      210
2013-07      207
2011-03      201
2020-04      153
2021-11      141
2012-06      120
2016-10      104
2017-11       73
2022-11       47
2017-03       45
2018-11       32
2013 07        7
2017 03        5
2022-05        2
Name: count, dtype: int64


FECHA ALTA 2024/11


fecha_alta
2024-11    19141
2010-07    14201
2019-11    10410
2014-12     5952
2020-11      529
2023-11      347
2018-03      257
2019-04      233
2011-03      180
2016-01      177
2013-07      154
2024-05      144
2021-05      138
2020-04      129
2012-06      100
2016-10       82
2021-11       81
2017-11       63
2017-03       44
2022-11       37
2018-11       32
2013 07        5
2017 03        3
2022-05        2
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 10. REGISTROS EXCLUSIVOS Y FECHA_ALTA
# ============================================================

solo_corte_1 = morelia_corte_1[
    morelia_corte_1["id"].astype(str).isin(
        ids_corte_1 - ids_corte_2
    )
].copy()

solo_corte_2 = morelia_corte_2[
    morelia_corte_2["id"].astype(str).isin(
        ids_corte_2 - ids_corte_1
    )
].copy()

print(f"Solo {ETIQUETA_1}:", len(solo_corte_1))
print(f"Solo {ETIQUETA_2}:", len(solo_corte_2))

print(f"\nFECHA ALTA - SOLO {ETIQUETA_1}")
display(
    solo_corte_1["fecha_alta"]
    .value_counts(dropna=False)
)

print(f"\nFECHA ALTA - SOLO {ETIQUETA_2}")
display(
    solo_corte_2["fecha_alta"]
    .value_counts(dropna=False)
)

Solo 2024/05: 15736
Solo 2024/11: 19134

FECHA ALTA - SOLO 2024/05


fecha_alta
2019-11    8940
2010-07    3546
2014-12    2379
2020-11     357
2021-05      97
2023-11      84
2024-05      63
2021-11      60
2013-07      53
2016-01      35
2020-04      24
2016-10      22
2011-03      21
2012-06      20
2019-04      10
2017-11      10
2022-11      10
2013 07       2
2017 03       2
2017-03       1
Name: count, dtype: int64


FECHA ALTA - SOLO 2024/11


fecha_alta
2024-11    19125
2019-11        8
2014-12        1
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 11. HUELLA COMPUESTA
# ============================================================

campos_huella = [
    "latitud",
    "longitud",
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "localidad"
]

def crear_huella_generica(df):
    copia = df.copy()

    for campo in campos_huella:
        copia[campo + "_norm"] = (
            copia[campo].apply(normalizar_valor)
        )

    columnas_norm = [
        campo + "_norm"
        for campo in campos_huella
    ]

    copia["HUELLA"] = (
        copia[columnas_norm]
        .agg("|".join, axis=1)
    )

    return copia

corte_1_h = crear_huella_generica(solo_corte_1)
corte_2_h = crear_huella_generica(solo_corte_2)

frecuencias_corte_1 = (
    corte_1_h["HUELLA"]
    .value_counts()
    .rename("n_corte_1")
)

frecuencias_corte_2 = (
    corte_2_h["HUELLA"]
    .value_counts()
    .rename("n_corte_2")
)

comparacion_huellas = pd.concat(
    [frecuencias_corte_1, frecuencias_corte_2],
    axis=1
).fillna(0).astype(int)

comparacion_huellas["coincidencias_posibles"] = (
    comparacion_huellas[
        ["n_corte_1", "n_corte_2"]
    ].min(axis=1)
)

print(
    "Huellas distintas presentes en ambos:",
    (
        (comparacion_huellas["n_corte_1"] > 0)
        &
        (comparacion_huellas["n_corte_2"] > 0)
    ).sum()
)

print(
    "Máximo de coincidencias exactas uno-a-uno:",
    comparacion_huellas[
        "coincidencias_posibles"
    ].sum()
)

Huellas distintas presentes en ambos: 0
Máximo de coincidencias exactas uno-a-uno: 0


In [ ]:
# ============================================================
# PASO 12. CAMBIO SCIAN 2018 -> SCIAN 2023
# EN ID PERSISTENTES
# ============================================================

ids_comunes = ids_corte_1 & ids_corte_2

comunes_05 = (
    morelia_corte_1[
        morelia_corte_1["id"].astype(str).isin(ids_comunes)
    ][["id", "codigo_act"]]
    .copy()
)

comunes_11 = (
    morelia_corte_2[
        morelia_corte_2["id"].astype(str).isin(ids_comunes)
    ][["id", "codigo_act"]]
    .copy()
)

comparacion_codigos_2024 = pd.merge(
    comunes_05,
    comunes_11,
    on="id",
    suffixes=("_SCIAN2018", "_SCIAN2023")
)

comparacion_codigos_2024["Cambio_codigo"] = (
    comparacion_codigos_2024["codigo_act_SCIAN2018"].astype(str)
    !=
    comparacion_codigos_2024["codigo_act_SCIAN2023"].astype(str)
)

print("ID comunes:", len(comparacion_codigos_2024))

print(
    "Con mismo código:",
    (~comparacion_codigos_2024["Cambio_codigo"]).sum()
)

print(
    "Con código diferente:",
    comparacion_codigos_2024["Cambio_codigo"].sum()
)

ID comunes: 33307
Con mismo código: 31247
Con código diferente: 2060


### Decisión metodológica para 2024

Se auditaron las ediciones DENUE 2024/05 y DENUE 2024/11.

El corte de mayo contiene 49,043 registros correspondientes a Morelia,
mientras que el corte de noviembre contiene 52,441, lo que representa una
variación neta de 3,398 registros.

Se identificaron 33,307 ID comunes entre ambos cortes, 15,736 registros
presentes únicamente en mayo y 19,134 presentes únicamente en noviembre.

De los 19,134 registros exclusivos de noviembre, 19,125 presentan fecha de
alta 2024-11. Estas altas corresponden a incorporaciones al DENUE y no se
interpretan automáticamente como nacimientos empresariales.

La comparación mediante una huella compuesta de ubicación, nombre del
establecimiento y domicilio no identificó coincidencias exactas entre los
registros exclusivos de ambos cortes, por lo que no se detectaron simples
cambios de identificador entre estos grupos.

Entre los 33,307 registros que conservaron el mismo ID, 31,247 conservaron
también el mismo código de actividad económica y 2,060 presentaron un código
diferente.

La edición 2024/05 utiliza SCIAN 2018, mientras que la edición 2024/11
corresponde a SCIAN 2023. Por ello, los cambios de código observados no se
interpretan automáticamente como cambios reales en la actividad económica
del establecimiento. Su homologación se realizará posteriormente mediante
las tablas oficiales de equivalencia entre SCIAN 2018 y SCIAN 2023.

Por tratarse además de una actualización amplia vinculada con los Censos
Económicos 2024, las diferencias entre ambos cortes no se utilizan de manera
directa para inferir nacimientos, cierres o permanencia empresarial.

Para la serie anual se selecciona DENUE 2024/11 como corte representativo
de 2024, conservando DENUE 2024/05 como corte intraanual auditado.

In [ ]:
# ============================================================
# PASO 13. SELECCIONAR CORTE ANUAL 2024
# ============================================================

CORTE_SELECCIONADO = ETIQUETA_2
morelia_seleccionado = morelia_corte_2.copy()

print("Año:", ANIO)
print("Corte seleccionado:", CORTE_SELECCIONADO)
print("Registros Morelia:", len(morelia_seleccionado))

archivo_salida = MORELIA / f"DENUE_{ANIO}_MORELIA.csv"

morelia_seleccionado.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Archivo Morelia guardado:")
print(archivo_salida)

Año: 2024
Corte seleccionado: 2024/11
Registros Morelia: 52441

✓ Archivo Morelia guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2024_MORELIA.csv


In [ ]:
# ============================================================
# PASO 14. CONTEO POR CLASE SCIAN 2023
# ============================================================

conteo_scian_anual = (
    morelia_seleccionado
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_seleccionado))

print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

display(conteo_scian_anual.head(20))

Total Morelia: 52441
Clases SCIAN distintas: 710


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
291,461110,3855,461110,46
684,812110,3128,812110,81
311,463211,1557,463211,46
651,722514,1501,722514,72
560,621211,1284,621211,62
650,722513,1181,722513,72
70,311830,1073,311830,31
333,465311,977,465311,46
657,811111,927,811111,81
295,461130,874,461130,46


In [ ]:
# ============================================================
# PASO 15. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(sectores_servicios)
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 19761

SERVICIOS
Clases: 325
Establecimientos: 25170

SERVICIOS CON MÁS DE 50
Clases: 76
Establecimientos: 22596


In [ ]:
print(
    "Clases SCIAN totales 2024/11:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Clases SCIAN totales 2024/11: 710


In [ ]:
# ============================================================
# PASO 16. GUARDAR PRODUCTOS 2024
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Productos {ANIO} guardados.")

✓ Productos 2024 guardados.


In [ ]:
# ============================================================
# PASO 17. ACTUALIZAR RESUMEN HISTÓRICO 2024
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),
    "Total_Morelia": len(morelia_seleccionado),
    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),
    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),
    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),
    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),
    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),
    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),
    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),
    "Version_SCIAN": "SCIAN 2023",
    "Observacion_metodologica":
        "Se auditaron 2024/05 y 2024/11; se seleccionó "
        "2024/11. Actualización amplia y cambio de SCIAN 2018 a SCIAN 2023."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen actualizado hasta 2024.


In [ ]:
# ============================================================
# PASO 18. ACTUALIZAR BITÁCORA 2024
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == "2024_05",
    "Estado"
] = "Auditado - no seleccionado"

cortes_denue.loc[
    cortes_denue["Corte"] == "2024_11",
    "Estado"
] = "Procesado - corte seleccionado"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora 2024 actualizada.")

✓ Bitácora 2024 actualizada.


# DENUE 2025

In [ ]:
# ============================================================
# DENUE 2025
# CONFIGURACIÓN DEL AÑO - UN SOLO CORTE
# ============================================================

ANIO = "2025"
CORTE = "05"

ETIQUETA = f"{ANIO}/{CORTE}"
CARPETA_CORTE = ORIGINALES / f"{ANIO}_{CORTE}"

print("Año de trabajo:", ANIO)
print("Corte:", ETIQUETA)
print("Carpeta existe:", CARPETA_CORTE.exists())

Año de trabajo: 2025
Corte: 2025/05
Carpeta existe: True


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ZIP Y REVISAR CONTENIDO
# ============================================================

zip_anual = next(CARPETA_CORTE.glob("*.zip"))

print("ZIP:", zip_anual.name)

with zipfile.ZipFile(zip_anual, "r") as z:
    print("\nContenido:")
    for nombre in z.namelist():
        print("-", nombre)

ZIP: denue_16_0525_csv.zip

Contenido:
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 2. DETECTAR CSV PRINCIPAL
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:
    csv_anual = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print("CSV principal:")
print(csv_anual)

CSV principal:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 3. INSPECCIÓN ESTRUCTURAL
# ============================================================

for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
    try:
        with zipfile.ZipFile(zip_anual, "r") as z:
            with z.open(csv_anual) as archivo:
                muestra_anual = pd.read_csv(
                    archivo,
                    nrows=5,
                    encoding=codificacion,
                    dtype=str
                )

        encoding_anual = codificacion
        break

    except UnicodeDecodeError:
        continue

print("Codificación:", encoding_anual)
print("Columnas:", len(muestra_anual.columns))

print("\nColumnas:")
for i, c in enumerate(muestra_anual.columns, start=1):
    print(i, repr(c))

Codificación: latin1
Columnas: 42

Columnas:
1 'id'
2 'clee'
3 'nom_estab'
4 'raz_social'
5 'codigo_act'
6 'nombre_act'
7 'per_ocu'
8 'tipo_vial'
9 'nom_vial'
10 'tipo_v_e_1'
11 'nom_v_e_1'
12 'tipo_v_e_2'
13 'nom_v_e_2'
14 'tipo_v_e_3'
15 'nom_v_e_3'
16 'numero_ext'
17 'letra_ext'
18 'edificio'
19 'edificio_e'
20 'numero_int'
21 'letra_int'
22 'tipo_asent'
23 'nomb_asent'
24 'tipoCenCom'
25 'nom_CenCom'
26 'num_local'
27 'cod_postal'
28 'cve_ent'
29 'entidad'
30 'cve_mun'
31 'municipio'
32 'cve_loc'
33 'localidad'
34 'ageb'
35 'manzana'
36 'telefono'
37 'correoelec'
38 'www'
39 'tipoUniEco'
40 'latitud'
41 'longitud'
42 'fecha_alta'


In [ ]:
# ============================================================
# PASO 4. REVISAR METADATOS
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:

    candidatos_meta = [
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".txt")
        and "metadato" in nombre.lower()
    ]

    if candidatos_meta:

        contenido = z.read(candidatos_meta[0])

        for enc in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                texto_meta = contenido.decode(enc)
                break
            except UnicodeDecodeError:
                continue

        print(f"===== METADATOS {ETIQUETA} =====")

        for linea in texto_meta.splitlines():

            linea_norm = normalizar_texto(linea)

            if any(
                palabra in linea_norm
                for palabra in [
                    "description",
                    "actualiz",
                    "censo",
                    "operativo",
                    "scian",
                    "version"
                ]
            ):
                print(linea)

    else:
        print("No se encontró archivo de metadatos TXT.")

===== METADATOS 2025/05 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes.


In [ ]:
# ============================================================
# PASO 5. CARGAR DENUE COMPLETO
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:
    with z.open(csv_anual) as archivo:
        denue_anual = pd.read_csv(
            archivo,
            encoding=encoding_anual,
            dtype=str
        )

print("Filas Michoacán:", len(denue_anual))
print("Columnas:", len(denue_anual.columns))

Filas Michoacán: 285392
Columnas: 42


In [ ]:
# ============================================================
# PASO 6. FILTRAR MORELIA
# ============================================================

denue_anual["cve_mun"] = (
    denue_anual["cve_mun"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

denue_anual["municipio_norm"] = (
    denue_anual["municipio"]
    .astype(str)
    .str.strip()
    .str.upper()
)

morelia_anual = denue_anual[
    denue_anual["cve_mun"] == "053"
].copy()

print("Registros Morelia:", len(morelia_anual))

print(
    "Discrepancias clave 053 vs MORELIA:",
    (
        (denue_anual["cve_mun"] == "053")
        !=
        (denue_anual["municipio_norm"] == "MORELIA")
    ).sum()
)

Registros Morelia: 52775
Discrepancias clave 053 vs MORELIA: 0


In [ ]:
# ============================================================
# PASO 7. AUDITORÍA BÁSICA
# ============================================================

print("Registros Morelia:", len(morelia_anual))
print("ID no nulos:", morelia_anual["id"].notna().sum())
print("ID únicos:", morelia_anual["id"].nunique())

print("\nFECHA ALTA:")
display(
    morelia_anual["fecha_alta"]
    .value_counts(dropna=False)
)

Registros Morelia: 52775
ID no nulos: 52775
ID únicos: 52775

FECHA ALTA:


fecha_alta
2024-11    18877
2010-07    14022
2019-11    10178
2014-12     5840
2025-04     1380
2020-11      430
2023-11      317
2018-03      257
2019-04      235
2016-01      153
2011-03      150
2013-07      148
2021-05      129
2024-05      123
2020-04      106
2012-06       95
2021-11       82
2016-10       77
2017-11       60
2017-03       42
2022-11       30
2018-11       30
2025-05        6
2013 07        4
2017 03        2
2022-05        2
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 8. CONTEO POR CLASE SCIAN
# ============================================================

conteo_scian_anual = (
    morelia_anual
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_anual))
print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

display(conteo_scian_anual.head(20))

Total Morelia: 52775
Clases SCIAN distintas: 697


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
283,461110,4109,461110,46
671,812110,3211,812110,81
303,463211,1604,463211,46
638,722514,1501,722514,72
547,621211,1324,621211,62
637,722513,1225,722513,72
67,311830,1078,311830,31
325,465311,973,465311,46
644,811111,920,811111,81
287,461130,887,461130,46


In [ ]:
# ============================================================
# PASO 9. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(
        sectores_servicios
    )
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 19994

SERVICIOS
Clases: 320
Establecimientos: 25359

SERVICIOS CON MÁS DE 50
Clases: 73
Establecimientos: 22748


In [ ]:
# ============================================================
# PASO 10. GUARDAR MORELIA 2025
# ============================================================

archivo_salida = MORELIA / f"DENUE_{ANIO}_MORELIA.csv"

morelia_anual.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ DENUE {ANIO} Morelia guardado.")

✓ DENUE 2025 Morelia guardado.


In [ ]:
# ============================================================
# PASO 11. GUARDAR PRODUCTOS DEL AÑO
# ============================================================

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Productos {ANIO} guardados.")

✓ Productos 2025 guardados.


In [ ]:
# ============================================================
# PASO 12. ACTUALIZAR RESUMEN HISTÓRICO
# ============================================================

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),

    "Total_Morelia":
        len(morelia_anual),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN":
        "SCIAN 2023",

    "Observacion_metodologica":
        "Único corte disponible para 2025: DENUE 2025/05."
}])

resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen actualizado hasta 2025.


In [ ]:
# ============================================================
# PASO 13. ACTUALIZAR BITÁCORA
# ============================================================

cortes_denue.loc[
    cortes_denue["Corte"] == "2025_05",
    "Estado"
] = "Procesado - único corte disponible"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora 2025 actualizada.")

✓ Bitácora 2025 actualizada.


### Tratamiento metodológico de DENUE 2025

Para 2025 se dispone únicamente de la edición DENUE 2025/05 dentro de la
serie histórica analizada. Por esta razón, no fue posible realizar una
auditoría intraanual entre dos cortes como en los años que cuentan con dos
ediciones.

Se verificó la estructura del archivo, la identificación territorial de los
establecimientos correspondientes al municipio de Morelia, la unicidad de
los identificadores y las fechas de incorporación registradas en el DENUE.

La edición 2025/05 utiliza SCIAN 2023, por lo que sus clases de actividad son
comparables dentro de esta misma versión del clasificador. Las comparaciones
con años clasificados mediante versiones anteriores del SCIAN requerirán la
homologación correspondiente.

Las fechas de alta registradas en el archivo se interpretan como fechas de
incorporación de las unidades económicas al DENUE y no automáticamente como
fechas de nacimiento empresarial.

Para el análisis anual se utiliza DENUE 2025/05 como corte representativo de
2025, al ser la única edición disponible para ese año en la serie consultada.

# DENUE 2026    

In [ ]:
# ============================================================
# DENUE 2026
# CONFIGURACIÓN DEL AÑO - UN SOLO CORTE
# ============================================================

ANIO = "2026"
CORTE = "05"

ETIQUETA = f"{ANIO}/{CORTE}"
CARPETA_CORTE = ORIGINALES / f"{ANIO}_{CORTE}"

print("Año de trabajo:", ANIO)
print("Corte:", ETIQUETA)
print("Carpeta existe:", CARPETA_CORTE.exists())

Año de trabajo: 2026
Corte: 2026/05
Carpeta existe: True


In [ ]:
# ============================================================
# PASO 1. LOCALIZAR ZIP Y REVISAR CONTENIDO
# ============================================================

zip_anual = next(CARPETA_CORTE.glob("*.zip"))

print("ZIP:", zip_anual.name)

with zipfile.ZipFile(zip_anual, "r") as z:
    print("\nContenido:")
    for nombre in z.namelist():
        print("-", nombre)

ZIP: denue_16_csv.zip

Contenido:
- diccionario_de_datos/denue_diccionario_de_datos.csv
- conjunto_de_datos/denue_inegi_16_.csv
- metadatos/metadatos_denue.txt


In [ ]:
# ============================================================
# PASO 2. DETECTAR CSV PRINCIPAL
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:
    csv_anual = next(
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".csv")
        and "diccionario" not in nombre.lower()
    )

print("CSV principal:")
print(csv_anual)

CSV principal:
conjunto_de_datos/denue_inegi_16_.csv


In [ ]:
# ============================================================
# PASO 3. INSPECCIÓN ESTRUCTURAL
# ============================================================

for codificacion in ["utf-8-sig", "utf-8", "latin1"]:
    try:
        with zipfile.ZipFile(zip_anual, "r") as z:
            with z.open(csv_anual) as archivo:
                muestra_anual = pd.read_csv(
                    archivo,
                    nrows=5,
                    encoding=codificacion,
                    dtype=str
                )

        encoding_anual = codificacion
        break

    except UnicodeDecodeError:
        continue

print("Codificación:", encoding_anual)
print("Columnas:", len(muestra_anual.columns))

print("\nColumnas:")
for i, c in enumerate(muestra_anual.columns, start=1):
    print(i, repr(c))

Codificación: latin1
Columnas: 42

Columnas:
1 'id'
2 'clee'
3 'nom_estab'
4 'raz_social'
5 'codigo_act'
6 'nombre_act'
7 'per_ocu'
8 'tipo_vial'
9 'nom_vial'
10 'tipo_v_e_1'
11 'nom_v_e_1'
12 'tipo_v_e_2'
13 'nom_v_e_2'
14 'tipo_v_e_3'
15 'nom_v_e_3'
16 'numero_ext'
17 'letra_ext'
18 'edificio'
19 'edificio_e'
20 'numero_int'
21 'letra_int'
22 'tipo_asent'
23 'nomb_asent'
24 'tipoCenCom'
25 'nom_CenCom'
26 'num_local'
27 'cod_postal'
28 'cve_ent'
29 'entidad'
30 'cve_mun'
31 'municipio'
32 'cve_loc'
33 'localidad'
34 'ageb'
35 'manzana'
36 'telefono'
37 'correoelec'
38 'www'
39 'tipoUniEco'
40 'latitud'
41 'longitud'
42 'fecha_alta'


In [ ]:
# ============================================================
# PASO 4. REVISAR METADATOS
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:

    candidatos_meta = [
        nombre for nombre in z.namelist()
        if nombre.lower().endswith(".txt")
        and "metadato" in nombre.lower()
    ]

    if candidatos_meta:

        contenido = z.read(candidatos_meta[0])

        for enc in ["utf-8-sig", "utf-8", "latin1"]:
            try:
                texto_meta = contenido.decode(enc)
                break
            except UnicodeDecodeError:
                continue

        print(f"===== METADATOS {ETIQUETA} =====")

        for linea in texto_meta.splitlines():

            linea_norm = normalizar_texto(linea)

            if any(
                palabra in linea_norm
                for palabra in [
                    "description",
                    "actualiz",
                    "censo",
                    "operativo",
                    "scian",
                    "version"
                ]
            ):
                print(linea)

    else:
        print("No se encontró archivo de metadatos TXT.")

===== METADATOS 2026/05 =====
Description: El Directorio Estadístico Nacional de Unidades Económicas ofrece datos de identificación, ubicación, actividad económica y tamaño de los negocios activos en el territorio nacional, actualizados, fundamentalmente, en el segmento de los establecimientos grandes.


In [ ]:
# ============================================================
# PASO 5. CARGAR DENUE COMPLETO
# ============================================================

with zipfile.ZipFile(zip_anual, "r") as z:
    with z.open(csv_anual) as archivo:
        denue_anual = pd.read_csv(
            archivo,
            encoding=encoding_anual,
            dtype=str
        )

print("Filas Michoacán:", len(denue_anual))
print("Columnas:", len(denue_anual.columns))

Filas Michoacán: 287049
Columnas: 42


In [ ]:
# ============================================================
# PASO 6. FILTRAR MORELIA
# ============================================================

denue_anual["cve_mun"] = (
    denue_anual["cve_mun"]
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

denue_anual["municipio_norm"] = (
    denue_anual["municipio"]
    .astype(str)
    .str.strip()
    .str.upper()
)

morelia_anual = denue_anual[
    denue_anual["cve_mun"] == "053"
].copy()

print("Registros Morelia:", len(morelia_anual))

print(
    "Discrepancias clave 053 vs MORELIA:",
    (
        (denue_anual["cve_mun"] == "053")
        !=
        (denue_anual["municipio_norm"] == "MORELIA")
    ).sum()
)

Registros Morelia: 53208
Discrepancias clave 053 vs MORELIA: 0


In [ ]:
# ============================================================
# PASO 7. AUDITORÍA BÁSICA
# ============================================================

print("Registros Morelia:", len(morelia_anual))
print("ID no nulos:", morelia_anual["id"].notna().sum())
print("ID únicos:", morelia_anual["id"].nunique())

print("\nFECHA ALTA:")
display(
    morelia_anual["fecha_alta"]
    .value_counts(dropna=False)
)

Registros Morelia: 53208
ID no nulos: 53208
ID únicos: 53208

FECHA ALTA:


fecha_alta
2024-11    18875
2010-07    14001
2019-11    10168
2014-12     5822
2025-04     1375
2026-04      896
2020-11      426
2023-11      316
2016-01      154
2013-07      147
2011-03      142
2021-05      129
2024-05      122
2020-04      106
2012-06       95
2019-04       94
2021-11       82
2016-10       75
2017-11       60
2017-03       42
2018-11       30
2022-11       28
2018-03        9
2025-05        6
2013 07        4
2017 03        2
2022-05        2
Name: count, dtype: int64

In [ ]:
# ============================================================
# PASO 8. CONTEO POR CLASE SCIAN
# ============================================================

conteo_scian_anual = (
    morelia_anual
    .groupby("codigo_act", dropna=False)
    .size()
    .reset_index(name="Número de establecimientos")
)

conteo_scian_anual["SCIAN6"] = (
    conteo_scian_anual["codigo_act"]
    .astype(str)
    .str.strip()
)

conteo_scian_anual["Sector SCIAN"] = (
    conteo_scian_anual["SCIAN6"].str[:2]
)

conteo_scian_anual = conteo_scian_anual.sort_values(
    "Número de establecimientos",
    ascending=False
)

print("Total Morelia:", len(morelia_anual))

print(
    "Clases SCIAN distintas:",
    conteo_scian_anual["SCIAN6"].nunique()
)

display(conteo_scian_anual.head(20))

Total Morelia: 53208
Clases SCIAN distintas: 706


,codigo_act,Número de establecimientos,SCIAN6,Sector SCIAN
283,461110,4142,461110,46
680,812110,3218,812110,81
303,463211,1628,463211,46
647,722514,1505,722514,72
557,621211,1384,621211,62
646,722513,1228,722513,72
67,311830,1080,311830,31
325,465311,977,465311,46
653,811111,924,811111,81
287,461130,895,461130,46


In [ ]:
# ============================================================
# PASO 9. COMERCIO AL POR MENOR Y SERVICIOS
# ============================================================

comercio_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"] == "46"
].copy()

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

servicios_anual = conteo_scian_anual[
    conteo_scian_anual["Sector SCIAN"].isin(
        sectores_servicios
    )
].copy()

servicios_mayores_50_anual = servicios_anual[
    servicios_anual["Número de establecimientos"] > 50
].copy()

print("COMERCIO AL POR MENOR")
print("Clases:", comercio_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    comercio_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS")
print("Clases:", servicios_anual["SCIAN6"].nunique())
print(
    "Establecimientos:",
    servicios_anual["Número de establecimientos"].sum()
)

print("\nSERVICIOS CON MÁS DE 50")
print("Clases:", len(servicios_mayores_50_anual))
print(
    "Establecimientos:",
    servicios_mayores_50_anual[
        "Número de establecimientos"
    ].sum()
)

COMERCIO AL POR MENOR
Clases: 83
Establecimientos: 20292

SERVICIOS
Clases: 329
Establecimientos: 25383

SERVICIOS CON MÁS DE 50
Clases: 77
Establecimientos: 22811


In [ ]:
# ============================================================
# CONTROL FINAL 2026
# ============================================================

assert ANIO == "2026", f"ALTO: ANIO vale {ANIO}, no 2026"

print("Año confirmado:", ANIO)
print("Total Morelia:", len(morelia_anual))
print(
    "Clases SCIAN totales:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Año confirmado: 2026
Total Morelia: 53208
Clases SCIAN totales: 706


### Tratamiento metodológico de DENUE 2026

Para 2026 se dispone únicamente de la edición DENUE 2026/05 dentro de la
serie histórica analizada, por lo que no fue posible realizar una auditoría
intraanual entre dos cortes.

El archivo contiene 53,208 registros correspondientes al municipio de
Morelia. La identificación territorial mediante la clave municipal 053
coincidió completamente con el nombre del municipio, sin discrepancias.

Los 53,208 registros cuentan con identificador no nulo y único dentro de
la edición. La variable fecha de alta presenta diferentes cohortes de
incorporación al DENUE, entre ellas registros con fecha 2026-04. Estas fechas
se interpretan como fechas de incorporación al Directorio y no automáticamente
como fechas de nacimiento empresarial.

Para el análisis anual se utiliza DENUE 2026/05 como corte representativo
de 2026, al ser la única edición disponible para este año dentro de la serie
histórica analizada.

In [ ]:
# ============================================================
# PASO 10. GUARDAR MORELIA 2026
# ============================================================

assert ANIO == "2026", f"ALTO: ANIO vale {ANIO}"

archivo_salida = MORELIA / f"DENUE_{ANIO}_MORELIA.csv"

morelia_anual.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ DENUE {ANIO} Morelia guardado.")
print(archivo_salida)

✓ DENUE 2026 Morelia guardado.
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\01_MORELIA_LIMPIO\DENUE_2026_MORELIA.csv


In [ ]:
# ============================================================
# PASO 11. GUARDAR PRODUCTOS 2026
# ============================================================

assert ANIO == "2026", f"ALTO: ANIO vale {ANIO}"

conteo_scian_anual.to_csv(
    CONTEOS / f"CONTEO_SCIAN_{ANIO}_MORELIA.csv",
    index=False,
    encoding="utf-8-sig"
)

comercio_anual.to_csv(
    CONTEOS / f"COMERCIO_MINORISTA_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_anual.to_csv(
    CONTEOS / f"SERVICIOS_TODAS_CLASES_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

servicios_mayores_50_anual.to_csv(
    CONTEOS / f"SERVICIOS_MAYORES_50_{ANIO}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ Productos {ANIO} guardados correctamente.")

✓ Productos 2026 guardados correctamente.


In [ ]:
# ============================================================
# PASO 12. ACTUALIZAR RESUMEN HISTÓRICO
# ============================================================

assert ANIO == "2026", f"ALTO: ANIO vale {ANIO}"

nueva_fila = pd.DataFrame([{
    "Año": int(ANIO),

    "Total_Morelia":
        len(morelia_anual),

    "Clases_SCIAN_Total":
        conteo_scian_anual["SCIAN6"].nunique(),

    "Clases_Comercio_Minorista":
        comercio_anual["SCIAN6"].nunique(),

    "Establecimientos_Comercio_Minorista":
        comercio_anual["Número de establecimientos"].sum(),

    "Clases_Servicios":
        servicios_anual["SCIAN6"].nunique(),

    "Establecimientos_Servicios":
        servicios_anual["Número de establecimientos"].sum(),

    "Clases_Servicios_Mayores_50":
        len(servicios_mayores_50_anual),

    "Establecimientos_Servicios_Mayores_50":
        servicios_mayores_50_anual[
            "Número de establecimientos"
        ].sum(),

    "Version_SCIAN":
        "SCIAN 2023",

    "Observacion_metodologica":
        "Único corte disponible para 2026: DENUE 2026/05."
}])

# Evitar duplicar 2026
resumen_historico = resumen_historico[
    resumen_historico["Año"] != int(ANIO)
]

resumen_historico = pd.concat(
    [resumen_historico, nueva_fila],
    ignore_index=True
).sort_values("Año")

resumen_historico.to_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

display(resumen_historico)

print(f"✓ Resumen histórico actualizado hasta {ANIO}.")

,Año,Total_Morelia,Clases_SCIAN_Total,Clases_Comercio_Minorista,Establecimientos_Comercio_Minorista,Clases_Servicios,Establecimientos_Servicios,Clases_Servicios_Mayores_50,Establecimientos_Servicios_Mayores_50,Version_SCIAN,Observacion_metodologica
0,2010,41342,646,80,16693,296,17963,57,15274,SCIAN 2007,
1,2011,42004,670,79,16824,310,18269,59,15505,SCIAN 2007,
2,2012,42251,676,79,16860,314,18429,60,15600,SCIAN 2007,
3,2013,42672,684,79,17045,318,18752,62,15828,SCIAN 2007,
4,2015,44814,660,82,16955,314,20722,65,17981,SCIAN 2013,Ruptura respecto de 2013: actualización integr...
5,2016,45511,672,82,17313,320,21059,66,18225,SCIAN 2013,Se auditaron 2016/01 y 2016/10; se seleccionó ...
6,2017,45715,672,82,17427,319,21109,65,18192,SCIAN 2013,Se auditaron 2017/03 y 2017/11; se seleccionó ...
7,2018,45900,671,82,17452,317,21276,65,18360,SCIAN 2018,Se auditaron 2018/03 y 2018/11; se seleccionó ...
8,2019,47934,709,83,17642,332,22842,72,20155,SCIAN 2018,Se auditaron 2019/04 y 2019/11; se seleccionó ...
9,2020,48981,685,83,17984,327,23464,77,20876,SCIAN 2018,Se auditaron 2020/04 y 2020/11; se seleccionó ...


✓ Resumen histórico actualizado hasta 2026.


In [ ]:
# ============================================================
# PASO 13. ACTUALIZAR BITÁCORA
# ============================================================

assert ANIO == "2026", f"ALTO: ANIO vale {ANIO}"

cortes_denue.loc[
    cortes_denue["Corte"] == "2026_05",
    "Estado"
] = "Procesado - único corte disponible"

cortes_denue.to_csv(
    RESULTADOS / "BITACORA_EDICIONES_DENUE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Bitácora 2026 actualizada.")

✓ Bitácora 2026 actualizada.


In [ ]:
# ============================================================
# CONTROL FINAL 2026
# ============================================================

print("Año actual:", ANIO)
print("Total Morelia:", len(morelia_anual))
print(
    "Clases SCIAN totales 2026:",
    conteo_scian_anual["SCIAN6"].nunique()
)

Año actual: 2026
Total Morelia: 53208
Clases SCIAN totales 2026: 706


# Cierre del procesamiento histórico DENUE 2010–2026

Se completó el procesamiento de las ediciones históricas del DENUE
seleccionadas para el municipio de Morelia, Michoacán.

Para los años con más de una edición se realizaron auditorías intraanuales
antes de seleccionar el corte representativo. La selección no implica que
las diferencias entre ediciones correspondan directamente a nacimientos,
cierres o permanencia empresarial.

La serie contiene cambios de versión del SCIAN:

- 2010–2013: SCIAN 2007
- 2015–2017: SCIAN 2013
- 2018–2023: SCIAN 2018
- 2024–2026: SCIAN 2023

Por tanto, las clases SCIAN de seis dígitos no se compararán directamente
entre versiones distintas del clasificador. La siguiente etapa consiste
en homologar las actividades mediante las tablas oficiales de equivalencia
del SCIAN.

In [ ]:
# ============================================================
# CONTROL FINAL DE LA SERIE HISTÓRICA
# ============================================================

print("Años contenidos:")
print(resumen_historico["Año"].tolist())

print("\nNúmero de años:")
print(len(resumen_historico))

print("\nDuplicados de año:")
print(resumen_historico["Año"].duplicated().sum())

print("\nValores nulos por columna:")
display(resumen_historico.isna().sum())

print("\nVersiones SCIAN:")
display(
    resumen_historico[
        ["Año", "Version_SCIAN"]
    ]
)

Años contenidos:
[2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Número de años:
16

Duplicados de año:
0

Valores nulos por columna:


Año                                      0
Total_Morelia                            0
Clases_SCIAN_Total                       0
Clases_Comercio_Minorista                0
Establecimientos_Comercio_Minorista      0
Clases_Servicios                         0
Establecimientos_Servicios               0
Clases_Servicios_Mayores_50              0
Establecimientos_Servicios_Mayores_50    0
Version_SCIAN                            0
Observacion_metodologica                 0
dtype: int64


Versiones SCIAN:


,Año,Version_SCIAN
0,2010,SCIAN 2007
1,2011,SCIAN 2007
2,2012,SCIAN 2007
3,2013,SCIAN 2007
4,2015,SCIAN 2013
5,2016,SCIAN 2013
6,2017,SCIAN 2013
7,2018,SCIAN 2018
8,2019,SCIAN 2018
9,2020,SCIAN 2018


In [ ]:
# ============================================================
# GUARDAR TABLA MAESTRA FINAL PRE-HOMOLOGACIÓN
# ============================================================

archivo_maestro = (
    RESULTADOS /
    "TABLA_MAESTRA_DENUE_2010_2026_PRE_HOMOLOGACION.csv"
)

resumen_historico.to_csv(
    archivo_maestro,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Tabla maestra pre-homologación guardada:")
print(archivo_maestro)

✓ Tabla maestra pre-homologación guardada:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\05_RESULTADOS\TABLA_MAESTRA_DENUE_2010_2026_PRE_HOMOLOGACION.csv


In [ ]:
# ============================================================
# CONTROL FINAL DE LA SERIE HISTÓRICA DENUE 2010–2026
# ============================================================

print("Años contenidos:")
print(resumen_historico["Año"].tolist())

print("\nNúmero de años:")
print(len(resumen_historico))

print("\nDuplicados de año:")
print(resumen_historico["Año"].duplicated().sum())

print("\nValores nulos por columna:")
display(resumen_historico.isna().sum())

print("\nVersiones SCIAN por año:")
display(
    resumen_historico[
        ["Año", "Version_SCIAN"]
    ]
)

Años contenidos:
[2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Número de años:
16

Duplicados de año:
0

Valores nulos por columna:


Año                                      0
Total_Morelia                            0
Clases_SCIAN_Total                       0
Clases_Comercio_Minorista                0
Establecimientos_Comercio_Minorista      0
Clases_Servicios                         0
Establecimientos_Servicios               0
Clases_Servicios_Mayores_50              0
Establecimientos_Servicios_Mayores_50    0
Version_SCIAN                            0
Observacion_metodologica                 0
dtype: int64


Versiones SCIAN por año:


,Año,Version_SCIAN
0,2010,SCIAN 2007
1,2011,SCIAN 2007
2,2012,SCIAN 2007
3,2013,SCIAN 2007
4,2015,SCIAN 2013
5,2016,SCIAN 2013
6,2017,SCIAN 2013
7,2018,SCIAN 2018
8,2019,SCIAN 2018
9,2020,SCIAN 2018


In [ ]:
# ============================================================
# GUARDAR TABLA MAESTRA FINAL PRE-HOMOLOGACIÓN
# ============================================================

archivo_maestro = (
    RESULTADOS /
    "TABLA_MAESTRA_DENUE_2010_2026_PRE_HOMOLOGACION.csv"
)

resumen_historico.to_csv(
    archivo_maestro,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Tabla maestra pre-homologación guardada:")
print(archivo_maestro)

✓ Tabla maestra pre-homologación guardada:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\05_RESULTADOS\TABLA_MAESTRA_DENUE_2010_2026_PRE_HOMOLOGACION.csv


In [ ]:
# ============================================================
# COPIA DE SEGURIDAD DEL RESUMEN HISTÓRICO
# ============================================================

resumen_historico.to_csv(
    RESULTADOS /
    "BACKUP_RESUMEN_HISTORICO_DENUE_2010_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Copia de seguridad guardada.")

✓ Copia de seguridad guardada.


In [ ]:
# ============================================================
# GUARDAR TABLA MAESTRA PRE-HOMOLOGACIÓN
# ============================================================

archivo_maestro = (
    RESULTADOS /
    "TABLA_MAESTRA_DENUE_2010_2026_PRE_HOMOLOGACION.csv"
)

resumen_historico.to_csv(
    archivo_maestro,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Tabla maestra guardada:")
print(archivo_maestro)

✓ Tabla maestra guardada:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\05_RESULTADOS\TABLA_MAESTRA_DENUE_2010_2026_PRE_HOMOLOGACION.csv


In [ ]:
# ============================================================
# BACKUP FINAL DE LA FASE 1
# ============================================================

backup_maestro = (
    RESULTADOS /
    "BACKUP_TABLA_MAESTRA_DENUE_2010_2026.csv"
)

resumen_historico.to_csv(
    backup_maestro,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Backup guardado:")
print(backup_maestro)

✓ Backup guardado:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\05_RESULTADOS\BACKUP_TABLA_MAESTRA_DENUE_2010_2026.csv


# FASE 2. HOMOLOGACIÓN HISTÓRICA DEL SCIAN

La serie DENUE procesada utiliza cuatro versiones del SCIAN:

- 2010–2013: SCIAN 2007
- 2015–2017: SCIAN 2013
- 2018–2023: SCIAN 2018
- 2024–2026: SCIAN 2023

La homologación se realizará utilizando las tablas comparativas oficiales
publicadas por INEGI.

No se asumirá que códigos iguales o similares representan necesariamente
la misma actividad entre versiones.

Las correspondencias se clasificarán en:

1. Correspondencia directa 1 a 1.
2. División de una clase en varias clases.
3. Fusión de varias clases en una.
4. Correspondencia parcial.
5. Sin correspondencia directa.

Cuando una equivalencia no permita comparar directamente los datos a seis
dígitos, se construirá un nivel de agregación históricamente comparable en
lugar de asignar arbitrariamente una clase.

In [ ]:
# ============================================================
# FASE 2 - PASO 1
# CONSTRUIR INVENTARIO HISTÓRICO DE CÓDIGOS SCIAN
# ============================================================

años_version = {
    2010: "SCIAN 2007",
    2011: "SCIAN 2007",
    2012: "SCIAN 2007",
    2013: "SCIAN 2007",

    2015: "SCIAN 2013",
    2016: "SCIAN 2013",
    2017: "SCIAN 2013",

    2018: "SCIAN 2018",
    2019: "SCIAN 2018",
    2020: "SCIAN 2018",
    2021: "SCIAN 2018",
    2022: "SCIAN 2018",
    2023: "SCIAN 2018",

    2024: "SCIAN 2023",
    2025: "SCIAN 2023",
    2026: "SCIAN 2023"
}

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

filas_inventario = []

for año, version in años_version.items():

    ruta = CONTEOS / f"CONTEO_SCIAN_{año}_MORELIA.csv"

    df = pd.read_csv(
        ruta,
        dtype=str,
        encoding="utf-8-sig"
    )

    # Detectar automáticamente la columna del código SCIAN
    candidatos_codigo = [
        "SCIAN6",
        "codigo_act",
        "Código de la clase de actividad",
        "Código de la clase de actividad SCIAN"
    ]

    col_codigo = next(
        c for c in candidatos_codigo
        if c in df.columns
    )

    # Detectar columna de conteo
    col_conteo = "Número de establecimientos"

    df["SCIAN6_STD"] = (
        df[col_codigo]
        .astype(str)
        .str.strip()
        .str.zfill(6)
    )

    df["Sector_STD"] = df["SCIAN6_STD"].str[:2]

    # Comercio minorista + servicios
    df_interes = df[
        (df["Sector_STD"] == "46")
        |
        (df["Sector_STD"].isin(sectores_servicios))
    ].copy()

    for _, fila in df_interes.iterrows():

        filas_inventario.append({
            "Año": año,
            "Version_SCIAN": version,
            "SCIAN_original": fila["SCIAN6_STD"],
            "Sector": fila["Sector_STD"],
            "Establecimientos": int(
                float(fila[col_conteo])
            )
        })

inventario_scian = pd.DataFrame(filas_inventario)

display(inventario_scian.head(20))

print("Registros año-código:", len(inventario_scian))

,Año,Version_SCIAN,SCIAN_original,Sector,Establecimientos
0,2010,SCIAN 2007,461110,46,4179
1,2010,SCIAN 2007,722212,72,1975
2,2010,SCIAN 2007,722219,72,1707
3,2010,SCIAN 2007,812110,81,1582
4,2010,SCIAN 2007,463211,46,1481
5,2010,SCIAN 2007,461130,46,1174
6,2010,SCIAN 2007,621211,62,913
7,2010,SCIAN 2007,465311,46,849
8,2010,SCIAN 2007,811111,81,656
9,2010,SCIAN 2007,561432,56,578


Registros año-código: 6435


In [ ]:
# ============================================================
# FASE 2 - PASO 2
# CÓDIGOS ÚNICOS QUE REALMENTE APARECEN EN LA SERIE
# ============================================================

codigos_por_version = (
    inventario_scian[
        ["Version_SCIAN", "SCIAN_original", "Sector"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Version_SCIAN", "Sector", "SCIAN_original"]
    )
    .reset_index(drop=True)
)

print("Códigos únicos por versión:\n")

display(
    codigos_por_version
    .groupby("Version_SCIAN")
    .size()
    .reset_index(name="Número_de_códigos")
)

display(codigos_por_version.head(30))

Códigos únicos por versión:



,Version_SCIAN,Número_de_códigos
0,SCIAN 2007,399
1,SCIAN 2013,407
2,SCIAN 2018,442
3,SCIAN 2023,421


,Version_SCIAN,SCIAN_original,Sector
0,SCIAN 2007,461110,46
1,SCIAN 2007,461121,46
2,SCIAN 2007,461122,46
3,SCIAN 2007,461123,46
4,SCIAN 2007,461130,46
5,SCIAN 2007,461140,46
6,SCIAN 2007,461150,46
7,SCIAN 2007,461160,46
8,SCIAN 2007,461170,46
9,SCIAN 2007,461190,46


In [ ]:
# ============================================================
# FASE 2 - PASO 4
# DESCARGAR TABLAS OFICIALES DE EQUIVALENCIA SCIAN
# ============================================================

from urllib.request import urlretrieve

url_2007_2013 = (
    "https://www.inegi.org.mx/contenidos/app/scian/tablaxi.pdf"
)

url_2013_2018 = (
    "https://www.inegi.org.mx/contenidos/app/scian/tablaxiii.pdf"
)

url_2018_2023 = (
    "https://www.inegi.org.mx/contenidos/app/scian/tablaxv.pdf"
)

archivo_2007_2013 = (
    HOMOLOGACION / "SCIAN_2013_2007_OFICIAL.pdf"
)

archivo_2013_2018 = (
    HOMOLOGACION / "SCIAN_2018_2013_OFICIAL.pdf"
)

archivo_2018_2023 = (
    HOMOLOGACION / "SCIAN_2023_2018_OFICIAL.pdf"
)

urlretrieve(url_2007_2013, archivo_2007_2013)
urlretrieve(url_2013_2018, archivo_2013_2018)
urlretrieve(url_2018_2023, archivo_2018_2023)

print("✓ Tablas oficiales descargadas:")
print(archivo_2007_2013)
print(archivo_2013_2018)
print(archivo_2018_2023)

✓ Tablas oficiales descargadas:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN\SCIAN_2013_2007_OFICIAL.pdf
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN\SCIAN_2018_2013_OFICIAL.pdf
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN\SCIAN_2023_2018_OFICIAL.pdf


In [ ]:
%pip install pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.6 MB 4.2 MB/s eta 0:00:02
   ------ --------------------------------- 1.0/6.6 MB 3.4 MB/s eta 0:00:02
   ----------- ---------------------------- 1.8/6.6 MB 3.1 MB/s eta 0:00:02
   --------------- ------------------------ 2.6/6.6 MB 3.3 MB/s eta 0:00:02
   -------------------- ------------------- 3.4/6.6 MB 3.3 MB/s eta 0:00:01
   ------------------------- -------------- 4.2/6.6 MB 3.3 MB/s eta 0:00:01
   ---------------------------- ----------- 4.7/6.6 MB 3.4 MB/s eta 0:00:01
   --------------------------------- ------ 5.5/6.6 MB 3.4 MB/s eta 0:00:01
   -------------------------------------- - 6.3/6.6 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 3.3 MB/s  0:00:02
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 3.4 MB/s eta 0:00:01
   ------------- ------------

In [ ]:
import pdfplumber

print("pdfplumber listo.")

pdfplumber listo.


In [ ]:
# ============================================================
# FASE 2 - PASO 5B
# INSPECCIONAR TABLA OFICIAL SCIAN 2013-2007
# ============================================================

pdf_2007_2013 = HOMOLOGACION / "SCIAN_2013_2007_OFICIAL.pdf"

with pdfplumber.open(pdf_2007_2013) as pdf:
    
    print("Número de páginas:", len(pdf.pages))
    
    print("\nTexto de la primera página:\n")
    texto = pdf.pages[0].extract_text()
    print(texto[:4000])

Número de páginas: 16

Texto de la primera página:

Sistema de Clasificación Industrial de América del Norte,
México 2013, SCIAN México 2013-Sistema de Clasificación
Industrial de América del Norte, México 2007, SCIAN México 2007
SCIAN
México 2013
SCIAN
México 2007


In [ ]:
# ============================================================
# FASE 2 - PASO 5C
# PROBAR EXTRACCIÓN DE TABLAS
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:
    
    for numero_pagina in range(min(3, len(pdf.pages))):
        
        pagina = pdf.pages[numero_pagina]
        tablas = pagina.extract_tables()
        
        print("\n====================================")
        print("PÁGINA:", numero_pagina + 1)
        print("Tablas detectadas:", len(tablas))
        print("====================================")
        
        for i, tabla in enumerate(tablas):
            
            print("\nTABLA", i + 1)
            
            for fila in tabla[:10]:
                print(fila)


PÁGINA: 1
Tablas detectadas: 0

PÁGINA: 2
Tablas detectadas: 0

PÁGINA: 3
Tablas detectadas: 0


In [ ]:
# ============================================================
# FASE 2 - PASO 5D
# VER TEXTO DE LA PÁGINA 2
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    texto_pagina_2 = pdf.pages[1].extract_text()

    print(texto_pagina_2[:8000])

ÍNDICE
Introducción V
Sistema de Clasificación Industrial de América del Norte, México 2013, SCIAN México 2013-Sistema
de Clasificación Industrial de América del Norte, México 2007, SCIAN México 2007 1
11 Agricultura, cría y explotación de animales, aprovechamiento forestal, pesca y caza 2
31-33 Industrias manufactureras 2
43 Comercio al por mayor 3
46 Comercio al por menor 3
51 Información en medios masivos 4
52 Servicios financieros y de seguros 4
53 Servicios inmobiliarios y de alquiler de bienes muebles e intangibles 4
61 Servicios educativos 4
62 Servicios de salud y de asistencia social 4
72 Servicios de alojamiento temporal y de preparación de alimentos y bebidas 5
81 Otros servicios excepto actividades gubernamentales 6
Sistema de Clasificación Industrial de América del Norte, México 2007, SCIAN México 2007-Sistema
de Clasificación Industrial de América del Norte, México 2013, SCIAN México 2013 7
11 Agricultura, cría y explotación de animales, aprovechamiento forestal, pesca y 

In [ ]:
# ============================================================
# FASE 2 - PASO 5E
# LOCALIZAR AUTOMÁTICAMENTE LAS PÁGINAS CON CÓDIGOS SCIAN
# ============================================================

import re

patron_scian = re.compile(r"\b\d{6}\b")

with pdfplumber.open(pdf_2007_2013) as pdf:

    paginas_con_codigos = []

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""

        codigos = patron_scian.findall(texto)

        if codigos:
            paginas_con_codigos.append({
                "pagina_pdf": i + 1,
                "cantidad_codigos": len(codigos),
                "primeros_codigos": codigos[:10]
            })

for resultado in paginas_con_codigos:
    print(resultado)

{'pagina_pdf': 4, 'cantidad_codigos': 9, 'primeros_codigos': ['114119', '114119', '313111', '313111', '314911', '314999', '316991', '333241', '333210']}
{'pagina_pdf': 5, 'cantidad_codigos': 1, 'primeros_codigos': ['465915']}
{'pagina_pdf': 7, 'cantidad_codigos': 35, 'primeros_codigos': ['111214', '111214', '111218', '111214', '111942', '111942', '112512', '112512', '112999', '112999']}
{'pagina_pdf': 8, 'cantidad_codigos': 37, 'primeros_codigos': ['315110', '315110', '315191', '315191', '315192', '315192', '316991', '316991', '333241', '333210']}
{'pagina_pdf': 9, 'cantidad_codigos': 30, 'primeros_codigos': ['465215', '465215', '465915', '517110', '517111', '517112', '517210', '517210', '517410', '517410']}
{'pagina_pdf': 10, 'cantidad_codigos': 29, 'primeros_codigos': ['621115', '621111', '621113', '621116', '621112', '621114', '722511', '722110', '722219', '722512']}
{'pagina_pdf': 11, 'cantidad_codigos': 4, 'primeros_codigos': ['812321', '812320', '812322', '812320']}
{'pagina_pdf'

In [ ]:
# ============================================================
# FASE 2 - PASO 5F
# VER PRIMERA PÁGINA REAL DE CORRESPONDENCIAS
# ============================================================

primera_pagina_datos = paginas_con_codigos[0]["pagina_pdf"] - 1

with pdfplumber.open(pdf_2007_2013) as pdf:

    texto_datos = pdf.pages[
        primera_pagina_datos
    ].extract_text() or ""

    print(
        "Primera página de datos:",
        primera_pagina_datos + 1
    )

    print("\n")
    print(texto_datos[:12000])

Primera página de datos: 4


Cuando una categoría del clasificador base mantiene correspondencia con otra completa del
segundo clasificador, en la segunda columna queda vacío el espacio de la primera subcolumna,
y en las dos subcolumnas restantes sólo aparecen el código y el nombre de la categoría
equivalente.
Enseguida se muestran ejemplos de las principales variantes que pueden hallarse en las tablas:
Caso 1. La clase SCIAN México 2013 es igual en contenido a una clase SCIAN México 2007,
pero los títulos son distintos:
SCIAN México 2013 SCIAN México 2007
Código Nombre Parte Código Nombre
114119 Pesca y captura de otros peces, 114119 Pesca y captura de peces, crustáceos,
crustáceos, moluscos y otras especies moluscos y otras especies
Caso 2. La clase SCIAN México 2013 es equivalente a la suma de varias clases o partes de clases
SCIAN México 2007:
SCIAN México 2013 SCIAN México 2007
Código Nombre Parte Código Nombre
313111 Preparación e hilado de fibras 313111 Preparación e hilado de f

In [ ]:
# ============================================================
# FASE 2 - PASO 5G
# LOCALIZAR PRIMERA PÁGINA REAL DE CORRESPONDENCIAS
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    candidatas = []

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        codigos = patron_scian.findall(texto)

        # Buscamos páginas con muchos códigos
        # y descartamos las páginas explicativas
        if (
            len(codigos) >= 20
            and "Caso 1." not in texto
            and "Caso 2." not in texto
            and "Caso 3." not in texto
            and "ÍNDICE" not in texto
        ):
            candidatas.append({
                "pagina_pdf": i + 1,
                "cantidad_codigos": len(codigos),
                "primeros_codigos": codigos[:15]
            })

for x in candidatas:
    print(x)

{'pagina_pdf': 7, 'cantidad_codigos': 35, 'primeros_codigos': ['111214', '111214', '111218', '111214', '111942', '111942', '112512', '112512', '112999', '112999', '112999', '114119', '114119', '311350', '311320']}
{'pagina_pdf': 8, 'cantidad_codigos': 37, 'primeros_codigos': ['315110', '315110', '315191', '315191', '315192', '315192', '316991', '316991', '333241', '333210', '333242', '333220', '333243', '333291', '333244']}
{'pagina_pdf': 9, 'cantidad_codigos': 30, 'primeros_codigos': ['465215', '465215', '465915', '517110', '517111', '517112', '517210', '517210', '517410', '517410', '522460', '522490', '522490', '522490', '531116']}
{'pagina_pdf': 10, 'cantidad_codigos': 29, 'primeros_codigos': ['621115', '621111', '621113', '621116', '621112', '621114', '722511', '722110', '722219', '722512', '722110', '722219', '722513', '722110', '722219']}
{'pagina_pdf': 13, 'cantidad_codigos': 40, 'primeros_codigos': ['111214', '111214', '111218', '111942', '111942', '112512', '112512', '112999',

In [ ]:
# ============================================================
# VER LA PRIMERA PÁGINA CANDIDATA
# ============================================================

pagina_tabla = candidatas[0]["pagina_pdf"] - 1

with pdfplumber.open(pdf_2007_2013) as pdf:

    texto = pdf.pages[pagina_tabla].extract_text() or ""

    print("Página PDF:", pagina_tabla + 1)
    print("\n")
    
    lineas = texto.splitlines()

    for linea in lineas[:100]:
        print(linea)

Página PDF: 7


TABLA COMPARATIVA SCIAN México 2013‐SCIAN México 2007
SCIAN México 2013 SCIAN México 2007
Código Nombre Parte Código Nombre
11 AGRICULTURA, CRÍA Y EXPLOTACIÓN DE ANIMALES, APROVECHAMIENTO FORESTAL, PESCA Y CAZA
111214 Cultivo de melón p 1/2 111214 Cultivo de melón o sandía (cultivo de melón)
111218 Cultivo de sandía p 2/2 111214 Cultivo de melón o sandía (cultivo de sandía)
111942 Cultivo de pastos 111942 Cultivo de pastos y zacates
112512 Piscicultura y otra acuicultura, 112512 Piscicultura y otra acuicultura,
excepto camaronicultura excepto camaronicultura
p 1/2 112999 Explotación de otros animales (explotación de cocodrilos
en ambientes controlados)
112999 Explotación de otros animales p 2/2 112999 Explotación de otros animales (excepto explotación
de cocodrilos en ambientes controlados)
114119 Pesca y captura de otros peces, crustáceos, 114119 Pesca y captura de peces, crustáceos, moluscos
moluscos y otras especies y otras especies
31-33 INDUSTRIAS MANUFACTURERAS
31

In [ ]:
# ============================================================
# FASE 2 - PASO 5H
# PROBAR EXTRACCIÓN TABULAR EN LA PÁGINA REAL
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    pagina = pdf.pages[pagina_tabla]

    tablas = pagina.extract_tables()

    print("Tablas detectadas:", len(tablas))

    for i, tabla in enumerate(tablas):

        print("\n========================")
        print("TABLA", i + 1)
        print("========================")

        for fila in tabla[:20]:
            print(fila)

Tablas detectadas: 0


In [ ]:
# ============================================================
# FASE 2 - PASO 5I
# INSPECCIONAR POSICIONES DE TEXTO EN LA TABLA REAL
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    pagina = pdf.pages[pagina_tabla]

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    # Agrupar palabras aproximadamente por línea
    lineas = {}

    for palabra in palabras:
        top = round(palabra["top"] / 3) * 3

        if top not in lineas:
            lineas[top] = []

        lineas[top].append(
            (
                palabra["text"],
                round(palabra["x0"], 1)
            )
        )

    # Mostrar únicamente líneas donde aparezca
    # al menos un código SCIAN de 6 dígitos
    contador = 0

    for top in sorted(lineas):

        contenido = sorted(
            lineas[top],
            key=lambda x: x[1]
        )

        texto_linea = " ".join(
            palabra for palabra, x in contenido
        )

        if re.search(r"\b\d{6}\b", texto_linea):

            print("\nTOP:", top)

            for palabra, x in contenido:
                print(
                    f"{x:7.1f}  {palabra}"
                )

            contador += 1

            if contador >= 15:
                break


TOP: 105
   54.1  111214
  102.3  Cultivo
  130.9  de
  141.6  melón
  280.9  p
  287.7  1/2
  320.5  111214
  368.7  Cultivo
  397.4  de
  408.1  melón
  432.3  o
  438.7  sandía
  464.0  (cultivo
  491.0  de
  500.1  melón)
  573.8  M

TOP: 126
   54.1  111218
  102.3  Cultivo
  130.9  de
  141.6  sandía
  280.9  p
  287.7  2/2
  320.5  111214
  368.7  Cultivo
  397.4  de
  408.1  melón
  432.4  o
  438.7  sandía
  464.0  (cultivo
  491.0  de
  500.1  sandía)

TOP: 147
   54.1  111942
  102.3  Cultivo
  130.9  de
  141.7  pastos
  320.5  111942
  368.7  Cultivo
  397.4  de
  408.1  pastos
  432.7  y
  439.1  zacates
  575.8  led

TOP: 171
   54.1  112512
  102.3  Piscicultura
  146.0  y
  152.5  otra
  168.6  acuicultura,
  320.5  112512
  368.7  Piscicultura
  412.5  y
  418.9  otra
  435.1  acuicultura,

TOP: 192
  280.9  p
  287.7  1/2
  320.6  112999
  368.7  Explotación
  412.8  de
  423.5  otros
  443.4  animales
  477.0  (explotación
  518.2  de
  527.3  cocodrilos

TOP: 222


In [ ]:
# ============================================================
# FASE 2 - PASO 5J
# LOCALIZAR LOS DOS SENTIDOS DE LA TABLA
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        texto_norm = normalizar_texto(texto)

        if "tabla comparativa" in texto_norm:

            print("\nPÁGINA PDF:", i + 1)

            primeras_lineas = texto.splitlines()[:8]

            for linea in primeras_lineas:
                print(linea)


PÁGINA PDF: 3
INTRODUCCIÓN
Sistema de Clasificación Industrial de América del Norte, México 2013,
SCIAN México 2013-Sistema de Clasificación Industrial de América del
Norte, México 2007, SCIAN México 2007
Para lograr la relación de los clasificadores y por lo tanto de la información que se recoge con
ellos, se elaboran tablas comparativas o de equivalencia1 que muestran los vínculos entre las
categorías de los clasificadores en estudio.
Para ayudar en las comparaciones hacia el pasado se elaboran tablas comparativas entre las

PÁGINA PDF: 7
TABLA COMPARATIVA SCIAN México 2013‐SCIAN México 2007
SCIAN México 2013 SCIAN México 2007
Código Nombre Parte Código Nombre
11 AGRICULTURA, CRÍA Y EXPLOTACIÓN DE ANIMALES, APROVECHAMIENTO FORESTAL, PESCA Y CAZA
111214 Cultivo de melón p 1/2 111214 Cultivo de melón o sandía (cultivo de melón)
111218 Cultivo de sandía p 2/2 111214 Cultivo de melón o sandía (cultivo de sandía)
111942 Cultivo de pastos 111942 Cultivo de pastos y zacates
112512 Piscicul

In [ ]:
# ============================================================
# FASE 2 - PASO 5K
# FUNCIÓN PARA RECONSTRUIR COLUMNAS POR POSICIÓN X
# ============================================================

import re
import pandas as pd

patron_codigo = re.compile(r"^\d{6}$")

def extraer_lineas_scian(pagina):

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    # Agrupar palabras por línea visual
    lineas = {}

    for palabra in palabras:

        top = round(palabra["top"] / 3) * 3

        lineas.setdefault(top, []).append(palabra)

    resultados = []

    for top in sorted(lineas):

        palabras_linea = sorted(
            lineas[top],
            key=lambda w: w["x0"]
        )

        columnas = {
            "codigo_2013": [],
            "nombre_2013": [],
            "parte": [],
            "codigo_2007": [],
            "nombre_2007": []
        }

        for w in palabras_linea:

            x = w["x0"]
            txt = w["text"].strip()

            # Ignorar marca lateral del documento
            if x >= 565:
                continue

            if x < 90:
                columnas["codigo_2013"].append(txt)

            elif x < 275:
                columnas["nombre_2013"].append(txt)

            elif x < 317:
                columnas["parte"].append(txt)

            elif x < 360:
                columnas["codigo_2007"].append(txt)

            else:
                columnas["nombre_2007"].append(txt)

        fila = {
            clave: " ".join(valor).strip()
            for clave, valor in columnas.items()
        }

        # Conservar solo líneas con contenido útil
        if any(fila.values()):
            fila["top"] = top
            resultados.append(fila)

    return resultados

In [ ]:
print("Función creada correctamente:", callable(extraer_lineas_scian))

Función creada correctamente: True


In [ ]:
# ============================================================
# FASE 2 - PASO 5L
# PROBAR EXTRACTOR EN UNA PÁGINA REAL
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    filas_prueba = extraer_lineas_scian(
        pdf.pages[pagina_tabla]
    )

df_prueba = pd.DataFrame(filas_prueba)

print("Filas extraídas:", len(df_prueba))

display(
    df_prueba[
        [
            "codigo_2013",
            "nombre_2013",
            "parte",
            "codigo_2007",
            "nombre_2007"
        ]
    ].head(30)
)

Filas extraídas: 46


,codigo_2013,nombre_2013,parte,codigo_2007,nombre_2007
0,TABLA,COMPARATIVA SCIAN México,2013‐SCIAN,México,2007
1,,SCIAN México 2013,,,SCIAN México 2007
2,Código,Nombre,Parte,Código,Nombre
3,"11 AGRICULTURA,","CRÍA Y EXPLOTACIÓN DE ANIMALES,",APROVECHAMIENTO,,"FORESTAL, PESCA Y CAZA"
4,111214,Cultivo de melón,p 1/2,111214,Cultivo de melón o sandía (cultivo de melón)
5,111218,Cultivo de sandía,p 2/2,111214,Cultivo de melón o sandía (cultivo de sandía)
6,111942,Cultivo de pastos,,111942,Cultivo de pastos y zacates
7,112512,"Piscicultura y otra acuicultura,",,112512,"Piscicultura y otra acuicultura,"
8,,excepto camaronicultura,,,excepto camaronicultura
9,,,p 1/2,112999,Explotación de otros animales (explotación de ...


In [ ]:
# ============================================================
# FASE 2 - PASO 5M
# RECONSTRUIR REGISTROS LÓGICOS DE LA TABLA
# ============================================================

import re
import pandas as pd

patron_6 = re.compile(r"^\d{6}$")
patron_sector = re.compile(r"^(\d{2}|\d{2}-\d{2})$")

def limpiar_texto(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def reconstruir_correspondencias(df_lineas):

    registros = []

    codigo_2013_activo = ""
    nombre_2013_activo = ""

    registro_actual = None

    for _, fila in df_lineas.iterrows():

        c13 = limpiar_texto(fila.get("codigo_2013", ""))
        n13 = limpiar_texto(fila.get("nombre_2013", ""))
        parte = limpiar_texto(fila.get("parte", ""))
        c07 = limpiar_texto(fila.get("codigo_2007", ""))
        n07 = limpiar_texto(fila.get("nombre_2007", ""))

        # -----------------------------------------
        # 1. Ignorar encabezados de sector
        # Ej.: 11, 31-33, 46, etc.
        # -----------------------------------------
        if patron_sector.match(c13):
            codigo_2013_activo = ""
            nombre_2013_activo = ""
            registro_actual = None
            continue

        # -----------------------------------------
        # 2. Si aparece un nuevo código 2013,
        # inicia una nueva clase activa
        # -----------------------------------------
        if patron_6.match(c13):

            codigo_2013_activo = c13
            nombre_2013_activo = n13

        # -----------------------------------------
        # 3. Si no hay código 2013 pero sí texto
        # en nombre 2013, es continuación del nombre
        # -----------------------------------------
        elif (
            codigo_2013_activo
            and n13
            and not c07
            and not parte
        ):
            nombre_2013_activo = limpiar_texto(
                nombre_2013_activo + " " + n13
            )

        # -----------------------------------------
        # 4. Si hay código 2007, inicia una
        # correspondencia nueva
        # -----------------------------------------
        if patron_6.match(c07):

            registro_actual = {
                "SCIAN_2013": codigo_2013_activo,
                "Nombre_2013": nombre_2013_activo,
                "Parte": parte,
                "SCIAN_2007": c07,
                "Nombre_2007": n07
            }

            registros.append(registro_actual)

        # -----------------------------------------
        # 5. Si no hay nuevo código 2007 pero sí
        # texto a la derecha, continúa su nombre
        # -----------------------------------------
        elif registro_actual is not None and n07:

            registro_actual["Nombre_2007"] = limpiar_texto(
                registro_actual["Nombre_2007"]
                + " "
                + n07
            )

        # -----------------------------------------
        # 6. Si aparece texto adicional del nombre
        # 2013 en la misma secuencia, actualizar
        # todos los registros de esa clase activa
        # -----------------------------------------
        if patron_6.match(c13) and n13:
            nombre_2013_activo = n13

    resultado = pd.DataFrame(registros)

    # Completar Nombre_2013 con la versión final
    # reconstruida para cada código
    nombres_2013 = {}

    codigo_activo = ""
    nombre_activo = ""

    for _, fila in df_lineas.iterrows():

        c13 = limpiar_texto(fila.get("codigo_2013", ""))
        n13 = limpiar_texto(fila.get("nombre_2013", ""))
        parte = limpiar_texto(fila.get("parte", ""))
        c07 = limpiar_texto(fila.get("codigo_2007", ""))

        if patron_sector.match(c13):
            codigo_activo = ""
            nombre_activo = ""
            continue

        if patron_6.match(c13):
            codigo_activo = c13
            nombre_activo = n13

        elif (
            codigo_activo
            and n13
            and not c07
            and not parte
        ):
            nombre_activo = limpiar_texto(
                nombre_activo + " " + n13
            )

        if codigo_activo:
            nombres_2013[codigo_activo] = nombre_activo

    resultado["Nombre_2013"] = (
        resultado["SCIAN_2013"]
        .map(nombres_2013)
        .fillna(resultado["Nombre_2013"])
    )

    return resultado

In [ ]:
# ============================================================
# FASE 2 - PASO 5N
# PROBAR LIMPIEZA EN LA PÁGINA DE EJEMPLO
# ============================================================

df_correspondencias_prueba = reconstruir_correspondencias(
    df_prueba
)

print(
    "Correspondencias reconstruidas:",
    len(df_correspondencias_prueba)
)

display(
    df_correspondencias_prueba.head(30)
)

Correspondencias reconstruidas: 20


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2007,Nombre_2007
0,111214,Cultivo de melón,p 1/2,111214,Cultivo de melón o sandía (cultivo de melón)
1,111218,Cultivo de sandía,p 2/2,111214,Cultivo de melón o sandía (cultivo de sandía)
2,111942,Cultivo de pastos,,111942,Cultivo de pastos y zacates
3,112512,"Piscicultura y otra acuicultura, excepto camar...",,112512,"Piscicultura y otra acuicultura, excepto camar..."
4,112512,"Piscicultura y otra acuicultura, excepto camar...",p 1/2,112999,Explotación de otros animales (explotación de ...
5,112999,Explotación de otros animales,p 2/2,112999,Explotación de otros animales (excepto explota...
6,114119,"Pesca y captura de otros peces, crustáceos, mo...",,114119,"Pesca y captura de peces, crustáceos, moluscos..."
7,311350,Elaboración de chocolate y productos de chocolate,,311320,Elaboración de chocolate y productos de chocol...
8,311350,Elaboración de chocolate y productos de chocolate,,311330,Elaboración de productos de chocolate a partir...
9,311412,Congelación de guisos y otros alimentos prepar...,,311412,Congelación de alimentos preparados


In [ ]:
# ============================================================
# FASE 2 - PASO 5O
# IDENTIFICAR PÁGINAS DE CADA SENTIDO DE CORRESPONDENCIA
# ============================================================

paginas_2013_2007 = []
paginas_2007_2013 = []

with pdfplumber.open(pdf_2007_2013) as pdf:

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        texto_norm = normalizar_texto(texto)

        if "tabla comparativa" not in texto_norm:
            continue

        pos_2013 = texto_norm.find("scian mexico 2013")
        pos_2007 = texto_norm.find("scian mexico 2007")

        if pos_2013 == -1 or pos_2007 == -1:
            continue

        if pos_2013 < pos_2007:
            paginas_2013_2007.append(i)

        elif pos_2007 < pos_2013:
            paginas_2007_2013.append(i)

print("Páginas PDF SCIAN 2013 -> SCIAN 2007:")
print([p + 1 for p in paginas_2013_2007])

print("\nPáginas PDF SCIAN 2007 -> SCIAN 2013:")
print([p + 1 for p in paginas_2007_2013])

Páginas PDF SCIAN 2013 -> SCIAN 2007:
[3, 7, 8, 9, 10, 11]

Páginas PDF SCIAN 2007 -> SCIAN 2013:
[13, 14, 15, 16]


In [ ]:
# ============================================================
# FASE 2 - PASO 5P
# VERIFICAR QUÉ PÁGINAS CONTIENEN DATOS REALES
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    for pagina_pdf in range(6, 17):

        pagina = pdf.pages[pagina_pdf - 1]
        texto = pagina.extract_text() or ""

        codigos = re.findall(r"\b\d{6}\b", texto)

        print(
            f"Página {pagina_pdf}: "
            f"{len(codigos)} códigos de 6 dígitos"
        )

Página 6: 0 códigos de 6 dígitos
Página 7: 35 códigos de 6 dígitos
Página 8: 37 códigos de 6 dígitos
Página 9: 30 códigos de 6 dígitos
Página 10: 29 códigos de 6 dígitos
Página 11: 4 códigos de 6 dígitos
Página 12: 0 códigos de 6 dígitos
Página 13: 40 códigos de 6 dígitos
Página 14: 34 códigos de 6 dígitos
Página 15: 30 códigos de 6 dígitos
Página 16: 22 códigos de 6 dígitos


In [ ]:
# ============================================================
# FASE 2 - PASO 5Q
# REVISAR INICIO DE LAS DOS TABLAS
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    for pagina_pdf in [6, 12]:

        texto = pdf.pages[pagina_pdf - 1].extract_text() or ""

        print("\n====================================")
        print("PÁGINA PDF:", pagina_pdf)
        print("====================================")

        for linea in texto.splitlines()[:30]:
            print(linea)


PÁGINA PDF: 6
Sistema de Clasificación Industrial de América del Norte,
México 2013, SCIAN México 2013-Sistema de Clasificación
Industrial de América del Norte, México 2007, SCIAN México 2007
SCIAN
México 2013
SCIAN
México 2007

PÁGINA PDF: 12
Sistema de Clasificación Industrial de América del Norte,
México 2007, SCIAN México 2007-Sistema de Clasificación
Industrial de América del Norte, México 2013, SCIAN México 2013
SCIAN
México 2007
SCIAN
México 2013


In [ ]:
# ============================================================
# FASE 2 - PASO 5R
# EXTRAER TABLA COMPLETA SCIAN 2013 -> SCIAN 2007
# PÁGINAS PDF 7 A 11
# ============================================================

paginas_datos_2013_2007 = [7, 8, 9, 10, 11]

todos_los_registros = []

with pdfplumber.open(pdf_2007_2013) as pdf:

    for pagina_pdf in paginas_datos_2013_2007:

        # PDF empieza en 1; Python en 0
        pagina = pdf.pages[pagina_pdf - 1]

        lineas_pagina = extraer_lineas_scian(pagina)

        df_lineas_pagina = pd.DataFrame(lineas_pagina)

        df_registros_pagina = reconstruir_correspondencias(
            df_lineas_pagina
        )

        df_registros_pagina["Pagina_PDF"] = pagina_pdf

        todos_los_registros.append(
            df_registros_pagina
        )

correspondencias_2013_2007 = pd.concat(
    todos_los_registros,
    ignore_index=True
)

print(
    "Correspondencias extraídas:",
    len(correspondencias_2013_2007)
)

display(
    correspondencias_2013_2007.head(30)
)

Correspondencias extraídas: 75


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2007,Nombre_2007,Pagina_PDF
0,111214,Cultivo de melón,p 1/2,111214,Cultivo de melón o sandía (cultivo de melón),7
1,111218,Cultivo de sandía,p 2/2,111214,Cultivo de melón o sandía (cultivo de sandía),7
2,111942,Cultivo de pastos,,111942,Cultivo de pastos y zacates,7
3,112512,"Piscicultura y otra acuicultura, excepto camar...",,112512,"Piscicultura y otra acuicultura, excepto camar...",7
4,112512,"Piscicultura y otra acuicultura, excepto camar...",p 1/2,112999,Explotación de otros animales (explotación de ...,7
5,112999,Explotación de otros animales,p 2/2,112999,Explotación de otros animales (excepto explota...,7
6,114119,"Pesca y captura de otros peces, crustáceos, mo...",,114119,"Pesca y captura de peces, crustáceos, moluscos...",7
7,311350,Elaboración de chocolate y productos de chocolate,,311320,Elaboración de chocolate y productos de chocol...,7
8,311350,Elaboración de chocolate y productos de chocolate,,311330,Elaboración de productos de chocolate a partir...,7
9,311412,Congelación de guisos y otros alimentos prepar...,,311412,Congelación de alimentos preparados,7


In [ ]:
# ============================================================
# FASE 2 - PASO 5S
# VALIDAR CÓDIGOS EXTRAÍDOS
# ============================================================

patron_codigo_6 = r"^\d{6}$"

validos_2013 = (
    correspondencias_2013_2007["SCIAN_2013"]
    .astype(str)
    .str.match(patron_codigo_6)
)

validos_2007 = (
    correspondencias_2013_2007["SCIAN_2007"]
    .astype(str)
    .str.match(patron_codigo_6)
)

print("Total de registros:",
      len(correspondencias_2013_2007))

print(
    "Código SCIAN 2013 válido:",
    validos_2013.sum()
)

print(
    "Código SCIAN 2007 válido:",
    validos_2007.sum()
)

print(
    "Registros con ambos códigos válidos:",
    (validos_2013 & validos_2007).sum()
)

print(
    "Registros problemáticos:",
    (~(validos_2013 & validos_2007)).sum()
)

Total de registros: 75
Código SCIAN 2013 válido: 75
Código SCIAN 2007 válido: 75
Registros con ambos códigos válidos: 75
Registros problemáticos: 0


In [ ]:
# ============================================================
# FASE 2 - PASO 5T
# REVISAR DUPLICADOS Y CAMPO "PARTE"
# ============================================================

print("Duplicados exactos:")
print(
    correspondencias_2013_2007.duplicated(
        subset=[
            "SCIAN_2013",
            "Parte",
            "SCIAN_2007"
        ]
    ).sum()
)

print("\nValores del campo Parte:")
display(
    correspondencias_2013_2007["Parte"]
    .value_counts(dropna=False)
    .head(20)
)

Duplicados exactos:
0

Valores del campo Parte:


Parte
         34
p 2/2    16
p 1/2    15
p 1/6     1
p 2/5     1
p 2/6     1
p 3/5     1
p 3/6     1
p 4/5     1
p 4/6     1
p 5/5     1
p 5/6     1
p 6/6     1
Name: count, dtype: int64

In [ ]:
# ============================================================
# FASE 2 - PASO 5U
# GUARDAR EXTRACCIÓN OFICIAL 2013 -> 2007
# ============================================================

ruta_correspondencia_2013_2007 = (
    HOMOLOGACION /
    "CORRESPONDENCIA_OFICIAL_SCIAN_2013_2007_EXTRAIDA.csv"
)

correspondencias_2013_2007.to_csv(
    ruta_correspondencia_2013_2007,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Correspondencia 2013→2007 guardada:")
print(ruta_correspondencia_2013_2007)

✓ Correspondencia 2013→2007 guardada:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN\CORRESPONDENCIA_OFICIAL_SCIAN_2013_2007_EXTRAIDA.csv


In [ ]:
# ============================================================
# FASE 2 - PASO 5T
# REVISAR DUPLICADOS Y CAMPO PARTE
# ============================================================

print("Duplicados exactos:")

print(
    correspondencias_2013_2007.duplicated(
        subset=[
            "SCIAN_2013",
            "Parte",
            "SCIAN_2007"
        ]
    ).sum()
)

print("\nValores del campo Parte:")

display(
    correspondencias_2013_2007["Parte"]
    .value_counts(dropna=False)
    .head(20)
)

Duplicados exactos:
0

Valores del campo Parte:


Parte
         34
p 2/2    16
p 1/2    15
p 1/6     1
p 2/5     1
p 2/6     1
p 3/5     1
p 3/6     1
p 4/5     1
p 4/6     1
p 5/5     1
p 5/6     1
p 6/6     1
Name: count, dtype: int64

In [ ]:
# ============================================================
# FASE 2 - CONTROL FINAL DE LA EXTRACCIÓN 2013 -> 2007
# ============================================================

validos_2013 = (
    correspondencias_2013_2007["SCIAN_2013"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

validos_2007 = (
    correspondencias_2013_2007["SCIAN_2007"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

print("Correspondencias extraídas:",
      len(correspondencias_2013_2007))

print("Ambos códigos válidos:",
      (validos_2013 & validos_2007).sum())

print("Problemáticos:",
      (~(validos_2013 & validos_2007)).sum())

Correspondencias extraídas: 75
Ambos códigos válidos: 75
Problemáticos: 0


In [ ]:
# ============================================================
# FASE 2 - PASO 6A
# GUARDAR CORRESPONDENCIA OFICIAL EXTRAÍDA 2013 -> 2007
# ============================================================

ruta_correspondencia_2013_2007 = (
    HOMOLOGACION /
    "CORRESPONDENCIA_OFICIAL_SCIAN_2013_2007_EXTRAIDA.csv"
)

correspondencias_2013_2007.to_csv(
    ruta_correspondencia_2013_2007,
    index=False,
    encoding="utf-8-sig"
)

print("✓ Correspondencia oficial cruda guardada.")

✓ Correspondencia oficial cruda guardada.


In [ ]:
# ============================================================
# FASE 2 - PASO 6B
# CLASIFICAR CARDINALIDAD DE LAS CORRESPONDENCIAS
# ============================================================

df = correspondencias_2013_2007.copy()

df["Parte"] = (
    df["Parte"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Cuántos códigos 2007 distintos se relacionan con cada código 2013
n_2007_por_2013 = (
    df.groupby("SCIAN_2013")["SCIAN_2007"]
    .nunique()
)

# Cuántos códigos 2013 distintos se relacionan con cada código 2007
n_2013_por_2007 = (
    df.groupby("SCIAN_2007")["SCIAN_2013"]
    .nunique()
)

df["N_2007_por_2013"] = (
    df["SCIAN_2013"]
    .map(n_2007_por_2013)
)

df["N_2013_por_2007"] = (
    df["SCIAN_2007"]
    .map(n_2013_por_2007)
)

def clasificar_cardinalidad(fila):

    a = fila["N_2007_por_2013"]
    b = fila["N_2013_por_2007"]

    if a == 1 and b == 1:
        return "1 a 1"

    elif a > 1 and b == 1:
        return "N a 1 (varios 2007 forman un 2013)"

    elif a == 1 and b > 1:
        return "1 a N (un 2007 se divide en varios 2013)"

    else:
        return "N a N"

df["Cardinalidad"] = df.apply(
    clasificar_cardinalidad,
    axis=1
)

df["Correspondencia_parcial"] = (
    df["Parte"] != ""
)

display(
    df[
        [
            "SCIAN_2013",
            "SCIAN_2007",
            "Parte",
            "Cardinalidad",
            "Correspondencia_parcial"
        ]
    ].head(30)
)

,SCIAN_2013,SCIAN_2007,Parte,Cardinalidad,Correspondencia_parcial
0,111214,111214,p 1/2,1 a N (un 2007 se divide en varios 2013),True
1,111218,111214,p 2/2,1 a N (un 2007 se divide en varios 2013),True
2,111942,111942,,1 a 1,False
3,112512,112512,,N a 1 (varios 2007 forman un 2013),False
4,112512,112999,p 1/2,N a N,True
5,112999,112999,p 2/2,1 a N (un 2007 se divide en varios 2013),True
6,114119,114119,,1 a 1,False
7,311350,311320,,N a 1 (varios 2007 forman un 2013),False
8,311350,311330,,N a 1 (varios 2007 forman un 2013),False
9,311412,311412,,1 a 1,False


In [ ]:
# ============================================================
# FASE 2 - PASO 6C
# RESUMEN DE TIPOS DE CORRESPONDENCIA
# ============================================================

print("CARDINALIDADES:")

display(
    df["Cardinalidad"]
    .value_counts()
    .reset_index()
)

print("\nCORRESPONDENCIAS PARCIALES:")

display(
    df["Correspondencia_parcial"]
    .value_counts()
    .reset_index()
)

CARDINALIDADES:


,Cardinalidad,count
0,1 a 1,23
1,1 a N (un 2007 se divide en varios 2013),22
2,N a N,21
3,N a 1 (varios 2007 forman un 2013),9



CORRESPONDENCIAS PARCIALES:


,Correspondencia_parcial,count
0,True,41
1,False,34


In [ ]:
# ============================================================
# FASE 2 - PASO 6D
# GUARDAR CORRESPONDENCIAS CLASIFICADAS
# ============================================================

correspondencias_2013_2007_clasificadas = df.copy()

correspondencias_2013_2007_clasificadas.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Correspondencias clasificadas guardadas.")

✓ Correspondencias clasificadas guardadas.


In [ ]:
# ============================================================
# RECUPERAR CÓDIGOS ÚNICOS POR VERSIÓN SCIAN
# ============================================================

codigos_por_version = pd.read_csv(
    HOMOLOGACION / "CODIGOS_UNICOS_POR_VERSION_SCIAN.csv",
    dtype=str,
    encoding="utf-8-sig"
)

print("Archivo recuperado correctamente.")
print("Registros:", len(codigos_por_version))

display(
    codigos_por_version
    .groupby("Version_SCIAN")
    .size()
    .reset_index(name="Número_de_códigos")
)

NameError: name 'pd' is not defined

In [ ]:
# ============================================================
# FASE 2 - PASO 6E
# CÓDIGOS OBSERVADOS EN MORELIA: SCIAN 2007 Y SCIAN 2013
# ============================================================

codigos_2007 = set(
    codigos_por_version.loc[
        codigos_por_version["Version_SCIAN"] == "SCIAN 2007",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

codigos_2013 = set(
    codigos_por_version.loc[
        codigos_por_version["Version_SCIAN"] == "SCIAN 2013",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

print("Códigos observados SCIAN 2007:", len(codigos_2007))
print("Códigos observados SCIAN 2013:", len(codigos_2013))

print(
    "Códigos exactamente iguales presentes en ambas versiones:",
    len(codigos_2007 & codigos_2013)
)

NameError: name 'codigos_por_version' is not defined

In [ ]:
# ============================================================
# RECUPERAR ENTORNO DE TRABAJO DENUE
# Ejecutar después de reiniciar el kernel
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
import zipfile
import pdfplumber

# Carpeta base del proyecto
BASE = Path.cwd()

# Rutas principales
ORIGINALES = BASE / "00_ORIGINALES"
MORELIA = BASE / "01_MORELIA_LIMPIO"
CONTEOS = BASE / "02_CONTEOS_SCIAN"
HOMOLOGACION = BASE / "03_HOMOLOGACION_SCIAN"
COMPARACIONES = BASE / "04_COMPARACIONES"
RESULTADOS = BASE / "05_RESULTADOS"

print("✓ Entorno recuperado.")
print("Carpeta base:")
print(BASE)

✓ Entorno recuperado.
Carpeta base:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS


In [ ]:
# ============================================================
# DIAGNÓSTICO DE RUTA - HOMOLOGACIÓN SCIAN
# ============================================================

print("Carpeta actual:")
print(Path.cwd())

print("\nRuta HOMOLOGACION:")
print(HOMOLOGACION)

print("\n¿Existe la carpeta HOMOLOGACION?")
print(HOMOLOGACION.exists())

print("\nArchivos que realmente existen dentro:")

if HOMOLOGACION.exists():
    for archivo in HOMOLOGACION.iterdir():
        print("-", archivo.name)

Carpeta actual:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS

Ruta HOMOLOGACION:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN

¿Existe la carpeta HOMOLOGACION?
True

Archivos que realmente existen dentro:
- CORRESPONDENCIA_OFICIAL_SCIAN_2013_2007_EXTRAIDA.csv
- CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA.csv
- SCIAN_2013_2007_OFICIAL.pdf
- SCIAN_2018_2013_OFICIAL.pdf
- SCIAN_2023_2018_OFICIAL.pdf
- TRANSICIONES_SCIAN_2013_2018_OBSERVADAS.csv


In [ ]:
# ============================================================
# RECUPERACIÓN COMPLETA DE LA FASE 2 - HOMOLOGACIÓN SCIAN
# Ejecutar después de reiniciar el kernel
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
import zipfile
import pdfplumber

# ------------------------------------------------------------
# 1. RECUPERAR RUTAS DEL PROYECTO
# ------------------------------------------------------------

BASE = Path.cwd()

ORIGINALES = BASE / "00_ORIGINALES"
MORELIA = BASE / "01_MORELIA_LIMPIO"
CONTEOS = BASE / "02_CONTEOS_SCIAN"
HOMOLOGACION = BASE / "03_HOMOLOGACION_SCIAN"
COMPARACIONES = BASE / "04_COMPARACIONES"
RESULTADOS = BASE / "05_RESULTADOS"

print("Carpeta base:")
print(BASE)

# ------------------------------------------------------------
# 2. DEFINIR AÑOS Y VERSIONES SCIAN
# ------------------------------------------------------------

años_version = {
    2010: "SCIAN 2007",
    2011: "SCIAN 2007",
    2012: "SCIAN 2007",
    2013: "SCIAN 2007",

    2015: "SCIAN 2013",
    2016: "SCIAN 2013",
    2017: "SCIAN 2013",

    2018: "SCIAN 2018",
    2019: "SCIAN 2018",
    2020: "SCIAN 2018",
    2021: "SCIAN 2018",
    2022: "SCIAN 2018",
    2023: "SCIAN 2018",

    2024: "SCIAN 2023",
    2025: "SCIAN 2023",
    2026: "SCIAN 2023"
}

sectores_servicios = [
    "48", "49", "51", "52", "53", "54",
    "56", "61", "62", "71", "72", "81"
]

# ------------------------------------------------------------
# 3. RECONSTRUIR INVENTARIO HISTÓRICO
# ------------------------------------------------------------

filas_inventario = []
faltantes = []

for año, version in años_version.items():

    ruta = CONTEOS / f"CONTEO_SCIAN_{año}_MORELIA.csv"

    if not ruta.exists():
        faltantes.append(ruta.name)
        continue

    df = pd.read_csv(
        ruta,
        dtype=str,
        encoding="utf-8-sig"
    )

    candidatos_codigo = [
        "SCIAN6",
        "codigo_act",
        "Código de la clase de actividad",
        "Código de la clase de actividad SCIAN"
    ]

    col_codigo = next(
        (c for c in candidatos_codigo if c in df.columns),
        None
    )

    if col_codigo is None:
        raise ValueError(
            f"No encontré columna SCIAN en {ruta.name}"
        )

    col_conteo = "Número de establecimientos"

    if col_conteo not in df.columns:
        raise ValueError(
            f"No encontré '{col_conteo}' en {ruta.name}"
        )

    df["SCIAN6_STD"] = (
        df[col_codigo]
        .astype(str)
        .str.strip()
        .str.zfill(6)
    )

    df["Sector_STD"] = (
        df["SCIAN6_STD"].str[:2]
    )

    df["Establecimientos_STD"] = pd.to_numeric(
        df[col_conteo],
        errors="coerce"
    ).fillna(0).astype(int)

    # Solo comercio al por menor + servicios
    df_interes = df[
        (df["Sector_STD"] == "46")
        |
        (df["Sector_STD"].isin(sectores_servicios))
    ].copy()

    for _, fila in df_interes.iterrows():

        filas_inventario.append({
            "Año": año,
            "Version_SCIAN": version,
            "SCIAN_original": fila["SCIAN6_STD"],
            "Sector": fila["Sector_STD"],
            "Establecimientos": fila["Establecimientos_STD"]
        })

inventario_scian = pd.DataFrame(filas_inventario)

print("\nRegistros año-código reconstruidos:")
print(len(inventario_scian))

if faltantes:
    print("\nATENCIÓN - archivos faltantes:")
    for archivo in faltantes:
        print("-", archivo)
else:
    print("✓ Todos los conteos anuales fueron encontrados.")

# ------------------------------------------------------------
# 4. RECONSTRUIR CÓDIGOS ÚNICOS POR VERSIÓN
# ------------------------------------------------------------

codigos_por_version = (
    inventario_scian[
        ["Version_SCIAN", "SCIAN_original", "Sector"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Version_SCIAN", "Sector", "SCIAN_original"]
    )
    .reset_index(drop=True)
)

resumen_codigos = (
    codigos_por_version
    .groupby("Version_SCIAN")
    .size()
    .reset_index(name="Número_de_códigos")
)

print("\nCódigos únicos por versión:")
display(resumen_codigos)

# ------------------------------------------------------------
# 5. GUARDAR PARA NO VOLVER A PERDERLOS
# ------------------------------------------------------------

inventario_scian.to_csv(
    HOMOLOGACION /
    "INVENTARIO_HISTORICO_SCIAN_COMERCIO_SERVICIOS.csv",
    index=False,
    encoding="utf-8-sig"
)

codigos_por_version.to_csv(
    HOMOLOGACION /
    "CODIGOS_UNICOS_POR_VERSION_SCIAN.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Inventario histórico guardado.")
print("✓ Códigos únicos por versión guardados.")

# ------------------------------------------------------------
# 6. RECUPERAR CORRESPONDENCIA 2013-2007 YA CLASIFICADA
# ------------------------------------------------------------

correspondencias_2013_2007_clasificadas = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA.csv",
    dtype=str,
    encoding="utf-8-sig"
)

print(
    "\nCorrespondencias 2013-2007 recuperadas:",
    len(correspondencias_2013_2007_clasificadas)
)

print("\n✓ FASE 2 recuperada correctamente.")

Carpeta base:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS

Registros año-código reconstruidos:
6435
✓ Todos los conteos anuales fueron encontrados.

Códigos únicos por versión:


,Version_SCIAN,Número_de_códigos
0,SCIAN 2007,399
1,SCIAN 2013,407
2,SCIAN 2018,442
3,SCIAN 2023,421



✓ Inventario histórico guardado.
✓ Códigos únicos por versión guardados.

Correspondencias 2013-2007 recuperadas: 75

✓ FASE 2 recuperada correctamente.


# INICIO DE SESIÓN / RECUPERACIÓN DEL ENTORNO

In [ ]:
# ============================================================
# INICIO DE SESIÓN / RECUPERACIÓN DEL ENTORNO
# Ejecutar SIEMPRE al abrir el notebook
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
import zipfile
import pdfplumber

BASE = Path.cwd()

ORIGINALES = BASE / "00_ORIGINALES"
MORELIA = BASE / "01_MORELIA_LIMPIO"
CONTEOS = BASE / "02_CONTEOS_SCIAN"
HOMOLOGACION = BASE / "03_HOMOLOGACION_SCIAN"
COMPARACIONES = BASE / "04_COMPARACIONES"
RESULTADOS = BASE / "05_RESULTADOS"

# Recuperar resumen histórico
resumen_historico = pd.read_csv(
    RESULTADOS / "RESUMEN_HISTORICO_DENUE.csv",
    encoding="utf-8-sig"
)

# Recuperar inventario SCIAN
inventario_scian = pd.read_csv(
    HOMOLOGACION / "INVENTARIO_HISTORICO_SCIAN_COMERCIO_SERVICIOS.csv",
    dtype=str,
    encoding="utf-8-sig"
)

# Recuperar códigos únicos por versión
codigos_por_version = pd.read_csv(
    HOMOLOGACION / "CODIGOS_UNICOS_POR_VERSION_SCIAN.csv",
    dtype=str,
    encoding="utf-8-sig"
)

# Recuperar correspondencia SCIAN 2013-2007 clasificada
correspondencias_2013_2007_clasificadas = pd.read_csv(
    HOMOLOGACION / "CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA.csv",
    dtype=str,
    encoding="utf-8-sig"
)

print("✓ Entorno recuperado correctamente.")
print("BASE:", BASE)
print("Años en resumen:", len(resumen_historico))
print("Inventario año-código:", len(inventario_scian))
print("Códigos únicos:", len(codigos_por_version))
print("Correspondencias 2013-2007:",
      len(correspondencias_2013_2007_clasificadas))

✓ Entorno recuperado correctamente.
BASE: c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS
Años en resumen: 16
Inventario año-código: 6435
Códigos únicos: 1669
Correspondencias 2013-2007: 75


In [ ]:
# ============================================================
# FASE 2 - PASO 6E
# CÓDIGOS OBSERVADOS EN MORELIA: SCIAN 2007 Y SCIAN 2013
# ============================================================

codigos_2007 = set(
    codigos_por_version.loc[
        codigos_por_version["Version_SCIAN"] == "SCIAN 2007",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

codigos_2013 = set(
    codigos_por_version.loc[
        codigos_por_version["Version_SCIAN"] == "SCIAN 2013",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

print("Códigos observados SCIAN 2007:", len(codigos_2007))
print("Códigos observados SCIAN 2013:", len(codigos_2013))

print(
    "Códigos con el mismo número presentes en ambas versiones:",
    len(codigos_2007 & codigos_2013)
)

Códigos observados SCIAN 2007: 399
Códigos observados SCIAN 2013: 407
Códigos con el mismo número presentes en ambas versiones: 362


In [ ]:
# ============================================================
# FASE 2 - PASO 6F
# CRUZAR CÓDIGOS OBSERVADOS CON LA TABLA OFICIAL
# ============================================================

codigos_oficiales_2007 = set(
    correspondencias_2013_2007_clasificadas["SCIAN_2007"]
    .astype(str)
    .str.strip()
)

codigos_oficiales_2013 = set(
    correspondencias_2013_2007_clasificadas["SCIAN_2013"]
    .astype(str)
    .str.strip()
)

print(
    "2007 observados que aparecen explícitamente en la tabla oficial:",
    len(codigos_2007 & codigos_oficiales_2007)
)

print(
    "2013 observados que aparecen explícitamente en la tabla oficial:",
    len(codigos_2013 & codigos_oficiales_2013)
)

print(
    "2007 observados que NO aparecen explícitamente:",
    len(codigos_2007 - codigos_oficiales_2007)
)

print(
    "2013 observados que NO aparecen explícitamente:",
    len(codigos_2013 - codigos_oficiales_2013)
)

2007 observados que aparecen explícitamente en la tabla oficial: 18
2013 observados que aparecen explícitamente en la tabla oficial: 25
2007 observados que NO aparecen explícitamente: 381
2013 observados que NO aparecen explícitamente: 382


In [ ]:
# ============================================================
# FASE 2 - PASO 6G
# DIAGNÓSTICO PRELIMINAR SCIAN 2007 -> 2013
# ============================================================

diagnostico_2007 = (
    codigos_por_version[
        codigos_por_version["Version_SCIAN"] == "SCIAN 2007"
    ]
    .copy()
)

def diagnosticar_2007(codigo):

    codigo = str(codigo).strip()

    if codigo in codigos_oficiales_2007:
        return "Cambio explícito en tabla oficial"

    elif codigo in codigos_2013:
        return "Mismo código - candidato sin cambio"

    else:
        return "Revisión necesaria"

diagnostico_2007["Diagnostico_2007_2013"] = (
    diagnostico_2007["SCIAN_original"]
    .apply(diagnosticar_2007)
)

display(
    diagnostico_2007[
        "Diagnostico_2007_2013"
    ]
    .value_counts()
    .reset_index()
)

,Diagnostico_2007_2013,count
0,Mismo código - candidato sin cambio,352
1,Revisión necesaria,29
2,Cambio explícito en tabla oficial,18


In [ ]:
# ============================================================
# FASE 2 - PASO 6H
# DIAGNÓSTICO PRELIMINAR SCIAN 2013 FRENTE A 2007
# ============================================================

diagnostico_2013 = (
    codigos_por_version[
        codigos_por_version["Version_SCIAN"] == "SCIAN 2013"
    ]
    .copy()
)

def diagnosticar_2013(codigo):

    codigo = str(codigo).strip()

    if codigo in codigos_oficiales_2013:
        return "Cambio explícito en tabla oficial"

    elif codigo in codigos_2007:
        return "Mismo código - candidato sin cambio"

    else:
        return "Revisión necesaria"

diagnostico_2013["Diagnostico_2007_2013"] = (
    diagnostico_2013["SCIAN_original"]
    .apply(diagnosticar_2013)
)

display(
    diagnostico_2013[
        "Diagnostico_2007_2013"
    ]
    .value_counts()
    .reset_index()
)

,Diagnostico_2007_2013,count
0,Mismo código - candidato sin cambio,352
1,Revisión necesaria,30
2,Cambio explícito en tabla oficial,25


In [ ]:
# ============================================================
# RECUPERAR HERRAMIENTAS PARA EXTRAER TABLAS SCIAN 2007-2013
# ============================================================

import re
import pandas as pd
import pdfplumber

# Ruta del PDF oficial que YA está guardado
pdf_2007_2013 = (
    HOMOLOGACION / "SCIAN_2013_2007_OFICIAL.pdf"
)

print("PDF encontrado:", pdf_2007_2013.exists())
print(pdf_2007_2013)


# ------------------------------------------------------------
# FUNCIÓN 1. Extraer líneas por posición dentro del PDF
# ------------------------------------------------------------

def extraer_lineas_scian(pagina):

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    lineas = {}

    for palabra in palabras:

        top = round(palabra["top"] / 3) * 3

        lineas.setdefault(top, []).append(palabra)

    resultados = []

    for top in sorted(lineas):

        palabras_linea = sorted(
            lineas[top],
            key=lambda w: w["x0"]
        )

        columnas = {
            "codigo_2013": [],
            "nombre_2013": [],
            "parte": [],
            "codigo_2007": [],
            "nombre_2007": []
        }

        for w in palabras_linea:

            x = w["x0"]
            txt = w["text"].strip()

            # Ignorar marcas laterales
            if x >= 565:
                continue

            if x < 90:
                columnas["codigo_2013"].append(txt)

            elif x < 275:
                columnas["nombre_2013"].append(txt)

            elif x < 317:
                columnas["parte"].append(txt)

            elif x < 360:
                columnas["codigo_2007"].append(txt)

            else:
                columnas["nombre_2007"].append(txt)

        fila = {
            clave: " ".join(valor).strip()
            for clave, valor in columnas.items()
        }

        if any(fila.values()):
            fila["top"] = top
            resultados.append(fila)

    return resultados


# ------------------------------------------------------------
# FUNCIÓN 2. Reconstruir registros lógicos
# ------------------------------------------------------------

patron_6 = re.compile(r"^\d{6}$")
patron_sector = re.compile(r"^(\d{2}|\d{2}-\d{2})$")

def limpiar_texto(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def reconstruir_correspondencias(df_lineas):

    registros = []

    codigo_2013_activo = ""
    nombre_2013_activo = ""
    registro_actual = None

    for _, fila in df_lineas.iterrows():

        c13 = limpiar_texto(fila.get("codigo_2013", ""))
        n13 = limpiar_texto(fila.get("nombre_2013", ""))
        parte = limpiar_texto(fila.get("parte", ""))
        c07 = limpiar_texto(fila.get("codigo_2007", ""))
        n07 = limpiar_texto(fila.get("nombre_2007", ""))

        if patron_sector.match(c13):
            codigo_2013_activo = ""
            nombre_2013_activo = ""
            registro_actual = None
            continue

        if patron_6.match(c13):

            codigo_2013_activo = c13
            nombre_2013_activo = n13

        elif (
            codigo_2013_activo
            and n13
            and not c07
            and not parte
        ):
            nombre_2013_activo = limpiar_texto(
                nombre_2013_activo + " " + n13
            )

        if patron_6.match(c07):

            registro_actual = {
                "SCIAN_2013": codigo_2013_activo,
                "Nombre_2013": nombre_2013_activo,
                "Parte": parte,
                "SCIAN_2007": c07,
                "Nombre_2007": n07
            }

            registros.append(registro_actual)

        elif registro_actual is not None and n07:

            registro_actual["Nombre_2007"] = limpiar_texto(
                registro_actual["Nombre_2007"]
                + " "
                + n07
            )

    return pd.DataFrame(registros)


print("\n✓ Herramientas de extracción recuperadas.")

PDF encontrado: True
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN\SCIAN_2013_2007_OFICIAL.pdf

✓ Herramientas de extracción recuperadas.


In [ ]:
# ============================================================
# FASE 2 - PASO 7A
# EXTRAER TABLA INVERSA SCIAN 2007 -> SCIAN 2013
# PÁGINAS PDF 13 A 16
# ============================================================

paginas_datos_2007_2013 = [13, 14, 15, 16]

registros_inversos = []

with pdfplumber.open(pdf_2007_2013) as pdf:

    for pagina_pdf in paginas_datos_2007_2013:

        pagina = pdf.pages[pagina_pdf - 1]

        lineas_pagina = extraer_lineas_scian(pagina)

        df_lineas = pd.DataFrame(lineas_pagina)

        # La función cree que izquierda=2013 y derecha=2007,
        # pero en estas páginas es al revés.
        temporal = reconstruir_correspondencias(df_lineas)

        temporal["Pagina_PDF"] = pagina_pdf

        registros_inversos.append(temporal)

tabla_inversa_temporal = pd.concat(
    registros_inversos,
    ignore_index=True
)

# Corregir nombres de columnas
correspondencias_2007_2013 = (
    tabla_inversa_temporal.rename(
        columns={
            "SCIAN_2013": "SCIAN_2007",
            "Nombre_2013": "Nombre_2007",
            "SCIAN_2007": "SCIAN_2013",
            "Nombre_2007": "Nombre_2013"
        }
    )
)

print(
    "Correspondencias inversas extraídas:",
    len(correspondencias_2007_2013)
)

display(correspondencias_2007_2013.head(30))

Correspondencias inversas extraídas: 76


,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013,Pagina_PDF
0,111214,Cultivo de melón o sandía,,111214,Cultivo de melón,13
1,111214,Cultivo de melón o sandía,,111218,Cultivo de sandía,13
2,111942,Cultivo de pastos y zacates,,111942,Cultivo de pastos,13
3,112512,"Piscicultura y otra acuicultura,",p 1/2,112512,"Piscicultura y otra acuicultura, excepto camar...",13
4,112999,Explotación de otros animales,,112999,Explotación de otros animales,13
5,112999,Explotación de otros animales,p 2/2,112512,"Piscicultura y otra acuicultura, excepto camar...",13
6,114119,"Pesca y captura de peces, crustáceos,",,114119,"Pesca y captura de otros peces, crustáceos, mo...",13
7,311320,Elaboración de chocolate y productos,p 1/2,311350,Elaboración de chocolate y productos de chocol...,13
8,311330,Elaboración de productos de chocolate a partir,p 2/2,311350,Elaboración de chocolate y productos de chocol...,13
9,311412,,,311412,,13


In [ ]:
# ============================================================
# FASE 2 - PASO 7B
# VALIDAR TABLA INVERSA
# ============================================================

valido_2007 = (
    correspondencias_2007_2013["SCIAN_2007"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

valido_2013 = (
    correspondencias_2007_2013["SCIAN_2013"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

print(
    "Correspondencias extraídas:",
    len(correspondencias_2007_2013)
)

print(
    "Ambos códigos válidos:",
    (valido_2007 & valido_2013).sum()
)

print(
    "Problemáticos:",
    (~(valido_2007 & valido_2013)).sum()
)

print(
    "Duplicados exactos:",
    correspondencias_2007_2013.duplicated(
        subset=[
            "SCIAN_2007",
            "Parte",
            "SCIAN_2013"
        ]
    ).sum()
)

print("\nValores de Parte:")
display(
    correspondencias_2007_2013["Parte"]
    .fillna("")
    .value_counts()
    .head(20)
)

Correspondencias extraídas: 76
Ambos códigos válidos: 76
Problemáticos: 0
Duplicados exactos: 0

Valores de Parte:


Parte
         46
p 1/2    13
p 2/2    13
p 1/4     1
p 2/4     1
p 3/4     1
p 4/4     1
Name: count, dtype: int64

In [ ]:
# ============================================================
# FASE 2 - PASO 7C
# GUARDAR TABLA INVERSA SCIAN 2007 -> 2013
# ============================================================

correspondencias_2007_2013.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_OFICIAL_SCIAN_2007_2013_EXTRAIDA.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Tabla inversa 2007→2013 guardada.")

✓ Tabla inversa 2007→2013 guardada.


In [ ]:
# ============================================================
# FASE 2 - PASO 7D
# COMPARAR PARES DE CÓDIGOS EN AMBOS SENTIDOS
# ============================================================

pares_directos = set(
    zip(
        correspondencias_2013_2007_clasificadas["SCIAN_2007"]
        .astype(str).str.strip(),

        correspondencias_2013_2007_clasificadas["SCIAN_2013"]
        .astype(str).str.strip()
    )
)

pares_inversos = set(
    zip(
        correspondencias_2007_2013["SCIAN_2007"]
        .astype(str).str.strip(),

        correspondencias_2007_2013["SCIAN_2013"]
        .astype(str).str.strip()
    )
)

print("Pares únicos tabla 2013→2007:", len(pares_directos))
print("Pares únicos tabla 2007→2013:", len(pares_inversos))

print("\nPares presentes en ambas tablas:")
print(len(pares_directos & pares_inversos))

print("\nSolo en tabla 2013→2007:")
print(len(pares_directos - pares_inversos))

print("\nSolo en tabla 2007→2013:")
print(len(pares_inversos - pares_directos))

Pares únicos tabla 2013→2007: 75
Pares únicos tabla 2007→2013: 76

Pares presentes en ambas tablas:
75

Solo en tabla 2013→2007:
0

Solo en tabla 2007→2013:
1


In [ ]:
# ============================================================
# FASE 2 - PASO 7E
# MOSTRAR PARES ASIMÉTRICOS
# ============================================================

solo_directa = sorted(
    pares_directos - pares_inversos
)

solo_inversa = sorted(
    pares_inversos - pares_directos
)

print("SOLO EN 2013→2007:")
for par in solo_directa:
    print(par)

print("\nSOLO EN 2007→2013:")
for par in solo_inversa:
    print(par)

SOLO EN 2013→2007:

SOLO EN 2007→2013:
('812320', '465915')


In [ ]:
# ============================================================
# FASE 2 - PASO 7F
# CONSTRUIR MAPA CONSOLIDADO 2007 ↔ 2013
# ============================================================

todos_pares = sorted(
    pares_directos | pares_inversos
)

mapa_2007_2013 = pd.DataFrame(
    todos_pares,
    columns=[
        "SCIAN_2007",
        "SCIAN_2013"
    ]
)

mapa_2007_2013["En_tabla_2013_2007"] = (
    mapa_2007_2013.apply(
        lambda x:
        (
            x["SCIAN_2007"],
            x["SCIAN_2013"]
        ) in pares_directos,
        axis=1
    )
)

mapa_2007_2013["En_tabla_2007_2013"] = (
    mapa_2007_2013.apply(
        lambda x:
        (
            x["SCIAN_2007"],
            x["SCIAN_2013"]
        ) in pares_inversos,
        axis=1
    )
)

mapa_2007_2013["Fuente_relacion"] = (
    mapa_2007_2013.apply(
        lambda x:
        "Ambas tablas"
        if (
            x["En_tabla_2013_2007"]
            and x["En_tabla_2007_2013"]
        )
        else (
            "Solo tabla 2013→2007"
            if x["En_tabla_2013_2007"]
            else "Solo tabla 2007→2013"
        ),
        axis=1
    )
)

display(
    mapa_2007_2013["Fuente_relacion"]
    .value_counts()
    .reset_index()
)

display(mapa_2007_2013.head(30))

,Fuente_relacion,count
0,Ambas tablas,75
1,Solo tabla 2007→2013,1


,SCIAN_2007,SCIAN_2013,En_tabla_2013_2007,En_tabla_2007_2013,Fuente_relacion
0,111214,111214,True,True,Ambas tablas
1,111214,111218,True,True,Ambas tablas
2,111942,111942,True,True,Ambas tablas
3,112512,112512,True,True,Ambas tablas
4,112999,112512,True,True,Ambas tablas
5,112999,112999,True,True,Ambas tablas
6,114119,114119,True,True,Ambas tablas
7,311320,311350,True,True,Ambas tablas
8,311330,311350,True,True,Ambas tablas
9,311412,311412,True,True,Ambas tablas


In [ ]:
# ============================================================
# FASE 2 - PASO 7G
# CORREGIR ARTEFACTO DE EXTRACCIÓN "YYYYYY"
# ============================================================

antes = len(correspondencias_2007_2013)

correspondencias_2007_2013 = (
    correspondencias_2007_2013[
        ~(
            (correspondencias_2007_2013["SCIAN_2007"].astype(str) == "812320")
            &
            (correspondencias_2007_2013["SCIAN_2013"].astype(str) == "465915")
        )
    ]
    .copy()
    .reset_index(drop=True)
)

despues = len(correspondencias_2007_2013)

print("Registros antes:", antes)
print("Registros después:", despues)
print("Registros eliminados:", antes - despues)

Registros antes: 76
Registros después: 75
Registros eliminados: 1


In [ ]:
# ============================================================
# FASE 2 - PASO 7H
# VOLVER A COMPARAR AMBAS TABLAS
# ============================================================

pares_directos = set(
    zip(
        correspondencias_2013_2007_clasificadas["SCIAN_2007"]
        .astype(str).str.strip(),

        correspondencias_2013_2007_clasificadas["SCIAN_2013"]
        .astype(str).str.strip()
    )
)

pares_inversos = set(
    zip(
        correspondencias_2007_2013["SCIAN_2007"]
        .astype(str).str.strip(),

        correspondencias_2007_2013["SCIAN_2013"]
        .astype(str).str.strip()
    )
)

print("Pares tabla 2013→2007:", len(pares_directos))
print("Pares tabla 2007→2013:", len(pares_inversos))
print("Pares comunes:", len(pares_directos & pares_inversos))
print("Solo directa:", len(pares_directos - pares_inversos))
print("Solo inversa:", len(pares_inversos - pares_directos))

Pares tabla 2013→2007: 75
Pares tabla 2007→2013: 75
Pares comunes: 75
Solo directa: 0
Solo inversa: 0


In [ ]:
correspondencias_2007_2013.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_OFICIAL_SCIAN_2007_2013_EXTRAIDA.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Tabla inversa corregida y guardada.")

✓ Tabla inversa corregida y guardada.


In [ ]:
# ============================================================
# FASE 2 - PASO 7I
# PARSER MEJORADO PARA MARCADORES ESPECIALES
# ============================================================

patron_6 = re.compile(r"^\d{6}$")
patron_sector = re.compile(r"^(\d{2}|\d{2}-\d{2})$")
patron_marcador = re.compile(r"^[A-Z]{6}$")


def limpiar_texto(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def reconstruir_correspondencias(df_lineas):

    registros = []
    marcadores_especiales = []

    codigo_izq_activo = ""
    nombre_izq_activo = ""
    registro_actual = None

    for _, fila in df_lineas.iterrows():

        c_izq = limpiar_texto(
            fila.get("codigo_2013", "")
        )

        n_izq = limpiar_texto(
            fila.get("nombre_2013", "")
        )

        parte = limpiar_texto(
            fila.get("parte", "")
        )

        c_der = limpiar_texto(
            fila.get("codigo_2007", "")
        )

        n_der = limpiar_texto(
            fila.get("nombre_2007", "")
        )

        # -----------------------------------------
        # 1. Encabezados de sector
        # -----------------------------------------
        if patron_sector.match(c_izq):

            codigo_izq_activo = ""
            nombre_izq_activo = ""
            registro_actual = None
            continue

        # -----------------------------------------
        # 2. Marcadores especiales: YYYYYY, etc.
        # No arrastrar el código anterior
        # -----------------------------------------
        if (
            patron_marcador.match(c_izq)
            and not patron_6.match(c_izq)
        ):

            codigo_izq_activo = ""
            nombre_izq_activo = ""
            registro_actual = None

            marcadores_especiales.append({
                "Marcador": c_izq,
                "Nombre_marcador": n_izq,
                "Parte": parte,
                "Codigo_derecha": c_der,
                "Nombre_derecha": n_der
            })

            continue

        # -----------------------------------------
        # 3. Nuevo código válido del lado izquierdo
        # -----------------------------------------
        if patron_6.match(c_izq):

            codigo_izq_activo = c_izq
            nombre_izq_activo = n_izq

        # -----------------------------------------
        # 4. Continuación del nombre izquierdo
        # -----------------------------------------
        elif (
            codigo_izq_activo
            and n_izq
            and not c_der
            and not parte
        ):

            nombre_izq_activo = limpiar_texto(
                nombre_izq_activo + " " + n_izq
            )

        # -----------------------------------------
        # 5. Nueva correspondencia válida
        # Solo si existen códigos válidos
        # en ambos lados
        # -----------------------------------------
        if (
            codigo_izq_activo
            and patron_6.match(c_der)
        ):

            registro_actual = {
                "SCIAN_2013": codigo_izq_activo,
                "Nombre_2013": nombre_izq_activo,
                "Parte": parte,
                "SCIAN_2007": c_der,
                "Nombre_2007": n_der
            }

            registros.append(registro_actual)

        # -----------------------------------------
        # 6. Continuación del nombre derecho
        # -----------------------------------------
        elif (
            registro_actual is not None
            and n_der
        ):

            registro_actual["Nombre_2007"] = (
                limpiar_texto(
                    registro_actual["Nombre_2007"]
                    + " "
                    + n_der
                )
            )

    return (
        pd.DataFrame(registros),
        pd.DataFrame(marcadores_especiales)
    )


print("✓ Parser mejorado cargado.")

✓ Parser mejorado cargado.


In [ ]:
correspondencias, marcadores = reconstruir_correspondencias(df_lineas)

In [ ]:
# ============================================================
# FASE 2 - PASO 7J
# GUARDAR MAPA VALIDADO SCIAN 2007 ↔ 2013
# ============================================================

pares_directos = set(
    zip(
        correspondencias_2013_2007_clasificadas["SCIAN_2007"]
        .astype(str)
        .str.strip(),

        correspondencias_2013_2007_clasificadas["SCIAN_2013"]
        .astype(str)
        .str.strip()
    )
)

mapa_validado_2007_2013 = pd.DataFrame(
    sorted(pares_directos),
    columns=[
        "SCIAN_2007",
        "SCIAN_2013"
    ]
)

mapa_validado_2007_2013.to_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2007_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "✓ Mapa validado 2007↔2013 guardado:",
    len(mapa_validado_2007_2013),
    "pares"
)

✓ Mapa validado 2007↔2013 guardado: 75 pares


## Homologación SCIAN 2007–2013

Para vincular las clases de actividad utilizadas en las ediciones históricas
del DENUE clasificadas mediante SCIAN 2007 con aquellas clasificadas mediante
SCIAN 2013, se utilizaron las tablas comparativas oficiales publicadas por
INEGI en ambos sentidos:

- SCIAN 2013 → SCIAN 2007
- SCIAN 2007 → SCIAN 2013

Las tablas oficiales fueron extraídas programáticamente a partir de los
documentos PDF, conservando el código de origen, el código de destino y el
campo "Parte", utilizado por INEGI para identificar correspondencias parciales.

La extracción SCIAN 2013 → SCIAN 2007 produjo 75 relaciones válidas, sin
registros problemáticos ni duplicados exactos. La tabla inversa produjo
inicialmente 76 relaciones; posteriormente se identificó un artefacto de
extracción asociado al marcador especial "YYYYYY", que no corresponde a una
clase SCIAN de seis dígitos. Una vez corregido, la tabla inversa quedó también
integrada por 75 relaciones válidas.

La comparación entre ambas direcciones confirmó:

- 75 pares de correspondencia en SCIAN 2013 → SCIAN 2007.
- 75 pares de correspondencia en SCIAN 2007 → SCIAN 2013.
- 75 pares coincidentes entre ambas tablas.
- 0 relaciones exclusivas de una sola dirección.
- 0 duplicados exactos.

Entre las relaciones extraídas se identificaron correspondencias de tipo
uno a uno, divisiones de una clase en varias, fusiones de varias clases en una
y relaciones de mayor complejidad. Asimismo, se conservaron las
correspondencias parciales señaladas por INEGI mediante valores como
"p 1/2", "p 2/2", "p 1/6", entre otros.

Por tanto, el puente SCIAN 2007–2013 queda validado para continuar con la
homologación histórica. Las relaciones parciales o de cardinalidad distinta
de uno a uno no serán transformadas mediante repartos arbitrarios de
establecimientos; serán tratadas posteriormente mediante criterios de
agregación comparables.

# Homologación SCIAN 2013–2018

In [ ]:
# ============================================================
# FASE 2 - PASO 8A
# LOCALIZAR PDF OFICIAL SCIAN 2018-2013
# ============================================================

pdf_2013_2018 = (
    HOMOLOGACION / "SCIAN_2018_2013_OFICIAL.pdf"
)

print("PDF encontrado:", pdf_2013_2018.exists())
print(pdf_2013_2018)

PDF encontrado: True
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN\SCIAN_2018_2013_OFICIAL.pdf


In [ ]:
# ============================================================
# FASE 2 - PASO 8B
# LOCALIZAR PÁGINAS CON CÓDIGOS SCIAN DE 6 DÍGITOS
# ============================================================

patron_scian = re.compile(r"\b\d{6}\b")

with pdfplumber.open(pdf_2013_2018) as pdf:

    print("Número total de páginas:", len(pdf.pages))
    print("\nPáginas con códigos de 6 dígitos:")

    paginas_con_codigos_2013_2018 = []

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        codigos = patron_scian.findall(texto)

        if codigos:

            paginas_con_codigos_2013_2018.append(i + 1)

            print(
                f"Página {i+1}: "
                f"{len(codigos)} códigos"
            )

Número total de páginas: 18

Páginas con códigos de 6 dígitos:
Página 5: 9 códigos
Página 6: 3 códigos
Página 8: 28 códigos
Página 9: 34 códigos
Página 10: 32 códigos
Página 11: 23 códigos
Página 12: 23 códigos
Página 13: 2 códigos
Página 15: 32 códigos
Página 16: 35 códigos
Página 17: 30 códigos
Página 18: 22 códigos


In [ ]:
# ============================================================
# RECUPERAR FUNCIÓN normalizar_texto
# ============================================================

import unicodedata

def normalizar_texto(texto):

    if texto is None:
        return ""

    texto = str(texto)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    return texto.lower().strip()


print("✓ normalizar_texto recuperada.")

✓ normalizar_texto recuperada.


In [ ]:
print(normalizar_texto("SCIAN México 2018"))

scian mexico 2018


In [ ]:
# ============================================================
# FASE 2 - PASO 8C
# IDENTIFICAR SENTIDO DE LAS TABLAS
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        texto_norm = normalizar_texto(texto)

        if (
            "scian mexico 2018" in texto_norm
            and "scian mexico 2013" in texto_norm
        ):

            print("\n====================================")
            print("PÁGINA PDF:", i + 1)
            print("====================================")

            for linea in texto.splitlines()[:12]:
                print(linea)


PÁGINA PDF: 2
ÍNDICE
Introducción V
Sistema de Clasificación Industrial de América del Norte, México 2018, SCIAN México
2018-Sistema de Clasificación Industrial de América del Norte, México 2013,
SCIAN México 2013 1
11 Agricultura, cría y explotación de animales, aprovechamiento forestal, pesca y caza 2
21 Minería 2
22 Generación, transmisión, distribución y comercialización de energía eléctrica,
suministro de agua y de gas natural por ductos al consumidor final 2
31-33 Industrias manufactureras 3
46 Comercio al por menor 3
48-49 Transportes, correos y almacenamiento 4

PÁGINA PDF: 4
TABLAS COMPARATIVAS
Para lograr la relación de los clasificadores y por lo tanto de la información que se recoge con ellos, se
elaboran tablas comparativas o de equivalencia1 que muestran los vínculos entre las categorías de los
clasificadores en estudio.
Para ayudar en las comparaciones hacia el pasado se elaboran tablas comparativas entre las categorías del
SCIAN y las de otros clasificadores de activid

In [ ]:
# ============================================================
# FASE 2 - PASO 8D
# BUSCAR PÁGINAS CON MUCHOS CÓDIGOS
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    candidatas = []

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        codigos = patron_scian.findall(texto)

        if (
            len(codigos) >= 15
            and "Caso 1." not in texto
            and "Caso 2." not in texto
            and "Caso 3." not in texto
            and "ÍNDICE" not in texto
        ):

            candidatas.append({
                "pagina_pdf": i + 1,
                "cantidad_codigos": len(codigos),
                "primeros_codigos": codigos[:10]
            })

for x in candidatas:
    print(x)

{'pagina_pdf': 8, 'cantidad_codigos': 28, 'primeros_codigos': ['111411', '111410', '111412', '111410', '111413', '111410', '111414', '111410', '111415', '111410']}
{'pagina_pdf': 9, 'cantidad_codigos': 34, 'primeros_codigos': ['221114', '221110', '221119', '221110', '221121', '221110', '221120', '221122', '221110', '221120']}
{'pagina_pdf': 10, 'cantidad_codigos': 32, 'primeros_codigos': ['468414', '468419', '468419', '468419', '485111', '485111', '485115', '485111', '512250', '512210']}
{'pagina_pdf': 11, 'cantidad_codigos': 23, 'primeros_codigos': ['561520', '561520', '562111', '562111', '562112', '931510', '562121', '562112', '562122', '931510']}
{'pagina_pdf': 12, 'cantidad_codigos': 23, 'primeros_codigos': ['562921', '562111', '562112', '562922', '931510', '562998', '562111', '562112', '562999', '931510']}
{'pagina_pdf': 15, 'cantidad_codigos': 32, 'primeros_codigos': ['111410', '111411', '111412', '111413', '111414', '111415', '111416', '111419', '111422', '111422']}
{'pagina_pdf

In [ ]:
# ============================================================
# FASE 2 - PASO 8E
# INSPECCIONAR POSICIONES DE LAS COLUMNAS
# ============================================================

pagina_prueba_2013_2018 = candidatas[0]["pagina_pdf"] - 1

with pdfplumber.open(pdf_2013_2018) as pdf:

    pagina = pdf.pages[pagina_prueba_2013_2018]

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    lineas = {}

    for palabra in palabras:

        top = round(palabra["top"] / 3) * 3

        lineas.setdefault(top, []).append(
            (
                palabra["text"],
                round(palabra["x0"], 1)
            )
        )

    contador = 0

    for top in sorted(lineas):

        contenido = sorted(
            lineas[top],
            key=lambda x: x[1]
        )

        texto_linea = " ".join(
            palabra for palabra, x in contenido
        )

        if re.search(r"\b\d{6}\b", texto_linea):

            print("\nTOP:", top)

            for palabra, x in contenido:
                print(f"{x:7.1f}  {palabra}")

            contador += 1

            if contador >= 10:
                break


TOP: 123
   59.2  111411
  107.4  Cultivo
  134.1  de
  145.2  jitomate
  175.5  en
  186.6  invernaderos
  235.0  y
  241.3  otras
  286.0  p
  292.7  1/7
  325.8  111410
  374.0  Cultivo
  400.7  de
  411.8  productos
  449.1  alimenticios
  493.1  en
  504.2  invernaderos
  571.3  N

TOP: 162
   59.3  111412
  107.4  Cultivo
  134.1  de
  145.2  fresa
  165.2  en
  176.4  invernaderos
  224.8  y
  231.1  otras
  286.0  p
  292.7  2/7
  325.9  111410
  374.0  Cultivo
  400.7  de
  411.8  productos
  449.1  alimenticios
  493.1  en
  504.2  invernaderos

TOP: 204
   59.3  111413
  107.5  Cultivo
  134.1  de
  145.2  bayas
  168.8  (berries)
  200.8  en
  211.9  invernaderos
  286.0  p
  292.7  3/7
  325.9  111410
  374.1  Cultivo
  400.7  de
  411.8  productos
  449.1  alimenticios
  493.1  en
  504.3  invernaderos

TOP: 252
   59.3  111414
  107.5  Cultivo
  134.1  de
  145.3  chile
  163.9  en
  175.1  invernaderos
  223.5  y
  229.7  otras
  286.1  p
  292.7  4/7
  325.9  111410
 

In [ ]:
# ============================================================
# FASE 2 - PASO 8F
# IDENTIFICAR LAS PÁGINAS DE CADA TABLA
# SCIAN 2018 ↔ SCIAN 2013
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    print("TOTAL DE PÁGINAS:", len(pdf.pages))

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        codigos = patron_scian.findall(texto)

        # Solo páginas con bastante contenido clasificatorio
        if len(codigos) >= 10:

            lineas = [
                linea.strip()
                for linea in texto.splitlines()
                if linea.strip()
            ]

            print("\n======================================")
            print("PÁGINA:", i + 1)
            print("CÓDIGOS DE 6 DÍGITOS:", len(codigos))
            print("--------------------------------------")

            for linea in lineas[:6]:
                print(linea)

TOTAL DE PÁGINAS: 18

PÁGINA: 8
CÓDIGOS DE 6 DÍGITOS: 28
--------------------------------------
TABLA COMPARATIVA I. SCIAN México 2018-SCIAN México 2013
SCIAN México 2018 SCIAN México 2013
Código Nombre Parte Código Nombre
11 AGRICULTURA, CRÍA Y EXPLOTACIÓN DE ANIMALES, APROVECHAMIENTO FORESTAL, PESCA Y CAZA
111411 Cultivo de jitomate en invernaderos y otras p 1/7 111410 Cultivo de productos alimenticios en invernaderos
estructuras agrícolas protegidas (cultivo de jitomate en invernaderos y otras

PÁGINA: 9
CÓDIGOS DE 6 DÍGITOS: 34
--------------------------------------
TABLA COMPARATIVA I. SCIAN México 2018-SCIAN México 2013 (Continúa)
SCIAN México 2018 SCIAN México 2013
Código Nombre Parte Código Nombre
221114 Generación de electricidad a partir de p 4/8 221110 Generación, transmisión y distribución de energía
energía eólica eléctrica (generación de electricidad a partir de
energía eólica)

PÁGINA: 10
CÓDIGOS DE 6 DÍGITOS: 32
--------------------------------------
TABLA COMPARATIVA I

In [ ]:
# ============================================================
# FASE 2 - PASO 8G
# RESUMEN COMPACTO DE PÁGINAS Y SENTIDO DE LAS TABLAS
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        texto_norm = normalizar_texto(texto)

        codigos = patron_scian.findall(texto)

        if len(codigos) >= 10:

            if "tabla comparativa i." in texto_norm:
                sentido = "2018 -> 2013"

            elif "tabla comparativa ii." in texto_norm:
                sentido = "2013 -> 2018"

            else:
                sentido = "REVISAR"

            print(
                f"Página {i+1:02d} | "
                f"Códigos: {len(codigos):3d} | "
                f"{sentido}"
            )

Página 08 | Códigos:  28 | 2018 -> 2013
Página 09 | Códigos:  34 | 2018 -> 2013
Página 10 | Códigos:  32 | 2018 -> 2013
Página 11 | Códigos:  23 | 2018 -> 2013
Página 12 | Códigos:  23 | 2018 -> 2013
Página 15 | Códigos:  32 | 2013 -> 2018
Página 16 | Códigos:  35 | 2013 -> 2018
Página 17 | Códigos:  30 | 2013 -> 2018
Página 18 | Códigos:  22 | 2013 -> 2018


In [ ]:
# ============================================================
# FASE 2 - PASO 8H
# PARSER GENÉRICO PARA TABLAS COMPARATIVAS SCIAN
# ============================================================

def extraer_lineas_scian_generico(pagina):

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    lineas = {}

    for palabra in palabras:

        top = round(palabra["top"] / 3) * 3
        lineas.setdefault(top, []).append(palabra)

    resultados = []

    for top in sorted(lineas):

        palabras_linea = sorted(
            lineas[top],
            key=lambda w: w["x0"]
        )

        columnas = {
            "codigo_izq": [],
            "nombre_izq": [],
            "parte": [],
            "codigo_der": [],
            "nombre_der": []
        }

        for w in palabras_linea:

            x = w["x0"]
            txt = w["text"].strip()

            # Ignorar marcas laterales
            if x >= 565:
                continue

            if x < 90:
                columnas["codigo_izq"].append(txt)

            elif x < 280:
                columnas["nombre_izq"].append(txt)

            elif x < 320:
                columnas["parte"].append(txt)

            elif x < 365:
                columnas["codigo_der"].append(txt)

            else:
                columnas["nombre_der"].append(txt)

        fila = {
            clave: " ".join(valor).strip()
            for clave, valor in columnas.items()
        }

        if any(fila.values()):
            fila["top"] = top
            resultados.append(fila)

    return pd.DataFrame(resultados)


def reconstruir_correspondencias_generico(df_lineas):

    registros = []
    marcadores = []

    codigo_izq_activo = ""
    nombre_izq_activo = ""
    registro_actual = None

    patron_codigo = re.compile(r"^\d{6}$")
    patron_sector = re.compile(r"^(\d{2}|\d{2}-\d{2})$")
    patron_marcador = re.compile(r"^[A-Z]{6}$")

    for _, fila in df_lineas.iterrows():

        cizq = limpiar_texto(fila.get("codigo_izq", ""))
        nizq = limpiar_texto(fila.get("nombre_izq", ""))
        parte = limpiar_texto(fila.get("parte", ""))
        cder = limpiar_texto(fila.get("codigo_der", ""))
        nder = limpiar_texto(fila.get("nombre_der", ""))

        # Encabezado de sector
        if patron_sector.match(cizq):
            codigo_izq_activo = ""
            nombre_izq_activo = ""
            registro_actual = None
            continue

        # Marcador especial en lado izquierdo, ej. YYYYYY
        if patron_marcador.match(cizq) and not patron_codigo.match(cizq):

            marcadores.append({
                "Lado": "izquierdo",
                "Marcador": cizq,
                "Nombre": nizq,
                "Parte": parte,
                "Codigo_otro_lado": cder,
                "Nombre_otro_lado": nder
            })

            codigo_izq_activo = ""
            nombre_izq_activo = ""
            registro_actual = None
            continue

        # Marcador especial en lado derecho
        if patron_marcador.match(cder) and not patron_codigo.match(cder):

            marcadores.append({
                "Lado": "derecho",
                "Marcador": cder,
                "Nombre": nder,
                "Parte": parte,
                "Codigo_otro_lado": cizq,
                "Nombre_otro_lado": nizq
            })

            registro_actual = None
            continue

        # Nuevo código izquierdo
        if patron_codigo.match(cizq):

            codigo_izq_activo = cizq
            nombre_izq_activo = nizq

        # Continuación del nombre izquierdo
        elif (
            codigo_izq_activo
            and nizq
            and not cder
            and not parte
        ):

            nombre_izq_activo = limpiar_texto(
                nombre_izq_activo + " " + nizq
            )

        # Correspondencia válida
        if (
            codigo_izq_activo
            and patron_codigo.match(cder)
        ):

            registro_actual = {
                "Codigo_izq": codigo_izq_activo,
                "Nombre_izq": nombre_izq_activo,
                "Parte": parte,
                "Codigo_der": cder,
                "Nombre_der": nder
            }

            registros.append(registro_actual)

        # Continuación del nombre derecho
        elif registro_actual is not None and nder:

            registro_actual["Nombre_der"] = limpiar_texto(
                registro_actual["Nombre_der"] + " " + nder
            )

    return (
        pd.DataFrame(registros),
        pd.DataFrame(marcadores)
    )


print("✓ Parser genérico SCIAN cargado.")

✓ Parser genérico SCIAN cargado.


In [ ]:
# ============================================================
# FASE 2 - PASO 8I
# EXTRAER SCIAN 2018 -> SCIAN 2013
# ============================================================

paginas_2018_2013 = [8, 9, 10, 11, 12]

registros = []
marcadores = []

with pdfplumber.open(pdf_2013_2018) as pdf:

    for pagina_pdf in paginas_2018_2013:

        pagina = pdf.pages[pagina_pdf - 1]

        df_lineas = extraer_lineas_scian_generico(pagina)

        tabla, marcas = reconstruir_correspondencias_generico(
            df_lineas
        )

        tabla["Pagina_PDF"] = pagina_pdf
        registros.append(tabla)

        if not marcas.empty:
            marcas["Pagina_PDF"] = pagina_pdf
            marcadores.append(marcas)


correspondencias_2018_2013 = pd.concat(
    registros,
    ignore_index=True
)

correspondencias_2018_2013 = (
    correspondencias_2018_2013.rename(
        columns={
            "Codigo_izq": "SCIAN_2018",
            "Nombre_izq": "Nombre_2018",
            "Codigo_der": "SCIAN_2013",
            "Nombre_der": "Nombre_2013"
        }
    )
)

if marcadores:
    marcadores_2018_2013 = pd.concat(
        marcadores,
        ignore_index=True
    )
else:
    marcadores_2018_2013 = pd.DataFrame()


print(
    "Correspondencias extraídas:",
    len(correspondencias_2018_2013)
)

print(
    "Marcadores especiales detectados:",
    len(marcadores_2018_2013)
)

display(correspondencias_2018_2013.head(25))

if not marcadores_2018_2013.empty:
    print("\nMARCADORES ESPECIALES:")
    display(marcadores_2018_2013)

Correspondencias extraídas: 76
Marcadores especiales detectados: 2


,SCIAN_2018,Nombre_2018,Parte,SCIAN_2013,Nombre_2013,Pagina_PDF
0,111411,Cultivo de jitomate en invernaderos y otras,p 1/7,111410,Cultivo de productos alimenticios en invernade...,8
1,111412,Cultivo de fresa en invernaderos y otras,p 2/7,111410,Cultivo de productos alimenticios en invernade...,8
2,111413,Cultivo de bayas (berries) en invernaderos,p 3/7,111410,Cultivo de productos alimenticios en invernade...,8
3,111414,Cultivo de chile en invernaderos y otras,p 4/7,111410,Cultivo de productos alimenticios en invernade...,8
4,111415,Cultivo de manzana en invernaderos y otras,p 5/7,111410,Cultivo de productos alimenticios en invernade...,8
5,111416,Cultivo de pepino en invernaderos y otras,p 6/7,111410,Cultivo de productos alimenticios en invernade...,8
6,111419,Cultivo de otros productos alimenticios en,p 7/7,111410,Cultivo de productos alimenticios en invernade...,8
7,111422,Floricultura en invernaderos y otras,,111422,Floricultura en invernadero Otros cultivos no ...,8
8,111429,,,111429,viveros Extracción de petróleo y gas (extracci...,8
9,211111,,,211110,petróleo y gas natural asociado),8



MARCADORES ESPECIALES:


,Lado,Marcador,Nombre,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,derecho,NNNNNN,Ninguno,p 1/2,525110,Sociedades de inversión especializadas en,10
1,derecho,NNNNNN,Ninguno,p 2/2,525210,Fondos de inversión,10


In [ ]:
# ============================================================
# FASE 2 - PASO 8J
# VALIDAR EXTRACCIÓN SCIAN 2018 -> 2013
# ============================================================

valido_2018 = (
    correspondencias_2018_2013["SCIAN_2018"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

valido_2013 = (
    correspondencias_2018_2013["SCIAN_2013"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

print(
    "Correspondencias extraídas:",
    len(correspondencias_2018_2013)
)

print(
    "Ambos códigos válidos:",
    (valido_2018 & valido_2013).sum()
)

print(
    "Problemáticos:",
    (~(valido_2018 & valido_2013)).sum()
)

print(
    "Duplicados exactos:",
    correspondencias_2018_2013.duplicated(
        subset=[
            "SCIAN_2018",
            "Parte",
            "SCIAN_2013"
        ]
    ).sum()
)

print("\nValores del campo Parte:")

display(
    correspondencias_2018_2013["Parte"]
    .fillna("")
    .value_counts()
)

print("\nMarcadores especiales detectados:")

display(marcadores_2018_2013)

Correspondencias extraídas: 76
Ambos códigos válidos: 76
Problemáticos: 0
Duplicados exactos: 0

Valores del campo Parte:


Parte
         33
p 2/2     6
p 1/2     6
p 1/8     2
p 2/8     2
p 3/8     2
p 4/8     2
p 5/8     2
p 6/8     2
p 7/8     2
p 1/5     2
p 2/5     2
p 3/5     2
p 1/7     1
p 2/7     1
p 3/7     1
p 4/7     1
p 5/7     1
p 6/7     1
p 7/7     1
p 1/3     1
p 8/8     1
p 4/5     1
p 5/5     1
Name: count, dtype: int64


Marcadores especiales detectados:


,Lado,Marcador,Nombre,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,derecho,NNNNNN,Ninguno,p 1/2,525110,Sociedades de inversión especializadas en,10
1,derecho,NNNNNN,Ninguno,p 2/2,525210,Fondos de inversión,10


In [ ]:
# ============================================================
# FASE 2 - PASO 8K
# GUARDAR EXTRACCIÓN SCIAN 2018 -> 2013
# ============================================================

correspondencias_2018_2013.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_OFICIAL_SCIAN_2018_2013_EXTRAIDA.csv",
    index=False,
    encoding="utf-8-sig"
)

marcadores_2018_2013.to_csv(
    HOMOLOGACION /
    "MARCADORES_ESPECIALES_SCIAN_2018_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Correspondencias 2018→2013 guardadas:", len(correspondencias_2018_2013))
print("✓ Marcadores especiales guardados:", len(marcadores_2018_2013))

✓ Correspondencias 2018→2013 guardadas: 76
✓ Marcadores especiales guardados: 2


In [ ]:
# ============================================================
# FASE 2 - PASO 8L
# EXTRAER SCIAN 2013 -> SCIAN 2018
# PÁGINAS 15 A 18
# ============================================================

paginas_2013_2018 = [15, 16, 17, 18]

registros = []
marcadores = []

with pdfplumber.open(pdf_2013_2018) as pdf:

    for pagina_pdf in paginas_2013_2018:

        pagina = pdf.pages[pagina_pdf - 1]

        df_lineas = extraer_lineas_scian_generico(pagina)

        tabla, marcas = reconstruir_correspondencias_generico(
            df_lineas
        )

        tabla["Pagina_PDF"] = pagina_pdf
        registros.append(tabla)

        if not marcas.empty:
            marcas["Pagina_PDF"] = pagina_pdf
            marcadores.append(marcas)


correspondencias_2013_2018 = pd.concat(
    registros,
    ignore_index=True
)

# En estas páginas:
# izquierda = SCIAN 2013
# derecha   = SCIAN 2018

correspondencias_2013_2018 = (
    correspondencias_2013_2018.rename(
        columns={
            "Codigo_izq": "SCIAN_2013",
            "Nombre_izq": "Nombre_2013",
            "Codigo_der": "SCIAN_2018",
            "Nombre_der": "Nombre_2018"
        }
    )
)

if marcadores:
    marcadores_2013_2018 = pd.concat(
        marcadores,
        ignore_index=True
    )
else:
    marcadores_2013_2018 = pd.DataFrame()

print(
    "Correspondencias extraídas:",
    len(correspondencias_2013_2018)
)

print(
    "Marcadores especiales detectados:",
    len(marcadores_2013_2018)
)

display(correspondencias_2013_2018.head(25))

if not marcadores_2013_2018.empty:
    print("\nMARCADORES ESPECIALES:")
    display(marcadores_2013_2018)

Correspondencias extraídas: 75
Marcadores especiales detectados: 1


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018,Pagina_PDF
0,111410,Cultivo de productos alimenticios en,,111411,Cultivo de jitomate en invernaderos y otras es...,15
1,111410,Cultivo de productos alimenticios en invernaderos,,111412,Cultivo de fresa en invernaderos y otras estru...,15
2,111410,Cultivo de productos alimenticios en invernaderos,,111413,Cultivo de bayas (berries) en invernaderos y o...,15
3,111410,Cultivo de productos alimenticios en invernaderos,,111414,estructuras agrícolas protegidas,15
4,111410,Cultivo de productos alimenticios en invernaderos,,111415,Cultivo de manzana en invernaderos y otras est...,15
5,111410,Cultivo de productos alimenticios en invernaderos,,111416,Cultivo de pepino en invernaderos y otras estr...,15
6,111410,Cultivo de productos alimenticios en invernaderos,,111419,Cultivo de otros productos alimenticios en inv...,15
7,111422,Floricultura en invernadero,,111422,Floricultura en invernaderos y otras estructur...,15
8,111429,Otros cultivos no alimenticios en,,111429,Otros cultivos no alimenticios en invernaderos...,15
9,211110,Extracción de petróleo y gas,,211111,Extracción de petróleo y gas natural asociado,15



MARCADORES ESPECIALES:


,Lado,Marcador,Nombre,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,izquierdo,NNNNNN,,,525110,,17


In [ ]:
# ============================================================
# FASE 2 - PASO 8M
# VALIDAR SCIAN 2013 -> 2018
# ============================================================

valido_2013 = (
    correspondencias_2013_2018["SCIAN_2013"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

valido_2018 = (
    correspondencias_2013_2018["SCIAN_2018"]
    .astype(str)
    .str.match(r"^\d{6}$")
)

print(
    "Correspondencias extraídas:",
    len(correspondencias_2013_2018)
)

print(
    "Ambos códigos válidos:",
    (valido_2013 & valido_2018).sum()
)

print(
    "Problemáticos:",
    (~(valido_2013 & valido_2018)).sum()
)

print(
    "Duplicados exactos:",
    correspondencias_2013_2018.duplicated(
        subset=[
            "SCIAN_2013",
            "Parte",
            "SCIAN_2018"
        ]
    ).sum()
)

print("\nValores del campo Parte:")

display(
    correspondencias_2013_2018["Parte"]
    .fillna("")
    .value_counts()
)

print("\nMarcadores especiales detectados:")

display(marcadores_2013_2018)

Correspondencias extraídas: 75
Ambos códigos válidos: 75
Problemáticos: 0
Duplicados exactos: 0

Valores del campo Parte:


Parte
         55
p 1/2     9
p 2/2     6
p 1/4     1
p 2/4     1
p 4/4     1
p 2/3     1
p 3/3     1
Name: count, dtype: int64


Marcadores especiales detectados:


,Lado,Marcador,Nombre,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,izquierdo,NNNNNN,,,525110,,17


In [ ]:
# ============================================================
# FASE 2 - PASO 8N
# COMPARAR LOS PARES SCIAN 2013 ↔ 2018
# ============================================================

pares_2018_2013 = set(
    zip(
        correspondencias_2018_2013["SCIAN_2013"]
        .astype(str).str.strip(),

        correspondencias_2018_2013["SCIAN_2018"]
        .astype(str).str.strip()
    )
)

pares_2013_2018 = set(
    zip(
        correspondencias_2013_2018["SCIAN_2013"]
        .astype(str).str.strip(),

        correspondencias_2013_2018["SCIAN_2018"]
        .astype(str).str.strip()
    )
)

print("Pares tabla 2018→2013:", len(pares_2018_2013))
print("Pares tabla 2013→2018:", len(pares_2013_2018))

print(
    "Pares comunes:",
    len(pares_2018_2013 & pares_2013_2018)
)

print(
    "Solo en tabla 2018→2013:",
    len(pares_2018_2013 - pares_2013_2018)
)

print(
    "Solo en tabla 2013→2018:",
    len(pares_2013_2018 - pares_2018_2013)
)

Pares tabla 2018→2013: 76
Pares tabla 2013→2018: 75
Pares comunes: 74
Solo en tabla 2018→2013: 2
Solo en tabla 2013→2018: 1


In [ ]:
# ============================================================
# FASE 2 - PASO 8O
# MOSTRAR LAS DIFERENCIAS EXACTAS
# ============================================================

solo_2018_2013 = sorted(
    pares_2018_2013 - pares_2013_2018
)

solo_2013_2018 = sorted(
    pares_2013_2018 - pares_2018_2013
)

print("SOLO EN 2018→2013:")

for par in solo_2018_2013:
    print(par)

print("\nSOLO EN 2013→2018:")

for par in solo_2013_2018:
    print(par)

SOLO EN 2018→2013:
('562112', '562921')
('562112', '562998')

SOLO EN 2013→2018:
('931510', '931510')


In [ ]:
# ============================================================
# FASE 2 - PASO 8P
# AUDITAR 525110 Y 525210
# ============================================================

codigos_revision = ["525110", "525210"]

print("=== TABLA 2018 → 2013 ===")

display(
    correspondencias_2018_2013[
        correspondencias_2018_2013["SCIAN_2018"]
        .astype(str)
        .isin(codigos_revision)
        |
        correspondencias_2018_2013["SCIAN_2013"]
        .astype(str)
        .isin(codigos_revision)
    ]
)

print("\n=== MARCADORES 2018 → 2013 ===")

display(
    marcadores_2018_2013[
        marcadores_2018_2013["Codigo_otro_lado"]
        .astype(str)
        .isin(codigos_revision)
    ]
)

print("\n=== TABLA 2013 → 2018 ===")

display(
    correspondencias_2013_2018[
        correspondencias_2013_2018["SCIAN_2013"]
        .astype(str)
        .isin(codigos_revision)
        |
        correspondencias_2013_2018["SCIAN_2018"]
        .astype(str)
        .isin(codigos_revision)
    ]
)

print("\n=== MARCADORES 2013 → 2018 ===")

display(
    marcadores_2013_2018[
        marcadores_2013_2018["Codigo_otro_lado"]
        .astype(str)
        .isin(codigos_revision)
    ]
)

=== TABLA 2018 → 2013 ===


,SCIAN_2018,Nombre_2018,Parte,SCIAN_2013,Nombre_2013,Pagina_PDF



=== MARCADORES 2018 → 2013 ===


,Lado,Marcador,Nombre,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,derecho,NNNNNN,Ninguno,p 1/2,525110,Sociedades de inversión especializadas en,10
1,derecho,NNNNNN,Ninguno,p 2/2,525210,Fondos de inversión,10



=== TABLA 2013 → 2018 ===


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018,Pagina_PDF



=== MARCADORES 2013 → 2018 ===


,Lado,Marcador,Nombre,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,izquierdo,NNNNNN,,,525110,,17


In [ ]:
# ============================================================
# FASE 2 - PASO 8Q
# PARSER V2: CONSERVA CONTINUIDAD ENTRE PÁGINAS
# Y ENTRE MARCADORES ESPECIALES
# ============================================================

def reconstruir_correspondencias_generico_v2(df_lineas):

    registros = []
    marcadores = []

    codigo_izq_activo = ""
    nombre_izq_activo = ""
    tipo_izq_activo = None

    registro_actual = None

    patron_codigo = re.compile(r"^\d{6}$")
    patron_sector = re.compile(r"^(\d{2}|\d{2}-\d{2})$")
    patron_marcador = re.compile(r"^[A-Z]{6}$")

    for _, fila in df_lineas.iterrows():

        cizq = limpiar_texto(fila.get("codigo_izq", ""))
        nizq = limpiar_texto(fila.get("nombre_izq", ""))
        parte = limpiar_texto(fila.get("parte", ""))
        cder = limpiar_texto(fila.get("codigo_der", ""))
        nder = limpiar_texto(fila.get("nombre_der", ""))
        pagina = fila.get("Pagina_PDF", None)

        # ----------------------------------------------------
        # 1. ENCABEZADO DE SECTOR
        # ----------------------------------------------------
        if patron_sector.match(cizq):

            codigo_izq_activo = ""
            nombre_izq_activo = ""
            tipo_izq_activo = None
            registro_actual = None

            continue

        # ----------------------------------------------------
        # 2. NUEVO CÓDIGO NORMAL DEL LADO IZQUIERDO
        # ----------------------------------------------------
        if patron_codigo.match(cizq):

            codigo_izq_activo = cizq
            nombre_izq_activo = nizq
            tipo_izq_activo = "codigo"
            registro_actual = None

        # ----------------------------------------------------
        # 3. NUEVO MARCADOR ESPECIAL DEL LADO IZQUIERDO
        # Ejemplo: NNNNNN
        # IMPORTANTE: SE MANTIENE ACTIVO
        # ----------------------------------------------------
        elif patron_marcador.match(cizq):

            codigo_izq_activo = cizq
            nombre_izq_activo = nizq
            tipo_izq_activo = "marcador"
            registro_actual = None

        # ----------------------------------------------------
        # 4. CONTINUACIÓN DEL NOMBRE IZQUIERDO
        # ----------------------------------------------------
        elif (
            codigo_izq_activo
            and nizq
            and not cder
            and not parte
        ):

            nombre_izq_activo = limpiar_texto(
                nombre_izq_activo + " " + nizq
            )

        # ----------------------------------------------------
        # 5. CÓDIGO NORMAL DEL LADO DERECHO
        # ----------------------------------------------------
        if patron_codigo.match(cder):

            if tipo_izq_activo == "codigo":

                registro_actual = {
                    "Codigo_izq": codigo_izq_activo,
                    "Nombre_izq": nombre_izq_activo,
                    "Parte": parte,
                    "Codigo_der": cder,
                    "Nombre_der": nder,
                    "Pagina_PDF": pagina
                }

                registros.append(registro_actual)

            elif tipo_izq_activo == "marcador":

                registro_actual = {
                    "Lado_marcador": "izquierdo",
                    "Marcador": codigo_izq_activo,
                    "Nombre_marcador": nombre_izq_activo,
                    "Parte": parte,
                    "Codigo_otro_lado": cder,
                    "Nombre_otro_lado": nder,
                    "Pagina_PDF": pagina
                }

                marcadores.append(registro_actual)

        # ----------------------------------------------------
        # 6. MARCADOR ESPECIAL DEL LADO DERECHO
        # ----------------------------------------------------
        elif patron_marcador.match(cder):

            if tipo_izq_activo is not None:

                registro_actual = {
                    "Lado_marcador": "derecho",
                    "Codigo_otro_lado": codigo_izq_activo,
                    "Nombre_otro_lado": nombre_izq_activo,
                    "Parte": parte,
                    "Marcador": cder,
                    "Nombre_marcador": nder,
                    "Pagina_PDF": pagina
                }

                marcadores.append(registro_actual)

        # ----------------------------------------------------
        # 7. CONTINUACIÓN DEL NOMBRE DERECHO
        # ----------------------------------------------------
        elif (
            registro_actual is not None
            and nder
        ):

            if "Nombre_der" in registro_actual:

                registro_actual["Nombre_der"] = limpiar_texto(
                    registro_actual["Nombre_der"]
                    + " "
                    + nder
                )

            elif "Nombre_otro_lado" in registro_actual:

                registro_actual["Nombre_otro_lado"] = limpiar_texto(
                    registro_actual["Nombre_otro_lado"]
                    + " "
                    + nder
                )

    return (
        pd.DataFrame(registros),
        pd.DataFrame(marcadores)
    )


print("✓ Parser V2 cargado.")

✓ Parser V2 cargado.


In [ ]:
# ============================================================
# FASE 2 - PASO 8R
# REEXTRAER AMBOS SENTIDOS CON CONTINUIDAD ENTRE PÁGINAS
# ============================================================

def extraer_paginas_continuas(pdf_path, paginas):

    bloques = []

    with pdfplumber.open(pdf_path) as pdf:

        for pagina_pdf in paginas:

            pagina = pdf.pages[pagina_pdf - 1]

            df = extraer_lineas_scian_generico(pagina)

            df["Pagina_PDF"] = pagina_pdf

            bloques.append(df)

    return pd.concat(
        bloques,
        ignore_index=True
    )


# ------------------------------------------------------------
# A) SCIAN 2018 -> SCIAN 2013
# ------------------------------------------------------------

lineas_2018_2013_v2 = extraer_paginas_continuas(
    pdf_2013_2018,
    [8, 9, 10, 11, 12]
)

tabla_2018_2013_v2, marcadores_2018_2013_v2 = (
    reconstruir_correspondencias_generico_v2(
        lineas_2018_2013_v2
    )
)

correspondencias_2018_2013_v2 = (
    tabla_2018_2013_v2.rename(
        columns={
            "Codigo_izq": "SCIAN_2018",
            "Nombre_izq": "Nombre_2018",
            "Codigo_der": "SCIAN_2013",
            "Nombre_der": "Nombre_2013"
        }
    )
)


# ------------------------------------------------------------
# B) SCIAN 2013 -> SCIAN 2018
# Incluimos página 14 por seguridad.
# Si en tu PDF es portada/separador, simplemente no aportará pares.
# ------------------------------------------------------------

lineas_2013_2018_v2 = extraer_paginas_continuas(
    pdf_2013_2018,
    [14, 15, 16, 17, 18]
)

tabla_2013_2018_v2, marcadores_2013_2018_v2 = (
    reconstruir_correspondencias_generico_v2(
        lineas_2013_2018_v2
    )
)

correspondencias_2013_2018_v2 = (
    tabla_2013_2018_v2.rename(
        columns={
            "Codigo_izq": "SCIAN_2013",
            "Nombre_izq": "Nombre_2013",
            "Codigo_der": "SCIAN_2018",
            "Nombre_der": "Nombre_2018"
        }
    )
)


print("2018→2013")
print(
    "Correspondencias:",
    len(correspondencias_2018_2013_v2)
)
print(
    "Marcadores:",
    len(marcadores_2018_2013_v2)
)

print("\n2013→2018")
print(
    "Correspondencias:",
    len(correspondencias_2013_2018_v2)
)
print(
    "Marcadores:",
    len(marcadores_2013_2018_v2)
)

2018→2013
Correspondencias: 76
Marcadores: 2

2013→2018
Correspondencias: 77
Marcadores: 2


In [ ]:
# ============================================================
# FASE 2 - PASO 8S
# VALIDACIÓN CRUZADA V2
# ============================================================

pares_directos_v2 = set(
    zip(
        correspondencias_2018_2013_v2["SCIAN_2013"]
        .astype(str).str.strip(),

        correspondencias_2018_2013_v2["SCIAN_2018"]
        .astype(str).str.strip()
    )
)

pares_inversos_v2 = set(
    zip(
        correspondencias_2013_2018_v2["SCIAN_2013"]
        .astype(str).str.strip(),

        correspondencias_2013_2018_v2["SCIAN_2018"]
        .astype(str).str.strip()
    )
)

print("Pares 2018→2013:", len(pares_directos_v2))
print("Pares 2013→2018:", len(pares_inversos_v2))
print("Pares comunes:", len(pares_directos_v2 & pares_inversos_v2))
print("Solo directa:", len(pares_directos_v2 - pares_inversos_v2))
print("Solo inversa:", len(pares_inversos_v2 - pares_directos_v2))

print("\nSOLO DIRECTA:")
for x in sorted(pares_directos_v2 - pares_inversos_v2):
    print(x)

print("\nSOLO INVERSA:")
for x in sorted(pares_inversos_v2 - pares_directos_v2):
    print(x)

print("\nMARCADORES 2018→2013:")
display(marcadores_2018_2013_v2)

print("\nMARCADORES 2013→2018:")
display(marcadores_2013_2018_v2)

Pares 2018→2013: 76
Pares 2013→2018: 77
Pares comunes: 76
Solo directa: 0
Solo inversa: 1

SOLO DIRECTA:

SOLO INVERSA:
('931510', '931510')

MARCADORES 2018→2013:


,Lado_marcador,Codigo_otro_lado,Nombre_otro_lado,Parte,Marcador,Nombre_marcador,Pagina_PDF
0,derecho,525110,Sociedades de inversión especializadas en,p 1/2,NNNNNN,Ninguno,10
1,derecho,525210,Fondos de inversión,p 2/2,NNNNNN,Ninguno,10



MARCADORES 2013→2018:


,Lado_marcador,Marcador,Nombre_marcador,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,izquierdo,NNNNNN,,,525110,fondos para el retiro,17
1,izquierdo,NNNNNN,,,525210,Fondos de inversión,17


In [ ]:
# ============================================================
# FASE 2 - PASO 8T
# AUDITAR COORDENADAS DE 931510 EN PÁGINA 12
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    pagina = pdf.pages[11]   # página PDF 12

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    lineas = {}

    for palabra in palabras:

        top = round(palabra["top"] / 3) * 3

        lineas.setdefault(top, []).append(
            (
                palabra["text"],
                round(palabra["x0"], 1)
            )
        )

    for top in sorted(lineas):

        contenido = sorted(
            lineas[top],
            key=lambda x: x[1]
        )

        texto_linea = " ".join(
            palabra for palabra, x in contenido
        )

        if (
            "931510" in texto_linea
            or "8/8" in texto_linea
            or "Regulación" in texto_linea
        ):

            print("\nTOP:", top)

            for palabra, x in contenido:
                print(f"{x:7.1f}  {palabra}")


TOP: 198
   59.3  562922
  107.4  Recuperación
  159.0  de
  170.1  residuos
  202.6  por
  216.4  el
  224.8  sector
  286.0  p
  292.7  6/8
  325.9  931510
  374.0  Regulación
  416.3  y
  422.5  fomento
  453.6  de
  464.8  actividades
  507.0  para

TOP: 327
   59.3  562999
  107.5  Otros
  129.2  servicios
  162.6  de
  173.7  manejo
  202.2  de
  213.3  residuos
  245.7  por
  259.5  el
  286.0  p
  292.7  7/8
  325.9  931510
  374.1  Regulación
  416.3  y
  422.5  fomento
  453.7  de
  464.8  actividades
  507.0  para


In [ ]:
# ============================================================
# FASE 2 - PASO 8U
# VER CÓMO EL PARSER DIVIDIÓ LA FILA 931510
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    df_p12 = extraer_lineas_scian_generico(
        pdf.pages[11]
    )

display(
    df_p12[
        df_p12.astype(str)
        .apply(
            lambda fila:
            fila.str.contains(
                "931510|8/8|Regulación",
                regex=True,
                case=False
            ).any(),
            axis=1
        )
    ]
)

,codigo_izq,nombre_izq,parte,codigo_der,nombre_der,top
13,562922,Recuperación de residuos por el sector,p 6/8,931510,Regulación y fomento de actividades para,198
25,562999,Otros servicios de manejo de residuos por el,p 7/8,931510,Regulación y fomento de actividades para,327


## PUNTO DE REANUDACIÓN - 02/10/2026

Homologación SCIAN 2013–2018 en proceso.

Estado actual:
- SCIAN 2018→2013: 76 pares extraídos.
- SCIAN 2013→2018: 77 pares extraídos.
- 76 pares comunes.
- Diferencia pendiente: ('931510', '931510').
- Marcadores NNNNNN correctamente detectados para 525110 y 525210.
- Próximo paso: auditar coordenadas de 931510 en página 12 del PDF oficial y corregir el parser sin introducir relaciones manualmente.

In [ ]:
# ============================================================
# FASE 2 - PASO 8T
# AUDITAR COORDENADAS DE 931510 EN PÁGINA 12
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    pagina = pdf.pages[11]   # página PDF 12

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    lineas = {}

    for palabra in palabras:

        top = round(palabra["top"] / 3) * 3

        lineas.setdefault(top, []).append(
            (
                palabra["text"],
                round(palabra["x0"], 1)
            )
        )

    for top in sorted(lineas):

        contenido = sorted(
            lineas[top],
            key=lambda x: x[1]
        )

        texto_linea = " ".join(
            palabra for palabra, x in contenido
        )

        if (
            "931510" in texto_linea
            or "8/8" in texto_linea
            or "Regulación" in texto_linea
        ):

            print("\nTOP:", top)

            for palabra, x in contenido:
                print(f"{x:7.1f}  {palabra}")


TOP: 198
   59.3  562922
  107.4  Recuperación
  159.0  de
  170.1  residuos
  202.6  por
  216.4  el
  224.8  sector
  286.0  p
  292.7  6/8
  325.9  931510
  374.0  Regulación
  416.3  y
  422.5  fomento
  453.6  de
  464.8  actividades
  507.0  para

TOP: 327
   59.3  562999
  107.5  Otros
  129.2  servicios
  162.6  de
  173.7  manejo
  202.2  de
  213.3  residuos
  245.7  por
  259.5  el
  286.0  p
  292.7  7/8
  325.9  931510
  374.1  Regulación
  416.3  y
  422.5  fomento
  453.7  de
  464.8  actividades
  507.0  para


In [ ]:
# ============================================================
# FASE 2 - PASO 8U
# VER CÓMO EL PARSER DIVIDIÓ LA FILA 931510
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    df_p12 = extraer_lineas_scian_generico(
        pdf.pages[11]
    )

display(
    df_p12[
        df_p12.astype(str)
        .apply(
            lambda fila:
            fila.str.contains(
                "931510|8/8|Regulación",
                regex=True,
                case=False
            ).any(),
            axis=1
        )
    ]
)

,codigo_izq,nombre_izq,parte,codigo_der,nombre_der,top
13,562922,Recuperación de residuos por el sector,p 6/8,931510,Regulación y fomento de actividades para,198
25,562999,Otros servicios de manejo de residuos por el,p 7/8,931510,Regulación y fomento de actividades para,327


In [ ]:
# ============================================================
# FASE 2 - PASO 8V
# INSPECCIONAR TEXTO CRUDO ALREDEDOR DE 931510 / p 8/8
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    pagina = pdf.pages[11]   # página PDF 12

    texto = pagina.extract_text(
        x_tolerance=1,
        y_tolerance=3
    ) or ""

    lineas = texto.splitlines()

    for i, linea in enumerate(lineas):

        if (
            "931510" in linea
            or "8/8" in linea
            or "562922" in linea
            or "562999" in linea
        ):

            print("\n----------------------------------------")

            desde = max(0, i - 2)
            hasta = min(len(lineas), i + 3)

            for j in range(desde, hasta):
                print(f"{j:03d}: {lineas[j]}")


----------------------------------------
009: no peligrosos (recuperación de residuos no
010: peligrosos por el sector privado)
011: 562922 Recuperación de residuos por el sector p 6/8 931510 Regulación y fomento de actividades para
012: público mejorar y preservar el medio ambiente
013: (recuperación de residuos por el sector público)

----------------------------------------
020: peligrosos (otros servicios de manejo de residuos
021: no peligrosos por el sector privado)
022: 562999 Otros servicios de manejo de residuos por el p 7/8 931510 Regulación y fomento de actividades para
023: sector público mejorar y preservar el medio ambiente (otros
024: servicios de manejo de residuos por el sector


In [ ]:
# ============================================================
# PASO 8W
# BUSCAR ESPECÍFICAMENTE p 8/8 EN LAS LÍNEAS YA SEPARADAS
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    df_p12 = extraer_lineas_scian_generico(
        pdf.pages[11]
    )

display(
    df_p12[
        df_p12["parte"]
        .astype(str)
        .str.contains("8/8", regex=False)
    ]
)

,codigo_izq,nombre_izq,parte,codigo_der,nombre_der,top


In [ ]:
# ============================================================
# PASO 8X
# INSPECCIONAR PÁGINA 13 COMPLETA
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    pagina = pdf.pages[12]   # página PDF 13
    texto = pagina.extract_text() or ""

    print(texto)

TABLA COMPARATIVA I. SCIAN México 2018-SCIAN México 2013 (Concluye)
SCIAN México 2018 SCIAN México 2013
Código Nombre Parte Código Nombre
93 ACTIVIDADES LEGISLATIVAS, GUBERNAMENTALES, DE IMPARTICIÓN DE JUSTICIA Y DE ORGANISMOS INTERNACIONALES
Y EXTRATERRITORIALES
931510 Regulación y fomento de actividades para p 8/8 931510 Regulación y fomento de actividades para
mejorar y preservar el medio ambiente mejorar y preservar el medio ambiente
(establecimiento de normas y procedimientos
para mejorar y preservar el medio ambiente, y
administración y regulación de programas para el
manejo de residuos)
7
,etroN
led
acirémA
ed
lairtsudnI
nóicacifisalC
ed
ametsiS-8102
ocixéM
NAICS
,8102
ocixéM
,etroN
led
acirémA
ed
lairtsudnI
nóicacifisalC
ed
ametsiS
.IGENI
.3102
ocixéM
NAICS
,3102
ocixéM
INEGI. Sistema de Clasificación Industrial de América del Norte, SCIAN 2018. SNIEG. Información de Interés Nacional.


In [ ]:
# ============================================================
# PASO 8Y
# BUSCAR CÓDIGOS Y p 8/8 EN PÁGINA 13
# ============================================================

with pdfplumber.open(pdf_2013_2018) as pdf:

    pagina = pdf.pages[12]

    texto = pagina.extract_text() or ""

    print("Códigos de 6 dígitos:")
    print(re.findall(r"\b\d{6}\b", texto))

    print("\n¿Aparece 931510?")
    print("931510" in texto)

    print("\n¿Aparece 8/8?")
    print("8/8" in texto)

Códigos de 6 dígitos:
['931510', '931510']

¿Aparece 931510?
True

¿Aparece 8/8?
True


In [ ]:
lineas_2018_2013_v2 = extraer_paginas_continuas(
    pdf_2013_2018,
    [8, 9, 10, 11, 12, 13]
)

In [ ]:
tabla_2018_2013_v2, marcadores_2018_2013_v2 = (
    reconstruir_correspondencias_generico_v2(
        lineas_2018_2013_v2
    )
)

correspondencias_2018_2013_v2 = (
    tabla_2018_2013_v2.rename(
        columns={
            "Codigo_izq": "SCIAN_2018",
            "Nombre_izq": "Nombre_2018",
            "Codigo_der": "SCIAN_2013",
            "Nombre_der": "Nombre_2013"
        }
    )
)

In [ ]:
# ============================================================
# FASE 2 - PASO 8S
# VALIDACIÓN CRUZADA V2
# ============================================================

pares_directos_v2 = set(
    zip(
        correspondencias_2018_2013_v2["SCIAN_2013"]
        .astype(str).str.strip(),

        correspondencias_2018_2013_v2["SCIAN_2018"]
        .astype(str).str.strip()
    )
)

pares_inversos_v2 = set(
    zip(
        correspondencias_2013_2018_v2["SCIAN_2013"]
        .astype(str).str.strip(),

        correspondencias_2013_2018_v2["SCIAN_2018"]
        .astype(str).str.strip()
    )
)

print("Pares 2018→2013:", len(pares_directos_v2))
print("Pares 2013→2018:", len(pares_inversos_v2))
print("Pares comunes:", len(pares_directos_v2 & pares_inversos_v2))
print("Solo directa:", len(pares_directos_v2 - pares_inversos_v2))
print("Solo inversa:", len(pares_inversos_v2 - pares_directos_v2))

print("\nSOLO DIRECTA:")
for x in sorted(pares_directos_v2 - pares_inversos_v2):
    print(x)

print("\nSOLO INVERSA:")
for x in sorted(pares_inversos_v2 - pares_directos_v2):
    print(x)

print("\nMARCADORES 2018→2013:")
display(marcadores_2018_2013_v2)

print("\nMARCADORES 2013→2018:")
display(marcadores_2013_2018_v2)

Pares 2018→2013: 77
Pares 2013→2018: 77
Pares comunes: 77
Solo directa: 0
Solo inversa: 0

SOLO DIRECTA:

SOLO INVERSA:

MARCADORES 2018→2013:


,Lado_marcador,Codigo_otro_lado,Nombre_otro_lado,Parte,Marcador,Nombre_marcador,Pagina_PDF
0,derecho,525110,Sociedades de inversión especializadas en,p 1/2,NNNNNN,Ninguno,10
1,derecho,525210,Fondos de inversión,p 2/2,NNNNNN,Ninguno,10



MARCADORES 2013→2018:


,Lado_marcador,Marcador,Nombre_marcador,Parte,Codigo_otro_lado,Nombre_otro_lado,Pagina_PDF
0,izquierdo,NNNNNN,,,525110,fondos para el retiro,17
1,izquierdo,NNNNNN,,,525210,Fondos de inversión,17


In [ ]:
# ============================================================
# FASE 2 - PASO 8Z
# GUARDAR PUENTE VALIDADO SCIAN 2013 ↔ 2018
# ============================================================

# Correspondencias en ambos sentidos
correspondencias_2018_2013_v2.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

correspondencias_2013_2018_v2.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2013_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

# Marcadores especiales
marcadores_2018_2013_v2.to_csv(
    HOMOLOGACION /
    "MARCADORES_ESPECIALES_SCIAN_2018_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

marcadores_2013_2018_v2.to_csv(
    HOMOLOGACION /
    "MARCADORES_ESPECIALES_SCIAN_2013_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

# Mapa único validado
mapa_validado_2013_2018 = pd.DataFrame(
    sorted(pares_directos_v2),
    columns=[
        "SCIAN_2013",
        "SCIAN_2018"
    ]
)

mapa_validado_2013_2018.to_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2013_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "✓ Mapa validado SCIAN 2013↔2018 guardado:",
    len(mapa_validado_2013_2018),
    "pares"
)

✓ Mapa validado SCIAN 2013↔2018 guardado: 77 pares


## Homologación SCIAN 2013–2018

Para vincular las clases de actividad correspondientes a SCIAN México 2013 y
SCIAN México 2018 se utilizaron las tablas comparativas oficiales de INEGI en
ambos sentidos:

- SCIAN 2018 → SCIAN 2013
- SCIAN 2013 → SCIAN 2018

La extracción se realizó programáticamente a partir del documento oficial,
preservando los códigos, nombres de las clases y el campo "Parte".

Durante la auditoría se detectó que algunas relaciones continuaban entre
páginas consecutivas del documento, por lo que el procedimiento de extracción
fue ajustado para conservar la continuidad de los registros entre páginas.

Una vez corregido y validado el procedimiento, se obtuvieron:

- 77 pares válidos en SCIAN 2018 → SCIAN 2013.
- 77 pares válidos en SCIAN 2013 → SCIAN 2018.
- 77 pares coincidentes entre ambas tablas.
- 0 relaciones exclusivas de una sola dirección.
- 0 pares discrepantes.

Asimismo, se identificó el marcador especial "NNNNNN", el cual fue conservado
separadamente y no fue tratado como una clase SCIAN de seis dígitos. Dicho
marcador se encontró asociado a las clases 525110 y 525210 y fue corroborado
en ambas direcciones de la tabla oficial.

Por tanto, el puente SCIAN 2013–2018 queda validado para su utilización en la
homologación histórica. Las relaciones parciales indicadas mediante el campo
"Parte" se conservarán sin realizar distribuciones arbitrarias de
establecimientos.

# Homologación SCIAN 2018–2023  

In [77]:
# ============================================================
# FASE 2 - PASO 9A
# LOCALIZAR PDF OFICIAL SCIAN 2023-2018
# ============================================================

pdf_2018_2023 = (
    HOMOLOGACION / "SCIAN_2023_2018_OFICIAL.pdf"
)

print("PDF encontrado:", pdf_2018_2023.exists())
print(pdf_2018_2023)

PDF encontrado: True
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\03_HOMOLOGACION_SCIAN\SCIAN_2023_2018_OFICIAL.pdf


In [78]:
# ============================================================
# FASE 2 - PASO 9B
# AUDITAR PÁGINAS DEL PDF SCIAN 2023-2018
# ============================================================

patron_scian = re.compile(r"\b\d{6}\b")

with pdfplumber.open(pdf_2018_2023) as pdf:

    print("TOTAL DE PÁGINAS:", len(pdf.pages))

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        texto_norm = normalizar_texto(texto)

        codigos = patron_scian.findall(texto)

        if len(codigos) >= 5:

            if (
                "scian mexico 2023" in texto_norm
                and "scian mexico 2018" in texto_norm
            ):

                primeras = [
                    x.strip()
                    for x in texto.splitlines()[:5]
                    if x.strip()
                ]

                print("\n======================================")
                print(
                    f"PÁGINA {i+1:02d} | "
                    f"CÓDIGOS: {len(codigos)}"
                )

                for linea in primeras:
                    print(linea)

TOTAL DE PÁGINAS: 45

PÁGINA 05 | CÓDIGOS: 6
Tablas comparativas
Caso 1. La clase SCIAN México 2023 es igual en contenido a una clase SCIAN México 2018, pero los títulos son
distintos:
SCIAN México 2023 SCIAN México 2018
Código Nombre Parte Código Nombre

PÁGINA 08 | CÓDIGOS: 34
TABLA COMPARATIVA I. SCIAN México 2023-SCIAN México 2018
SCIAN México 2023 SCIAN México 2018
Código Nombre Parte Código Nombre
11 AGRICULTURA, CRÍA Y EXPLOTACIÓN DE ANIMALES, APROVECHAMIENTO FORESTAL, PESCA Y CAZA
112513 Camaronicultura y acuicultura de otros p 1/2 112511 Camaronicultura (camaronicultura y acuicultura

PÁGINA 09 | CÓDIGOS: 37
TABLA COMPARATIVA I. SCIAN México 2023-SCIAN México 2018 (Continúa)
SCIAN México 2023 SCIAN México 2018
Código Nombre Parte Código Nombre
313230 Fabricación de telas no tejidas (comprimidas) 313230 Fabricación de telas no tejidas (comprimidas)
p 3/9 314993 Fabricación de productos textiles reciclados

PÁGINA 10 | CÓDIGOS: 22
TABLA COMPARATIVA I. SCIAN México 2023-SCIAN Méx

In [79]:
# ============================================================
# FASE 2 - PASO 9C
# IDENTIFICAR SENTIDO DE CADA TABLA
# ============================================================

with pdfplumber.open(pdf_2018_2023) as pdf:

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        texto_norm = normalizar_texto(texto)

        codigos = patron_scian.findall(texto)

        if len(codigos) >= 5:

            sentido = "REVISAR"

            if "tabla comparativa i." in texto_norm:
                sentido = "TABLA I"

            elif "tabla comparativa ii." in texto_norm:
                sentido = "TABLA II"

            print(
                f"Página {i+1:02d} | "
                f"Códigos: {len(codigos):3d} | "
                f"{sentido}"
            )

Página 05 | Códigos:   6 | REVISAR
Página 08 | Códigos:  34 | TABLA I
Página 09 | Códigos:  37 | TABLA I
Página 10 | Códigos:  22 | TABLA I
Página 11 | Códigos:  21 | TABLA I
Página 12 | Códigos:  24 | TABLA I
Página 13 | Códigos:  24 | TABLA I
Página 14 | Códigos:  24 | TABLA I
Página 15 | Códigos:  21 | TABLA I
Página 16 | Códigos:  24 | TABLA I
Página 17 | Códigos:  21 | TABLA I
Página 18 | Códigos:  21 | TABLA I
Página 19 | Códigos:  24 | TABLA I
Página 20 | Códigos:  21 | TABLA I
Página 21 | Códigos:  23 | TABLA I
Página 22 | Códigos:  27 | TABLA I
Página 23 | Códigos:  30 | TABLA I
Página 24 | Códigos:  21 | TABLA I
Página 26 | Códigos:  34 | TABLA II
Página 27 | Códigos:  30 | TABLA II
Página 28 | Códigos:  20 | TABLA II
Página 29 | Códigos:  20 | TABLA II
Página 30 | Códigos:  22 | TABLA II
Página 31 | Códigos:  22 | TABLA II
Página 32 | Códigos:  22 | TABLA II
Página 33 | Códigos:  22 | TABLA II
Página 34 | Códigos:  20 | TABLA II
Página 35 | Códigos:  18 | TABLA II
Página 36 

In [80]:
# ============================================================
# FASE 2 - PASO 9D
# INSPECCIONAR COORDENADAS DE COLUMNAS
# ============================================================

with pdfplumber.open(pdf_2018_2023) as pdf:

    for i, pagina in enumerate(pdf.pages):

        texto = pagina.extract_text() or ""
        codigos = patron_scian.findall(texto)

        if len(codigos) >= 15:

            pagina_prueba = i
            break

    pagina = pdf.pages[pagina_prueba]

    palabras = pagina.extract_words(
        x_tolerance=1,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False
    )

    lineas = {}

    for palabra in palabras:

        top = round(palabra["top"] / 3) * 3

        lineas.setdefault(top, []).append(
            (
                palabra["text"],
                round(palabra["x0"], 1)
            )
        )

    contador = 0

    print("Página de prueba:", pagina_prueba + 1)

    for top in sorted(lineas):

        contenido = sorted(
            lineas[top],
            key=lambda x: x[1]
        )

        texto_linea = " ".join(
            palabra for palabra, x in contenido
        )

        if re.search(r"\b\d{6}\b", texto_linea):

            print("\nTOP:", top)

            for palabra, x in contenido:
                print(f"{x:7.1f}  {palabra}")

            contador += 1

            if contador >= 8:
                break

Página de prueba: 8

TOP: 123
   59.7  112513
  107.9  Camaronicultura
  168.8  y
  175.1  acuicultura
  216.0  de
  227.1  otros
  286.5  p
  293.2  1/2
  326.3  112511
  374.5  Camaronicultura
  435.4  (camaronicultura
  497.2  y
  503.5  acuicultura

TOP: 153
   59.8  112514
  107.9  Camaronicultura
  168.8  y
  175.1  acuicultura
  216.0  de
  227.1  otros
  286.5  p
  293.2  2/2
  326.4  112511
  374.5  Camaronicultura
  435.5  (camaronicultura
  497.2  y
  503.5  acuicultura

TOP: 186
   59.8  112515
  108.0  Piscicultura
  151.1  en
  162.2  agua
  182.2  dulce
  286.5  p
  293.2  1/3
  326.4  112512
  374.6  Piscicultura
  417.7  y
  423.9  otra
  439.9  acuicultura,
  483.0  excepto

TOP: 216
   59.8  112516
  108.0  Piscicultura
  151.1  en
  162.2  agua
  182.2  salada
  286.5  p
  293.2  2/3
  326.4  112512
  374.6  Piscicultura
  417.7  y
  423.9  otra
  439.9  acuicultura,
  483.0  excepto

TOP: 249
   59.8  112519
  108.0  Otra
  125.7  acuicultura
  286.5  p
  293.2  3/

In [81]:
# ============================================================
# FASE 2 - PASO 9G
# EXTRAER SCIAN 2023 -> SCIAN 2018
# PÁGINAS 8 A 24
# ============================================================

lineas_2023_2018_v2 = extraer_paginas_continuas(
    pdf_2018_2023,
    list(range(8, 25))
)

tabla_2023_2018_v2, marcadores_2023_2018_v2 = (
    reconstruir_correspondencias_generico_v2(
        lineas_2023_2018_v2
    )
)

correspondencias_2023_2018_v2 = (
    tabla_2023_2018_v2.rename(
        columns={
            "Codigo_izq": "SCIAN_2023",
            "Nombre_izq": "Nombre_2023",
            "Codigo_der": "SCIAN_2018",
            "Nombre_der": "Nombre_2018"
        }
    )
)

print(
    "Correspondencias 2023→2018:",
    len(correspondencias_2023_2018_v2)
)

print(
    "Marcadores especiales:",
    len(marcadores_2023_2018_v2)
)

Correspondencias 2023→2018: 265
Marcadores especiales: 0


In [82]:
# ============================================================
# FASE 2 - PASO 9H
# EXTRAER SCIAN 2018 -> SCIAN 2023
# PÁGINAS 26 A 45
# ============================================================

lineas_2018_2023_v2 = extraer_paginas_continuas(
    pdf_2018_2023,
    list(range(26, 46))
)

tabla_2018_2023_v2, marcadores_2018_2023_v2 = (
    reconstruir_correspondencias_generico_v2(
        lineas_2018_2023_v2
    )
)

correspondencias_2018_2023_v2 = (
    tabla_2018_2023_v2.rename(
        columns={
            "Codigo_izq": "SCIAN_2018",
            "Nombre_izq": "Nombre_2018",
            "Codigo_der": "SCIAN_2023",
            "Nombre_der": "Nombre_2023"
        }
    )
)

print(
    "Correspondencias 2018→2023:",
    len(correspondencias_2018_2023_v2)
)

print(
    "Marcadores especiales:",
    len(marcadores_2018_2023_v2)
)

Correspondencias 2018→2023: 265
Marcadores especiales: 0


In [83]:
# ============================================================
# FASE 2 - PASO 9I
# VALIDACIÓN BÁSICA DE LAS DOS EXTRACCIONES
# ============================================================

validos_directa = (
    correspondencias_2023_2018_v2["SCIAN_2023"]
    .astype(str).str.match(r"^\d{6}$")
    &
    correspondencias_2023_2018_v2["SCIAN_2018"]
    .astype(str).str.match(r"^\d{6}$")
)

validos_inversa = (
    correspondencias_2018_2023_v2["SCIAN_2018"]
    .astype(str).str.match(r"^\d{6}$")
    &
    correspondencias_2018_2023_v2["SCIAN_2023"]
    .astype(str).str.match(r"^\d{6}$")
)

print("=== 2023 → 2018 ===")
print("Total:", len(correspondencias_2023_2018_v2))
print("Válidos:", validos_directa.sum())
print("Problemáticos:", (~validos_directa).sum())

print(
    "Duplicados exactos:",
    correspondencias_2023_2018_v2.duplicated(
        subset=["SCIAN_2023", "Parte", "SCIAN_2018"]
    ).sum()
)

print("\n=== 2018 → 2023 ===")
print("Total:", len(correspondencias_2018_2023_v2))
print("Válidos:", validos_inversa.sum())
print("Problemáticos:", (~validos_inversa).sum())

print(
    "Duplicados exactos:",
    correspondencias_2018_2023_v2.duplicated(
        subset=["SCIAN_2018", "Parte", "SCIAN_2023"]
    ).sum()
)

print("\nMarcadores 2023→2018:")
display(marcadores_2023_2018_v2)

print("\nMarcadores 2018→2023:")
display(marcadores_2018_2023_v2)

=== 2023 → 2018 ===
Total: 265
Válidos: 265
Problemáticos: 0
Duplicados exactos: 0

=== 2018 → 2023 ===
Total: 265
Válidos: 265
Problemáticos: 0
Duplicados exactos: 0

Marcadores 2023→2018:


""



Marcadores 2018→2023:


""


In [84]:
# ============================================================
# FASE 2 - PASO 9J
# VALIDACIÓN CRUZADA SCIAN 2018 ↔ 2023
# ============================================================

pares_2023_2018 = set(
    zip(
        correspondencias_2023_2018_v2["SCIAN_2018"]
        .astype(str).str.strip(),

        correspondencias_2023_2018_v2["SCIAN_2023"]
        .astype(str).str.strip()
    )
)

pares_2018_2023 = set(
    zip(
        correspondencias_2018_2023_v2["SCIAN_2018"]
        .astype(str).str.strip(),

        correspondencias_2018_2023_v2["SCIAN_2023"]
        .astype(str).str.strip()
    )
)

print("Pares 2023→2018:", len(pares_2023_2018))
print("Pares 2018→2023:", len(pares_2018_2023))
print(
    "Pares comunes:",
    len(pares_2023_2018 & pares_2018_2023)
)
print(
    "Solo 2023→2018:",
    len(pares_2023_2018 - pares_2018_2023)
)
print(
    "Solo 2018→2023:",
    len(pares_2018_2023 - pares_2023_2018)
)

print("\nSOLO 2023→2018:")
for x in sorted(pares_2023_2018 - pares_2018_2023):
    print(x)

print("\nSOLO 2018→2023:")
for x in sorted(pares_2018_2023 - pares_2023_2018):
    print(x)

Pares 2023→2018: 265
Pares 2018→2023: 265
Pares comunes: 265
Solo 2023→2018: 0
Solo 2018→2023: 0

SOLO 2023→2018:

SOLO 2018→2023:


In [85]:
# ============================================================
# FASE 2 - PASO 9K
# GUARDAR PUENTE VALIDADO SCIAN 2018 ↔ 2023
# ============================================================

correspondencias_2023_2018_v2.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2023_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

correspondencias_2018_2023_v2.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

# Crear mapa único validado
mapa_validado_2018_2023 = pd.DataFrame(
    sorted(pares_2023_2018),
    columns=[
        "SCIAN_2018",
        "SCIAN_2023"
    ]
)

mapa_validado_2018_2023.to_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2018_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "✓ Mapa validado SCIAN 2018↔2023 guardado:",
    len(mapa_validado_2018_2023),
    "pares"
)

✓ Mapa validado SCIAN 2018↔2023 guardado: 265 pares


## Homologación SCIAN 2018–2023

Para vincular las clases de actividad correspondientes a SCIAN México 2018 y
SCIAN México 2023 se utilizaron las tablas comparativas oficiales de INEGI en
ambos sentidos:

- SCIAN 2023 → SCIAN 2018
- SCIAN 2018 → SCIAN 2023

La extracción se realizó programáticamente a partir del documento oficial,
conservando los códigos, nombres de las clases y el campo "Parte".

La Tabla Comparativa I, correspondiente a SCIAN 2023 → SCIAN 2018, fue
identificada en las páginas 8 a 24 del documento, mientras que la Tabla
Comparativa II, correspondiente a SCIAN 2018 → SCIAN 2023, se localizó en
las páginas 26 a 45.

La validación produjo los siguientes resultados:

- 265 pares válidos en SCIAN 2023 → SCIAN 2018.
- 265 pares válidos en SCIAN 2018 → SCIAN 2023.
- 265 pares coincidentes entre ambas tablas.
- 0 relaciones exclusivas de una sola dirección.
- 0 registros problemáticos.
- 0 duplicados exactos.
- 0 marcadores especiales detectados.

La coincidencia total entre ambas direcciones permite considerar validado el
puente SCIAN 2018–2023 para la homologación histórica.

Las correspondencias parciales y las relaciones de división o fusión entre
clases se conservarán explícitamente y no se realizarán distribuciones
arbitrarias de establecimientos.

# Integración de la homologación histórica SCIAN 2007–2023

# Diagnóstico de homologación de los códigos observados en Morelia

Una vez validados los tres puentes oficiales entre versiones del SCIAN, se
contrastan las clases efectivamente observadas en los establecimientos de
comercio y servicios de Morelia con las correspondencias oficiales.

El objetivo de esta etapa es distinguir entre:

- clases con cambio explícito en las tablas oficiales;
- clases cuyo código permanece numéricamente igual entre versiones;
- clases involucradas en divisiones, fusiones o relaciones complejas;
- clases sin correspondencia directa que requieren revisión específica.

No se asignarán equivalencias automáticas cuando la correspondencia oficial
sea parcial o involucre más de una clase.

In [86]:
# ============================================================
# FASE 2 - PASO 10A
# CARGAR MAPAS VALIDADOS E INVENTARIO OBSERVADO
# ============================================================

mapa_2007_2013 = pd.read_csv(
    HOMOLOGACION / "MAPA_VALIDADO_SCIAN_2007_2013.csv",
    dtype=str
)

mapa_2013_2018 = pd.read_csv(
    HOMOLOGACION / "MAPA_VALIDADO_SCIAN_2013_2018.csv",
    dtype=str
)

mapa_2018_2023 = pd.read_csv(
    HOMOLOGACION / "MAPA_VALIDADO_SCIAN_2018_2023.csv",
    dtype=str
)

codigos_por_version = pd.read_csv(
    HOMOLOGACION / "CODIGOS_UNICOS_POR_VERSION_SCIAN.csv",
    dtype=str
)

print("Mapas cargados:")
print("2007↔2013:", len(mapa_2007_2013))
print("2013↔2018:", len(mapa_2013_2018))
print("2018↔2023:", len(mapa_2018_2023))

print("\nCódigos observados por versión:")
display(
    codigos_por_version.groupby("Version_SCIAN")
    .size()
    .reset_index(name="Codigos_observados")
)

Mapas cargados:
2007↔2013: 75
2013↔2018: 77
2018↔2023: 265

Códigos observados por versión:


,Version_SCIAN,Codigos_observados
0,SCIAN 2007,399
1,SCIAN 2013,407
2,SCIAN 2018,442
3,SCIAN 2023,421


In [87]:
# ============================================================
# FASE 2 - PASO 10B
# CLASIFICAR CARDINALIDAD DE LOS TRES MAPAS
# ============================================================

def clasificar_cardinalidad(df, origen, destino):

    salida = df.copy()

    n_destinos = (
        salida.groupby(origen)[destino]
        .nunique()
        .rename("N_destinos")
    )

    n_origenes = (
        salida.groupby(destino)[origen]
        .nunique()
        .rename("N_origenes")
    )

    salida = salida.merge(
        n_destinos,
        on=origen,
        how="left"
    )

    salida = salida.merge(
        n_origenes,
        on=destino,
        how="left"
    )

    def tipo(row):

        if row["N_destinos"] == 1 and row["N_origenes"] == 1:
            return "1 a 1"

        elif row["N_destinos"] > 1 and row["N_origenes"] == 1:
            return "1 a N"

        elif row["N_destinos"] == 1 and row["N_origenes"] > 1:
            return "N a 1"

        else:
            return "N a N"

    salida["Cardinalidad"] = salida.apply(
        tipo,
        axis=1
    )

    return salida


mapa_2007_2013_clasificado = clasificar_cardinalidad(
    mapa_2007_2013,
    "SCIAN_2007",
    "SCIAN_2013"
)

mapa_2013_2018_clasificado = clasificar_cardinalidad(
    mapa_2013_2018,
    "SCIAN_2013",
    "SCIAN_2018"
)

mapa_2018_2023_clasificado = clasificar_cardinalidad(
    mapa_2018_2023,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("=== 2007 → 2013 ===")
display(
    mapa_2007_2013_clasificado["Cardinalidad"]
    .value_counts()
    .reset_index()
)

print("\n=== 2013 → 2018 ===")
display(
    mapa_2013_2018_clasificado["Cardinalidad"]
    .value_counts()
    .reset_index()
)

print("\n=== 2018 → 2023 ===")
display(
    mapa_2018_2023_clasificado["Cardinalidad"]
    .value_counts()
    .reset_index()
)

=== 2007 → 2013 ===


,Cardinalidad,count
0,1 a 1,23
1,1 a N,22
2,N a N,21
3,N a 1,9



=== 2013 → 2018 ===


,Cardinalidad,count
0,1 a N,35
1,1 a 1,17
2,N a N,15
3,N a 1,10



=== 2018 → 2023 ===


,Cardinalidad,count
0,N a N,115
1,N a 1,104
2,1 a 1,32
3,1 a N,14


In [88]:
# ============================================================
# FASE 2 - PASO 10D
# EJECUTAR DIAGNÓSTICO EN LOS TRES PUENTES
# ============================================================

diag_2007_2013 = diagnosticar_puente(
    codigos_por_version,
    "SCIAN 2007",
    "SCIAN 2013",
    mapa_2007_2013_clasificado,
    "SCIAN_2007",
    "SCIAN_2013"
)

diag_2013_2018 = diagnosticar_puente(
    codigos_por_version,
    "SCIAN 2013",
    "SCIAN 2018",
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

diag_2018_2023 = diagnosticar_puente(
    codigos_por_version,
    "SCIAN 2018",
    "SCIAN 2023",
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("=== SCIAN 2007 → 2013 ===")
display(
    diag_2007_2013["Diagnostico"]
    .value_counts()
    .reset_index()
)

print("\n=== SCIAN 2013 → 2018 ===")
display(
    diag_2013_2018["Diagnostico"]
    .value_counts()
    .reset_index()
)

print("\n=== SCIAN 2018 → 2023 ===")
display(
    diag_2018_2023["Diagnostico"]
    .value_counts()
    .reset_index()
)

NameError: name 'diagnosticar_puente' is not defined

In [89]:
# ============================================================
# FASE 2 - PASOS 10C + 10D + 10E
# DEFINIR FUNCIÓN, EJECUTAR DIAGNÓSTICOS Y GUARDAR
# ============================================================

def diagnosticar_puente(
    codigos_df,
    version_origen,
    version_destino,
    mapa,
    col_origen,
    col_destino
):

    observados_origen = set(
        codigos_df.loc[
            codigos_df["Version_SCIAN"] == version_origen,
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    observados_destino = set(
        codigos_df.loc[
            codigos_df["Version_SCIAN"] == version_destino,
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    explicitos = set(
        mapa[col_origen]
        .astype(str)
        .str.strip()
    )

    resultados = []

    for codigo in sorted(observados_origen):

        if codigo in explicitos:

            filas = mapa[
                mapa[col_origen]
                .astype(str)
                .str.strip() == codigo
            ]

            destinos = sorted(
                filas[col_destino]
                .astype(str)
                .str.strip()
                .unique()
            )

            cardinalidades = sorted(
                filas["Cardinalidad"]
                .astype(str)
                .unique()
            )

            diagnostico = "Cambio explícito oficial"
            destino_texto = " | ".join(destinos)
            cardinalidad = " | ".join(cardinalidades)

        elif codigo in observados_destino:

            diagnostico = "Mismo código - candidato sin cambio"
            destino_texto = codigo
            cardinalidad = "1 a 1 candidato"

        else:

            diagnostico = "Revisión necesaria"
            destino_texto = ""
            cardinalidad = ""

        resultados.append({
            "Version_origen": version_origen,
            "SCIAN_origen": codigo,
            "Version_destino": version_destino,
            "SCIAN_destino_propuesto": destino_texto,
            "Diagnostico": diagnostico,
            "Cardinalidad": cardinalidad
        })

    return pd.DataFrame(resultados)


# ------------------------------------------------------------
# EJECUTAR LOS TRES PUENTES
# ------------------------------------------------------------

diag_2007_2013 = diagnosticar_puente(
    codigos_por_version,
    "SCIAN 2007",
    "SCIAN 2013",
    mapa_2007_2013_clasificado,
    "SCIAN_2007",
    "SCIAN_2013"
)

diag_2013_2018 = diagnosticar_puente(
    codigos_por_version,
    "SCIAN 2013",
    "SCIAN 2018",
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

diag_2018_2023 = diagnosticar_puente(
    codigos_por_version,
    "SCIAN 2018",
    "SCIAN 2023",
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)


# ------------------------------------------------------------
# MOSTRAR RESULTADOS
# ------------------------------------------------------------

print("=== SCIAN 2007 → 2013 ===")
display(
    diag_2007_2013["Diagnostico"]
    .value_counts()
    .reset_index()
)

print("\n=== SCIAN 2013 → 2018 ===")
display(
    diag_2013_2018["Diagnostico"]
    .value_counts()
    .reset_index()
)

print("\n=== SCIAN 2018 → 2023 ===")
display(
    diag_2018_2023["Diagnostico"]
    .value_counts()
    .reset_index()
)


# ------------------------------------------------------------
# GUARDAR
# ------------------------------------------------------------

diag_2007_2013.to_csv(
    HOMOLOGACION / "DIAGNOSTICO_OBSERVADOS_SCIAN_2007_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

diag_2013_2018.to_csv(
    HOMOLOGACION / "DIAGNOSTICO_OBSERVADOS_SCIAN_2013_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

diag_2018_2023.to_csv(
    HOMOLOGACION / "DIAGNOSTICO_OBSERVADOS_SCIAN_2018_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Diagnósticos guardados correctamente.")

=== SCIAN 2007 → 2013 ===


,Diagnostico,count
0,Mismo código - candidato sin cambio,352
1,Revisión necesaria,29
2,Cambio explícito oficial,18



=== SCIAN 2013 → 2018 ===


,Diagnostico,count
0,Mismo código - candidato sin cambio,383
1,Cambio explícito oficial,21
2,Revisión necesaria,3



=== SCIAN 2018 → 2023 ===


,Diagnostico,count
0,Mismo código - candidato sin cambio,305
1,Cambio explícito oficial,109
2,Revisión necesaria,28



✓ Diagnósticos guardados correctamente.


## Separación entre cambio de clasificación y presencia empírica

La ausencia de una clase SCIAN en los establecimientos observados de Morelia
en una versión posterior no implica necesariamente que dicha clase haya sido
eliminada o modificada por el SCIAN.

Por esta razón se separan dos dimensiones:

1. Cambio clasificatorio: determinado exclusivamente a partir de las tablas
   comparativas oficiales del SCIAN.
2. Presencia empírica: determinada a partir de la existencia o ausencia de
   establecimientos de dicha clase en los cortes históricos del DENUE para
   Morelia.

Una clase que no aparece en la tabla oficial de cambios se identifica como
"sin cambio explícito en tabla oficial", independientemente de que dicha clase
haya sido o no observada en Morelia durante la versión posterior.

In [90]:
# ============================================================
# FASE 2 - PASO 10F
# DIAGNÓSTICO CORREGIDO:
# CAMBIO OFICIAL ≠ PRESENCIA EN MORELIA
# ============================================================

def diagnosticar_puente_corregido(
    codigos_df,
    version_origen,
    version_destino,
    mapa,
    col_origen,
    col_destino
):

    observados_origen = set(
        codigos_df.loc[
            codigos_df["Version_SCIAN"] == version_origen,
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    observados_destino = set(
        codigos_df.loc[
            codigos_df["Version_SCIAN"] == version_destino,
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    mapa = mapa.copy()

    mapa[col_origen] = (
        mapa[col_origen]
        .astype(str)
        .str.strip()
    )

    mapa[col_destino] = (
        mapa[col_destino]
        .astype(str)
        .str.strip()
    )

    resultados = []

    for codigo in sorted(observados_origen):

        filas = mapa[
            mapa[col_origen] == codigo
        ]

        # ----------------------------------------------------
        # A. CAMBIO EXPLÍCITO EN TABLA OFICIAL
        # ----------------------------------------------------
        if not filas.empty:

            destinos = sorted(
                filas[col_destino]
                .dropna()
                .unique()
            )

            cardinalidades = sorted(
                filas["Cardinalidad"]
                .dropna()
                .astype(str)
                .unique()
            )

            estado_oficial = (
                "Cambio explícito oficial"
            )

            destino_propuesto = " | ".join(destinos)

            cardinalidad = " | ".join(
                cardinalidades
            )

            destinos_observados = sorted(
                set(destinos)
                & observados_destino
            )

            alguno_observado = (
                len(destinos_observados) > 0
            )

        # ----------------------------------------------------
        # B. NO APARECE EN TABLA DE CAMBIOS
        # ----------------------------------------------------
        else:

            estado_oficial = (
                "Sin cambio explícito en tabla oficial"
            )

            destino_propuesto = codigo

            cardinalidad = (
                "Sin cambio explícito"
            )

            destinos_observados = (
                [codigo]
                if codigo in observados_destino
                else []
            )

            alguno_observado = (
                codigo in observados_destino
            )

        resultados.append({
            "Version_origen": version_origen,
            "SCIAN_origen": codigo,
            "Version_destino": version_destino,

            "Estado_oficial":
                estado_oficial,

            "SCIAN_destino_oficial":
                destino_propuesto,

            "Cardinalidad":
                cardinalidad,

            "Destino_observado_en_Morelia":
                alguno_observado,

            "Codigos_destino_observados":
                " | ".join(destinos_observados)
        })

    return pd.DataFrame(resultados)

In [91]:
# ============================================================
# FASE 2 - PASO 10G
# EJECUTAR DIAGNÓSTICO CORREGIDO
# ============================================================

diag2_2007_2013 = diagnosticar_puente_corregido(
    codigos_por_version,
    "SCIAN 2007",
    "SCIAN 2013",
    mapa_2007_2013_clasificado,
    "SCIAN_2007",
    "SCIAN_2013"
)

diag2_2013_2018 = diagnosticar_puente_corregido(
    codigos_por_version,
    "SCIAN 2013",
    "SCIAN 2018",
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

diag2_2018_2023 = diagnosticar_puente_corregido(
    codigos_por_version,
    "SCIAN 2018",
    "SCIAN 2023",
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)


for titulo, df in [
    ("2007 → 2013", diag2_2007_2013),
    ("2013 → 2018", diag2_2013_2018),
    ("2018 → 2023", diag2_2018_2023)
]:

    print("\n===================================")
    print(titulo)
    print("===================================")

    print("\nESTADO OFICIAL:")
    display(
        df["Estado_oficial"]
        .value_counts()
        .reset_index()
    )

    print("\nPRESENCIA DEL DESTINO EN MORELIA:")
    display(
        df["Destino_observado_en_Morelia"]
        .value_counts()
        .reset_index()
    )


2007 → 2013

ESTADO OFICIAL:


,Estado_oficial,count
0,Sin cambio explícito en tabla oficial,381
1,Cambio explícito oficial,18



PRESENCIA DEL DESTINO EN MORELIA:


,Destino_observado_en_Morelia,count
0,True,369
1,False,30



2013 → 2018

ESTADO OFICIAL:


...

,Destino_observado_en_Morelia,count
0,True,404
1,False,3



2018 → 2023

ESTADO OFICIAL:


,Estado_oficial,count
0,Sin cambio explícito en tabla oficial,333
1,Cambio explícito oficial,109



PRESENCIA DEL DESTINO EN MORELIA:


,Destino_observado_en_Morelia,count
0,True,412
1,False,30


In [92]:
# ============================================================
# FASE 2 - PASO 10H
# GUARDAR DIAGNÓSTICOS CORREGIDOS
# ============================================================

diag2_2007_2013.to_csv(
    HOMOLOGACION /
    "DIAGNOSTICO_CORREGIDO_SCIAN_2007_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

diag2_2013_2018.to_csv(
    HOMOLOGACION /
    "DIAGNOSTICO_CORREGIDO_SCIAN_2013_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

diag2_2018_2023.to_csv(
    HOMOLOGACION /
    "DIAGNOSTICO_CORREGIDO_SCIAN_2018_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Diagnósticos corregidos guardados.")

✓ Diagnósticos corregidos guardados.


## Tipología de los cambios explícitos observados

Las clases de actividad que participan en modificaciones explícitas entre
versiones del SCIAN se clasifican según la estructura de su correspondencia:

- 1 a 1: una clase de origen corresponde a una sola clase de destino.
- 1 a N: una clase de origen se divide entre varias clases de destino.
- N a 1: varias clases de origen convergen en una sola clase de destino.
- N a N: existen relaciones múltiples tanto en el origen como en el destino.

La clasificación se realiza a nivel de clase de origen y no a nivel de fila de
la tabla de correspondencias, con el propósito de evitar que una misma clase
sea contabilizada varias veces cuando se relaciona con más de un destino.

In [93]:
# ============================================================
# FASE 2 - PASO 10I
# CLASIFICAR LOS CAMBIOS EXPLÍCITOS A NIVEL DE CÓDIGO ORIGEN
# ============================================================

def resumen_cambios_por_origen(
    diagnostico,
    mapa,
    col_origen,
    col_destino
):

    codigos_explicitos = set(
        diagnostico.loc[
            diagnostico["Estado_oficial"] ==
            "Cambio explícito oficial",
            "SCIAN_origen"
        ]
        .astype(str)
        .str.strip()
    )

    temp = mapa[
        mapa[col_origen]
        .astype(str)
        .str.strip()
        .isin(codigos_explicitos)
    ].copy()

    temp[col_origen] = (
        temp[col_origen]
        .astype(str)
        .str.strip()
    )

    temp[col_destino] = (
        temp[col_destino]
        .astype(str)
        .str.strip()
    )

    # Número de destinos de cada clase de origen
    n_destinos = (
        temp.groupby(col_origen)[col_destino]
        .nunique()
        .rename("N_destinos")
    )

    # Número de orígenes que llegan a cada destino
    n_origenes_destino = (
        mapa.groupby(col_destino)[col_origen]
        .nunique()
        .rename("N_origenes_destino")
    )

    temp = temp.merge(
        n_destinos,
        on=col_origen,
        how="left"
    )

    temp = temp.merge(
        n_origenes_destino,
        on=col_destino,
        how="left"
    )

    filas_resumen = []

    for codigo, grupo in temp.groupby(col_origen):

        destinos = sorted(
            grupo[col_destino].unique()
        )

        n_dest = len(destinos)

        max_origenes = (
            grupo["N_origenes_destino"].max()
        )

        if n_dest == 1 and max_origenes == 1:
            tipo = "1 a 1"

        elif n_dest > 1 and max_origenes == 1:
            tipo = "1 a N"

        elif n_dest == 1 and max_origenes > 1:
            tipo = "N a 1"

        else:
            tipo = "N a N"

        filas_resumen.append({
            "SCIAN_origen": codigo,
            "N_destinos": n_dest,
            "SCIAN_destinos": " | ".join(destinos),
            "Max_origenes_por_destino": int(max_origenes),
            "Tipo_relacion": tipo
        })

    return pd.DataFrame(filas_resumen)

In [94]:
# ============================================================
# FASE 2 - PASO 10J
# RESUMIR COMPLEJIDAD DE LOS CAMBIOS EXPLÍCITOS
# ============================================================

cambios_2007_2013 = resumen_cambios_por_origen(
    diag2_2007_2013,
    mapa_2007_2013_clasificado,
    "SCIAN_2007",
    "SCIAN_2013"
)

cambios_2013_2018 = resumen_cambios_por_origen(
    diag2_2013_2018,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

cambios_2018_2023 = resumen_cambios_por_origen(
    diag2_2018_2023,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("=== 2007 → 2013 ===")
display(
    cambios_2007_2013["Tipo_relacion"]
    .value_counts()
    .reset_index()
)

print("\n=== 2013 → 2018 ===")
display(
    cambios_2013_2018["Tipo_relacion"]
    .value_counts()
    .reset_index()
)

print("\n=== 2018 → 2023 ===")
display(
    cambios_2018_2023["Tipo_relacion"]
    .value_counts()
    .reset_index()
)

=== 2007 → 2013 ===


,Tipo_relacion,count
0,N a N,6
1,1 a 1,5
2,1 a N,4
3,N a 1,3



=== 2013 → 2018 ===


,Tipo_relacion,count
0,1 a 1,11
1,N a 1,5
2,N a N,3
3,1 a N,2



=== 2018 → 2023 ===


,Tipo_relacion,count
0,N a 1,90
1,1 a 1,11
2,N a N,8


In [95]:
# ============================================================
# FASE 2 - PASO 10K
# GUARDAR TIPOLOGÍA DE CAMBIOS
# ============================================================

cambios_2007_2013.to_csv(
    HOMOLOGACION /
    "CAMBIOS_EXPLICITOS_OBSERVADOS_2007_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

cambios_2013_2018.to_csv(
    HOMOLOGACION /
    "CAMBIOS_EXPLICITOS_OBSERVADOS_2013_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

cambios_2018_2023.to_csv(
    HOMOLOGACION /
    "CAMBIOS_EXPLICITOS_OBSERVADOS_2018_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Tipologías de cambios guardadas.")

✓ Tipologías de cambios guardadas.


## Identificación de correspondencias complejas

Las correspondencias explícitas se distinguen según su posibilidad de
homologación.

Las relaciones 1 a 1 permiten establecer una correspondencia directa, mientras
que las relaciones N a 1 permiten agregar varias clases de origen hacia una
misma categoría de destino.

En contraste, las relaciones 1 a N y N a N requieren un tratamiento específico,
debido a que no es posible distribuir retrospectivamente los establecimientos
entre varias clases de destino sin información adicional.

Por esta razón, estas relaciones se identifican separadamente antes de construir
los grupos de homologación histórica.

In [96]:
# ============================================================
# FASE 2 - PASO 10L
# CLASIFICAR OPERATIVAMENTE LOS CAMBIOS EXPLÍCITOS
# ============================================================

def preparar_cambios(df, puente):

    temp = df.copy()

    temp["Puente"] = puente

    temp["Tratamiento"] = temp["Tipo_relacion"].map({
        "1 a 1": "Correspondencia directa",
        "N a 1": "Agregación directa al destino",
        "1 a N": "Requiere grupo homologado - división",
        "N a N": "Requiere grupo homologado - relación compleja"
    })

    return temp


cambios_operativos = pd.concat(
    [
        preparar_cambios(
            cambios_2007_2013,
            "2007 → 2013"
        ),

        preparar_cambios(
            cambios_2013_2018,
            "2013 → 2018"
        ),

        preparar_cambios(
            cambios_2018_2023,
            "2018 → 2023"
        )
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# RESUMEN GENERAL
# ------------------------------------------------------------

print("=== RESUMEN OPERATIVO ===")

display(
    cambios_operativos.groupby(
        ["Puente", "Tratamiento"]
    )
    .size()
    .reset_index(name="Cantidad")
)


# ------------------------------------------------------------
# SOLO RELACIONES COMPLEJAS
# ------------------------------------------------------------

complejos = cambios_operativos[
    cambios_operativos["Tipo_relacion"]
    .isin(["1 a N", "N a N"])
].copy()

print(
    "\nTotal de clases de origen que requieren "
    "tratamiento especial:",
    len(complejos)
)

display(
    complejos[
        [
            "Puente",
            "SCIAN_origen",
            "N_destinos",
            "SCIAN_destinos",
            "Max_origenes_por_destino",
            "Tipo_relacion"
        ]
    ]
    .sort_values(
        ["Puente", "SCIAN_origen"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# GUARDAR
# ------------------------------------------------------------

cambios_operativos.to_csv(
    HOMOLOGACION /
    "CAMBIOS_EXPLICITOS_CLASIFICACION_OPERATIVA.csv",
    index=False,
    encoding="utf-8-sig"
)

complejos.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIAS_COMPLEJAS_PARA_HOMOLOGACION.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\n✓ Clasificación operativa guardada."
)

print(
    "✓ Correspondencias complejas guardadas:",
    len(complejos)
)

=== RESUMEN OPERATIVO ===


,Puente,Tratamiento,Cantidad
0,2007 → 2013,Agregación directa al destino,3
1,2007 → 2013,Correspondencia directa,5
2,2007 → 2013,Requiere grupo homologado - división,4
3,2007 → 2013,Requiere grupo homologado - relación compleja,6
4,2013 → 2018,Agregación directa al destino,5
5,2013 → 2018,Correspondencia directa,11
6,2013 → 2018,Requiere grupo homologado - división,2
7,2013 → 2018,Requiere grupo homologado - relación compleja,3
8,2018 → 2023,Agregación directa al destino,90
9,2018 → 2023,Correspondencia directa,11



Total de clases de origen que requieren tratamiento especial: 23


,Puente,SCIAN_origen,N_destinos,SCIAN_destinos,Max_origenes_por_destino,Tipo_relacion
0,2007 → 2013,465215,2,465213 | 465215,2,N a N
1,2007 → 2013,522490,2,522460 | 522490,1,1 a N
2,2007 → 2013,531119,2,531116 | 531119,1,1 a N
3,2007 → 2013,621111,2,621111 | 621115,2,N a N
4,2007 → 2013,621112,2,621112 | 621116,2,N a N
5,2007 → 2013,621113,2,621113 | 621115,2,N a N
6,2007 → 2013,722110,5,722511 | 722512 | 722513 | 722514 | 722515,2,N a N
7,2007 → 2013,722212,2,722517 | 722518,1,1 a N
8,2007 → 2013,722219,6,722511 | 722512 | 722513 | 722514 | 722515 | 7...,2,N a N
9,2007 → 2013,812320,2,812321 | 812322,1,1 a N



✓ Clasificación operativa guardada.
✓ Correspondencias complejas guardadas: 23


## Construcción de grupos de correspondencia compleja

Las clases identificadas como relaciones 1 a N o N a N pueden formar parte
de un mismo conjunto de correspondencias. Por ello, antes de definir una
regla de homologación se identifican los componentes conectados completos
de cada puente entre versiones del SCIAN.

Un componente reúne todas las clases de origen y destino vinculadas entre sí
por las correspondencias oficiales. Este procedimiento permite evitar el
tratamiento aislado de clases que en realidad pertenecen a una misma
reestructuración clasificatoria.

La construcción de estos grupos no implica todavía una homologación.
Su propósito es delimitar la unidad mínima de análisis para evaluar
posteriormente si es posible conservar el nivel de clase o si se requiere
una agregación superior.

In [97]:
# ============================================================
# FASE 2 - PASO 10M
# CONSTRUIR COMPONENTES CONECTADOS DE RELACIONES COMPLEJAS
# ============================================================

from collections import defaultdict


def construir_componentes_complejos(
    mapa,
    cambios,
    col_origen,
    col_destino,
    puente
):

    # Códigos de origen que ya identificamos como complejos
    objetivos = set(
        cambios.loc[
            cambios["Tipo_relacion"].isin(["1 a N", "N a N"]),
            "SCIAN_origen"
        ]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # Construir grafo bipartito
    # O = versión origen
    # D = versión destino
    # --------------------------------------------------------

    grafo = defaultdict(set)

    for _, fila in mapa.iterrows():

        origen = str(fila[col_origen]).strip()
        destino = str(fila[col_destino]).strip()

        nodo_o = "O::" + origen
        nodo_d = "D::" + destino

        grafo[nodo_o].add(nodo_d)
        grafo[nodo_d].add(nodo_o)

    visitados = set()
    componentes = []

    grupo_n = 1

    for codigo in sorted(objetivos):

        inicio = "O::" + codigo

        if inicio in visitados:
            continue

        # Búsqueda de todo el componente conectado
        pila = [inicio]
        componente = set()

        while pila:

            nodo = pila.pop()

            if nodo in componente:
                continue

            componente.add(nodo)
            visitados.add(nodo)

            for vecino in grafo[nodo]:
                if vecino not in componente:
                    pila.append(vecino)

        origenes = sorted(
            nodo.replace("O::", "")
            for nodo in componente
            if nodo.startswith("O::")
        )

        destinos = sorted(
            nodo.replace("D::", "")
            for nodo in componente
            if nodo.startswith("D::")
        )

        origenes_complejos = sorted(
            set(origenes) & objetivos
        )

        componentes.append({
            "Puente": puente,
            "Grupo": f"G{grupo_n:02d}",
            "N_origenes": len(origenes),
            "N_destinos": len(destinos),
            "N_origenes_complejos_observados":
                len(origenes_complejos),
            "SCIAN_origenes":
                " | ".join(origenes),
            "SCIAN_destinos":
                " | ".join(destinos),
            "Orígenes_complejos_observados":
                " | ".join(origenes_complejos),
            "Total_codigos_componente":
                len(origenes) + len(destinos)
        })

        grupo_n += 1

    return pd.DataFrame(componentes)

In [98]:
# ============================================================
# FASE 2 - PASO 10N
# COMPONENTES COMPLEJOS EN LOS TRES PUENTES
# ============================================================

componentes_2007_2013 = construir_componentes_complejos(
    mapa_2007_2013_clasificado,
    cambios_2007_2013,
    "SCIAN_2007",
    "SCIAN_2013",
    "2007 → 2013"
)

componentes_2013_2018 = construir_componentes_complejos(
    mapa_2013_2018_clasificado,
    cambios_2013_2018,
    "SCIAN_2013",
    "SCIAN_2018",
    "2013 → 2018"
)

componentes_2018_2023 = construir_componentes_complejos(
    mapa_2018_2023_clasificado,
    cambios_2018_2023,
    "SCIAN_2018",
    "SCIAN_2023",
    "2018 → 2023"
)


componentes_complejos = pd.concat(
    [
        componentes_2007_2013,
        componentes_2013_2018,
        componentes_2018_2023
    ],
    ignore_index=True
)


print("=== NÚMERO DE GRUPOS COMPLEJOS ===")

display(
    componentes_complejos.groupby("Puente")
    .size()
    .reset_index(name="Grupos_complejos")
)

print(
    "\nTotal de grupos complejos:",
    len(componentes_complejos)
)

print("\n=== TAMAÑO DE LOS COMPONENTES ===")

display(
    componentes_complejos[
        [
            "Puente",
            "Grupo",
            "N_origenes",
            "N_destinos",
            "N_origenes_complejos_observados",
            "Total_codigos_componente"
        ]
    ]
    .sort_values(
        "Total_codigos_componente",
        ascending=False
    )
    .reset_index(drop=True)
)

=== NÚMERO DE GRUPOS COMPLEJOS ===


,Puente,Grupos_complejos
0,2007 → 2013,8
1,2013 → 2018,4
2,2018 → 2023,3



Total de grupos complejos: 15

=== TAMAÑO DE LOS COMPONENTES ===


,Puente,Grupo,N_origenes,N_destinos,N_origenes_complejos_observados,Total_codigos_componente
0,2018 → 2023,G01,84,84,1,168
1,2018 → 2023,G02,17,15,6,32
2,2013 → 2018,G03,2,7,2,9
3,2007 → 2013,G06,2,6,2,8
4,2013 → 2018,G04,2,3,1,5
5,2007 → 2013,G04,2,3,2,5
6,2007 → 2013,G05,2,3,1,5
7,2018 → 2023,G03,2,2,1,4
8,2007 → 2013,G01,2,2,1,4
9,2007 → 2013,G02,1,2,1,3


In [99]:
# ============================================================
# FASE 2 - PASO 10O
# GUARDAR COMPONENTES COMPLEJOS
# ============================================================

componentes_2007_2013.to_csv(
    HOMOLOGACION /
    "COMPONENTES_COMPLEJOS_SCIAN_2007_2013.csv",
    index=False,
    encoding="utf-8-sig"
)

componentes_2013_2018.to_csv(
    HOMOLOGACION /
    "COMPONENTES_COMPLEJOS_SCIAN_2013_2018.csv",
    index=False,
    encoding="utf-8-sig"
)

componentes_2018_2023.to_csv(
    HOMOLOGACION /
    "COMPONENTES_COMPLEJOS_SCIAN_2018_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

componentes_complejos.to_csv(
    HOMOLOGACION /
    "COMPONENTES_COMPLEJOS_SCIAN_CONSOLIDADOS.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Componentes complejos guardados.")

✓ Componentes complejos guardados.


## Peso empírico de los grupos complejos en Morelia

La existencia de una correspondencia compleja entre versiones del SCIAN no
implica que todas las clases involucradas tengan la misma importancia en la
estructura económica observada en Morelia.

Por ello, antes de establecer una regla de homologación, se cuantifica el
número de establecimientos asociado a cada componente complejo en los años
correspondientes.

Este análisis permite distinguir entre problemas clasificatorios de impacto
reducido y reestructuraciones que afectan una proporción importante de los
establecimientos analizados.

In [100]:
# ============================================================
# FASE 2 - PASO 10P
# VER ESTRUCTURA DEL INVENTARIO HISTÓRICO
# ============================================================

print("Columnas de inventario_scian:")
print(inventario_scian.columns.tolist())

print("\nPrimeras filas:")
display(inventario_scian.head())

Columnas de inventario_scian:
['Año', 'Version_SCIAN', 'SCIAN_original', 'Sector', 'Establecimientos']

Primeras filas:


,Año,Version_SCIAN,SCIAN_original,Sector,Establecimientos
0,2010,SCIAN 2007,461110,46,4179
1,2010,SCIAN 2007,722212,72,1975
2,2010,SCIAN 2007,722219,72,1707
3,2010,SCIAN 2007,812110,81,1582
4,2010,SCIAN 2007,463211,46,1481


## Cuantificación del peso de los componentes complejos

Para cada componente complejo se cuantifica su presencia en los establecimientos
observados de comercio y servicios de Morelia.

La medición respeta la versión del SCIAN utilizada en cada edición histórica:
los códigos de origen se emplean durante los años clasificados con la versión
anterior y los códigos de destino durante los años clasificados con la versión
posterior.

El peso de cada componente se calcula como la proporción de establecimientos
pertenecientes al grupo respecto del total de establecimientos de comercio y
servicios incluidos en el análisis para cada año.

Este procedimiento permite evaluar la importancia empírica de cada problema
de homologación antes de definir una regla definitiva de agregación.

In [101]:
# ============================================================
# FASE 2 - PASO 10Q
# CONSTRUIR MEMBRESÍA DE LOS COMPONENTES COMPLEJOS
# ============================================================

def expandir_componentes(
    componentes,
    version_origen,
    version_destino
):

    filas = []

    for _, row in componentes.iterrows():

        puente = row["Puente"]
        grupo = row["Grupo"]

        # ID único porque G01, G02, etc. se repiten entre puentes
        grupo_id = f"{puente} | {grupo}"

        origenes = [
            x.strip()
            for x in str(row["SCIAN_origenes"]).split("|")
            if x.strip()
        ]

        destinos = [
            x.strip()
            for x in str(row["SCIAN_destinos"]).split("|")
            if x.strip()
        ]

        for codigo in origenes:

            filas.append({
                "Grupo_ID": grupo_id,
                "Puente": puente,
                "Grupo": grupo,
                "Version_SCIAN": version_origen,
                "Lado": "Origen",
                "SCIAN_original": codigo
            })

        for codigo in destinos:

            filas.append({
                "Grupo_ID": grupo_id,
                "Puente": puente,
                "Grupo": grupo,
                "Version_SCIAN": version_destino,
                "Lado": "Destino",
                "SCIAN_original": codigo
            })

    return pd.DataFrame(filas)


membresia_2007_2013 = expandir_componentes(
    componentes_2007_2013,
    "SCIAN 2007",
    "SCIAN 2013"
)

membresia_2013_2018 = expandir_componentes(
    componentes_2013_2018,
    "SCIAN 2013",
    "SCIAN 2018"
)

membresia_2018_2023 = expandir_componentes(
    componentes_2018_2023,
    "SCIAN 2018",
    "SCIAN 2023"
)

membresia_componentes = pd.concat(
    [
        membresia_2007_2013,
        membresia_2013_2018,
        membresia_2018_2023
    ],
    ignore_index=True
)

print(
    "Relaciones grupo-versión-código:",
    len(membresia_componentes)
)

print(
    "Grupos únicos:",
    membresia_componentes["Grupo_ID"].nunique()
)

display(membresia_componentes.head(20))

Relaciones grupo-versión-código: 258
Grupos únicos: 15


,Grupo_ID,Puente,Grupo,Version_SCIAN,Lado,SCIAN_original
0,2007 → 2013 | G01,2007 → 2013,G01,SCIAN 2007,Origen,465213
1,2007 → 2013 | G01,2007 → 2013,G01,SCIAN 2007,Origen,465215
2,2007 → 2013 | G01,2007 → 2013,G01,SCIAN 2013,Destino,465213
3,2007 → 2013 | G01,2007 → 2013,G01,SCIAN 2013,Destino,465215
4,2007 → 2013 | G02,2007 → 2013,G02,SCIAN 2007,Origen,522490
5,2007 → 2013 | G02,2007 → 2013,G02,SCIAN 2013,Destino,522460
6,2007 → 2013 | G02,2007 → 2013,G02,SCIAN 2013,Destino,522490
7,2007 → 2013 | G03,2007 → 2013,G03,SCIAN 2007,Origen,531119
8,2007 → 2013 | G03,2007 → 2013,G03,SCIAN 2013,Destino,531116
9,2007 → 2013 | G03,2007 → 2013,G03,SCIAN 2013,Destino,531119


In [103]:
# ============================================================
# FASE 2 - PASO 10R
# PESO ANUAL DE CADA COMPONENTE COMPLEJO EN MORELIA
# ============================================================

# Asegurar tipos correctos
inventario_scian["SCIAN_original"] = (
    inventario_scian["SCIAN_original"]
    .astype(str)
    .str.strip()
)

inventario_scian["Version_SCIAN"] = (
    inventario_scian["Version_SCIAN"]
    .astype(str)
    .str.strip()
)

inventario_scian["Establecimientos"] = pd.to_numeric(
    inventario_scian["Establecimientos"],
    errors="coerce"
).fillna(0)


# ------------------------------------------------------------
# Denominador anual:
# total comercio + servicios incluidos en nuestro inventario
# ------------------------------------------------------------

totales_anuales = (
    inventario_scian.groupby("Año")["Establecimientos"]
    .sum()
    .reset_index(name="Total_comercio_servicios")
)


# ------------------------------------------------------------
# Cruzar códigos de cada componente con inventario histórico
# ------------------------------------------------------------

peso_detalle = membresia_componentes.merge(
    inventario_scian,
    on=[
        "Version_SCIAN",
        "SCIAN_original"
    ],
    how="left"
)


# Solo observaciones realmente presentes en un año
peso_detalle = peso_detalle[
    peso_detalle["Año"].notna()
].copy()

peso_detalle["Año"] = (
    peso_detalle["Año"]
    .astype(int)
)


# ------------------------------------------------------------
# Sumar establecimientos por componente y año
# ------------------------------------------------------------

peso_componentes_anual = (
    peso_detalle.groupby(
        [
            "Grupo_ID",
            "Puente",
            "Grupo",
            "Año",
            "Version_SCIAN"
        ],
        as_index=False
    )["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos":
            "Establecimientos_grupo"
        }
    )
)


# Añadir denominador anual
peso_componentes_anual = (
    peso_componentes_anual.merge(
        totales_anuales,
        on="Año",
        how="left"
    )
)


# Porcentaje del universo comercio + servicios analizado
peso_componentes_anual[
    "Porcentaje_comercio_servicios"
] = (
    peso_componentes_anual[
        "Establecimientos_grupo"
    ]
    /
    peso_componentes_anual[
        "Total_comercio_servicios"
    ]
    * 100
)


display(
    peso_componentes_anual.sort_values(
        [
            "Puente",
            "Grupo",
            "Año"
        ]
    )
)

ValueError: You are trying to merge on int64 and str columns for key 'Año'. If you wish to proceed you should use pd.concat

In [104]:
# ============================================================
# CORRECCIÓN PASO 10R
# UNIFICAR TIPO DE DATO DE "Año"
# ============================================================

# Asegurar que Año sea numérico en el inventario
inventario_scian["Año"] = pd.to_numeric(
    inventario_scian["Año"],
    errors="coerce"
).astype("Int64")

# Asegurar establecimientos numéricos
inventario_scian["Establecimientos"] = pd.to_numeric(
    inventario_scian["Establecimientos"],
    errors="coerce"
).fillna(0)

# ------------------------------------------------------------
# DENOMINADOR ANUAL
# Total de establecimientos de comercio + servicios
# incluidos en nuestro inventario
# ------------------------------------------------------------

totales_anuales = (
    inventario_scian
    .groupby("Año", as_index=False)["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos": "Total_comercio_servicios"
        }
    )
)

# ------------------------------------------------------------
# CRUZAR COMPONENTES CON INVENTARIO
# ------------------------------------------------------------

peso_detalle = membresia_componentes.merge(
    inventario_scian,
    on=[
        "Version_SCIAN",
        "SCIAN_original"
    ],
    how="left"
)

# Quitar códigos que no aparecen empíricamente
peso_detalle = peso_detalle[
    peso_detalle["Año"].notna()
].copy()

# Mismo tipo de dato en ambas tablas
peso_detalle["Año"] = pd.to_numeric(
    peso_detalle["Año"],
    errors="coerce"
).astype("Int64")

totales_anuales["Año"] = pd.to_numeric(
    totales_anuales["Año"],
    errors="coerce"
).astype("Int64")

# ------------------------------------------------------------
# SUMAR ESTABLECIMIENTOS POR GRUPO Y AÑO
# ------------------------------------------------------------

peso_componentes_anual = (
    peso_detalle
    .groupby(
        [
            "Grupo_ID",
            "Puente",
            "Grupo",
            "Año",
            "Version_SCIAN"
        ],
        as_index=False
    )["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos":
            "Establecimientos_grupo"
        }
    )
)

# ------------------------------------------------------------
# AGREGAR TOTAL ANUAL
# ------------------------------------------------------------

peso_componentes_anual = (
    peso_componentes_anual.merge(
        totales_anuales,
        on="Año",
        how="left"
    )
)

# ------------------------------------------------------------
# CALCULAR PORCENTAJE
# ------------------------------------------------------------

peso_componentes_anual[
    "Porcentaje_comercio_servicios"
] = (
    peso_componentes_anual[
        "Establecimientos_grupo"
    ]
    /
    peso_componentes_anual[
        "Total_comercio_servicios"
    ]
    * 100
)

print("✓ Peso anual calculado correctamente.")

display(
    peso_componentes_anual.sort_values(
        [
            "Puente",
            "Grupo",
            "Año"
        ]
    )
)

✓ Peso anual calculado correctamente.


,Grupo_ID,Puente,Grupo,Año,Version_SCIAN,Establecimientos_grupo,Total_comercio_servicios,Porcentaje_comercio_servicios
0,2007 → 2013 | G01,2007 → 2013,G01,2010,SCIAN 2007,100.0,34656,0.288550
1,2007 → 2013 | G01,2007 → 2013,G01,2011,SCIAN 2007,102.0,35093,0.290656
2,2007 → 2013 | G01,2007 → 2013,G01,2012,SCIAN 2007,102.0,35289,0.289042
3,2007 → 2013 | G01,2007 → 2013,G01,2013,SCIAN 2007,108.0,35797,0.301701
4,2007 → 2013 | G01,2007 → 2013,G01,2015,SCIAN 2013,136.0,37677,0.360963
...,...,...,...,...,...,...,...,...
114,2018 → 2023 | G03,2018 → 2023,G03,2022,SCIAN 2018,430.0,41303,1.041087
115,2018 → 2023 | G03,2018 → 2023,G03,2023,SCIAN 2018,432.0,41323,1.045423
116,2018 → 2023 | G03,2018 → 2023,G03,2024,SCIAN 2023,448.0,44931,0.997084
117,2018 → 2023 | G03,2018 → 2023,G03,2025,SCIAN 2023,452.0,45353,0.996626


In [105]:
resumen_peso_componentes = (
    peso_componentes_anual.groupby(
        [
            "Grupo_ID",
            "Puente",
            "Grupo"
        ]
    )
    .agg(
        Min_establecimientos=(
            "Establecimientos_grupo",
            "min"
        ),
        Max_establecimientos=(
            "Establecimientos_grupo",
            "max"
        ),
        Promedio_establecimientos=(
            "Establecimientos_grupo",
            "mean"
        ),
        Min_porcentaje=(
            "Porcentaje_comercio_servicios",
            "min"
        ),
        Max_porcentaje=(
            "Porcentaje_comercio_servicios",
            "max"
        ),
        Promedio_porcentaje=(
            "Porcentaje_comercio_servicios",
            "mean"
        )
    )
    .reset_index()
)

tam_componentes = (
    componentes_complejos[
        [
            "Puente",
            "Grupo",
            "N_origenes",
            "N_destinos",
            "Total_codigos_componente"
        ]
    ]
    .copy()
)

resumen_peso_componentes = (
    resumen_peso_componentes.merge(
        tam_componentes,
        on=[
            "Puente",
            "Grupo"
        ],
        how="left"
    )
)

display(
    resumen_peso_componentes[
        [
            "Puente",
            "Grupo",
            "N_origenes",
            "N_destinos",
            "Max_establecimientos",
            "Max_porcentaje",
            "Promedio_porcentaje",
            "Total_codigos_componente"
        ]
    ]
    .sort_values(
        "Max_porcentaje",
        ascending=False
    )
    .reset_index(drop=True)
)


,Puente,Grupo,N_origenes,N_destinos,Max_establecimientos,Max_porcentaje,Promedio_porcentaje,Total_codigos_componente
0,2018 → 2023,G01,84,84,20292.0,45.063004,43.913381,168
1,2007 → 2013,G06,2,6,3977.0,10.470579,7.597193,8
2,2007 → 2013,G07,1,2,1977.0,5.698869,4.464446,3
3,2007 → 2013,G04,2,3,1032.0,2.712373,2.672423,5
4,2018 → 2023,G03,2,2,452.0,1.052268,1.025690,4
5,2013 → 2018,G04,2,3,380.0,0.942219,0.921798,5
6,2007 → 2013,G01,2,2,146.0,0.378867,0.325319,4
7,2013 → 2018,G02,1,2,55.0,0.133204,0.113331,3
8,2007 → 2013,G02,1,2,45.0,0.116774,0.068587,3
9,2018 → 2023,G02,17,15,45.0,0.116195,0.088387,32


## Verificación del componente del comercio al por menor

El componente G01 del puente SCIAN 2018–2023 presenta una estructura
excepcionalmente grande, con 84 clases de origen y 84 clases de destino.

Debido a su magnitud, se verifica si este componente corresponde al conjunto
del comercio al por menor observado en Morelia. Para ello se compara, año por
año, el número de establecimientos pertenecientes al componente con el total
de establecimientos clasificados en el sector 46.

Esta comprobación permitirá determinar si la complejidad observada corresponde
a una reestructuración interna del sector y no a una pérdida de comparabilidad
del agregado comercio al por menor.

In [106]:
# ============================================================
# FASE 2 - PASO 10V
# ¿G01 2018-2023 = COMERCIO AL POR MENOR?
# ============================================================

# Asegurar Sector como texto
inventario_scian["Sector"] = (
    inventario_scian["Sector"]
    .astype(str)
    .str.strip()
)

# Total anual del sector 46
comercio_anual = (
    inventario_scian[
        inventario_scian["Sector"] == "46"
    ]
    .groupby(
        ["Año", "Version_SCIAN"],
        as_index=False
    )["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos":
            "Total_sector_46"
        }
    )
)

# G01 del puente 2018 → 2023
g01_comercio = (
    peso_componentes_anual[
        peso_componentes_anual["Grupo_ID"]
        == "2018 → 2023 | G01"
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "Establecimientos_grupo"
        ]
    ]
    .copy()
)

# Comparar
verificacion_g01 = g01_comercio.merge(
    comercio_anual,
    on=[
        "Año",
        "Version_SCIAN"
    ],
    how="left"
)

verificacion_g01["Diferencia"] = (
    verificacion_g01["Establecimientos_grupo"]
    -
    verificacion_g01["Total_sector_46"]
)

verificacion_g01["Coincide_exacto"] = (
    verificacion_g01["Diferencia"] == 0
)

display(verificacion_g01)

print(
    "\nAños con coincidencia exacta:",
    verificacion_g01["Coincide_exacto"].sum(),
    "de",
    len(verificacion_g01)
)

print(
    "Diferencia absoluta total:",
    verificacion_g01["Diferencia"].abs().sum()
)

,Año,Version_SCIAN,Establecimientos_grupo,Total_sector_46,Diferencia,Coincide_exacto
0,2018,SCIAN 2018,17452.0,17452,0.0,True
1,2019,SCIAN 2018,17642.0,17642,0.0,True
2,2020,SCIAN 2018,17984.0,17984,0.0,True
3,2021,SCIAN 2018,17992.0,17992,0.0,True
4,2022,SCIAN 2018,17997.0,17997,0.0,True
5,2023,SCIAN 2018,17996.0,17996,0.0,True
6,2024,SCIAN 2023,19761.0,19761,0.0,True
7,2025,SCIAN 2023,19994.0,19994,0.0,True
8,2026,SCIAN 2023,20292.0,20292,0.0,True



Años con coincidencia exacta: 9 de 9
Diferencia absoluta total: 0.0


In [107]:
# ============================================================
# FASE 2 - PASO 10W
# VERIFICAR COBERTURA DE CLASES DEL SECTOR 46
# ============================================================

for version in ["SCIAN 2018", "SCIAN 2023"]:

    codigos_comercio = set(
        inventario_scian.loc[
            (
                inventario_scian["Version_SCIAN"] == version
            )
            &
            (
                inventario_scian["Sector"] == "46"
            ),
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    codigos_g01 = set(
        membresia_componentes.loc[
            (
                membresia_componentes["Grupo_ID"]
                == "2018 → 2023 | G01"
            )
            &
            (
                membresia_componentes["Version_SCIAN"]
                == version
            ),
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    print("\n==============================")
    print(version)
    print("==============================")

    print(
        "Clases de comercio observadas:",
        len(codigos_comercio)
    )

    print(
        "Clases del G01:",
        len(codigos_g01)
    )

    print(
        "Clases observadas de comercio fuera de G01:",
        len(codigos_comercio - codigos_g01)
    )

    if codigos_comercio - codigos_g01:
        print(
            sorted(
                codigos_comercio - codigos_g01
            )
        )


SCIAN 2018
Clases de comercio observadas: 83
Clases del G01: 84
Clases observadas de comercio fuera de G01: 0

SCIAN 2023
Clases de comercio observadas: 83
Clases del G01: 84
Clases observadas de comercio fuera de G01: 0


In [108]:
# ============================================================
# FASE 2 - PASO 10X
# IDENTIFICAR CLASES DEL G01 NO OBSERVADAS EN MORELIA
# ============================================================

for version in ["SCIAN 2018", "SCIAN 2023"]:

    codigos_comercio = set(
        inventario_scian.loc[
            (
                inventario_scian["Version_SCIAN"] == version
            )
            &
            (
                inventario_scian["Sector"] == "46"
            ),
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    codigos_g01 = set(
        membresia_componentes.loc[
            (
                membresia_componentes["Grupo_ID"]
                == "2018 → 2023 | G01"
            )
            &
            (
                membresia_componentes["Version_SCIAN"]
                == version
            ),
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    print("\n", version)

    print(
        "Clases G01 no observadas en Morelia:",
        sorted(codigos_g01 - codigos_comercio)
    )


 SCIAN 2018
Clases G01 no observadas en Morelia: ['468319']

 SCIAN 2023
Clases G01 no observadas en Morelia: ['468319']


In [109]:
# ============================================================
# FASE 2 - PASO 10Y
# COMPROBAR QUE G01 = TOTAL OBSERVADO DEL SECTOR 46
# ============================================================

comercio_anual = (
    inventario_scian[
        inventario_scian["Sector"] == "46"
    ]
    .groupby(
        ["Año", "Version_SCIAN"],
        as_index=False
    )["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos":
            "Total_sector_46"
        }
    )
)

g01_comercio = (
    peso_componentes_anual[
        peso_componentes_anual["Grupo_ID"]
        == "2018 → 2023 | G01"
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "Establecimientos_grupo"
        ]
    ]
    .copy()
)

verificacion_g01 = g01_comercio.merge(
    comercio_anual,
    on=["Año", "Version_SCIAN"],
    how="left"
)

verificacion_g01["Diferencia"] = (
    verificacion_g01["Establecimientos_grupo"]
    - verificacion_g01["Total_sector_46"]
)

verificacion_g01["Coincide_exacto"] = (
    verificacion_g01["Diferencia"] == 0
)

display(verificacion_g01)

print(
    "\nAños con coincidencia exacta:",
    verificacion_g01["Coincide_exacto"].sum(),
    "de",
    len(verificacion_g01)
)

print(
    "Diferencia absoluta total:",
    verificacion_g01["Diferencia"].abs().sum()
)

,Año,Version_SCIAN,Establecimientos_grupo,Total_sector_46,Diferencia,Coincide_exacto
0,2018,SCIAN 2018,17452.0,17452,0.0,True
1,2019,SCIAN 2018,17642.0,17642,0.0,True
2,2020,SCIAN 2018,17984.0,17984,0.0,True
3,2021,SCIAN 2018,17992.0,17992,0.0,True
4,2022,SCIAN 2018,17997.0,17997,0.0,True
5,2023,SCIAN 2018,17996.0,17996,0.0,True
6,2024,SCIAN 2023,19761.0,19761,0.0,True
7,2025,SCIAN 2023,19994.0,19994,0.0,True
8,2026,SCIAN 2023,20292.0,20292,0.0,True



Años con coincidencia exacta: 9 de 9
Diferencia absoluta total: 0.0


### Resolución del componente G01 del puente SCIAN 2018–2023

El componente G01 contiene 84 clases de SCIAN 2018 y 84 clases de SCIAN
2023. Al contrastarlo con los establecimientos observados en Morelia se
comprobó que incluye la totalidad de las clases de comercio al por menor
presentes en ambas versiones.

En SCIAN 2018 se observaron 83 clases del sector 46 y en SCIAN 2023 también
83; todas ellas pertenecen al componente G01. La única clase integrante del
componente no observada en Morelia en ninguna de las dos versiones fue
468319.

La comparación anual entre los establecimientos del componente G01 y el
total del sector 46 mostró coincidencia exacta en los nueve años analizados
(2018–2026), con una diferencia absoluta acumulada igual a cero.

En consecuencia, la reestructuración interna de las clases de comercio al
por menor entre SCIAN 2018 y SCIAN 2023 no afecta la comparabilidad del
agregado sectorial. Para este puente, el sector 46 se considera la unidad
homologada para el análisis longitudinal agregado del comercio al por menor.

Las clases de seis dígitos se conservarán para los análisis internos de cada
versión del SCIAN, pero no se asumirán como directamente comparables entre
2018 y 2023 cuando formen parte de esta reestructuración.

In [110]:
# ============================================================
# FASE 2 - PASO 11A
# IDENTIFICAR SECTORES DE LOS 14 COMPONENTES PENDIENTES
# ============================================================

membresia_componentes["SCIAN_original"] = (
    membresia_componentes["SCIAN_original"]
    .astype(str)
    .str.strip()
)

membresia_componentes["Sector_codigo"] = (
    membresia_componentes["SCIAN_original"]
    .str[:2]
)

# Sectores del lado origen
sectores_origen = (
    membresia_componentes[
        membresia_componentes["Lado"] == "Origen"
    ]
    .groupby(
        ["Grupo_ID", "Puente", "Grupo"]
    )["Sector_codigo"]
    .agg(
        lambda x:
        " | ".join(sorted(set(x)))
    )
    .reset_index(
        name="Sectores_origen"
    )
)

# Sectores del lado destino
sectores_destino = (
    membresia_componentes[
        membresia_componentes["Lado"] == "Destino"
    ]
    .groupby(
        ["Grupo_ID", "Puente", "Grupo"]
    )["Sector_codigo"]
    .agg(
        lambda x:
        " | ".join(sorted(set(x)))
    )
    .reset_index(
        name="Sectores_destino"
    )
)

sectores_componentes = (
    sectores_origen.merge(
        sectores_destino,
        on=[
            "Grupo_ID",
            "Puente",
            "Grupo"
        ],
        how="outer"
    )
)

# Incorporar tamaño y peso empírico
revision_componentes = (
    resumen_peso_componentes.merge(
        sectores_componentes,
        on=[
            "Grupo_ID",
            "Puente",
            "Grupo"
        ],
        how="left"
    )
)

# Quitar G01 2018→2023, ya resuelto
revision_componentes = revision_componentes[
    revision_componentes["Grupo_ID"]
    != "2018 → 2023 | G01"
].copy()

print(
    "Componentes pendientes:",
    len(revision_componentes)
)

display(
    revision_componentes[
        [
            "Puente",
            "Grupo",
            "Sectores_origen",
            "Sectores_destino",
            "N_origenes",
            "N_destinos",
            "Max_establecimientos",
            "Max_porcentaje",
            "Promedio_porcentaje"
        ]
    ]
    .sort_values(
        "Max_porcentaje",
        ascending=False
    )
    .reset_index(drop=True)
)

Componentes pendientes: 14


,Puente,Grupo,Sectores_origen,Sectores_destino,N_origenes,N_destinos,Max_establecimientos,Max_porcentaje,Promedio_porcentaje
0,2007 → 2013,G06,72,72,2,6,3977.0,10.470579,7.597193
1,2007 → 2013,G07,72,72,1,2,1977.0,5.698869,4.464446
2,2007 → 2013,G04,62,62,2,3,1032.0,2.712373,2.672423
3,2018 → 2023,G03,81,81,2,2,452.0,1.052268,1.025690
4,2013 → 2018,G04,33 | 81,33 | 81,2,3,380.0,0.942219,0.921798
5,2007 → 2013,G01,46,46,2,2,146.0,0.378867,0.325319
6,2013 → 2018,G02,48,48,1,2,55.0,0.133204,0.113331
7,2007 → 2013,G02,52,52,1,2,45.0,0.116774,0.068587
8,2018 → 2023,G02,51,51,17,15,45.0,0.116195,0.088387
9,2013 → 2018,G01,46,46,1,2,31.0,0.080444,0.066084


In [111]:
# ============================================================
# PASO 11B
# GUARDAR RESULTADOS
# ============================================================

verificacion_g01.to_csv(
    HOMOLOGACION /
    "VERIFICACION_G01_COMERCIO_SCIAN_2018_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

revision_componentes.to_csv(
    HOMOLOGACION /
    "COMPONENTES_COMPLEJOS_PENDIENTES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ G01 documentado como resuelto a nivel sector 46.")
print(
    "✓ Componentes complejos pendientes:",
    len(revision_componentes)
)

✓ G01 documentado como resuelto a nivel sector 46.
✓ Componentes complejos pendientes: 14


## Revisión específica de los componentes de mayor impacto

Los componentes complejos se revisan de manera individual comenzando por
aquellos con mayor peso empírico en Morelia.

Los grupos G06 y G07 del puente SCIAN 2007–2013 pertenecen al sector 72 y
concentran la mayor proporción de establecimientos entre los componentes
complejos pendientes.

Antes de definir una regla de homologación se examinan sus códigos, nombres
de clase, correspondencias oficiales y evolución anual de establecimientos.

In [112]:
# ============================================================
# FASE 2 - PASO 11C
# ABRIR G06 Y G07 DEL PUENTE SCIAN 2007 → 2013
# ============================================================

grupos_72 = [
    "2007 → 2013 | G06",
    "2007 → 2013 | G07"
]

detalle_grupos_72 = (
    membresia_componentes[
        membresia_componentes["Grupo_ID"].isin(grupos_72)
    ]
    [
        [
            "Grupo_ID",
            "Version_SCIAN",
            "Lado",
            "SCIAN_original"
        ]
    ]
    .sort_values(
        [
            "Grupo_ID",
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
    .reset_index(drop=True)
)

display(detalle_grupos_72)

,Grupo_ID,Version_SCIAN,Lado,SCIAN_original
0,2007 → 2013 | G06,SCIAN 2007,Origen,722110
1,2007 → 2013 | G06,SCIAN 2007,Origen,722219
2,2007 → 2013 | G06,SCIAN 2013,Destino,722511
3,2007 → 2013 | G06,SCIAN 2013,Destino,722512
4,2007 → 2013 | G06,SCIAN 2013,Destino,722513
5,2007 → 2013 | G06,SCIAN 2013,Destino,722514
6,2007 → 2013 | G06,SCIAN 2013,Destino,722515
7,2007 → 2013 | G06,SCIAN 2013,Destino,722519
8,2007 → 2013 | G07,SCIAN 2007,Origen,722212
9,2007 → 2013 | G07,SCIAN 2013,Destino,722517


In [113]:
# ============================================================
# PASO 11D
# RECUPERAR NOMBRES OFICIALES DE LAS CLASES G06 Y G07
# ============================================================

corr_nombres_2007_2013 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA.csv",
    dtype=str
)

codigos_g06_g07_2007 = set(
    detalle_grupos_72.loc[
        detalle_grupos_72["Version_SCIAN"] == "SCIAN 2007",
        "SCIAN_original"
    ]
)

codigos_g06_g07_2013 = set(
    detalle_grupos_72.loc[
        detalle_grupos_72["Version_SCIAN"] == "SCIAN 2013",
        "SCIAN_original"
    ]
)

detalle_oficial_72 = corr_nombres_2007_2013[
    corr_nombres_2007_2013["SCIAN_2007"]
    .astype(str)
    .isin(codigos_g06_g07_2007)
    |
    corr_nombres_2007_2013["SCIAN_2013"]
    .astype(str)
    .isin(codigos_g06_g07_2013)
].copy()

display(
    detalle_oficial_72[
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ]
    .sort_values(
        ["SCIAN_2007", "SCIAN_2013"]
    )
    .reset_index(drop=True)
)

,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
0,722110,con servicio completo de preparación de alimen...,NaN,722511,de alimentos a la carta o de comida corrida
1,722110,Restaurantes con servicio completo (restaurant...,p 2/5,722512,Restaurantes con servicio de preparación de pe...
2,722110,Restaurantes con servicio completo (restaurant...,p 3/5,722513,Restaurantes con servicio de preparación de an...
3,722110,Restaurantes con servicio completo (restaurant...,p 4/5,722514,Restaurantes con servicio de preparación de ta...
4,722110,Restaurantes con servicio completo (cafeterías...,p 5/5,722515,"Cafeterías, fuentes de sodas, neverías, refres..."
5,722212,Restaurantes de comida para llevar (restaurant...,p 1/2,722517,Restaurantes con servicio de preparación de pi...
6,722212,Restaurantes de comida para llevar (restaurant...,p 2/2,722518,Restaurantes que preparan otro tipo de aliment...
7,722219,Otros restaurantes con servicio limitado (rest...,p 1/6,722511,de alimentos a la carta o de comida corrida
8,722219,Otros restaurantes con servicio limitado (rest...,p 2/6,722512,Restaurantes con servicio de preparación de pe...
9,722219,Otros restaurantes con servicio limitado (rest...,p 3/6,722513,Restaurantes con servicio de preparación de an...


In [114]:
# ============================================================
# PASO 11E
# EVOLUCIÓN ANUAL DE LOS CÓDIGOS DE G06 Y G07
# ============================================================

codigos_72_todos = set(
    detalle_grupos_72["SCIAN_original"]
    .astype(str)
    .str.strip()
)

evolucion_72 = (
    inventario_scian[
        inventario_scian["SCIAN_original"]
        .astype(str)
        .str.strip()
        .isin(codigos_72_todos)
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "SCIAN_original",
            "Establecimientos"
        ]
    ]
    .sort_values(
        [
            "Año",
            "SCIAN_original"
        ]
    )
    .reset_index(drop=True)
)

display(evolucion_72)

,Año,Version_SCIAN,SCIAN_original,Establecimientos
0,2010,SCIAN 2007,722110,217
1,2010,SCIAN 2007,722212,1975
2,2010,SCIAN 2007,722219,1707
3,2011,SCIAN 2007,722110,222
4,2011,SCIAN 2007,722212,1975
...,...,...,...,...
103,2026,SCIAN 2023,722514,1505
104,2026,SCIAN 2023,722515,845
105,2026,SCIAN 2023,722517,762
106,2026,SCIAN 2023,722518,545


In [115]:
# ============================================================
# PASO 11F
# TABLA AÑO × CÓDIGO
# ============================================================

tabla_72 = (
    evolucion_72.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_72)

SCIAN_original,Año,Version_SCIAN,722110,722212,722219,722511,722512,722513,722514,722515,722517,722518,722519
0,2010,SCIAN 2007,217,1975,1707,0,0,0,0,0,0,0,0
1,2011,SCIAN 2007,222,1975,1708,0,0,0,0,0,0,0,0
2,2012,SCIAN 2007,225,1977,1718,0,0,0,0,0,0,0,0
3,2013,SCIAN 2007,233,1971,1729,0,0,0,0,0,0,0,0
4,2015,SCIAN 2013,0,0,0,485,165,968,1287,594,616,504,446
5,2016,SCIAN 2013,0,0,0,485,166,958,1298,592,610,512,473
6,2017,SCIAN 2013,0,0,0,489,166,958,1297,594,613,512,473
7,2018,SCIAN 2018,0,0,0,494,166,958,1297,589,616,512,473
8,2019,SCIAN 2018,0,0,0,477,170,1064,1409,752,616,713,468
9,2020,SCIAN 2018,0,0,0,536,176,1076,1394,725,686,730,536


In [116]:
# ============================================================
# FASE 2 - PASO 11G
# AUDITAR RELACIÓN 722110 -> 722511 Y SU CAMPO "Parte"
# ============================================================

with pdfplumber.open(pdf_2007_2013) as pdf:

    for numero_pagina, pagina in enumerate(pdf.pages, start=1):

        texto = pagina.extract_text() or ""

        if "722110" in texto and "722511" in texto:

            print("\n====================================")
            print("PÁGINA:", numero_pagina)
            print("====================================")

            lineas = texto.splitlines()

            for i, linea in enumerate(lineas):

                if "722110" in linea or "722511" in linea:

                    desde = max(0, i - 2)
                    hasta = min(len(lineas), i + 3)

                    for j in range(desde, hasta):
                        print(f"{j:03d}: {lineas[j]}")

                    print("--------------------------------")


PÁGINA: 10
014: del sector público)
015: 72 SERVICIOS DE ALOJAMIENTO TEMPORAL Y DE PREPARACIÓN DE ALIMENTOS Y BEBIDAS
016: 722511 Restaurantes con servicio de preparación p 1/5 722110 Restaurantes con servicio completo (restaurantes
017: de alimentos a la carta o de comida corrida con servicio completo de preparación de alimentos
018: a la carta o de comida corrida)
--------------------------------
020: con servicio limitado de preparación de alimentos a la carta
021: o de comida corrida)
022: 722512 Restaurantes con servicio de preparación p 2/5 722110 Restaurantes con servicio completo (restaurantes
023: de pescados y mariscos con servicio completo de preparación de pescados y mariscos)
024: p 2/6 722219 Otros restaurantes con servicio limitado (restaurantes
--------------------------------
024: p 2/6 722219 Otros restaurantes con servicio limitado (restaurantes
025: con servicio limitado de preparación de pescados y mariscos)
026: 722513 Restaurantes con servicio de preparación p 3

In [117]:
# ============================================================
# FASE 2 - PASO 11H
# CONSTRUIR SERIES HOMOLOGADAS G06 Y G07
# ============================================================

def total_grupo_por_version(
    df,
    codigos_2007,
    codigos_2013_mas
):

    temp = df.copy()

    def pertenece(row):

        codigo = str(row["SCIAN_original"]).strip()
        version = str(row["Version_SCIAN"]).strip()

        if version == "SCIAN 2007":
            return codigo in codigos_2007

        if version in [
            "SCIAN 2013",
            "SCIAN 2018",
            "SCIAN 2023"
        ]:
            return codigo in codigos_2013_mas

        return False

    temp = temp[
        temp.apply(pertenece, axis=1)
    ].copy()

    return (
        temp.groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )


# ------------------------------------------------------------
# G06
# 2007: 722110 + 722219
# 2013 en adelante:
# 722511, 722512, 722513, 722514, 722515, 722519
# ------------------------------------------------------------

serie_G06 = total_grupo_por_version(
    inventario_scian,
    {"722110", "722219"},
    {
        "722511",
        "722512",
        "722513",
        "722514",
        "722515",
        "722519"
    }
)

serie_G06["Grupo"] = "G06"


# ------------------------------------------------------------
# G07
# 2007: 722212
# 2013 en adelante: 722517 + 722518
# ------------------------------------------------------------

serie_G07 = total_grupo_por_version(
    inventario_scian,
    {"722212"},
    {
        "722517",
        "722518"
    }
)

serie_G07["Grupo"] = "G07"


series_homologadas_72 = pd.concat(
    [
        serie_G06,
        serie_G07
    ],
    ignore_index=True
)

display(
    series_homologadas_72[
        [
            "Grupo",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values(
        ["Grupo", "Año"]
    )
)

,Grupo,Año,Version_SCIAN,Establecimientos
0,G06,2010,SCIAN 2007,1924
1,G06,2011,SCIAN 2007,1930
2,G06,2012,SCIAN 2007,1943
3,G06,2013,SCIAN 2007,1962
4,G06,2015,SCIAN 2013,3945
5,G06,2016,SCIAN 2013,3972
6,G06,2017,SCIAN 2013,3977
7,G06,2018,SCIAN 2018,3977
8,G06,2019,SCIAN 2018,4340
9,G06,2020,SCIAN 2018,4443


In [118]:
# ============================================================
# FASE 2 - PASO 11I
# CORREGIR METADATO "Parte" VALIDADO EN PDF OFICIAL
# 722110 -> 722511 = p 1/5
# ============================================================

corr_2007_2013_validada = (
    corr_nombres_2007_2013.copy()
)

mask = (
    corr_2007_2013_validada["SCIAN_2007"]
    .astype(str).str.strip().eq("722110")
    &
    corr_2007_2013_validada["SCIAN_2013"]
    .astype(str).str.strip().eq("722511")
)

print("Filas encontradas:", mask.sum())

print("\nANTES:")
display(
    corr_2007_2013_validada.loc[
        mask,
        [
            "SCIAN_2007",
            "Parte",
            "SCIAN_2013"
        ]
    ]
)

corr_2007_2013_validada.loc[
    mask,
    "Parte"
] = "p 1/5"

print("\nDESPUÉS:")
display(
    corr_2007_2013_validada.loc[
        mask,
        [
            "SCIAN_2007",
            "Parte",
            "SCIAN_2013"
        ]
    ]
)

corr_2007_2013_validada.to_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA_VALIDADA.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Corrección validada y guardada.")

Filas encontradas: 1

ANTES:


,SCIAN_2007,Parte,SCIAN_2013
59,722110,NaN,722511



DESPUÉS:


,SCIAN_2007,Parte,SCIAN_2013
59,722110,p 1/5,722511



✓ Corrección validada y guardada.


In [119]:
# ============================================================
# FASE 2 - PASO 11J
# PROPAGAR G06 Y G07 A TRAVÉS DE SCIAN 2018 Y 2023
# ============================================================

G06_2013 = {
    "722511",
    "722512",
    "722513",
    "722514",
    "722515",
    "722519"
}

G07_2013 = {
    "722517",
    "722518"
}


def propagar_codigos(
    codigos_origen,
    mapa,
    col_origen,
    col_destino
):

    resultado = set()
    detalle = []

    for codigo in sorted(codigos_origen):

        filas = mapa[
            mapa[col_origen]
            .astype(str)
            .str.strip()
            .eq(codigo)
        ]

        if filas.empty:

            # No aparece en tabla oficial de cambios:
            # se conserva el mismo código
            resultado.add(codigo)

            detalle.append({
                "Codigo_origen": codigo,
                "Cambio_explicito": False,
                "Codigo_destino": codigo
            })

        else:

            destinos = sorted(
                filas[col_destino]
                .astype(str)
                .str.strip()
                .unique()
            )

            resultado.update(destinos)

            for destino in destinos:

                detalle.append({
                    "Codigo_origen": codigo,
                    "Cambio_explicito": True,
                    "Codigo_destino": destino
                })

    return resultado, pd.DataFrame(detalle)


# ------------------------------------------------------------
# G06: 2013 -> 2018
# ------------------------------------------------------------

G06_2018, detalle_G06_13_18 = propagar_codigos(
    G06_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

# G06: 2018 -> 2023
G06_2023, detalle_G06_18_23 = propagar_codigos(
    G06_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)


# ------------------------------------------------------------
# G07: 2013 -> 2018
# ------------------------------------------------------------

G07_2018, detalle_G07_13_18 = propagar_codigos(
    G07_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

# G07: 2018 -> 2023
G07_2023, detalle_G07_18_23 = propagar_codigos(
    G07_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)


print("========== G06 ==========")
print("SCIAN 2013:", sorted(G06_2013))
print("SCIAN 2018:", sorted(G06_2018))
print("SCIAN 2023:", sorted(G06_2023))

print("\nCambios explícitos 2013→2018:")
display(
    detalle_G06_13_18[
        detalle_G06_13_18["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2018→2023:")
display(
    detalle_G06_18_23[
        detalle_G06_18_23["Cambio_explicito"]
    ]
)


print("\n========== G07 ==========")
print("SCIAN 2013:", sorted(G07_2013))
print("SCIAN 2018:", sorted(G07_2018))
print("SCIAN 2023:", sorted(G07_2023))

print("\nCambios explícitos 2013→2018:")
display(
    detalle_G07_13_18[
        detalle_G07_13_18["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2018→2023:")
display(
    detalle_G07_18_23[
        detalle_G07_18_23["Cambio_explicito"]
    ]
)

========== G06 ==========
SCIAN 2013: ['722511', '722512', '722513', '722514', '722515', '722519']
SCIAN 2018: ['722511', '722512', '722513', '722514', '722515', '722519']
SCIAN 2023: ['722511', '722512', '722513', '722514', '722515', '722519']

Cambios explícitos 2013→2018:


,Codigo_origen,Cambio_explicito,Codigo_destino



Cambios explícitos 2018→2023:


,Codigo_origen,Cambio_explicito,Codigo_destino



========== G07 ==========
SCIAN 2013: ['722517', '722518']
SCIAN 2018: ['722517', '722518']
SCIAN 2023: ['722517', '722518']

Cambios explícitos 2013→2018:


,Codigo_origen,Cambio_explicito,Codigo_destino



Cambios explícitos 2018→2023:


,Codigo_origen,Cambio_explicito,Codigo_destino


### Resolución de los componentes G06 y G07 del sector 72

Los componentes G06 y G07 corresponden a reestructuraciones de clases del
sector 72 ocurridas entre SCIAN México 2007 y SCIAN México 2013.

El componente G06 vincula las clases 722110 y 722219 de SCIAN 2007 con las
clases 722511, 722512, 722513, 722514, 722515 y 722519 de SCIAN 2013.

El componente G07 vincula la clase 722212 de SCIAN 2007 con las clases
722517 y 722518 de SCIAN 2013.

La propagación de ambos componentes a través de los puentes SCIAN
2013–2018 y SCIAN 2018–2023 confirmó que las clases resultantes no presentan
cambios explícitos adicionales. Por tanto, los conjuntos definidos en SCIAN
2013 permanecen sin modificación en SCIAN 2018 y SCIAN 2023.

En consecuencia, ambos componentes pueden tratarse como grupos históricos
homologados. La homologación se realiza mediante agregación de las clases
integrantes de cada grupo, sin distribuir retrospectivamente establecimientos
entre las clases resultantes.

Esta decisión permite conservar un nivel de detalle superior al sector 72
completo y mantener la trazabilidad de los establecimientos a lo largo de la
serie histórica.

In [120]:
# ============================================================
# FASE 2 - PASO 11K
# SERIES HISTÓRICAS HOMOLOGADAS DEFINITIVAS: G06 Y G07
# ============================================================

grupos_homologados_72 = {
    
    "G06": {
        "SCIAN 2007": {
            "722110",
            "722219"
        },
        "SCIAN 2013": {
            "722511",
            "722512",
            "722513",
            "722514",
            "722515",
            "722519"
        },
        "SCIAN 2018": {
            "722511",
            "722512",
            "722513",
            "722514",
            "722515",
            "722519"
        },
        "SCIAN 2023": {
            "722511",
            "722512",
            "722513",
            "722514",
            "722515",
            "722519"
        }
    },

    "G07": {
        "SCIAN 2007": {
            "722212"
        },
        "SCIAN 2013": {
            "722517",
            "722518"
        },
        "SCIAN 2018": {
            "722517",
            "722518"
        },
        "SCIAN 2023": {
            "722517",
            "722518"
        }
    }
}


filas = []

for grupo, versiones in grupos_homologados_72.items():

    for version, codigos in versiones.items():

        temp = inventario_scian[
            (
                inventario_scian["Version_SCIAN"] == version
            )
            &
            (
                inventario_scian["SCIAN_original"]
                .astype(str)
                .isin(codigos)
            )
        ].copy()

        resumen = (
            temp.groupby(
                ["Año", "Version_SCIAN"],
                as_index=False
            )["Establecimientos"]
            .sum()
        )

        resumen["Grupo_homologado"] = grupo

        filas.append(resumen)


series_homologadas_72_final = pd.concat(
    filas,
    ignore_index=True
)

series_homologadas_72_final = (
    series_homologadas_72_final[
        [
            "Grupo_homologado",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values(
        [
            "Grupo_homologado",
            "Año"
        ]
    )
    .reset_index(drop=True)
)

display(series_homologadas_72_final)

,Grupo_homologado,Año,Version_SCIAN,Establecimientos
0,G06,2010,SCIAN 2007,1924
1,G06,2011,SCIAN 2007,1930
2,G06,2012,SCIAN 2007,1943
3,G06,2013,SCIAN 2007,1962
4,G06,2015,SCIAN 2013,3945
5,G06,2016,SCIAN 2013,3972
6,G06,2017,SCIAN 2013,3977
7,G06,2018,SCIAN 2018,3977
8,G06,2019,SCIAN 2018,4340
9,G06,2020,SCIAN 2018,4443


In [121]:
# ============================================================
# PASO 11L
# CONTROL DE SERIES HOMOLOGADAS
# ============================================================

print("Años G06:")
print(
    series_homologadas_72_final.loc[
        series_homologadas_72_final["Grupo_homologado"] == "G06",
        "Año"
    ].tolist()
)

print("\nAños G07:")
print(
    series_homologadas_72_final.loc[
        series_homologadas_72_final["Grupo_homologado"] == "G07",
        "Año"
    ].tolist()
)

print("\nNulos:")
print(
    series_homologadas_72_final.isna().sum()
)

print("\nDuplicados grupo-año:")
print(
    series_homologadas_72_final.duplicated(
        subset=[
            "Grupo_homologado",
            "Año"
        ]
    ).sum()
)

Años G06:
[2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Años G07:
[2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Nulos:
Grupo_homologado    0
Año                 0
Version_SCIAN       0
Establecimientos    0
dtype: int64

Duplicados grupo-año:
0


In [122]:
# ============================================================
# PASO 11M
# GUARDAR G06 Y G07 RESUELTOS
# ============================================================

series_homologadas_72_final.to_csv(
    HOMOLOGACION /
    "SERIES_HOMOLOGADAS_G06_G07_SECTOR72.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ G06 y G07 homologados y guardados.")

✓ G06 y G07 homologados y guardados.


### Revisión del componente G04 del sector 62

El componente G04 del puente SCIAN 2007–2013 pertenece al sector 62,
Servicios de salud y de asistencia social.

Debido a que presenta una correspondencia compleja entre clases de origen
y destino, se revisan sus códigos, definiciones y presencia empírica antes
de establecer una regla de homologación.

Posteriormente se verifica si las clases resultantes vuelven a presentar
modificaciones en SCIAN 2018 o SCIAN 2023.

In [123]:
# ============================================================
# FASE 2 - PASO 11N
# ABRIR COMPONENTE G04 DEL PUENTE 2007 → 2013
# ============================================================

grupo_revision = "2007 → 2013 | G04"

detalle_G04_62 = (
    membresia_componentes[
        membresia_componentes["Grupo_ID"] == grupo_revision
    ]
    [
        [
            "Grupo_ID",
            "Version_SCIAN",
            "Lado",
            "SCIAN_original"
        ]
    ]
    .sort_values(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
    .reset_index(drop=True)
)

display(detalle_G04_62)

,Grupo_ID,Version_SCIAN,Lado,SCIAN_original
0,2007 → 2013 | G04,SCIAN 2007,Origen,621111
1,2007 → 2013 | G04,SCIAN 2007,Origen,621113
2,2007 → 2013 | G04,SCIAN 2013,Destino,621111
3,2007 → 2013 | G04,SCIAN 2013,Destino,621113
4,2007 → 2013 | G04,SCIAN 2013,Destino,621115


In [124]:
# ============================================================
# PASO 11O
# NOMBRES OFICIALES Y CAMPO "PARTE"
# ============================================================

corr_2007_2013_validada = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA_VALIDADA.csv",
    dtype=str
)

codigos_G04_2007 = set(
    detalle_G04_62.loc[
        detalle_G04_62["Version_SCIAN"] == "SCIAN 2007",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

codigos_G04_2013 = set(
    detalle_G04_62.loc[
        detalle_G04_62["Version_SCIAN"] == "SCIAN 2013",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

detalle_oficial_G04 = (
    corr_2007_2013_validada[
        corr_2007_2013_validada["SCIAN_2007"]
        .astype(str).str.strip()
        .isin(codigos_G04_2007)
        |
        corr_2007_2013_validada["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(codigos_G04_2013)
    ]
    [
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ]
    .sort_values(
        ["SCIAN_2007", "SCIAN_2013"]
    )
    .reset_index(drop=True)
)

display(detalle_oficial_G04)

,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
0,621111,Consultorios de medicina general del sector pr...,p 1/2,621111,Consultorios de medicina general del sector pr...
1,621111,Consultorios de medicina general del sector pr...,p 2/2,621115,Clínicas de consultorios médicos del sector pr...
2,621113,Consultorios de medicina especializada del sec...,p 1/2,621113,Consultorios de medicina especializada del sec...
3,621113,Consultorios de medicina especializada del sec...,p 2/2,621115,Clínicas de consultorios médicos del sector pr...


In [125]:
# ============================================================
# PASO 11P
# EVOLUCIÓN ANUAL DEL COMPONENTE G04
# ============================================================

codigos_G04_todos = set(
    detalle_G04_62["SCIAN_original"]
    .astype(str)
    .str.strip()
)

evolucion_G04 = (
    inventario_scian[
        inventario_scian["SCIAN_original"]
        .astype(str)
        .str.strip()
        .isin(codigos_G04_todos)
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "SCIAN_original",
            "Establecimientos"
        ]
    ]
    .sort_values(
        ["Año", "SCIAN_original"]
    )
    .reset_index(drop=True)
)

tabla_G04 = (
    evolucion_G04.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_G04)

SCIAN_original,Año,Version_SCIAN,621111,621113,621115
0,2010,SCIAN 2007,441,499,0
1,2011,SCIAN 2007,441,500,0
2,2012,SCIAN 2007,442,501,0
3,2013,SCIAN 2007,447,502,0
4,2015,SCIAN 2013,402,579,9
5,2016,SCIAN 2013,434,586,10
6,2017,SCIAN 2013,435,587,10
7,2018,SCIAN 2018,433,587,10
8,2019,SCIAN 2018,401,566,23
9,2020,SCIAN 2018,414,576,44


In [126]:
# ============================================================
# PASO 11Q
# PROPAGAR G04 HACIA SCIAN 2018 Y SCIAN 2023
# ============================================================

G04_2013 = codigos_G04_2013

G04_2018, detalle_G04_13_18 = propagar_codigos(
    G04_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

G04_2023, detalle_G04_18_23 = propagar_codigos(
    G04_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("SCIAN 2007:", sorted(codigos_G04_2007))
print("SCIAN 2013:", sorted(G04_2013))
print("SCIAN 2018:", sorted(G04_2018))
print("SCIAN 2023:", sorted(G04_2023))

print("\nCambios explícitos 2013→2018:")
display(
    detalle_G04_13_18[
        detalle_G04_13_18["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2018→2023:")
display(
    detalle_G04_18_23[
        detalle_G04_18_23["Cambio_explicito"]
    ]
)

SCIAN 2007: ['621111', '621113']
SCIAN 2013: ['621111', '621113', '621115']
SCIAN 2018: ['621111', '621113', '621115']
SCIAN 2023: ['621111', '621113', '621115']

Cambios explícitos 2013→2018:


,Codigo_origen,Cambio_explicito,Codigo_destino



Cambios explícitos 2018→2023:


,Codigo_origen,Cambio_explicito,Codigo_destino


In [127]:
display(detalle_oficial_G04)
display(tabla_G04)

,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
0,621111,Consultorios de medicina general del sector pr...,p 1/2,621111,Consultorios de medicina general del sector pr...
1,621111,Consultorios de medicina general del sector pr...,p 2/2,621115,Clínicas de consultorios médicos del sector pr...
2,621113,Consultorios de medicina especializada del sec...,p 1/2,621113,Consultorios de medicina especializada del sec...
3,621113,Consultorios de medicina especializada del sec...,p 2/2,621115,Clínicas de consultorios médicos del sector pr...


SCIAN_original,Año,Version_SCIAN,621111,621113,621115
0,2010,SCIAN 2007,441,499,0
1,2011,SCIAN 2007,441,500,0
2,2012,SCIAN 2007,442,501,0
3,2013,SCIAN 2007,447,502,0
4,2015,SCIAN 2013,402,579,9
5,2016,SCIAN 2013,434,586,10
6,2017,SCIAN 2013,435,587,10
7,2018,SCIAN 2018,433,587,10
8,2019,SCIAN 2018,401,566,23
9,2020,SCIAN 2018,414,576,44


### Resolución del componente G04 del sector 62

El componente G04 del puente SCIAN 2007–2013 corresponde a una
reestructuración de actividades de consultorios y clínicas médicas.

En SCIAN 2007 el componente estaba integrado por las clases 621111 y
621113. En SCIAN 2013, estas actividades se vinculan con las clases
621111, 621113 y 621115.

La clase 621115 recibe contenido procedente parcialmente de las dos
clases existentes en SCIAN 2007, por lo que no resulta metodológicamente
adecuado distribuir retrospectivamente sus establecimientos entre ellas.

La propagación del componente a través de los puentes SCIAN 2013–2018
y SCIAN 2018–2023 mostró que las clases 621111, 621113 y 621115 no
presentan cambios explícitos adicionales.

Por tanto, el componente se considera homologable mediante la agregación
del conjunto completo:

- SCIAN 2007: 621111 + 621113.
- SCIAN 2013: 621111 + 621113 + 621115.
- SCIAN 2018: 621111 + 621113 + 621115.
- SCIAN 2023: 621111 + 621113 + 621115.

La serie resultante conserva el contenido conjunto del componente sin
realizar repartos artificiales entre clases.

In [128]:
# ============================================================
# FASE 2 - PASO 11R
# SERIE HISTÓRICA HOMOLOGADA DEFINITIVA G04 - SECTOR 62
# ============================================================

grupo_homologado_G04 = {

    "SCIAN 2007": {
        "621111",
        "621113"
    },

    "SCIAN 2013": {
        "621111",
        "621113",
        "621115"
    },

    "SCIAN 2018": {
        "621111",
        "621113",
        "621115"
    },

    "SCIAN 2023": {
        "621111",
        "621113",
        "621115"
    }
}


filas_G04 = []

for version, codigos in grupo_homologado_G04.items():

    temp = inventario_scian[
        (
            inventario_scian["Version_SCIAN"] == version
        )
        &
        (
            inventario_scian["SCIAN_original"]
            .astype(str)
            .isin(codigos)
        )
    ].copy()

    resumen = (
        temp.groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Grupo_homologado"] = "G04_SECTOR62"

    filas_G04.append(resumen)


serie_homologada_G04 = pd.concat(
    filas_G04,
    ignore_index=True
)

serie_homologada_G04 = (
    serie_homologada_G04[
        [
            "Grupo_homologado",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_homologada_G04)

,Grupo_homologado,Año,Version_SCIAN,Establecimientos
0,G04_SECTOR62,2010,SCIAN 2007,940
1,G04_SECTOR62,2011,SCIAN 2007,941
2,G04_SECTOR62,2012,SCIAN 2007,943
3,G04_SECTOR62,2013,SCIAN 2007,949
4,G04_SECTOR62,2015,SCIAN 2013,990
5,G04_SECTOR62,2016,SCIAN 2013,1030
6,G04_SECTOR62,2017,SCIAN 2013,1032
7,G04_SECTOR62,2018,SCIAN 2018,1030
8,G04_SECTOR62,2019,SCIAN 2018,990
9,G04_SECTOR62,2020,SCIAN 2018,1034


In [129]:
# ============================================================
# PASO 11S
# CONTROL DE G04
# ============================================================

print(
    "Años:",
    serie_homologada_G04["Año"].tolist()
)

print(
    "Número de años:",
    serie_homologada_G04["Año"].nunique()
)

print(
    "Nulos:",
    serie_homologada_G04.isna().sum().sum()
)

print(
    "Duplicados año:",
    serie_homologada_G04.duplicated(
        subset=["Año"]
    ).sum()
)

Años: [2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Número de años: 16
Nulos: 0
Duplicados año: 0


In [130]:
# ============================================================
# PASO 11T
# GUARDAR G04 RESUELTO
# ============================================================

serie_homologada_G04.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_G04_SECTOR62.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ G04 sector 62 homologado y guardado.")

✓ G04 sector 62 homologado y guardado.


### Revisión del componente G03 del sector 81

El componente G03 del puente SCIAN 2018–2023 pertenece al sector 81,
Otros servicios excepto actividades gubernamentales.

Debido a que presenta una relación compleja entre clases de SCIAN 2018
y SCIAN 2023, se revisan sus correspondencias oficiales, su peso empírico
y su evolución histórica.

Dado que la modificación ocurre en el último puente de la serie, las clases
del componente también se rastrean retrospectivamente hacia SCIAN 2013
y SCIAN 2007 para determinar si es posible construir un grupo homologado
para todo el periodo 2010–2026.

In [131]:
# ============================================================
# FASE 2 - PASO 11U
# ABRIR G03 DEL PUENTE SCIAN 2018 → 2023 - SECTOR 81
# ============================================================

grupo_revision = "2018 → 2023 | G03"

detalle_G03_81 = (
    membresia_componentes[
        membresia_componentes["Grupo_ID"] == grupo_revision
    ]
    [
        [
            "Grupo_ID",
            "Version_SCIAN",
            "Lado",
            "SCIAN_original"
        ]
    ]
    .sort_values(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
    .reset_index(drop=True)
)

display(detalle_G03_81)

,Grupo_ID,Version_SCIAN,Lado,SCIAN_original
0,2018 → 2023 | G03,SCIAN 2018,Origen,811121
1,2018 → 2023 | G03,SCIAN 2018,Origen,811129
2,2018 → 2023 | G03,SCIAN 2023,Destino,811121
3,2018 → 2023 | G03,SCIAN 2023,Destino,811129


In [132]:
# ============================================================
# PASO 11V
# NOMBRES Y CORRESPONDENCIAS OFICIALES DE G03
# ============================================================

corr_2018_2023_validada = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

codigos_G03_2018 = set(
    detalle_G03_81.loc[
        detalle_G03_81["Version_SCIAN"] == "SCIAN 2018",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

codigos_G03_2023 = set(
    detalle_G03_81.loc[
        detalle_G03_81["Version_SCIAN"] == "SCIAN 2023",
        "SCIAN_original"
    ]
    .astype(str)
    .str.strip()
)

detalle_oficial_G03 = (
    corr_2018_2023_validada[
        corr_2018_2023_validada["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(codigos_G03_2018)
        |
        corr_2018_2023_validada["SCIAN_2023"]
        .astype(str).str.strip()
        .isin(codigos_G03_2023)
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
    .sort_values(
        ["SCIAN_2018", "SCIAN_2023"]
    )
    .reset_index(drop=True)
)

display(detalle_oficial_G03)

,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023
0,811121,Hojalatería y pintura de automóviles y,p 1/2,811121,Hojalatería y pintura de automóviles y camione...
1,811129,Instalación de cristales y otras reparaciones,p 2/2,811121,Hojalatería y pintura de automóviles y camione...
2,811129,Instalación de cristales y otras reparaciones ...,NaN,811129,Instalación de cristales y otras reparaciones ...


In [133]:
# ============================================================
# PASO 11W
# EVOLUCIÓN EMPÍRICA DE G03
# ============================================================

codigos_G03_actuales = (
    codigos_G03_2018
    |
    codigos_G03_2023
)

evolucion_G03 = (
    inventario_scian[
        inventario_scian["SCIAN_original"]
        .astype(str)
        .str.strip()
        .isin(codigos_G03_actuales)
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "SCIAN_original",
            "Establecimientos"
        ]
    ]
    .sort_values(
        ["Año", "SCIAN_original"]
    )
    .reset_index(drop=True)
)

tabla_G03 = (
    evolucion_G03.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_G03)

SCIAN_original,Año,Version_SCIAN,811121,811129
0,2010,SCIAN 2007,335,31
1,2011,SCIAN 2007,336,31
2,2012,SCIAN 2007,337,31
3,2013,SCIAN 2007,337,31
4,2015,SCIAN 2013,355,43
5,2016,SCIAN 2013,355,44
6,2017,SCIAN 2013,355,44
7,2018,SCIAN 2018,355,44
8,2019,SCIAN 2018,374,52
9,2020,SCIAN 2018,378,52


In [134]:
# ============================================================
# PASO 11X
# FUNCIÓN PARA RETROCEDER CÓDIGOS ENTRE VERSIONES SCIAN
# ============================================================

def retroceder_codigos(
    codigos_destino,
    mapa,
    col_origen,
    col_destino
):

    resultado = set()
    detalle = []

    for codigo in sorted(codigos_destino):

        filas = mapa[
            mapa[col_destino]
            .astype(str)
            .str.strip()
            .eq(codigo)
        ]

        if filas.empty:

            # No aparece en tabla oficial de cambios:
            # se conserva el mismo código hacia atrás
            resultado.add(codigo)

            detalle.append({
                "Codigo_destino": codigo,
                "Cambio_explicito": False,
                "Codigo_origen": codigo
            })

        else:

            origenes = sorted(
                filas[col_origen]
                .astype(str)
                .str.strip()
                .unique()
            )

            resultado.update(origenes)

            for origen in origenes:

                detalle.append({
                    "Codigo_destino": codigo,
                    "Cambio_explicito": True,
                    "Codigo_origen": origen
                })

    return resultado, pd.DataFrame(detalle)


# SCIAN 2018 -> SCIAN 2013
G03_2013, detalle_G03_18_13 = retroceder_codigos(
    codigos_G03_2018,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

# SCIAN 2013 -> SCIAN 2007
G03_2007, detalle_G03_13_07 = retroceder_codigos(
    G03_2013,
    mapa_2007_2013_clasificado,
    "SCIAN_2007",
    "SCIAN_2013"
)


print("========== G03 SECTOR 81 ==========")

print("SCIAN 2007:", sorted(G03_2007))
print("SCIAN 2013:", sorted(G03_2013))
print("SCIAN 2018:", sorted(codigos_G03_2018))
print("SCIAN 2023:", sorted(codigos_G03_2023))

print("\nCambios explícitos al retroceder 2018→2013:")
display(
    detalle_G03_18_13[
        detalle_G03_18_13["Cambio_explicito"]
    ]
)

print("\nCambios explícitos al retroceder 2013→2007:")
display(
    detalle_G03_13_07[
        detalle_G03_13_07["Cambio_explicito"]
    ]
)

========== G03 SECTOR 81 ==========
SCIAN 2007: ['811121', '811129']
SCIAN 2013: ['811121', '811129']
SCIAN 2018: ['811121', '811129']
SCIAN 2023: ['811121', '811129']

Cambios explícitos al retroceder 2018→2013:


,Codigo_destino,Cambio_explicito,Codigo_origen
0,811121,True,811121



Cambios explícitos al retroceder 2013→2007:


,Codigo_destino,Cambio_explicito,Codigo_origen


## Componentes longitudinales de homologación SCIAN

Las correspondencias entre versiones consecutivas del SCIAN pueden
interactuar entre sí a lo largo del tiempo. Una clase involucrada en una
reestructuración entre dos versiones puede participar nuevamente en otra
modificación posterior.

Por ello, además del análisis de cada puente por separado, se construye un
grafo longitudinal que integra simultáneamente las correspondencias oficiales
SCIAN 2007–2013, SCIAN 2013–2018 y SCIAN 2018–2023.

Cada nodo representa una clase identificada por su versión del SCIAN y cada
arista representa una correspondencia oficial entre versiones consecutivas.

Los componentes conectados resultantes permiten identificar el conjunto
completo de clases que deben considerarse conjuntamente para mantener la
comparabilidad histórica y evitar homologaciones parciales.

In [135]:
# ============================================================
# FASE 2 - PASO 12A
# CONSTRUIR GRAFO LONGITUDINAL SCIAN 2007–2023
# ============================================================

from collections import defaultdict

grafo_longitudinal = defaultdict(set)


def agregar_puente(
    mapa,
    version_origen,
    version_destino,
    col_origen,
    col_destino
):

    for _, fila in mapa.iterrows():

        codigo_o = str(fila[col_origen]).strip()
        codigo_d = str(fila[col_destino]).strip()

        nodo_o = f"{version_origen}::{codigo_o}"
        nodo_d = f"{version_destino}::{codigo_d}"

        grafo_longitudinal[nodo_o].add(nodo_d)
        grafo_longitudinal[nodo_d].add(nodo_o)


# 2007 ↔ 2013
agregar_puente(
    mapa_2007_2013_clasificado,
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN_2007",
    "SCIAN_2013"
)

# 2013 ↔ 2018
agregar_puente(
    mapa_2013_2018_clasificado,
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN_2013",
    "SCIAN_2018"
)

# 2018 ↔ 2023
agregar_puente(
    mapa_2018_2023_clasificado,
    "SCIAN 2018",
    "SCIAN 2023",
    "SCIAN_2018",
    "SCIAN_2023"
)

print(
    "Nodos en grafo longitudinal:",
    len(grafo_longitudinal)
)

Nodos en grafo longitudinal: 508


In [136]:
# ============================================================
# PASO 12B
# COMPONENTE LONGITUDINAL DEL ACTUAL G03 - SECTOR 81
# ============================================================

semillas = {
    "SCIAN 2018::811121",
    "SCIAN 2018::811129"
}

visitados = set()
pila = list(semillas)

while pila:

    nodo = pila.pop()

    if nodo in visitados:
        continue

    visitados.add(nodo)

    for vecino in grafo_longitudinal[nodo]:

        if vecino not in visitados:
            pila.append(vecino)


componentes_por_version = {}

for nodo in sorted(visitados):

    version, codigo = nodo.split("::")

    componentes_por_version.setdefault(
        version,
        []
    ).append(codigo)


print("=== COMPONENTE LONGITUDINAL DE G03 ===")

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    codigos = sorted(
        componentes_por_version.get(
            version,
            []
        )
    )

    print(
        version,
        f"({len(codigos)} clases):",
        codigos
    )

print(
    "\nTotal de nodos del componente:",
    len(visitados)
)

=== COMPONENTE LONGITUDINAL DE G03 ===
SCIAN 2007 (0 clases): []
SCIAN 2013 (2 clases): ['336210', '811121']
SCIAN 2018 (4 clases): ['336210', '811121', '811123', '811129']
SCIAN 2023 (2 clases): ['811121', '811129']

Total de nodos del componente: 8


In [137]:
# ============================================================
# PASO 12C
# VER QUÉ GRUPOS COMPLEJOS YA IDENTIFICADOS TOCAN ESTE
# COMPONENTE LONGITUDINAL
# ============================================================

filas_solapadas = []

for _, fila in membresia_componentes.iterrows():

    nodo = (
        str(fila["Version_SCIAN"]).strip()
        + "::"
        + str(fila["SCIAN_original"]).strip()
    )

    if nodo in visitados:

        filas_solapadas.append({
            "Grupo_ID": fila["Grupo_ID"],
            "Puente": fila["Puente"],
            "Grupo": fila["Grupo"],
            "Version_SCIAN": fila["Version_SCIAN"],
            "SCIAN_original": fila["SCIAN_original"],
            "Lado": fila["Lado"]
        })


solapamientos_G03 = pd.DataFrame(
    filas_solapadas
)

print("Grupos complejos conectados:")

display(
    solapamientos_G03[
        [
            "Grupo_ID",
            "Puente",
            "Grupo"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["Puente", "Grupo"]
    )
    .reset_index(drop=True)
)

print("\nCódigos involucrados:")

display(
    solapamientos_G03.sort_values(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
    .reset_index(drop=True)
)

Grupos complejos conectados:


,Grupo_ID,Puente,Grupo
0,2013 → 2018 | G04,2013 → 2018,G04
1,2018 → 2023 | G03,2018 → 2023,G03



Códigos involucrados:


,Grupo_ID,Puente,Grupo,Version_SCIAN,SCIAN_original,Lado
0,2013 → 2018 | G04,2013 → 2018,G04,SCIAN 2013,336210,Origen
1,2013 → 2018 | G04,2013 → 2018,G04,SCIAN 2013,811121,Origen
2,2013 → 2018 | G04,2013 → 2018,G04,SCIAN 2018,336210,Destino
3,2013 → 2018 | G04,2013 → 2018,G04,SCIAN 2018,811121,Destino
4,2018 → 2023 | G03,2018 → 2023,G03,SCIAN 2018,811121,Origen
5,2013 → 2018 | G04,2013 → 2018,G04,SCIAN 2018,811123,Destino
6,2018 → 2023 | G03,2018 → 2023,G03,SCIAN 2018,811129,Origen
7,2018 → 2023 | G03,2018 → 2023,G03,SCIAN 2023,811121,Destino
8,2018 → 2023 | G03,2018 → 2023,G03,SCIAN 2023,811129,Destino


In [138]:
# ============================================================
# FASE 2 - PASO 12D
# AUDITAR LA FAMILIA LONGITUDINAL 33–81
# ============================================================

# ------------------------------------------------------------
# A) Correspondencias oficiales 2013 -> 2018
# ------------------------------------------------------------

corr_13_18_nombres = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2013_2018.csv",
    dtype=str
)

codigos_familia_2013 = {
    "336210",
    "811121"
}

codigos_familia_2018 = {
    "336210",
    "811121",
    "811123",
    "811129"
}

detalle_familia_13_18 = (
    corr_13_18_nombres[
        corr_13_18_nombres["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(codigos_familia_2013)
        |
        corr_13_18_nombres["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(codigos_familia_2018)
    ]
    [
        [
            "SCIAN_2013",
            "Nombre_2013",
            "Parte",
            "SCIAN_2018",
            "Nombre_2018"
        ]
    ]
    .sort_values(
        ["SCIAN_2013", "SCIAN_2018"]
    )
    .reset_index(drop=True)
)

print("=== SCIAN 2013 → 2018 ===")
display(detalle_familia_13_18)


# ------------------------------------------------------------
# B) Correspondencias oficiales 2018 -> 2023
# ------------------------------------------------------------

corr_18_23_nombres = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

codigos_familia_2023 = {
    "811121",
    "811129"
}

detalle_familia_18_23 = (
    corr_18_23_nombres[
        corr_18_23_nombres["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(codigos_familia_2018)
        |
        corr_18_23_nombres["SCIAN_2023"]
        .astype(str).str.strip()
        .isin(codigos_familia_2023)
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
    .sort_values(
        ["SCIAN_2018", "SCIAN_2023"]
    )
    .reset_index(drop=True)
)

print("\n=== SCIAN 2018 → 2023 ===")
display(detalle_familia_18_23)

=== SCIAN 2013 → 2018 ===


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018
0,336210,Fabricación de carrocerías y remolques,NaN,336210,Fabricación de carrocerías y remolques
1,336210,Fabricación de carrocerías y remolques,p 1/2,811123,Servicios de blindaje y de adaptación de autom...
2,811121,Hojalatería y pintura de automóviles y,NaN,811121,Hojalatería y pintura de automóviles y camiones
3,811121,Hojalatería y pintura de automóviles y camiones,p 2/2,811123,Servicios de blindaje y de adaptación de autom...



=== SCIAN 2018 → 2023 ===


,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023
0,811121,Hojalatería y pintura de automóviles y,p 1/2,811121,Hojalatería y pintura de automóviles y camione...
1,811129,Instalación de cristales y otras reparaciones,p 2/2,811121,Hojalatería y pintura de automóviles y camione...
2,811129,Instalación de cristales y otras reparaciones ...,NaN,811129,Instalación de cristales y otras reparaciones ...


In [139]:
# ============================================================
# PASO 12E
# PRESENCIA EMPÍRICA DE LA FAMILIA EN NUESTRO UNIVERSO
# ============================================================

codigos_familia = {
    "336210",
    "811121",
    "811123",
    "811129"
}

presencia_familia = (
    inventario_scian[
        inventario_scian["SCIAN_original"]
        .astype(str).str.strip()
        .isin(codigos_familia)
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "SCIAN_original",
            "Sector",
            "Establecimientos"
        ]
    ]
    .sort_values(
        ["Año", "SCIAN_original"]
    )
    .reset_index(drop=True)
)

display(presencia_familia)

print("\nCódigos de la familia presentes en nuestro inventario:")
print(
    sorted(
        presencia_familia["SCIAN_original"]
        .astype(str)
        .unique()
    )
)

,Año,Version_SCIAN,SCIAN_original,Sector,Establecimientos
0,2010,SCIAN 2007,811121,81,335
1,2010,SCIAN 2007,811129,81,31
2,2011,SCIAN 2007,811121,81,336
3,2011,SCIAN 2007,811129,81,31
4,2012,SCIAN 2007,811121,81,337
5,2012,SCIAN 2007,811129,81,31
6,2013,SCIAN 2007,811121,81,337
7,2013,SCIAN 2007,811129,81,31
8,2015,SCIAN 2013,811121,81,355
9,2015,SCIAN 2013,811129,81,43



Códigos de la familia presentes en nuestro inventario:
['811121', '811123', '811129']


### Tratamiento del componente longitudinal de los servicios automotrices

La integración de los puentes SCIAN 2013–2018 y SCIAN 2018–2023 mostró
que los componentes inicialmente identificados como G04 y G03 forman una
misma familia longitudinal.

Entre SCIAN 2013 y SCIAN 2018, la clase 811123 recibe contenido procedente
tanto de la clase 811121, perteneciente al sector servicios, como de la clase
336210, perteneciente al sector manufacturero. Debido a que la manufactura
no forma parte del universo definido para esta investigación, no es posible
determinar con los datos del DENUE qué proporción de los establecimientos
clasificados en 811123 procede de cada actividad de origen.

En consecuencia, no se realiza una asignación retrospectiva arbitraria.

Para cuantificar esta incertidumbre se construyen dos series:

- Límite inferior: clases 811121 y 811129.
- Límite superior: clases 811121, 811129 y, a partir de SCIAN 2018,
  la clase 811123.

La diferencia entre ambos límites representa el número máximo de
establecimientos cuya adscripción longitudinal al universo de servicios no
puede determinarse de manera inequívoca debido al cambio clasificatorio.

Este componente se clasifica como una homologación con incertidumbre
acotada por transferencia intersectorial.

In [140]:

# ============================================================
# FASE 2 - PASO 12F
# SERIE HOMOLOGADA CON INCERTIDUMBRE ACOTADA
# FAMILIA 811121 / 811129 / 811123
# ============================================================

filas = []

for año in sorted(inventario_scian["Año"].dropna().unique()):

    temp = inventario_scian[
        inventario_scian["Año"] == año
    ].copy()

    version = temp["Version_SCIAN"].iloc[0]

    # Núcleo comparable
    codigos_base = {
        "811121",
        "811129"
    }

    limite_inferior = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .isin(codigos_base)
        ]["Establecimientos"]
        .sum()
    )

    # 811123 sólo genera incertidumbre desde SCIAN 2018
    if version in [
        "SCIAN 2018",
        "SCIAN 2023"
    ]:

        ambiguo_811123 = (
            temp[
                temp["SCIAN_original"]
                .astype(str)
                .eq("811123")
            ]["Establecimientos"]
            .sum()
        )

    else:
        ambiguo_811123 = 0

    limite_superior = (
        limite_inferior
        + ambiguo_811123
    )

    filas.append({
        "Año": int(año),
        "Version_SCIAN": version,
        "Limite_inferior": limite_inferior,
        "Establecimientos_ambiguos_811123":
            ambiguo_811123,
        "Limite_superior": limite_superior,
        "Amplitud_incertidumbre":
            limite_superior - limite_inferior
    })


serie_familia_81_acotada = pd.DataFrame(filas)

display(serie_familia_81_acotada)

,Año,Version_SCIAN,Limite_inferior,Establecimientos_ambiguos_811123,Limite_superior,Amplitud_incertidumbre
0,2010,SCIAN 2007,366,0,366,0
1,2011,SCIAN 2007,367,0,367,0
2,2012,SCIAN 2007,368,0,368,0
3,2013,SCIAN 2007,368,0,368,0
4,2015,SCIAN 2013,398,0,398,0
5,2016,SCIAN 2013,399,0,399,0
6,2017,SCIAN 2013,399,0,399,0
7,2018,SCIAN 2018,399,0,399,0
8,2019,SCIAN 2018,426,1,427,1
9,2020,SCIAN 2018,430,1,431,1


In [141]:
# ============================================================
# PASO 12G
# CUANTIFICAR LA INCERTIDUMBRE
# ============================================================

totales_anuales_aux = (
    inventario_scian
    .groupby("Año", as_index=False)["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos":
            "Total_comercio_servicios"
        }
    )
)

serie_familia_81_acotada = (
    serie_familia_81_acotada.merge(
        totales_anuales_aux,
        on="Año",
        how="left"
    )
)

serie_familia_81_acotada[
    "Incertidumbre_pct_universo"
] = (
    serie_familia_81_acotada[
        "Amplitud_incertidumbre"
    ]
    /
    serie_familia_81_acotada[
        "Total_comercio_servicios"
    ]
    * 100
)

display(
    serie_familia_81_acotada[
        [
            "Año",
            "Version_SCIAN",
            "Limite_inferior",
            "Establecimientos_ambiguos_811123",
            "Limite_superior",
            "Incertidumbre_pct_universo"
        ]
    ]
)

print(
    "\nMáxima amplitud de incertidumbre:",
    serie_familia_81_acotada[
        "Amplitud_incertidumbre"
    ].max(),
    "establecimientos"
)

print(
    "Máxima incertidumbre como % del universo:",
    serie_familia_81_acotada[
        "Incertidumbre_pct_universo"
    ].max()
)

,Año,Version_SCIAN,Limite_inferior,Establecimientos_ambiguos_811123,Limite_superior,Incertidumbre_pct_universo
0,2010,SCIAN 2007,366,0,366,0.000000
1,2011,SCIAN 2007,367,0,367,0.000000
2,2012,SCIAN 2007,368,0,368,0.000000
3,2013,SCIAN 2007,368,0,368,0.000000
4,2015,SCIAN 2013,398,0,398,0.000000
5,2016,SCIAN 2013,399,0,399,0.000000
6,2017,SCIAN 2013,399,0,399,0.000000
7,2018,SCIAN 2018,399,0,399,0.000000
8,2019,SCIAN 2018,426,1,427,0.002470
9,2020,SCIAN 2018,430,1,431,0.002413



Máxima amplitud de incertidumbre: 1 establecimientos
Máxima incertidumbre como % del universo: 0.0024701116490465367


In [142]:
# ============================================================
# PASO 12H
# GUARDAR FAMILIA CON INCERTIDUMBRE ACOTADA
# ============================================================

serie_familia_81_acotada.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_ACOTADA_FAMILIA_811121_811129_811123.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "✓ Familia sector 81 guardada "
    "como homologación con incertidumbre acotada."
)

✓ Familia sector 81 guardada como homologación con incertidumbre acotada.


## Consolidación de componentes longitudinales complejos

El análisis individual mostró que componentes identificados en distintos
puentes entre versiones del SCIAN pueden pertenecer a una misma familia
longitudinal.

Por esta razón, las correspondencias complejas se consolidan mediante un
grafo único que integra los cambios SCIAN 2007–2013, 2013–2018 y
2018–2023.

El objetivo es identificar familias completas de clases relacionadas a lo
largo del periodo, evitando tratar como independientes modificaciones que
forman parte de una misma trayectoria clasificatoria.

In [143]:
# ============================================================
# FASE 2 - PASO 12I
# CONSOLIDAR TODOS LOS COMPONENTES COMPLEJOS
# EN FAMILIAS LONGITUDINALES
# ============================================================

# ------------------------------------------------------------
# 1. Nodos semilla:
# todos los códigos pertenecientes a los 15 componentes
# complejos detectados anteriormente
# ------------------------------------------------------------

semillas_complejas = set()

for _, fila in membresia_componentes.iterrows():

    nodo = (
        str(fila["Version_SCIAN"]).strip()
        + "::"
        + str(fila["SCIAN_original"]).strip()
    )

    semillas_complejas.add(nodo)


# ------------------------------------------------------------
# 2. Obtener componentes conectados longitudinales
# ------------------------------------------------------------

visitados_global = set()
familias = []

numero_familia = 1

for semilla in sorted(semillas_complejas):

    if semilla in visitados_global:
        continue

    pila = [semilla]
    componente = set()

    while pila:

        nodo = pila.pop()

        if nodo in componente:
            continue

        componente.add(nodo)
        visitados_global.add(nodo)

        for vecino in grafo_longitudinal[nodo]:

            if vecino not in componente:
                pila.append(vecino)

    # Solo conservar componentes que tocan
    # alguno de nuestros componentes complejos
    if not (componente & semillas_complejas):
        continue

    registro = {
        "Familia": f"FL{numero_familia:02d}",
        "Total_nodos": len(componente)
    }

    for version in [
        "SCIAN 2007",
        "SCIAN 2013",
        "SCIAN 2018",
        "SCIAN 2023"
    ]:

        codigos = sorted(
            nodo.split("::")[1]
            for nodo in componente
            if nodo.startswith(version + "::")
        )

        registro[
            f"N_{version.replace(' ', '_')}"
        ] = len(codigos)

        registro[
            f"Codigos_{version.replace(' ', '_')}"
        ] = " | ".join(codigos)

    # Qué grupos puente originales toca
    grupos_tocados = sorted(
        set(
            membresia_componentes.loc[
                membresia_componentes.apply(
                    lambda r:
                    (
                        str(r["Version_SCIAN"]).strip()
                        + "::"
                        + str(r["SCIAN_original"]).strip()
                    )
                    in componente,
                    axis=1
                ),
                "Grupo_ID"
            ]
        )
    )

    registro["Grupos_originales"] = " || ".join(
        grupos_tocados
    )

    registro["N_grupos_originales"] = len(
        grupos_tocados
    )

    familias.append(registro)

    numero_familia += 1


familias_longitudinales = pd.DataFrame(familias)

print(
    "Componentes complejos originales:",
    componentes_complejos.shape[0]
)

print(
    "Familias longitudinales resultantes:",
    len(familias_longitudinales)
)

display(
    familias_longitudinales[
        [
            "Familia",
            "N_grupos_originales",
            "Total_nodos",
            "N_SCIAN_2007",
            "N_SCIAN_2013",
            "N_SCIAN_2018",
            "N_SCIAN_2023",
            "Grupos_originales"
        ]
    ]
    .sort_values(
        "Total_nodos",
        ascending=False
    )
    .reset_index(drop=True)
)

Componentes complejos originales: 15
Familias longitudinales resultantes: 12


,Familia,N_grupos_originales,Total_nodos,N_SCIAN_2007,N_SCIAN_2013,N_SCIAN_2018,N_SCIAN_2023,Grupos_originales
0,FL01,3,173,2,3,84,84,2007 → 2013 | G01 || 2013 → 2018 | G01 || 2018...
1,FL12,1,32,0,0,17,15,2018 → 2023 | G02
2,FL11,1,10,0,2,7,1,2013 → 2018 | G03
3,FL06,1,8,2,6,0,0,2007 → 2013 | G06
4,FL09,2,8,0,2,4,2,2013 → 2018 | G04 || 2018 → 2023 | G03
5,FL02,1,7,1,5,1,0,2007 → 2013 | G02
6,FL05,1,5,2,3,0,0,2007 → 2013 | G05
7,FL04,1,5,2,3,0,0,2007 → 2013 | G04
8,FL03,1,3,1,2,0,0,2007 → 2013 | G03
9,FL07,1,3,1,2,0,0,2007 → 2013 | G07


In [144]:
# ============================================================
# PASO 12K
# GUARDAR FAMILIAS LONGITUDINALES
# ============================================================

familias_longitudinales.to_csv(
    HOMOLOGACION /
    "FAMILIAS_LONGITUDINALES_COMPLEJAS.csv",
    index=False,
    encoding="utf-8-sig"
)

sectores_familias_longitudinales.to_csv(
    HOMOLOGACION /
    "SECTORES_FAMILIAS_LONGITUDINALES_COMPLEJAS.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Familias longitudinales guardadas.")

NameError: name 'sectores_familias_longitudinales' is not defined

In [145]:
# ============================================================
# RECUPERACIÓN PASOS 12J + 12K
# CREAR SECTORES POR FAMILIA Y GUARDAR RESULTADOS
# ============================================================

filas = []

for _, fam in familias_longitudinales.iterrows():

    familia = fam["Familia"]

    sectores = set()
    codigos_totales = []

    for version in [
        "SCIAN 2007",
        "SCIAN 2013",
        "SCIAN 2018",
        "SCIAN 2023"
    ]:

        columna = (
            "Codigos_"
            + version.replace(" ", "_")
        )

        texto = str(fam[columna])

        codigos = [
            x.strip()
            for x in texto.split("|")
            if x.strip()
            and x.strip().lower() != "nan"
        ]

        codigos_totales.extend(codigos)

        for codigo in codigos:
            sectores.add(codigo[:2])

    filas.append({
        "Familia": familia,
        "Sectores": " | ".join(sorted(sectores)),
        "N_sectores": len(sectores),
        "Total_codigos_distintos": len(set(codigos_totales)),
        "Grupos_originales": fam["Grupos_originales"]
    })


sectores_familias_longitudinales = pd.DataFrame(filas)


print("=== SECTORES POR FAMILIA LONGITUDINAL ===")

display(
    sectores_familias_longitudinales
    .sort_values(
        [
            "N_sectores",
            "Total_codigos_distintos"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# GUARDAR
# ------------------------------------------------------------

familias_longitudinales.to_csv(
    HOMOLOGACION /
    "FAMILIAS_LONGITUDINALES_COMPLEJAS.csv",
    index=False,
    encoding="utf-8-sig"
)

sectores_familias_longitudinales.to_csv(
    HOMOLOGACION /
    "SECTORES_FAMILIAS_LONGITUDINALES_COMPLEJAS.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Familias longitudinales guardadas.")
print(
    "✓ Familias registradas:",
    len(familias_longitudinales)
)

=== SECTORES POR FAMILIA LONGITUDINAL ===


,Familia,Sectores,N_sectores,Total_codigos_distintos,Grupos_originales
0,FL09,33 | 81,2,4,2013 → 2018 | G04 || 2018 → 2023 | G03
1,FL01,46,1,84,2007 → 2013 | G01 || 2013 → 2018 | G01 || 2018...
2,FL12,51,1,32,2018 → 2023 | G02
3,FL06,72,1,8,2007 → 2013 | G06
4,FL11,56,1,8,2013 → 2018 | G03
5,FL02,52,1,5,2007 → 2013 | G02
6,FL04,62,1,3,2007 → 2013 | G04
7,FL05,62,1,3,2007 → 2013 | G05
8,FL07,72,1,3,2007 → 2013 | G07
9,FL08,81,1,3,2007 → 2013 | G08



✓ Familias longitudinales guardadas.
✓ Familias registradas: 12


In [146]:
# ============================================================
# PASO 12K
# GUARDAR FAMILIAS LONGITUDINALES
# ============================================================

familias_longitudinales.to_csv(
    HOMOLOGACION /
    "FAMILIAS_LONGITUDINALES_COMPLEJAS.csv",
    index=False,
    encoding="utf-8-sig"
)

sectores_familias_longitudinales.to_csv(
    HOMOLOGACION /
    "SECTORES_FAMILIAS_LONGITUDINALES_COMPLEJAS.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Familias longitudinales guardadas.")

✓ Familias longitudinales guardadas.


## Consolidación longitudinal con continuidad de códigos sin cambio

Las tablas comparativas oficiales identifican principalmente las clases que
presentan modificaciones entre versiones del SCIAN. Por ello, una clase que
conserva su código y no presenta una modificación explícita puede no aparecer
en dichas tablas.

Para evitar que esta característica fragmente artificialmente los componentes
longitudinales, se incorporan vínculos de continuidad entre códigos observados
con el mismo número en versiones consecutivas, siempre que dichos códigos no
formen parte de una modificación explícita que contradiga dicha continuidad.

Posteriormente se reconstruyen los componentes longitudinales antes de adoptar
una clasificación definitiva de las familias de homologación.

In [147]:
# ============================================================
# FASE 2 - PASO 12L
# GRAFO LONGITUDINAL V2
# CORRESPONDENCIAS OFICIALES + CONTINUIDAD DE CÓDIGOS
# ============================================================

from collections import defaultdict

grafo_v2 = defaultdict(set)


def conectar(grafo, nodo_a, nodo_b):

    grafo[nodo_a].add(nodo_b)
    grafo[nodo_b].add(nodo_a)


# ------------------------------------------------------------
# 1. RELACIONES EXPLÍCITAS OFICIALES
# ------------------------------------------------------------

def agregar_mapa(
    grafo,
    mapa,
    version_a,
    version_b,
    col_a,
    col_b
):

    for _, fila in mapa.iterrows():

        ca = str(fila[col_a]).strip()
        cb = str(fila[col_b]).strip()

        conectar(
            grafo,
            f"{version_a}::{ca}",
            f"{version_b}::{cb}"
        )


agregar_mapa(
    grafo_v2,
    mapa_2007_2013_clasificado,
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN_2007",
    "SCIAN_2013"
)

agregar_mapa(
    grafo_v2,
    mapa_2013_2018_clasificado,
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN_2013",
    "SCIAN_2018"
)

agregar_mapa(
    grafo_v2,
    mapa_2018_2023_clasificado,
    "SCIAN 2018",
    "SCIAN 2023",
    "SCIAN_2018",
    "SCIAN_2023"
)


# ------------------------------------------------------------
# 2. CÓDIGOS OBSERVADOS POR VERSIÓN
# ------------------------------------------------------------

def codigos_observados(version):

    return set(
        codigos_por_version.loc[
            codigos_por_version["Version_SCIAN"] == version,
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )


obs_2007 = codigos_observados("SCIAN 2007")
obs_2013 = codigos_observados("SCIAN 2013")
obs_2018 = codigos_observados("SCIAN 2018")
obs_2023 = codigos_observados("SCIAN 2023")


# ------------------------------------------------------------
# 3. AGREGAR CONTINUIDAD SOLO CUANDO NO EXISTE
#    CAMBIO EXPLÍCITO PARA ESE CÓDIGO
# ------------------------------------------------------------

def agregar_continuidad(
    grafo,
    observados_a,
    observados_b,
    mapa,
    version_a,
    version_b,
    col_a,
    col_b
):

    explicitos_a = set(
        mapa[col_a]
        .astype(str)
        .str.strip()
    )

    explicitos_b = set(
        mapa[col_b]
        .astype(str)
        .str.strip()
    )

    comunes = (
        observados_a
        &
        observados_b
    )

    agregados = 0

    for codigo in comunes:

        # Sólo continuidad pura:
        # no aparece involucrado en cambio oficial
        if (
            codigo not in explicitos_a
            and codigo not in explicitos_b
        ):

            conectar(
                grafo,
                f"{version_a}::{codigo}",
                f"{version_b}::{codigo}"
            )

            agregados += 1

    return agregados


n_07_13 = agregar_continuidad(
    grafo_v2,
    obs_2007,
    obs_2013,
    mapa_2007_2013_clasificado,
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN_2007",
    "SCIAN_2013"
)

n_13_18 = agregar_continuidad(
    grafo_v2,
    obs_2013,
    obs_2018,
    mapa_2013_2018_clasificado,
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN_2013",
    "SCIAN_2018"
)

n_18_23 = agregar_continuidad(
    grafo_v2,
    obs_2018,
    obs_2023,
    mapa_2018_2023_clasificado,
    "SCIAN 2018",
    "SCIAN 2023",
    "SCIAN_2018",
    "SCIAN_2023"
)

print("Continuidades añadidas:")
print("2007→2013:", n_07_13)
print("2013→2018:", n_13_18)
print("2018→2023:", n_18_23)

print("\nNodos del grafo V2:", len(grafo_v2))

Continuidades añadidas:
2007→2013: 352
2013→2018: 383
2018→2023: 305

Nodos del grafo V2: 1830


In [148]:
# ============================================================
# FASE 2 - PASO 12M
# RECONSTRUIR FAMILIAS LONGITUDINALES CON GRAFO V2
# ============================================================

semillas_complejas = set()

for _, fila in membresia_componentes.iterrows():

    semillas_complejas.add(
        str(fila["Version_SCIAN"]).strip()
        + "::"
        + str(fila["SCIAN_original"]).strip()
    )


visitados = set()
familias_v2 = []

numero = 1

for semilla in sorted(semillas_complejas):

    if semilla in visitados:
        continue

    pila = [semilla]
    componente = set()

    while pila:

        nodo = pila.pop()

        if nodo in componente:
            continue

        componente.add(nodo)
        visitados.add(nodo)

        for vecino in grafo_v2[nodo]:

            if vecino not in componente:
                pila.append(vecino)

    if not (componente & semillas_complejas):
        continue

    registro = {
        "Familia": f"FL{numero:02d}",
        "Total_nodos": len(componente)
    }

    for version in [
        "SCIAN 2007",
        "SCIAN 2013",
        "SCIAN 2018",
        "SCIAN 2023"
    ]:

        codigos = sorted(
            nodo.split("::")[1]
            for nodo in componente
            if nodo.startswith(version + "::")
        )

        registro[
            "N_" + version.replace(" ", "_")
        ] = len(codigos)

        registro[
            "Codigos_" + version.replace(" ", "_")
        ] = " | ".join(codigos)

    grupos_tocados = sorted(
        set(
            membresia_componentes.loc[
                membresia_componentes.apply(
                    lambda r:
                    (
                        str(r["Version_SCIAN"]).strip()
                        + "::"
                        + str(r["SCIAN_original"]).strip()
                    ) in componente,
                    axis=1
                ),
                "Grupo_ID"
            ]
        )
    )

    registro["N_grupos_originales"] = len(
        grupos_tocados
    )

    registro["Grupos_originales"] = (
        " || ".join(grupos_tocados)
    )

    familias_v2.append(registro)

    numero += 1


familias_longitudinales_v2 = pd.DataFrame(
    familias_v2
)

print(
    "Familias anteriores:",
    len(familias_longitudinales)
)

print(
    "Familias V2:",
    len(familias_longitudinales_v2)
)

display(
    familias_longitudinales_v2[
        [
            "Familia",
            "N_grupos_originales",
            "Total_nodos",
            "N_SCIAN_2007",
            "N_SCIAN_2013",
            "N_SCIAN_2018",
            "N_SCIAN_2023",
            "Grupos_originales"
        ]
    ]
    .sort_values(
        "Total_nodos",
        ascending=False
    )
    .reset_index(drop=True)
)

Familias anteriores: 12
Familias V2: 12


,Familia,N_grupos_originales,Total_nodos,N_SCIAN_2007,N_SCIAN_2013,N_SCIAN_2018,N_SCIAN_2023,Grupos_originales
0,FL01,3,330,80,82,84,84,2007 → 2013 | G01 || 2013 → 2018 | G01 || 2018...
1,FL12,1,51,9,10,17,15,2018 → 2023 | G02
2,FL06,1,20,2,6,6,6,2007 → 2013 | G06
3,FL11,1,16,2,2,7,5,2013 → 2018 | G03
4,FL09,2,12,2,3,4,3,2013 → 2018 | G04 || 2018 → 2023 | G03
5,FL02,1,11,2,5,2,2,2007 → 2013 | G02
6,FL04,1,11,2,3,3,3,2007 → 2013 | G04
7,FL05,1,9,2,3,2,2,2007 → 2013 | G05
8,FL08,1,7,1,2,2,2,2007 → 2013 | G08
9,FL07,1,7,1,2,2,2,2007 → 2013 | G07


## Consolidación longitudinal con continuidad de códigos sin cambio

Las tablas comparativas oficiales identifican principalmente las clases que
presentan modificaciones entre versiones del SCIAN. Por ello, una clase que
conserva su código y no presenta una modificación explícita puede no aparecer
en dichas tablas.

Para evitar que esta característica fragmente artificialmente los componentes
longitudinales, se incorporan vínculos de continuidad entre códigos observados
con el mismo número en versiones consecutivas, siempre que dichos códigos no
formen parte de una modificación explícita que contradiga dicha continuidad.

Posteriormente se reconstruyen los componentes longitudinales antes de adoptar
una clasificación definitiva de las familias de homologación.

In [149]:
# ============================================================
# FASE 2 - PASO 12L
# GRAFO LONGITUDINAL V2
# CORRESPONDENCIAS OFICIALES + CONTINUIDAD DE CÓDIGOS
# ============================================================

from collections import defaultdict

grafo_v2 = defaultdict(set)


def conectar(grafo, nodo_a, nodo_b):

    grafo[nodo_a].add(nodo_b)
    grafo[nodo_b].add(nodo_a)


# ------------------------------------------------------------
# 1. RELACIONES EXPLÍCITAS OFICIALES
# ------------------------------------------------------------

def agregar_mapa(
    grafo,
    mapa,
    version_a,
    version_b,
    col_a,
    col_b
):

    for _, fila in mapa.iterrows():

        ca = str(fila[col_a]).strip()
        cb = str(fila[col_b]).strip()

        conectar(
            grafo,
            f"{version_a}::{ca}",
            f"{version_b}::{cb}"
        )


agregar_mapa(
    grafo_v2,
    mapa_2007_2013_clasificado,
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN_2007",
    "SCIAN_2013"
)

agregar_mapa(
    grafo_v2,
    mapa_2013_2018_clasificado,
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN_2013",
    "SCIAN_2018"
)

agregar_mapa(
    grafo_v2,
    mapa_2018_2023_clasificado,
    "SCIAN 2018",
    "SCIAN 2023",
    "SCIAN_2018",
    "SCIAN_2023"
)


# ------------------------------------------------------------
# 2. CÓDIGOS OBSERVADOS POR VERSIÓN
# ------------------------------------------------------------

def codigos_observados(version):

    return set(
        codigos_por_version.loc[
            codigos_por_version["Version_SCIAN"] == version,
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )


obs_2007 = codigos_observados("SCIAN 2007")
obs_2013 = codigos_observados("SCIAN 2013")
obs_2018 = codigos_observados("SCIAN 2018")
obs_2023 = codigos_observados("SCIAN 2023")


# ------------------------------------------------------------
# 3. AGREGAR CONTINUIDAD SOLO CUANDO NO EXISTE
#    CAMBIO EXPLÍCITO PARA ESE CÓDIGO
# ------------------------------------------------------------

def agregar_continuidad(
    grafo,
    observados_a,
    observados_b,
    mapa,
    version_a,
    version_b,
    col_a,
    col_b
):

    explicitos_a = set(
        mapa[col_a]
        .astype(str)
        .str.strip()
    )

    explicitos_b = set(
        mapa[col_b]
        .astype(str)
        .str.strip()
    )

    comunes = (
        observados_a
        &
        observados_b
    )

    agregados = 0

    for codigo in comunes:

        # Sólo continuidad pura:
        # no aparece involucrado en cambio oficial
        if (
            codigo not in explicitos_a
            and codigo not in explicitos_b
        ):

            conectar(
                grafo,
                f"{version_a}::{codigo}",
                f"{version_b}::{codigo}"
            )

            agregados += 1

    return agregados


n_07_13 = agregar_continuidad(
    grafo_v2,
    obs_2007,
    obs_2013,
    mapa_2007_2013_clasificado,
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN_2007",
    "SCIAN_2013"
)

n_13_18 = agregar_continuidad(
    grafo_v2,
    obs_2013,
    obs_2018,
    mapa_2013_2018_clasificado,
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN_2013",
    "SCIAN_2018"
)

n_18_23 = agregar_continuidad(
    grafo_v2,
    obs_2018,
    obs_2023,
    mapa_2018_2023_clasificado,
    "SCIAN 2018",
    "SCIAN 2023",
    "SCIAN_2018",
    "SCIAN_2023"
)

print("Continuidades añadidas:")
print("2007→2013:", n_07_13)
print("2013→2018:", n_13_18)
print("2018→2023:", n_18_23)

print("\nNodos del grafo V2:", len(grafo_v2))

Continuidades añadidas:
2007→2013: 352
2013→2018: 383
2018→2023: 305

Nodos del grafo V2: 1830


In [150]:
# ============================================================
# FASE 2 - PASO 12M
# RECONSTRUIR FAMILIAS LONGITUDINALES CON GRAFO V2
# ============================================================

semillas_complejas = set()

for _, fila in membresia_componentes.iterrows():

    semillas_complejas.add(
        str(fila["Version_SCIAN"]).strip()
        + "::"
        + str(fila["SCIAN_original"]).strip()
    )


visitados = set()
familias_v2 = []

numero = 1

for semilla in sorted(semillas_complejas):

    if semilla in visitados:
        continue

    pila = [semilla]
    componente = set()

    while pila:

        nodo = pila.pop()

        if nodo in componente:
            continue

        componente.add(nodo)
        visitados.add(nodo)

        for vecino in grafo_v2[nodo]:

            if vecino not in componente:
                pila.append(vecino)

    if not (componente & semillas_complejas):
        continue

    registro = {
        "Familia": f"FL{numero:02d}",
        "Total_nodos": len(componente)
    }

    for version in [
        "SCIAN 2007",
        "SCIAN 2013",
        "SCIAN 2018",
        "SCIAN 2023"
    ]:

        codigos = sorted(
            nodo.split("::")[1]
            for nodo in componente
            if nodo.startswith(version + "::")
        )

        registro[
            "N_" + version.replace(" ", "_")
        ] = len(codigos)

        registro[
            "Codigos_" + version.replace(" ", "_")
        ] = " | ".join(codigos)

    grupos_tocados = sorted(
        set(
            membresia_componentes.loc[
                membresia_componentes.apply(
                    lambda r:
                    (
                        str(r["Version_SCIAN"]).strip()
                        + "::"
                        + str(r["SCIAN_original"]).strip()
                    ) in componente,
                    axis=1
                ),
                "Grupo_ID"
            ]
        )
    )

    registro["N_grupos_originales"] = len(
        grupos_tocados
    )

    registro["Grupos_originales"] = (
        " || ".join(grupos_tocados)
    )

    familias_v2.append(registro)

    numero += 1


familias_longitudinales_v2 = pd.DataFrame(
    familias_v2
)

print(
    "Familias anteriores:",
    len(familias_longitudinales)
)

print(
    "Familias V2:",
    len(familias_longitudinales_v2)
)

display(
    familias_longitudinales_v2[
        [
            "Familia",
            "N_grupos_originales",
            "Total_nodos",
            "N_SCIAN_2007",
            "N_SCIAN_2013",
            "N_SCIAN_2018",
            "N_SCIAN_2023",
            "Grupos_originales"
        ]
    ]
    .sort_values(
        "Total_nodos",
        ascending=False
    )
    .reset_index(drop=True)
)

Familias anteriores: 12
Familias V2: 12


,Familia,N_grupos_originales,Total_nodos,N_SCIAN_2007,N_SCIAN_2013,N_SCIAN_2018,N_SCIAN_2023,Grupos_originales
0,FL01,3,330,80,82,84,84,2007 → 2013 | G01 || 2013 → 2018 | G01 || 2018...
1,FL12,1,51,9,10,17,15,2018 → 2023 | G02
2,FL06,1,20,2,6,6,6,2007 → 2013 | G06
3,FL11,1,16,2,2,7,5,2013 → 2018 | G03
4,FL09,2,12,2,3,4,3,2013 → 2018 | G04 || 2018 → 2023 | G03
5,FL02,1,11,2,5,2,2,2007 → 2013 | G02
6,FL04,1,11,2,3,3,3,2007 → 2013 | G04
7,FL05,1,9,2,3,2,2,2007 → 2013 | G05
8,FL08,1,7,1,2,2,2,2007 → 2013 | G08
9,FL07,1,7,1,2,2,2,2007 → 2013 | G07


In [151]:
# ============================================================
# PASO 12N
# SECTORES DE LAS FAMILIAS LONGITUDINALES V2
# ============================================================

filas = []

for _, fam in familias_longitudinales_v2.iterrows():

    sectores = set()
    codigos_totales = set()

    for version in [
        "SCIAN 2007",
        "SCIAN 2013",
        "SCIAN 2018",
        "SCIAN 2023"
    ]:

        columna = (
            "Codigos_"
            + version.replace(" ", "_")
        )

        codigos = [
            x.strip()
            for x in str(fam[columna]).split("|")
            if x.strip()
            and x.strip().lower() != "nan"
        ]

        for codigo in codigos:
            codigos_totales.add(codigo)
            sectores.add(codigo[:2])

    filas.append({
        "Familia": fam["Familia"],
        "Sectores": " | ".join(sorted(sectores)),
        "N_sectores": len(sectores),
        "Codigos_distintos": len(codigos_totales),
        "Grupos_originales": fam["Grupos_originales"]
    })


sectores_familias_v2 = pd.DataFrame(filas)

display(
    sectores_familias_v2
    .sort_values(
        [
            "N_sectores",
            "Codigos_distintos"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

,Familia,Sectores,N_sectores,Codigos_distintos,Grupos_originales
0,FL09,33 | 81,2,4,2013 → 2018 | G04 || 2018 → 2023 | G03
1,FL01,46,1,84,2007 → 2013 | G01 || 2013 → 2018 | G01 || 2018...
2,FL12,51,1,32,2018 → 2023 | G02
3,FL06,72,1,8,2007 → 2013 | G06
4,FL11,56,1,8,2013 → 2018 | G03
5,FL02,52,1,5,2007 → 2013 | G02
6,FL04,62,1,3,2007 → 2013 | G04
7,FL05,62,1,3,2007 → 2013 | G05
8,FL07,72,1,3,2007 → 2013 | G07
9,FL08,81,1,3,2007 → 2013 | G08


In [152]:
# ============================================================
# FASE 2 - PASO 12O
# GUARDAR FAMILIAS LONGITUDINALES V2
# ============================================================

familias_longitudinales_v2.to_csv(
    HOMOLOGACION /
    "FAMILIAS_LONGITUDINALES_COMPLEJAS_V2.csv",
    index=False,
    encoding="utf-8-sig"
)

sectores_familias_v2.to_csv(
    HOMOLOGACION /
    "SECTORES_FAMILIAS_LONGITUDINALES_V2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Familias longitudinales V2 guardadas.")
print("✓ Total definitivo provisional:", len(familias_longitudinales_v2))

✓ Familias longitudinales V2 guardadas.
✓ Total definitivo provisional: 12


In [153]:
# ============================================================
# FASE 2 - PASO 12P
# EXTRAER MEMBRESÍA COMPLETA DE FL01 - SECTOR 46
# ============================================================

fl01 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL01"
    ]
    .iloc[0]
)

membresia_FL01 = {}

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = (
        "Codigos_"
        + version.replace(" ", "_")
    )

    codigos = {
        x.strip()
        for x in str(fl01[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    }

    membresia_FL01[version] = codigos

    print(
        version,
        "→",
        len(codigos),
        "clases en FL01"
    )

SCIAN 2007 → 80 clases en FL01
SCIAN 2013 → 82 clases en FL01
SCIAN 2018 → 84 clases en FL01
SCIAN 2023 → 84 clases en FL01


In [154]:
# ============================================================
# PASO 12Q
# COBERTURA DE CLASES OBSERVADAS DEL SECTOR 46
# ============================================================

inventario_scian["Sector"] = (
    inventario_scian["Sector"]
    .astype(str)
    .str.strip()
)

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    observadas_46 = set(
        inventario_scian.loc[
            (
                inventario_scian["Version_SCIAN"] == version
            )
            &
            (
                inventario_scian["Sector"] == "46"
            ),
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    familia = membresia_FL01[version]

    print("\n=================================")
    print(version)
    print("=================================")

    print(
        "Clases sector 46 observadas:",
        len(observadas_46)
    )

    print(
        "Clases FL01:",
        len(familia)
    )

    print(
        "Clases observadas fuera de FL01:",
        len(observadas_46 - familia)
    )

    print(
        "Clases FL01 no observadas en Morelia:",
        len(familia - observadas_46)
    )

    if observadas_46 - familia:
        print(
            "FUERA:",
            sorted(observadas_46 - familia)
        )

    if familia - observadas_46:
        print(
            "NO OBSERVADAS:",
            sorted(familia - observadas_46)
        )


SCIAN 2007
Clases sector 46 observadas: 80
Clases FL01: 80
Clases observadas fuera de FL01: 0
Clases FL01 no observadas en Morelia: 0

SCIAN 2013
Clases sector 46 observadas: 82
Clases FL01: 82
Clases observadas fuera de FL01: 0
Clases FL01 no observadas en Morelia: 0

SCIAN 2018
Clases sector 46 observadas: 83
Clases FL01: 84
Clases observadas fuera de FL01: 0
Clases FL01 no observadas en Morelia: 1
NO OBSERVADAS: ['468319']

SCIAN 2023
Clases sector 46 observadas: 83
Clases FL01: 84
Clases observadas fuera de FL01: 0
Clases FL01 no observadas en Morelia: 1
NO OBSERVADAS: ['468319']


In [155]:
# ============================================================
# PASO 12R
# VERIFICACIÓN ANUAL FL01 = SECTOR 46
# TODA LA SERIE HISTÓRICA
# ============================================================

filas = []

for año in sorted(
    inventario_scian["Año"]
    .dropna()
    .astype(int)
    .unique()
):

    temp = inventario_scian[
        inventario_scian["Año"].astype(int) == año
    ].copy()

    version = temp["Version_SCIAN"].iloc[0]

    codigos_fl01 = membresia_FL01[version]

    total_sector_46 = (
        temp[
            temp["Sector"] == "46"
        ]["Establecimientos"]
        .sum()
    )

    total_fl01 = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos_fl01)
        ]["Establecimientos"]
        .sum()
    )

    filas.append({
        "Año": año,
        "Version_SCIAN": version,
        "Establecimientos_FL01": total_fl01,
        "Total_sector_46": total_sector_46,
        "Diferencia": total_fl01 - total_sector_46,
        "Coincide_exacto":
            total_fl01 == total_sector_46
    })


verificacion_FL01_historica = pd.DataFrame(filas)

display(verificacion_FL01_historica)

print(
    "\nCoincidencias exactas:",
    verificacion_FL01_historica[
        "Coincide_exacto"
    ].sum(),
    "de",
    len(verificacion_FL01_historica)
)

print(
    "Diferencia absoluta acumulada:",
    verificacion_FL01_historica[
        "Diferencia"
    ].abs().sum()
)

,Año,Version_SCIAN,Establecimientos_FL01,Total_sector_46,Diferencia,Coincide_exacto
0,2010,SCIAN 2007,16693,16693,0,True
1,2011,SCIAN 2007,16824,16824,0,True
2,2012,SCIAN 2007,16860,16860,0,True
3,2013,SCIAN 2007,17045,17045,0,True
4,2015,SCIAN 2013,16955,16955,0,True
5,2016,SCIAN 2013,17313,17313,0,True
6,2017,SCIAN 2013,17427,17427,0,True
7,2018,SCIAN 2018,17452,17452,0,True
8,2019,SCIAN 2018,17642,17642,0,True
9,2020,SCIAN 2018,17984,17984,0,True



Coincidencias exactas: 16 de 16
Diferencia absoluta acumulada: 0


In [156]:
verificacion_FL01_historica.to_csv(
    HOMOLOGACION /
    "VERIFICACION_HISTORICA_FL01_SECTOR46_2010_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Verificación histórica FL01 guardada.")

✓ Verificación histórica FL01 guardada.


### Resolución de la familia longitudinal FL01: comercio al por menor

La familia longitudinal FL01 integra las reestructuraciones del sector 46,
Comercio al por menor, identificadas en los tres cambios de versión del SCIAN:
2007–2013, 2013–2018 y 2018–2023.

La integración longitudinal mostró que la familia está formada por:

- 80 clases en SCIAN 2007.
- 82 clases en SCIAN 2013.
- 84 clases en SCIAN 2018.
- 84 clases en SCIAN 2023.

Al contrastar la familia con las clases efectivamente observadas en Morelia,
se comprobó que todas las clases del sector 46 presentes en las cuatro
versiones pertenecen a FL01. La clase 468319 forma parte de la familia
clasificatoria en SCIAN 2018 y SCIAN 2023, pero no fue observada en Morelia.

La comparación del número de establecimientos de FL01 con el total anual
del sector 46 mostró coincidencia exacta en los 16 cortes anuales utilizados
entre 2010 y 2026. La diferencia absoluta acumulada fue igual a cero.

Por tanto, FL01 se considera resuelta mediante homologación a nivel del
sector 46, Comercio al por menor. Las clases de seis dígitos se conservan
para análisis internos de cada versión del SCIAN, pero no se asume
comparabilidad longitudinal directa entre clases individuales cuando forman
parte de las reestructuraciones identificadas.

Esta solución permite mantener la comparabilidad histórica completa del
agregado comercio al por menor sin realizar distribuciones artificiales de
establecimientos.

In [157]:
# ============================================================
# FASE 2 - PASO 12S
# CONTROL DE FAMILIAS LONGITUDINALES
# ============================================================

control_familias = sectores_familias_v2.copy()

control_familias["Estado"] = "Pendiente"
control_familias["Tratamiento"] = ""
control_familias["Observacion"] = ""

# ------------------------------------------------------------
# FL01 - COMERCIO
# ------------------------------------------------------------

mask = control_familias["Familia"] == "FL01"

control_familias.loc[mask, "Estado"] = "RESUELTA"
control_familias.loc[mask, "Tratamiento"] = (
    "Homologación a nivel sector 46"
)
control_familias.loc[mask, "Observacion"] = (
    "Coincidencia exacta 16/16 años; diferencia acumulada = 0"
)

# ------------------------------------------------------------
# FL04 - SECTOR 62
# ------------------------------------------------------------

mask = control_familias["Familia"] == "FL04"

control_familias.loc[mask, "Estado"] = "RESUELTA"
control_familias.loc[mask, "Tratamiento"] = (
    "Grupo homologado 621111+621113 / "
    "621111+621113+621115"
)

# ------------------------------------------------------------
# FL06 - SECTOR 72
# ------------------------------------------------------------

mask = control_familias["Familia"] == "FL06"

control_familias.loc[mask, "Estado"] = "RESUELTA"
control_familias.loc[mask, "Tratamiento"] = (
    "Grupo histórico homologado G06"
)

# ------------------------------------------------------------
# FL07 - SECTOR 72
# ------------------------------------------------------------

mask = control_familias["Familia"] == "FL07"

control_familias.loc[mask, "Estado"] = "RESUELTA"
control_familias.loc[mask, "Tratamiento"] = (
    "Grupo histórico homologado G07"
)

# ------------------------------------------------------------
# FL09 - SECTORES 33 / 81
# ------------------------------------------------------------

mask = control_familias["Familia"] == "FL09"

control_familias.loc[mask, "Estado"] = "RESUELTA CON INCERTIDUMBRE"
control_familias.loc[mask, "Tratamiento"] = (
    "Intervalo inferior-superior"
)
control_familias.loc[mask, "Observacion"] = (
    "Transferencia intersectorial; incertidumbre máxima = "
    "1 establecimiento (0.00247% del universo)"
)

display(
    control_familias[
        [
            "Familia",
            "Sectores",
            "Grupos_originales",
            "Estado",
            "Tratamiento",
            "Observacion"
        ]
    ]
    .sort_values("Familia")
    .reset_index(drop=True)
)

print(
    "\nResueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente").sum()
)

,Familia,Sectores,Grupos_originales,Estado,Tratamiento,Observacion
0,FL01,46,2007 → 2013 | G01 || 2013 → 2018 | G01 || 2018...,RESUELTA,Homologación a nivel sector 46,Coincidencia exacta 16/16 años; diferencia acu...
1,FL02,52,2007 → 2013 | G02,Pendiente,,
2,FL03,53,2007 → 2013 | G03,Pendiente,,
3,FL04,62,2007 → 2013 | G04,RESUELTA,Grupo homologado 621111+621113 / 621111+621113...,
4,FL05,62,2007 → 2013 | G05,Pendiente,,
5,FL06,72,2007 → 2013 | G06,RESUELTA,Grupo histórico homologado G06,
6,FL07,72,2007 → 2013 | G07,RESUELTA,Grupo histórico homologado G07,
7,FL08,81,2007 → 2013 | G08,Pendiente,,
8,FL09,33 | 81,2013 → 2018 | G04 || 2018 → 2023 | G03,RESUELTA CON INCERTIDUMBRE,Intervalo inferior-superior,Transferencia intersectorial; incertidumbre má...
9,FL10,48,2013 → 2018 | G02,Pendiente,,



Resueltas: 5
Pendientes: 7


In [158]:
control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Control de familias guardado.")

✓ Control de familias guardado.


In [159]:
# ============================================================
# FASE 2 - PASO 13A
# INSPECCIONAR FL10 - SECTOR 48
# ============================================================

fila_FL10 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL10"
    ]
    .iloc[0]
)

print("=== FL10 - SECTOR 48 ===")

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = (
        "Codigos_"
        + version.replace(" ", "_")
    )

    print(
        version + ":",
        fila_FL10[columna]
    )

=== FL10 - SECTOR 48 ===
SCIAN 2007: 485111
SCIAN 2013: 485111
SCIAN 2018: 485111 | 485115
SCIAN 2023: 485111


In [160]:
# ============================================================
# PASO 13B
# EVOLUCIÓN EMPÍRICA DE FL10
# ============================================================

codigos_FL10 = set()

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = (
        "Codigos_"
        + version.replace(" ", "_")
    )

    codigos_FL10.update(
        x.strip()
        for x in str(fila_FL10[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    )


evolucion_FL10 = (
    inventario_scian[
        inventario_scian["SCIAN_original"]
        .astype(str)
        .str.strip()
        .isin(codigos_FL10)
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "SCIAN_original",
            "Establecimientos"
        ]
    ]
    .sort_values(
        ["Año", "SCIAN_original"]
    )
    .reset_index(drop=True)
)

tabla_FL10 = (
    evolucion_FL10.pivot_table(
        index=["Año", "Version_SCIAN"],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_FL10)

SCIAN_original,Año,Version_SCIAN,485111
0,2010,SCIAN 2007,77
1,2011,SCIAN 2007,77
2,2012,SCIAN 2007,78
3,2013,SCIAN 2007,92
4,2015,SCIAN 2013,40
5,2016,SCIAN 2013,37
6,2017,SCIAN 2013,36
7,2018,SCIAN 2018,36
8,2019,SCIAN 2018,41
9,2020,SCIAN 2018,55


In [161]:
# ============================================================
# FASE 2 - PASO 13C
# AUDITORÍA COMPLETA DE FL10 - SECTOR 48
# ============================================================

fila_FL10 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL10"
    ]
    .iloc[0]
)

print("========== MEMBRESÍA FL10 ==========")

codigos_FL10_version = {}

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = "Codigos_" + version.replace(" ", "_")

    codigos = [
        x.strip()
        for x in str(fila_FL10[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    ]

    codigos_FL10_version[version] = set(codigos)

    print(version + ":", codigos)


# ------------------------------------------------------------
# RELACIÓN OFICIAL 2013 -> 2018
# ------------------------------------------------------------

corr_FL10 = (
    pd.read_csv(
        HOMOLOGACION /
        "CORRESPONDENCIA_VALIDADA_SCIAN_2013_2018.csv",
        dtype=str
    )
)

codigos_2013 = codigos_FL10_version["SCIAN 2013"]
codigos_2018 = codigos_FL10_version["SCIAN 2018"]

detalle_FL10 = corr_FL10[
    corr_FL10["SCIAN_2013"]
    .astype(str).str.strip()
    .isin(codigos_2013)
    |
    corr_FL10["SCIAN_2018"]
    .astype(str).str.strip()
    .isin(codigos_2018)
].copy()

print("\n========== CORRESPONDENCIA OFICIAL ==========")

display(
    detalle_FL10[
        [
            "SCIAN_2013",
            "Nombre_2013",
            "Parte",
            "SCIAN_2018",
            "Nombre_2018"
        ]
    ]
)


# ------------------------------------------------------------
# PRESENCIA REAL DE CADA CÓDIGO EN MORELIA
# ------------------------------------------------------------

todos_codigos_FL10 = set().union(
    *codigos_FL10_version.values()
)

presencia_FL10 = (
    inventario_scian[
        inventario_scian["SCIAN_original"]
        .astype(str).str.strip()
        .isin(todos_codigos_FL10)
    ]
    .groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ],
        as_index=False
    )
    .agg(
        Años_observados=("Año", "nunique"),
        Total_establecimientos=("Establecimientos", "sum"),
        Max_establecimientos=("Establecimientos", "max")
    )
)

print("\n========== PRESENCIA EN MORELIA ==========")

display(presencia_FL10)

========== MEMBRESÍA FL10 ==========
SCIAN 2007: ['485111']
SCIAN 2013: ['485111']
SCIAN 2018: ['485111', '485115']
SCIAN 2023: ['485111']

========== CORRESPONDENCIA OFICIAL ==========


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018
36,485111,Transporte colectivo urbano y suburbano de,NaN,485111,Transporte colectivo urbano y suburbano de pas...
37,485111,Transporte colectivo urbano y suburbano de pas...,NaN,485115,Transporte colectivo urbano y suburbano de pas...



========== PRESENCIA EN MORELIA ==========


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,485111,4,324,92
1,SCIAN 2013,485111,3,113,40
2,SCIAN 2018,485111,6,296,55
3,SCIAN 2023,485111,3,208,74


In [162]:
print("========== CORRESPONDENCIA OFICIAL ==========")

display(
    detalle_FL10[
        [
            "SCIAN_2013",
            "Nombre_2013",
            "Parte",
            "SCIAN_2018",
            "Nombre_2018"
        ]
    ]
)

print("\n========== PRESENCIA EN MORELIA ==========")

display(presencia_FL10)

========== CORRESPONDENCIA OFICIAL ==========


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018
36,485111,Transporte colectivo urbano y suburbano de,NaN,485111,Transporte colectivo urbano y suburbano de pas...
37,485111,Transporte colectivo urbano y suburbano de pas...,NaN,485115,Transporte colectivo urbano y suburbano de pas...



========== PRESENCIA EN MORELIA ==========


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,485111,4,324,92
1,SCIAN 2013,485111,3,113,40
2,SCIAN 2018,485111,6,296,55
3,SCIAN 2023,485111,3,208,74


In [163]:
# ============================================================
# FASE 2 - PASO 13D
# AUDITORÍA FINAL DE FL10 - 485111 / 485115
# ============================================================

# ------------------------------------------------------------
# 1. TEXTO CRUDO DEL PDF SCIAN 2013 -> 2018
# ------------------------------------------------------------

print("========== PDF 2013 -> 2018 ==========")

with pdfplumber.open(pdf_2013_2018) as pdf:

    for numero_pagina, pagina in enumerate(pdf.pages, start=1):

        texto = pagina.extract_text() or ""

        if "485111" in texto or "485115" in texto:

            print("\nPÁGINA:", numero_pagina)

            lineas = texto.splitlines()

            for i, linea in enumerate(lineas):

                if "485111" in linea or "485115" in linea:

                    desde = max(0, i - 2)
                    hasta = min(len(lineas), i + 3)

                    for j in range(desde, hasta):
                        print(f"{j:03d}: {lineas[j]}")

                    print("--------------------------------")


# ------------------------------------------------------------
# 2. CORRESPONDENCIA OFICIAL SCIAN 2018 -> 2023
# ------------------------------------------------------------

print("\n\n========== SCIAN 2018 -> 2023 ==========")

corr_18_23_FL10 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

detalle_FL10_18_23 = (
    corr_18_23_FL10[
        corr_18_23_FL10["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(["485111", "485115"])
        |
        corr_18_23_FL10["SCIAN_2023"]
        .astype(str).str.strip()
        .eq("485111")
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
    .sort_values(
        ["SCIAN_2018", "SCIAN_2023"]
    )
    .reset_index(drop=True)
)

display(detalle_FL10_18_23)

========== PDF 2013 -> 2018 ==========

PÁGINA: 10
008: carbón vegetal y leña)
009: 48-49 TRANSPORTES, CORREOS Y ALMACENAMIENTO
010: 485111 Transporte colectivo urbano y suburbano de p 1/2 485111 Transporte colectivo urbano y suburbano de
011: pasajeros en autobuses de ruta fija pasajeros en autobuses de ruta fija (transporte
012: colectivo urbano y suburbano de pasajeros en
--------------------------------
013: autobuses y midibuses, con horarios regulares y
014: rutas fija)
015: 485115 Transporte colectivo urbano y suburbano p 2/2 485111 Transporte colectivo urbano y suburbano de
016: de pasajeros en autobuses que transitan en pasajeros en autobuses de ruta fija (transporte
017: carril exclusivo colectivo urbano y suburbano de pasajeros en
--------------------------------

PÁGINA: 16
026: natural vehicular
027: 48-49 TRANSPORTES, CORREOS Y ALMACENAMIENTO
028: 485111 Transporte colectivo urbano y suburbano de 485111 Transporte colectivo urbano y suburbano de
029: pasajeros en autobuse

,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023


In [164]:
# ============================================================
# PASO 13D
# AUDITAR 485111 / 485115 DIRECTAMENTE EN LOS PDF OFICIALES
# ============================================================

print("========== PDF SCIAN 2013 -> 2018 ==========")

with pdfplumber.open(pdf_2013_2018) as pdf:

    for numero_pagina, pagina in enumerate(pdf.pages, start=1):

        texto = pagina.extract_text() or ""

        if "485111" in texto or "485115" in texto:

            print("\nPÁGINA:", numero_pagina)

            lineas = texto.splitlines()

            for i, linea in enumerate(lineas):

                if "485111" in linea or "485115" in linea:

                    desde = max(0, i - 2)
                    hasta = min(len(lineas), i + 3)

                    for j in range(desde, hasta):
                        print(f"{j:03d}: {lineas[j]}")

                    print("--------------------------------")


print("\n========== TABLA SCIAN 2018 -> 2023 ==========")

corr_18_23_FL10 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

detalle_FL10_18_23 = (
    corr_18_23_FL10[
        corr_18_23_FL10["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(["485111", "485115"])
        |
        corr_18_23_FL10["SCIAN_2023"]
        .astype(str).str.strip()
        .eq("485111")
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
    .sort_values(
        ["SCIAN_2018", "SCIAN_2023"]
    )
    .reset_index(drop=True)
)

display(detalle_FL10_18_23)

========== PDF SCIAN 2013 -> 2018 ==========

PÁGINA: 10
008: carbón vegetal y leña)
009: 48-49 TRANSPORTES, CORREOS Y ALMACENAMIENTO
010: 485111 Transporte colectivo urbano y suburbano de p 1/2 485111 Transporte colectivo urbano y suburbano de
011: pasajeros en autobuses de ruta fija pasajeros en autobuses de ruta fija (transporte
012: colectivo urbano y suburbano de pasajeros en
--------------------------------
013: autobuses y midibuses, con horarios regulares y
014: rutas fija)
015: 485115 Transporte colectivo urbano y suburbano p 2/2 485111 Transporte colectivo urbano y suburbano de
016: de pasajeros en autobuses que transitan en pasajeros en autobuses de ruta fija (transporte
017: carril exclusivo colectivo urbano y suburbano de pasajeros en
--------------------------------

PÁGINA: 16
026: natural vehicular
027: 48-49 TRANSPORTES, CORREOS Y ALMACENAMIENTO
028: 485111 Transporte colectivo urbano y suburbano de 485111 Transporte colectivo urbano y suburbano de
029: pasajeros en au

,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023


In [165]:
# ============================================================
# PASO 13D
# AUDITAR 485111 / 485115 DIRECTAMENTE EN LOS PDF OFICIALES
# ============================================================

print("========== PDF SCIAN 2013 -> 2018 ==========")

with pdfplumber.open(pdf_2013_2018) as pdf:

    for numero_pagina, pagina in enumerate(pdf.pages, start=1):

        texto = pagina.extract_text() or ""

        if "485111" in texto or "485115" in texto:

            print("\nPÁGINA:", numero_pagina)

            lineas = texto.splitlines()

            for i, linea in enumerate(lineas):

                if "485111" in linea or "485115" in linea:

                    desde = max(0, i - 2)
                    hasta = min(len(lineas), i + 3)

                    for j in range(desde, hasta):
                        print(f"{j:03d}: {lineas[j]}")

                    print("--------------------------------")


print("\n========== TABLA SCIAN 2018 -> 2023 ==========")

corr_18_23_FL10 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

detalle_FL10_18_23 = (
    corr_18_23_FL10[
        corr_18_23_FL10["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(["485111", "485115"])
        |
        corr_18_23_FL10["SCIAN_2023"]
        .astype(str).str.strip()
        .eq("485111")
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
    .sort_values(
        ["SCIAN_2018", "SCIAN_2023"]
    )
    .reset_index(drop=True)
)

display(detalle_FL10_18_23)

========== PDF SCIAN 2013 -> 2018 ==========

PÁGINA: 10
008: carbón vegetal y leña)
009: 48-49 TRANSPORTES, CORREOS Y ALMACENAMIENTO
010: 485111 Transporte colectivo urbano y suburbano de p 1/2 485111 Transporte colectivo urbano y suburbano de
011: pasajeros en autobuses de ruta fija pasajeros en autobuses de ruta fija (transporte
012: colectivo urbano y suburbano de pasajeros en
--------------------------------
013: autobuses y midibuses, con horarios regulares y
014: rutas fija)
015: 485115 Transporte colectivo urbano y suburbano p 2/2 485111 Transporte colectivo urbano y suburbano de
016: de pasajeros en autobuses que transitan en pasajeros en autobuses de ruta fija (transporte
017: carril exclusivo colectivo urbano y suburbano de pasajeros en
--------------------------------

PÁGINA: 16
026: natural vehicular
027: 48-49 TRANSPORTES, CORREOS Y ALMACENAMIENTO
028: 485111 Transporte colectivo urbano y suburbano de 485111 Transporte colectivo urbano y suburbano de
029: pasajeros en au

,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023


### Resolución de FL10: transporte colectivo urbano y suburbano

La familia FL10 corresponde a una división de la clase 485111 de SCIAN
2013 en dos clases en SCIAN 2018:

- 485111, Transporte colectivo urbano y suburbano de pasajeros en
  autobuses de ruta fija.
- 485115, Transporte colectivo urbano y suburbano de pasajeros en
  autobuses que transitan en carril exclusivo.

La tabla comparativa oficial identifica estas relaciones como p 1/2 y
p 2/2, respectivamente.

Ambas clases permanecen vigentes en SCIAN 2023. Sin embargo, en los
cortes históricos del DENUE correspondientes a Morelia no se observaron
establecimientos clasificados en 485115.

Por tanto, aunque la modificación clasificatoria debe conservarse en la
documentación metodológica, su impacto empírico en la serie analizada es
igual a cero. El grupo homologado se define como 485111 en SCIAN 2007 y
2013, y como 485111 + 485115 en SCIAN 2018 y 2023.

Debido a que 485115 no presenta establecimientos observados en Morelia,
los valores del grupo homologado coinciden con los registros de 485111 en
los años correspondientes.

In [166]:
# ============================================================
# FASE 2 - PASO 13E
# SERIE HOMOLOGADA DEFINITIVA FL10 - SECTOR 48
# ============================================================

FL10_codigos = {

    "SCIAN 2007": {
        "485111"
    },

    "SCIAN 2013": {
        "485111"
    },

    "SCIAN 2018": {
        "485111",
        "485115"
    },

    "SCIAN 2023": {
        "485111",
        "485115"
    }
}


filas_FL10 = []

for version, codigos in FL10_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"].eq(version)
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL10"
    resumen["Sector"] = "48"

    filas_FL10.append(resumen)


serie_FL10 = pd.concat(
    filas_FL10,
    ignore_index=True
)

serie_FL10 = (
    serie_FL10[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL10)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL10,48,2010,SCIAN 2007,77
1,FL10,48,2011,SCIAN 2007,77
2,FL10,48,2012,SCIAN 2007,78
3,FL10,48,2013,SCIAN 2007,92
4,FL10,48,2015,SCIAN 2013,40
5,FL10,48,2016,SCIAN 2013,37
6,FL10,48,2017,SCIAN 2013,36
7,FL10,48,2018,SCIAN 2018,36
8,FL10,48,2019,SCIAN 2018,41
9,FL10,48,2020,SCIAN 2018,55


In [167]:
# ============================================================
# PASO 13F
# CONFIRMAR IMPACTO EMPÍRICO DE 485115
# ============================================================

presencia_485115 = inventario_scian[
    inventario_scian["SCIAN_original"]
    .astype(str)
    .str.strip()
    .eq("485115")
].copy()

print(
    "Filas observadas de 485115 en Morelia:",
    len(presencia_485115)
)

print(
    "Establecimientos observados de 485115:",
    presencia_485115["Establecimientos"].sum()
)

print(
    "\nAños FL10:",
    serie_FL10["Año"].tolist()
)

print(
    "Número de años:",
    serie_FL10["Año"].nunique()
)

print(
    "Nulos:",
    serie_FL10.isna().sum().sum()
)

print(
    "Duplicados año:",
    serie_FL10.duplicated(
        subset=["Año"]
    ).sum()
)

Filas observadas de 485115 en Morelia: 0
Establecimientos observados de 485115: 0

Años FL10: [2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Número de años: 16
Nulos: 0
Duplicados año: 0


In [168]:
# ============================================================
# PASO 13G
# GUARDAR FL10 Y MARCAR COMO RESUELTA
# ============================================================

serie_FL10.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_FL10_SECTOR48.csv",
    index=False,
    encoding="utf-8-sig"
)

mask = (
    control_familias["Familia"] == "FL10"
)

control_familias.loc[
    mask,
    "Estado"
] = "RESUELTA"

control_familias.loc[
    mask,
    "Tratamiento"
] = (
    "485111 en SCIAN 2007/2013; "
    "485111+485115 en SCIAN 2018/2023"
)

control_familias.loc[
    mask,
    "Observacion"
] = (
    "485115 pertenece a la familia oficial, "
    "pero registra 0 establecimientos observados en Morelia"
)

control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ FL10 homologada y guardada.")

print(
    "Resueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente")
    .sum()
)

✓ FL10 homologada y guardada.
Resueltas: 6
Pendientes: 6


### Revisión de la familia FL02: sector 52

La familia longitudinal FL02 pertenece al sector 52, Servicios financieros
y de seguros, y tiene su origen en una reestructuración identificada entre
SCIAN 2007 y SCIAN 2013.

Antes de establecer una regla de homologación se revisan las clases que
integran la familia en cada versión del SCIAN, sus correspondencias oficiales
y su presencia efectiva en los establecimientos observados de Morelia.

In [169]:
# ============================================================
# FASE 2 - PASO 13H
# INSPECCIONAR FL02 - SECTOR 52
# ============================================================

fila_FL02 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL02"
    ]
    .iloc[0]
)

print("========== MEMBRESÍA LONGITUDINAL FL02 ==========")

codigos_FL02_version = {}

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = (
        "Codigos_"
        + version.replace(" ", "_")
    )

    codigos = [
        x.strip()
        for x in str(fila_FL02[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    ]

    codigos_FL02_version[version] = set(codigos)

    print(version + ":", codigos)

========== MEMBRESÍA LONGITUDINAL FL02 ==========
SCIAN 2007: ['522430', '522490']
SCIAN 2013: ['522410', '522420', '522430', '522460', '522490']
SCIAN 2018: ['522460', '522490']
SCIAN 2023: ['522460', '522490']


In [170]:
# ============================================================
# PASO 13I
# CORRESPONDENCIAS OFICIALES FL02 - SCIAN 2007 → 2013
# ============================================================

corr_FL02 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_SCIAN_2013_2007_CLASIFICADA_VALIDADA.csv",
    dtype=str
)

codigos_FL02_2007 = codigos_FL02_version["SCIAN 2007"]
codigos_FL02_2013 = codigos_FL02_version["SCIAN 2013"]

detalle_FL02 = (
    corr_FL02[
        corr_FL02["SCIAN_2007"]
        .astype(str).str.strip()
        .isin(codigos_FL02_2007)
        |
        corr_FL02["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(codigos_FL02_2013)
    ]
    [
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ]
    .sort_values(
        ["SCIAN_2007", "SCIAN_2013"]
    )
    .reset_index(drop=True)
)

display(detalle_FL02)

,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
0,522490,Otras instituciones de intermediación creditic...,p 1/2,522460,Sociedades financieras de objeto múltiple
1,522490,Otras instituciones de intermediación creditic...,p 2/2,522490,Otras instituciones de intermediación creditic...


In [171]:
# ============================================================
# PASO 13J
# PRESENCIA EMPÍRICA Y EVOLUCIÓN DE FL02
# ============================================================

todos_codigos_FL02 = set().union(
    *codigos_FL02_version.values()
)

evolucion_FL02 = (
    inventario_scian[
        inventario_scian["SCIAN_original"]
        .astype(str)
        .str.strip()
        .isin(todos_codigos_FL02)
    ]
    [
        [
            "Año",
            "Version_SCIAN",
            "SCIAN_original",
            "Establecimientos"
        ]
    ]
    .sort_values(
        ["Año", "SCIAN_original"]
    )
    .reset_index(drop=True)
)

tabla_FL02 = (
    evolucion_FL02.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_FL02)

print("\nPRESENCIA POR CÓDIGO:")

display(
    evolucion_FL02.groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ],
        as_index=False
    )
    .agg(
        Años_observados=("Año", "nunique"),
        Total_establecimientos=("Establecimientos", "sum"),
        Max_establecimientos=("Establecimientos", "max")
    )
)

SCIAN_original,Año,Version_SCIAN,522420,522430,522460,522490
0,2010,SCIAN 2007,0,0,0,11
1,2011,SCIAN 2007,1,9,0,14
2,2012,SCIAN 2007,1,9,0,14
3,2013,SCIAN 2007,1,9,0,14
4,2015,SCIAN 2013,0,2,37,1
5,2016,SCIAN 2013,0,2,12,31
6,2017,SCIAN 2013,0,2,14,31
7,2018,SCIAN 2018,0,0,16,31
8,2019,SCIAN 2018,0,0,42,11
9,2020,SCIAN 2018,0,0,63,14



PRESENCIA POR CÓDIGO:


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,522420,3,3,1
1,SCIAN 2007,522430,3,27,9
2,SCIAN 2007,522490,4,53,14
3,SCIAN 2013,522430,3,6,2
4,SCIAN 2013,522460,3,63,37
5,SCIAN 2013,522490,3,63,31
6,SCIAN 2018,522460,6,306,63
7,SCIAN 2018,522490,6,93,31
8,SCIAN 2023,522460,3,169,63
9,SCIAN 2023,522490,3,69,26


In [172]:
# ============================================================
# FASE 2 - PASO 13J CORREGIDO
# PRESENCIA EMPÍRICA DE FL02 RESPETANDO CADA VERSIÓN SCIAN
# ============================================================

filas_FL02 = []

for version, codigos in codigos_FL02_version.items():

    temp = inventario_scian[
        (
            inventario_scian["Version_SCIAN"] == version
        )
        &
        (
            inventario_scian["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        )
    ].copy()

    filas_FL02.append(temp)


evolucion_FL02_correcta = pd.concat(
    filas_FL02,
    ignore_index=True
)

evolucion_FL02_correcta = (
    evolucion_FL02_correcta[
        [
            "Año",
            "Version_SCIAN",
            "SCIAN_original",
            "Establecimientos"
        ]
    ]
    .sort_values(
        ["Año", "SCIAN_original"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# TABLA AÑO × CÓDIGO
# ------------------------------------------------------------

tabla_FL02_correcta = (
    evolucion_FL02_correcta.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

print("========== MEMBRESÍA FL02 ==========")

for version, codigos in codigos_FL02_version.items():
    print(
        version + ":",
        sorted(codigos)
    )

print("\n========== EVOLUCIÓN CORRECTA ==========")

display(tabla_FL02_correcta)


print("\n========== PRESENCIA CORRECTA POR CÓDIGO ==========")

display(
    evolucion_FL02_correcta.groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ],
        as_index=False
    )
    .agg(
        Años_observados=("Año", "nunique"),
        Total_establecimientos=(
            "Establecimientos",
            "sum"
        ),
        Max_establecimientos=(
            "Establecimientos",
            "max"
        )
    )
)

========== MEMBRESÍA FL02 ==========
SCIAN 2007: ['522430', '522490']
SCIAN 2013: ['522410', '522420', '522430', '522460', '522490']
SCIAN 2018: ['522460', '522490']
SCIAN 2023: ['522460', '522490']

========== EVOLUCIÓN CORRECTA ==========


SCIAN_original,Año,Version_SCIAN,522430,522460,522490
0,2010,SCIAN 2007,0,0,11
1,2011,SCIAN 2007,9,0,14
2,2012,SCIAN 2007,9,0,14
3,2013,SCIAN 2007,9,0,14
4,2015,SCIAN 2013,2,37,1
5,2016,SCIAN 2013,2,12,31
6,2017,SCIAN 2013,2,14,31
7,2018,SCIAN 2018,0,16,31
8,2019,SCIAN 2018,0,42,11
9,2020,SCIAN 2018,0,63,14



========== PRESENCIA CORRECTA POR CÓDIGO ==========


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,522430,3,27,9
1,SCIAN 2007,522490,4,53,14
2,SCIAN 2013,522430,3,6,2
3,SCIAN 2013,522460,3,63,37
4,SCIAN 2013,522490,3,63,31
5,SCIAN 2018,522460,6,306,63
6,SCIAN 2018,522490,6,93,31
7,SCIAN 2023,522460,3,169,63
8,SCIAN 2023,522490,3,69,26


In [173]:
print("========== CORRESPONDENCIA OFICIAL FL02 ==========")

display(
    detalle_FL02[
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ]
)

========== CORRESPONDENCIA OFICIAL FL02 ==========


,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
0,522490,Otras instituciones de intermediación creditic...,p 1/2,522460,Sociedades financieras de objeto múltiple
1,522490,Otras instituciones de intermediación creditic...,p 2/2,522490,Otras instituciones de intermediación creditic...


In [174]:
# ============================================================
# FASE 2 - PASO 13K
# VER CORRESPONDENCIA COMPLETA Y PROPAGAR FL02
# ============================================================

print("========== CORRESPONDENCIA COMPLETA 2007 → 2013 ==========")

print(
    detalle_FL02[
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# PROPAGAR TODO EL CONJUNTO 2013 HACIA 2018
# ------------------------------------------------------------

FL02_2007 = {
    "522430",
    "522490"
}

FL02_2013 = {
    "522410",
    "522420",
    "522430",
    "522460",
    "522490"
}

FL02_2018, detalle_FL02_13_18 = propagar_codigos(
    FL02_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

FL02_2023, detalle_FL02_18_23 = propagar_codigos(
    FL02_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)


print("\n========== FAMILIA PROPAGADA ==========")

print("SCIAN 2007:", sorted(FL02_2007))
print("SCIAN 2013:", sorted(FL02_2013))
print("SCIAN 2018:", sorted(FL02_2018))
print("SCIAN 2023:", sorted(FL02_2023))


print("\n========== CAMBIOS EXPLÍCITOS 2013 → 2018 ==========")

display(
    detalle_FL02_13_18[
        detalle_FL02_13_18["Cambio_explicito"]
    ]
)


print("\n========== CAMBIOS EXPLÍCITOS 2018 → 2023 ==========")

display(
    detalle_FL02_18_23[
        detalle_FL02_18_23["Cambio_explicito"]
    ]
)

========== CORRESPONDENCIA COMPLETA 2007 → 2013 ==========
SCIAN_2007                                                                                                                   Nombre_2007 Parte SCIAN_2013                                                               Nombre_2013
    522490         Otras instituciones de intermediación crediticia y financiera no bursátil (sociedades financieras de objeto múltiple) p 1/2     522460                                 Sociedades financieras de objeto múltiple
    522490 Otras instituciones de intermediación crediticia y financiera no bursátil (excepto sociedades financieras de objeto múltiple) p 2/2     522490 Otras instituciones de intermediación crediticia y financiera no bursátil

========== FAMILIA PROPAGADA ==========
SCIAN 2007: ['522430', '522490']
SCIAN 2013: ['522410', '522420', '522430', '522460', '522490']
SCIAN 2018: ['522460', '522490']
SCIAN 2023: ['522460', '522490']

========== CAMBIOS EXPLÍCITOS 2013 → 2018 ==========


,Codigo_origen,Cambio_explicito,Codigo_destino
0,522410,True,522460
1,522420,True,522460
2,522430,True,522460
3,522460,True,522460



========== CAMBIOS EXPLÍCITOS 2018 → 2023 ==========


,Codigo_origen,Cambio_explicito,Codigo_destino


### Resolución de la familia FL02 del sector 52

La familia longitudinal FL02 corresponde a una reestructuración de clases
del sector 52, Servicios financieros y de seguros.

En SCIAN 2007, las clases observadas que integran la familia son 522430 y
522490. Entre SCIAN 2007 y SCIAN 2013, la clase 522490 se divide
parcialmente en 522460 y 522490, mientras que otras clases relacionadas
permanecen dentro del componente longitudinal.

En SCIAN 2013, la familia queda integrada por las clases 522410, 522420,
522430, 522460 y 522490.

La transición SCIAN 2013–2018 concentra las clases 522410, 522420,
522430 y 522460 en la clase 522460, mientras que 522490 permanece dentro
del componente. En SCIAN 2018 y SCIAN 2023, por tanto, la familia queda
representada por las clases 522460 y 522490.

Esta estructura permite homologar el componente mediante agregación de las
clases que forman la familia en cada versión del SCIAN, sin realizar
distribuciones retrospectivas de establecimientos.

La familia homologada queda definida de la siguiente manera:

SCIAN 2007: 522430 + 522490.
SCIAN 2013: 522410 + 522420 + 522430 + 522460 + 522490.
SCIAN 2018: 522460 + 522490.
SCIAN 2023: 522460 + 522490.

In [175]:
# ============================================================
# FASE 2 - PASO 13L
# SERIE HISTÓRICA HOMOLOGADA FL02 - SECTOR 52
# ============================================================

FL02_codigos = {

    "SCIAN 2007": {
        "522430",
        "522490"
    },

    "SCIAN 2013": {
        "522410",
        "522420",
        "522430",
        "522460",
        "522490"
    },

    "SCIAN 2018": {
        "522460",
        "522490"
    },

    "SCIAN 2023": {
        "522460",
        "522490"
    }
}


filas_FL02 = []

for version, codigos in FL02_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    # Sumar códigos observados.
    # Los códigos de la familia no observados aportan cero.
    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL02"
    resumen["Sector"] = "52"

    filas_FL02.append(resumen)


serie_FL02 = pd.concat(
    filas_FL02,
    ignore_index=True
)

serie_FL02 = (
    serie_FL02[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL02)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL02,52,2010,SCIAN 2007,11
1,FL02,52,2011,SCIAN 2007,23
2,FL02,52,2012,SCIAN 2007,23
3,FL02,52,2013,SCIAN 2007,23
4,FL02,52,2015,SCIAN 2013,40
5,FL02,52,2016,SCIAN 2013,45
6,FL02,52,2017,SCIAN 2013,47
7,FL02,52,2018,SCIAN 2018,47
8,FL02,52,2019,SCIAN 2018,53
9,FL02,52,2020,SCIAN 2018,77


In [176]:
# ============================================================
# PASO 13M
# CONTROL DE LA SERIE FL02
# ============================================================

print("Años:")
print(serie_FL02["Año"].tolist())

print(
    "\nNúmero de años:",
    serie_FL02["Año"].nunique()
)

print(
    "Nulos:",
    serie_FL02.isna().sum().sum()
)

print(
    "Duplicados año:",
    serie_FL02.duplicated(
        subset=["Año"]
    ).sum()
)

print("\nSerie:")
display(
    serie_FL02[
        [
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
)

Años:
[2010, 2011, 2012, 2013, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Número de años: 16
Nulos: 0
Duplicados año: 0

Serie:


,Año,Version_SCIAN,Establecimientos
0,2010,SCIAN 2007,11
1,2011,SCIAN 2007,23
2,2012,SCIAN 2007,23
3,2013,SCIAN 2007,23
4,2015,SCIAN 2013,40
5,2016,SCIAN 2013,45
6,2017,SCIAN 2013,47
7,2018,SCIAN 2018,47
8,2019,SCIAN 2018,53
9,2020,SCIAN 2018,77


In [177]:
# ============================================================
# PASO 13N
# GUARDAR FL02 Y MARCAR COMO RESUELTA
# ============================================================

serie_FL02.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_FL02_SECTOR52.csv",
    index=False,
    encoding="utf-8-sig"
)

mask = control_familias["Familia"] == "FL02"

control_familias.loc[
    mask,
    "Estado"
] = "RESUELTA"

control_familias.loc[
    mask,
    "Tratamiento"
] = (
    "Agregación longitudinal por versión: "
    "2007=522430+522490; "
    "2013=522410+522420+522430+522460+522490; "
    "2018/2023=522460+522490"
)

control_familias.loc[
    mask,
    "Observacion"
] = (
    "Fusión N→1 entre SCIAN 2013 y 2018; "
    "sin reparto retrospectivo"
)

control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ FL02 sector 52 homologada y guardada.")

print(
    "Resueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente")
    .sum()
)

✓ FL02 sector 52 homologada y guardada.
Resueltas: 7
Pendientes: 5


### Revisión de la familia FL03: sector 53

La familia longitudinal FL03 pertenece al sector 53, Servicios inmobiliarios
y de alquiler de bienes muebles e intangibles.

Debido a que presenta una correspondencia compleja entre versiones del SCIAN,
se revisan las clases que integran la familia, sus correspondencias oficiales
y su presencia empírica en Morelia antes de establecer la regla definitiva
de homologación.

In [178]:
# ============================================================
# FASE 2 - PASO 13O
# INSPECCIONAR FL03 - SECTOR 53
# ============================================================

fila_FL03 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL03"
    ]
    .iloc[0]
)

print("========== MEMBRESÍA LONGITUDINAL FL03 ==========")

codigos_FL03_version = {}

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = (
        "Codigos_"
        + version.replace(" ", "_")
    )

    codigos = [
        x.strip()
        for x in str(fila_FL03[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    ]

    codigos_FL03_version[version] = set(codigos)

    print(version + ":", codigos)

========== MEMBRESÍA LONGITUDINAL FL03 ==========
SCIAN 2007: ['531119']
SCIAN 2013: ['531116', '531119']
SCIAN 2018: ['531119']
SCIAN 2023: ['531119']


In [179]:
# ============================================================
# PASO 13P
# CORRESPONDENCIAS OFICIALES DE FL03
# ============================================================

todos_FL03 = set().union(
    *codigos_FL03_version.values()
)

print("========== 2007 → 2013 ==========")

display(
    corr_2007_2013_validada[
        corr_2007_2013_validada["SCIAN_2007"]
        .astype(str).str.strip()
        .isin(todos_FL03)
        |
        corr_2007_2013_validada["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(todos_FL03)
    ]
    [
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ]
)

print("\n========== 2013 → 2018 ==========")

corr_13_18_FL03 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2013_2018.csv",
    dtype=str
)

display(
    corr_13_18_FL03[
        corr_13_18_FL03["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(todos_FL03)
        |
        corr_13_18_FL03["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(todos_FL03)
    ]
    [
        [
            "SCIAN_2013",
            "Nombre_2013",
            "Parte",
            "SCIAN_2018",
            "Nombre_2018"
        ]
    ]
)

print("\n========== 2018 → 2023 ==========")

corr_18_23_FL03 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

display(
    corr_18_23_FL03[
        corr_18_23_FL03["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(todos_FL03)
        |
        corr_18_23_FL03["SCIAN_2023"]
        .astype(str).str.strip()
        .isin(todos_FL03)
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
)

========== 2007 → 2013 ==========


,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
47,531119,Alquiler sin intermediación de otros bienes ra...,p 1/2,531116,Alquiler sin intermediación de edificios indus...
48,531119,Alquiler sin intermediación de otros bienes ra...,p 2/2,531119,Alquiler sin intermediación de otros bienes ra...



========== 2013 → 2018 ==========


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018



========== 2018 → 2023 ==========


,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023


In [180]:
# ============================================================
# PASO 13Q
# EVOLUCIÓN EMPÍRICA CORRECTA DE FL03
# ============================================================

filas_FL03 = []

for version, codigos in codigos_FL03_version.items():

    temp = inventario_scian[
        (
            inventario_scian["Version_SCIAN"] == version
        )
        &
        (
            inventario_scian["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        )
    ].copy()

    filas_FL03.append(temp)

evolucion_FL03 = pd.concat(
    filas_FL03,
    ignore_index=True
)

tabla_FL03 = (
    evolucion_FL03.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_FL03)

print("\nPRESENCIA POR CÓDIGO:")

display(
    evolucion_FL03.groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ],
        as_index=False
    )
    .agg(
        Años_observados=("Año", "nunique"),
        Total_establecimientos=("Establecimientos", "sum"),
        Max_establecimientos=("Establecimientos", "max")
    )
)

SCIAN_original,Año,Version_SCIAN,531119
0,2010,SCIAN 2007,8
1,2011,SCIAN 2007,8
2,2012,SCIAN 2007,9
3,2013,SCIAN 2007,9
4,2015,SCIAN 2013,4
5,2016,SCIAN 2013,5
6,2017,SCIAN 2013,5
7,2018,SCIAN 2018,5
8,2019,SCIAN 2018,1
9,2020,SCIAN 2018,1



PRESENCIA POR CÓDIGO:


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,531119,4,34,9
1,SCIAN 2013,531119,3,14,5
2,SCIAN 2018,531119,6,12,5
3,SCIAN 2023,531119,3,7,3


### Resolución de la familia FL03 del sector 53

La familia FL03 corresponde a una división de la clase 531119 de SCIAN 2007.

Entre SCIAN 2007 y SCIAN 2013, parte de su contenido se separa hacia la
clase 531116, mientras que el resto permanece en 531119. Las tablas
comparativas SCIAN 2013–2018 y SCIAN 2018–2023 no registran cambios
explícitos adicionales para estas clases.

Por tanto, la familia homologada queda definida como:

- SCIAN 2007: 531119.
- SCIAN 2013: 531116 + 531119.
- SCIAN 2018: 531116 + 531119.
- SCIAN 2023: 531116 + 531119.

En los cortes históricos del DENUE correspondientes a Morelia se identificó
una observación de la clase 531116, correspondiente a un establecimiento.
Por tanto, esta clase forma parte tanto de la familia clasificatoria FL03
como de la serie empírica observada.    

La homologación se realiza mediante agregación del conjunto completo, sin
distribuir retrospectivamente los establecimientos de 531119.

In [4]:
# ============================================================
# FASE 2 - PASO 13R
# PROPAGACIÓN CLASIFICATORIA CORRECTA DE FL03
# ============================================================

FL03_2007 = {
    "531119"
}

FL03_2013 = {
    "531116",
    "531119"
}

FL03_2018, detalle_FL03_13_18 = propagar_codigos(
    FL03_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

FL03_2023, detalle_FL03_18_23 = propagar_codigos(
    FL03_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("SCIAN 2007:", sorted(FL03_2007))
print("SCIAN 2013:", sorted(FL03_2013))
print("SCIAN 2018:", sorted(FL03_2018))
print("SCIAN 2023:", sorted(FL03_2023))

print("\nCambios explícitos 2013→2018:")
display(
    detalle_FL03_13_18[
        detalle_FL03_13_18["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2018→2023:")
display(
    detalle_FL03_18_23[
        detalle_FL03_18_23["Cambio_explicito"]
    ]
)

NameError: name 'propagar_codigos' is not defined

In [ ]:
# ============================================================
# PASO 13S
# SERIE HISTÓRICA HOMOLOGADA FL03 - SECTOR 53
# ============================================================

FL03_codigos = {

    "SCIAN 2007": {
        "531119"
    },

    "SCIAN 2013": {
        "531116",
        "531119"
    },

    "SCIAN 2018": {
        "531116",
        "531119"
    },

    "SCIAN 2023": {
        "531116",
        "531119"
    }
}


filas_FL03 = []

for version, codigos in FL03_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL03"
    resumen["Sector"] = "53"

    filas_FL03.append(resumen)


serie_FL03 = pd.concat(
    filas_FL03,
    ignore_index=True
)

serie_FL03 = (
    serie_FL03[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL03)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL03,53,2010,SCIAN 2007,8
1,FL03,53,2011,SCIAN 2007,8
2,FL03,53,2012,SCIAN 2007,9
3,FL03,53,2013,SCIAN 2007,9
4,FL03,53,2015,SCIAN 2013,4
5,FL03,53,2016,SCIAN 2013,5
6,FL03,53,2017,SCIAN 2013,5
7,FL03,53,2018,SCIAN 2018,5
8,FL03,53,2019,SCIAN 2018,1
9,FL03,53,2020,SCIAN 2018,1


In [ ]:
# ============================================================
# PASO 13T
# AUDITORÍA DE 531116
# ============================================================

presencia_531116 = inventario_scian[
    inventario_scian["SCIAN_original"]
    .astype(str)
    .str.strip()
    .eq("531116")
]

print(
    "Filas observadas de 531116:",
    len(presencia_531116)
)

print(
    "Establecimientos observados de 531116:",
    presencia_531116["Establecimientos"].sum()
)

print(
    "\nNúmero de años FL03:",
    serie_FL03["Año"].nunique()
)

print(
    "Nulos:",
    serie_FL03.isna().sum().sum()
)

print(
    "Duplicados:",
    serie_FL03.duplicated(
        subset=["Año"]
    ).sum()
)

Filas observadas de 531116: 1
Establecimientos observados de 531116: 1

Número de años FL03: 16
Nulos: 0
Duplicados: 0


In [ ]:
# ============================================================
# PASO 13U
# GUARDAR FL03 Y ACTUALIZAR CONTROL
# ============================================================

serie_FL03.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_FL03_SECTOR53.csv",
    index=False,
    encoding="utf-8-sig"
)

mask = control_familias["Familia"] == "FL03"

control_familias.loc[
    mask,
    "Estado"
] = "RESUELTA"

control_familias.loc[
    mask,
    "Tratamiento"
] = (
    "2007=531119; "
    "2013/2018/2023=531116+531119"
)

control_familias.loc[
    mask,
    "Observacion"
] = (
    "531116 pertenece a la familia clasificatoria, "
    "pero registra 0 establecimientos observados en Morelia"
)

control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ FL03 sector 53 homologada y guardada.")

print(
    "Resueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente")
    .sum()
)

✓ FL03 sector 53 homologada y guardada.
Resueltas: 8
Pendientes: 4


### Revisión de la familia FL05: sector 62

La familia longitudinal FL05 pertenece al sector 62, Servicios de salud y de
asistencia social.

Se revisan las clases involucradas, sus correspondencias oficiales y su
presencia efectiva en Morelia para determinar la regla de homologación
aplicable a toda la serie histórica.

In [185]:
# ============================================================
# FASE 2 - PASO 14A
# INSPECCIONAR FL05 - SECTOR 62
# ============================================================

fila_FL05 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL05"
    ]
    .iloc[0]
)

print("========== MEMBRESÍA LONGITUDINAL FL05 ==========")

codigos_FL05_version = {}

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = "Codigos_" + version.replace(" ", "_")

    codigos = [
        x.strip()
        for x in str(fila_FL05[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    ]

    codigos_FL05_version[version] = set(codigos)

    print(version + ":", codigos)

========== MEMBRESÍA LONGITUDINAL FL05 ==========
SCIAN 2007: ['621112', '621114']
SCIAN 2013: ['621112', '621114', '621116']
SCIAN 2018: ['621112', '621116']
SCIAN 2023: ['621112', '621116']


In [186]:
# ============================================================
# PASO 14B
# CORRESPONDENCIAS OFICIALES DE FL05
# ============================================================

todos_FL05 = set().union(
    *codigos_FL05_version.values()
)

print("========== 2007 → 2013 ==========")

detalle_FL05_07_13 = (
    corr_2007_2013_validada[
        corr_2007_2013_validada["SCIAN_2007"]
        .astype(str).str.strip()
        .isin(todos_FL05)
        |
        corr_2007_2013_validada["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(todos_FL05)
    ]
    [
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ]
)

display(detalle_FL05_07_13)


print("\n========== 2013 → 2018 ==========")

corr_13_18_FL05 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2013_2018.csv",
    dtype=str
)

detalle_FL05_13_18 = (
    corr_13_18_FL05[
        corr_13_18_FL05["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(todos_FL05)
        |
        corr_13_18_FL05["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(todos_FL05)
    ]
    [
        [
            "SCIAN_2013",
            "Nombre_2013",
            "Parte",
            "SCIAN_2018",
            "Nombre_2018"
        ]
    ]
)

display(detalle_FL05_13_18)


print("\n========== 2018 → 2023 ==========")

corr_18_23_FL05 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

detalle_FL05_18_23 = (
    corr_18_23_FL05[
        corr_18_23_FL05["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(todos_FL05)
        |
        corr_18_23_FL05["SCIAN_2023"]
        .astype(str).str.strip()
        .isin(todos_FL05)
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
)

display(detalle_FL05_18_23)

========== 2007 → 2013 ==========


,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
52,621112,(consultorios de medicina general del sector p...,NaN,621112,del sector público
54,621114,Consultorios de medicina especializada del sec...,p 1/2,621114,Consultorios de medicina especializada del sec...
57,621112,Consultorios de medicina general del sector pú...,p 2/2,621116,Clínicas de consultorios médicos del sector pú...
58,621114,Consultorios de medicina especializada del sec...,p 2/2,621116,Clínicas de consultorios médicos del sector pú...



========== 2013 → 2018 ==========


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018



========== 2018 → 2023 ==========


,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023


In [187]:
# ============================================================
# PASO 14C
# EVOLUCIÓN EMPÍRICA CORRECTA DE FL05
# ============================================================

filas_FL05 = []

for version, codigos in codigos_FL05_version.items():

    temp = inventario_scian[
        (
            inventario_scian["Version_SCIAN"] == version
        )
        &
        (
            inventario_scian["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        )
    ].copy()

    filas_FL05.append(temp)


evolucion_FL05 = pd.concat(
    filas_FL05,
    ignore_index=True
)

tabla_FL05 = (
    evolucion_FL05.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_FL05)

print("\nPRESENCIA POR CÓDIGO:")

display(
    evolucion_FL05.groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ],
        as_index=False
    )
    .agg(
        Años_observados=("Año", "nunique"),
        Total_establecimientos=("Establecimientos", "sum"),
        Max_establecimientos=("Establecimientos", "max")
    )
)

SCIAN_original,Año,Version_SCIAN,621112,621116
0,2010,SCIAN 2007,11,0
1,2011,SCIAN 2007,11,0
2,2012,SCIAN 2007,11,0
3,2013,SCIAN 2007,11,0
4,2015,SCIAN 2013,11,7
5,2016,SCIAN 2013,10,9
6,2017,SCIAN 2013,10,9
7,2018,SCIAN 2018,10,9
8,2019,SCIAN 2018,13,16
9,2020,SCIAN 2018,16,16



PRESENCIA POR CÓDIGO:


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,621112,4,44,11
1,SCIAN 2013,621112,3,31,11
2,SCIAN 2013,621116,3,25,9
3,SCIAN 2018,621112,6,87,16
4,SCIAN 2018,621116,6,89,16
5,SCIAN 2023,621112,3,36,14
6,SCIAN 2023,621116,3,63,21


### Resolución de la familia FL03 del sector 53

La familia FL03 corresponde a una división de la clase 531119 de SCIAN 2007.

Entre SCIAN 2007 y SCIAN 2013, parte de su contenido se separa hacia la
clase 531116, mientras que el resto permanece en 531119. Las tablas
comparativas SCIAN 2013–2018 y SCIAN 2018–2023 no registran cambios
explícitos adicionales para estas clases.

Por tanto, la familia homologada queda definida como:

- SCIAN 2007: 531119.
- SCIAN 2013: 531116 + 531119.
- SCIAN 2018: 531116 + 531119.
- SCIAN 2023: 531116 + 531119.

En los cortes históricos del DENUE correspondientes a Morelia se identificó
una observación de la clase 531116, correspondiente a un establecimiento.
Por tanto, esta clase forma parte tanto de la familia clasificatoria FL03
como de la serie empírica observada.

La homologación se realiza mediante agregación del conjunto completo,
sin distribuir retrospectivamente los establecimientos de 531119.

In [188]:
# ============================================================
# FASE 2 - PASO 13R
# PROPAGACIÓN CLASIFICATORIA CORRECTA DE FL03
# ============================================================

FL03_2007 = {
    "531119"
}

FL03_2013 = {
    "531116",
    "531119"
}

FL03_2018, detalle_FL03_13_18 = propagar_codigos(
    FL03_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

FL03_2023, detalle_FL03_18_23 = propagar_codigos(
    FL03_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("SCIAN 2007:", sorted(FL03_2007))
print("SCIAN 2013:", sorted(FL03_2013))
print("SCIAN 2018:", sorted(FL03_2018))
print("SCIAN 2023:", sorted(FL03_2023))

print("\nCambios explícitos 2013→2018:")
display(
    detalle_FL03_13_18[
        detalle_FL03_13_18["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2018→2023:")
display(
    detalle_FL03_18_23[
        detalle_FL03_18_23["Cambio_explicito"]
    ]
)

SCIAN 2007: ['531119']
SCIAN 2013: ['531116', '531119']
SCIAN 2018: ['531116', '531119']
SCIAN 2023: ['531116', '531119']

Cambios explícitos 2013→2018:


,Codigo_origen,Cambio_explicito,Codigo_destino



Cambios explícitos 2018→2023:


,Codigo_origen,Cambio_explicito,Codigo_destino


In [189]:
# ============================================================
# PASO 13S
# SERIE HISTÓRICA HOMOLOGADA FL03 - SECTOR 53
# ============================================================

FL03_codigos = {

    "SCIAN 2007": {
        "531119"
    },

    "SCIAN 2013": {
        "531116",
        "531119"
    },

    "SCIAN 2018": {
        "531116",
        "531119"
    },

    "SCIAN 2023": {
        "531116",
        "531119"
    }
}


filas_FL03 = []

for version, codigos in FL03_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL03"
    resumen["Sector"] = "53"

    filas_FL03.append(resumen)


serie_FL03 = pd.concat(
    filas_FL03,
    ignore_index=True
)

serie_FL03 = (
    serie_FL03[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL03)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL03,53,2010,SCIAN 2007,8
1,FL03,53,2011,SCIAN 2007,8
2,FL03,53,2012,SCIAN 2007,9
3,FL03,53,2013,SCIAN 2007,9
4,FL03,53,2015,SCIAN 2013,4
5,FL03,53,2016,SCIAN 2013,5
6,FL03,53,2017,SCIAN 2013,5
7,FL03,53,2018,SCIAN 2018,5
8,FL03,53,2019,SCIAN 2018,1
9,FL03,53,2020,SCIAN 2018,1


In [190]:
# ============================================================
# PASO 13T
# AUDITORÍA DE 531116
# ============================================================

presencia_531116 = inventario_scian[
    inventario_scian["SCIAN_original"]
    .astype(str)
    .str.strip()
    .eq("531116")
]

print(
    "Filas observadas de 531116:",
    len(presencia_531116)
)

print(
    "Establecimientos observados de 531116:",
    presencia_531116["Establecimientos"].sum()
)

print(
    "\nNúmero de años FL03:",
    serie_FL03["Año"].nunique()
)

print(
    "Nulos:",
    serie_FL03.isna().sum().sum()
)

print(
    "Duplicados:",
    serie_FL03.duplicated(
        subset=["Año"]
    ).sum()
)

Filas observadas de 531116: 1
Establecimientos observados de 531116: 1

Número de años FL03: 16
Nulos: 0
Duplicados: 0


In [192]:
# ============================================================
# RECUPERACIÓN COMPLETA FL05
# RECONSTRUIR + PROPAGAR + AUDITAR PDF
# ============================================================

from pathlib import Path
import pandas as pd
import pdfplumber

# Recuperar ruta si hiciera falta
if "HOMOLOGACION" not in globals():
    HOMOLOGACION = Path.cwd() / "03_HOMOLOGACION_SCIAN"


# ------------------------------------------------------------
# 1. CARGAR MAPAS VALIDADOS
# ------------------------------------------------------------

mapa_FL05_13_18 = pd.read_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2013_2018.csv",
    dtype=str
)

mapa_FL05_18_23 = pd.read_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2018_2023.csv",
    dtype=str
)


# ------------------------------------------------------------
# 2. FUNCIÓN LOCAL DE PROPAGACIÓN
# ------------------------------------------------------------

def propagar_FL05(
    codigos_origen,
    mapa,
    col_origen,
    col_destino
):

    resultado = set()
    detalle = []

    for codigo in sorted(codigos_origen):

        filas = mapa[
            mapa[col_origen]
            .astype(str)
            .str.strip()
            .eq(codigo)
        ]

        if filas.empty:

            # No aparece en tabla oficial de cambios:
            # conserva el mismo código
            resultado.add(codigo)

            detalle.append({
                "Codigo_origen": codigo,
                "Cambio_explicito": False,
                "Codigo_destino": codigo
            })

        else:

            destinos = sorted(
                filas[col_destino]
                .astype(str)
                .str.strip()
                .unique()
            )

            resultado.update(destinos)

            for destino in destinos:

                detalle.append({
                    "Codigo_origen": codigo,
                    "Cambio_explicito": True,
                    "Codigo_destino": destino
                })

    return resultado, pd.DataFrame(detalle)


# ------------------------------------------------------------
# 3. DEFINIR FL05
# ------------------------------------------------------------

FL05_2007 = {
    "621112",
    "621114"
}

FL05_2013 = {
    "621112",
    "621114",
    "621116"
}


# ------------------------------------------------------------
# 4. PROPAGAR A 2018 Y 2023
# ------------------------------------------------------------

FL05_2018, detalle_FL05_13_18 = propagar_FL05(
    FL05_2013,
    mapa_FL05_13_18,
    "SCIAN_2013",
    "SCIAN_2018"
)

FL05_2023, detalle_FL05_18_23 = propagar_FL05(
    FL05_2018,
    mapa_FL05_18_23,
    "SCIAN_2018",
    "SCIAN_2023"
)


print("========== PROPAGACIÓN FL05 ==========")

print("SCIAN 2007:", sorted(FL05_2007))
print("SCIAN 2013:", sorted(FL05_2013))
print("SCIAN 2018:", sorted(FL05_2018))
print("SCIAN 2023:", sorted(FL05_2023))


print("\nCambios explícitos 2013→2018:")

display(
    detalle_FL05_13_18[
        detalle_FL05_13_18["Cambio_explicito"]
    ]
)


print("\nCambios explícitos 2018→2023:")

display(
    detalle_FL05_18_23[
        detalle_FL05_18_23["Cambio_explicito"]
    ]
)


# ------------------------------------------------------------
# 5. AUDITAR 621112 DIRECTAMENTE EN PDF OFICIAL 2007-2013
# ------------------------------------------------------------

pdf_2007_2013 = (
    HOMOLOGACION /
    "SCIAN_2013_2007_OFICIAL.pdf"
)

print("\n========== AUDITORÍA PDF 621112 ==========")

with pdfplumber.open(pdf_2007_2013) as pdf:

    for numero_pagina, pagina in enumerate(
        pdf.pages,
        start=1
    ):

        texto = pagina.extract_text() or ""

        if "621112" in texto and "621116" in texto:

            lineas = texto.splitlines()

            for i, linea in enumerate(lineas):

                if (
                    "621112" in linea
                    or "621116" in linea
                ):

                    desde = max(0, i - 1)
                    hasta = min(len(lineas), i + 2)

                    print(
                        f"\nPágina {numero_pagina}"
                    )

                    for j in range(
                        desde,
                        hasta
                    ):
                        print(
                            f"{j:03d}: {lineas[j]}"
                        )

========== PROPAGACIÓN FL05 ==========
SCIAN 2007: ['621112', '621114']
SCIAN 2013: ['621112', '621114', '621116']
SCIAN 2018: ['621112', '621114', '621116']
SCIAN 2023: ['621112', '621114', '621116']

Cambios explícitos 2013→2018:


,Codigo_origen,Cambio_explicito,Codigo_destino



Cambios explícitos 2018→2023:


,Codigo_origen,Cambio_explicito,Codigo_destino



========== AUDITORÍA PDF 621112 ==========

Página 10
008: del sector privado)
009: 621116 Clínicas de consultorios médicos p 2/2 621112 Consultorios de medicina general del sector
010: del sector público público (clínicas de consultorios médicos generales

Página 15
031: del sector privado)
032: 621112 Consultorios de medicina general 621112 Consultorios de medicina general del sector público
033: del sector público p 1/2 621116 Clínicas de consultorios médicos del sector

Página 15
032: 621112 Consultorios de medicina general 621112 Consultorios de medicina general del sector público
033: del sector público p 1/2 621116 Clínicas de consultorios médicos del sector
034: público (clínicas de consultorios médicos generales

Página 15
042: del sector público del sector público
043: p 2/2 621116 Clínicas de consultorios médicos del sector
044: público (clínicas de consultorios médicos especializados


### Resolución de la familia FL05 del sector 62

La familia FL05 corresponde a una reestructuración de actividades de
consultorios y clínicas médicas del sector público.

En SCIAN 2007 la familia se encuentra integrada por las clases 621112 y
621114. En SCIAN 2013 se incorpora la clase 621116, cuyo contenido procede
parcialmente de las actividades clasificadas anteriormente en 621112 y
621114.

La revisión directa de las tablas oficiales confirmó que la correspondencia
621112–621112 no presenta indicador de parcialidad, mientras que las
relaciones que contribuyen a la formación de 621116 se identifican mediante
los indicadores de parte correspondientes.

Las transiciones SCIAN 2013–2018 y SCIAN 2018–2023 no presentan cambios
explícitos adicionales para estas clases.

Por tanto, la familia homologada queda definida como:

- SCIAN 2007: 621112 + 621114.
- SCIAN 2013: 621112 + 621114 + 621116.
- SCIAN 2018: 621112 + 621114 + 621116.
- SCIAN 2023: 621112 + 621114 + 621116.

La homologación se realiza mediante agregación del conjunto completo, sin
distribuir retrospectivamente los establecimientos de la clase 621116.

In [193]:
# ============================================================
# FASE 2 - PASO 14F
# SERIE HISTÓRICA HOMOLOGADA FL05 - SECTOR 62
# ============================================================

FL05_codigos = {

    "SCIAN 2007": {
        "621112",
        "621114"
    },

    "SCIAN 2013": {
        "621112",
        "621114",
        "621116"
    },

    "SCIAN 2018": {
        "621112",
        "621114",
        "621116"
    },

    "SCIAN 2023": {
        "621112",
        "621114",
        "621116"
    }
}

filas_FL05 = []

for version, codigos in FL05_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL05"
    resumen["Sector"] = "62"

    filas_FL05.append(resumen)


serie_FL05 = pd.concat(
    filas_FL05,
    ignore_index=True
)

serie_FL05 = (
    serie_FL05[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL05)

print("\nNúmero de años:", serie_FL05["Año"].nunique())
print("Nulos:", serie_FL05.isna().sum().sum())
print(
    "Duplicados:",
    serie_FL05.duplicated(subset=["Año"]).sum()
)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL05,62,2010,SCIAN 2007,11
1,FL05,62,2011,SCIAN 2007,11
2,FL05,62,2012,SCIAN 2007,11
3,FL05,62,2013,SCIAN 2007,11
4,FL05,62,2015,SCIAN 2013,18
5,FL05,62,2016,SCIAN 2013,19
6,FL05,62,2017,SCIAN 2013,19
7,FL05,62,2018,SCIAN 2018,19
8,FL05,62,2019,SCIAN 2018,30
9,FL05,62,2020,SCIAN 2018,33



Número de años: 16
Nulos: 0
Duplicados: 0


In [194]:
# ============================================================
# PASO 14G
# GUARDAR FL05 Y ACTUALIZAR CONTROL
# ============================================================

serie_FL05.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_FL05_SECTOR62.csv",
    index=False,
    encoding="utf-8-sig"
)

mask = control_familias["Familia"] == "FL05"

control_familias.loc[
    mask,
    "Estado"
] = "RESUELTA"

control_familias.loc[
    mask,
    "Tratamiento"
] = (
    "2007=621112+621114; "
    "2013/2018/2023=621112+621114+621116"
)

control_familias.loc[
    mask,
    "Observacion"
] = (
    "621116 recibe contenido de clases previas; "
    "homologación mediante agregación completa"
)

control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ FL05 sector 62 homologada y guardada.")

print(
    "Resueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente").sum()
)

✓ FL05 sector 62 homologada y guardada.
Resueltas: 9
Pendientes: 3


In [195]:
# ============================================================
# FASE 2 - PASO 15A
# INSPECCIONAR FL08 - SECTOR 81
# ============================================================

fila_FL08 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL08"
    ]
    .iloc[0]
)

print("========== MEMBRESÍA LONGITUDINAL FL08 ==========")

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = "Codigos_" + version.replace(" ", "_")

    print(
        version + ":",
        fila_FL08[columna]
    )

========== MEMBRESÍA LONGITUDINAL FL08 ==========
SCIAN 2007: 812320
SCIAN 2013: 812321 | 812322
SCIAN 2018: 812321 | 812322
SCIAN 2023: 812321 | 812322


### Revisión de la familia FL08: sector 81

La familia longitudinal FL08 pertenece al sector 81, Otros servicios excepto
actividades gubernamentales.

La familia presenta una división de una clase de SCIAN 2007 en dos clases
a partir de SCIAN 2013. Se revisan las correspondencias oficiales y la
presencia empírica de las clases involucradas antes de establecer la regla
definitiva de homologación.

In [196]:
# ============================================================
# FASE 2 - PASO 15B
# CORRESPONDENCIA OFICIAL FL08
# ============================================================

FL08_2007 = {"812320"}
FL08_2013 = {"812321", "812322"}

detalle_FL08 = (
    corr_2007_2013_validada[
        corr_2007_2013_validada["SCIAN_2007"]
        .astype(str).str.strip()
        .isin(FL08_2007)
        |
        corr_2007_2013_validada["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(FL08_2013)
    ]
    [
        [
            "SCIAN_2007",
            "Nombre_2007",
            "Parte",
            "SCIAN_2013",
            "Nombre_2013"
        ]
    ]
    .sort_values(
        ["SCIAN_2007", "SCIAN_2013"]
    )
    .reset_index(drop=True)
)

display(detalle_FL08)

,SCIAN_2007,Nombre_2007,Parte,SCIAN_2013,Nombre_2013
0,812320,Administración de cementerios (administración ...,p 1/2,812321,Administración de cementerios pertenecientes a...
1,812320,Administración de cementerios (administración ...,p 2/2,812322,Administración de cementerios pertenecientes a...


In [197]:
# ============================================================
# PASO 15C
# PROPAGAR FL08 HACIA 2018 Y 2023
# ============================================================

FL08_2018, detalle_FL08_13_18 = propagar_codigos(
    FL08_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

FL08_2023, detalle_FL08_18_23 = propagar_codigos(
    FL08_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("SCIAN 2007:", sorted(FL08_2007))
print("SCIAN 2013:", sorted(FL08_2013))
print("SCIAN 2018:", sorted(FL08_2018))
print("SCIAN 2023:", sorted(FL08_2023))

print("\nCambios explícitos 2013→2018:")
display(
    detalle_FL08_13_18[
        detalle_FL08_13_18["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2018→2023:")
display(
    detalle_FL08_18_23[
        detalle_FL08_18_23["Cambio_explicito"]
    ]
)

SCIAN 2007: ['812320']
SCIAN 2013: ['812321', '812322']
SCIAN 2018: ['812321', '812322']
SCIAN 2023: ['812321', '812322']

Cambios explícitos 2013→2018:


,Codigo_origen,Cambio_explicito,Codigo_destino



Cambios explícitos 2018→2023:


,Codigo_origen,Cambio_explicito,Codigo_destino


In [198]:
# ============================================================
# PASO 15D
# EVOLUCIÓN EMPÍRICA FL08
# ============================================================

FL08_codigos = {
    "SCIAN 2007": {"812320"},
    "SCIAN 2013": {"812321", "812322"},
    "SCIAN 2018": {"812321", "812322"},
    "SCIAN 2023": {"812321", "812322"}
}

filas_FL08 = []

for version, codigos in FL08_codigos.items():

    temp = inventario_scian[
        (
            inventario_scian["Version_SCIAN"] == version
        )
        &
        (
            inventario_scian["SCIAN_original"]
            .astype(str).str.strip()
            .isin(codigos)
        )
    ].copy()

    filas_FL08.append(temp)

evolucion_FL08 = pd.concat(
    filas_FL08,
    ignore_index=True
)

tabla_FL08 = (
    evolucion_FL08.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_FL08)

print("\nPRESENCIA POR CÓDIGO:")

display(
    evolucion_FL08.groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ],
        as_index=False
    )
    .agg(
        Años_observados=("Año", "nunique"),
        Total_establecimientos=("Establecimientos", "sum"),
        Max_establecimientos=("Establecimientos", "max")
    )
)

SCIAN_original,Año,Version_SCIAN,812320,812321,812322
0,2010,SCIAN 2007,9,0,0
1,2011,SCIAN 2007,9,0,0
2,2012,SCIAN 2007,9,0,0
3,2013,SCIAN 2007,9,0,0
4,2015,SCIAN 2013,0,2,4
5,2016,SCIAN 2013,0,3,3
6,2017,SCIAN 2013,0,3,3
7,2018,SCIAN 2018,0,3,3
8,2019,SCIAN 2018,0,2,5
9,2020,SCIAN 2018,0,1,5



PRESENCIA POR CÓDIGO:


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,812320,4,36,9
1,SCIAN 2013,812321,3,8,3
2,SCIAN 2013,812322,3,10,4
3,SCIAN 2018,812321,6,9,3
4,SCIAN 2018,812322,6,28,5
5,SCIAN 2023,812321,3,7,3
6,SCIAN 2023,812322,3,17,6


### Resolución de la familia FL08 del sector 81

La familia FL08 corresponde a una división de la clase 812320 de SCIAN
2007 en las clases 812321 y 812322 a partir de SCIAN 2013.

La propagación de la familia hacia SCIAN 2018 y SCIAN 2023 confirmó que
las clases 812321 y 812322 no presentan cambios explícitos adicionales.

Por tanto, la familia homologada queda definida como:

- SCIAN 2007: 812320.
- SCIAN 2013: 812321 + 812322.
- SCIAN 2018: 812321 + 812322.
- SCIAN 2023: 812321 + 812322.

La homologación se realiza mediante agregación de las clases resultantes,
sin distribuir retrospectivamente los establecimientos de la clase 812320.

In [199]:
# ============================================================
# FASE 2 - PASO 15E
# SERIE HISTÓRICA HOMOLOGADA FL08 - SECTOR 81
# ============================================================

FL08_codigos = {

    "SCIAN 2007": {
        "812320"
    },

    "SCIAN 2013": {
        "812321",
        "812322"
    },

    "SCIAN 2018": {
        "812321",
        "812322"
    },

    "SCIAN 2023": {
        "812321",
        "812322"
    }
}

filas_FL08 = []

for version, codigos in FL08_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL08"
    resumen["Sector"] = "81"

    filas_FL08.append(resumen)


serie_FL08 = pd.concat(
    filas_FL08,
    ignore_index=True
)

serie_FL08 = (
    serie_FL08[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL08)

print("\nNúmero de años:", serie_FL08["Año"].nunique())
print("Nulos:", serie_FL08.isna().sum().sum())
print(
    "Duplicados:",
    serie_FL08.duplicated(subset=["Año"]).sum()
)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL08,81,2010,SCIAN 2007,9
1,FL08,81,2011,SCIAN 2007,9
2,FL08,81,2012,SCIAN 2007,9
3,FL08,81,2013,SCIAN 2007,9
4,FL08,81,2015,SCIAN 2013,6
5,FL08,81,2016,SCIAN 2013,6
6,FL08,81,2017,SCIAN 2013,6
7,FL08,81,2018,SCIAN 2018,6
8,FL08,81,2019,SCIAN 2018,7
9,FL08,81,2020,SCIAN 2018,6



Número de años: 16
Nulos: 0
Duplicados: 0


In [201]:
# ============================================================
# PASO 15F
# GUARDAR FL08 Y ACTUALIZAR CONTROL
# ============================================================

serie_FL08.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_FL08_SECTOR81.csv",
    index=False,
    encoding="utf-8-sig"
)

mask = control_familias["Familia"] == "FL08"

control_familias.loc[
    mask,
    "Estado"
] = "RESUELTA"

control_familias.loc[
    mask,
    "Tratamiento"
] = (
    "2007=812320; "
    "2013/2018/2023=812321+812322"
)

control_familias.loc[
    mask,
    "Observacion"
] = (
    "División 1→2; homologación mediante agregación"
)

control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ FL08 sector 81 homologada y guardada.")

print(
    "Resueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente").sum()
)

✓ FL08 sector 81 homologada y guardada.
Resueltas: 10
Pendientes: 2


In [202]:
# ============================================================
# FASE 2 - PASO 16A
# INSPECCIONAR FL11 - SECTOR 56
# ============================================================

fila_FL11 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL11"
    ]
    .iloc[0]
)

print("========== MEMBRESÍA LONGITUDINAL FL11 ==========")

codigos_FL11_version = {}

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = (
        "Codigos_"
        + version.replace(" ", "_")
    )

    codigos = [
        x.strip()
        for x in str(fila_FL11[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    ]

    codigos_FL11_version[version] = set(codigos)

    print(version + ":", codigos)

========== MEMBRESÍA LONGITUDINAL FL11 ==========
SCIAN 2007: ['562111', '562112']
SCIAN 2013: ['562111', '562112']
SCIAN 2018: ['562111', '562121', '562211', '562221', '562911', '562921', '562998']
SCIAN 2023: ['562111', '562121', '562221', '562911', '562998']


### Revisión de la familia FL11: sector 56

La familia longitudinal FL11 pertenece al sector 56, Servicios de apoyo a
los negocios y manejo de residuos, y servicios de remediación.

La estructura preliminar muestra una expansión importante entre SCIAN 2013
y SCIAN 2018, seguida de una nueva reorganización entre SCIAN 2018 y
SCIAN 2023.

Por esta razón, la familia se reconstruye utilizando directamente las
correspondencias oficiales entre versiones, distinguiendo la estructura
clasificatoria completa de la presencia efectiva de establecimientos en
Morelia.

In [204]:
# ============================================================
# FASE 2 - PASO 16B
# RECONSTRUIR FL11 MEDIANTE LOS MAPAS OFICIALES
# ============================================================

FL11_2013 = {
    "562111",
    "562112"
}

# Retroceder hacia SCIAN 2007
FL11_2007, detalle_FL11_13_07 = retroceder_codigos(
    FL11_2013,
    mapa_2007_2013_clasificado,
    "SCIAN_2007",
    "SCIAN_2013"
)

# Propagar hacia SCIAN 2018
FL11_2018, detalle_FL11_13_18 = propagar_codigos(
    FL11_2013,
    mapa_2013_2018_clasificado,
    "SCIAN_2013",
    "SCIAN_2018"
)

# Propagar hacia SCIAN 2023
FL11_2023, detalle_FL11_18_23 = propagar_codigos(
    FL11_2018,
    mapa_2018_2023_clasificado,
    "SCIAN_2018",
    "SCIAN_2023"
)

print("========== FL11 RECONSTRUIDA ==========")

print("SCIAN 2007:", sorted(FL11_2007))
print("SCIAN 2013:", sorted(FL11_2013))
print("SCIAN 2018:", sorted(FL11_2018))
print("SCIAN 2023:", sorted(FL11_2023))

print("\nCambios explícitos 2007→2013:")
display(
    detalle_FL11_13_07[
        detalle_FL11_13_07["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2013→2018:")
display(
    detalle_FL11_13_18[
        detalle_FL11_13_18["Cambio_explicito"]
    ]
)

print("\nCambios explícitos 2018→2023:")
display(
    detalle_FL11_18_23[
        detalle_FL11_18_23["Cambio_explicito"]
    ]
)

========== FL11 RECONSTRUIDA ==========
SCIAN 2007: ['562111', '562112']
SCIAN 2013: ['562111', '562112']
SCIAN 2018: ['562111', '562121', '562211', '562221', '562911', '562921', '562998']
SCIAN 2023: ['562111', '562121', '562211', '562221', '562911', '562921', '562998']

Cambios explícitos 2007→2013:


,Codigo_destino,Cambio_explicito,Codigo_origen



Cambios explícitos 2013→2018:


,Codigo_origen,Cambio_explicito,Codigo_destino
0,562111,True,562111
1,562111,True,562211
2,562111,True,562911
3,562111,True,562921
4,562111,True,562998
5,562112,True,562121
6,562112,True,562221
7,562112,True,562911
8,562112,True,562921
9,562112,True,562998



Cambios explícitos 2018→2023:


,Codigo_origen,Cambio_explicito,Codigo_destino
4,562911,True,562911


In [205]:
# ============================================================
# PASO 16C
# CORRESPONDENCIAS OFICIALES CON NOMBRES - FL11
# ============================================================

print("========== SCIAN 2013 → 2018 ==========")

corr_FL11_13_18 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2013_2018.csv",
    dtype=str
)

display(
    corr_FL11_13_18[
        corr_FL11_13_18["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(FL11_2013)
        |
        corr_FL11_13_18["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(FL11_2018)
    ]
    [
        [
            "SCIAN_2013",
            "Nombre_2013",
            "Parte",
            "SCIAN_2018",
            "Nombre_2018"
        ]
    ]
    .sort_values(
        ["SCIAN_2013", "SCIAN_2018"]
    )
    .reset_index(drop=True)
)


print("\n========== SCIAN 2018 → 2023 ==========")

corr_FL11_18_23 = pd.read_csv(
    HOMOLOGACION /
    "CORRESPONDENCIA_VALIDADA_SCIAN_2018_2023.csv",
    dtype=str
)

display(
    corr_FL11_18_23[
        corr_FL11_18_23["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(FL11_2018)
        |
        corr_FL11_18_23["SCIAN_2023"]
        .astype(str).str.strip()
        .isin(FL11_2023)
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
    .sort_values(
        ["SCIAN_2018", "SCIAN_2023"]
    )
    .reset_index(drop=True)
)

========== SCIAN 2013 → 2018 ==========


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018
0,562111,Manejo de residuos peligrosos y servicios de,NaN,562111,Recolección de residuos peligrosos por el sect...
1,562111,Manejo de residuos peligrosos y servicios de r...,NaN,562211,Tratamiento y disposición final de residuos pe...
2,562111,Manejo de residuos peligrosos y servicios de r...,p 1/2,562911,Servicios de remediación de residuos por el se...
3,562111,Manejo de residuos peligrosos y servicios de r...,p 1/2,562921,Recuperación de residuos por el sector privado...
4,562111,Manejo de residuos peligrosos y servicios de r...,p 1/2,562998,Otros servicios de manejo de residuos por el s...
5,562112,NaN,NaN,562121,sector privado
6,562112,servicios de remediación a zonas dañadas por d...,NaN,562221,Tratamiento y disposición final de residuos no...
7,562112,servicios de remediación a zonas dañadas por d...,p 2/2,562911,Servicios de remediación de residuos por el se...
8,562112,servicios de remediación a zonas dañadas por d...,p 2/2,562921,Recuperación de residuos por el sector privado...
9,562112,servicios de remediación a zonas dañadas por d...,p 2/2,562998,Otros servicios de manejo de residuos por el s...



========== SCIAN 2018 → 2023 ==========


,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023
0,562911,Servicios de remediación de residuos por el,NaN,562911,Servicios de remediación por el sector privado


In [206]:
# ============================================================
# PASO 16D
# EVOLUCIÓN EMPÍRICA CORRECTA DE FL11
# ============================================================

FL11_codigos = {
    "SCIAN 2007": FL11_2007,
    "SCIAN 2013": FL11_2013,
    "SCIAN 2018": FL11_2018,
    "SCIAN 2023": FL11_2023
}

filas_FL11 = []

for version, codigos in FL11_codigos.items():

    temp = inventario_scian[
        (
            inventario_scian["Version_SCIAN"] == version
        )
        &
        (
            inventario_scian["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        )
    ].copy()

    filas_FL11.append(temp)

evolucion_FL11 = pd.concat(
    filas_FL11,
    ignore_index=True
)

tabla_FL11 = (
    evolucion_FL11.pivot_table(
        index=[
            "Año",
            "Version_SCIAN"
        ],
        columns="SCIAN_original",
        values="Establecimientos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

display(tabla_FL11)

print("\nPRESENCIA POR CÓDIGO:")

display(
    evolucion_FL11.groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ],
        as_index=False
    )
    .agg(
        Años_observados=("Año", "nunique"),
        Total_establecimientos=("Establecimientos", "sum"),
        Max_establecimientos=("Establecimientos", "max")
    )
)

SCIAN_original,Año,Version_SCIAN,562111,562112,562121,562221,562911,562998
0,2010,SCIAN 2007,1,4,0,0,0,0
1,2011,SCIAN 2007,1,5,0,0,0,0
2,2012,SCIAN 2007,1,5,0,0,0,0
3,2013,SCIAN 2007,1,6,0,0,0,0
4,2015,SCIAN 2013,1,6,0,0,0,0
5,2016,SCIAN 2013,2,7,0,0,0,0
6,2017,SCIAN 2013,2,7,0,0,0,0
7,2018,SCIAN 2018,2,0,0,0,0,0
8,2019,SCIAN 2018,0,0,2,2,2,2
9,2020,SCIAN 2018,1,0,2,2,2,2



PRESENCIA POR CÓDIGO:


,Version_SCIAN,SCIAN_original,Años_observados,Total_establecimientos,Max_establecimientos
0,SCIAN 2007,562111,4,4,1
1,SCIAN 2007,562112,4,20,6
2,SCIAN 2013,562111,3,5,2
3,SCIAN 2013,562112,3,20,7
4,SCIAN 2018,562111,5,6,2
5,SCIAN 2018,562121,5,10,2
6,SCIAN 2018,562221,4,8,2
7,SCIAN 2018,562911,5,10,2
8,SCIAN 2018,562998,5,10,2
9,SCIAN 2023,562111,3,3,1


In [207]:
# ============================================================
# FASE 2 - PASO 16E
# CIERRE DE AUDITORÍA OFICIAL FL11
# ============================================================

print("========== FAMILIA RECONSTRUIDA ==========")

print("SCIAN 2007:", sorted(FL11_2007))
print("SCIAN 2013:", sorted(FL11_2013))
print("SCIAN 2018:", sorted(FL11_2018))
print("SCIAN 2023:", sorted(FL11_2023))


print("\n========== OFICIAL SCIAN 2013 → 2018 ==========")

display(
    corr_FL11_13_18[
        corr_FL11_13_18["SCIAN_2013"]
        .astype(str).str.strip()
        .isin(FL11_2013)
        |
        corr_FL11_13_18["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(FL11_2018)
    ]
    [
        [
            "SCIAN_2013",
            "Nombre_2013",
            "Parte",
            "SCIAN_2018",
            "Nombre_2018"
        ]
    ]
    .sort_values(
        ["SCIAN_2013", "SCIAN_2018"]
    )
    .reset_index(drop=True)
)


print("\n========== OFICIAL SCIAN 2018 → 2023 ==========")

display(
    corr_FL11_18_23[
        corr_FL11_18_23["SCIAN_2018"]
        .astype(str).str.strip()
        .isin(FL11_2018)
        |
        corr_FL11_18_23["SCIAN_2023"]
        .astype(str).str.strip()
        .isin(FL11_2023)
    ]
    [
        [
            "SCIAN_2018",
            "Nombre_2018",
            "Parte",
            "SCIAN_2023",
            "Nombre_2023"
        ]
    ]
    .sort_values(
        ["SCIAN_2018", "SCIAN_2023"]
    )
    .reset_index(drop=True)
)

========== FAMILIA RECONSTRUIDA ==========
SCIAN 2007: ['562111', '562112']
SCIAN 2013: ['562111', '562112']
SCIAN 2018: ['562111', '562121', '562211', '562221', '562911', '562921', '562998']
SCIAN 2023: ['562111', '562121', '562211', '562221', '562911', '562921', '562998']

========== OFICIAL SCIAN 2013 → 2018 ==========


,SCIAN_2013,Nombre_2013,Parte,SCIAN_2018,Nombre_2018
0,562111,Manejo de residuos peligrosos y servicios de,NaN,562111,Recolección de residuos peligrosos por el sect...
1,562111,Manejo de residuos peligrosos y servicios de r...,NaN,562211,Tratamiento y disposición final de residuos pe...
2,562111,Manejo de residuos peligrosos y servicios de r...,p 1/2,562911,Servicios de remediación de residuos por el se...
3,562111,Manejo de residuos peligrosos y servicios de r...,p 1/2,562921,Recuperación de residuos por el sector privado...
4,562111,Manejo de residuos peligrosos y servicios de r...,p 1/2,562998,Otros servicios de manejo de residuos por el s...
5,562112,NaN,NaN,562121,sector privado
6,562112,servicios de remediación a zonas dañadas por d...,NaN,562221,Tratamiento y disposición final de residuos no...
7,562112,servicios de remediación a zonas dañadas por d...,p 2/2,562911,Servicios de remediación de residuos por el se...
8,562112,servicios de remediación a zonas dañadas por d...,p 2/2,562921,Recuperación de residuos por el sector privado...
9,562112,servicios de remediación a zonas dañadas por d...,p 2/2,562998,Otros servicios de manejo de residuos por el s...



========== OFICIAL SCIAN 2018 → 2023 ==========


,SCIAN_2018,Nombre_2018,Parte,SCIAN_2023,Nombre_2023
0,562911,Servicios de remediación de residuos por el,NaN,562911,Servicios de remediación por el sector privado


### Resolución de la familia FL11 del sector 56

La familia FL11 corresponde a una reestructuración amplia de las actividades
de manejo de residuos y servicios de remediación.

En SCIAN 2007 y SCIAN 2013, la familia se encuentra representada por las
clases 562111 y 562112.

Entre SCIAN 2013 y SCIAN 2018, el contenido de estas dos clases se
redistribuye entre las clases 562111, 562121, 562211, 562221, 562911,
562921 y 562998. Algunas de las clases resultantes reciben contenido
procedente de ambas clases de origen, por lo que no es posible realizar una
asignación retrospectiva individual sin introducir supuestos arbitrarios.

En SCIAN 2023 se conserva el mismo conjunto de clases que en SCIAN 2018.
La clase 562911 presenta una correspondencia explícita con el mismo código,
sin alterar la composición general de la familia.

Por tanto, la familia homologada se define como:

- SCIAN 2007: 562111 + 562112.
- SCIAN 2013: 562111 + 562112.
- SCIAN 2018: 562111 + 562121 + 562211 + 562221 + 562911 + 562921 + 562998.
- SCIAN 2023: 562111 + 562121 + 562211 + 562221 + 562911 + 562921 + 562998.

La homologación se realiza mediante agregación del componente completo,
sin distribuir retrospectivamente los establecimientos entre las clases
resultantes.

In [208]:
# ============================================================
# FASE 2 - PASO 16F
# SERIE HISTÓRICA HOMOLOGADA FL11 - SECTOR 56
# ============================================================

FL11_codigos = {

    "SCIAN 2007": {
        "562111",
        "562112"
    },

    "SCIAN 2013": {
        "562111",
        "562112"
    },

    "SCIAN 2018": {
        "562111",
        "562121",
        "562211",
        "562221",
        "562911",
        "562921",
        "562998"
    },

    "SCIAN 2023": {
        "562111",
        "562121",
        "562211",
        "562221",
        "562911",
        "562921",
        "562998"
    }
}

filas_FL11 = []

for version, codigos in FL11_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL11"
    resumen["Sector"] = "56"

    filas_FL11.append(resumen)


serie_FL11 = pd.concat(
    filas_FL11,
    ignore_index=True
)

serie_FL11 = (
    serie_FL11[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL11)

print("\nNúmero de años:", serie_FL11["Año"].nunique())
print("Nulos:", serie_FL11.isna().sum().sum())
print(
    "Duplicados:",
    serie_FL11.duplicated(subset=["Año"]).sum()
)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL11,56,2010,SCIAN 2007,5
1,FL11,56,2011,SCIAN 2007,6
2,FL11,56,2012,SCIAN 2007,6
3,FL11,56,2013,SCIAN 2007,7
4,FL11,56,2015,SCIAN 2013,7
5,FL11,56,2016,SCIAN 2013,9
6,FL11,56,2017,SCIAN 2013,9
7,FL11,56,2018,SCIAN 2018,2
8,FL11,56,2019,SCIAN 2018,8
9,FL11,56,2020,SCIAN 2018,9



Número de años: 16
Nulos: 0
Duplicados: 0


In [209]:
# ============================================================
# PASO 16G
# GUARDAR FL11 Y ACTUALIZAR CONTROL
# ============================================================

serie_FL11.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_FL11_SECTOR56.csv",
    index=False,
    encoding="utf-8-sig"
)

mask = control_familias["Familia"] == "FL11"

control_familias.loc[
    mask,
    "Estado"
] = "RESUELTA"

control_familias.loc[
    mask,
    "Tratamiento"
] = (
    "2007/2013=562111+562112; "
    "2018/2023=562111+562121+562211+562221+"
    "562911+562921+562998"
)

control_familias.loc[
    mask,
    "Observacion"
] = (
    "Reestructuración 2→7; homologación mediante "
    "agregación del componente completo"
)

control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ FL11 sector 56 homologada y guardada.")

print(
    "Resueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente").sum()
)

✓ FL11 sector 56 homologada y guardada.
Resueltas: 11
Pendientes: 1


In [210]:
# ============================================================
# FASE 2 - PASO 17A
# INSPECCIONAR FL12 - SECTOR 51
# ============================================================

fila_FL12 = (
    familias_longitudinales_v2[
        familias_longitudinales_v2["Familia"] == "FL12"
    ]
    .iloc[0]
)

print("========== MEMBRESÍA LONGITUDINAL FL12 ==========")

codigos_FL12_version = {}

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:

    columna = "Codigos_" + version.replace(" ", "_")

    codigos = [
        x.strip()
        for x in str(fila_FL12[columna]).split("|")
        if x.strip()
        and x.strip().lower() != "nan"
    ]

    codigos_FL12_version[version] = set(codigos)

    print(version + ":", codigos)

========== MEMBRESÍA LONGITUDINAL FL12 ==========
SCIAN 2007: ['511112', '511121', '511131', '511132', '511210', '515110', '515120', '519110', '519130']
SCIAN 2013: ['511112', '511121', '511131', '511132', '511210', '515110', '515120', '515210', '519110', '519130']
SCIAN 2018: ['511111', '511112', '511121', '511122', '511131', '511132', '511141', '511142', '511191', '511192', '511210', '515110', '515120', '515210', '519110', '519130', '519190']
SCIAN 2023: ['513111', '513112', '513121', '513122', '513131', '513132', '513141', '513142', '513191', '513192', '513210', '516110', '516120', '516210', '519290']


### Revisión de la familia FL12: sector 51

La familia longitudinal FL12 pertenece al sector 51, Información en medios
masivos, y presenta modificaciones sucesivas a través de las distintas
versiones del SCIAN.

Debido al número de clases involucradas y a la modificación de diversos
códigos entre SCIAN 2018 y SCIAN 2023, la familia se reconstruye mediante
una clausura bidireccional de las correspondencias oficiales.

Este procedimiento incorpora todas las clases conectadas directa o
indirectamente con el componente inicial y permite distinguir la estructura
clasificatoria completa de las clases efectivamente observadas en Morelia.

In [213]:
# ============================================================
# FASE 2 - PASO 17B
# CLAUSURA LONGITUDINAL COMPLETA DE FL12
# ============================================================

def cierre_puente(
    conjunto_a,
    conjunto_b,
    mapa,
    col_a,
    col_b
):

    cambio = True

    while cambio:

        cambio = False

        # ----------------------------------------------------
        # A -> B
        # ----------------------------------------------------
        for codigo in list(conjunto_a):

            filas = mapa[
                mapa[col_a]
                .astype(str)
                .str.strip()
                .eq(codigo)
            ]

            if filas.empty:

                # Sin cambio explícito:
                # conservar mismo código
                if codigo not in conjunto_b:
                    conjunto_b.add(codigo)
                    cambio = True

            else:

                destinos = set(
                    filas[col_b]
                    .astype(str)
                    .str.strip()
                )

                nuevos = destinos - conjunto_b

                if nuevos:
                    conjunto_b.update(nuevos)
                    cambio = True

        # ----------------------------------------------------
        # B -> A
        # ----------------------------------------------------
        for codigo in list(conjunto_b):

            filas = mapa[
                mapa[col_b]
                .astype(str)
                .str.strip()
                .eq(codigo)
            ]

            if filas.empty:

                # Sin cambio explícito:
                # conservar mismo código hacia atrás
                if codigo not in conjunto_a:
                    conjunto_a.add(codigo)
                    cambio = True

            else:

                origenes = set(
                    filas[col_a]
                    .astype(str)
                    .str.strip()
                )

                nuevos = origenes - conjunto_a

                if nuevos:
                    conjunto_a.update(nuevos)
                    cambio = True

    return conjunto_a, conjunto_b

In [214]:
# ============================================================
# PASO 17C
# RECONSTRUIR FL12 EN LAS CUATRO VERSIONES
# ============================================================

FL12_2007 = set(
    codigos_FL12_version["SCIAN 2007"]
)

FL12_2013 = set(
    codigos_FL12_version["SCIAN 2013"]
)

FL12_2018 = set(
    codigos_FL12_version["SCIAN 2018"]
)

FL12_2023 = set(
    codigos_FL12_version["SCIAN 2023"]
)


cambio_global = True

while cambio_global:

    antes = (
        len(FL12_2007),
        len(FL12_2013),
        len(FL12_2018),
        len(FL12_2023)
    )

    FL12_2007, FL12_2013 = cierre_puente(
        FL12_2007,
        FL12_2013,
        mapa_2007_2013_clasificado,
        "SCIAN_2007",
        "SCIAN_2013"
    )

    FL12_2013, FL12_2018 = cierre_puente(
        FL12_2013,
        FL12_2018,
        mapa_2013_2018_clasificado,
        "SCIAN_2013",
        "SCIAN_2018"
    )

    FL12_2018, FL12_2023 = cierre_puente(
        FL12_2018,
        FL12_2023,
        mapa_2018_2023_clasificado,
        "SCIAN_2018",
        "SCIAN_2023"
    )

    despues = (
        len(FL12_2007),
        len(FL12_2013),
        len(FL12_2018),
        len(FL12_2023)
    )

    cambio_global = (
        despues != antes
    )


print("========== FL12 COMPLETA ==========")

print(
    "SCIAN 2007:",
    len(FL12_2007),
    sorted(FL12_2007)
)

print(
    "\nSCIAN 2013:",
    len(FL12_2013),
    sorted(FL12_2013)
)

print(
    "\nSCIAN 2018:",
    len(FL12_2018),
    sorted(FL12_2018)
)

print(
    "\nSCIAN 2023:",
    len(FL12_2023),
    sorted(FL12_2023)
)

========== FL12 COMPLETA ==========
SCIAN 2007: 17 ['511111', '511112', '511121', '511122', '511131', '511132', '511141', '511142', '511191', '511192', '511210', '515110', '515120', '515210', '519110', '519130', '519190']

SCIAN 2013: 17 ['511111', '511112', '511121', '511122', '511131', '511132', '511141', '511142', '511191', '511192', '511210', '515110', '515120', '515210', '519110', '519130', '519190']

SCIAN 2018: 17 ['511111', '511112', '511121', '511122', '511131', '511132', '511141', '511142', '511191', '511192', '511210', '515110', '515120', '515210', '519110', '519130', '519190']

SCIAN 2023: 15 ['513111', '513112', '513121', '513122', '513131', '513132', '513141', '513142', '513191', '513192', '513210', '516110', '516120', '516210', '519290']


In [216]:
# ============================================================
# PASO 17D
# VERIFICAR SECTORES DE FL12
# ============================================================

for version, codigos in [
    ("SCIAN 2007", FL12_2007),
    ("SCIAN 2013", FL12_2013),
    ("SCIAN 2018", FL12_2018),
    ("SCIAN 2023", FL12_2023)
]:

    sectores = sorted(
        set(
            codigo[:2]
            for codigo in codigos
        )
    )

    print(
        version,
        "| Clases:",
        len(codigos),
        "| Sectores:",
        sectores
    )

SCIAN 2007 | Clases: 17 | Sectores: ['51']
SCIAN 2013 | Clases: 17 | Sectores: ['51']
SCIAN 2018 | Clases: 17 | Sectores: ['51']
SCIAN 2023 | Clases: 15 | Sectores: ['51']


In [217]:
# ============================================================
# PASO 17E
# COBERTURA EMPÍRICA DE FL12 EN MORELIA
# ============================================================

FL12_por_version = {
    "SCIAN 2007": FL12_2007,
    "SCIAN 2013": FL12_2013,
    "SCIAN 2018": FL12_2018,
    "SCIAN 2023": FL12_2023
}

for version, codigos in FL12_por_version.items():

    observados = set(
        inventario_scian.loc[
            inventario_scian["Version_SCIAN"] == version,
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    observados_familia = (
        codigos & observados
    )

    no_observados = (
        codigos - observados
    )

    print("\n==============================")
    print(version)
    print("==============================")

    print(
        "Clases de la familia:",
        len(codigos)
    )

    print(
        "Clases observadas en Morelia:",
        len(observados_familia)
    )

    print(
        "Clases de la familia no observadas:",
        len(no_observados)
    )

    if no_observados:
        print(
            "No observadas:",
            sorted(no_observados)
        )


SCIAN 2007
Clases de la familia: 17
Clases observadas en Morelia: 12
Clases de la familia no observadas: 5
No observadas: ['511141', '511142', '511191', '511192', '515210']

SCIAN 2013
Clases de la familia: 17
Clases observadas en Morelia: 10
Clases de la familia no observadas: 7
No observadas: ['511111', '511122', '511141', '511142', '511191', '511192', '519190']

SCIAN 2018
Clases de la familia: 17
Clases observadas en Morelia: 13
Clases de la familia no observadas: 4
No observadas: ['511122', '511141', '511142', '511191']

SCIAN 2023
Clases de la familia: 15
Clases observadas en Morelia: 9
Clases de la familia no observadas: 6
No observadas: ['513122', '513141', '513142', '513191', '513192', '519290']


In [218]:
# ============================================================
# PASO 17F
# SERIE AGREGADA PRELIMINAR FL12
# ============================================================

filas_FL12 = []

for version, codigos in FL12_por_version.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL12"
    resumen["Sector"] = "51"

    filas_FL12.append(resumen)


serie_FL12_preliminar = pd.concat(
    filas_FL12,
    ignore_index=True
)

serie_FL12_preliminar = (
    serie_FL12_preliminar[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL12_preliminar)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL12,51,2010,SCIAN 2007,40
1,FL12,51,2011,SCIAN 2007,48
2,FL12,51,2012,SCIAN 2007,52
3,FL12,51,2013,SCIAN 2007,52
4,FL12,51,2015,SCIAN 2013,44
5,FL12,51,2016,SCIAN 2013,45
6,FL12,51,2017,SCIAN 2013,45
7,FL12,51,2018,SCIAN 2018,45
8,FL12,51,2019,SCIAN 2018,34
9,FL12,51,2020,SCIAN 2018,38


### Verificación de cobertura de FL12 en el sector 51

La clausura longitudinal de FL12 produjo un conjunto integrado exclusivamente
por clases del sector 51 en las cuatro versiones del SCIAN.

Antes de definir la unidad homologada definitiva se verifica si este componente
comprende la totalidad de las clases del sector 51 observadas en Morelia o
solamente una parte de ellas.

La comprobación se realiza tanto por número de clases como por número anual
de establecimientos.

In [219]:
# ============================================================
# FASE 2 - PASO 17G
# ¿FL12 REPRESENTA TODO EL SECTOR 51 OBSERVADO?
# ============================================================

FL12_por_version = {
    "SCIAN 2007": FL12_2007,
    "SCIAN 2013": FL12_2013,
    "SCIAN 2018": FL12_2018,
    "SCIAN 2023": FL12_2023
}

for version, familia in FL12_por_version.items():

    observadas_51 = set(
        inventario_scian.loc[
            (
                inventario_scian["Version_SCIAN"] == version
            )
            &
            (
                inventario_scian["Sector"]
                .astype(str)
                .str.strip()
                .eq("51")
            ),
            "SCIAN_original"
        ]
        .astype(str)
        .str.strip()
    )

    print("\n=================================")
    print(version)
    print("=================================")

    print(
        "Clases sector 51 observadas:",
        len(observadas_51)
    )

    print(
        "Clases de FL12:",
        len(familia)
    )

    print(
        "Clases observadas fuera de FL12:",
        len(observadas_51 - familia)
    )

    if observadas_51 - familia:
        print(
            "FUERA DE FL12:",
            sorted(observadas_51 - familia)
        )

    print(
        "Clases FL12 no observadas en Morelia:",
        len(familia - observadas_51)
    )

    if familia - observadas_51:
        print(
            "NO OBSERVADAS:",
            sorted(familia - observadas_51)
        )


SCIAN 2007
Clases sector 51 observadas: 25
Clases de FL12: 17
Clases observadas fuera de FL12: 13
FUERA DE FL12: ['512111', '512113', '512130', '512210', '512290', '517111', '517112', '517210', '517410', '517910', '518210', '519121', '519122']
Clases FL12 no observadas en Morelia: 5
NO OBSERVADAS: ['511141', '511142', '511191', '511192', '515210']

SCIAN 2013
Clases sector 51 observadas: 21
Clases de FL12: 17
Clases observadas fuera de FL12: 11
FUERA DE FL12: ['512112', '512113', '512130', '512210', '512290', '517110', '517210', '517910', '518210', '519121', '519122']
Clases FL12 no observadas en Morelia: 7
NO OBSERVADAS: ['511111', '511122', '511141', '511142', '511191', '511192', '519190']

SCIAN 2018
Clases sector 51 observadas: 25
Clases de FL12: 17
Clases observadas fuera de FL12: 12
FUERA DE FL12: ['512112', '512113', '512130', '512250', '512290', '517311', '517312', '517410', '517910', '518210', '519121', '519122']
Clases FL12 no observadas en Morelia: 4
NO OBSERVADAS: ['511122

In [220]:
# ============================================================
# PASO 17H
# COMPARAR FL12 CONTRA EL TOTAL ANUAL DEL SECTOR 51
# ============================================================

filas = []

for año in sorted(
    inventario_scian["Año"]
    .dropna()
    .astype(int)
    .unique()
):

    temp = inventario_scian[
        inventario_scian["Año"].astype(int) == año
    ].copy()

    version = temp["Version_SCIAN"].iloc[0]

    codigos_fl12 = FL12_por_version[version]

    total_sector_51 = (
        temp[
            temp["Sector"]
            .astype(str)
            .str.strip()
            .eq("51")
        ]["Establecimientos"]
        .sum()
    )

    total_fl12 = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos_fl12)
        ]["Establecimientos"]
        .sum()
    )

    filas.append({
        "Año": año,
        "Version_SCIAN": version,
        "Establecimientos_FL12": total_fl12,
        "Total_sector_51": total_sector_51,
        "Diferencia": total_fl12 - total_sector_51,
        "Coincide_exacto": total_fl12 == total_sector_51
    })


verificacion_FL12_sector51 = pd.DataFrame(filas)

display(verificacion_FL12_sector51)

print(
    "\nCoincidencias exactas:",
    verificacion_FL12_sector51[
        "Coincide_exacto"
    ].sum(),
    "de",
    len(verificacion_FL12_sector51)
)

print(
    "Diferencia absoluta acumulada:",
    verificacion_FL12_sector51[
        "Diferencia"
    ].abs().sum()
)

,Año,Version_SCIAN,Establecimientos_FL12,Total_sector_51,Diferencia,Coincide_exacto
0,2010,SCIAN 2007,40,141,-101,False
1,2011,SCIAN 2007,48,166,-118,False
2,2012,SCIAN 2007,52,174,-122,False
3,2013,SCIAN 2007,52,181,-129,False
4,2015,SCIAN 2013,44,143,-99,False
5,2016,SCIAN 2013,45,182,-137,False
6,2017,SCIAN 2013,45,189,-144,False
7,2018,SCIAN 2018,45,189,-144,False
8,2019,SCIAN 2018,34,136,-102,False
9,2020,SCIAN 2018,38,191,-153,False



Coincidencias exactas: 0 de 16
Diferencia absoluta acumulada: 2089


### Delimitación definitiva de FL12 dentro del sector 51

La familia FL12 no comprende la totalidad del sector 51. La comparación
con las clases y establecimientos observados en Morelia mostró la existencia
de diversas actividades del sector 51 que permanecen fuera del componente.

En consecuencia, FL12 no se homologa al nivel agregado del sector 51.
Se conserva como una subfamilia longitudinal específica integrada únicamente
por las clases conectadas mediante las correspondencias oficiales entre
versiones del SCIAN.

Antes de adoptar esta delimitación definitiva se verifica que el componente
sea cerrado, es decir, que ninguna de sus clases presente correspondencias
oficiales hacia clases externas a la familia.

In [221]:
# ============================================================
# FASE 2 - PASO 17I
# VERIFICAR QUE FL12 SEA UN COMPONENTE CERRADO
# ============================================================

def auditar_cierre(
    mapa,
    conjunto_origen,
    conjunto_destino,
    col_origen,
    col_destino,
    nombre_puente
):

    temp = mapa.copy()

    temp[col_origen] = (
        temp[col_origen].astype(str).str.strip()
    )

    temp[col_destino] = (
        temp[col_destino].astype(str).str.strip()
    )

    # Relaciones que salen desde la familia hacia fuera
    salidas = temp[
        temp[col_origen].isin(conjunto_origen)
        &
        ~temp[col_destino].isin(conjunto_destino)
    ].copy()

    # Relaciones que entran desde fuera hacia la familia
    entradas = temp[
        temp[col_destino].isin(conjunto_destino)
        &
        ~temp[col_origen].isin(conjunto_origen)
    ].copy()

    print("\n======================================")
    print(nombre_puente)
    print("======================================")

    print(
        "Relaciones de la familia hacia códigos externos:",
        len(salidas)
    )

    print(
        "Relaciones desde códigos externos hacia la familia:",
        len(entradas)
    )

    if not salidas.empty:
        print("\nSALIDAS EXTERNAS:")
        display(salidas)

    if not entradas.empty:
        print("\nENTRADAS EXTERNAS:")
        display(entradas)

    return len(salidas), len(entradas)


auditoria_FL12 = []

auditoria_FL12.append(
    auditar_cierre(
        mapa_2007_2013_clasificado,
        FL12_2007,
        FL12_2013,
        "SCIAN_2007",
        "SCIAN_2013",
        "SCIAN 2007 → 2013"
    )
)

auditoria_FL12.append(
    auditar_cierre(
        mapa_2013_2018_clasificado,
        FL12_2013,
        FL12_2018,
        "SCIAN_2013",
        "SCIAN_2018",
        "SCIAN 2013 → 2018"
    )
)

auditoria_FL12.append(
    auditar_cierre(
        mapa_2018_2023_clasificado,
        FL12_2018,
        FL12_2023,
        "SCIAN_2018",
        "SCIAN_2023",
        "SCIAN 2018 → 2023"
    )
)

print("\nAUDITORÍA GLOBAL:")
print(auditoria_FL12)


SCIAN 2007 → 2013
Relaciones de la familia hacia códigos externos: 0
Relaciones desde códigos externos hacia la familia: 0

SCIAN 2013 → 2018
Relaciones de la familia hacia códigos externos: 0
Relaciones desde códigos externos hacia la familia: 0

SCIAN 2018 → 2023
Relaciones de la familia hacia códigos externos: 0
Relaciones desde códigos externos hacia la familia: 0

AUDITORÍA GLOBAL:
[(0, 0), (0, 0), (0, 0)]


In [222]:
# ============================================================
# PASO 17J
# SERIE HISTÓRICA HOMOLOGADA DEFINITIVA FL12
# ============================================================

FL12_codigos = {
    "SCIAN 2007": FL12_2007,
    "SCIAN 2013": FL12_2013,
    "SCIAN 2018": FL12_2018,
    "SCIAN 2023": FL12_2023
}

filas_FL12 = []

for version, codigos in FL12_codigos.items():

    temp = inventario_scian[
        inventario_scian["Version_SCIAN"] == version
    ].copy()

    resumen = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(codigos)
        ]
        .groupby(
            ["Año", "Version_SCIAN"],
            as_index=False
        )["Establecimientos"]
        .sum()
    )

    resumen["Familia"] = "FL12"
    resumen["Sector"] = "51"

    filas_FL12.append(resumen)


serie_FL12 = pd.concat(
    filas_FL12,
    ignore_index=True
)

serie_FL12 = (
    serie_FL12[
        [
            "Familia",
            "Sector",
            "Año",
            "Version_SCIAN",
            "Establecimientos"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

display(serie_FL12)

print("\nNúmero de años:", serie_FL12["Año"].nunique())
print("Nulos:", serie_FL12.isna().sum().sum())
print(
    "Duplicados:",
    serie_FL12.duplicated(subset=["Año"]).sum()
)

,Familia,Sector,Año,Version_SCIAN,Establecimientos
0,FL12,51,2010,SCIAN 2007,40
1,FL12,51,2011,SCIAN 2007,48
2,FL12,51,2012,SCIAN 2007,52
3,FL12,51,2013,SCIAN 2007,52
4,FL12,51,2015,SCIAN 2013,44
5,FL12,51,2016,SCIAN 2013,45
6,FL12,51,2017,SCIAN 2013,45
7,FL12,51,2018,SCIAN 2018,45
8,FL12,51,2019,SCIAN 2018,34
9,FL12,51,2020,SCIAN 2018,38



Número de años: 16
Nulos: 0
Duplicados: 0


In [223]:
# ============================================================
# PASO 17K
# GUARDAR FL12 Y CERRAR LAS 12 FAMILIAS
# ============================================================

serie_FL12.to_csv(
    HOMOLOGACION /
    "SERIE_HOMOLOGADA_FL12_SECTOR51.csv",
    index=False,
    encoding="utf-8-sig"
)

verificacion_FL12_sector51.to_csv(
    HOMOLOGACION /
    "VERIFICACION_FL12_FRENTE_SECTOR51.csv",
    index=False,
    encoding="utf-8-sig"
)

mask = control_familias["Familia"] == "FL12"

control_familias.loc[
    mask,
    "Estado"
] = "RESUELTA"

control_familias.loc[
    mask,
    "Tratamiento"
] = (
    "Subfamilia longitudinal cerrada: "
    "17 clases en SCIAN 2007/2013/2018 "
    "y 15 clases en SCIAN 2023"
)

control_familias.loc[
    mask,
    "Observacion"
] = (
    "No equivale al sector 51 completo; "
    "se conserva como componente homologado específico"
)

control_familias.to_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ FL12 homologada y guardada.")

print(
    "Resueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente")
    .sum()
)

✓ FL12 homologada y guardada.
Resueltas: 12
Pendientes: 0


# Control final de la homologación de correspondencias complejas

Una vez resueltas las familias longitudinales derivadas de las
correspondencias complejas entre versiones del SCIAN, se realiza una
auditoría final de cobertura.

El objetivo es comprobar que todos los componentes complejos identificados
originalmente en los puentes SCIAN 2007–2013, 2013–2018 y 2018–2023 han
sido incorporados a una familia longitudinal con una regla explícita de
tratamiento.

Este control precede a la construcción de la tabla maestra de homologación
para la serie histórica del DENUE.

In [224]:
# ============================================================
# FASE 2 - PASO 18A
# CONTROL GENERAL DE LAS 12 FAMILIAS
# ============================================================

print("Total de familias:", len(control_familias))

print("\nEstados:")
display(
    control_familias["Estado"]
    .value_counts()
    .reset_index()
)

print(
    "\nResueltas:",
    control_familias["Estado"]
    .str.startswith("RESUELTA")
    .sum()
)

print(
    "Pendientes:",
    (control_familias["Estado"] == "Pendiente")
    .sum()
)

display(
    control_familias[
        [
            "Familia",
            "Sectores",
            "Estado",
            "Tratamiento",
            "Observacion"
        ]
    ]
    .sort_values("Familia")
    .reset_index(drop=True)
)

Total de familias: 12

Estados:


,Estado,count
0,RESUELTA,11
1,RESUELTA CON INCERTIDUMBRE,1



Resueltas: 12
Pendientes: 0


,Familia,Sectores,Estado,Tratamiento,Observacion
0,FL01,46,RESUELTA,Homologación a nivel sector 46,Coincidencia exacta 16/16 años; diferencia acu...
1,FL02,52,RESUELTA,Agregación longitudinal por versión: 2007=5224...,Fusión N→1 entre SCIAN 2013 y 2018; sin repart...
2,FL03,53,RESUELTA,2007=531119; 2013/2018/2023=531116+531119,"531116 pertenece a la familia clasificatoria, ..."
3,FL04,62,RESUELTA,Grupo homologado 621111+621113 / 621111+621113...,
4,FL05,62,RESUELTA,2007=621112+621114; 2013/2018/2023=621112+6211...,621116 recibe contenido de clases previas; hom...
5,FL06,72,RESUELTA,Grupo histórico homologado G06,
6,FL07,72,RESUELTA,Grupo histórico homologado G07,
7,FL08,81,RESUELTA,2007=812320; 2013/2018/2023=812321+812322,División 1→2; homologación mediante agregación
8,FL09,33 | 81,RESUELTA CON INCERTIDUMBRE,Intervalo inferior-superior,Transferencia intersectorial; incertidumbre má...
9,FL10,48,RESUELTA,485111 en SCIAN 2007/2013; 485111+485115 en SC...,"485115 pertenece a la familia oficial, pero re..."


In [225]:
# ============================================================
# FASE 2 - PASO 18B
# COBERTURA DE LOS 15 COMPONENTES COMPLEJOS ORIGINALES
# ============================================================

grupos_originales_detectados = set(
    componentes_complejos["Puente"]
    + " | "
    + componentes_complejos["Grupo"]
)

grupos_cubiertos = set()

for texto in familias_longitudinales_v2[
    "Grupos_originales"
].dropna():

    partes = [
        x.strip()
        for x in str(texto).split("||")
        if x.strip()
    ]

    grupos_cubiertos.update(partes)


faltantes = (
    grupos_originales_detectados
    - grupos_cubiertos
)

extras = (
    grupos_cubiertos
    - grupos_originales_detectados
)

print(
    "Componentes complejos originales:",
    len(grupos_originales_detectados)
)

print(
    "Componentes cubiertos por familias:",
    len(
        grupos_originales_detectados
        & grupos_cubiertos
    )
)

print(
    "Componentes faltantes:",
    len(faltantes)
)

print(
    "Componentes extraños/no esperados:",
    len(extras)
)

if faltantes:
    print("\nFALTANTES:")
    for x in sorted(faltantes):
        print(x)

if extras:
    print("\nEXTRAS:")
    for x in sorted(extras):
        print(x)

Componentes complejos originales: 15
Componentes cubiertos por familias: 15
Componentes faltantes: 0
Componentes extraños/no esperados: 0


In [226]:
# ============================================================
# FASE 2 - PASO 18C
# BITÁCORA FINAL DE FAMILIAS COMPLEJAS
# ============================================================

bitacora_familias = control_familias.copy()

bitacora_familias["Cobertura_final"] = (
    bitacora_familias["Estado"]
    .str.startswith("RESUELTA")
)

bitacora_familias.to_csv(
    HOMOLOGACION /
    "BITACORA_FINAL_FAMILIAS_COMPLEJAS.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "✓ Bitácora final guardada."
)

print(
    "✓ Familias cerradas:",
    bitacora_familias[
        "Cobertura_final"
    ].sum(),
    "de",
    len(bitacora_familias)
)

✓ Bitácora final guardada.
✓ Familias cerradas: 12 de 12


In [227]:
# ============================================================
# FASE 2 - PASO 18B
# COBERTURA DE LOS 15 COMPONENTES COMPLEJOS ORIGINALES
# ============================================================

grupos_originales_detectados = set(
    componentes_complejos["Puente"].astype(str).str.strip()
    + " | "
    + componentes_complejos["Grupo"].astype(str).str.strip()
)

grupos_cubiertos = set()

for texto in familias_longitudinales_v2[
    "Grupos_originales"
].dropna():

    partes = [
        x.strip()
        for x in str(texto).split("||")
        if x.strip()
    ]

    grupos_cubiertos.update(partes)


faltantes = (
    grupos_originales_detectados
    - grupos_cubiertos
)

extras = (
    grupos_cubiertos
    - grupos_originales_detectados
)


print(
    "Componentes complejos originales:",
    len(grupos_originales_detectados)
)

print(
    "Componentes cubiertos por familias:",
    len(
        grupos_originales_detectados
        & grupos_cubiertos
    )
)

print(
    "Componentes faltantes:",
    len(faltantes)
)

print(
    "Componentes extraños/no esperados:",
    len(extras)
)


if faltantes:
    print("\nFALTANTES:")
    for x in sorted(faltantes):
        print(x)

if extras:
    print("\nEXTRAS:")
    for x in sorted(extras):
        print(x)

Componentes complejos originales: 15
Componentes cubiertos por familias: 15
Componentes faltantes: 0
Componentes extraños/no esperados: 0


# Tabla maestra de homologación SCIAN 2007–2023

Una vez resueltas las correspondencias complejas, se construye un sistema
único de homologación para todas las clases de comercio y servicios
observadas en Morelia.

La tabla maestra integra las correspondencias oficiales entre versiones
con la continuidad de las clases que no presentan cambios explícitos.

Cada combinación versión SCIAN–código original se asigna a una única
unidad longitudinal de homologación. Las unidades pueden representar una
clase sin cambio, una correspondencia directa, una agregación de varias
clases o una de las familias longitudinales complejas previamente resueltas.

La tabla maestra conserva siempre el código SCIAN original y no modifica
los archivos históricos del DENUE.


In [228]:
# ============================================================
# FASE 2 - PASO 18D
# CONSTRUIR TABLA MAESTRA DE HOMOLOGACIÓN SCIAN 2007–2023
# ============================================================

from collections import defaultdict
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1. RECARGAR INSUMOS DESDE DISCO
# ------------------------------------------------------------

mapa_maestro_07_13 = pd.read_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2007_2013.csv",
    dtype=str
)

mapa_maestro_13_18 = pd.read_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2013_2018.csv",
    dtype=str
)

mapa_maestro_18_23 = pd.read_csv(
    HOMOLOGACION /
    "MAPA_VALIDADO_SCIAN_2018_2023.csv",
    dtype=str
)

codigos_maestros = pd.read_csv(
    HOMOLOGACION /
    "CODIGOS_UNICOS_POR_VERSION_SCIAN.csv",
    dtype=str
)

familias_maestras = pd.read_csv(
    HOMOLOGACION /
    "FAMILIAS_LONGITUDINALES_COMPLEJAS_V2.csv",
    dtype=str
)

control_maestro = pd.read_csv(
    HOMOLOGACION /
    "CONTROL_FAMILIAS_LONGITUDINALES.csv",
    dtype=str
)


# Normalizar
for df in [
    mapa_maestro_07_13,
    mapa_maestro_13_18,
    mapa_maestro_18_23,
    codigos_maestros
]:
    for col in df.columns:
        if "SCIAN" in col:
            df[col] = df[col].astype(str).str.strip()


versiones = [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]


# ------------------------------------------------------------
# 2. CREAR GRAFO MAESTRO
# ------------------------------------------------------------

grafo_maestro = defaultdict(set)
nodos_maestros = set()

def conectar_maestro(a, b):
    grafo_maestro[a].add(b)
    grafo_maestro[b].add(a)
    nodos_maestros.add(a)
    nodos_maestros.add(b)


# Añadir todos los códigos observados como nodos
for _, row in codigos_maestros.iterrows():

    nodo = (
        str(row["Version_SCIAN"]).strip()
        + "::"
        + str(row["SCIAN_original"]).strip()
    )

    nodos_maestros.add(nodo)


# ------------------------------------------------------------
# 3. AGREGAR TODAS LAS CORRESPONDENCIAS OFICIALES
# ------------------------------------------------------------

def agregar_puente_maestro(
    mapa,
    version_a,
    version_b,
    col_a,
    col_b
):

    for _, row in mapa.iterrows():

        ca = str(row[col_a]).strip()
        cb = str(row[col_b]).strip()

        conectar_maestro(
            f"{version_a}::{ca}",
            f"{version_b}::{cb}"
        )


agregar_puente_maestro(
    mapa_maestro_07_13,
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN_2007",
    "SCIAN_2013"
)

agregar_puente_maestro(
    mapa_maestro_13_18,
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN_2013",
    "SCIAN_2018"
)

agregar_puente_maestro(
    mapa_maestro_18_23,
    "SCIAN 2018",
    "SCIAN 2023",
    "SCIAN_2018",
    "SCIAN_2023"
)


# ------------------------------------------------------------
# 4. CÓDIGOS DISPONIBLES POR VERSIÓN
#    Incluye observados + códigos de tablas oficiales
# ------------------------------------------------------------

codigos_version = {
    version: set()
    for version in versiones
}

for nodo in nodos_maestros:

    version, codigo = nodo.split("::")

    if version in codigos_version:
        codigos_version[version].add(codigo)


# ------------------------------------------------------------
# 5. AGREGAR CONTINUIDAD DE CÓDIGOS SIN CAMBIO EXPLÍCITO
# ------------------------------------------------------------

def continuidad_maestra(
    mapa,
    version_a,
    version_b,
    col_a,
    col_b
):

    explicitos_a = set(
        mapa[col_a].astype(str).str.strip()
    )

    explicitos_b = set(
        mapa[col_b].astype(str).str.strip()
    )

    universo = (
        codigos_version[version_a]
        |
        codigos_version[version_b]
    )

    agregadas = 0

    for codigo in universo:

        # Sólo enlazamos automáticamente cuando el código
        # no participa en una modificación oficial.
        if (
            codigo not in explicitos_a
            and codigo not in explicitos_b
        ):

            nodo_a = f"{version_a}::{codigo}"
            nodo_b = f"{version_b}::{codigo}"

            conectar_maestro(
                nodo_a,
                nodo_b
            )

            codigos_version[version_a].add(codigo)
            codigos_version[version_b].add(codigo)

            agregadas += 1

    return agregadas


cont_07_13 = continuidad_maestra(
    mapa_maestro_07_13,
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN_2007",
    "SCIAN_2013"
)

cont_13_18 = continuidad_maestra(
    mapa_maestro_13_18,
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN_2013",
    "SCIAN_2018"
)

cont_18_23 = continuidad_maestra(
    mapa_maestro_18_23,
    "SCIAN 2018",
    "SCIAN 2023",
    "SCIAN_2018",
    "SCIAN_2023"
)

print("Continuidades maestras añadidas:")
print("2007→2013:", cont_07_13)
print("2013→2018:", cont_13_18)
print("2018→2023:", cont_18_23)


# ------------------------------------------------------------
# 6. OBTENER COMPONENTES CONECTADOS
# ------------------------------------------------------------

visitados = set()
componentes = []

for inicio in sorted(nodos_maestros):

    if inicio in visitados:
        continue

    pila = [inicio]
    componente = set()

    while pila:

        nodo = pila.pop()

        if nodo in componente:
            continue

        componente.add(nodo)
        visitados.add(nodo)

        for vecino in grafo_maestro[nodo]:

            if vecino not in componente:
                pila.append(vecino)

    componentes.append(componente)


print(
    "\nComponentes longitudinales totales:",
    len(componentes)
)


# ------------------------------------------------------------
# 7. MAPEAR LOS NODOS DE LAS FAMILIAS FL01–FL12
# ------------------------------------------------------------

nodo_a_familia = {}

for _, fam in familias_maestras.iterrows():

    nombre_familia = fam["Familia"]

    for version in versiones:

        columna = (
            "Codigos_"
            + version.replace(" ", "_")
        )

        if columna not in fam.index:
            continue

        texto = str(fam[columna])

        codigos = [
            x.strip()
            for x in texto.split("|")
            if x.strip()
            and x.strip().lower() != "nan"
        ]

        for codigo in codigos:

            nodo_a_familia[
                f"{version}::{codigo}"
            ] = nombre_familia


# ------------------------------------------------------------
# 8. ASIGNAR ID ESTABLE A CADA COMPONENTE
# ------------------------------------------------------------

componentes_ordenados = sorted(
    componentes,
    key=lambda comp: sorted(comp)[0]
)

nodo_a_unidad = {}
resumen_unidades = []

contador_simple = 1

for componente in componentes_ordenados:

    familias_en_componente = sorted(
        {
            nodo_a_familia[nodo]
            for nodo in componente
            if nodo in nodo_a_familia
        }
    )

    # Si toca una familia compleja ya resuelta,
    # usamos su FL como identificador.
    if len(familias_en_componente) == 1:

        unidad = familias_en_componente[0]
        familia = familias_en_componente[0]
        tipo_base = "Familia compleja resuelta"

    elif len(familias_en_componente) > 1:

        # Esto NO debería ocurrir.
        unidad = (
            "ERROR_FAMILIAS_"
            + "_".join(familias_en_componente)
        )
        familia = " | ".join(familias_en_componente)
        tipo_base = "REVISAR"

    else:

        unidad = f"UH{contador_simple:04d}"
        familia = ""
        contador_simple += 1

        # Clasificar estructura simple
        por_version = {}

        for version in versiones:

            codigos = {
                nodo.split("::")[1]
                for nodo in componente
                if nodo.startswith(version + "::")
            }

            por_version[version] = codigos

        tamaños = [
            len(x)
            for x in por_version.values()
            if len(x) > 0
        ]

        codigos_distintos = {
            nodo.split("::")[1]
            for nodo in componente
        }

        if len(codigos_distintos) == 1:

            tipo_base = (
                "Continuidad sin cambio explícito"
            )

        elif tamaños and max(tamaños) == 1:

            tipo_base = (
                "Correspondencia directa 1 a 1"
            )

        else:

            tipo_base = (
                "Agregación directa"
            )

    for nodo in componente:
        nodo_a_unidad[nodo] = unidad

    resumen = {
        "Unidad_homologada": unidad,
        "Familia": familia,
        "Tipo_unidad": tipo_base,
        "Total_nodos": len(componente)
    }

    sectores = set()

    for version in versiones:

        codigos = sorted(
            nodo.split("::")[1]
            for nodo in componente
            if nodo.startswith(version + "::")
        )

        resumen[
            "N_" + version.replace(" ", "_")
        ] = len(codigos)

        resumen[
            "Codigos_" + version.replace(" ", "_")
        ] = " | ".join(codigos)

        for codigo in codigos:
            sectores.add(codigo[:2])

    resumen["Sectores_componente"] = (
        " | ".join(sorted(sectores))
    )

    resumen_unidades.append(resumen)


resumen_unidades = pd.DataFrame(
    resumen_unidades
)


# ------------------------------------------------------------
# 9. CREAR TABLA MAESTRA SOLO PARA LOS CÓDIGOS OBSERVADOS
# ------------------------------------------------------------

tabla_maestra_homologacion = (
    codigos_maestros.copy()
)

tabla_maestra_homologacion["Nodo"] = (
    tabla_maestra_homologacion["Version_SCIAN"]
    .astype(str).str.strip()
    + "::"
    + tabla_maestra_homologacion["SCIAN_original"]
    .astype(str).str.strip()
)

tabla_maestra_homologacion[
    "Unidad_homologada"
] = (
    tabla_maestra_homologacion["Nodo"]
    .map(nodo_a_unidad)
)

tabla_maestra_homologacion = (
    tabla_maestra_homologacion.merge(
        resumen_unidades[
            [
                "Unidad_homologada",
                "Familia",
                "Tipo_unidad",
                "Sectores_componente"
            ]
        ],
        on="Unidad_homologada",
        how="left"
    )
)


# ------------------------------------------------------------
# 10. AUDITORÍA FINAL DE COBERTURA
# ------------------------------------------------------------

print("\n========== CONTROL TABLA MAESTRA ==========")

print(
    "Filas versión-código observadas:",
    len(tabla_maestra_homologacion)
)

print(
    "Unidades homologadas:",
    tabla_maestra_homologacion[
        "Unidad_homologada"
    ].nunique()
)

print(
    "Códigos sin unidad homologada:",
    tabla_maestra_homologacion[
        "Unidad_homologada"
    ].isna().sum()
)

print(
    "Duplicados versión-código:",
    tabla_maestra_homologacion.duplicated(
        subset=[
            "Version_SCIAN",
            "SCIAN_original"
        ]
    ).sum()
)

print(
    "Componentes con más de una FL:",
    resumen_unidades[
        "Unidad_homologada"
        .replace("Unidad_homologada", "Unidad_homologada")
    ]
    .astype(str)
    .str.startswith("ERROR_FAMILIAS_")
    .sum()
)


print("\n========== TIPOS DE UNIDAD ==========")

display(
    tabla_maestra_homologacion[
        [
            "Unidad_homologada",
            "Tipo_unidad"
        ]
    ]
    .drop_duplicates()
    ["Tipo_unidad"]
    .value_counts()
    .reset_index()
)


print("\n========== FAMILIAS COMPLEJAS PRESENTES ==========")

familias_presentes = sorted(
    tabla_maestra_homologacion.loc[
        tabla_maestra_homologacion[
            "Familia"
        ].notna()
        &
        tabla_maestra_homologacion[
            "Familia"
        ].ne(""),
        "Familia"
    ]
    .unique()
)

print(familias_presentes)
print(
    "Número de familias:",
    len(familias_presentes)
)

Continuidades maestras añadidas:
2007→2013: 429
2013→2018: 500
2018→2023: 415

Componentes longitudinales totales: 404

========== CONTROL TABLA MAESTRA ==========
Filas versión-código observadas: 1669
Unidades homologadas: 350
Códigos sin unidad homologada: 0
Duplicados versión-código: 0
Componentes con más de una FL: 0

========== TIPOS DE UNIDAD ==========


,Tipo_unidad,count
0,Continuidad sin cambio explícito,328
1,Familia compleja resuelta,12
2,Correspondencia directa 1 a 1,6
3,Agregación directa,4



========== FAMILIAS COMPLEJAS PRESENTES ==========
['FL01', 'FL02', 'FL03', 'FL04', 'FL05', 'FL06', 'FL07', 'FL08', 'FL09', 'FL10', 'FL11', 'FL12']
Número de familias: 12


In [229]:
# ============================================================
# FASE 2 - PASO 18E
# GUARDAR TABLA MAESTRA DE HOMOLOGACIÓN
# ============================================================

tabla_maestra_homologacion.to_csv(
    HOMOLOGACION /
    "TABLA_MAESTRA_HOMOLOGACION_SCIAN_2007_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_unidades.to_csv(
    HOMOLOGACION /
    "RESUMEN_UNIDADES_HOMOLOGADAS_SCIAN_2007_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Tabla maestra guardada.")
print("✓ Resumen de unidades homologadas guardado.")
print("Unidades homologadas:", tabla_maestra_homologacion["Unidad_homologada"].nunique())

✓ Tabla maestra guardada.
✓ Resumen de unidades homologadas guardado.
Unidades homologadas: 350


In [230]:
# ============================================================
# FASE 2 - PASO 18F
# INCORPORAR REGLAS DE TRATAMIENTO A LA TABLA MAESTRA
# ============================================================

tabla_maestra_homologacion["Regla_calculo"] = "SUMA_DIRECTA"
tabla_maestra_homologacion["Observacion_metodologica"] = ""

# FL09: incertidumbre intersectorial
mask_fl09 = (
    tabla_maestra_homologacion["Unidad_homologada"] == "FL09"
)

tabla_maestra_homologacion.loc[
    mask_fl09,
    "Regla_calculo"
] = "INTERVALO_INFERIOR_SUPERIOR"

tabla_maestra_homologacion.loc[
    mask_fl09,
    "Observacion_metodologica"
] = (
    "811123 presenta transferencia intersectorial 33-81; "
    "reportar límite inferior y superior"
)

# FL01: dejar explícito que su unidad comparable es el sector 46
mask_fl01 = (
    tabla_maestra_homologacion["Unidad_homologada"] == "FL01"
)

tabla_maestra_homologacion.loc[
    mask_fl01,
    "Observacion_metodologica"
] = (
    "Familia equivalente al sector 46 observado; "
    "validación exacta 16/16 años"
)

tabla_maestra_homologacion.to_csv(
    HOMOLOGACION /
    "TABLA_MAESTRA_HOMOLOGACION_SCIAN_2007_2023.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    tabla_maestra_homologacion[
        tabla_maestra_homologacion["Unidad_homologada"]
        .isin(["FL01", "FL09"])
    ]
    [
        [
            "Version_SCIAN",
            "SCIAN_original",
            "Unidad_homologada",
            "Tipo_unidad",
            "Regla_calculo",
            "Observacion_metodologica"
        ]
    ]
    .sort_values(
        ["Unidad_homologada", "Version_SCIAN", "SCIAN_original"]
    )
)

,Version_SCIAN,SCIAN_original,Unidad_homologada,Tipo_unidad,Regla_calculo,Observacion_metodologica
0,SCIAN 2007,461110,FL01,Familia compleja resuelta,SUMA_DIRECTA,Familia equivalente al sector 46 observado; va...
1,SCIAN 2007,461121,FL01,Familia compleja resuelta,SUMA_DIRECTA,Familia equivalente al sector 46 observado; va...
2,SCIAN 2007,461122,FL01,Familia compleja resuelta,SUMA_DIRECTA,Familia equivalente al sector 46 observado; va...
3,SCIAN 2007,461123,FL01,Familia compleja resuelta,SUMA_DIRECTA,Familia equivalente al sector 46 observado; va...
4,SCIAN 2007,461130,FL01,Familia compleja resuelta,SUMA_DIRECTA,Familia equivalente al sector 46 observado; va...
...,...,...,...,...,...,...
1214,SCIAN 2018,811123,FL09,Familia compleja resuelta,INTERVALO_INFERIOR_SUPERIOR,811123 presenta transferencia intersectorial 3...
1215,SCIAN 2018,811129,FL09,Familia compleja resuelta,INTERVALO_INFERIOR_SUPERIOR,811123 presenta transferencia intersectorial 3...
1633,SCIAN 2023,811121,FL09,Familia compleja resuelta,INTERVALO_INFERIOR_SUPERIOR,811123 presenta transferencia intersectorial 3...
1635,SCIAN 2023,811123,FL09,Familia compleja resuelta,INTERVALO_INFERIOR_SUPERIOR,811123 presenta transferencia intersectorial 3...


In [231]:
# ============================================================
# FASE 2 - PASO 18G
# APLICAR TABLA MAESTRA AL INVENTARIO HISTÓRICO
# ============================================================

inventario_homologado = inventario_scian.copy()

inventario_homologado["Version_SCIAN"] = (
    inventario_homologado["Version_SCIAN"]
    .astype(str)
    .str.strip()
)

inventario_homologado["SCIAN_original"] = (
    inventario_homologado["SCIAN_original"]
    .astype(str)
    .str.strip()
)

diccionario_homologacion = (
    tabla_maestra_homologacion[
        [
            "Version_SCIAN",
            "SCIAN_original",
            "Unidad_homologada",
            "Familia",
            "Tipo_unidad",
            "Sectores_componente",
            "Regla_calculo",
            "Observacion_metodologica"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
)

inventario_homologado = (
    inventario_homologado.merge(
        diccionario_homologacion,
        on=[
            "Version_SCIAN",
            "SCIAN_original"
        ],
        how="left",
        validate="many_to_one"
    )
)

print("========== CONTROL DE APLICACIÓN ==========")

print(
    "Registros originales:",
    len(inventario_scian)
)

print(
    "Registros homologados:",
    len(inventario_homologado)
)

print(
    "Registros sin unidad homologada:",
    inventario_homologado[
        "Unidad_homologada"
    ].isna().sum()
)

print(
    "Años:",
    inventario_homologado["Año"]
    .nunique()
)

print(
    "Unidades homologadas utilizadas:",
    inventario_homologado[
        "Unidad_homologada"
    ].nunique()
)

========== CONTROL DE APLICACIÓN ==========
Registros originales: 6435
Registros homologados: 6435
Registros sin unidad homologada: 0
Años: 16
Unidades homologadas utilizadas: 350


In [232]:
# ============================================================
# PASO 18H
# CONTROL DE CONSERVACIÓN DE ESTABLECIMIENTOS
# ============================================================

antes = (
    inventario_scian
    .groupby("Año", as_index=False)["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos": "Total_antes"
        }
    )
)

despues = (
    inventario_homologado
    .groupby("Año", as_index=False)["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos": "Total_despues"
        }
    )
)

control_conservacion = antes.merge(
    despues,
    on="Año",
    how="outer"
)

control_conservacion["Diferencia"] = (
    control_conservacion["Total_despues"]
    - control_conservacion["Total_antes"]
)

control_conservacion["Coincide"] = (
    control_conservacion["Diferencia"] == 0
)

display(control_conservacion)

print(
    "\nAños con conservación exacta:",
    control_conservacion["Coincide"].sum(),
    "de",
    len(control_conservacion)
)

print(
    "Diferencia absoluta total:",
    control_conservacion["Diferencia"].abs().sum()
)

,Año,Total_antes,Total_despues,Diferencia,Coincide
0,2010,34656,34656,0,True
1,2011,35093,35093,0,True
2,2012,35289,35289,0,True
3,2013,35797,35797,0,True
4,2015,37677,37677,0,True
5,2016,38372,38372,0,True
6,2017,38536,38536,0,True
7,2018,38728,38728,0,True
8,2019,40484,40484,0,True
9,2020,41448,41448,0,True



Años con conservación exacta: 16 de 16
Diferencia absoluta total: 0


## Verificación de cobertura temporal de las unidades homologadas

La conservación exacta del número de establecimientos confirma que la
aplicación del diccionario de homologación no genera pérdidas ni
duplicaciones de registros.

Antes de construir el panel longitudinal, se verifica que cada unidad
homologada disponga de una representación clasificatoria en las distintas
versiones del SCIAN.

Esta distinción es necesaria porque la ausencia de establecimientos
observados en una unidad durante un año puede interpretarse como un valor
cero únicamente cuando dicha unidad cuenta con una correspondencia
clasificatoria válida para la versión SCIAN correspondiente.

In [233]:
# ============================================================
# FASE 2 - PASO 18I
# COBERTURA TEMPORAL DE LAS 350 UNIDADES HOMOLOGADAS
# ============================================================

unidades_utilizadas = set(
    tabla_maestra_homologacion[
        "Unidad_homologada"
    ].dropna().unique()
)

cobertura_unidades = (
    resumen_unidades[
        resumen_unidades["Unidad_homologada"]
        .isin(unidades_utilizadas)
    ]
    .copy()
)

columnas_n = [
    "N_SCIAN_2007",
    "N_SCIAN_2013",
    "N_SCIAN_2018",
    "N_SCIAN_2023"
]

for col in columnas_n:
    cobertura_unidades[col] = pd.to_numeric(
        cobertura_unidades[col],
        errors="coerce"
    ).fillna(0).astype(int)


cobertura_unidades["Versiones_con_representacion"] = (
    (cobertura_unidades["N_SCIAN_2007"] > 0).astype(int)
    +
    (cobertura_unidades["N_SCIAN_2013"] > 0).astype(int)
    +
    (cobertura_unidades["N_SCIAN_2018"] > 0).astype(int)
    +
    (cobertura_unidades["N_SCIAN_2023"] > 0).astype(int)
)

cobertura_unidades["Cobertura_4_versiones"] = (
    cobertura_unidades[
        "Versiones_con_representacion"
    ] == 4
)

print("Unidades homologadas utilizadas:", len(cobertura_unidades))

print(
    "Con representación en las 4 versiones:",
    cobertura_unidades["Cobertura_4_versiones"].sum()
)

print(
    "Sin representación en alguna versión:",
    (~cobertura_unidades["Cobertura_4_versiones"]).sum()
)

print("\nDISTRIBUCIÓN:")
display(
    cobertura_unidades[
        "Versiones_con_representacion"
    ]
    .value_counts()
    .sort_index(ascending=False)
    .reset_index()
)

print("\nUNIDADES QUE NO CUBREN LAS 4 VERSIONES:")

display(
    cobertura_unidades[
        ~cobertura_unidades["Cobertura_4_versiones"]
    ]
    [
        [
            "Unidad_homologada",
            "Familia",
            "Tipo_unidad",
            "Sectores_componente",
            "N_SCIAN_2007",
            "N_SCIAN_2013",
            "N_SCIAN_2018",
            "N_SCIAN_2023",
            "Versiones_con_representacion"
        ]
    ]
    .sort_values(
        [
            "Versiones_con_representacion",
            "Unidad_homologada"
        ]
    )
    .reset_index(drop=True)
)

Unidades homologadas utilizadas: 350
Con representación en las 4 versiones: 329
Sin representación en alguna versión: 21

DISTRIBUCIÓN:


,Versiones_con_representacion,count
0,4,329
1,3,17
2,2,4



UNIDADES QUE NO CUBREN LAS 4 VERSIONES:


,Unidad_homologada,Familia,Tipo_unidad,Sectores_componente,N_SCIAN_2007,N_SCIAN_2013,N_SCIAN_2018,N_SCIAN_2023,Versiones_con_representacion
0,UH0389,,Continuidad sin cambio explícito,48,0,0,1,1,2
1,UH0390,,Continuidad sin cambio explícito,54,0,0,1,1,2
2,UH0391,,Continuidad sin cambio explícito,71,0,0,1,1,2
3,UH0392,,Continuidad sin cambio explícito,71,0,0,1,1,2
4,UH0370,,Continuidad sin cambio explícito,48,0,1,1,1,3
5,UH0371,,Continuidad sin cambio explícito,52,0,1,1,1,3
6,UH0372,,Continuidad sin cambio explícito,53,0,1,1,1,3
7,UH0373,,Continuidad sin cambio explícito,53,0,1,1,1,3
8,UH0374,,Continuidad sin cambio explícito,53,0,1,1,1,3
9,UH0375,,Continuidad sin cambio explícito,54,0,1,1,1,3


In [234]:
# ============================================================
# PASO 18J
# CONTROL FINAL DEL MERGE 6,435 REGISTROS
# ============================================================

print("Registros inventario original:", len(inventario_scian))
print("Registros inventario homologado:", len(inventario_homologado))

print(
    "Sin unidad homologada:",
    inventario_homologado["Unidad_homologada"]
    .isna().sum()
)

print(
    "Años:",
    inventario_homologado["Año"].nunique()
)

print(
    "Unidades homologadas utilizadas:",
    inventario_homologado[
        "Unidad_homologada"
    ].nunique()
)

Registros inventario original: 6435
Registros inventario homologado: 6435
Sin unidad homologada: 0
Años: 16
Unidades homologadas utilizadas: 350


In [235]:
# ============================================================
# FASE 2 - PASO 18I
# COBERTURA TEMPORAL DE LAS 350 UNIDADES HOMOLOGADAS
# ============================================================

unidades_utilizadas = set(
    tabla_maestra_homologacion[
        "Unidad_homologada"
    ].dropna().unique()
)

cobertura_unidades = (
    resumen_unidades[
        resumen_unidades["Unidad_homologada"]
        .isin(unidades_utilizadas)
    ]
    .copy()
)

columnas_n = [
    "N_SCIAN_2007",
    "N_SCIAN_2013",
    "N_SCIAN_2018",
    "N_SCIAN_2023"
]

for col in columnas_n:
    cobertura_unidades[col] = pd.to_numeric(
        cobertura_unidades[col],
        errors="coerce"
    ).fillna(0).astype(int)

cobertura_unidades["Versiones_con_representacion"] = (
    (cobertura_unidades["N_SCIAN_2007"] > 0).astype(int)
    +
    (cobertura_unidades["N_SCIAN_2013"] > 0).astype(int)
    +
    (cobertura_unidades["N_SCIAN_2018"] > 0).astype(int)
    +
    (cobertura_unidades["N_SCIAN_2023"] > 0).astype(int)
)

cobertura_unidades["Cobertura_4_versiones"] = (
    cobertura_unidades["Versiones_con_representacion"] == 4
)

print(
    "Unidades homologadas utilizadas:",
    len(cobertura_unidades)
)

print(
    "Con representación en las 4 versiones:",
    cobertura_unidades["Cobertura_4_versiones"].sum()
)

print(
    "Sin representación en alguna versión:",
    (~cobertura_unidades["Cobertura_4_versiones"]).sum()
)

print("\nDISTRIBUCIÓN:")
display(
    cobertura_unidades[
        "Versiones_con_representacion"
    ]
    .value_counts()
    .sort_index(ascending=False)
    .reset_index()
)

print("\nUNIDADES QUE NO CUBREN LAS 4 VERSIONES:")

display(
    cobertura_unidades[
        ~cobertura_unidades["Cobertura_4_versiones"]
    ]
    [
        [
            "Unidad_homologada",
            "Familia",
            "Tipo_unidad",
            "Sectores_componente",
            "N_SCIAN_2007",
            "N_SCIAN_2013",
            "N_SCIAN_2018",
            "N_SCIAN_2023",
            "Versiones_con_representacion"
        ]
    ]
    .sort_values(
        [
            "Versiones_con_representacion",
            "Unidad_homologada"
        ]
    )
    .reset_index(drop=True)
)

Unidades homologadas utilizadas: 350
Con representación en las 4 versiones: 329
Sin representación en alguna versión: 21

DISTRIBUCIÓN:


,Versiones_con_representacion,count
0,4,329
1,3,17
2,2,4



UNIDADES QUE NO CUBREN LAS 4 VERSIONES:


,Unidad_homologada,Familia,Tipo_unidad,Sectores_componente,N_SCIAN_2007,N_SCIAN_2013,N_SCIAN_2018,N_SCIAN_2023,Versiones_con_representacion
0,UH0389,,Continuidad sin cambio explícito,48,0,0,1,1,2
1,UH0390,,Continuidad sin cambio explícito,54,0,0,1,1,2
2,UH0391,,Continuidad sin cambio explícito,71,0,0,1,1,2
3,UH0392,,Continuidad sin cambio explícito,71,0,0,1,1,2
4,UH0370,,Continuidad sin cambio explícito,48,0,1,1,1,3
5,UH0371,,Continuidad sin cambio explícito,52,0,1,1,1,3
6,UH0372,,Continuidad sin cambio explícito,53,0,1,1,1,3
7,UH0373,,Continuidad sin cambio explícito,53,0,1,1,1,3
8,UH0374,,Continuidad sin cambio explícito,53,0,1,1,1,3
9,UH0375,,Continuidad sin cambio explícito,54,0,1,1,1,3


## Auditoría de unidades con cobertura clasificatoria incompleta

De las 350 unidades homologadas construidas, 329 cuentan con representación
en las cuatro versiones del SCIAN utilizadas en la serie histórica.

Las 21 unidades restantes no se interpretan automáticamente como actividades
inexistentes en las versiones faltantes. La ausencia de un nodo en el grafo
puede deberse a que la clase no fue observada en Morelia o a que no aparece
en las tablas comparativas por no haber presentado un cambio explícito.

Por esta razón, estas unidades se contrastan con la estructura completa de
cada versión oficial del SCIAN antes de asignar valores cero o considerarlas
no definidas para determinados periodos.

In [238]:
# ============================================================
# FASE 2 - PASO 18K
# IDENTIFICAR LAS 21 UNIDADES CON COBERTURA INCOMPLETA
# ============================================================

unidades_incompletas = (
    cobertura_unidades[
        ~cobertura_unidades["Cobertura_4_versiones"]
    ]
    .copy()
)

versiones = [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]


def versiones_faltantes(row):

    faltan = []

    if row["N_SCIAN_2007"] == 0:
        faltan.append("SCIAN 2007")

    if row["N_SCIAN_2013"] == 0:
        faltan.append("SCIAN 2013")

    if row["N_SCIAN_2018"] == 0:
        faltan.append("SCIAN 2018")

    if row["N_SCIAN_2023"] == 0:
        faltan.append("SCIAN 2023")

    return " | ".join(faltan)


unidades_incompletas["Versiones_faltantes"] = (
    unidades_incompletas.apply(
        versiones_faltantes,
        axis=1
    )
)


# ============================================================
# CORRECCIÓN PASO 18K
# ORDENAR ANTES DE SELECCIONAR COLUMNAS
# ============================================================

columnas_mostrar = [
    "Unidad_homologada",
    "Sectores_componente",
    "Tipo_unidad",

    "Codigos_SCIAN_2007",
    "Codigos_SCIAN_2013",
    "Codigos_SCIAN_2018",
    "Codigos_SCIAN_2023",

    "N_SCIAN_2007",
    "N_SCIAN_2013",
    "N_SCIAN_2018",
    "N_SCIAN_2023",

    "Versiones_faltantes"
]

unidades_incompletas_ordenadas = (
    unidades_incompletas
    .sort_values(
        [
            "Versiones_con_representacion",
            "Unidad_homologada"
        ]
    )
    .reset_index(drop=True)
)

display(
    unidades_incompletas_ordenadas[
        columnas_mostrar
    ]
)

print(
    "\nTotal por auditar:",
    len(unidades_incompletas_ordenadas)
)

,Unidad_homologada,Sectores_componente,Tipo_unidad,Codigos_SCIAN_2007,Codigos_SCIAN_2013,Codigos_SCIAN_2018,Codigos_SCIAN_2023,N_SCIAN_2007,N_SCIAN_2013,N_SCIAN_2018,N_SCIAN_2023,Versiones_faltantes
0,UH0389,48,Continuidad sin cambio explícito,,,484233,484233,0,0,1,1,SCIAN 2007 | SCIAN 2013
1,UH0390,54,Continuidad sin cambio explícito,,,541930,541930,0,0,1,1,SCIAN 2007 | SCIAN 2013
2,UH0391,71,Continuidad sin cambio explícito,,,711132,711132,0,0,1,1,SCIAN 2007 | SCIAN 2013
3,UH0392,71,Continuidad sin cambio explícito,,,713114,713114,0,0,1,1,SCIAN 2007 | SCIAN 2013
4,UH0370,48,Continuidad sin cambio explícito,,488190,488190,488190,0,1,1,1,SCIAN 2007
5,UH0371,52,Continuidad sin cambio explícito,,523990,523990,523990,0,1,1,1,SCIAN 2007
6,UH0372,53,Continuidad sin cambio explícito,,531111,531111,531111,0,1,1,1,SCIAN 2007
7,UH0373,53,Continuidad sin cambio explícito,,532122,532122,532122,0,1,1,1,SCIAN 2007
8,UH0374,53,Continuidad sin cambio explícito,,532491,532491,532491,0,1,1,1,SCIAN 2007
9,UH0375,54,Continuidad sin cambio explícito,,541820,541820,541820,0,1,1,1,SCIAN 2007



Total por auditar: 21


In [239]:
unidades_incompletas_ordenadas.to_csv(
    HOMOLOGACION /
    "AUDITORIA_21_UNIDADES_COBERTURA_INCOMPLETA.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Auditoría de 21 unidades guardada.")

✓ Auditoría de 21 unidades guardada.


In [240]:
# ============================================================
# FASE 2 - PASO 18M
# DESCARGAR CATÁLOGOS COMPLETOS OFICIALES SCIAN 2007 Y 2013
# ============================================================

import requests
from pathlib import Path

url_scian_2007 = (
    "https://www.inegi.org.mx/contenido/productos/prod_serv/"
    "contenidos/espanol/bvinegi/productos/historicos/1329/"
    "702825023614/702825023614_1.pdf"
)

url_scian_2013 = (
    "https://www.inegi.org.mx/contenidos/productos/prod_serv/"
    "contenidos/espanol/bvinegi/productos/clasificadores/"
    "SCIAN/SCIAN_2013/702825051693.pdf"
)

catalogo_2007 = (
    HOMOLOGACION / "CATALOGO_COMPLETO_SCIAN_2007.pdf"
)

catalogo_2013 = (
    HOMOLOGACION / "CATALOGO_COMPLETO_SCIAN_2013.pdf"
)


def descargar_pdf(url, destino):

    if destino.exists():
        print("Ya existe:", destino.name)
        return

    r = requests.get(
        url,
        timeout=120
    )

    r.raise_for_status()

    destino.write_bytes(r.content)

    print(
        "✓ Descargado:",
        destino.name,
        "|",
        round(len(r.content) / 1024 / 1024, 2),
        "MB"
    )


descargar_pdf(
    url_scian_2007,
    catalogo_2007
)

descargar_pdf(
    url_scian_2013,
    catalogo_2013
)

print("\nSCIAN 2007 existe:", catalogo_2007.exists())
print("SCIAN 2013 existe:", catalogo_2013.exists())

✓ Descargado: CATALOGO_COMPLETO_SCIAN_2007.pdf | 40.42 MB
✓ Descargado: CATALOGO_COMPLETO_SCIAN_2013.pdf | 6.69 MB

SCIAN 2007 existe: True
SCIAN 2013 existe: True


In [241]:
# ============================================================
# PASO 18N
# EXTRAER CATÁLOGO DE CÓDIGOS DE 6 DÍGITOS
# ============================================================

import re
import pdfplumber


def extraer_codigos_catalogo(pdf_path):

    codigos = set()

    with pdfplumber.open(pdf_path) as pdf:

        print(
            pdf_path.name,
            "| páginas:",
            len(pdf.pages)
        )

        for i, pagina in enumerate(
            pdf.pages,
            start=1
        ):

            texto = pagina.extract_text() or ""

            encontrados = re.findall(
                r"\b\d{6}\b",
                texto
            )

            codigos.update(encontrados)

            if i % 100 == 0:
                print(
                    "Procesadas:",
                    i,
                    "páginas"
                )

    return codigos


codigos_catalogo_2007 = (
    extraer_codigos_catalogo(
        catalogo_2007
    )
)

codigos_catalogo_2013 = (
    extraer_codigos_catalogo(
        catalogo_2013
    )
)

print(
    "\nCódigos de 6 dígitos encontrados SCIAN 2007:",
    len(codigos_catalogo_2007)
)

print(
    "Códigos de 6 dígitos encontrados SCIAN 2013:",
    len(codigos_catalogo_2013)
)

CATALOGO_COMPLETO_SCIAN_2007.pdf | páginas: 633
Procesadas: 100 páginas
Procesadas: 200 páginas
Procesadas: 300 páginas
Procesadas: 400 páginas
Procesadas: 500 páginas
Procesadas: 600 páginas
CATALOGO_COMPLETO_SCIAN_2013.pdf | páginas: 596
Procesadas: 100 páginas
Procesadas: 200 páginas
Procesadas: 300 páginas
Procesadas: 400 páginas
Procesadas: 500 páginas

Códigos de 6 dígitos encontrados SCIAN 2007: 1094
Códigos de 6 dígitos encontrados SCIAN 2013: 1076


In [242]:
# ============================================================
# PASO 18O
# AUDITAR LAS 21 UNIDADES CONTRA CATÁLOGOS COMPLETOS
# ============================================================

resultados_auditoria = []

for _, row in unidades_incompletas_ordenadas.iterrows():

    unidad = row["Unidad_homologada"]

    # Tomamos cualquier código conocido de la unidad.
    candidatos = []

    for col in [
        "Codigos_SCIAN_2007",
        "Codigos_SCIAN_2013",
        "Codigos_SCIAN_2018",
        "Codigos_SCIAN_2023"
    ]:

        texto = str(row[col])

        candidatos.extend(
            [
                x.strip()
                for x in texto.split("|")
                if x.strip()
                and x.strip().lower() != "nan"
            ]
        )

    candidatos = sorted(
        set(candidatos)
    )

    # En estas 21 unidades son continuidades simples,
    # así que debería haber un código principal.
    codigo = candidatos[0] if candidatos else ""

    resultados_auditoria.append({
        "Unidad_homologada": unidad,
        "Codigo": codigo,

        "Existe_SCIAN_2007":
            codigo in codigos_catalogo_2007,

        "Existe_SCIAN_2013":
            codigo in codigos_catalogo_2013,

        "Versiones_faltantes_grafo":
            row["Versiones_faltantes"]
    })


auditoria_catalogos_21 = pd.DataFrame(
    resultados_auditoria
)

display(auditoria_catalogos_21)

,Unidad_homologada,Codigo,Existe_SCIAN_2007,Existe_SCIAN_2013,Versiones_faltantes_grafo
0,UH0389,484233,True,True,SCIAN 2007 | SCIAN 2013
1,UH0390,541930,True,True,SCIAN 2007 | SCIAN 2013
2,UH0391,711132,True,True,SCIAN 2007 | SCIAN 2013
3,UH0392,713114,True,True,SCIAN 2007 | SCIAN 2013
4,UH0370,488190,True,True,SCIAN 2007
5,UH0371,523990,True,True,SCIAN 2007
6,UH0372,531111,True,True,SCIAN 2007
7,UH0373,532122,True,True,SCIAN 2007
8,UH0374,532491,True,True,SCIAN 2007
9,UH0375,541820,True,True,SCIAN 2007


In [243]:
# ============================================================
# PASO 18P
# CLASIFICAR RESULTADO DE LA AUDITORÍA
# ============================================================

def clasificar_auditoria(row):

    faltantes = row[
        "Versiones_faltantes_grafo"
    ]

    existe07 = row[
        "Existe_SCIAN_2007"
    ]

    existe13 = row[
        "Existe_SCIAN_2013"
    ]

    if (
        "SCIAN 2007" in faltantes
        and existe07
    ):
        return (
            "Completar continuidad en SCIAN 2007"
        )

    if (
        "SCIAN 2007" in faltantes
        and not existe07
        and
        "SCIAN 2013" not in faltantes
    ):
        return (
            "No existía como clase en SCIAN 2007"
        )

    if (
        "SCIAN 2013" in faltantes
        and existe13
    ):
        return (
            "Completar continuidad en SCIAN 2013"
        )

    if (
        "SCIAN 2013" in faltantes
        and not existe13
    ):
        return (
            "No existía como clase en SCIAN 2013"
        )

    return "REVISAR"


auditoria_catalogos_21[
    "Resultado"
] = auditoria_catalogos_21.apply(
    clasificar_auditoria,
    axis=1
)

display(
    auditoria_catalogos_21[
        [
            "Unidad_homologada",
            "Codigo",
            "Existe_SCIAN_2007",
            "Existe_SCIAN_2013",
            "Versiones_faltantes_grafo",
            "Resultado"
        ]
    ]
)

print("\nRESUMEN:")

display(
    auditoria_catalogos_21[
        "Resultado"
    ]
    .value_counts()
    .reset_index()
)

,Unidad_homologada,Codigo,Existe_SCIAN_2007,Existe_SCIAN_2013,Versiones_faltantes_grafo,Resultado
0,UH0389,484233,True,True,SCIAN 2007 | SCIAN 2013,Completar continuidad en SCIAN 2007
1,UH0390,541930,True,True,SCIAN 2007 | SCIAN 2013,Completar continuidad en SCIAN 2007
2,UH0391,711132,True,True,SCIAN 2007 | SCIAN 2013,Completar continuidad en SCIAN 2007
3,UH0392,713114,True,True,SCIAN 2007 | SCIAN 2013,Completar continuidad en SCIAN 2007
4,UH0370,488190,True,True,SCIAN 2007,Completar continuidad en SCIAN 2007
5,UH0371,523990,True,True,SCIAN 2007,Completar continuidad en SCIAN 2007
6,UH0372,531111,True,True,SCIAN 2007,Completar continuidad en SCIAN 2007
7,UH0373,532122,True,True,SCIAN 2007,Completar continuidad en SCIAN 2007
8,UH0374,532491,True,True,SCIAN 2007,Completar continuidad en SCIAN 2007
9,UH0375,541820,True,True,SCIAN 2007,Completar continuidad en SCIAN 2007



RESUMEN:


,Resultado,count
0,Completar continuidad en SCIAN 2007,21


In [244]:
# ============================================================
# FASE 2 - PASO 18Q
# AUDITAR CAMBIOS EXPLÍCITOS DE LAS 21 UNIDADES
# ============================================================

codigos_21 = set(
    auditoria_catalogos_21["Codigo"]
    .astype(str)
    .str.strip()
)

filas_auditoria = []

for codigo in sorted(codigos_21):

    cambio_07_13 = (
        codigo in set(
            mapa_maestro_07_13["SCIAN_2007"]
            .astype(str).str.strip()
        )
        or
        codigo in set(
            mapa_maestro_07_13["SCIAN_2013"]
            .astype(str).str.strip()
        )
    )

    cambio_13_18 = (
        codigo in set(
            mapa_maestro_13_18["SCIAN_2013"]
            .astype(str).str.strip()
        )
        or
        codigo in set(
            mapa_maestro_13_18["SCIAN_2018"]
            .astype(str).str.strip()
        )
    )

    filas_auditoria.append({
        "Codigo": codigo,
        "Cambio_explicito_2007_2013": cambio_07_13,
        "Cambio_explicito_2013_2018": cambio_13_18
    })


auditoria_cambios_21 = pd.DataFrame(
    filas_auditoria
)

display(auditoria_cambios_21)

print(
    "\nCódigos con cambio explícito 2007→2013:",
    auditoria_cambios_21[
        "Cambio_explicito_2007_2013"
    ].sum()
)

print(
    "Códigos con cambio explícito 2013→2018:",
    auditoria_cambios_21[
        "Cambio_explicito_2013_2018"
    ].sum()
)

,Codigo,Cambio_explicito_2007_2013,Cambio_explicito_2013_2018
0,484233,False,False
1,488190,False,False
2,523990,False,False
3,531111,False,False
4,532122,False,False
5,532491,False,False
6,541820,False,False
7,541860,False,False
8,541930,False,False
9,611432,False,False



Códigos con cambio explícito 2007→2013: 0
Códigos con cambio explícito 2013→2018: 0


In [245]:
# ============================================================
# FASE 2 - PASO 18R
# COMPLETAR LAS 21 UNIDADES EN LAS VERSIONES FALTANTES
# ============================================================

tabla_maestra_completa = (
    tabla_maestra_homologacion.copy()
)

filas_nuevas = []

for _, row in auditoria_catalogos_21.iterrows():

    unidad = row["Unidad_homologada"]
    codigo = str(row["Codigo"]).strip()

    # Recuperar metadatos de esa unidad
    referencia = (
        tabla_maestra_homologacion[
            tabla_maestra_homologacion[
                "Unidad_homologada"
            ] == unidad
        ]
        .iloc[0]
    )

    versiones_agregar = []

    if row["Existe_SCIAN_2007"]:
        versiones_agregar.append("SCIAN 2007")

    if row["Existe_SCIAN_2013"]:
        versiones_agregar.append("SCIAN 2013")

    # También conservar 2018/2023 si ya forman parte
    # de la continuidad conocida
    versiones_agregar.extend(
        ["SCIAN 2018", "SCIAN 2023"]
    )

    versiones_agregar = sorted(
        set(versiones_agregar)
    )

    for version in versiones_agregar:

        ya_existe = (
            (
                tabla_maestra_completa[
                    "Version_SCIAN"
                ] == version
            )
            &
            (
                tabla_maestra_completa[
                    "SCIAN_original"
                ].astype(str).str.strip()
                == codigo
            )
        ).any()

        if not ya_existe:

            filas_nuevas.append({
                "Version_SCIAN": version,
                "SCIAN_original": codigo,
                "Unidad_homologada": unidad,
                "Familia": referencia["Familia"],
                "Tipo_unidad": referencia["Tipo_unidad"],
                "Sectores_componente":
                    referencia["Sectores_componente"],
                "Regla_calculo":
                    referencia["Regla_calculo"],
                "Observacion_metodologica":
                    (
                        "Continuidad completada mediante "
                        "catálogo oficial SCIAN; "
                        "sin establecimiento observado en "
                        "Morelia en la versión correspondiente"
                    )
            })


filas_nuevas_df = pd.DataFrame(filas_nuevas)

print(
    "Combinaciones versión-código añadidas:",
    len(filas_nuevas_df)
)

display(filas_nuevas_df)


tabla_maestra_completa = pd.concat(
    [
        tabla_maestra_completa,
        filas_nuevas_df
    ],
    ignore_index=True
)

tabla_maestra_completa = (
    tabla_maestra_completa
    .drop_duplicates(
        subset=[
            "Version_SCIAN",
            "SCIAN_original"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "\nFilas tabla observada:",
    len(tabla_maestra_homologacion)
)

print(
    "Filas tabla completa:",
    len(tabla_maestra_completa)
)

Combinaciones versión-código añadidas: 50


,Version_SCIAN,SCIAN_original,Unidad_homologada,Familia,Tipo_unidad,Sectores_componente,Regla_calculo,Observacion_metodologica
0,SCIAN 2007,484233,UH0389,,Continuidad sin cambio explícito,48,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
1,SCIAN 2013,484233,UH0389,,Continuidad sin cambio explícito,48,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
2,SCIAN 2018,484233,UH0389,,Continuidad sin cambio explícito,48,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
3,SCIAN 2007,541930,UH0390,,Continuidad sin cambio explícito,54,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
4,SCIAN 2013,541930,UH0390,,Continuidad sin cambio explícito,54,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
5,SCIAN 2018,541930,UH0390,,Continuidad sin cambio explícito,54,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
6,SCIAN 2007,711132,UH0391,,Continuidad sin cambio explícito,71,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
7,SCIAN 2013,711132,UH0391,,Continuidad sin cambio explícito,71,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
8,SCIAN 2018,711132,UH0391,,Continuidad sin cambio explícito,71,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...
9,SCIAN 2007,713114,UH0392,,Continuidad sin cambio explícito,71,SUMA_DIRECTA,Continuidad completada mediante catálogo ofici...



Filas tabla observada: 1669
Filas tabla completa: 1719


In [246]:
# ============================================================
# PASO 18S
# REVALIDAR COBERTURA DE LAS 350 UNIDADES
# ============================================================

cobertura_final = (
    tabla_maestra_completa.groupby(
        [
            "Unidad_homologada",
            "Version_SCIAN"
        ]
    )
    .size()
    .reset_index(name="N_codigos")
)

matriz_cobertura_final = (
    cobertura_final.pivot_table(
        index="Unidad_homologada",
        columns="Version_SCIAN",
        values="N_codigos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for version in [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]:
    if version not in matriz_cobertura_final.columns:
        matriz_cobertura_final[version] = 0


matriz_cobertura_final[
    "Versiones_con_representacion"
] = (
    (
        matriz_cobertura_final["SCIAN 2007"] > 0
    ).astype(int)
    +
    (
        matriz_cobertura_final["SCIAN 2013"] > 0
    ).astype(int)
    +
    (
        matriz_cobertura_final["SCIAN 2018"] > 0
    ).astype(int)
    +
    (
        matriz_cobertura_final["SCIAN 2023"] > 0
    ).astype(int)
)

print(
    "Unidades homologadas:",
    matriz_cobertura_final[
        "Unidad_homologada"
    ].nunique()
)

print(
    "Con cobertura en 4 versiones:",
    (
        matriz_cobertura_final[
            "Versiones_con_representacion"
        ] == 4
    ).sum()
)

print(
    "Todavía incompletas:",
    (
        matriz_cobertura_final[
            "Versiones_con_representacion"
        ] < 4
    ).sum()
)

print("\nDISTRIBUCIÓN:")

display(
    matriz_cobertura_final[
        "Versiones_con_representacion"
    ]
    .value_counts()
    .sort_index(ascending=False)
    .reset_index()
)

Unidades homologadas: 350
Con cobertura en 4 versiones: 286
Todavía incompletas: 64

DISTRIBUCIÓN:


,Versiones_con_representacion,count
0,4,286
1,3,34
2,2,20
3,1,10


In [247]:
# ============================================================
# FASE 2 - PASO 18R CORREGIDO
# AUDITAR LAS 21 CLASES ANTES DE COMPLETAR CONTINUIDAD
# ============================================================

codigos_21 = set(
    auditoria_catalogos_21["Codigo"]
    .astype(str)
    .str.strip()
)

exp_07_origen = set(
    mapa_maestro_07_13["SCIAN_2007"]
    .astype(str).str.strip()
)

exp_07_destino = set(
    mapa_maestro_07_13["SCIAN_2013"]
    .astype(str).str.strip()
)

exp_13_origen = set(
    mapa_maestro_13_18["SCIAN_2013"]
    .astype(str).str.strip()
)

exp_13_destino = set(
    mapa_maestro_13_18["SCIAN_2018"]
    .astype(str).str.strip()
)

auditoria_cambio_21 = []

for codigo in sorted(codigos_21):

    auditoria_cambio_21.append({
        "Codigo": codigo,
        "Cambio_explicito_2007_2013":
            (
                codigo in exp_07_origen
                or codigo in exp_07_destino
            ),
        "Cambio_explicito_2013_2018":
            (
                codigo in exp_13_origen
                or codigo in exp_13_destino
            )
    })

auditoria_cambio_21 = pd.DataFrame(
    auditoria_cambio_21
)

display(auditoria_cambio_21)

print(
    "\nCon cambio explícito 2007→2013:",
    auditoria_cambio_21[
        "Cambio_explicito_2007_2013"
    ].sum()
)

print(
    "Con cambio explícito 2013→2018:",
    auditoria_cambio_21[
        "Cambio_explicito_2013_2018"
    ].sum()
)

,Codigo,Cambio_explicito_2007_2013,Cambio_explicito_2013_2018
0,484233,False,False
1,488190,False,False
2,523990,False,False
3,531111,False,False
4,532122,False,False
5,532491,False,False
6,541820,False,False
7,541860,False,False
8,541930,False,False
9,611432,False,False



Con cambio explícito 2007→2013: 0
Con cambio explícito 2013→2018: 0


In [248]:
# ============================================================
# FASE 2 - PASO 18S
# CREAR DICCIONARIO CLASIFICATORIO COMPLETO
# DESDE resumen_unidades
# ============================================================

versiones = [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]

filas_diccionario = []

for _, row in resumen_unidades.iterrows():

    unidad = row["Unidad_homologada"]

    for version in versiones:

        columna = (
            "Codigos_"
            + version.replace(" ", "_")
        )

        texto = str(row[columna])

        codigos = [
            x.strip()
            for x in texto.split("|")
            if x.strip()
            and x.strip().lower() != "nan"
        ]

        for codigo in codigos:

            filas_diccionario.append({
                "Unidad_homologada": unidad,
                "Version_SCIAN": version,
                "SCIAN_original": codigo,
                "Origen_asignacion":
                    "Grafo homologación"
            })


diccionario_clasificatorio = pd.DataFrame(
    filas_diccionario
)

print(
    "Filas generadas desde resumen_unidades:",
    len(diccionario_clasificatorio)
)

print(
    "Unidades:",
    diccionario_clasificatorio[
        "Unidad_homologada"
    ].nunique()
)

print(
    "Duplicados versión-código:",
    diccionario_clasificatorio.duplicated(
        subset=[
            "Version_SCIAN",
            "SCIAN_original"
        ]
    ).sum()
)

Filas generadas desde resumen_unidades: 2155
Unidades: 404
Duplicados versión-código: 0


In [249]:
# ============================================================
# FASE 2 - PASO 18T
# COMPLETAR LAS 21 CONTINUIDADES VALIDADAS EN CATÁLOGOS
# ============================================================

filas_complementarias = []

for _, row in auditoria_catalogos_21.iterrows():

    unidad = row["Unidad_homologada"]
    codigo = str(row["Codigo"]).strip()

    faltantes = [
        x.strip()
        for x in str(
            row["Versiones_faltantes_grafo"]
        ).split("|")
        if x.strip()
    ]

    for version in faltantes:

        permitido = False

        if (
            version == "SCIAN 2007"
            and row["Existe_SCIAN_2007"]
        ):
            permitido = True

        elif (
            version == "SCIAN 2013"
            and row["Existe_SCIAN_2013"]
        ):
            permitido = True

        if permitido:

            filas_complementarias.append({
                "Unidad_homologada": unidad,
                "Version_SCIAN": version,
                "SCIAN_original": codigo,
                "Origen_asignacion":
                    "Catálogo oficial - continuidad auditada"
            })


complementarias_21 = pd.DataFrame(
    filas_complementarias
)

print(
    "Combinaciones añadidas:",
    len(complementarias_21)
)

display(complementarias_21)

Combinaciones añadidas: 25


,Unidad_homologada,Version_SCIAN,SCIAN_original,Origen_asignacion
0,UH0389,SCIAN 2007,484233,Catálogo oficial - continuidad auditada
1,UH0389,SCIAN 2013,484233,Catálogo oficial - continuidad auditada
2,UH0390,SCIAN 2007,541930,Catálogo oficial - continuidad auditada
3,UH0390,SCIAN 2013,541930,Catálogo oficial - continuidad auditada
4,UH0391,SCIAN 2007,711132,Catálogo oficial - continuidad auditada
5,UH0391,SCIAN 2013,711132,Catálogo oficial - continuidad auditada
6,UH0392,SCIAN 2007,713114,Catálogo oficial - continuidad auditada
7,UH0392,SCIAN 2013,713114,Catálogo oficial - continuidad auditada
8,UH0370,SCIAN 2007,488190,Catálogo oficial - continuidad auditada
9,UH0371,SCIAN 2007,523990,Catálogo oficial - continuidad auditada


In [250]:
# ============================================================
# FASE 2 - PASO 18U
# DICCIONARIO CLASIFICATORIO DEFINITIVO
# ============================================================

diccionario_clasificatorio_completo = pd.concat(
    [
        diccionario_clasificatorio,
        complementarias_21
    ],
    ignore_index=True
)

diccionario_clasificatorio_completo = (
    diccionario_clasificatorio_completo
    .drop_duplicates(
        subset=[
            "Unidad_homologada",
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# CONTROL: un mismo código-versión no puede pertenecer
# a dos unidades distintas
# ------------------------------------------------------------

conflictos = (
    diccionario_clasificatorio_completo
    .groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )["Unidad_homologada"]
    .nunique()
    .reset_index(
        name="N_unidades"
    )
)

conflictos = conflictos[
    conflictos["N_unidades"] > 1
]


print(
    "Conflictos código-versión:",
    len(conflictos)
)

if not conflictos.empty:
    display(conflictos)

Conflictos código-versión: 0


In [251]:
# ============================================================
# FASE 2 - PASO 18V
# COBERTURA FINAL CORRECTA DE LAS 350 UNIDADES
# ============================================================

cobertura_final = (
    diccionario_clasificatorio_completo
    .groupby(
        [
            "Unidad_homologada",
            "Version_SCIAN"
        ]
    )
    .size()
    .reset_index(name="N_codigos")
)

matriz_cobertura_final = (
    cobertura_final.pivot_table(
        index="Unidad_homologada",
        columns="Version_SCIAN",
        values="N_codigos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for version in versiones:

    if version not in matriz_cobertura_final.columns:
        matriz_cobertura_final[version] = 0


matriz_cobertura_final[
    "Versiones_con_representacion"
] = sum(
    (
        matriz_cobertura_final[version] > 0
    ).astype(int)
    for version in versiones
)


print(
    "Unidades homologadas:",
    matriz_cobertura_final[
        "Unidad_homologada"
    ].nunique()
)

print(
    "Con cobertura en 4 versiones:",
    (
        matriz_cobertura_final[
            "Versiones_con_representacion"
        ] == 4
    ).sum()
)

print(
    "Todavía incompletas:",
    (
        matriz_cobertura_final[
            "Versiones_con_representacion"
        ] < 4
    ).sum()
)

print("\nDISTRIBUCIÓN:")

display(
    matriz_cobertura_final[
        "Versiones_con_representacion"
    ]
    .value_counts()
    .sort_index(ascending=False)
    .reset_index()
)

Unidades homologadas: 404
Con cobertura en 4 versiones: 383
Todavía incompletas: 21

DISTRIBUCIÓN:


,Versiones_con_representacion,count
0,4,383
1,3,21


In [252]:
# ============================================================
# FASE 2 - PASO 18X
# CORRECCIÓN DEFINITIVA:
# RECONSTRUIR SOLO LAS 350 UNIDADES UTILIZADAS EN MORELIA
# ============================================================

# ------------------------------------------------------------
# 1. IDENTIFICAR LAS 350 UNIDADES REALMENTE UTILIZADAS
# ------------------------------------------------------------

unidades_utilizadas = set(
    tabla_maestra_homologacion[
        "Unidad_homologada"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

print(
    "Unidades utilizadas en Morelia:",
    len(unidades_utilizadas)
)


# ------------------------------------------------------------
# 2. FILTRAR resumen_unidades A ESAS 350
# ------------------------------------------------------------

resumen_350 = (
    resumen_unidades[
        resumen_unidades["Unidad_homologada"]
        .astype(str)
        .str.strip()
        .isin(unidades_utilizadas)
    ]
    .copy()
)

print(
    "Unidades recuperadas desde resumen_unidades:",
    resumen_350["Unidad_homologada"].nunique()
)


# ------------------------------------------------------------
# 3. CONSTRUIR DICCIONARIO CLASIFICATORIO DE LAS 350
# ------------------------------------------------------------

versiones = [
    "SCIAN 2007",
    "SCIAN 2013",
    "SCIAN 2018",
    "SCIAN 2023"
]

filas_diccionario_350 = []

for _, row in resumen_350.iterrows():

    unidad = str(
        row["Unidad_homologada"]
    ).strip()

    for version in versiones:

        columna = (
            "Codigos_"
            + version.replace(" ", "_")
        )

        texto = str(row[columna])

        codigos = [
            x.strip()
            for x in texto.split("|")
            if x.strip()
            and x.strip().lower() != "nan"
        ]

        for codigo in codigos:

            filas_diccionario_350.append({
                "Unidad_homologada": unidad,
                "Version_SCIAN": version,
                "SCIAN_original": codigo,
                "Origen_asignacion":
                    "Grafo homologación"
            })


diccionario_350 = pd.DataFrame(
    filas_diccionario_350
)

print(
    "\nUnidades en diccionario base:",
    diccionario_350[
        "Unidad_homologada"
    ].nunique()
)


# ------------------------------------------------------------
# 4. VERIFICAR QUE LAS 21 CLASES AUDITADAS NO TENGAN
#    CAMBIOS EXPLÍCITOS QUE CONTRADIGAN LA CONTINUIDAD
# ------------------------------------------------------------

codigos_21 = set(
    auditoria_catalogos_21["Codigo"]
    .astype(str)
    .str.strip()
)

exp_07_13 = (
    set(
        mapa_maestro_07_13["SCIAN_2007"]
        .astype(str).str.strip()
    )
    |
    set(
        mapa_maestro_07_13["SCIAN_2013"]
        .astype(str).str.strip()
    )
)

exp_13_18 = (
    set(
        mapa_maestro_13_18["SCIAN_2013"]
        .astype(str).str.strip()
    )
    |
    set(
        mapa_maestro_13_18["SCIAN_2018"]
        .astype(str).str.strip()
    )
)

problematicos_21 = []

for codigo in sorted(codigos_21):

    if (
        codigo in exp_07_13
        or codigo in exp_13_18
    ):
        problematicos_21.append(codigo)


print(
    "\nCódigos de las 21 unidades con cambio explícito:",
    len(problematicos_21)
)

if problematicos_21:
    print(problematicos_21)


# ------------------------------------------------------------
# 5. COMPLETAR SOLO LAS VERSIONES FALTANTES YA VALIDADAS
#    EN LOS CATÁLOGOS OFICIALES
# ------------------------------------------------------------

filas_complementarias = []

if len(problematicos_21) == 0:

    for _, row in auditoria_catalogos_21.iterrows():

        unidad = str(
            row["Unidad_homologada"]
        ).strip()

        codigo = str(
            row["Codigo"]
        ).strip()

        # Seguridad: solo unidades realmente utilizadas
        if unidad not in unidades_utilizadas:
            continue

        faltantes = [
            x.strip()
            for x in str(
                row["Versiones_faltantes_grafo"]
            ).split("|")
            if x.strip()
        ]

        for version in faltantes:

            permitido = False

            if (
                version == "SCIAN 2007"
                and bool(row["Existe_SCIAN_2007"])
            ):
                permitido = True

            elif (
                version == "SCIAN 2013"
                and bool(row["Existe_SCIAN_2013"])
            ):
                permitido = True

            if permitido:

                filas_complementarias.append({
                    "Unidad_homologada": unidad,
                    "Version_SCIAN": version,
                    "SCIAN_original": codigo,
                    "Origen_asignacion":
                        "Catálogo oficial - continuidad auditada"
                })


complementarias_21 = pd.DataFrame(
    filas_complementarias
)

print(
    "Combinaciones complementarias añadidas:",
    len(complementarias_21)
)


# ------------------------------------------------------------
# 6. UNIR Y ELIMINAR DUPLICADOS
# ------------------------------------------------------------

diccionario_350_completo = pd.concat(
    [
        diccionario_350,
        complementarias_21
    ],
    ignore_index=True
)

diccionario_350_completo = (
    diccionario_350_completo
    .drop_duplicates(
        subset=[
            "Unidad_homologada",
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. CONTROL DE CONFLICTOS:
#    UN CÓDIGO-VERSIÓN NO PUEDE IR A DOS UNIDADES
# ------------------------------------------------------------

conflictos = (
    diccionario_350_completo
    .groupby(
        [
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )["Unidad_homologada"]
    .nunique()
    .reset_index(
        name="N_unidades"
    )
)

conflictos = conflictos[
    conflictos["N_unidades"] > 1
]

print(
    "\nConflictos código-versión:",
    len(conflictos)
)

if not conflictos.empty:
    display(conflictos)


# ------------------------------------------------------------
# 8. COBERTURA FINAL
# ------------------------------------------------------------

cobertura_350 = (
    diccionario_350_completo
    .groupby(
        [
            "Unidad_homologada",
            "Version_SCIAN"
        ]
    )
    .size()
    .reset_index(name="N_codigos")
)

matriz_cobertura_350 = (
    cobertura_350.pivot_table(
        index="Unidad_homologada",
        columns="Version_SCIAN",
        values="N_codigos",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

for version in versiones:

    if version not in matriz_cobertura_350.columns:
        matriz_cobertura_350[version] = 0


matriz_cobertura_350[
    "Versiones_con_representacion"
] = sum(
    (
        matriz_cobertura_350[version] > 0
    ).astype(int)
    for version in versiones
)


print("\n========== COBERTURA FINAL CORREGIDA ==========")

print(
    "Unidades homologadas:",
    matriz_cobertura_350[
        "Unidad_homologada"
    ].nunique()
)

print(
    "Con cobertura en 4 versiones:",
    (
        matriz_cobertura_350[
            "Versiones_con_representacion"
        ] == 4
    ).sum()
)

print(
    "Todavía incompletas:",
    (
        matriz_cobertura_350[
            "Versiones_con_representacion"
        ] < 4
    ).sum()
)

print("\nDISTRIBUCIÓN:")

display(
    matriz_cobertura_350[
        "Versiones_con_representacion"
    ]
    .value_counts()
    .sort_index(ascending=False)
    .reset_index()
)


# ------------------------------------------------------------
# 9. SI QUEDARA ALGUNA INCOMPLETA, MOSTRARLA
# ------------------------------------------------------------

incompletas_finales = (
    matriz_cobertura_350[
        matriz_cobertura_350[
            "Versiones_con_representacion"
        ] < 4
    ]
    .copy()
)

if not incompletas_finales.empty:

    print("\nUNIDADES TODAVÍA INCOMPLETAS:")

    display(incompletas_finales)

Unidades utilizadas en Morelia: 350
Unidades recuperadas desde resumen_unidades: 350

Unidades en diccionario base: 350

Códigos de las 21 unidades con cambio explícito: 0
Combinaciones complementarias añadidas: 25

Conflictos código-versión: 0

========== COBERTURA FINAL CORREGIDA ==========
Unidades homologadas: 350
Con cobertura en 4 versiones: 350
Todavía incompletas: 0

DISTRIBUCIÓN:


,Versiones_con_representacion,count
0,4,350


# Construcción del panel histórico homologado 2010–2026

Una vez comprobada la representación clasificatoria de las 350 unidades
homologadas en las cuatro versiones del SCIAN, se construye el panel histórico
correspondiente a los 16 cortes anuales seleccionados del DENUE.

Cada unidad homologada se representa en cada año de la serie. Cuando una
unidad cuenta con representación clasificatoria válida para la versión SCIAN
correspondiente pero no presenta establecimientos observados en Morelia, se
asigna un valor igual a cero.

Para la familia FL09 se conserva la incertidumbre previamente identificada
mediante un límite inferior y un límite superior. Para las demás unidades,
ambos límites son idénticos.

De esta manera, la estructura final permite distinguir entre ausencia
empírica de establecimientos e inexistencia clasificatoria, evitando la
introducción de ceros artificiales.

In [253]:
# ============================================================
# FASE 2 - PASO 19A
# GUARDAR DICCIONARIO CLASIFICATORIO DEFINITIVO
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

if "RESULTADOS" not in globals():
    RESULTADOS = Path.cwd() / "05_RESULTADOS"

diccionario_350_completo.to_csv(
    HOMOLOGACION /
    "DICCIONARIO_CLASIFICATORIO_HOMOLOGADO_350_UNIDADES.csv",
    index=False,
    encoding="utf-8-sig"
)

matriz_cobertura_350.to_csv(
    HOMOLOGACION /
    "CONTROL_COBERTURA_FINAL_350_UNIDADES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Diccionario definitivo guardado.")
print("✓ Cobertura final guardada.")

✓ Diccionario definitivo guardado.
✓ Cobertura final guardada.


In [254]:
# ============================================================
# FASE 2 - PASO 19B
# APLICAR DICCIONARIO DEFINITIVO A LOS 6,435 REGISTROS
# ============================================================

mapa_final = (
    diccionario_350_completo[
        [
            "Version_SCIAN",
            "SCIAN_original",
            "Unidad_homologada"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Version_SCIAN",
            "SCIAN_original"
        ]
    )
)

inventario_homologado_final = (
    inventario_scian.merge(
        mapa_final,
        on=[
            "Version_SCIAN",
            "SCIAN_original"
        ],
        how="left",
        validate="many_to_one"
    )
)

print("Registros originales:", len(inventario_scian))
print("Registros homologados:", len(inventario_homologado_final))

print(
    "Sin unidad homologada:",
    inventario_homologado_final[
        "Unidad_homologada"
    ].isna().sum()
)

print(
    "Unidades utilizadas:",
    inventario_homologado_final[
        "Unidad_homologada"
    ].nunique()
)

Registros originales: 6435
Registros homologados: 6435
Sin unidad homologada: 0
Unidades utilizadas: 350


In [255]:
# ============================================================
# FASE 2 - PASO 19C
# CREAR MALLA 350 UNIDADES × 16 AÑOS
# ============================================================

años = sorted(
    inventario_scian["Año"]
    .dropna()
    .astype(int)
    .unique()
)

unidades = sorted(
    unidades_utilizadas
)

print("Años:", años)
print("Número de años:", len(años))
print("Unidades:", len(unidades))

print(
    "Observaciones esperadas:",
    len(años) * len(unidades)
)


# ------------------------------------------------------------
# Una versión SCIAN por cada año
# ------------------------------------------------------------

año_version = (
    inventario_scian[
        [
            "Año",
            "Version_SCIAN"
        ]
    ]
    .drop_duplicates()
    .copy()
)

año_version["Año"] = (
    año_version["Año"]
    .astype(int)
)

print(
    "Duplicados año-versión:",
    año_version.duplicated(
        subset=["Año"]
    ).sum()
)


# ------------------------------------------------------------
# Producto cartesiano unidad × año
# ------------------------------------------------------------

panel_base = pd.MultiIndex.from_product(
    [
        unidades,
        años
    ],
    names=[
        "Unidad_homologada",
        "Año"
    ]
).to_frame(index=False)

panel_base = panel_base.merge(
    año_version,
    on="Año",
    how="left",
    validate="many_to_one"
)

print(
    "\nFilas del panel base:",
    len(panel_base)
)

Años: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Número de años: 16
Unidades: 350
Observaciones esperadas: 5600
Duplicados año-versión: 0

Filas del panel base: 5600


In [256]:
# ============================================================
# FASE 2 - PASO 19D
# AGREGAR ESTABLECIMIENTOS POR UNIDAD HOMOLOGADA Y AÑO
# ============================================================

conteos_unidad_año = (
    inventario_homologado_final
    .groupby(
        [
            "Unidad_homologada",
            "Año"
        ],
        as_index=False
    )["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos":
            "Establecimientos_agregados"
        }
    )
)

conteos_unidad_año["Año"] = (
    conteos_unidad_año["Año"]
    .astype(int)
)


panel_homologado = panel_base.merge(
    conteos_unidad_año,
    on=[
        "Unidad_homologada",
        "Año"
    ],
    how="left",
    validate="one_to_one"
)

# Ahora sí: NA = unidad existente pero no observada.
panel_homologado[
    "Establecimientos_agregados"
] = (
    panel_homologado[
        "Establecimientos_agregados"
    ]
    .fillna(0)
    .astype(int)
)


print(
    "Filas:",
    len(panel_homologado)
)

print(
    "Nulos en establecimientos:",
    panel_homologado[
        "Establecimientos_agregados"
    ].isna().sum()
)

print(
    "Ceros introducidos por ausencia empírica:",
    (
        panel_homologado[
            "Establecimientos_agregados"
        ] == 0
    ).sum()
)

Filas: 5600
Nulos en establecimientos: 0
Ceros introducidos por ausencia empírica: 833


In [257]:
# ============================================================
# FASE 2 - PASO 19E
# INCORPORAR METADATOS DE LAS 350 UNIDADES
# ============================================================

meta_unidades = (
    resumen_350[
        [
            "Unidad_homologada",
            "Familia",
            "Tipo_unidad",
            "Sectores_componente"
        ]
    ]
    .drop_duplicates(
        subset=["Unidad_homologada"]
    )
)

panel_homologado = panel_homologado.merge(
    meta_unidades,
    on="Unidad_homologada",
    how="left",
    validate="many_to_one"
)

print(
    "Unidades sin metadatos:",
    panel_homologado[
        "Tipo_unidad"
    ].isna().sum()
)

Unidades sin metadatos: 0


In [258]:
# ============================================================
# FASE 2 - PASO 19F
# INCORPORAR INTERVALO DE INCERTIDUMBRE DE FL09
# ============================================================

# Para todas las unidades normales:
panel_homologado["Establecimientos_min"] = (
    panel_homologado[
        "Establecimientos_agregados"
    ]
)

panel_homologado["Establecimientos_max"] = (
    panel_homologado[
        "Establecimientos_agregados"
    ]
)

panel_homologado["Incertidumbre"] = 0


# ------------------------------------------------------------
# Reconstruir FL09 directamente desde el inventario
# ------------------------------------------------------------

filas_fl09 = []

for año in años:

    temp = inventario_scian[
        inventario_scian["Año"].astype(int) == año
    ].copy()

    version = temp["Version_SCIAN"].iloc[0]

    limite_inferior = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .isin(
                [
                    "811121",
                    "811129"
                ]
            )
        ]["Establecimientos"]
        .sum()
    )

    # Sólo genera ambigüedad cuando 811123 existe
    ambiguo = (
        temp[
            temp["SCIAN_original"]
            .astype(str)
            .str.strip()
            .eq("811123")
        ]["Establecimientos"]
        .sum()
    )

    limite_superior = (
        limite_inferior
        + ambiguo
    )

    filas_fl09.append({
        "Año": año,
        "FL09_min": int(limite_inferior),
        "FL09_max": int(limite_superior),
        "FL09_incertidumbre": int(ambiguo)
    })


intervalo_FL09 = pd.DataFrame(
    filas_fl09
)

display(intervalo_FL09)

,Año,FL09_min,FL09_max,FL09_incertidumbre
0,2010,366,366,0
1,2011,367,367,0
2,2012,368,368,0
3,2013,368,368,0
4,2015,398,398,0
5,2016,399,399,0
6,2017,399,399,0
7,2018,399,399,0
8,2019,426,427,1
9,2020,430,431,1


In [259]:
# ============================================================
# PASO 19G
# SOBRESCRIBIR LOS LÍMITES DE FL09
# ============================================================

panel_homologado = panel_homologado.merge(
    intervalo_FL09,
    on="Año",
    how="left"
)

mask_fl09 = (
    panel_homologado[
        "Unidad_homologada"
    ] == "FL09"
)

panel_homologado.loc[
    mask_fl09,
    "Establecimientos_min"
] = (
    panel_homologado.loc[
        mask_fl09,
        "FL09_min"
    ]
)

panel_homologado.loc[
    mask_fl09,
    "Establecimientos_max"
] = (
    panel_homologado.loc[
        mask_fl09,
        "FL09_max"
    ]
)

panel_homologado.loc[
    mask_fl09,
    "Incertidumbre"
] = (
    panel_homologado.loc[
        mask_fl09,
        "FL09_incertidumbre"
    ]
)

panel_homologado = panel_homologado.drop(
    columns=[
        "FL09_min",
        "FL09_max",
        "FL09_incertidumbre"
    ]
)

In [260]:
# ============================================================
# FASE 2 - PASO 19H
# AUDITORÍA FINAL DEL PANEL 350 × 16
# ============================================================

print("========== ESTRUCTURA ==========")

print(
    "Filas:",
    len(panel_homologado)
)

print(
    "Unidades:",
    panel_homologado[
        "Unidad_homologada"
    ].nunique()
)

print(
    "Años:",
    panel_homologado[
        "Año"
    ].nunique()
)

print(
    "Duplicados unidad-año:",
    panel_homologado.duplicated(
        subset=[
            "Unidad_homologada",
            "Año"
        ]
    ).sum()
)

print(
    "Nulos en mínimo:",
    panel_homologado[
        "Establecimientos_min"
    ].isna().sum()
)

print(
    "Nulos en máximo:",
    panel_homologado[
        "Establecimientos_max"
    ].isna().sum()
)


# ------------------------------------------------------------
# CONTROL DE TOTALES POR AÑO
# ------------------------------------------------------------

totales_panel = (
    panel_homologado.groupby(
        "Año",
        as_index=False
    )
    .agg(
        Total_min=(
            "Establecimientos_min",
            "sum"
        ),
        Total_max=(
            "Establecimientos_max",
            "sum"
        )
    )
)

totales_originales = (
    inventario_scian.groupby(
        "Año",
        as_index=False
    )["Establecimientos"]
    .sum()
    .rename(
        columns={
            "Establecimientos":
            "Total_original"
        }
    )
)

control_panel = (
    totales_panel.merge(
        totales_originales,
        on="Año",
        how="left"
    )
)

control_panel[
    "Diferencia_max_original"
] = (
    control_panel["Total_max"]
    -
    control_panel["Total_original"]
)

control_panel[
    "Amplitud_total"
] = (
    control_panel["Total_max"]
    -
    control_panel["Total_min"]
)

display(control_panel)

print(
    "\nAños donde máximo = total original:",
    (
        control_panel[
            "Diferencia_max_original"
        ] == 0
    ).sum(),
    "de",
    len(control_panel)
)

print(
    "Máxima amplitud total por FL09:",
    control_panel[
        "Amplitud_total"
    ].max()
)

========== ESTRUCTURA ==========
Filas: 5600
Unidades: 350
Años: 16
Duplicados unidad-año: 0
Nulos en mínimo: 0
Nulos en máximo: 0


,Año,Total_min,Total_max,Total_original,Diferencia_max_original,Amplitud_total
0,2010,34656,34656,34656,0,0
1,2011,35093,35093,35093,0,0
2,2012,35289,35289,35289,0,0
3,2013,35797,35797,35797,0,0
4,2015,37677,37677,37677,0,0
5,2016,38372,38372,38372,0,0
6,2017,38536,38536,38536,0,0
7,2018,38728,38728,38728,0,0
8,2019,40483,40484,40484,0,1
9,2020,41447,41448,41448,0,1



Años donde máximo = total original: 16 de 16
Máxima amplitud total por FL09: 1


In [261]:
# ============================================================
# PASO 19I
# GUARDAR PANEL HISTÓRICO HOMOLOGADO
# ============================================================

panel_homologado.to_csv(
    RESULTADOS /
    "PANEL_HISTORICO_HOMOLOGADO_350_UNIDADES_2010_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

inventario_homologado_final.to_csv(
    RESULTADOS /
    "INVENTARIO_HISTORICO_HOMOLOGADO_6435_REGISTROS.csv",
    index=False,
    encoding="utf-8-sig"
)

control_panel.to_csv(
    RESULTADOS /
    "CONTROL_TOTALES_PANEL_HOMOLOGADO.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Panel 350 × 16 guardado.")
print("✓ Inventario homologado de 6,435 registros guardado.")
print("✓ Control de totales guardado.")

✓ Panel 350 × 16 guardado.
✓ Inventario homologado de 6,435 registros guardado.
✓ Control de totales guardado.


In [262]:
# ============================================================
# FASE 2 - PASO 19I
# GUARDAR PRODUCTOS FINALES DE HOMOLOGACIÓN
# ============================================================

panel_homologado.to_csv(
    RESULTADOS /
    "PANEL_HISTORICO_HOMOLOGADO_350_UNIDADES_2010_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

inventario_homologado_final.to_csv(
    RESULTADOS /
    "INVENTARIO_HISTORICO_HOMOLOGADO_6435_REGISTROS.csv",
    index=False,
    encoding="utf-8-sig"
)

control_panel.to_csv(
    RESULTADOS /
    "CONTROL_TOTALES_PANEL_HOMOLOGADO.csv",
    index=False,
    encoding="utf-8-sig"
)

diccionario_350_completo.to_csv(
    HOMOLOGACION /
    "DICCIONARIO_CLASIFICATORIO_HOMOLOGADO_350_UNIDADES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Panel histórico homologado guardado.")
print("✓ Inventario histórico homologado guardado.")
print("✓ Control de totales guardado.")
print("✓ Diccionario clasificatorio definitivo guardado.")

✓ Panel histórico homologado guardado.
✓ Inventario histórico homologado guardado.
✓ Control de totales guardado.
✓ Diccionario clasificatorio definitivo guardado.


# Cierre de la fase de homologación SCIAN

La serie histórica del DENUE utilizada en la investigación atraviesa cuatro
versiones del Sistema de Clasificación Industrial de América del Norte:
SCIAN 2007, SCIAN 2013, SCIAN 2018 y SCIAN 2023.

Para asegurar la comparabilidad longitudinal se utilizaron las tablas
comparativas oficiales entre versiones consecutivas y se construyeron
componentes longitudinales de correspondencia.

El procedimiento permitió establecer 350 unidades homologadas para las
actividades de comercio y servicios observadas en Morelia.

El panel final contiene 350 unidades homologadas en 16 cortes históricos,
para un total de 5,600 observaciones unidad-año.

Los controles finales mostraron:

- 350 unidades con representación clasificatoria en las cuatro versiones
  del SCIAN.
- 0 registros sin unidad homologada.
- 0 duplicados unidad-año.
- 0 valores nulos en los límites de establecimientos.
- Conservación exacta del total original de establecimientos en los 16
  cortes mediante el límite superior.
- Una única incertidumbre clasificatoria acotada en la familia FL09,
  equivalente como máximo a un establecimiento por año.

La homologación no modifica los códigos originales ni los archivos históricos
del DENUE. Los códigos originales se conservan para garantizar trazabilidad y
reproducibilidad.

A partir de esta base se inicia el análisis de la dinámica histórica de los
establecimientos de comercio y servicios en Morelia.

# FASE 3. Dinámica histórica de los establecimientos de comercio y servicios en Morelia

In [263]:
# ============================================================
# FASE 3 - PASO 1
# SERIE AGREGADA HOMOLOGADA 2010–2026
# ============================================================

serie_general = (
    control_panel[
        [
            "Año",
            "Total_min",
            "Total_max",
            "Total_original",
            "Amplitud_total"
        ]
    ]
    .sort_values("Año")
    .reset_index(drop=True)
)

# Para la serie descriptiva principal conservamos el total observado
serie_general["Establecimientos"] = (
    serie_general["Total_original"]
)

# Variación entre cortes consecutivos disponibles
serie_general["Cambio_absoluto"] = (
    serie_general["Establecimientos"].diff()
)

serie_general["Cambio_porcentual"] = (
    serie_general["Establecimientos"]
    .pct_change()
    * 100
)

# Distancia temporal entre cortes
serie_general["Años_desde_corte_anterior"] = (
    serie_general["Año"].diff()
)

display(serie_general)

,Año,Total_min,Total_max,Total_original,Amplitud_total,Establecimientos,Cambio_absoluto,Cambio_porcentual,Años_desde_corte_anterior
0,2010,34656,34656,34656,0,34656,NaN,NaN,NaN
1,2011,35093,35093,35093,0,35093,437.0,1.260965,1.0
2,2012,35289,35289,35289,0,35289,196.0,0.558516,1.0
3,2013,35797,35797,35797,0,35797,508.0,1.439542,1.0
4,2015,37677,37677,37677,0,37677,1880.0,5.251837,2.0
5,2016,38372,38372,38372,0,38372,695.0,1.844627,1.0
6,2017,38536,38536,38536,0,38536,164.0,0.427395,1.0
7,2018,38728,38728,38728,0,38728,192.0,0.498235,1.0
8,2019,40483,40484,40484,1,40484,1756.0,4.534187,1.0
9,2020,41447,41448,41448,1,41448,964.0,2.381188,1.0


In [264]:
# ============================================================
# FASE 3 - PASO 2
# IDENTIFICAR INTERVALOS QUE REQUIEREN CAUTELA
# ============================================================

def clasificar_intervalo(row):

    año = int(row["Año"])

    if año == 2010:
        return "Inicio de serie"

    if año == 2015:
        return "Intervalo 2013–2015; no existe corte seleccionado 2014"

    if año == 2019:
        return "Ruptura de actualización DENUE asociada al levantamiento censal 2019"

    if año == 2024:
        return "Actualización amplia DENUE y transición SCIAN 2018–2023"

    return "Comparación entre cortes consecutivos seleccionados"


serie_general["Nota_intervalo"] = (
    serie_general.apply(
        clasificar_intervalo,
        axis=1
    )
)

display(
    serie_general[
        [
            "Año",
            "Establecimientos",
            "Cambio_absoluto",
            "Cambio_porcentual",
            "Años_desde_corte_anterior",
            "Nota_intervalo"
        ]
    ]
)

,Año,Establecimientos,Cambio_absoluto,Cambio_porcentual,Años_desde_corte_anterior,Nota_intervalo
0,2010,34656,NaN,NaN,NaN,Inicio de serie
1,2011,35093,437.0,1.260965,1.0,Comparación entre cortes consecutivos seleccio...
2,2012,35289,196.0,0.558516,1.0,Comparación entre cortes consecutivos seleccio...
3,2013,35797,508.0,1.439542,1.0,Comparación entre cortes consecutivos seleccio...
4,2015,37677,1880.0,5.251837,2.0,Intervalo 2013–2015; no existe corte seleccion...
5,2016,38372,695.0,1.844627,1.0,Comparación entre cortes consecutivos seleccio...
6,2017,38536,164.0,0.427395,1.0,Comparación entre cortes consecutivos seleccio...
7,2018,38728,192.0,0.498235,1.0,Comparación entre cortes consecutivos seleccio...
8,2019,40484,1756.0,4.534187,1.0,Ruptura de actualización DENUE asociada al lev...
9,2020,41448,964.0,2.381188,1.0,Comparación entre cortes consecutivos seleccio...


In [265]:
serie_general.to_csv(
    RESULTADOS /
    "SERIE_GENERAL_HOMOLOGADA_MORELIA_2010_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Serie general homologada guardada.")

✓ Serie general homologada guardada.


## Evolución diferenciada de comercio y servicios

Una vez construida la serie general homologada, se analiza por separado la
evolución de los establecimientos de comercio al por menor y de servicios.

Esta separación permite identificar si las variaciones observadas en el total
provienen principalmente del sector comercial, del conjunto de actividades
de servicios o de ambos.

Los intervalos asociados con actualizaciones amplias del DENUE se mantienen
identificados y no se interpretan automáticamente como crecimiento o
contracción económica.

In [266]:
# ============================================================
# FASE 3 - PASO 3
# SERIE HISTÓRICA: COMERCIO VS SERVICIOS
# ============================================================

# Asegurar sector como texto
inventario_homologado_final["Sector"] = (
    inventario_homologado_final["Sector"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Clasificar macrogrupo
# ------------------------------------------------------------

inventario_homologado_final["Macrogrupo"] = np.where(
    inventario_homologado_final["Sector"] == "46",
    "Comercio al por menor",
    "Servicios"
)

# ------------------------------------------------------------
# Sumar establecimientos por año y macrogrupo
# ------------------------------------------------------------

serie_macrogrupos = (
    inventario_homologado_final
    .groupby(
        [
            "Año",
            "Macrogrupo"
        ],
        as_index=False
    )["Establecimientos"]
    .sum()
)

serie_macrogrupos["Año"] = (
    serie_macrogrupos["Año"]
    .astype(int)
)

# ------------------------------------------------------------
# Variación dentro de cada macrogrupo
# ------------------------------------------------------------

serie_macrogrupos = (
    serie_macrogrupos
    .sort_values(
        [
            "Macrogrupo",
            "Año"
        ]
    )
    .reset_index(drop=True)
)

serie_macrogrupos["Cambio_absoluto"] = (
    serie_macrogrupos
    .groupby("Macrogrupo")["Establecimientos"]
    .diff()
)

serie_macrogrupos["Cambio_porcentual"] = (
    serie_macrogrupos
    .groupby("Macrogrupo")["Establecimientos"]
    .pct_change()
    * 100
)

serie_macrogrupos["Años_desde_corte_anterior"] = (
    serie_macrogrupos
    .groupby("Macrogrupo")["Año"]
    .diff()
)

display(serie_macrogrupos)

,Año,Macrogrupo,Establecimientos,Cambio_absoluto,Cambio_porcentual,Años_desde_corte_anterior
0,2010,Comercio al por menor,16693,NaN,NaN,NaN
1,2011,Comercio al por menor,16824,131.0,0.784760,1.0
2,2012,Comercio al por menor,16860,36.0,0.213980,1.0
3,2013,Comercio al por menor,17045,185.0,1.097272,1.0
4,2015,Comercio al por menor,16955,-90.0,-0.528014,2.0
5,2016,Comercio al por menor,17313,358.0,2.111472,1.0
6,2017,Comercio al por menor,17427,114.0,0.658465,1.0
7,2018,Comercio al por menor,17452,25.0,0.143456,1.0
8,2019,Comercio al por menor,17642,190.0,1.088700,1.0
9,2020,Comercio al por menor,17984,342.0,1.938556,1.0


In [267]:
# ============================================================
# FASE 3 - PASO 4
# TABLA COMPARATIVA COMERCIO VS SERVICIOS
# ============================================================

tabla_macrogrupos = (
    serie_macrogrupos.pivot(
        index="Año",
        columns="Macrogrupo",
        values="Establecimientos"
    )
    .reset_index()
)

tabla_macrogrupos["Total"] = (
    tabla_macrogrupos[
        "Comercio al por menor"
    ]
    +
    tabla_macrogrupos[
        "Servicios"
    ]
)

display(tabla_macrogrupos)

Macrogrupo,Año,Comercio al por menor,Servicios,Total
0,2010,16693,17963,34656
1,2011,16824,18269,35093
2,2012,16860,18429,35289
3,2013,17045,18752,35797
4,2015,16955,20722,37677
5,2016,17313,21059,38372
6,2017,17427,21109,38536
7,2018,17452,21276,38728
8,2019,17642,22842,40484
9,2020,17984,23464,41448


In [268]:
# ============================================================
# FASE 3 - PASO 5
# CONTROL: COMERCIO + SERVICIOS = TOTAL HOMOLOGADO
# ============================================================

control_macro = (
    tabla_macrogrupos[
        [
            "Año",
            "Total"
        ]
    ]
    .merge(
        serie_general[
            [
                "Año",
                "Establecimientos"
            ]
        ],
        on="Año",
        how="left"
    )
)

control_macro["Diferencia"] = (
    control_macro["Total"]
    -
    control_macro["Establecimientos"]
)

display(control_macro)

print(
    "\nAños con coincidencia exacta:",
    (control_macro["Diferencia"] == 0).sum(),
    "de",
    len(control_macro)
)

print(
    "Diferencia absoluta acumulada:",
    control_macro["Diferencia"].abs().sum()
)

,Año,Total,Establecimientos,Diferencia
0,2010,34656,34656,0
1,2011,35093,35093,0
2,2012,35289,35289,0
3,2013,35797,35797,0
4,2015,37677,37677,0
5,2016,38372,38372,0
6,2017,38536,38536,0
7,2018,38728,38728,0
8,2019,40484,40484,0
9,2020,41448,41448,0



Años con coincidencia exacta: 16 de 16
Diferencia absoluta acumulada: 0


In [269]:
# ============================================================
# FASE 3 - PASO 6
# GUARDAR SERIES DE MACROGRUPOS
# ============================================================

serie_macrogrupos.to_csv(
    RESULTADOS /
    "SERIE_HISTORICA_COMERCIO_VS_SERVICIOS_2010_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_macrogrupos.to_csv(
    RESULTADOS /
    "TABLA_COMPARATIVA_COMERCIO_VS_SERVICIOS_2010_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

control_macro.to_csv(
    RESULTADOS /
    "CONTROL_COMERCIO_SERVICIOS_VS_TOTAL.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Series de comercio y servicios guardadas.")

✓ Series de comercio y servicios guardadas.


## Dinámica histórica por sector de servicios

El comportamiento agregado de los servicios presenta variaciones importantes
en determinados cortes de la serie, particularmente en 2015, 2019, 2021 y
2024.

Para identificar el origen sectorial de estas variaciones, el conjunto de
servicios se desagrega por sector SCIAN. Los sectores 48 y 49 se integran
como el sector combinado 48-49, Transportes, correos y almacenamiento,
de acuerdo con la estructura sectorial del SCIAN.

El análisis se realiza sobre los establecimientos observados en cada corte y
no atribuye automáticamente las variaciones a nacimientos, cierres o
crecimiento económico, especialmente en los intervalos asociados con
actualizaciones amplias del DENUE.

In [270]:
# ============================================================
# FASE 3 - PASO 7
# SERIE HISTÓRICA POR SECTOR DE SERVICIOS
# ============================================================

servicios_sector = inventario_homologado_final[
    inventario_homologado_final["Sector"] != "46"
].copy()

# SCIAN considera 48-49 como un sector conjunto
servicios_sector["Sector_analisis"] = (
    servicios_sector["Sector"]
    .replace({
        "48": "48-49",
        "49": "48-49"
    })
)

nombres_sector = {
    "48-49": "Transportes, correos y almacenamiento",
    "51": "Información en medios masivos",
    "52": "Servicios financieros y de seguros",
    "53": "Servicios inmobiliarios y de alquiler",
    "54": "Servicios profesionales, científicos y técnicos",
    "56": "Servicios de apoyo a los negocios y manejo de residuos",
    "61": "Servicios educativos",
    "62": "Servicios de salud y asistencia social",
    "71": "Servicios de esparcimiento, culturales y deportivos",
    "72": "Alojamiento temporal y preparación de alimentos y bebidas",
    "81": "Otros servicios excepto actividades gubernamentales"
}

servicios_sector["Nombre_sector"] = (
    servicios_sector["Sector_analisis"]
    .map(nombres_sector)
)

serie_servicios_sector = (
    servicios_sector
    .groupby(
        [
            "Año",
            "Sector_analisis",
            "Nombre_sector"
        ],
        as_index=False
    )["Establecimientos"]
    .sum()
)

serie_servicios_sector["Año"] = (
    serie_servicios_sector["Año"]
    .astype(int)
)

serie_servicios_sector = (
    serie_servicios_sector
    .sort_values(
        [
            "Sector_analisis",
            "Año"
        ]
    )
    .reset_index(drop=True)
)

serie_servicios_sector["Cambio_absoluto"] = (
    serie_servicios_sector
    .groupby("Sector_analisis")["Establecimientos"]
    .diff()
)

serie_servicios_sector["Cambio_porcentual"] = (
    serie_servicios_sector
    .groupby("Sector_analisis")["Establecimientos"]
    .pct_change()
    * 100
)

display(serie_servicios_sector)

,Año,Sector_analisis,Nombre_sector,Establecimientos,Cambio_absoluto,Cambio_porcentual
0,2010,48-49,"Transportes, correos y almacenamiento",297,NaN,NaN
1,2011,48-49,"Transportes, correos y almacenamiento",327,30.0,10.101010
2,2012,48-49,"Transportes, correos y almacenamiento",341,14.0,4.281346
3,2013,48-49,"Transportes, correos y almacenamiento",459,118.0,34.604106
4,2015,48-49,"Transportes, correos y almacenamiento",224,-235.0,-51.198257
...,...,...,...,...,...,...
171,2022,81,Otros servicios excepto actividades gubernamen...,7316,1.0,0.013671
172,2023,81,Otros servicios excepto actividades gubernamen...,7317,1.0,0.013669
173,2024,81,Otros servicios excepto actividades gubernamen...,8433,1116.0,15.252153
174,2025,81,Otros servicios excepto actividades gubernamen...,8559,126.0,1.494130


In [271]:
# ============================================================
# FASE 3 - PASO 8
# TABLA AÑO × SECTOR DE SERVICIOS
# ============================================================

tabla_servicios_sector = (
    serie_servicios_sector.pivot(
        index="Año",
        columns="Sector_analisis",
        values="Establecimientos"
    )
    .reset_index()
)

display(tabla_servicios_sector)

Sector_analisis,Año,48-49,51,52,53,54,56,61,62,71,72,81
0,2010,297,141,189,651,1352,1064,1088,2724,520,4357,5580
1,2011,327,166,398,654,1364,1070,1089,2732,521,4365,5583
2,2012,341,174,402,659,1420,1083,1100,2747,526,4386,5591
3,2013,459,181,410,666,1479,1107,1107,2760,526,4407,5650
4,2015,224,143,589,671,1230,1068,1224,3009,520,5569,6475
5,2016,251,182,616,676,1263,1074,1275,3071,559,5597,6495
6,2017,253,189,621,681,1277,1081,1277,3074,559,5604,6493
7,2018,253,189,788,681,1277,1080,1277,3072,559,5607,6493
8,2019,270,136,723,700,1289,832,1298,3325,631,6357,7281
9,2020,272,191,914,730,1304,874,1385,3404,591,6448,7351


In [277]:
# ============================================================
# FASE 3 - PASO 9 CORREGIDO COMPLETO
# ¿QUÉ SECTORES EXPLICAN 2015, 2019, 2021 Y 2024?
# ============================================================

años_clave = [2015, 2019, 2021, 2024]


# ------------------------------------------------------------
# 1. TOMAR LOS CAMBIOS SECTORIALES DE LOS AÑOS CLAVE
# ------------------------------------------------------------

cambios_clave = (
    serie_servicios_sector[
        serie_servicios_sector["Año"].isin(años_clave)
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. CAMBIO TOTAL DE SERVICIOS EN CADA AÑO CLAVE
# ------------------------------------------------------------

cambio_total_servicios = (
    serie_macrogrupos[
        (
            serie_macrogrupos["Macrogrupo"] == "Servicios"
        )
        &
        (
            serie_macrogrupos["Año"].isin(años_clave)
        )
    ]
    [
        [
            "Año",
            "Cambio_absoluto"
        ]
    ]
    .rename(
        columns={
            "Cambio_absoluto": "Cambio_total_servicios"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 3. UNIR CAMBIO SECTORIAL CON CAMBIO TOTAL DE SERVICIOS
# ------------------------------------------------------------

cambios_clave = cambios_clave.merge(
    cambio_total_servicios,
    on="Año",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 4. CALCULAR APORTE DE CADA SECTOR
# ------------------------------------------------------------

cambios_clave[
    "Aporte_pct_al_cambio_servicios"
] = (
    cambios_clave["Cambio_absoluto"]
    /
    cambios_clave["Cambio_total_servicios"]
    * 100
)

cambios_clave[
    "Magnitud_cambio"
] = (
    cambios_clave["Cambio_absoluto"].abs()
)


# ------------------------------------------------------------
# 5. ORDENAR
# ------------------------------------------------------------

cambios_clave_ordenados = (
    cambios_clave
    .sort_values(
        by=[
            "Año",
            "Magnitud_cambio"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. MOSTRAR
# ------------------------------------------------------------

display(
    cambios_clave_ordenados[
        [
            "Año",
            "Sector_analisis",
            "Nombre_sector",
            "Establecimientos",
            "Cambio_absoluto",
            "Cambio_porcentual",
            "Cambio_total_servicios",
            "Aporte_pct_al_cambio_servicios"
        ]
    ]
)


# ------------------------------------------------------------
# 7. GUARDAR
# ------------------------------------------------------------

cambios_clave_ordenados.to_csv(
    RESULTADOS /
    "SECTORES_EXPLICATIVOS_CAMBIOS_CLAVE_2015_2019_2021_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Análisis sectorial de cambios clave guardado.")

,Año,Sector_analisis,Nombre_sector,Establecimientos,Cambio_absoluto,Cambio_porcentual,Cambio_total_servicios,Aporte_pct_al_cambio_servicios
0,2015,72,Alojamiento temporal y preparación de alimento...,5569,1162.0,26.367143,1970.0,58.984772
1,2015,81,Otros servicios excepto actividades gubernamen...,6475,825.0,14.601770,1970.0,41.878173
2,2015,54,"Servicios profesionales, científicos y técnicos",1230,-249.0,-16.835700,1970.0,-12.639594
3,2015,62,Servicios de salud y asistencia social,3009,249.0,9.021739,1970.0,12.639594
4,2015,48-49,"Transportes, correos y almacenamiento",224,-235.0,-51.198257,1970.0,-11.928934
5,2015,52,Servicios financieros y de seguros,589,179.0,43.658537,1970.0,9.086294
6,2015,61,Servicios educativos,1224,117.0,10.569106,1970.0,5.939086
7,2015,56,Servicios de apoyo a los negocios y manejo de ...,1068,-39.0,-3.523035,1970.0,-1.979695
8,2015,51,Información en medios masivos,143,-38.0,-20.994475,1970.0,-1.928934
9,2015,71,"Servicios de esparcimiento, culturales y depor...",520,-6.0,-1.140684,1970.0,-0.304569


✓ Análisis sectorial de cambios clave guardado.


In [278]:
# ============================================================
# FASE 3 - PASO 11
# MEJORAR INTERPRETACIÓN DE LA CONTRIBUCIÓN SECTORIAL
# ============================================================

cambios_clave_ordenados[
    "Contribucion_al_cambio_neto_pct"
] = (
    cambios_clave_ordenados[
        "Aporte_pct_al_cambio_servicios"
    ]
)

cambios_clave_ordenados["Sentido"] = np.select(
    [
        cambios_clave_ordenados["Cambio_absoluto"] > 0,
        cambios_clave_ordenados["Cambio_absoluto"] < 0
    ],
    [
        "Aumento",
        "Disminución"
    ],
    default="Sin cambio"
)

display(
    cambios_clave_ordenados[
        [
            "Año",
            "Sector_analisis",
            "Nombre_sector",
            "Cambio_absoluto",
            "Sentido",
            "Cambio_total_servicios",
            "Contribucion_al_cambio_neto_pct"
        ]
    ]
)

,Año,Sector_analisis,Nombre_sector,Cambio_absoluto,Sentido,Cambio_total_servicios,Contribucion_al_cambio_neto_pct
0,2015,72,Alojamiento temporal y preparación de alimento...,1162.0,Aumento,1970.0,58.984772
1,2015,81,Otros servicios excepto actividades gubernamen...,825.0,Aumento,1970.0,41.878173
2,2015,54,"Servicios profesionales, científicos y técnicos",-249.0,Disminución,1970.0,-12.639594
3,2015,62,Servicios de salud y asistencia social,249.0,Aumento,1970.0,12.639594
4,2015,48-49,"Transportes, correos y almacenamiento",-235.0,Disminución,1970.0,-11.928934
5,2015,52,Servicios financieros y de seguros,179.0,Aumento,1970.0,9.086294
6,2015,61,Servicios educativos,117.0,Aumento,1970.0,5.939086
7,2015,56,Servicios de apoyo a los negocios y manejo de ...,-39.0,Disminución,1970.0,-1.979695
8,2015,51,Información en medios masivos,-38.0,Disminución,1970.0,-1.928934
9,2015,71,"Servicios de esparcimiento, culturales y depor...",-6.0,Disminución,1970.0,-0.304569


In [279]:
# ============================================================
# FASE 3 - PASO 12
# TOP 3 AUMENTOS Y DISMINUCIONES POR CORTE CLAVE
# ============================================================

filas_top = []

for año in años_clave:

    temp = cambios_clave_ordenados[
        cambios_clave_ordenados["Año"] == año
    ].copy()

    # --------------------------------------------------------
    # TOP 3 AUMENTOS
    # --------------------------------------------------------

    aumentos = (
        temp[
            temp["Cambio_absoluto"] > 0
        ]
        .nlargest(
            3,
            "Cambio_absoluto"
        )
    )

    for posicion, (_, fila) in enumerate(
        aumentos.iterrows(),
        start=1
    ):

        filas_top.append({
            "Año": año,
            "Sentido": "Aumento",
            "Posicion": posicion,
            "Sector": fila["Sector_analisis"],
            "Nombre_sector": fila["Nombre_sector"],
            "Cambio_absoluto": fila["Cambio_absoluto"],
            "Contribucion_pct":
                fila["Contribucion_al_cambio_neto_pct"]
        })

    # --------------------------------------------------------
    # TOP 3 DISMINUCIONES
    # --------------------------------------------------------

    disminuciones = (
        temp[
            temp["Cambio_absoluto"] < 0
        ]
        .nsmallest(
            3,
            "Cambio_absoluto"
        )
    )

    for posicion, (_, fila) in enumerate(
        disminuciones.iterrows(),
        start=1
    ):

        filas_top.append({
            "Año": año,
            "Sentido": "Disminución",
            "Posicion": posicion,
            "Sector": fila["Sector_analisis"],
            "Nombre_sector": fila["Nombre_sector"],
            "Cambio_absoluto": fila["Cambio_absoluto"],
            "Contribucion_pct":
                fila["Contribucion_al_cambio_neto_pct"]
        })


top_cambios_sectoriales = pd.DataFrame(
    filas_top
)

display(top_cambios_sectoriales)

,Año,Sentido,Posicion,Sector,Nombre_sector,Cambio_absoluto,Contribucion_pct
0,2015,Aumento,1,72,Alojamiento temporal y preparación de alimento...,1162.0,58.984772
1,2015,Aumento,2,81,Otros servicios excepto actividades gubernamen...,825.0,41.878173
2,2015,Aumento,3,62,Servicios de salud y asistencia social,249.0,12.639594
3,2015,Disminución,1,54,"Servicios profesionales, científicos y técnicos",-249.0,-12.639594
4,2015,Disminución,2,48-49,"Transportes, correos y almacenamiento",-235.0,-11.928934
5,2015,Disminución,3,56,Servicios de apoyo a los negocios y manejo de ...,-39.0,-1.979695
6,2019,Aumento,1,81,Otros servicios excepto actividades gubernamen...,788.0,50.319285
7,2019,Aumento,2,72,Alojamiento temporal y preparación de alimento...,750.0,47.892720
8,2019,Aumento,3,62,Servicios de salud y asistencia social,253.0,16.155811
9,2019,Disminución,1,56,Servicios de apoyo a los negocios y manejo de ...,-248.0,-15.836526


In [280]:
top_cambios_sectoriales.to_csv(
    RESULTADOS /
    "TOP3_AUMENTOS_DISMINUCIONES_SERVICIOS_CORTES_CLAVE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Top 3 sectorial guardado.")

✓ Top 3 sectorial guardado.


## Actividades específicas asociadas con los cambios sectoriales

Los sectores 72, 81 y 62 presentan una participación recurrente en las
principales variaciones observadas en los cortes de 2015, 2019, 2021 y 2024.

Para identificar las actividades que explican estos movimientos, el análisis
se desagrega al nivel de unidad homologada.

La comparación utiliza las diferencias entre cortes consecutivos disponibles.
En particular, la variación de 2015 corresponde al intervalo 2013–2015 y,
por tanto, representa un cambio acumulado de dos años y no una variación anual.

In [281]:
# ============================================================
# FASE 3 - PASO 13
# PANEL UNIDAD-AÑO PARA SECTORES 72, 81 Y 62
# ============================================================

sectores_clave = ["72", "81", "62"]

base_clave = (
    inventario_homologado_final[
        inventario_homologado_final["Sector"]
        .astype(str)
        .isin(sectores_clave)
    ]
    .copy()
)

base_clave["Año"] = (
    base_clave["Año"]
    .astype(int)
)

base_clave["Sector"] = (
    base_clave["Sector"]
    .astype(str)
)

# ------------------------------------------------------------
# Total real por unidad-año-sector
# ------------------------------------------------------------

conteos_unidad_sector = (
    base_clave.groupby(
        [
            "Sector",
            "Unidad_homologada",
            "Año"
        ],
        as_index=False
    )["Establecimientos"]
    .sum()
)

# ------------------------------------------------------------
# Crear una malla completa por sector:
# unidad × todos los años
# ------------------------------------------------------------

filas_panel = []

for sector in sectores_clave:

    unidades_sector = sorted(
        base_clave.loc[
            base_clave["Sector"] == sector,
            "Unidad_homologada"
        ]
        .dropna()
        .unique()
    )

    malla = pd.MultiIndex.from_product(
        [
            [sector],
            unidades_sector,
            años
        ],
        names=[
            "Sector",
            "Unidad_homologada",
            "Año"
        ]
    ).to_frame(index=False)

    filas_panel.append(malla)

panel_unidad_sector = pd.concat(
    filas_panel,
    ignore_index=True
)

panel_unidad_sector = panel_unidad_sector.merge(
    conteos_unidad_sector,
    on=[
        "Sector",
        "Unidad_homologada",
        "Año"
    ],
    how="left",
    validate="one_to_one"
)

panel_unidad_sector["Establecimientos"] = (
    panel_unidad_sector["Establecimientos"]
    .fillna(0)
    .astype(int)
)

print(
    "Filas panel:",
    len(panel_unidad_sector)
)

print(
    "Nulos:",
    panel_unidad_sector["Establecimientos"]
    .isna().sum()
)

Filas panel: 1760
Nulos: 0


In [282]:
# ============================================================
# FASE 3 - PASO 14
# CAMBIO ENTRE CORTES POR UNIDAD HOMOLOGADA
# ============================================================

panel_unidad_sector = (
    panel_unidad_sector
    .sort_values(
        [
            "Sector",
            "Unidad_homologada",
            "Año"
        ]
    )
    .reset_index(drop=True)
)

panel_unidad_sector["Cambio_absoluto"] = (
    panel_unidad_sector
    .groupby(
        [
            "Sector",
            "Unidad_homologada"
        ]
    )["Establecimientos"]
    .diff()
)

panel_unidad_sector[
    "Años_desde_corte_anterior"
] = (
    panel_unidad_sector
    .groupby(
        [
            "Sector",
            "Unidad_homologada"
        ]
    )["Año"]
    .diff()
)

display(
    panel_unidad_sector.head(30)
)

,Sector,Unidad_homologada,Año,Establecimientos,Cambio_absoluto,Años_desde_corte_anterior
0,62,FL04,2010,940,NaN,NaN
1,62,FL04,2011,941,1.0,1.0
2,62,FL04,2012,943,2.0,1.0
3,62,FL04,2013,949,6.0,1.0
4,62,FL04,2015,990,41.0,2.0
5,62,FL04,2016,1030,40.0,1.0
6,62,FL04,2017,1032,2.0,1.0
7,62,FL04,2018,1030,-2.0,1.0
8,62,FL04,2019,990,-40.0,1.0
9,62,FL04,2020,1034,44.0,1.0


In [283]:
# ============================================================
# FASE 3 - PASO 15
# TOP UNIDADES HOMOLOGADAS EN 2015, 2019, 2021 Y 2024
# ============================================================

años_clave_unidades = [
    2015,
    2019,
    2021,
    2024
]

filas_top_unidades = []

for año in años_clave_unidades:

    for sector in sectores_clave:

        temp = panel_unidad_sector[
            (
                panel_unidad_sector["Año"] == año
            )
            &
            (
                panel_unidad_sector["Sector"] == sector
            )
        ].copy()

        # Top 5 aumentos
        aumentos = (
            temp[
                temp["Cambio_absoluto"] > 0
            ]
            .nlargest(
                5,
                "Cambio_absoluto"
            )
        )

        for pos, (_, fila) in enumerate(
            aumentos.iterrows(),
            start=1
        ):

            filas_top_unidades.append({
                "Año": año,
                "Sector": sector,
                "Sentido": "Aumento",
                "Posicion": pos,
                "Unidad_homologada":
                    fila["Unidad_homologada"],
                "Establecimientos":
                    fila["Establecimientos"],
                "Cambio_absoluto":
                    fila["Cambio_absoluto"],
                "Años_intervalo":
                    fila["Años_desde_corte_anterior"]
            })

        # Top 5 disminuciones
        disminuciones = (
            temp[
                temp["Cambio_absoluto"] < 0
            ]
            .nsmallest(
                5,
                "Cambio_absoluto"
            )
        )

        for pos, (_, fila) in enumerate(
            disminuciones.iterrows(),
            start=1
        ):

            filas_top_unidades.append({
                "Año": año,
                "Sector": sector,
                "Sentido": "Disminución",
                "Posicion": pos,
                "Unidad_homologada":
                    fila["Unidad_homologada"],
                "Establecimientos":
                    fila["Establecimientos"],
                "Cambio_absoluto":
                    fila["Cambio_absoluto"],
                "Años_intervalo":
                    fila["Años_desde_corte_anterior"]
            })


top_unidades_clave = pd.DataFrame(
    filas_top_unidades
)

display(
    top_unidades_clave.sort_values(
        [
            "Año",
            "Sector",
            "Sentido",
            "Posicion"
        ]
    )
)

,Año,Sector,Sentido,Posicion,Unidad_homologada,Establecimientos,Cambio_absoluto,Años_intervalo
19,2015,62,Aumento,1,UH0221,1055,141.0,2.0
20,2015,62,Aumento,2,FL04,990,41.0,2.0
21,2015,62,Aumento,3,UH0225,100,21.0,2.0
22,2015,62,Aumento,4,UH0230,77,20.0,2.0
23,2015,62,Aumento,5,UH0229,36,14.0,2.0
...,...,...,...,...,...,...,...,...
95,2024,81,Disminución,1,UH0323,92,-18.0,1.0
96,2024,81,Disminución,2,UH0344,30,-12.0,1.0
97,2024,81,Disminución,3,UH0331,97,-11.0,1.0
98,2024,81,Disminución,4,UH0349,104,-10.0,1.0


In [284]:
# ============================================================
# FASE 3 - PASO 16
# AÑADIR CÓDIGOS SCIAN A LAS UNIDADES DESTACADAS
# ============================================================

codigos_por_unidad = (
    diccionario_350_completo.groupby(
        "Unidad_homologada"
    )["SCIAN_original"]
    .agg(
        lambda x:
        " | ".join(
            sorted(
                set(
                    str(v).strip()
                    for v in x
                )
            )
        )
    )
    .reset_index(
        name="Codigos_SCIAN"
    )
)

top_unidades_clave = (
    top_unidades_clave.merge(
        codigos_por_unidad,
        on="Unidad_homologada",
        how="left",
        validate="many_to_one"
    )
)

display(
    top_unidades_clave[
        [
            "Año",
            "Sector",
            "Sentido",
            "Posicion",
            "Unidad_homologada",
            "Codigos_SCIAN",
            "Cambio_absoluto",
            "Establecimientos",
            "Años_intervalo"
        ]
    ]
    .sort_values(
        [
            "Año",
            "Sector",
            "Sentido",
            "Posicion"
        ]
    )
)

,Año,Sector,Sentido,Posicion,Unidad_homologada,Codigos_SCIAN,Cambio_absoluto,Establecimientos,Años_intervalo
19,2015,62,Aumento,1,UH0221,621211,141.0,1055,2.0
20,2015,62,Aumento,2,FL04,621111 | 621113 | 621115,41.0,990,2.0
21,2015,62,Aumento,3,UH0225,621331,21.0,100,2.0
22,2015,62,Aumento,4,UH0230,621398,20.0,77,2.0
23,2015,62,Aumento,5,UH0229,621391,14.0,36,2.0
...,...,...,...,...,...,...,...,...,...
95,2024,81,Disminución,1,UH0323,811211,-18.0,92,1.0
96,2024,81,Disminución,2,UH0344,813110,-12.0,30,1.0
97,2024,81,Disminución,3,UH0331,811430,-11.0,97,1.0
98,2024,81,Disminución,4,UH0349,813230,-10.0,104,1.0


In [285]:
top_unidades_clave.to_csv(
    RESULTADOS /
    "TOP_UNIDADES_HOMOLOGADAS_SECTORES_72_81_62_CORTES_CLAVE.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Unidades explicativas de los sectores 72, 81 y 62 guardadas.")

✓ Unidades explicativas de los sectores 72, 81 y 62 guardadas.


### Principales unidades responsables de las variaciones sectoriales

Para facilitar la interpretación, las variaciones a nivel de unidad
homologada se resumen por sector y corte.

Para cada combinación año-sector se identifica la unidad con el mayor
incremento, la unidad con la mayor disminución y la contribución conjunta
de los tres mayores aumentos y disminuciones al cambio neto del sector.

Este procedimiento permite distinguir entre variaciones concentradas en
pocas actividades y movimientos distribuidos entre numerosas unidades.

In [286]:
# ============================================================
# FASE 3 - PASO 17
# RESUMEN COMPACTO DE PRINCIPALES UNIDADES EXPLICATIVAS
# ============================================================

# Cambio total de cada sector en los años clave
cambio_sector_clave = (
    serie_servicios_sector[
        (
            serie_servicios_sector["Año"]
            .isin(años_clave_unidades)
        )
        &
        (
            serie_servicios_sector["Sector_analisis"]
            .isin(sectores_clave)
        )
    ]
    [
        [
            "Año",
            "Sector_analisis",
            "Cambio_absoluto"
        ]
    ]
    .rename(
        columns={
            "Sector_analisis": "Sector",
            "Cambio_absoluto": "Cambio_sector"
        }
    )
)

filas_resumen = []

for año in años_clave_unidades:

    for sector in sectores_clave:

        temp = panel_unidad_sector[
            (
                panel_unidad_sector["Año"] == año
            )
            &
            (
                panel_unidad_sector["Sector"] == sector
            )
        ].copy()

        # Cambio total del sector
        fila_cambio = cambio_sector_clave[
            (
                cambio_sector_clave["Año"] == año
            )
            &
            (
                cambio_sector_clave["Sector"] == sector
            )
        ]

        if fila_cambio.empty:
            continue

        cambio_sector = float(
            fila_cambio["Cambio_sector"].iloc[0]
        )

        # -----------------------------
        # AUMENTOS
        # -----------------------------
        aumentos = (
            temp[temp["Cambio_absoluto"] > 0]
            .nlargest(3, "Cambio_absoluto")
        )

        if not aumentos.empty:

            principal_aumento = aumentos.iloc[0]

            unidad_aumento = (
                principal_aumento["Unidad_homologada"]
            )

            cambio_aumento = float(
                principal_aumento["Cambio_absoluto"]
            )

            top3_aumentos = float(
                aumentos["Cambio_absoluto"].sum()
            )

        else:

            unidad_aumento = ""
            cambio_aumento = 0
            top3_aumentos = 0

        # -----------------------------
        # DISMINUCIONES
        # -----------------------------
        disminuciones = (
            temp[temp["Cambio_absoluto"] < 0]
            .nsmallest(3, "Cambio_absoluto")
        )

        if not disminuciones.empty:

            principal_disminucion = (
                disminuciones.iloc[0]
            )

            unidad_disminucion = (
                principal_disminucion[
                    "Unidad_homologada"
                ]
            )

            cambio_disminucion = float(
                principal_disminucion[
                    "Cambio_absoluto"
                ]
            )

            top3_disminuciones = float(
                disminuciones[
                    "Cambio_absoluto"
                ].sum()
            )

        else:

            unidad_disminucion = ""
            cambio_disminucion = 0
            top3_disminuciones = 0

        filas_resumen.append({
            "Año": año,
            "Sector": sector,

            "Cambio_sector":
                cambio_sector,

            "Principal_aumento":
                unidad_aumento,

            "Cambio_principal_aumento":
                cambio_aumento,

            "Top3_aumentos":
                top3_aumentos,

            "Principal_disminucion":
                unidad_disminucion,

            "Cambio_principal_disminucion":
                cambio_disminucion,

            "Top3_disminuciones":
                top3_disminuciones
        })


resumen_unidades_clave = pd.DataFrame(
    filas_resumen
)

In [287]:
# ============================================================
# FASE 3 - PASO 18
# AÑADIR CÓDIGOS A LAS UNIDADES PRINCIPALES
# ============================================================

dicc_codigos = dict(
    zip(
        codigos_por_unidad[
            "Unidad_homologada"
        ],
        codigos_por_unidad[
            "Codigos_SCIAN"
        ]
    )
)

resumen_unidades_clave[
    "Codigos_principal_aumento"
] = (
    resumen_unidades_clave[
        "Principal_aumento"
    ].map(dicc_codigos)
)

resumen_unidades_clave[
    "Codigos_principal_disminucion"
] = (
    resumen_unidades_clave[
        "Principal_disminucion"
    ].map(dicc_codigos)
)


# Contribución del principal movimiento al cambio sectorial
resumen_unidades_clave[
    "Aporte_principal_aumento_pct"
] = np.where(
    resumen_unidades_clave[
        "Cambio_sector"
    ] != 0,

    resumen_unidades_clave[
        "Cambio_principal_aumento"
    ]
    /
    resumen_unidades_clave[
        "Cambio_sector"
    ]
    * 100,

    np.nan
)

resumen_unidades_clave[
    "Aporte_principal_disminucion_pct"
] = np.where(
    resumen_unidades_clave[
        "Cambio_sector"
    ] != 0,

    resumen_unidades_clave[
        "Cambio_principal_disminucion"
    ]
    /
    resumen_unidades_clave[
        "Cambio_sector"
    ]
    * 100,

    np.nan
)


display(
    resumen_unidades_clave[
        [
            "Año",
            "Sector",
            "Cambio_sector",

            "Principal_aumento",
            "Codigos_principal_aumento",
            "Cambio_principal_aumento",

            "Principal_disminucion",
            "Codigos_principal_disminucion",
            "Cambio_principal_disminucion",

            "Top3_aumentos",
            "Top3_disminuciones"
        ]
    ]
    .sort_values(
        [
            "Año",
            "Sector"
        ]
    )
    .reset_index(drop=True)
)

,Año,Sector,Cambio_sector,Principal_aumento,Codigos_principal_aumento,Cambio_principal_aumento,Principal_disminucion,Codigos_principal_disminucion,Cambio_principal_disminucion,Top3_aumentos,Top3_disminuciones
0,2015,62,249.0,UH0221,621211,141.0,UH0251,623992,-22.0,203.0,-44.0
1,2015,72,1162.0,FL06,722110 | 722219 | 722511 | 722512 | 722513 | 7...,1983.0,FL07,722212 | 722517 | 722518,-851.0,2093.0,-953.0
2,2015,81,825.0,UH0335,811499,186.0,UH0324,811219,-35.0,458.0,-78.0
3,2019,62,253.0,UH0221,621211,103.0,FL04,621111 | 621113 | 621115,-40.0,181.0,-74.0
4,2019,72,750.0,FL06,722110 | 722219 | 722511 | 722512 | 722513 | 7...,363.0,UH0302,721113,-10.0,714.0,-16.0
5,2019,81,788.0,UH0336,812110,561.0,UH0335,811499,-87.0,703.0,-141.0
6,2021,62,-34.0,UH0235,621511,4.0,UH0221,621211,-15.0,9.0,-33.0
7,2021,72,-42.0,UH0308,722310,3.0,FL06,722110 | 722219 | 722511 | 722512 | 722513 | 7...,-25.0,4.0,-45.0
8,2021,81,-36.0,UH0330,811420,1.0,UH0336,812110,-14.0,2.0,-22.0
9,2024,62,424.0,FL04,621111 | 621113 | 621115,134.0,UH0266,624411,-14.0,293.0,-30.0


In [288]:
resumen_unidades_clave.to_csv(
    RESULTADOS /
    "RESUMEN_UNIDADES_EXPLICATIVAS_SECTORES_62_72_81.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Resumen compacto de unidades explicativas guardado.")

✓ Resumen compacto de unidades explicativas guardado.


In [289]:
# ============================================================
# FASE 3 - PASO 19
# LOCALIZAR ARCHIVOS LIMPIOS 2018 Y 2019
# ============================================================

print("Archivos 2018:")
for archivo in sorted(MORELIA.glob("*2018*.csv")):
    print(" -", archivo.name)

print("\nArchivos 2019:")
for archivo in sorted(MORELIA.glob("*2019*.csv")):
    print(" -", archivo.name)

Archivos 2018:
 - DENUE_2018_MORELIA.csv

Archivos 2019:
 - DENUE_2019_MORELIA.csv


In [290]:
# ============================================================
# FASE 3 - PASO 20
# CARGAR DENUE 2018 Y 2019 Y AISLAR SCIAN 812110
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

archivo_2018 = MORELIA / "DENUE_2018_MORELIA.csv"
archivo_2019 = MORELIA / "DENUE_2019_MORELIA.csv"

df18 = pd.read_csv(
    archivo_2018,
    dtype=str,
    low_memory=False
)

df19 = pd.read_csv(
    archivo_2019,
    dtype=str,
    low_memory=False
)

print("2018 registros totales:", len(df18))
print("2019 registros totales:", len(df19))

print("\nColumnas 2018:")
print(df18.columns.tolist())

print("\nColumnas 2019:")
print(df19.columns.tolist())

2018 registros totales: 45900
2019 registros totales: 47934

Columnas 2018:
['id', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1', 'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext', 'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int', 'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local', 'cod_postal', 'cve_ent', 'entidad', 'cve_mun', 'municipio', 'cve_loc', 'localidad', 'ageb', 'manzana', 'telefono', 'correoelec', 'www', 'tipoUniEco', 'latitud', 'longitud', 'fecha_alta', 'municipio_norm']

Columnas 2019:
['id', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1', 'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext', 'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int', 'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local', 'cod_postal', 'cve_ent', 'entidad', 'cve_mun', 

In [291]:
# ============================================================
# FASE 3 - PASO 21
# DETECTAR COLUMNAS SCIAN E IDENTIFICADOR
# ============================================================

def detectar_columna(df, candidatos):

    for col in candidatos:
        if col in df.columns:
            return col

    return None


candidatos_scian = [
    "codigo_act",
    "SCIAN6",
    "Código de la clase de actividad",
    "Código de la clase de actividad SCIAN"
]

candidatos_id = [
    "id",
    "ID",
    "Id",
    "llave_denue",
    "Llave DENUE",
    "llave_DENUE"
]


scian18 = detectar_columna(
    df18,
    candidatos_scian
)

scian19 = detectar_columna(
    df19,
    candidatos_scian
)

id18 = detectar_columna(
    df18,
    candidatos_id
)

id19 = detectar_columna(
    df19,
    candidatos_id
)


print("SCIAN 2018:", scian18)
print("SCIAN 2019:", scian19)
print("ID 2018:", id18)
print("ID 2019:", id19)

SCIAN 2018: codigo_act
SCIAN 2019: codigo_act
ID 2018: id
ID 2019: id


In [292]:
# ============================================================
# FASE 3 - PASO 22
# FILTRAR SCIAN 812110
# ============================================================

codigo_objetivo = "812110"

d18 = df18[
    df18[scian18]
    .astype(str)
    .str.strip()
    .eq(codigo_objetivo)
].copy()

d19 = df19[
    df19[scian19]
    .astype(str)
    .str.strip()
    .eq(codigo_objetivo)
].copy()


print("812110 en 2018:", len(d18))
print("812110 en 2019:", len(d19))
print("Diferencia:", len(d19) - len(d18))

print("\nIDs nulos 2018:", d18[id18].isna().sum())
print("IDs nulos 2019:", d19[id19].isna().sum())

print(
    "IDs duplicados 2018:",
    d18[id18].duplicated().sum()
)

print(
    "IDs duplicados 2019:",
    d19[id19].duplicated().sum()
)

812110 en 2018: 1767
812110 en 2019: 2328
Diferencia: 561

IDs nulos 2018: 0
IDs nulos 2019: 0
IDs duplicados 2018: 0
IDs duplicados 2019: 0


In [293]:
# ============================================================
# FASE 3 - PASO 23
# COMPARAR 812110 ENTRE 2018 Y 2019 POR ID DENUE
# ============================================================

ids18 = set(
    d18[id18]
    .dropna()
    .astype(str)
    .str.strip()
)

ids19 = set(
    d19[id19]
    .dropna()
    .astype(str)
    .str.strip()
)

comunes = ids18 & ids19
solo18 = ids18 - ids19
solo19 = ids19 - ids18

print("IDs 2018:", len(ids18))
print("IDs 2019:", len(ids19))

print("\nPresentes en ambos cortes:", len(comunes))
print("Solo en 2018:", len(solo18))
print("Solo en 2019:", len(solo19))

print(
    "\nBalance solo2019 - solo2018:",
    len(solo19) - len(solo18)
)

IDs 2018: 1767
IDs 2019: 2328

Presentes en ambos cortes: 935
Solo en 2018: 832
Solo en 2019: 1393

Balance solo2019 - solo2018: 561


In [294]:
# ============================================================
# FASE 3 - PASO 23
# COMPARAR 812110 ENTRE 2018 Y 2019 POR ID DENUE
# ============================================================

ids18 = set(
    d18[id18]
    .dropna()
    .astype(str)
    .str.strip()
)

ids19 = set(
    d19[id19]
    .dropna()
    .astype(str)
    .str.strip()
)

comunes = ids18 & ids19
solo18 = ids18 - ids19
solo19 = ids19 - ids18

print("IDs 2018:", len(ids18))
print("IDs 2019:", len(ids19))

print("\nPresentes en ambos cortes:", len(comunes))
print("Solo en 2018:", len(solo18))
print("Solo en 2019:", len(solo19))

print(
    "\nBalance solo2019 - solo2018:",
    len(solo19) - len(solo18)
)

IDs 2018: 1767
IDs 2019: 2328

Presentes en ambos cortes: 935
Solo en 2018: 832
Solo en 2019: 1393

Balance solo2019 - solo2018: 561


In [295]:
# ============================================================
# FASE 3 - PASO 24
# BUSCAR POSIBLES MISMOS ESTABLECIMIENTOS CON CAMBIO DE ID
# MEDIANTE HUELLA EXACTA
# ============================================================

import re
import unicodedata


# ------------------------------------------------------------
# 1. AISLAR EXCLUSIVOS POR ID
# ------------------------------------------------------------

solo18_df = d18[
    d18[id18]
    .astype(str)
    .str.strip()
    .isin(solo18)
].copy()

solo19_df = d19[
    d19[id19]
    .astype(str)
    .str.strip()
    .isin(solo19)
].copy()

print("Exclusivos 2018 por ID:", len(solo18_df))
print("Exclusivos 2019 por ID:", len(solo19_df))


# ------------------------------------------------------------
# 2. FUNCIÓN DE NORMALIZACIÓN
# ------------------------------------------------------------

def normalizar_huella(valor):

    if pd.isna(valor):
        return ""

    texto = str(valor).strip().upper()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto


# ------------------------------------------------------------
# 3. DETECTAR COLUMNAS COMUNES ÚTILES
# ------------------------------------------------------------

candidatos_huella = [
    "nom_estab",
    "nom_vial",
    "numero_ext",
    "numero_int",
    "cod_postal",
    "localidad",
    "municipio",
    "latitud",
    "longitud"
]

columnas_huella = [
    col
    for col in candidatos_huella
    if col in solo18_df.columns
    and col in solo19_df.columns
]

print("\nColumnas utilizadas para huella:")
print(columnas_huella)

Exclusivos 2018 por ID: 832
Exclusivos 2019 por ID: 1393

Columnas utilizadas para huella:
['nom_estab', 'nom_vial', 'numero_ext', 'numero_int', 'cod_postal', 'localidad', 'municipio', 'latitud', 'longitud']


In [296]:
# ============================================================
# FASE 3 - PASO 25
# CONSTRUIR HUELLA EXACTA
# ============================================================

def construir_huella(df, columnas):

    partes = []

    for col in columnas:

        partes.append(
            df[col]
            .apply(normalizar_huella)
        )

    huella = partes[0]

    for parte in partes[1:]:
        huella = huella + "||" + parte

    return huella


solo18_df["HUELLA_EXACTA"] = construir_huella(
    solo18_df,
    columnas_huella
)

solo19_df["HUELLA_EXACTA"] = construir_huella(
    solo19_df,
    columnas_huella
)


print(
    "Huellas únicas 2018:",
    solo18_df["HUELLA_EXACTA"].nunique()
)

print(
    "Huellas únicas 2019:",
    solo19_df["HUELLA_EXACTA"].nunique()
)

Huellas únicas 2018: 832
Huellas únicas 2019: 1392


In [297]:
# ============================================================
# FASE 3 - PASO 26
# COINCIDENCIAS EXACTAS 1 A 1 ENTRE EXCLUSIVOS
# ============================================================

conteo18 = (
    solo18_df["HUELLA_EXACTA"]
    .value_counts()
)

conteo19 = (
    solo19_df["HUELLA_EXACTA"]
    .value_counts()
)

huellas_comunes = (
    set(conteo18.index)
    &
    set(conteo19.index)
)

huellas_1a1 = {
    h
    for h in huellas_comunes
    if conteo18[h] == 1
    and conteo19[h] == 1
}

print(
    "Huellas exactas comunes:",
    len(huellas_comunes)
)

print(
    "Coincidencias exactas 1 a 1:",
    len(huellas_1a1)
)


# ------------------------------------------------------------
# EXTRAER PARES
# ------------------------------------------------------------

match18 = (
    solo18_df[
        solo18_df["HUELLA_EXACTA"]
        .isin(huellas_1a1)
    ]
    [
        [
            id18,
            "HUELLA_EXACTA"
        ]
    ]
    .rename(
        columns={
            id18: "ID_2018"
        }
    )
)

match19 = (
    solo19_df[
        solo19_df["HUELLA_EXACTA"]
        .isin(huellas_1a1)
    ]
    [
        [
            id19,
            "HUELLA_EXACTA"
        ]
    ]
    .rename(
        columns={
            id19: "ID_2019"
        }
    )
)

pares_cambio_id = match18.merge(
    match19,
    on="HUELLA_EXACTA",
    how="inner",
    validate="one_to_one"
)

print(
    "\nPosibles mismos establecimientos "
    "con cambio de ID:",
    len(pares_cambio_id)
)

display(
    pares_cambio_id.head(20)
)

Huellas exactas comunes: 1
Coincidencias exactas 1 a 1: 1

Posibles mismos establecimientos con cambio de ID: 1


,ID_2018,HUELLA_EXACTA,ID_2019
0,2583057,ESTETICA SIMONE||ANASTASIO BUSTAMANTE||80||||5...,8749175


In [298]:
# ============================================================
# FASE 3 - PASO 26
# COINCIDENCIAS EXACTAS 1 A 1 ENTRE EXCLUSIVOS
# ============================================================

conteo18 = (
    solo18_df["HUELLA_EXACTA"]
    .value_counts()
)

conteo19 = (
    solo19_df["HUELLA_EXACTA"]
    .value_counts()
)

huellas_comunes = (
    set(conteo18.index)
    &
    set(conteo19.index)
)

huellas_1a1 = {
    h
    for h in huellas_comunes
    if conteo18[h] == 1
    and conteo19[h] == 1
}

print(
    "Huellas exactas comunes:",
    len(huellas_comunes)
)

print(
    "Coincidencias exactas 1 a 1:",
    len(huellas_1a1)
)


# ------------------------------------------------------------
# EXTRAER PARES
# ------------------------------------------------------------

match18 = (
    solo18_df[
        solo18_df["HUELLA_EXACTA"]
        .isin(huellas_1a1)
    ]
    [
        [
            id18,
            "HUELLA_EXACTA"
        ]
    ]
    .rename(
        columns={
            id18: "ID_2018"
        }
    )
)

match19 = (
    solo19_df[
        solo19_df["HUELLA_EXACTA"]
        .isin(huellas_1a1)
    ]
    [
        [
            id19,
            "HUELLA_EXACTA"
        ]
    ]
    .rename(
        columns={
            id19: "ID_2019"
        }
    )
)

pares_cambio_id = match18.merge(
    match19,
    on="HUELLA_EXACTA",
    how="inner",
    validate="one_to_one"
)

print(
    "\nPosibles mismos establecimientos "
    "con cambio de ID:",
    len(pares_cambio_id)
)

display(
    pares_cambio_id.head(20)
)

Huellas exactas comunes: 1
Coincidencias exactas 1 a 1: 1

Posibles mismos establecimientos con cambio de ID: 1


,ID_2018,HUELLA_EXACTA,ID_2019
0,2583057,ESTETICA SIMONE||ANASTASIO BUSTAMANTE||80||||5...,8749175


In [299]:
# ============================================================
# FASE 3 - PASO 27
# BALANCE DESPUÉS DE SEPARAR CAMBIOS DE ID
# ============================================================

n_cambios_id = len(pares_cambio_id)

solo18_sin_match = (
    len(solo18_df)
    - n_cambios_id
)

solo19_sin_match = (
    len(solo19_df)
    - n_cambios_id
)

print("Exclusivos 2018 originales:", len(solo18_df))
print("Exclusivos 2019 originales:", len(solo19_df))

print(
    "\nPosibles cambios de ID exactos:",
    n_cambios_id
)

print(
    "Solo 2018 después de descontarlos:",
    solo18_sin_match
)

print(
    "Solo 2019 después de descontarlos:",
    solo19_sin_match
)

print(
    "\nBalance neto restante:",
    solo19_sin_match - solo18_sin_match
)

Exclusivos 2018 originales: 832
Exclusivos 2019 originales: 1393

Posibles cambios de ID exactos: 1
Solo 2018 después de descontarlos: 831
Solo 2019 después de descontarlos: 1392

Balance neto restante: 561


In [300]:
# ============================================================
# FASE 3 - PASO 28
# HUELLA REDUCIDA PARA DETECTAR CAMBIOS MENORES DE REGISTRO
# ============================================================

columnas_huella_reducida = [
    col
    for col in [
        "nom_estab",
        "nom_vial",
        "numero_ext",
        "localidad"
    ]
    if col in solo18_df.columns
    and col in solo19_df.columns
]

print(
    "Columnas de huella reducida:",
    columnas_huella_reducida
)


solo18_df["HUELLA_REDUCIDA"] = construir_huella(
    solo18_df,
    columnas_huella_reducida
)

solo19_df["HUELLA_REDUCIDA"] = construir_huella(
    solo19_df,
    columnas_huella_reducida
)


# Quitar de esta segunda búsqueda el par
# que ya resolvimos con huella exacta
ids18_ya_match = set(
    pares_cambio_id["ID_2018"]
    .astype(str)
)

ids19_ya_match = set(
    pares_cambio_id["ID_2019"]
    .astype(str)
)

resto18 = solo18_df[
    ~solo18_df[id18]
    .astype(str)
    .isin(ids18_ya_match)
].copy()

resto19 = solo19_df[
    ~solo19_df[id19]
    .astype(str)
    .isin(ids19_ya_match)
].copy()


conteo18_red = (
    resto18["HUELLA_REDUCIDA"]
    .value_counts()
)

conteo19_red = (
    resto19["HUELLA_REDUCIDA"]
    .value_counts()
)

huellas_red_comunes = (
    set(conteo18_red.index)
    &
    set(conteo19_red.index)
)

huellas_red_1a1 = {
    h
    for h in huellas_red_comunes
    if conteo18_red[h] == 1
    and conteo19_red[h] == 1
}


print(
    "\nHuellas reducidas comunes:",
    len(huellas_red_comunes)
)

print(
    "Coincidencias reducidas únicas 1 a 1:",
    len(huellas_red_1a1)
)

Columnas de huella reducida: ['nom_estab', 'nom_vial', 'numero_ext', 'localidad']

Huellas reducidas comunes: 2
Coincidencias reducidas únicas 1 a 1: 2


In [301]:
# ============================================================
# FASE 3 - PASO 29
# PARES 1 A 1 CON HUELLA REDUCIDA
# ============================================================

match18_red = (
    resto18[
        resto18["HUELLA_REDUCIDA"]
        .isin(huellas_red_1a1)
    ]
    [
        [
            id18,
            "HUELLA_REDUCIDA"
        ]
    ]
    .rename(
        columns={
            id18: "ID_2018"
        }
    )
)

match19_red = (
    resto19[
        resto19["HUELLA_REDUCIDA"]
        .isin(huellas_red_1a1)
    ]
    [
        [
            id19,
            "HUELLA_REDUCIDA"
        ]
    ]
    .rename(
        columns={
            id19: "ID_2019"
        }
    )
)

pares_reducidos = match18_red.merge(
    match19_red,
    on="HUELLA_REDUCIDA",
    how="inner",
    validate="one_to_one"
)

print(
    "Posibles mismos establecimientos "
    "con cambios menores en el registro:",
    len(pares_reducidos)
)

display(
    pares_reducidos.head(30)
)

Posibles mismos establecimientos con cambios menores en el registro: 2


,ID_2018,HUELLA_REDUCIDA,ID_2019
0,2612777,ESTETICA ARA||VICENTE GUERRERO||391||MORELIA,8072000
1,2613825,ESTETICA CIRI||TORREON NUEVO||521||MORELIA,8452308


In [302]:
# ============================================================
# FASE 3 - PASO 30
# BALANCE DESPUÉS DE LAS DOS HUELLA
# ============================================================

n_exactos = len(pares_cambio_id)
n_reducidos = len(pares_reducidos)

total_posibles_mismos = (
    n_exactos + n_reducidos
)

solo18_final = (
    len(solo18_df)
    - total_posibles_mismos
)

solo19_final = (
    len(solo19_df)
    - total_posibles_mismos
)

print("Coincidencias exactas:", n_exactos)
print("Coincidencias reducidas adicionales:", n_reducidos)

print(
    "\nTotal posibles mismos establecimientos "
    "con ID diferente:",
    total_posibles_mismos
)

print(
    "Solo 2018 después de ambas huellas:",
    solo18_final
)

print(
    "Solo 2019 después de ambas huellas:",
    solo19_final
)

print(
    "Balance neto:",
    solo19_final - solo18_final
)

Coincidencias exactas: 1
Coincidencias reducidas adicionales: 2

Total posibles mismos establecimientos con ID diferente: 3
Solo 2018 después de ambas huellas: 829
Solo 2019 después de ambas huellas: 1390
Balance neto: 561


In [303]:
# ============================================================
# FASE 3 - PASO 31
# EXCLUSIVOS FINALES DESPUÉS DE DESCONTAR POSIBLES CAMBIOS DE ID
# ============================================================

ids18_match = set(
    pares_cambio_id["ID_2018"].astype(str)
) | set(
    pares_reducidos["ID_2018"].astype(str)
)

ids19_match = set(
    pares_cambio_id["ID_2019"].astype(str)
) | set(
    pares_reducidos["ID_2019"].astype(str)
)

solo18_final_df = solo18_df[
    ~solo18_df[id18]
    .astype(str)
    .isin(ids18_match)
].copy()

solo19_final_df = solo19_df[
    ~solo19_df[id19]
    .astype(str)
    .isin(ids19_match)
].copy()

print("Solo 2018 final:", len(solo18_final_df))
print("Solo 2019 final:", len(solo19_final_df))
print(
    "Balance:",
    len(solo19_final_df) - len(solo18_final_df)
)

Solo 2018 final: 829
Solo 2019 final: 1390
Balance: 561


In [304]:
# ============================================================
# FASE 3 - PASO 32
# DISTRIBUCIÓN DE FECHA_ALTA EN EXCLUSIVOS DE 2019
# ============================================================

candidatos_fecha = [
    "fecha_alta",
    "Fecha_alta",
    "FECHA_ALTA",
    "fechaAlta"
]

fecha19 = detectar_columna(
    solo19_final_df,
    candidatos_fecha
)

print("Columna fecha_alta detectada:", fecha19)

if fecha19 is not None:

    print("\nValores de fecha_alta:")
    
    display(
        solo19_final_df[fecha19]
        .fillna("SIN DATO")
        .astype(str)
        .str.strip()
        .value_counts()
        .reset_index()
        .head(30)
    )

Columna fecha_alta detectada: fecha_alta

Valores de fecha_alta:


,fecha_alta,count
0,2019-11,1363
1,2014-12,16
2,2010-07,11


In [305]:
# ============================================================
# FASE 3 - PASO 33
# ¿CUÁNTOS EXCLUSIVOS DE 2019 TIENEN FECHA_ALTA 2019-11?
# ============================================================

def es_alta_2019_11(valor):

    if pd.isna(valor):
        return False

    texto = str(valor).strip().upper()

    return (
        "2019-11" in texto
        or
        (
            "2019" in texto
            and "NOVIEMBRE" in texto
        )
    )


solo19_final_df["Alta_2019_11"] = (
    solo19_final_df[fecha19]
    .apply(es_alta_2019_11)
)

n_alta_2019_11 = (
    solo19_final_df["Alta_2019_11"].sum()
)

n_otras_fechas = (
    len(solo19_final_df)
    - n_alta_2019_11
)

print(
    "Exclusivos 2019 finales:",
    len(solo19_final_df)
)

print(
    "Con fecha_alta 2019-11:",
    n_alta_2019_11
)

print(
    "Con otra fecha o sin dato:",
    n_otras_fechas
)

print(
    "Porcentaje con fecha_alta 2019-11:",
    round(
        n_alta_2019_11
        /
        len(solo19_final_df)
        * 100,
        2
    ),
    "%"
)

Exclusivos 2019 finales: 1390
Con fecha_alta 2019-11: 1363
Con otra fecha o sin dato: 27
Porcentaje con fecha_alta 2019-11: 98.06 %


In [306]:
# ============================================================
# PASO 34
# FECHA_ALTA 2019-11 EN TODO EL STOCK 812110 DE 2019
# ============================================================

d19["Alta_2019_11"] = (
    d19[fecha19]
    .apply(es_alta_2019_11)
)

print(
    "Total 812110 en 2019:",
    len(d19)
)

print(
    "Con fecha_alta 2019-11:",
    d19["Alta_2019_11"].sum()
)

print(
    "Porcentaje:",
    round(
        d19["Alta_2019_11"].mean() * 100,
        2
    ),
    "%"
)

Total 812110 en 2019: 2328
Con fecha_alta 2019-11: 1366
Porcentaje: 58.68 %


In [307]:
# ============================================================
# FASE 3 - PASO 35
# FECHA_ALTA ENTRE LOS EXCLUSIVOS FINALES DE 2019
# ============================================================

solo19_final_df["Alta_2019_11"] = (
    solo19_final_df[fecha19]
    .apply(es_alta_2019_11)
)

n_exclusivos_alta_2019 = int(
    solo19_final_df["Alta_2019_11"].sum()
)

n_exclusivos_otras = (
    len(solo19_final_df)
    - n_exclusivos_alta_2019
)

print(
    "Exclusivos finales de 2019:",
    len(solo19_final_df)
)

print(
    "Con fecha_alta 2019-11:",
    n_exclusivos_alta_2019
)

print(
    "Con otra fecha o sin dato:",
    n_exclusivos_otras
)

print(
    "Porcentaje de exclusivos con fecha_alta 2019-11:",
    round(
        n_exclusivos_alta_2019
        / len(solo19_final_df)
        * 100,
        2
    ),
    "%"
)

print("\nDistribución completa de fecha_alta:")

display(
    solo19_final_df[fecha19]
    .fillna("SIN DATO")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
    .head(20)
)

Exclusivos finales de 2019: 1390
Con fecha_alta 2019-11: 1363
Con otra fecha o sin dato: 27
Porcentaje de exclusivos con fecha_alta 2019-11: 98.06 %

Distribución completa de fecha_alta:


,fecha_alta,count
0,2019-11,1363
1,2014-12,16
2,2010-07,11


In [308]:
# ============================================================
# FASE 3 - PASO 36
# FECHA_ALTA DE LOS REGISTROS QUE ESTABAN EN 2018
# Y YA NO APARECEN EN 2019
# ============================================================

fecha18 = detectar_columna(
    solo18_final_df,
    candidatos_fecha
)

print("Columna fecha_alta 2018:", fecha18)

display(
    solo18_final_df[fecha18]
    .fillna("SIN DATO")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
    .head(20)
)

Columna fecha_alta 2018: fecha_alta


,fecha_alta,count
0,2014-12,484
1,2010-07,342
2,2013-07,3


In [309]:
# ============================================================
# FASE 3 - PASO 37
# DISTRIBUCIÓN DE FECHA_ALTA EN LOS 1,390 EXCLUSIVOS DE 2019
# ============================================================

distribucion_alta_solo19 = (
    solo19_final_df[fecha19]
    .fillna("SIN DATO")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)

distribucion_alta_solo19.columns = [
    "fecha_alta",
    "Establecimientos"
]

distribucion_alta_solo19["Porcentaje"] = (
    distribucion_alta_solo19["Establecimientos"]
    / len(solo19_final_df)
    * 100
)

display(distribucion_alta_solo19)

print(
    "\nTotal exclusivos 2019:",
    len(solo19_final_df)
)

print(
    "Con fecha_alta 2019-11:",
    (
        solo19_final_df[fecha19]
        .astype(str)
        .str.strip()
        .eq("2019-11")
        .sum()
    )
)

,fecha_alta,Establecimientos,Porcentaje
0,2019-11,1363,98.057554
1,2014-12,16,1.151079
2,2010-07,11,0.791367



Total exclusivos 2019: 1390
Con fecha_alta 2019-11: 1363


In [310]:
# ============================================================
# FASE 3 - PASO 37
# DISTRIBUCIÓN DE FECHA_ALTA EN LOS 1,390 EXCLUSIVOS DE 2019
# ============================================================

distribucion_alta_solo19 = (
    solo19_final_df[fecha19]
    .fillna("SIN DATO")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)

distribucion_alta_solo19.columns = [
    "fecha_alta",
    "Establecimientos"
]

distribucion_alta_solo19["Porcentaje"] = (
    distribucion_alta_solo19["Establecimientos"]
    / len(solo19_final_df)
    * 100
)

display(distribucion_alta_solo19)

print(
    "\nTotal exclusivos 2019:",
    len(solo19_final_df)
)

print(
    "Con fecha_alta 2019-11:",
    (
        solo19_final_df[fecha19]
        .astype(str)
        .str.strip()
        .eq("2019-11")
        .sum()
    )
)

,fecha_alta,Establecimientos,Porcentaje
0,2019-11,1363,98.057554
1,2014-12,16,1.151079
2,2010-07,11,0.791367



Total exclusivos 2019: 1390
Con fecha_alta 2019-11: 1363


In [311]:
# ============================================================
# FASE 3 - PASO 38
# AUDITAR IDs CONTINUOS CON FECHA_ALTA 2019-11
# ============================================================

# IDs presentes en ambos años
ids_comunes = set(
    d18[id18].astype(str).str.strip()
) & set(
    d19[id19].astype(str).str.strip()
)

# Registros 2019 presentes también en 2018
# pero con fecha_alta = 2019-11
casos_fecha_reasignada_19 = d19[
    d19[id19].astype(str).str.strip().isin(ids_comunes)
    &
    d19[fecha19].astype(str).str.strip().eq("2019-11")
].copy()

print(
    "IDs presentes en 2018 y 2019 "
    "con fecha_alta 2019-11 en 2019:",
    len(casos_fecha_reasignada_19)
)

ids_revision = set(
    casos_fecha_reasignada_19[id19]
    .astype(str)
    .str.strip()
)

cols_revision = [
    col for col in [
        "id",
        "nom_estab",
        "codigo_act",
        "nom_vial",
        "numero_ext",
        "cod_postal",
        "latitud",
        "longitud",
        "fecha_alta"
    ]
    if col in d18.columns
]

rev18 = d18[
    d18[id18].astype(str).str.strip().isin(ids_revision)
][cols_revision].copy()

rev18["Año_corte"] = 2018


cols_revision19 = [
    col for col in [
        "id",
        "nom_estab",
        "codigo_act",
        "nom_vial",
        "numero_ext",
        "cod_postal",
        "latitud",
        "longitud",
        "fecha_alta"
    ]
    if col in d19.columns
]

rev19 = d19[
    d19[id19].astype(str).str.strip().isin(ids_revision)
][cols_revision19].copy()

rev19["Año_corte"] = 2019


revision_fecha_alta = pd.concat(
    [rev18, rev19],
    ignore_index=True
)

display(
    revision_fecha_alta.sort_values(
        ["id", "Año_corte"]
    )
)


IDs presentes en 2018 y 2019 con fecha_alta 2019-11 en 2019: 0


,id,nom_estab,codigo_act,nom_vial,numero_ext,cod_postal,latitud,longitud,fecha_alta,Año_corte


In [312]:
# ============================================================
# FASE 3 - PASO 39
# AUDITAR LOS 3 POSIBLES CAMBIOS DE ID
# Y SU FECHA_ALTA EN 2018 Y 2019
# ============================================================

# Unir los 3 pares identificados
pares_exactos_aux = (
    pares_cambio_id[
        ["ID_2018", "ID_2019"]
    ]
    .copy()
)

pares_exactos_aux["Tipo_match"] = (
    "Huella exacta"
)

pares_reducidos_aux = (
    pares_reducidos[
        ["ID_2018", "ID_2019"]
    ]
    .copy()
)

pares_reducidos_aux["Tipo_match"] = (
    "Huella reducida"
)

pares_3 = pd.concat(
    [
        pares_exactos_aux,
        pares_reducidos_aux
    ],
    ignore_index=True
)

pares_3["ID_2018"] = (
    pares_3["ID_2018"]
    .astype(str)
    .str.strip()
)

pares_3["ID_2019"] = (
    pares_3["ID_2019"]
    .astype(str)
    .str.strip()
)

display(pares_3)

,ID_2018,ID_2019,Tipo_match
0,2583057,8749175,Huella exacta
1,2612777,8072000,Huella reducida
2,2613825,8452308,Huella reducida


In [313]:
# ============================================================
# FASE 3 - PASO 40
# COMPARAR DATOS DE LOS 3 PARES ENTRE CORTES
# ============================================================

cols_info = [
    "nom_estab",
    "codigo_act",
    "nom_vial",
    "numero_ext",
    "numero_int",
    "cod_postal",
    "localidad",
    "latitud",
    "longitud",
    "fecha_alta"
]

cols18_info = [
    c for c in cols_info
    if c in d18.columns
]

cols19_info = [
    c for c in cols_info
    if c in d19.columns
]


info18 = (
    d18[
        d18[id18]
        .astype(str)
        .str.strip()
        .isin(set(pares_3["ID_2018"]))
    ]
    [[id18] + cols18_info]
    .copy()
)

info18[id18] = (
    info18[id18]
    .astype(str)
    .str.strip()
)

info18 = info18.rename(
    columns={
        id18: "ID_2018"
    }
)


info19 = (
    d19[
        d19[id19]
        .astype(str)
        .str.strip()
        .isin(set(pares_3["ID_2019"]))
    ]
    [[id19] + cols19_info]
    .copy()
)

info19[id19] = (
    info19[id19]
    .astype(str)
    .str.strip()
)

info19 = info19.rename(
    columns={
        id19: "ID_2019"
    }
)


revision_3 = (
    pares_3
    .merge(
        info18,
        on="ID_2018",
        how="left"
    )
    .merge(
        info19,
        on="ID_2019",
        how="left",
        suffixes=("_2018", "_2019")
    )
)

display(
    revision_3[
        [
            "Tipo_match",
            "ID_2018",
            "ID_2019",
            "nom_estab_2018",
            "nom_estab_2019",
            "fecha_alta_2018",
            "fecha_alta_2019"
        ]
    ]
)

,Tipo_match,ID_2018,ID_2019,nom_estab_2018,nom_estab_2019,fecha_alta_2018,fecha_alta_2019
0,Huella exacta,2583057,8749175,ESTÉTICA SIMONE,ESTÉTICA SIMONE,2010-07,2019-11
1,Huella reducida,2612777,8072000,ESTÉTICA ARA,ESTÉTICA ARA,2014-12,2019-11
2,Huella reducida,2613825,8452308,ESTÉTICA CIRI,ESTÉTICA CIRI,2014-12,2019-11


In [314]:
# ============================================================
# FASE 3 - PASO 41
# GUARDAR AUDITORÍA MICRO 812110: 2018-2019
# ============================================================

revision_3.to_csv(
    RESULTADOS /
    "AUDITORIA_CAMBIOS_ID_812110_2018_2019.csv",
    index=False,
    encoding="utf-8-sig"
)

distribucion_alta_solo19.to_csv(
    RESULTADOS /
    "DISTRIBUCION_FECHA_ALTA_812110_EXCLUSIVOS_2019.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Auditoría 812110 2018-2019 guardada.")

✓ Auditoría 812110 2018-2019 guardada.


### Auditoría micro del código SCIAN 812110 entre 2018 y 2019

El código SCIAN 812110 pasó de 1,767 establecimientos en 2018 a 2,328 en
2019, lo que representa una diferencia neta de 561 registros.

La comparación longitudinal por identificador mostró 935 registros presentes
en ambos cortes, 832 exclusivos de 2018 y 1,393 exclusivos de 2019.
Mediante huellas de identificación se detectaron únicamente tres posibles
correspondencias de establecimientos con cambio de ID.

Después de descontar estos casos permanecieron 829 registros exclusivos de
2018 y 1,390 exclusivos de 2019. De estos últimos, 1,363 (98.06%) presentaron
fecha_alta igual a 2019-11.

Asimismo, en los tres casos identificados con posible cambio de ID, la
fecha_alta registrada en 2018 fue sustituida por 2019-11 en el corte de 2019.

Estos resultados indican que el campo fecha_alta no debe interpretarse
automáticamente como fecha de nacimiento empresarial ni como una fecha
histórica inmutable. Asimismo, el incremento neto observado en 2019 debe
analizarse considerando una incorporación masiva de registros asociada con
la actualización del DENUE, y no exclusivamente como crecimiento del número
de establecimientos.

In [316]:
# ============================================================
# FASE 3 - PASO 42
# PREPARAR AUDITORÍA 2023-2024 DE UH0336 / SCIAN 812110
# ============================================================

print("Representación clasificatoria de UH0336:")

display(
    diccionario_350_completo[
        diccionario_350_completo["Unidad_homologada"] == "UH0336"
    ]
    .sort_values("Version_SCIAN")
)

print("\nArchivos 2023:")
for archivo in sorted(MORELIA.glob("*2023*.csv")):
    print(" -", archivo.name)

print("\nArchivos 2024:")
for archivo in sorted(MORELIA.glob("*2024*.csv")):
    print(" -", archivo.name)

Representación clasificatoria de UH0336:


,Unidad_homologada,Version_SCIAN,SCIAN_original,Origen_asignacion
1717,UH0336,SCIAN 2007,812110,Grafo homologación
1718,UH0336,SCIAN 2013,812110,Grafo homologación
1719,UH0336,SCIAN 2018,812110,Grafo homologación
1720,UH0336,SCIAN 2023,812110,Grafo homologación



Archivos 2023:
 - DENUE_2023_MORELIA.csv

Archivos 2024:
 - DENUE_2024_MORELIA.csv


In [317]:
# ============================================================
# FASE 3 - PASO 43
# CARGAR 2023 Y 2024 Y AISLAR SCIAN 812110
# ============================================================

archivo_2023 = MORELIA / "DENUE_2023_MORELIA.csv"
archivo_2024 = MORELIA / "DENUE_2024_MORELIA.csv"

df23 = pd.read_csv(
    archivo_2023,
    dtype=str,
    low_memory=False
)

df24 = pd.read_csv(
    archivo_2024,
    dtype=str,
    low_memory=False
)

# Detectar columnas
scian23 = detectar_columna(
    df23,
    candidatos_scian
)

scian24 = detectar_columna(
    df24,
    candidatos_scian
)

id23 = detectar_columna(
    df23,
    candidatos_id
)

id24 = detectar_columna(
    df24,
    candidatos_id
)

print("SCIAN 2023:", scian23)
print("SCIAN 2024:", scian24)
print("ID 2023:", id23)
print("ID 2024:", id24)

# Filtrar 812110
d23 = df23[
    df23[scian23]
    .astype(str)
    .str.strip()
    .eq("812110")
].copy()

d24 = df24[
    df24[scian24]
    .astype(str)
    .str.strip()
    .eq("812110")
].copy()

print("\n812110 en 2023:", len(d23))
print("812110 en 2024:", len(d24))
print("Diferencia:", len(d24) - len(d23))

print("\nIDs nulos 2023:", d23[id23].isna().sum())
print("IDs nulos 2024:", d24[id24].isna().sum())

print(
    "IDs duplicados 2023:",
    d23[id23].duplicated().sum()
)

print(
    "IDs duplicados 2024:",
    d24[id24].duplicated().sum()
)

SCIAN 2023: codigo_act
SCIAN 2024: codigo_act
ID 2023: id
ID 2024: id

812110 en 2023: 2334
812110 en 2024: 3128
Diferencia: 794

IDs nulos 2023: 0
IDs nulos 2024: 0
IDs duplicados 2023: 0
IDs duplicados 2024: 0


In [318]:
# ============================================================
# FASE 3 - PASO 44
# COMPARAR 812110 ENTRE 2023 Y 2024 POR ID
# ============================================================

ids23 = set(
    d23[id23]
    .dropna()
    .astype(str)
    .str.strip()
)

ids24 = set(
    d24[id24]
    .dropna()
    .astype(str)
    .str.strip()
)

comunes_23_24 = ids23 & ids24
solo23 = ids23 - ids24
solo24 = ids24 - ids23

print("IDs 2023:", len(ids23))
print("IDs 2024:", len(ids24))

print(
    "\nPresentes en ambos cortes:",
    len(comunes_23_24)
)

print(
    "Solo en 2023:",
    len(solo23)
)

print(
    "Solo en 2024:",
    len(solo24)
)

print(
    "\nBalance solo2024 - solo2023:",
    len(solo24) - len(solo23)
)

IDs 2023: 2334
IDs 2024: 3128

Presentes en ambos cortes: 1390
Solo en 2023: 944
Solo en 2024: 1738

Balance solo2024 - solo2023: 794


In [319]:
# ============================================================
# FASE 3 - PASO 45
# HUELLA EXACTA ENTRE EXCLUSIVOS 2023 Y 2024
# ============================================================

solo23_df = d23[
    d23[id23]
    .astype(str)
    .str.strip()
    .isin(solo23)
].copy()

solo24_df = d24[
    d24[id24]
    .astype(str)
    .str.strip()
    .isin(solo24)
].copy()

columnas_huella_23_24 = [
    col
    for col in [
        "nom_estab",
        "nom_vial",
        "numero_ext",
        "numero_int",
        "cod_postal",
        "localidad",
        "municipio",
        "latitud",
        "longitud"
    ]
    if col in solo23_df.columns
    and col in solo24_df.columns
]

print("Exclusivos 2023:", len(solo23_df))
print("Exclusivos 2024:", len(solo24_df))

print("\nColumnas de huella:")
print(columnas_huella_23_24)

solo23_df["HUELLA_EXACTA"] = construir_huella(
    solo23_df,
    columnas_huella_23_24
)

solo24_df["HUELLA_EXACTA"] = construir_huella(
    solo24_df,
    columnas_huella_23_24
)

conteo23 = (
    solo23_df["HUELLA_EXACTA"]
    .value_counts()
)

conteo24 = (
    solo24_df["HUELLA_EXACTA"]
    .value_counts()
)

huellas_comunes_23_24 = (
    set(conteo23.index)
    &
    set(conteo24.index)
)

huellas_1a1_23_24 = {
    h
    for h in huellas_comunes_23_24
    if conteo23[h] == 1
    and conteo24[h] == 1
}

print(
    "\nHuellas exactas comunes:",
    len(huellas_comunes_23_24)
)

print(
    "Coincidencias exactas 1 a 1:",
    len(huellas_1a1_23_24)
)

Exclusivos 2023: 944
Exclusivos 2024: 1738

Columnas de huella:
['nom_estab', 'nom_vial', 'numero_ext', 'numero_int', 'cod_postal', 'localidad', 'municipio', 'latitud', 'longitud']

Huellas exactas comunes: 0
Coincidencias exactas 1 a 1: 0


In [320]:
# ============================================================
# FASE 3 - PASO 46
# HUELLA REDUCIDA ENTRE EXCLUSIVOS 2023 Y 2024
# ============================================================

columnas_huella_reducida_23_24 = [
    col
    for col in [
        "nom_estab",
        "nom_vial",
        "numero_ext",
        "localidad"
    ]
    if col in solo23_df.columns
    and col in solo24_df.columns
]

print(
    "Columnas de huella reducida:",
    columnas_huella_reducida_23_24
)

solo23_df["HUELLA_REDUCIDA"] = construir_huella(
    solo23_df,
    columnas_huella_reducida_23_24
)

solo24_df["HUELLA_REDUCIDA"] = construir_huella(
    solo24_df,
    columnas_huella_reducida_23_24
)

conteo23_red = (
    solo23_df["HUELLA_REDUCIDA"]
    .value_counts()
)

conteo24_red = (
    solo24_df["HUELLA_REDUCIDA"]
    .value_counts()
)

huellas_red_comunes_23_24 = (
    set(conteo23_red.index)
    &
    set(conteo24_red.index)
)

huellas_red_1a1_23_24 = {
    h
    for h in huellas_red_comunes_23_24
    if conteo23_red[h] == 1
    and conteo24_red[h] == 1
}

print(
    "\nHuellas reducidas comunes:",
    len(huellas_red_comunes_23_24)
)

print(
    "Coincidencias reducidas únicas 1 a 1:",
    len(huellas_red_1a1_23_24)
)

Columnas de huella reducida: ['nom_estab', 'nom_vial', 'numero_ext', 'localidad']

Huellas reducidas comunes: 1
Coincidencias reducidas únicas 1 a 1: 1


In [321]:
# ============================================================
# FASE 3 - PASO 47
# PARES CON HUELLA REDUCIDA 2023-2024
# ============================================================

match23_red = (
    solo23_df[
        solo23_df["HUELLA_REDUCIDA"]
        .isin(huellas_red_1a1_23_24)
    ]
    [[id23, "HUELLA_REDUCIDA"]]
    .rename(
        columns={
            id23: "ID_2023"
        }
    )
)

match24_red = (
    solo24_df[
        solo24_df["HUELLA_REDUCIDA"]
        .isin(huellas_red_1a1_23_24)
    ]
    [[id24, "HUELLA_REDUCIDA"]]
    .rename(
        columns={
            id24: "ID_2024"
        }
    )
)

pares_reducidos_23_24 = match23_red.merge(
    match24_red,
    on="HUELLA_REDUCIDA",
    how="inner",
    validate="one_to_one"
)

print(
    "Posibles mismos establecimientos "
    "con cambio de ID:",
    len(pares_reducidos_23_24)
)

display(
    pares_reducidos_23_24.head(30)
)

Posibles mismos establecimientos con cambio de ID: 1


,ID_2023,HUELLA_REDUCIDA,ID_2024
0,2574745,PELUQUERIA||LAZARO CARDENAS||526||MORELIA,6976777


In [322]:
# ============================================================
# FASE 3 - PASO 48
# FECHA_ALTA DE LOS EXCLUSIVOS DE 2024
# ============================================================

fecha24 = detectar_columna(
    solo24_df,
    candidatos_fecha
)

print(
    "Columna fecha_alta 2024:",
    fecha24
)

distribucion_alta_solo24 = (
    solo24_df[fecha24]
    .fillna("SIN DATO")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)

distribucion_alta_solo24.columns = [
    "fecha_alta",
    "Establecimientos"
]

distribucion_alta_solo24["Porcentaje"] = (
    distribucion_alta_solo24[
        "Establecimientos"
    ]
    /
    len(solo24_df)
    * 100
)

display(
    distribucion_alta_solo24.head(20)
)


Columna fecha_alta 2024: fecha_alta


,fecha_alta,Establecimientos,Porcentaje
0,2024-11,1688,97.123130
1,2019-11,30,1.726122
2,2010-07,12,0.690449
3,2014-12,5,0.287687
4,2020-11,2,0.115075
5,2021-05,1,0.057537


In [323]:
# ============================================================
# FASE 3 - PASO 49
# EXCLUSIVOS FINALES 2023-2024
# ============================================================

ids23_match = set(
    pares_reducidos_23_24["ID_2023"]
    .astype(str)
    .str.strip()
)

ids24_match = set(
    pares_reducidos_23_24["ID_2024"]
    .astype(str)
    .str.strip()
)

solo23_final_df = solo23_df[
    ~solo23_df[id23]
    .astype(str)
    .str.strip()
    .isin(ids23_match)
].copy()

solo24_final_df = solo24_df[
    ~solo24_df[id24]
    .astype(str)
    .str.strip()
    .isin(ids24_match)
].copy()

print("Solo 2023 final:", len(solo23_final_df))
print("Solo 2024 final:", len(solo24_final_df))

print(
    "Balance final:",
    len(solo24_final_df)
    - len(solo23_final_df)
)

Solo 2023 final: 943
Solo 2024 final: 1737
Balance final: 794


In [324]:
# ============================================================
# FASE 3 - PASO 50
# FECHA_ALTA DE LOS EXCLUSIVOS FINALES
# ============================================================

fecha23 = detectar_columna(
    solo23_final_df,
    candidatos_fecha
)

fecha24 = detectar_columna(
    solo24_final_df,
    candidatos_fecha
)

print("=== SOLO 2023 ===")

display(
    solo23_final_df[fecha23]
    .fillna("SIN DATO")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
    .head(20)
)

print("\n=== SOLO 2024 ===")

distribucion_final_2024 = (
    solo24_final_df[fecha24]
    .fillna("SIN DATO")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)

distribucion_final_2024.columns = [
    "fecha_alta",
    "Establecimientos"
]

distribucion_final_2024["Porcentaje"] = (
    distribucion_final_2024["Establecimientos"]
    / len(solo24_final_df)
    * 100
)

display(distribucion_final_2024)

=== SOLO 2023 ===


,fecha_alta,count
0,2019-11,634
1,2010-07,167
2,2014-12,127
3,2020-11,9
4,2021-05,3
5,2021-11,2
6,2023-11,1



=== SOLO 2024 ===


,fecha_alta,Establecimientos,Porcentaje
0,2024-11,1688,97.179044
1,2019-11,29,1.669545
2,2010-07,12,0.690846
3,2014-12,5,0.287853
4,2020-11,2,0.115141
5,2021-05,1,0.057571


In [325]:
# ============================================================
# FASE 3 - PASO 51
# AUDITAR EL ÚNICO POSIBLE CAMBIO DE ID 2023-2024
# ============================================================

id_2023_revision = str(
    pares_reducidos_23_24.iloc[0]["ID_2023"]
).strip()

id_2024_revision = str(
    pares_reducidos_23_24.iloc[0]["ID_2024"]
).strip()

cols_revision = [
    "id",
    "nom_estab",
    "codigo_act",
    "nom_vial",
    "numero_ext",
    "numero_int",
    "cod_postal",
    "localidad",
    "latitud",
    "longitud",
    "fecha_alta"
]

cols23 = [
    c for c in cols_revision
    if c in d23.columns
]

cols24 = [
    c for c in cols_revision
    if c in d24.columns
]

caso23 = d23[
    d23[id23]
    .astype(str)
    .str.strip()
    .eq(id_2023_revision)
][cols23].copy()

caso23["Año_corte"] = 2023

caso24 = d24[
    d24[id24]
    .astype(str)
    .str.strip()
    .eq(id_2024_revision)
][cols24].copy()

caso24["Año_corte"] = 2024

display(
    pd.concat(
        [caso23, caso24],
        ignore_index=True
    )
)

,id,nom_estab,codigo_act,nom_vial,numero_ext,numero_int,cod_postal,localidad,latitud,longitud,fecha_alta,Año_corte
0,2574745,PELUQUERIA,812110,LAZARO CARDENAS,526,53,58000,Morelia,19.69598865000000032,-101.18876489000000163,2010-07,2023
1,6976777,PELUQUERIA,812110,LAZARO CARDENAS,526,52,58000,Morelia ...,19.69598592,-101.18877214,2019-11,2024


In [326]:
# ============================================================
# FASE 3 - PASO 52
# RESUMEN COMPARATIVO DE LAS RUPTURAS 2019 Y 2024
# SCIAN 812110
# ============================================================

resumen_rupturas = pd.DataFrame([
    {
        "Transicion": "2018-2019",
        "SCIAN": "812110",

        "Establecimientos_inicial":
            len(d18),

        "Establecimientos_final":
            len(d19),

        "Cambio_neto":
            len(d19) - len(d18),

        "IDs_presentes_en_ambos":
            len(comunes),

        "Exclusivos_inicial_raw":
            len(solo18),

        "Exclusivos_final_raw":
            len(solo19),

        "Posibles_cambios_ID":
            len(pares_cambio_id)
            + len(pares_reducidos),

        "Exclusivos_inicial_final":
            len(solo18_final_df),

        "Exclusivos_final_final":
            len(solo19_final_df),

        "Fecha_alta_del_corte":
            "2019-11",

        "Exclusivos_con_fecha_del_corte":
            (
                solo19_final_df[fecha19]
                .astype(str)
                .str.strip()
                .eq("2019-11")
                .sum()
            )
    },

    {
        "Transicion": "2023-2024",
        "SCIAN": "812110",

        "Establecimientos_inicial":
            len(d23),

        "Establecimientos_final":
            len(d24),

        "Cambio_neto":
            len(d24) - len(d23),

        "IDs_presentes_en_ambos":
            len(comunes_23_24),

        "Exclusivos_inicial_raw":
            len(solo23),

        "Exclusivos_final_raw":
            len(solo24),

        "Posibles_cambios_ID":
            len(pares_reducidos_23_24),

        "Exclusivos_inicial_final":
            len(solo23_final_df),

        "Exclusivos_final_final":
            len(solo24_final_df),

        "Fecha_alta_del_corte":
            "2024-11",

        "Exclusivos_con_fecha_del_corte":
            (
                solo24_final_df[fecha24]
                .astype(str)
                .str.strip()
                .eq("2024-11")
                .sum()
            )
    }
])


resumen_rupturas[
    "Pct_exclusivos_con_fecha_del_corte"
] = (
    resumen_rupturas[
        "Exclusivos_con_fecha_del_corte"
    ]
    /
    resumen_rupturas[
        "Exclusivos_final_final"
    ]
    * 100
)

display(resumen_rupturas)

,Transicion,SCIAN,Establecimientos_inicial,Establecimientos_final,Cambio_neto,IDs_presentes_en_ambos,Exclusivos_inicial_raw,Exclusivos_final_raw,Posibles_cambios_ID,Exclusivos_inicial_final,Exclusivos_final_final,Fecha_alta_del_corte,Exclusivos_con_fecha_del_corte,Pct_exclusivos_con_fecha_del_corte
0,2018-2019,812110,1767,2328,561,935,832,1393,3,829,1390,2019-11,1363,98.057554
1,2023-2024,812110,2334,3128,794,1390,944,1738,1,943,1737,2024-11,1688,97.179044


In [327]:
resumen_rupturas.to_csv(
    RESULTADOS /
    "RESUMEN_AUDITORIA_RUPTURAS_812110_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

distribucion_final_2024.to_csv(
    RESULTADOS /
    "DISTRIBUCION_FECHA_ALTA_812110_EXCLUSIVOS_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✓ Auditorías 2019 y 2024 consolidadas.")

✓ Auditorías 2019 y 2024 consolidadas.


### Auditoría de las discontinuidades de 2019 y 2024

La revisión a nivel de establecimiento del código SCIAN 812110 mostró que
los incrementos observados en 2019 y 2024 no corresponden únicamente a una
expansión neta del número de establecimientos.

Entre 2018 y 2019 el número de registros aumentó de 1,767 a 2,328
(+561). Sin embargo, la comparación identificó simultáneamente 829 registros
exclusivos del corte inicial y 1,390 exclusivos del corte final, después de
descontar tres posibles correspondencias con cambio de identificador. De los
registros exclusivos de 2019, 1,363 (98.06 %) presentan fecha_alta 2019-11.

Entre 2023 y 2024 el número de registros aumentó de 2,334 a 3,128
(+794). Después de descontar una posible correspondencia con cambio de ID,
se observaron 943 registros exclusivos de 2023 y 1,737 exclusivos de 2024.
De estos últimos, 1,688 (97.18 %) presentan fecha_alta 2024-11.

Asimismo, se identificaron casos en los que un establecimiento aparentemente
continuo recibió un nuevo identificador y una fecha_alta diferente. Por ello,
el campo fecha_alta no se interpreta como fecha de nacimiento empresarial ni
como una característica histórica necesariamente inmutable.

En consecuencia, las variaciones de 2019 y 2024 se consideran discontinuidades
asociadas con una recomposición amplia del registro y deben distinguirse de
las variaciones ordinarias entre cortes del DENUE.

In [328]:
# ============================================================
# FASE 3 - CIERRE FINAL
# CONSOLIDACIÓN DE RUPTURAS 2019 Y 2024
# ============================================================

# ------------------------------------------------------------
# 1. RESUMEN COMPARATIVO
# ------------------------------------------------------------

resumen_rupturas = pd.DataFrame([
    {
        "Transicion": "2018-2019",
        "SCIAN": "812110",
        "Establecimientos_inicial": len(d18),
        "Establecimientos_final": len(d19),
        "Cambio_neto": len(d19) - len(d18),

        "IDs_presentes_en_ambos": len(comunes),

        "Exclusivos_inicial_raw": len(solo18),
        "Exclusivos_final_raw": len(solo19),

        "Posibles_cambios_ID":
            len(pares_cambio_id)
            + len(pares_reducidos),

        "Exclusivos_inicial_final":
            len(solo18_final_df),

        "Exclusivos_final_final":
            len(solo19_final_df),

        "Fecha_alta_corte": "2019-11",

        "Exclusivos_con_fecha_corte":
            int(
                solo19_final_df[fecha19]
                .astype(str)
                .str.strip()
                .eq("2019-11")
                .sum()
            )
    },

    {
        "Transicion": "2023-2024",
        "SCIAN": "812110",
        "Establecimientos_inicial": len(d23),
        "Establecimientos_final": len(d24),
        "Cambio_neto": len(d24) - len(d23),

        "IDs_presentes_en_ambos":
            len(comunes_23_24),

        "Exclusivos_inicial_raw":
            len(solo23),

        "Exclusivos_final_raw":
            len(solo24),

        "Posibles_cambios_ID":
            len(pares_reducidos_23_24),

        "Exclusivos_inicial_final":
            len(solo23_final_df),

        "Exclusivos_final_final":
            len(solo24_final_df),

        "Fecha_alta_corte": "2024-11",

        "Exclusivos_con_fecha_corte":
            int(
                solo24_final_df[fecha24]
                .astype(str)
                .str.strip()
                .eq("2024-11")
                .sum()
            )
    }
])

resumen_rupturas[
    "Pct_exclusivos_con_fecha_corte"
] = (
    resumen_rupturas[
        "Exclusivos_con_fecha_corte"
    ]
    /
    resumen_rupturas[
        "Exclusivos_final_final"
    ]
    * 100
).round(2)


# ------------------------------------------------------------
# 2. GUARDAR RESULTADOS
# ------------------------------------------------------------

resumen_rupturas.to_csv(
    RESULTADOS /
    "RESUMEN_AUDITORIA_RUPTURAS_812110_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

distribucion_alta_solo19.to_csv(
    RESULTADOS /
    "DISTRIBUCION_FECHA_ALTA_812110_EXCLUSIVOS_2019.csv",
    index=False,
    encoding="utf-8-sig"
)

distribucion_final_2024.to_csv(
    RESULTADOS /
    "DISTRIBUCION_FECHA_ALTA_812110_EXCLUSIVOS_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

revision_3.to_csv(
    RESULTADOS /
    "AUDITORIA_CAMBIOS_ID_812110_2018_2019.csv",
    index=False,
    encoding="utf-8-sig"
)

pares_reducidos_23_24.to_csv(
    RESULTADOS /
    "AUDITORIA_CAMBIOS_ID_812110_2023_2024.csv",
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. CONTROLES FINALES
# ------------------------------------------------------------

print("=" * 70)
print("CIERRE FASE 3 - AUDITORÍA DE RUPTURAS")
print("=" * 70)

display(resumen_rupturas)

print("\nCONTROLES:")

print(
    "2019 balance:",
    len(solo19_final_df)
    - len(solo18_final_df),
    "| cambio observado:",
    len(d19) - len(d18)
)

print(
    "2024 balance:",
    len(solo24_final_df)
    - len(solo23_final_df),
    "| cambio observado:",
    len(d24) - len(d23)
)

print(
    "\n2019 % exclusivos con fecha del corte:",
    resumen_rupturas.loc[
        resumen_rupturas["Transicion"] == "2018-2019",
        "Pct_exclusivos_con_fecha_corte"
    ].iloc[0],
    "%"
)

print(
    "2024 % exclusivos con fecha del corte:",
    resumen_rupturas.loc[
        resumen_rupturas["Transicion"] == "2023-2024",
        "Pct_exclusivos_con_fecha_corte"
    ].iloc[0],
    "%"
)

print("\n✓ Archivos finales guardados.")
print("✓ Ruptura 2019 auditada.")
print("✓ Ruptura 2024 auditada.")
print("✓ FASE 3 CERRADA.")

CIERRE FASE 3 - AUDITORÍA DE RUPTURAS


,Transicion,SCIAN,Establecimientos_inicial,Establecimientos_final,Cambio_neto,IDs_presentes_en_ambos,Exclusivos_inicial_raw,Exclusivos_final_raw,Posibles_cambios_ID,Exclusivos_inicial_final,Exclusivos_final_final,Fecha_alta_corte,Exclusivos_con_fecha_corte,Pct_exclusivos_con_fecha_corte
0,2018-2019,812110,1767,2328,561,935,832,1393,3,829,1390,2019-11,1363,98.06
1,2023-2024,812110,2334,3128,794,1390,944,1738,1,943,1737,2024-11,1688,97.18



CONTROLES:
2019 balance: 561 | cambio observado: 561
2024 balance: 794 | cambio observado: 794

2019 % exclusivos con fecha del corte: 98.06 %
2024 % exclusivos con fecha del corte: 97.18 %

✓ Archivos finales guardados.
✓ Ruptura 2019 auditada.
✓ Ruptura 2024 auditada.
✓ FASE 3 CERRADA.
